# Nucleotide Transformer v2 50M — DIMER E2E promoter fine-tuning tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-nucleotide--transformer--genomics--pipeline-181717?logo=github)](https://github.com/kurtvalcorza/nucleotide-transformer-genomics-pipeline) [![Model card](https://img.shields.io/badge/Model%20card-MODEL__CARD.md-blue)](https://github.com/kurtvalcorza/nucleotide-transformer-genomics-pipeline/blob/main/MODEL_CARD.md) [![Licence](https://img.shields.io/badge/weights-CC%20BY--NC--SA%204.0-orange)](https://creativecommons.org/licenses/by-nc-sa/4.0/) [![Upstream](https://img.shields.io/badge/Hugging%20Face-InstaDeepAI%2Fnucleotide--transformer--v2--50m--multi--species-yellow)](https://huggingface.co/InstaDeepAI/nucleotide-transformer-v2-50m-multi-species)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** DNA sequence representation (one 512-d mean-pooled vector per sequence) and bounded supervised fine-tuning of a promoter classifier — a mean-pooled head plus the last two encoder blocks — on labelled `{id, sequence, label}` records, using the pinned `InstaDeepAI/nucleotide-transformer-v2-50m-multi-species` weights (CC BY-NC-SA 4.0)

**This notebook is standalone.** It carries the repository's package (3 modules under `src/nucleotide_transformer_genomics_pipeline/`, at revision `a24afa29a199`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `81b29e5786726d891dbf929404ef20adca5b36f1` (~224 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned `InstaDeepAI/nucleotide-transformer-v2-50m-multi-species` snapshot (a 224 MB `model.safetensors` and the two model-code files, all re-hashed before the code is imported), reads the 2,200-sequence human promoter sample that is carried **inline** in the package (no data download at all), validates and splits it 1,600 / 200 / 400 without leakage, embeds a few sequences and predicts a masked 6-mer through the inference contract with an input manifest and a rejection probe, scores two non-neural baselines and a logistic probe on the **frozen** embeddings over the 400 held-out sequences, runs a bounded fine-tuning of the mean-pooled head plus the last two encoder blocks with cross-entropy and epoch selection on validation MCC, scores the held-out sequences again, exports the adapter as safetensors with a manifest, and reloads that artifact into a fresh pipeline to verify prediction parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5). On an RTX 5070 Ti the fine-tuning took 34 s and the whole path about two minutes after the snapshot download; every sequence is 47 tokens long, so CPU is workable — a CUDA runtime is used automatically when present.

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4 and re-run from that cell to upload one CSV with a header naming `sequence` and `label` (0/1; an optional `id`) — at least eight sequences of 12..6,000 A/C/G/T/N bases with both labels present. The records pass through the same validation, sequence-disjoint split, baselines, frozen probe, fine-tuning, held-out evaluation, artifact export and reload-parity cells as the promoter sample. Uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

`InstaDeepAI/nucleotide-transformer-v2-50m-multi-species` is the smallest Nucleotide Transformer v2 (Dalla-Torre et al., 2024): a 12-block, 512-wide encoder with rotary positions and a bias-free SwiGLU feed-forward, pre-trained by masked 6-mer prediction on 850 genomes (55,904,972 parameters as loaded with its masked-LM head; the encoder alone is 53,534,401). Its weights are published under **CC BY-NC-SA 4.0** — attribution, **non-commercial**, share-alike — and every embedding, probe and adapter below inherits those conditions. The checkpoint cannot be loaded by the native Transformers ESM classes (its `config.json` carries an `auto_map` and the native block has no bias-free SwiGLU), so it ships its own `modeling_esm.py`; Section 3 verifies that file's digest before it is imported.

What this notebook adds to representation is **adaptation with labelled sequences**. The task is the human non-TATA promoter benchmark of Genomic Benchmarks: 251-base windows of the GRCh38 reference that either centre on a promoter from the Eukaryotic Promoter Database or contain none. Promoters are GC-rich, so the composition-only baseline is not trivial — a single GC threshold fitted on the training split scores **0.715** accuracy / MCC 0.431 on the 400 held-out windows — and a logistic probe on the *frozen* mean-pooled embeddings already reaches 0.815 / 0.632. So the honest question is narrow: does a bounded fine-tuning of a small mean-pooled head plus the last two encoder blocks (8,660,482 of 53.8 M parameters) on 1,600 windows move the held-out **MCC** past the frozen probe and both baselines? The build record measured 0.8425 / **0.693** — a gain of 0.06 MCC over the frozen probe on one seeded draw. Nothing here is a claim about your sequences: it is one split of one benchmark's labelling convention.

**Snapshot note:** the pinned revision ships `model.safetensors` (an 8-file manifest: weights, config, tokenizer files, README and the two model-code files); the upstream repository also hosts a `pytorch_model.bin` and a JAX checkpoint, which are not in the manifest and are never staged or loaded, so no pickle is opened anywhere. The adapter written in Section 9 is safetensors too.

**Learning objectives:** install the pinned runtime; read what the carried package guarantees, including where its remote-code perimeter begins and ends; stage and digest-verify the immutable upstream snapshot; read a digest-pinned, licence-traced promoter sample carried inline, validate it and split it without leakage; embed sequences and predict a masked 6-mer through the public API and read the output contract correctly (a representation is not a prediction; the masked distribution is the pre-training objective, not a classifier); measure two non-neural baselines and a logistic probe on the frozen embeddings; run a bounded fine-tuning of a mean-pooled head plus the last two blocks with a stated loss, explicit hyperparameters and validation-based epoch selection; evaluate on a sequence-disjoint test split with accuracy and MCC; and export a safetensors adapter that reloads against the pinned base with verified parity and carries the base licence with it.

**This notebook does not demonstrate:** multi-class or multi-label genomics tasks, token-level prediction (splice sites, variant effects), sequences longer than 6,000 bases, generation, fine-tuning of the embeddings or of the first ten encoder blocks, LoRA, the other seventeen tasks of the Nucleotide Transformer benchmark or any benchmark proper (only one seeded 2,200-sequence draw of one Genomic Benchmarks task is scored here), calibration of the class probabilities, and any claim that a promoter classifier transfers to other regulatory elements or other species. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU (float32) and uses CUDA automatically when available. Every sequence is 251 bases = 47 tokens, so the model is cheap per step: the build record measured 3 s for the frozen probe and 34 s for the six-epoch fine-tuning with per-epoch validation on an RTX 5070 Ti (558 MiB peak), and 213 s for the same fine-tuning on the build workstation's CPU (19 s for the probe). Expect a few tens of minutes on a 2-vCPU hosted runtime. The pinned `torch==2.14.0` install and the 224 MB checkpoint are the large downloads of the run; the data is carried inline.
- **Knowledge:** basic Python; what a DNA sequence, GC content and a promoter are; what accuracy and Matthews correlation measure and why MCC is 0 for a constant predictor; why a probe on frozen embeddings and a fine-tuned model answer different questions.
- **Data contract:** records are `{id, sequence, label}` — `sequence` a string of A/C/G/T/N of 12..6,000 bases (case-insensitive; `N` is tokenised base by base), `label` 0 or 1, `id` matching `[A-Za-z0-9_.:-]{1,64}` and unique; a dataset needs 8..20,000 records and a training split needs both labels; splitting de-duplicates by exact sequence so no sequence lands in two splits. BYOD accepts one CSV with a header naming `sequence` and `label` (and optionally `id`).
- **Validation is structural, not biological:** every sequence is checked for alphabet, length and label, but nothing checks that a label is right or that a window is a promoter — a mislabelled set is fine-tuned on without complaint.
- **Licence:** the weights are **CC BY-NC-SA 4.0**. Copying, redistribution and adaptation are permitted for non-commercial purposes with attribution and under the same licence; **commercial use is not permitted under this licence**. The adapter this notebook writes is a derivative of the weights and carries the same conditions, recorded in its manifest.
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there. The default path uploads nothing.
- **External access (data):** none. The 2,200 sequences are carried inline in the carried `samples.py` with their digest. Their provenance is recorded there: interval lists from `ML-Bioinfo-CEITEC/genomic_benchmarks` (Apache-2.0) at commit `605d8539…` (four gzipped CSVs pinned by size and SHA-256) and bases from the GRCh38 human reference genome (Genome Reference Consortium, public domain) read through Ensembl REST; `tools/pin_sample.py` in the repository reproduces the draw from those sources and asserts the digest. Nothing is redistributed beyond 2,200 windows of a public-domain reference.
- **External access:** the Hugging Face Hub only, to fetch the pinned `InstaDeepAI/nucleotide-transformer-v2-50m-multi-species` snapshot (~224 MB in total) at revision `81b29e578672…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's pyproject.toml at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'safetensors==0.8.0',
    'huggingface-hub==0.36.2',
    'numpy==2.5.3',
]
NOTEBOOK_SOURCE = {
    'repository': 'nucleotide-transformer-genomics-pipeline',
    'repository_revision': 'a24afa29a199e5703b51bf6b22e4b143871a8b31',
    'embedded_module': 'src/nucleotide_transformer_genomics_pipeline/pipeline.py',
    'embedded_modules': ['src/nucleotide_transformer_genomics_pipeline/metrics.py', 'src/nucleotide_transformer_genomics_pipeline/pipeline.py', 'src/nucleotide_transformer_genomics_pipeline/samples.py'],
    'module_sha256': '6ca7d1d36d9248efa7c6979386bc74842869c63f0d8535fb311832d489c769af',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/nucleotide_transformer_genomics_pipeline/` @ `a24afa29a199`)

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/nucleotide_transformer_genomics_pipeline/metrics.py`

In [ ]:
"""Binary sequence-classification measures and two non-neural baselines, in plain Python.

``classification_metrics`` scores predicted labels against gold labels: accuracy, Matthews correlation
(MCC, the headline metric — it is 0 for any constant or chance predictor and symmetric in the two
classes), per-class precision / recall / F1 and the confusion counts. ``majority_baseline`` predicts the
training majority class; ``gc_threshold_baseline`` predicts from GC content alone with the threshold and
direction chosen on the training split. Promoters are GC-rich, so the second baseline is the honest
composition-only reference any sequence model must beat.
"""

from __future__ import annotations

# ruff: noqa: E501  -- the inline sample block and contract lines are kept at the fleet width
import math
from collections.abc import Mapping, Sequence
from typing import Any

LABELS = (0, 1)


def gc_content(sequence: str) -> float:
    """Fraction of G and C bases (case-insensitive); ``N`` counts toward the length, not the GC total."""
    upper = sequence.upper()
    return (upper.count("G") + upper.count("C")) / len(upper) if upper else 0.0


def classification_metrics(predictions: Sequence[int], gold: Sequence[int]) -> dict[str, Any]:
    """Accuracy, MCC, per-class precision/recall/F1 and the confusion counts for binary labels."""
    if len(predictions) != len(gold):
        raise ValueError(f"{len(predictions)} predictions for {len(gold)} labels")
    if not gold:
        raise ValueError("no records to score")
    for value in (*predictions, *gold):
        if value not in LABELS:
            raise ValueError(f"labels must be 0 or 1, got {value!r}")
    tp = sum(1 for p, g in zip(predictions, gold, strict=True) if p == 1 and g == 1)
    tn = sum(1 for p, g in zip(predictions, gold, strict=True) if p == 0 and g == 0)
    fp = sum(1 for p, g in zip(predictions, gold, strict=True) if p == 1 and g == 0)
    fn = sum(1 for p, g in zip(predictions, gold, strict=True) if p == 0 and g == 1)
    denominator = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    mcc = (tp * tn - fp * fn) / denominator if denominator else 0.0

    def _prf(t: int, f_pos: int, f_neg: int) -> dict[str, float]:
        precision = t / (t + f_pos) if t + f_pos else 0.0
        recall = t / (t + f_neg) if t + f_neg else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        return {"precision": round(precision, 4), "recall": round(recall, 4), "f1": round(f1, 4)}

    return {
        "n": len(gold),
        "accuracy": round((tp + tn) / len(gold), 4),
        "mcc": round(mcc, 4),
        "positive": _prf(tp, fp, fn),
        "negative": _prf(tn, fn, fp),
        "confusion": {"tp": tp, "tn": tn, "fp": fp, "fn": fn},
    }


def majority_baseline(train: Sequence[Mapping[str, Any]], test: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """Predict the training majority label (ties go to 1) for every test record."""
    positives = sum(int(r["label"]) for r in train)
    label = 1 if positives * 2 >= len(train) else 0
    scored = classification_metrics([label] * len(test), [int(r["label"]) for r in test])
    return {"baseline": "majority", "label": label, **scored}


def gc_threshold_baseline(train: Sequence[Mapping[str, Any]], test: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """Predict from GC content alone: the midpoint threshold and direction that maximise training accuracy."""
    train_gc = [(gc_content(r["sequence"]), int(r["label"])) for r in train]
    observed = sorted({gc for gc, _ in train_gc})
    # thresholds halfway between neighbouring observed values, plus one below and one above the range
    candidates = [observed[0] - 1e-9, *((a + b) / 2 for a, b in zip(observed, observed[1:], strict=False)), observed[-1] + 1e-9]
    best = (-1.0, candidates[0], 1)
    for threshold in candidates:
        for direction in (1, -1):  # 1: GC >= threshold -> positive; -1: GC >= threshold -> negative
            correct = sum(1 for gc, label in train_gc if (int(gc >= threshold) if direction == 1 else int(gc < threshold)) == label)
            accuracy = correct / len(train_gc)
            if accuracy > best[0]:
                best = (accuracy, threshold, direction)
    _, threshold, direction = best
    predictions = [int(gc_content(r["sequence"]) >= threshold) if direction == 1 else int(gc_content(r["sequence"]) < threshold) for r in test]
    scored = classification_metrics(predictions, [int(r["label"]) for r in test])
    return {
        "baseline": "gc_threshold",
        "threshold": round(threshold, 4),
        "rule": "GC >= threshold -> positive" if direction == 1 else "GC < threshold -> positive",
        "train_accuracy": round(best[0], 4),
        **scored,
    }

**Module 2/3:** `src/nucleotide_transformer_genomics_pipeline/pipeline.py` (carried verbatim; see the note above)

In [ ]:
"""DNA sequence representation and bounded promoter fine-tuning over the pinned
``InstaDeepAI/nucleotide-transformer-v2-50m-multi-species`` checkpoint.

**Remote code, inside a verified perimeter.** This checkpoint cannot be loaded by the native Transformers ESM
classes: its ``config.json`` carries an ``auto_map`` and its feed-forward block is a bias-free SwiGLU the native
implementation does not have, so ``trust_remote_code=True`` is an architectural requirement. The two Python
files the loader executes (``modeling_esm.py``, ``esm_config.py``) are entries of the snapshot manifest, so
``from_pretrained`` refuses to run without a manifest, stages any absent file at the pinned revision, re-hashes
every file (``verify_snapshot``) and only then imports the model code. The tokenizer is the native
``EsmTokenizer`` and loads with ``trust_remote_code=False``. Digest verification proves the executed code is
the pinned upstream code byte for byte; it is not a safety claim about that code.

Weights are **CC BY-NC-SA 4.0** (attribution, non-commercial, share-alike). Everything derived from them -
embeddings, probes, adapters saved by ``save_artifact`` - carries the same conditions.

Two capabilities: ``embed`` (mean-pooled 512-d representations; the ``TASK-INFERENCE`` contract) and, after
``adapt`` or ``load_artifact``, ``predict`` / ``evaluate`` (binary sequence classification through a small
mean-pooled head trained together with the last encoder blocks). ``linear_probe`` scores the frozen model
on the same task without touching its weights - the honest reference an adaptation has to beat.
"""

from __future__ import annotations

# ruff: noqa: E501  -- adaptation-contract lines are kept at the fleet width
import hashlib
import json
import random
import time
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

MODEL_ID = "InstaDeepAI/nucleotide-transformer-v2-50m-multi-species"
MODEL_REVISION = "81b29e5786726d891dbf929404ef20adca5b36f1"
MODEL_LICENSE = "cc-by-nc-sa-4.0"
MODEL_KEY = "nt-v2-50m-multi-species"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"
WEIGHT_FILE = "model.safetensors"
REMOTE_CODE_FILES = ("modeling_esm.py", "esm_config.py")  # executed by the loader; manifest entries, verified before import
HIDDEN_SIZE = 512
NUM_LAYERS = 12
KMER = 6
MAX_TOKENS = 1_000  # the length the checkpoint was trained at (6,000 bases as 6-mers)
MAX_BASES = KMER * MAX_TOKENS
MIN_BASES = 12
MAX_SEQUENCES = 20_000  # per call
DEFAULT_BATCH_SIZE = 16
PARAMETER_COUNT = 55_904_972  # EsmForMaskedLM as loaded (encoder + masked-LM head)
ENCODER_PARAMETERS = 53_534_401  # the `esm` encoder the head reads
DEFAULT_TRAINED_LAYERS = 2  # last encoder blocks trained by `adapt` alongside the head
HEAD_HIDDEN = 512
NUM_LABELS = 2
LABEL_NAMES = ("no_promoter", "promoter")
ARTIFACT_FORMAT = f"org.valcorza.{MODEL_KEY}.adapter.v1"
ARTIFACT_VERSION = "1.0"
ADAPTER_WEIGHTS = "adapter.safetensors"
ADAPTER_MANIFEST = "manifest.json"
HEAD_PREFIX = "head."
MIN_SCORED_RECORDS = 50  # below this a scored set is labelled a small sample
MAX_EVAL_RECORDS = 20_000
POOLING = "mean of the last hidden state over non-padding tokens"


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _read_manifest(root: Path) -> dict[str, Any]:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    listed = {entry["path"] for entry in manifest["files"]}
    missing = [name for name in (WEIGHT_FILE, *REMOTE_CODE_FILES) if name not in listed]
    if missing:
        raise ValueError(f"manifest does not list {missing}; the remote code must be inside the verified perimeter")
    return manifest


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch. The manifest must
    list the weight file and both remote-code files, so a passing check means the code about to be imported
    is byte-identical to the pinned revision."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root)
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "remote_code_files": list(REMOTE_CODE_FILES),
        "total_bytes": manifest.get("totalBytes"),
    }


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights and the model code). Returns the relative paths fetched; `verify_snapshot`
    still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root)
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def validate_sequences(sequences: Any) -> list[str]:
    """Upper-cased A/C/G/T/N strings within the base limits; raises ValueError before any model import."""
    if isinstance(sequences, str | bytes) or not isinstance(sequences, Sequence):
        raise ValueError("sequences must be a list of str")
    if not 1 <= len(sequences) <= MAX_SEQUENCES:
        raise ValueError(f"{len(sequences)} sequences; 1..{MAX_SEQUENCES} per call")
    checked = []
    for index, sequence in enumerate(sequences):
        if not isinstance(sequence, str):
            raise ValueError(f"sequences[{index}] must be a str")
        upper = sequence.strip().upper()
        if not MIN_BASES <= len(upper) <= MAX_BASES:
            raise ValueError(f"sequences[{index}]: {len(upper)} bases; {MIN_BASES}..{MAX_BASES} are required")
        bad = sorted(set(upper) - set("ACGTN"))
        if bad:
            raise ValueError(f"sequences[{index}]: characters outside A/C/G/T/N: {bad[:5]}")
        checked.append(upper)
    return checked


def validate_inputs(sequences: Sequence[str], *, names: Sequence[str] | None = None) -> dict[str, Any]:
    """The input manifest the tutorial records before inference (DAT24): counts, lengths, GC, N content."""
    checked = validate_sequences(sequences)
    if names is not None and len(names) != len(checked):
        raise ValueError(f"{len(names)} names for {len(checked)} sequences")
    lengths = [len(s) for s in checked]
    gc = [(s.count("G") + s.count("C")) / len(s) for s in checked]
    return {
        "n_sequences": len(checked),
        "bases": {"min": min(lengths), "max": max(lengths), "total": sum(lengths)},
        "gc_fraction": {"min": round(min(gc), 4), "max": round(max(gc), 4)},
        "n_with_N": sum("N" in s for s in checked),
        "tokenization": f"{KMER}-mers over the checkpoint vocabulary, single bases around N and at a trailing remainder; ceiling {MAX_TOKENS} tokens = {MAX_BASES} bases",
        "names": list(names) if names is not None else None,
        "sequence_sha256": [hashlib.sha256(s.encode()).hexdigest() for s in checked],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    adapted: Mapping[str, Any],
    frozen: Mapping[str, Any],
    baselines: Sequence[Mapping[str, Any]],
    *,
    sample_kind: str = "pinned promoter sample",
) -> dict[str, Any]:
    """Structured verdict (EVAL21): the adapted model against the frozen probe and the non-neural baselines
    on the same held-out records, with the small-sample flag and no claim beyond the numbers."""
    n = int(adapted["n"])
    if n != int(frozen["n"]) or any(int(b["n"]) != n for b in baselines):
        raise ValueError("adapted, frozen and baseline metrics must score the same records")
    best_baseline = max(baselines, key=lambda b: b["mcc"])
    return {
        "n": n,
        "sample_kind": sample_kind,
        "small_sample": n < MIN_SCORED_RECORDS,
        "headline_metric": "mcc",
        "adapted": {"accuracy": adapted["accuracy"], "mcc": adapted["mcc"]},
        "frozen_probe": {"accuracy": frozen["accuracy"], "mcc": frozen["mcc"]},
        "best_baseline": {"name": best_baseline.get("baseline"), "accuracy": best_baseline["accuracy"], "mcc": best_baseline["mcc"]},
        "adapted_beats_frozen": adapted["mcc"] > frozen["mcc"],
        "adapted_beats_baselines": all(adapted["mcc"] > b["mcc"] for b in baselines),
        "frozen_beats_baselines": all(frozen["mcc"] > b["mcc"] for b in baselines),
        "mcc_gain_over_frozen": round(adapted["mcc"] - frozen["mcc"], 4),
        "note": "sample-sanity evidence on one seeded draw with no dispersion estimate; not a benchmark reproduction",
    }


def _trainable_names(model: Any, layers: int) -> list[str]:
    """Encoder tensors `adapt` trains: the last `layers` blocks of the 12-block encoder (0 = head only)."""
    if isinstance(layers, bool) or not isinstance(layers, int) or not 0 <= layers <= NUM_LAYERS:
        raise ValueError(f"layers must be an int in 0..{NUM_LAYERS}")
    wanted = {f"esm.encoder.layer.{NUM_LAYERS - 1 - k}." for k in range(layers)}
    names = [name for name, _ in model.named_parameters() if any(name.startswith(prefix) for prefix in wanted)]
    if layers == NUM_LAYERS:  # the whole encoder also trains its final layer norm (embeddings stay frozen)
        names += [name for name, _ in model.named_parameters() if name.startswith("esm.encoder.emb_layer_norm_after")]
    return names


def _check_artifact_manifest(manifest: Mapping[str, Any], artifact_dir: Path, base_sha256: str) -> None:
    if manifest.get("format") != ARTIFACT_FORMAT:
        raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
    base = manifest.get("base", {})
    if (base.get("model_id"), base.get("revision")) != (MODEL_ID, MODEL_REVISION):
        raise ValueError("artifact was trained on a different base model or revision")
    if base_sha256 and base.get("weight_sha256") and base["weight_sha256"] != base_sha256:
        raise ValueError("artifact base weight digest does not match the loaded snapshot")
    if manifest.get("license") != MODEL_LICENSE:
        raise ValueError(f"artifact licence {manifest.get('license')!r} != {MODEL_LICENSE!r}; the adapter inherits the base licence")
    files = {entry["path"]: entry for entry in manifest.get("files", [])}
    if ADAPTER_WEIGHTS not in files:
        raise ValueError(f"artifact manifest does not list {ADAPTER_WEIGHTS}")
    weights = artifact_dir / ADAPTER_WEIGHTS
    if not weights.is_file():
        raise FileNotFoundError(f"artifact weights missing: {weights}")
    if weights.stat().st_size != files[ADAPTER_WEIGHTS]["bytes"]:
        raise ValueError("artifact weights size does not match the manifest")
    if _sha256(weights) != files[ADAPTER_WEIGHTS]["sha256"]:
        raise ValueError("artifact weights digest does not match the manifest")


def _build_head(seed: int) -> Any:
    """The mean-pooled classification head, initialised from a fixed seed so two fresh pipelines agree."""
    import torch

    class PromoterHead(torch.nn.Module):
        def __init__(self) -> None:
            super().__init__()
            self.dense = torch.nn.Linear(HIDDEN_SIZE, HEAD_HIDDEN)
            self.out = torch.nn.Linear(HEAD_HIDDEN, NUM_LABELS)

        def forward(self, hidden: Any, attention_mask: Any) -> Any:
            mask = attention_mask.unsqueeze(-1).to(hidden.dtype)
            pooled = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1.0)
            return self.out(torch.tanh(self.dense(pooled)))

    generator_state = torch.random.get_rng_state()
    try:
        torch.manual_seed(seed)
        head = PromoterHead()
    finally:
        torch.random.set_rng_state(generator_state)
    return head


@dataclass
class NucleotideTransformerPipeline:
    """Sequence representations (`embed`) and, once adapted, promoter classification (`predict`) over the
    pinned Nucleotide Transformer v2 50M checkpoint loaded through its digest-verified remote code."""

    device: str
    _model: Any = field(default=None, repr=False)
    _tokenizer: Any = field(default=None, repr=False)
    _head: Any = field(default=None, repr=False)
    weight_sha256: str | None = None
    adapter: dict[str, Any] | None = None
    remote_code_executed: bool = True

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> NucleotideTransformerPipeline:
        """Stage (if allowed) and verify the snapshot, then load the tokenizer natively and the model through
        the verified remote code. There is no manifest-less path: without a digest check nothing is imported."""
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        stage_missing_files(root, allow_download=allow_download)
        snapshot = verify_snapshot(root)  # every file, including the two .py files, before any import
        with open(root / MANIFEST_NAME, encoding="utf-8") as handle:
            entries = json.load(handle).get("files", [])
        weight_sha256 = next((e["sha256"] for e in entries if e["path"] == WEIGHT_FILE), None)

        import torch
        from transformers import AutoModelForMaskedLM, AutoTokenizer

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        tokenizer = AutoTokenizer.from_pretrained(str(root), local_files_only=True, trust_remote_code=False)
        model = AutoModelForMaskedLM.from_pretrained(str(root), local_files_only=True, trust_remote_code=True, dtype=torch.float32)
        if type(model).__module__.split(".")[-1] != "modeling_esm":
            raise RuntimeError(f"expected the snapshot's modeling_esm module, loaded {type(model).__module__}")
        model = model.to(resolved_device).eval()
        for param in model.parameters():
            param.requires_grad_(False)
        pipe = cls(resolved_device, model, tokenizer, None, weight_sha256)
        pipe.snapshot = snapshot  # type: ignore[attr-defined]
        return pipe

    # ------------------------------------------------------------------ representation (TASK-INFERENCE)

    def _require_model(self) -> tuple[Any, Any]:
        if self._model is None or self._tokenizer is None:
            raise RuntimeError("no model loaded: construct with from_pretrained or from_artifact")
        return self._model, self._tokenizer

    def _encode(self, sequences: Sequence[str]) -> Any:
        _model, tokenizer = self._require_model()
        return tokenizer(list(sequences), return_tensors="pt", padding=True).to(self.device)

    def _hidden(self, encoded: Any) -> Any:
        model, _tokenizer = self._require_model()
        return model.esm(input_ids=encoded["input_ids"], attention_mask=encoded["attention_mask"]).last_hidden_state

    def token_counts(self, sequences: Sequence[str]) -> list[int]:
        """Tokens per sequence (6-mers plus single-base fallbacks, with the special tokens)."""
        checked = validate_sequences(sequences)
        _model, tokenizer = self._require_model()
        return [len(tokenizer(s)["input_ids"]) for s in checked]

    def embed(self, sequences: Sequence[str], *, batch_size: int = DEFAULT_BATCH_SIZE) -> list[list[float]]:
        """One mean-pooled 512-d vector per sequence (padding excluded) from the frozen encoder."""
        checked = validate_sequences(sequences)
        self._require_model()
        import torch

        vectors: list[list[float]] = []
        with torch.no_grad():  # not inference_mode: the remote code caches rotary tables on first use, and inference tensors cannot feed a later backward pass
            for start in range(0, len(checked), batch_size):
                encoded = self._encode(checked[start : start + batch_size])
                hidden = self._hidden(encoded)
                mask = encoded["attention_mask"].unsqueeze(-1).to(hidden.dtype)
                pooled = (hidden * mask).sum(1) / mask.sum(1)
                vectors.extend(pooled.float().cpu().tolist())
        return vectors

    def predict_masked(self, sequence: str, *, top_k: int = 5) -> dict[str, Any]:
        """Masked-token prediction for one sequence containing the tokenizer's mask token (the checkpoint's
        pre-training objective); returns the top-k tokens and probabilities at the masked position."""
        model, tokenizer = self._require_model()
        if not isinstance(sequence, str) or tokenizer.mask_token not in sequence:
            raise ValueError(f"sequence must contain the mask token {tokenizer.mask_token!r}")
        import torch

        encoded = tokenizer(sequence, return_tensors="pt").to(self.device)
        position = (encoded["input_ids"][0] == tokenizer.mask_token_id).nonzero()
        if len(position) != 1:
            raise ValueError("exactly one mask token is required")
        with torch.no_grad():  # not inference_mode: the remote code caches rotary tables on first use, and inference tensors cannot feed a later backward pass
            logits = model(**encoded).logits[0, int(position[0, 0])]
        probabilities = torch.softmax(logits.float(), dim=-1)
        values, indices = probabilities.topk(top_k)
        return {
            "tokens": tokenizer.convert_ids_to_tokens(indices.tolist()),
            "probabilities": [round(float(v), 6) for v in values.tolist()],
            "position": int(position[0, 0]),
        }

    # ------------------------------------------------------------------ frozen reference

    def linear_probe(
        self,
        train: Sequence[Mapping[str, Any]],
        test: Sequence[Mapping[str, Any]],
        *,
        l2: float = 1e-2,
        batch_size: int = DEFAULT_BATCH_SIZE,
    ) -> dict[str, Any]:
        """Logistic regression on standardised frozen mean-pooled embeddings (L-BFGS, L2-penalised, no seed
        dependence): the model's weights are untouched, so this is what the pre-trained representation alone
        knows about the task. Returns the test metrics plus the fitted probe's train accuracy."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import classification_metrics` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        train_checked = validate_dataset(train, require_both_labels=True)["records"]
        test_checked = validate_dataset(test, min_records=1)["records"]
        self._require_model()
        import torch

        started = time.perf_counter()
        x_train = torch.tensor(self.embed([r["sequence"] for r in train_checked], batch_size=batch_size))
        x_test = torch.tensor(self.embed([r["sequence"] for r in test_checked], batch_size=batch_size))
        mean, std = x_train.mean(0), x_train.std(0) + 1e-6
        x_train, x_test = (x_train - mean) / std, (x_test - mean) / std
        y_train = torch.tensor([float(r["label"]) for r in train_checked])
        weight = torch.zeros(HIDDEN_SIZE, requires_grad=True)
        bias = torch.zeros(1, requires_grad=True)
        optimizer = torch.optim.LBFGS([weight, bias], max_iter=300, line_search_fn="strong_wolfe")

        def closure() -> Any:
            optimizer.zero_grad()
            loss = torch.nn.functional.binary_cross_entropy_with_logits(x_train @ weight + bias, y_train) + l2 * (weight * weight).sum()
            loss.backward()
            return loss

        optimizer.step(closure)
        with torch.no_grad():
            train_predictions = ((x_train @ weight + bias) > 0).int().tolist()
            test_predictions = ((x_test @ weight + bias) > 0).int().tolist()
        scored = classification_metrics(test_predictions, [r["label"] for r in test_checked])
        return {
            "probe": "logistic regression on frozen mean-pooled embeddings",
            "l2": l2,
            "train_accuracy": classification_metrics(train_predictions, [r["label"] for r in train_checked])["accuracy"],
            "seconds": round(time.perf_counter() - started, 3),
            **scored,
        }

    # ------------------------------------------------------------------ classification (after adapt)

    def predict(self, sequences: Sequence[str], *, batch_size: int = DEFAULT_BATCH_SIZE) -> dict[str, Any]:
        """Labels and class probabilities for sequences; requires an adapter (`adapt` or `load_artifact`)."""
        checked = validate_sequences(sequences)
        self._require_model()
        if self._head is None or self.adapter is None:
            raise RuntimeError("no promoter head: call adapt() or load an artifact before predict()")
        import torch

        labels: list[int] = []
        probabilities: list[list[float]] = []
        self._head.eval()
        with torch.no_grad():  # not inference_mode: the remote code caches rotary tables on first use, and inference tensors cannot feed a later backward pass
            for start in range(0, len(checked), batch_size):
                encoded = self._encode(checked[start : start + batch_size])
                logits = self._head(self._hidden(encoded), encoded["attention_mask"])
                probs = torch.softmax(logits.float(), dim=-1)
                labels.extend(int(v) for v in probs.argmax(-1).tolist())
                probabilities.extend([round(float(p), 6) for p in row] for row in probs.tolist())
        return {"labels": labels, "label_names": [LABEL_NAMES[v] for v in labels], "probabilities": probabilities, "n": len(checked)}

    def evaluate(
        self,
        records: Sequence[Mapping[str, Any]],
        *,
        batch_size: int = DEFAULT_BATCH_SIZE,
        progress: Callable[[int, int], None] | None = None,
    ) -> dict[str, Any]:
        """Classification metrics of the adapted model over labelled records (validated first)."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import classification_metrics` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
        predictions: list[int] = []
        for start in range(0, len(checked), batch_size):
            predictions.extend(self.predict([r["sequence"] for r in checked[start : start + batch_size]], batch_size=batch_size)["labels"])
            if progress is not None:
                progress(min(start + batch_size, len(checked)), len(checked))
        return classification_metrics(predictions, [r["label"] for r in checked])

    def adapt(
        self,
        train: Sequence[Mapping[str, Any]],
        val: Sequence[Mapping[str, Any]] | None = None,
        *,
        epochs: int = 6,
        lr: float = 3e-5,
        layers: int = DEFAULT_TRAINED_LAYERS,
        batch_size: int = DEFAULT_BATCH_SIZE,
        seed: int = 0,
        progress: Callable[[Mapping[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded fine-tuning: a fresh mean-pooled head (seeded) plus the last `layers` encoder blocks are
        trained with cross-entropy, AdamW (no weight decay), gradient clipping at 1.0, seeded shuffling and no
        scheduler; embeddings and the other blocks stay frozen. Epoch 0 records the untrained head's
        validation metrics; the epoch with the highest validation MCC (ties: accuracy, then the earlier epoch)
        is kept, or the final one without a validation split. On any exception the frozen weights are
        restored and the previous adapter, if any, is kept."""
        model, _tokenizer = self._require_model()  # refuse before importing torch
        pass  # standalone rewrite (build_notebook.py): `from .metrics import classification_metrics` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if isinstance(epochs, bool) or not isinstance(epochs, int) or not 1 <= epochs <= 50:
            raise ValueError("epochs must be an int in 1..50")
        if not isinstance(lr, int | float) or not 0.0 < float(lr) <= 1e-2:
            raise ValueError("lr must be in (0, 1e-2]")
        if isinstance(batch_size, bool) or not isinstance(batch_size, int) or not 1 <= batch_size <= 256:
            raise ValueError("batch_size must be an int in 1..256")
        names = _trainable_names(model, layers)
        train_checked = validate_dataset(train, require_both_labels=True)["records"]
        val_checked = validate_dataset(val, min_records=1)["records"] if val is not None else None
        import torch

        name_set = set(names)
        frozen_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in name_set}
        previous_head, previous_adapter = self._head, self.adapter
        cudnn_flags = (torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark)
        torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark = True, False  # repeatable on one device
        history: list[dict[str, Any]] = []
        started = time.perf_counter()

        def _val() -> dict[str, Any] | None:
            if val_checked is None:
                return None
            scored = self.evaluate(val_checked, batch_size=max(batch_size, 32))
            return {"accuracy": scored["accuracy"], "mcc": scored["mcc"], "n": scored["n"]}

        def _score(entry: Mapping[str, Any]) -> tuple[float, float]:
            return (entry["val"]["mcc"], entry["val"]["accuracy"]) if entry["val"] else (0.0, 0.0)

        try:
            head = _build_head(seed).to(self.device)
            self._head = head
            self.adapter = {"provisional": True}  # lets evaluate() run during training
            for param in model.parameters():
                param.requires_grad_(False)
            params = list(head.parameters())
            for name, param in model.named_parameters():
                if name in name_set:
                    param.requires_grad_(True)
                    params.append(param)
            n_trainable = sum(p.numel() for p in params)
            entry = {"epoch": 0, "train_loss": None, "val": _val(), "note": "untrained head on the frozen encoder"}
            history.append(entry)
            if progress is not None:
                progress(entry)
            best_epoch, best_score = 0, _score(entry)
            best_state = ({k: v.detach().clone() for k, v in model.state_dict().items() if k in name_set}, {k: v.detach().clone() for k, v in head.state_dict().items()})
            optimizer = torch.optim.AdamW(params, lr=float(lr), weight_decay=0.0)
            rng = random.Random(seed)
            torch.manual_seed(seed)
            for epoch in range(1, epochs + 1):
                model.train()
                head.train()
                order = list(train_checked)
                rng.shuffle(order)
                losses = []
                for start in range(0, len(order), batch_size):
                    batch = order[start : start + batch_size]
                    encoded = self._encode([r["sequence"] for r in batch])
                    logits = head(self._hidden(encoded), encoded["attention_mask"])
                    targets = torch.tensor([r["label"] for r in batch], device=self.device)
                    loss = torch.nn.functional.cross_entropy(logits, targets)
                    optimizer.zero_grad(set_to_none=True)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(params, 1.0)
                    optimizer.step()
                    losses.append(float(loss.detach()))
                model.eval()
                head.eval()
                entry = {"epoch": epoch, "train_loss": round(sum(losses) / len(losses), 6), "val": _val()}
                history.append(entry)
                if progress is not None:
                    progress(entry)
                if val_checked is None or _score(entry) > best_score:
                    best_epoch, best_score = epoch, _score(entry)
                    best_state = ({k: v.detach().clone() for k, v in model.state_dict().items() if k in name_set}, {k: v.detach().clone() for k, v in head.state_dict().items()})
            model.load_state_dict(best_state[0], strict=False)
            head.load_state_dict(best_state[1])
            for param in model.parameters():
                param.requires_grad_(False)
            for param in head.parameters():
                param.requires_grad_(False)
            model.eval()
            head.eval()
        except BaseException:
            model.load_state_dict(frozen_state, strict=False)
            for param in model.parameters():
                param.requires_grad_(False)
            model.eval()
            self._head, self.adapter = previous_head, previous_adapter
            raise
        finally:
            torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark = cudnn_flags
        train_metrics = classification_metrics(self.predict([r["sequence"] for r in train_checked], batch_size=max(batch_size, 32))["labels"], [r["label"] for r in train_checked])
        self.adapter = {
            "task": "binary promoter classification",
            "pooling": POOLING,
            "layers": layers,
            "trainable_names": names,
            "n_trainable": n_trainable,
            "n_head": sum(p.numel() for p in head.parameters()),
            "n_total": sum(p.numel() for p in model.parameters()) + sum(p.numel() for p in head.parameters()),
            "epochs": epochs,
            "best_epoch": best_epoch,
            "selection": "highest validation MCC (ties: accuracy, earlier epoch)" if val_checked is not None else "final epoch (no validation split)",
            "loss": "cross-entropy over the two labels",
            "lr": float(lr),
            "batch_size": batch_size,
            "seed": seed,
            "n_train": len(train_checked),
            "n_val": len(val_checked) if val_checked is not None else 0,
            "train_accuracy": train_metrics["accuracy"],
            "history": history,
            "seconds": round(time.perf_counter() - started, 3),
        }
        return dict(self.adapter)

    # ------------------------------------------------------------------ artifacts

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Write the head and the trained encoder tensors as safetensors plus a manifest naming the base, the
        digests, the licence the adapter inherits and the training configuration. Requires a prior `adapt`."""
        model, _tokenizer = self._require_model()  # refuse before importing torch
        if self.adapter is None or self._head is None or self.adapter.get("provisional"):
            raise RuntimeError("nothing to save: call adapt() first")
        import torch
        from safetensors.torch import save_file

        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = list(self.adapter["trainable_names"])
        state = model.state_dict()
        tensors = {name: state[name].detach().cpu().contiguous() for name in names}
        tensors.update({HEAD_PREFIX + k: v.detach().cpu().contiguous() for k, v in self._head.state_dict().items()})
        weights = out / ADAPTER_WEIGHTS
        save_file(tensors, str(weights), metadata={"format": "pt"})
        manifest = {
            "format": ARTIFACT_FORMAT,
            "version": ARTIFACT_VERSION,
            "license": MODEL_LICENSE,
            "license_note": "derived from CC BY-NC-SA 4.0 weights: attribution, non-commercial use and share-alike apply to this adapter",
            "base": {"model_id": MODEL_ID, "revision": MODEL_REVISION, "weight_file": WEIGHT_FILE, "weight_sha256": self.weight_sha256, "remote_code_files": list(REMOTE_CODE_FILES)},
            "adapter": {k: v for k, v in self.adapter.items() if k not in ("history", "trainable_names")},
            "history": self.adapter["history"],
            "tensors": sorted(tensors),
            "label_names": list(LABEL_NAMES),
            "files": [{"path": ADAPTER_WEIGHTS, "bytes": weights.stat().st_size, "sha256": _sha256(weights)}],
            "torch": torch.__version__,
            "metadata": dict(metadata or {}),
        }
        with open(out / ADAPTER_MANIFEST, "w", encoding="utf-8") as handle:
            json.dump(manifest, handle, indent=2, ensure_ascii=False)
        return out

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Overlay a saved adapter onto this (freshly loaded) pipeline after checking its manifest, digest,
        licence and exact tensor set. Refuses encoder tensors outside the recorded trained blocks."""
        model, _tokenizer = self._require_model()  # refuse before importing safetensors
        from safetensors.torch import load_file

        artifact = Path(artifact_dir)
        manifest_path = artifact / ADAPTER_MANIFEST
        if not manifest_path.is_file():
            raise FileNotFoundError(f"artifact manifest missing: {manifest_path}")
        with open(manifest_path, encoding="utf-8") as handle:
            manifest = json.load(handle)
        _check_artifact_manifest(manifest, artifact, self.weight_sha256 or "")
        layers = manifest.get("adapter", {}).get("layers")
        expected_encoder = _trainable_names(model, layers)
        head = _build_head(int(manifest.get("adapter", {}).get("seed", 0))).to(self.device)
        expected = sorted(expected_encoder + [HEAD_PREFIX + k for k in head.state_dict()])
        if sorted(manifest["tensors"]) != expected:
            raise ValueError("artifact tensor set does not match its recorded configuration")
        tensors = load_file(str(artifact / ADAPTER_WEIGHTS))
        if sorted(tensors) != expected:
            raise ValueError("artifact tensor names differ from the manifest")
        state = model.state_dict()
        head_state = head.state_dict()
        for name, tensor in tensors.items():
            target = head_state[name[len(HEAD_PREFIX) :]] if name.startswith(HEAD_PREFIX) else state[name]
            if tuple(tensor.shape) != tuple(target.shape):
                raise ValueError(f"artifact tensor {name} has shape {tuple(tensor.shape)}, base has {tuple(target.shape)}")
        model.load_state_dict({k: v.to(state[k].device, state[k].dtype) for k, v in tensors.items() if not k.startswith(HEAD_PREFIX)}, strict=False)
        head.load_state_dict({k[len(HEAD_PREFIX) :]: v.to(self.device) for k, v in tensors.items() if k.startswith(HEAD_PREFIX)})
        for param in head.parameters():
            param.requires_grad_(False)
        model.eval()
        head.eval()
        self._head = head
        self.adapter = {**manifest["adapter"], "trainable_names": expected_encoder, "history": manifest.get("history", [])}
        return dict(self.adapter)

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        *,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> NucleotideTransformerPipeline:
        """Load the verified base snapshot, then overlay the adapter (verified before deserialising)."""
        pipe = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)
        pipe.load_artifact(artifact_dir)
        return pipe

**Module 3/3:** `src/nucleotide_transformer_genomics_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Labelled DNA-sequence datasets for the adaptation contract: the digest-pinned human non-TATA promoter
sample, structural validation of any ``{id, sequence, label}`` dataset, leakage checks, the seeded BYOD
split, and the CSV readers/writers the tutorial uses.

**Where the default sample comes from.** The interval lists are the ``human_nontata_promoters`` benchmark
of Genomic Benchmarks (Grešová et al., BMC Genomic Data 2023), read from the Apache-2.0 repository
``ML-Bioinfo-CEITEC/genomic_benchmarks`` at one immutable commit (four gzipped CSV files, pinned by size
and SHA-256 in ``CORPUS_FILES``). Positives are 251-base windows around human non-TATA promoters from the
Eukaryotic Promoter Database; negatives are 251-base windows of the same genome that contain no promoter.
The bases themselves are the GRCh38 human reference genome (Genome Reference Consortium; public domain),
read interval by interval through the Ensembl REST API (``ENSEMBL_REST``) with the origin's convention:
``-`` strand intervals are reverse-complemented. The sequences of the seeded draw are carried **inline** in
this module (``_INLINE_RECORDS``) so the notebook fetches no data at all, and every one of them was
cross-checked against the benchmark authors' own Hub re-upload when the pin was made
(``tools/pin_sample.py`` reproduces the draw from the origin and asserts ``SAMPLE_DIGEST``).

The draw keeps the origin's train/test separation: the train and validation records come from the origin
``train`` lists, the test records from the origin ``test`` lists, each class drawn with equal counts.
"""

from __future__ import annotations

# ruff: noqa: E501  -- the inline sample block and contract lines are kept at the fleet width
import csv
import hashlib
import io
import random
import re
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

# standalone rewrite (build_notebook.py): `from .pipeline import MAX_BASES, MIN_BASES, MODEL_ID` removed — names are kernel globals defined by the carried modules

CORPUS_NAME = "Genomic Benchmarks - human non-TATA promoters (interval lists) over the GRCh38 reference"
CORPUS_REPO = "ML-Bioinfo-CEITEC/genomic_benchmarks"
CORPUS_REVISION = "605d8539830e16c85abe7826990958303ffc5e1c"  # commit on GitHub the interval lists were read at
CORPUS_DATASET = "human_nontata_promoters"
CORPUS_LICENSE = (
    "Apache-2.0 (interval lists; Grešová et al. 2023, https://github.com/ML-Bioinfo-CEITEC/genomic_benchmarks); "
    "sequence bases from the GRCh38 human reference genome (Genome Reference Consortium, public domain) via Ensembl REST"
)
CORPUS_URL = f"https://raw.githubusercontent.com/{CORPUS_REPO}/{CORPUS_REVISION}/datasets/{CORPUS_DATASET}"
# path -> (bytes, sha256) of the four gzipped CSV interval lists at CORPUS_REVISION
CORPUS_FILES: dict[str, tuple[int, str]] = {
    "train/positive.csv.gz": (188_240, "8d8824518233a49cd02d9854a28154078bc69f3509719855dd4127a9024a44c9"),
    "train/negative.csv.gz": (137_784, "44b93cc1eb1273645d4123078809d9012bed3933605f4da906723a971b5a730e"),
    "test/positive.csv.gz": (63_260, "062277be939aea6538fe5e8a84792916835707e303e833d5ac1e6263c7416894"),
    "test/negative.csv.gz": (46_480, "3dd480a549f5e4b12770038e4ecc7f55cbec8101aa89b146fa9c4546ff7fad84"),
}
CORPUS_ROWS = {"train": 27_097, "test": 9_034}  # rows in the origin lists (positives 14,742 / 4,915)
REFERENCE_GENOME = "GRCh38 (Ensembl coord_system_version GRCh38; origin intervals are 0-based half-open)"
ENSEMBL_REST = "https://rest.ensembl.org/sequence/region/human"
INTERVAL_BASES = 251

SAMPLE_SEED = 42
SAMPLE_SPLIT = {"train": 1_600, "validation": 200, "test": 400}  # balanced: half positive, half negative
SAMPLE_DIGEST = "d913c4bd4cf74199456d24ced1f3ddc8f39e9241ab2eca132ffb9885efc62cfb"  # dataset_digest over the three default splits together; tests pin it
SAMPLE_SPLIT_DIGESTS = {"train": "3e4fe9685bab8554df451703ee65b288365f44d7a351ee9acc0a82f220c51810", "validation": "ccb6ac988348e1f2811fe3f7d4c7f76ef327e88f72a4344ec89dd0ba60761d8d", "test": "8384c09b5cba075000fe484f268adb6568c35acd602da8d47a31de36501fc747"}
MIN_RECORDS = 8
MAX_RECORDS = 20_000
ALPHABET = frozenset("ACGTN")
_ID_RE = re.compile(r"^[A-Za-z0-9_.:-]{1,64}$")
_SEQUENCE_RE = re.compile(r"^[ACGTN]+$")

INPUT_SCHEMA = {
    "record": {
        "id": "str matching [A-Za-z0-9_.:-]{1,64}",
        "sequence": f"str of A/C/G/T/N, {MIN_BASES}..{MAX_BASES} bases (case-insensitive)",
        "label": "0 (no promoter) or 1 (promoter)",
    },
    "dataset": f"{MIN_RECORDS}..{MAX_RECORDS} records with unique ids; a training split must contain both labels",
    "byod_csv": "columns id,sequence,label (header required); id may be omitted and is then row-numbered",
}


def _sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def _check_record(record: Any, index: int) -> dict[str, Any]:
    where = f"records[{index}]"
    if not isinstance(record, Mapping):
        raise ValueError(f"{where} must be a mapping with id/sequence/label")
    for key in ("id", "sequence", "label"):
        if key not in record:
            raise ValueError(f"{where} is missing {key!r}")
    rid = record["id"]
    if not isinstance(rid, str) or not _ID_RE.match(rid):
        raise ValueError(f"{where}: id must match {_ID_RE.pattern}")
    sequence = record["sequence"]
    if not isinstance(sequence, str):
        raise ValueError(f"{where}: sequence must be a str")
    sequence = sequence.strip().upper()
    if not MIN_BASES <= len(sequence) <= MAX_BASES:
        raise ValueError(f"{where}: {len(sequence)} bases; {MIN_BASES}..{MAX_BASES} are required")
    if not _SEQUENCE_RE.match(sequence):
        bad = sorted(set(sequence) - ALPHABET)
        raise ValueError(f"{where}: sequence contains characters outside A/C/G/T/N: {bad[:5]}")
    label = record["label"]
    if isinstance(label, bool) or label not in (0, 1, "0", "1"):
        raise ValueError(f"{where}: label must be 0 or 1, got {label!r}")
    item = {"id": rid, "sequence": sequence, "label": int(label)}
    for key in ("region", "start", "end", "strand", "source_split"):
        if key in record:
            item[key] = record[key]
    return item


def validate_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    min_records: int = MIN_RECORDS,
    max_records: int = MAX_RECORDS,
    require_both_labels: bool = False,
) -> dict[str, Any]:
    """Structural validation of a labelled DNA dataset; raises ValueError before any model import."""
    if isinstance(records, Mapping) or not isinstance(records, Sequence) or isinstance(records, str | bytes):
        raise ValueError("records must be a list of {id, sequence, label} mappings")
    if not min_records <= len(records) <= max_records:
        raise ValueError(f"{len(records)} records; {min_records}..{max_records} are required")
    checked, ids = [], set()
    for index, record in enumerate(records):
        item = _check_record(record, index)
        if item["id"] in ids:
            raise ValueError(f"duplicate id {item['id']!r}")
        ids.add(item["id"])
        checked.append(item)
    positives = sum(r["label"] for r in checked)
    if require_both_labels and positives in (0, len(checked)):
        raise ValueError("a training split must contain both labels")
    lengths = [len(r["sequence"]) for r in checked]
    gc = [(r["sequence"].count("G") + r["sequence"].count("C")) / len(r["sequence"]) for r in checked]
    return {
        "records": checked,
        "n_records": len(checked),
        "label_counts": {"0": len(checked) - positives, "1": positives},
        "bases": {"min": min(lengths), "max": max(lengths)},
        "gc_fraction": {"min": round(min(gc), 4), "mean": round(sum(gc) / len(gc), 4), "max": round(max(gc), 4)},
        "n_with_N": sum("N" in r["sequence"] for r in checked),
        "digest": dataset_digest(checked),
        "model_id": MODEL_ID,
    }


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    """Order-independent SHA-256 over ``id<TAB>sequence<TAB>label`` lines (sequences upper-cased)."""
    lines = sorted(f"{r['id']}\t{str(r['sequence']).upper()}\t{int(r['label'])}\n" for r in records)
    return _sha256_text("".join(lines))


def check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Assert no sequence (by exact bases) appears in two splits (leakage check)."""
    seen: dict[str, str] = {}
    for name, records in splits.items():
        for record in records:
            key = str(record["sequence"]).upper()
            if key in seen and seen[key] != name:
                raise ValueError(f"sequence {record['id']!r} appears in both {seen[key]} and {name}")
            seen[key] = name
    return {name: len(records) for name, records in splits.items()}


def split_dataset(
    records: Sequence[Mapping[str, Any]], *, val_fraction: float = 0.15, test_fraction: float = 0.2, seed: int = 0
) -> dict[str, list[dict[str, Any]]]:
    """Seeded shuffle of a BYOD dataset into train/validation/test after de-duplicating sequences."""
    if not (0.0 <= val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):
        raise ValueError("fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1")
    checked = validate_dataset(records)["records"]
    seen: set[str] = set()
    unique = []
    for record in checked:
        if record["sequence"] not in seen:
            seen.add(record["sequence"])
            unique.append(record)
    random.Random(seed).shuffle(unique)
    n = len(unique)
    n_test = max(1, round(n * test_fraction))
    n_val = round(n * val_fraction)
    if n - n_test - n_val < 2:
        raise ValueError(f"{n} distinct sequences are too few to split into train/validation/test")
    return {"test": unique[:n_test], "validation": unique[n_test : n_test + n_val], "train": unique[n_test + n_val :]}


def load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:
    """Records from a CSV with a header naming ``sequence`` and ``label`` (and optionally ``id``)."""
    source = Path(path)
    if not source.is_file():
        raise ValueError(f"{source} is not a file")
    with open(source, encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        names = {f.strip().lower() for f in reader.fieldnames or []}
        if not {"sequence", "label"} <= names:
            raise ValueError("BYOD CSV needs a header with at least the columns sequence,label")
        rows = list(reader)
    records = []
    for index, row in enumerate(rows):
        lower = {k.strip().lower(): (v or "").strip() for k, v in row.items() if k is not None}
        records.append({"id": lower.get("id") or f"row{index + 1:05d}", "sequence": lower["sequence"], "label": lower["label"]})
    return records


def write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    """A summary table (id, sequence, label, bases, GC fraction, provenance where known) - also a valid BYOD CSV."""
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8", newline="") as handle:
        writer = csv.writer(handle)
        writer.writerow(["id", "sequence", "label", "bases", "gc_fraction", "region", "start", "end", "strand", "source_split"])
        for r in records:
            seq = str(r["sequence"]).upper()
            gc = (seq.count("G") + seq.count("C")) / len(seq)
            writer.writerow([r["id"], seq, int(r["label"]), len(seq), f"{gc:.4f}", r.get("region", ""), r.get("start", ""), r.get("end", ""), r.get("strand", ""), r.get("source_split", "")])
    return out


def _parse_inline() -> list[dict[str, Any]]:
    records = []
    for line in _INLINE_RECORDS.strip().splitlines():
        rid, region, start, end, strand, label, split, sequence = line.split("\t")
        records.append(
            {
                "id": rid,
                "region": region,
                "start": int(start),
                "end": int(end),
                "strand": strand,
                "label": int(label),
                "split": split,
                "source_split": "test" if split == "test" else "train",
                "sequence": sequence,
            }
        )
    return records


def sample_dataset(*, verify: bool = True) -> dict[str, list[dict[str, Any]]]:
    """The pinned default draw as {train, validation, test} lists; refuses to return it if the inline data
    no longer hashes to ``SAMPLE_DIGEST`` (a silently edited sample would otherwise pass as the pinned one)."""
    records = _parse_inline()
    splits = {name: [r for r in records if r["split"] == name] for name in SAMPLE_SPLIT}
    if verify:
        digest = dataset_digest(records)
        if digest != SAMPLE_DIGEST:
            raise ValueError(f"inline sample digest {digest[:12]}... != pinned {SAMPLE_DIGEST[:12]}...; the module was edited")
        for name, expected in SAMPLE_SPLIT.items():
            if len(splits[name]) != expected:
                raise ValueError(f"{name}: {len(splits[name])} records, pinned {expected}")
    return splits


def sample_csv_text() -> str:
    """The inline records as CSV text (the BYOD format), for readers who want the data outside Python."""
    buffer = io.StringIO()
    writer = csv.writer(buffer)
    writer.writerow(["id", "sequence", "label", "split", "region", "start", "end", "strand"])
    for r in _parse_inline():
        writer.writerow([r["id"], r["sequence"], r["label"], r["split"], r["region"], r["start"], r["end"], r["strand"]])
    return buffer.getvalue()


# id, region, start, end, strand, label, split, sequence - one record per line (tab-separated); generated by
# tools/pin_sample.py from the origin interval lists at CORPUS_REVISION and the GRCh38 reference via Ensembl.
_INLINE_RECORDS = """
FP004746	chr4	67558547	67558798	+	1	train	CTAAGCACATGTGGATTACCCAGAGATTGCCCTCTGAAAGTCAGTCTACACCTGTTCTTTCTTACCTCACAAAAAGTAATGGAAAAAAAAGTGTGTGTGTGTGTGTGTGCGTCTGTGTGTGTGTGTGTCCTGTTGGTGGTAGTGTTGGTGGTTAAAAAGCAATTTGGGACTTCCTCTTTGAACAGTTGCCTTTTCCTCTCACAGAAGGAAGATTTCATTTTGTTTGAGACGAGAAACCAAACCACACACCA
FP001676	chr1	202348510	202348761	+	1	train	AGGTGACCCCACTGGGTCCCGTGGCGAAAGAGAGAGCCGCCGGTCCTCGCCTTTTTTCTGATGCATCCAGGGATTTGTAGTTCCCTTACGGCCACCAAGTGCACTTAGGGCAGGCCTCTTACTACGAGTCCCGTGATGACCCGAGGCCGGGCAGCGCCTGCGTATTGAAGCCGAGGGAGCGTGCGGGCGGTACTACTGGCGGGAGGAGTAAAGATGGCGGCGCGAGGGTCTCCGCCCTCTGCTCCGGGCTG
FP014177	chr15	96325875	96326126	+	1	train	TTTCCAGCCCCAAACAAGGTGTAACAACGCACTCTTCCTTCTAAGGAATGAGATGAGAGACAAGGATCACTCCAGACATCTCCTACCTACGGTTTGGGGTTTTTTTTCTTAAAGGCGAGGCTTGCATTCCTCAGCAGCTATGTACAAAGCTCCCTGAAACCTTGTCTCTCTAAAGTTAGTGTGCAGGGTTTTCCAAGGCTGAGAGAGCCTAATACATGGGGAAGCACTTCCTTGAGGTGGAAGATCTCTCC
FP000664	chr1	45687993	45688244	+	1	train	GGGCAGATTCTGCTGCCTGGGAGCCCTGGAATAGAACTTCAAGGACCGTTCAACATCTGGATTCAACTAGAAATTAAGAATGGCCCCCAAACAGAAAGCCGGGCCTAGGAACCAATTAGCGCAGGTCAAGTCCCTACCCGGACTCATTTTTTTTCAGCCCTCAGGAAGTAGGGGCCTCTCTGAATCCACTTGCCGGAAGTGCCTTTCCAGTGGACCTGGGCTGTTGTTGCGGTTGTTTTCCTTCTCTCCGT
FP010379	chr11	313839	314090	+	1	train	ACTTAGGAAGTCACTAGTCCTGACTTGAGTTTCTGATGAGGAAGCCTCTCTCCTTAGCCTTCAGCCTTTCCTCCCACCCTGCCATAAAGTAATTTGATCCTCAAGAAGTTAAACCACACCTCATTGGTCCCTGGCTAATTCACCAATTTACAAACAGCAGGAAATAGAAACTTAAGAGAAATACACACTTCTGAGAAACTGAAACGACAGGGGAAAGGAGGTCTCACTGAGCACCGTCCCAGCATCCGGAC
FP006921	chr6	131063194	131063445	-	1	train	GTCTCCGCGGCCGCGCTTGCCGAATCCCGCGGGAAAGGTCCCCCGCCCGCGCAGGGGCGGGGCCTGGGCGTCGGGGGCGGGACCGCCCGCGGGGGTTTCCGCGCCGGGGGCTCGTCCGGCGGGGTCGCGGGAGGGGCTTCCGCCGCGAGGGGGCGGGGCCGGCCCTCAGAGTCCTCTGACGGCCCCAGTCAGGGAATTTCAATTTGAAACCTAGCGGAGGGAGGAGGCAGGCGCGGCTGCCGGCGGCTGGG
FP004001	chr3	122564519	122564770	-	1	train	TGAGTCTCTTGGCAGTAGGGCCCCGTCCCTGTCCGCCACCGCACAGTGACATACTTTCTCCCGCCCTCTCCCCTAGGTCAGCTGGAACTGGGTCCGCCTGGAAACCCGGGGGCCGGGCCCCCAGAGGGCTTTCGCAGCCTGGCCGCCCGAAGCTCACCTGCCCTTTCACTGAACTCCACCCGGAAGGTGCCCGGGGCTTCGTGTTCCTGGGTGCTGACCGTGCACTCCCCGCCGCCCGAGGACTTAGAGCT
FP019684	chrX	149544250	149544501	+	1	train	GGCGGGAGAAAAGCAGGAGATCTGGAGCTCAGAAGGCTGCAGTGCAGACTGGGAAAGGAGTGAATACCAGGCAGCTTTGGCCCTATCAGGGCTTTGTCTCATGAGGGTGGGTATATTTTCTTAAAATCATAGAGCGTACTAGAGGTTGCGTTTATGTGTTGAAAGGAAGGACAGAGAGAGGAGTAGGGGCAGCTCAAGGGTGGAGTTCAGGGCCGGGGGCGGGGCCACGGGCCTGAGGGGTGAGGGCAGCC
FP001896	chr1	230855956	230856207	-	1	train	AGATTATGGCTTGGCGTTCTTTTCCCAGATTACCTCATTTAATCCTTACAACAGCCCTGCAAGGGAAGTATCATTTTGCTTTTACAAATGAGAAAACTGAGGCTCAAAAAGGCTGAGTCATCTTCCCATGGCCACACAGCTAATCATTGTTCAGGGCTGAGTCTGTGCCTGATTCTGCTGAGACCATCAAGCCCTCATCCACTCCTACTCATCTCATCTGTCTGAGTCCATTGGGGGGCCTCACCGGCATT
FP017423	chr19	45668020	45668271	+	1	train	GACAGCATGAGAGATTGTACACACTTGGTGCAGGGGTCCTCAGGACGATAAGGACAATTCAGTAACTGCCCTCCCTCATGACCTTGATGACTGCCCCCTGCTCGGCTCTCTGCCCCAGAGCTCCCCGCTTCCGTTCCCTGTCCCGCCTTGGCCCCGCCCCCTCCCATCACCCCGGGCTGCCAGCGCCTGTCACCTCTCCCAGAGCCGAGACAAGGCAGTTGGAGGCAGCGGTGGCAGGGGCTGCAGGAGCA
FP016649	chr19	5823829	5824080	+	1	train	CCTTTGCACTGGCTGTGTCCCCTGCCTGTGATGCCATTCTCCTCTGCCTGGCCAACTCCTACGTTTATTCAAGTCTGGACCTTGTCATCGGCTCCTCAGGAAGGCACTCCGGGACCCCCAGATGGGGGCGGTTCCCTGTGACTCCTGGCACGGAGGCCAACCCCTTCCTTGTTCAATGGTTCCTTGAGGGACCATTCCCATGTGATTATCGACCATTCGGCAGGCGTTCAAAGTCAAAGGCCCCACACTGA
FP000566	chr1	40450621	40450872	+	1	train	TAGTCCTTTGCTGATTCTAATGTCTTCTGCTCAGCATCTGCAGGGGCTGCTGAGAGTAAATACTTGGCGCCTCCAGCTGCTGGCCAAGGAGACAGATGGAGCTCAAGTTGGGAGATACGCCCTGAGAGCCGATGATAGACACAAGTCCAGATCTCGGATTTTGATACTGTATGTTCCCTGGGTTCCTGAGAGAGGACATTGAGGAGTAGGAGTCGGCGATTAAGGAGATCGGTACAATTGGGAAGCCTCCT
FP002475	chr2	86065636	86065887	-	1	train	TTACAGAAGAGCCAGGCCAGAAACGTGTGAAAAATACTAAAGCATAAAACCAAAGTATAAAAACAAGAACACAGCTGACAGCAGCCAGACCAAATTTAGGCATAAATGCCAAAAATATGCTGGAGTAACTAGAACCAGATGGGATTAAACAGAGAAGGCCGGGGGAACGGGATGAGTTTGAAGCAGTCTTGATATAGGATAGTGGGCAAATGGCAAACGTAGAAGCCAGTGCTAAACCCTGGTCATGTGCT
FP009732	chr10	35137431	35137682	+	1	train	ATGAGCTATACTTAGTATTAGATGATTTAGATAGCACTGCTTTCTCTTCAGTAGAAAATAATCAGAAAAATAGTTTATCATTTAAGTTTTGTCAATTTTTTTCCCTAGGAAGATAAGAAAAATACAGTAGAAGTAAGATTGACAAAAAACCGTACAGGGCTGTTTTTAGCTTTCTAGTCTTTTGGTTTTTTTTGGTGGGGGGGAGTTATTTTTTATTTTTAGTTTTATAGAAATTAGAATATTCTTCCTGG
FP008137	chr8	22023933	22024184	+	1	train	CCCAGCTTCCAGGAATAAAAGATTCTGACCTCTCCGTGGCAATGAGCCCCTGTCCCAGGGGTGTGCGGGAGATCCCTGGTTATCTGAGGTGTCTCCAGGGTATATCCGCTGAAACCCCACTTCCTCTTCACTGCCCACTGAGCCTGGGACCAGCTGCTGGTCACGTGCTTGGCCCTGTAGCGTCGCTAGCCGTGCTCCTCAGTGTGCTCCCCGCCCCCTGCCGCGGCGCCTCGCCTCCCGGCTCACCTCCC
FP013246	chr14	59484313	59484564	+	1	train	CATGGTCTGCGTCGGGGGAGACGAGTACGGTCCCGCAGCTATGGCTTCAAGCCCGACCCTCACCCACTGACTCCGCGGGAGGAGGGCGGGACGCTAACCAGCCACGTCCGGGGGGCGGGGTCTCGGAGCCTAAACCCGGAAGCGAGGGAGGAACTTCGGAGCTGTCGCCCGGGTTACCGGGAGGCGGAGCCGCCGAGCTCGCTGTGGCCCGGATGTTCGGTGCAGCTGCCAGATCCGCTGATCTAGTGCTT
FP015663	chr17	44004394	44004645	-	1	train	CCCCCCAAGAGTGGCAGTCTGTCTGGAGCGTCTGGCACTGGGGGGCTCTTAACTTGCCGTTGGGCGGGGCTGGACCTGGTCGGGGCGGGGTCCGGACAGGGGACCAGGAGTCTCCTCTGGGTGCGTTTGAGGAGGGTCAGGCCATGAGTCAGGGGGCGGGGCCACTAGGCTCCCCCGCCCCTGGAGGAACTGAACCCACTATCGGTCATGGGGCCGAGACTAAATGTGGCGGGTTGTCTTTAATCTGCTGC
FP019059	chrX	7927673	7927924	-	1	train	AAGTTCCCCAGAGTCATAGTCCAGCTTTTCCTTTAGATAGCACTTCTTAAAAAAACCAAAGGCACGCCTTGAAAAGCTGCGGGATGTCCAGCCCCAAAATAGCCGCTGGAGTTGGCCCCTAGGGGCAGCTGGACCGCCCAGGCGGGCGAAAGGGTGGAGAGGGAGAAGAACCACTTCCGGGAAACAGACGCGAAGAGGGTAAGGCACTTCCGGGGCACAGAAAATGAGGATTATTAAAGGTCAGTTGCTCG
FP016732	chr19	9498530	9498781	-	1	train	GTCCATTTCTCTCTTGCCCACCCAGGCACTGAGCCGACCCCTTGATACCTCCGCACCCACGCGGAGGGAAACTAGAATCCTGAAGAGTTCCATCCTGTCCAACTGCCCACACAACGAGCCACGCCCCTTCCAATGCCCCAAACGGAAGTGACGTTGCGGCGCGCCAGTTCTCTCTTTTGGTCGCTGGCGCCTGTTTCCTCAGAGCCACTTTTTCCTTTGTGTTGTCTGGTCGTTCACTACTTTGGTGAGGC
FP011560	chr12	4321012	4321263	+	1	train	CCTGCCGCGCGCCAATCACAGGCCGGCGCGCAGGGGCGCAAGCGCCGCAGCCCTGGGCGGGGCCCCCGTGGCTCCCTCGTCTCCCCCGCCCCGTCGGCCTCGCCCGGCCCTGAGTGGCCTTCGGGGATGACGTGCGAGGCCGCCTCGGCCTATGGCGGCGGAGCCGGCCGGCTGCTTGGCGGAAGTGGTGTGGGGGAGGTAGCCCGCAGTGCAGGGGCAGCGCGGCGCGGGGCCACCGACGGGACGCGGCT
FP004082	chr3	129567494	129567745	-	1	train	TGCAGATGGAGGAGATGGAATCTCAGATCCGAGAGGAAATCCGCAAAGGTAGTGGGCGGGGACCCTGCAGGGCTGGGCCTCAACTTTCCCTCCTAGGGGATGGGTTCTCGGGCCGCTCACGGCTGTCCGTGTGTCCGCCAGGCTTCGCTGAGCTGCAGACAGACATGACAGATCTCACCAAGGAGCTGAACCGCAGCCAGGGCATCCCCTTCCTGGAGTATAAGCACTTCGTGACCCGCACCTTCTTCCCC
FP016695	chr19	7699390	7699641	-	1	train	GGAAAGAGGGTGAATTTCTAAGAAAGGGACTGGTGTGAGTAAGGAGGTGAGGCCGGACTGACTTTCCTGGCACAGAGCCAGGAAGGAGTGGCAAATTGAGGGCCCCTCCTTTTTCTGATTCAACACCCTCCTGACAAAAAAAGAAAAAGAAAAAAAAAAAAAAACGGCTTCAGCTAGGGAGCGGGGAGCCCAATAGAGTCAGAGGCCAAATAGAACAGGAACTTGGAACAAGCAGAATTTAGCATAATGAA
FP019400	chrX	74614557	74614808	-	1	train	GGGCCTGTCACATGGTGGGCGTCATAACAGCTTTCCTTTTCCAAACCGCCGCCCGCCTTGCCGCGTCGTCTGCCGTCCCCCTCCTCCTCCCGCGGAGCCTTTTCGAGGAATCGACTGGGATTGGCTAATAGGCTGGGAGTGGGGCTGCGGTGCCGGGACGCGGCATCATCACTTCCTGTTGTGGGCAGTCGTTTCCTGTCGCTGCTTGGTAACAATGGGGAAGATAATGGCTGCCTGAGCAACGTCTCCGA
FP001551	chr1	174999747	174999998	+	1	train	GGCCCAGCGCGGTCAAATTATAATACATAAAAGTTGTCAGGGCGGAGAGCAAGACATTACTCTTCTCGGATTGCCGGTTCGCTCGCGAGACTTGAGCGTTGCTAGGAGATTCGGCAGGCGGGCGGAGCCAGACTCGGCGGGGCGGGGAGGGGTGGGGCTAGGCTCGGCGAGGCGAGGAAGGGTGGGTGGAGCCAGGCTTGGCGGGCTGTGCGTGCTCGCGGTGGGCGGTGGCGGCGGCTGCCTCGCGAAGG
FP017200	chr19	37594679	37594930	+	1	train	TACTTTTTCGATGAAGCAACCCAAAAGCTGCGAGCGGTTCCCGGTGAGGCCGCCCACTCACCTGGCCGGCGCAGACAAGCTCCGTGCGTCAAGACATAACAGCGTAAGTGTACGACGTTGCGCAGCGACGCGGGGGCCTTCGGGAAATGTAGTCTACAACTGGAAACCGGCCGGATCGTGTCTGCGCAGGCCCAGCAGCTAAGATCGGGTCCGGCGCTCCAGAACAGAACGATCCCTGAGGCTCCCTTGCT
FP000125	chr1	8878635	8878886	-	1	train	CCGCCTCAAGTCCGCGGGACGTCACCCCCCTTTCCACGCTACTGCAGCCGTCGCAGTCCCACCCCTTTCCGGGAGGTGAGGGAATGAGTGACGGCTCTCCCGACGAATGGCGAGGCGGAGCTGAGGGGGCGTGCCCCGGAGGCGGGAAGTGGGTGGGGCTCGCCTTAGCTAGGCAGGAAGTCGGCGCGGGCGGCGCGGACAGTATCTGTGGGTACCCGGAGCACGGAGATCTCGCCGGCTTTACGTTCACC
FP002844	chr2	170715138	170715389	+	1	train	AATCAGTTCTTTTATCCAGACCAACAAACACACCATAGGAGCTTTGTGGATTCAAAGGATTTGCTTTCGCTTCTGAAAGAGCCGCTATTCTTTGATGATTGGGTAGCGGCAAACTTCAAAGCCATAAATCTTCCCTCTGACTGGCTGGCGGCCCAGCAAAGTCCTTATCAAATTCTTGGAGGTGATCCTGAAGCGCTCAGACCGCGCGCGGGGCGAGCGAGCGGGGCGCGGCGAGGGGCAAGGGCGGGGAG
FP019565	chrX	119692760	119693011	-	1	train	CCGCCGCCGCCACATCGCGAGGAGGGCAGTGGGCAGCATCCAGGAGCCAAGGGCACGGTGGGACTTCTGCTCCCGGGGTCAGCAGCTCAGTCCCTCAGCCTGTCCCGGAAAACGGTGGTGGCGCGCGGGGGGCGGGGGGCGGCAGGGGGGCAGGCAAAACCTGCGGGCCGGCGGCTGAACTTGCATTGCCCAAGAGCCGCAGTGAGCGCCACCGCCTCCCTCGCTGCAGCTGCGGCCCCACGGCTGGGGCG
FP015143	chr17	4584328	4584579	+	1	train	CCGCACGCCGCTCCTCCGGGGTCGCAGGGCTCAGTTAAGAGCCGGGGAGGGGAGAGGCTGGGCGAGCTCTCCCTCCGCACCGTCCCGCTGGGCTCCAGGGCAGGGGGGTGTCCCGGAGTCGTCCCCGGCTCCGCCCCCGAGGCCCCGCCCCGCGCCCGCCTCCCCGGCAGCCCCTCGAGCGAAGCCAGCCACGGACGGCCAGTCGCGGCCCGGAGCTGCGGAGCTCGGATCTTCTCCCCCGTCTGGCCCGC
FP000808	chr1	65309185	65309436	+	1	train	CCCCTTTCTTCTCTGTCACACGCGTGCGAACACACCCTCCCAACCCACGTCGTAGTAGTCCCCTCCCCCGACCACCCCGCCCCGTTTCCCTTCAACCCCCCTCCTCACGAACCTCCCCTCCCTTCCTACCTCCCTTTCCTTCCCTCCCCGCGGGCCAGCGGCTTTGGCGCTGCCCTGCTGCGCGGCGGGGCGACGCGGCGACACCTATAGGATTGCTTCCGGGCACTCCTCAGCCAGGAGGTGCTGGCCTG
FP014991	chr16	86554675	86554926	-	1	train	GCCGTGCACGTGGGGGAGCCCGTGGGCCCCTTGAGGTCGGGGAAACTGAGGCTCAGGAACAGGATCGGAAGCCGGTTTGACCTACACGCGGGGGTCGCTTAACTACTGTTAAGCGGTTTCAGCATGGGCTCTGGAATTCCTTTGTATGTTATTAAGTGGGAATCTTTTGTTGCAGTAGGTGTCTCCAAACAGGACATACGTGAACAAATTTGGGGCTACATGGAATCACAAAATTTAGCTGACTTTCCCCG
FP018832	chr22	38057470	38057721	+	1	train	TGTGGGACCAACGCTTCCGGTGAGCGACAGAGGCAGCTCCCCAGGGCCTGGAGACCCGTGGGGCGGACTCTGGGATCTGAGCCTATCGCCCTGGCCTGGAGCCCCCCTTTGTACCTAGTAAGAATCACCTACCCAGCCCCCCACCCGGAGACTGTCCCATCCCACTCCAGAGGGAAGAGGAGGGCACTGCTGTCCCGTATGAGGTGGTGGAGGGAGGGGTGGCGTTGGCATTTAGGCACTTGGGTTCCTTA
FP004249	chr3	160449709	160449960	-	1	train	CCGTCCACCGTTTACTTTCGAACGAGCCCAGCAATAGTTCCCATTGGCCGATGCCTTGCAGTTGAATGTTGCCTGCAGACTGCGCCTGCGCTGCGAGGCTCCGCGAGGCCCCGCCCGGTCTGGGCACCTGAAGCCCTTCGGGGCAGAGGAGGGCGGGGACTCGGGGCGGCTCTCAGCATCCGCCTGGAGCTCGTGGCGCTGTGTTTCCGTGCTGTGGAGTTGCCTGGTCCGCTTCCTCCCCGCGGTGAGTC
FP003051	chr2	205682300	205682551	+	1	train	CGAGAGAAGCCGTCGCCGCCCCGCCCGAGCTGGGTGCCGCGGCACTCTGCTCTGCGCCGCGCGTCTCCGCGGCTGCCTCTTTAATCAGGAACACAGAAGGGGCGGGCCCTGAGCCGGCGCGTAATCGGATAGCTCTGCCGCGGCTCTGACATCCACATGCTGCTCGGCCTGAAAAGGCAGTGGGGAGCCGGAGGGGAGGCAGAGATCGCGAGCGAGGCACCAGCCTGCAGCCGGCCCCCAGCACATCCTCA
FP008397	chr8	73008663	73008914	+	1	train	AGCCCTGCCCTAGACTGGCGGCAAATTCAACGCCCAGAGGCAGGAACAGGTGTCGGCCAATCAGAAGCCGGAACAGGTGTCGGCCAATCAGAAGCCAGGCTTGAACCGCTCGCCCATGTCCCGCCGGCGGAAGAGACTCTCCGCGGCCACGCCCCGAGCCCTCGAATGCGAGCCAATCGTTGCTCGGCGCCTGAAGGGGCAGTACCCAAGCGAGCCATTTAACATGGCGGAGGATGTTTCCTCAGCGGCCC
FP010484	chr11	5754435	5754686	+	1	train	GGAATATGTCTTTTTATAAAATAATTGATGTTTGATTGGCATCTGTGTTGAAAACTTAGACTGCTCAATAAGAATTTTTTACTGAATCAATTAATTAATCAATCAATAAAACTAAACAGACAAACAAAATTATTTCTTGGCACTCAGAAATTTGGTTGTTGGACCATTAAAATGCATTATGGAATTTTTAAAAGTTGGGGGAGAGGGAGACAGTAAAAATAACCTATATTTTCTCTTGTTTTTTTTTTTAA
FP010239	chr10	114043665	114043916	+	1	train	GCTCGGGACTCTCAGGAGCCGCTCAATTGCCAACGGGAGGGGGGTGCGGGGAGTTGGAGGTGGGGGTGCAGACCAGACGGGGGCGTGCCTTTGCCCGGATTGGCTGCAGGAGCCTGACGCGAGGCCCCGGGGGTTGGCTTGGGGAGTGGGAGCGGGGTGGGGTGGGTGCTGGGTGCCGGAGCTGCGGGCCCGGCGCGCTCAGAAACATGCTGAAGTCCCGGCGGCTCTTCCAGCAGCGGCAGCGGCTCCAG
FP015780	chr17	49815337	49815588	+	1	train	TGAGTCTAAAATCTTAGACACTAAGGCTTAGAAAATCTGGTTTTTGTTTGTTTTGTTTTTAAATCGGGGAGTATCTACCAAAGCTCCTTCCAAATCTCAGGATTCTGGGAATTTGTAGTGTCTCTGGGTTTGTTGTACTTTATTTATTTGTACTGTCATGTAAATTTTTACATGAAGGAACAGCGTTCTGATCCACAAAGTGTATCTTTTCAGAGATAGAAGAATGAAGCTCCCTTCTGGAGCTGTGGGAG
FP000577	chr1	40981684	40981935	+	1	train	TTAATCAGCGTTTCTCTGATTTATATCCATAACACCCTTTTTGAGAATCCCATGTGCATTTTGTTGAAACCAGGGAGCATGCTTTTGGAAATGCTGCAGCAGAGGATGCAAAGAGCCTTTGGAGCGGGGAAGACTTCTGAGCATTCTTCCTTCCCTGCCAAGTGTGTTCTTTCTTGCTCTGGGGAGGGGGTGGGGCGGGGGAATGCTTCGAGCACAACTCCTTAGTTTTCTTGTTCCTTAGTTTATTTTTA
FP001158	chr1	147611320	147611571	+	1	train	TGCTCACCCTTGCCTGGTCCATGCTGCTTGCGGGGGAGGCTTGCATGCTTGGCAGGAAGTCCTAGTCACCCTCCAGCTGGTGTGGCCCCTGAGCCTTTGTCCACAATGGGAGCATCACCCTTTCACCCTGATGGAAGTGGAGGCATGGTCTAGCTGCTTCGCTGAGTTGGCTAGTGCTTTCACCTTGATCCTGTTGTGAGGGTGTTTTATGTCTCTGTTTTTGCTCCCAACTTTTTTTGGGTGGCCTTTTC
FP006967	chr6	138404140	138404391	+	1	train	GTGTGTGCCTTTCCTTCAAAAACAAAGGCGGGGCGGGGGCGGAAGTGCCTCAAAAGAAGGCGGGCCGAAGCGAGGTGCGGCTCCCTGTCGCCGCGGAGGGGCGGGGGCAGGACGCGCAACTCCGGCCGGAGCTGTCCGGGGTCGTGAGCCGGCCCCGCCTTGGTGGCGGCGCCCCCTCGCGGTCCAGAGGCAGACGCATCGGGTGGGCTCGGGTCTCCAGCCCGGCCGGGAGGAGGGACCGGGTCTGCGGA
FP001448	chr1	161159299	161159550	+	1	train	ACCTCAGGGACTTGCTTTGGAGTCATGGATTAAAGACCTCTCTGGGCAGCGTTGAGAACCAGCGACCCTAAACTTGGCTGCGTTTCCATGGTGTCGATCGCGGAGCCGTGGGGGCGGGCTCACGCAAGCGCCAAGGGACCGGTGGGAGGAGTTCGGGTGTGGATGCGCAGGCGCTTTGAGAGACGGTGAAGCCGGTGGCCGGTGGCCGGGCGGGACCAACAAAGATGGCGGCGGCCCCTGCGGCGGGAGCG
FP002460	chr2	85413965	85414216	-	1	train	CTCGCAAAGTTTACAGCACAGTAAGCTTGTCCTTAGCTTCCCCTATACCCTCCTTCCCACGCCCAATTCTAGGACCCTTTCGGGCGCCCCAGCCTCCCATAGGGGCTCCCCGCCAGCTCAGACCGGCCGGGCCCCTCCCTCTGACTCACCGGGGGCGGCTCCTGGCCGCGCACCTCCAGGGGGCGGGGGGCGGCTGAGACAGCGCCCAGGGCCTCGGAGCAAGGCGTTGGCAGGCAAGTAGTGCGCTTCGG
FP001760	chr1	209827818	209828069	+	1	train	GTTCTAGGGCTCTGAGAACACTAGTGAACGAACTCCCGTGCTTTCAAAGAGCTGCGGTAGGGGGCAGAACCGGGAACCGGATGTTCTAAGCCTGTCGTACGAGCGCGACGTAAAGCGGATCTGCTTTATGGCACCTTGCTTTCGCCGTAAAGCGCAGTCAGCGAGCCCACGTGCTTGTGTTGACTGGACAACTTCCTGGTGGAAAACCGCGACTCTTGCAAGTGGGCAAACTTGACGTTTTCGCTATGGGC
FP019399	chrX	74421292	74421543	+	1	train	CTGGAGTAGGTAGGCGAGTAAAAGGAGGAGGAAGGGCTGGGGAGAAGAGCGAGGCTGTAGCGGCTGCCTGTTGAGGGAGGAAGAGGTGGGGGGTGTGGAGGAGGGGGAGGAGAGAGAGATGACATGGGGAGAGGAGGAGGGGGGTTGGACGTGGGAGGAGGAGGAGAGGGCTCGAGGGACCGTCTGTCGCGGGACGGGCTGGCCAGCTGGGGCGCGGAGCCTGGAGGAGGAGGCAGCGGCAGCGGCAGCAG
FP003801	chr3	64687585	64687836	-	1	train	CTCAGCCGAGCGCAGACGGGACCCTCGCAGCGAGACCTCAGCGACTCCTAAAGTCAAAAGTTGGCGGCGGGCGCCGGGCTCCGCGCGCTCTCCACGGCCGCTGCCTCGCGTCGCCGCCGCAGCCAAGGAGGGCAGGAGGGAGGGGGGTGGGGGCAGCGGAGGGAGGGGTGGGAAGCACCATGCAGTTTGTATCCTGGGCCACACTGCTAACGCTCCTGGTGCGGGACCTGGCCGAGATGGGGAGCCCAGAC
FP003080	chr2	210009334	210009585	+	1	train	CTAGTGTGTGACTTGGGCAAGTCATAAAGCCTGAGGGTTTCTGATTTTATCATCTCTCCATTTCTACTTGTCATGGGTCGTGAATTACGGAGTCAAATCATTATGAAAATATATGTGACGTGATTTGTTGAAATGTCAAGGAGACATTGCCTAGTAAAACCAAAATGACCCATTATTTTTAGTATTATTTAATACTGATGATTAAGTATTAAATTGAAAAACATTGCAGACAATCCCCTCAACCTGTGTCA
FP011020	chr11	66347563	66347814	-	1	train	GGCGGGCGGGCGTCTGCACGGCGCGCGGCCGAGCGGCGGGGCCGTGGCTCCTCCTCCCTCGTAGGGGGCGAGCCGGCGCGCCGGGGGCTGGGGGCGGTGGCGGGCGGCGGGCGGCGGGCGGCGGGCGGCGGGCGGGGCGGCGCTCGGGGCTCGGCTGGCCTCGGCTCGCCTCGGCTGCGCTCGGCAGGCTGCGGTAAATCCGGGCTTGCGGCCGCTGGCGTAGTCTGTGGCCGGGTGGTCGTTGCTGCGCG
FP011345	chr11	112164038	112164289	-	1	train	AGCCCCAACTTTTACGGAAGAAAAGATTTCATGAAAATAGTGATATTACATTAAAAGAAGTACTCGTATCCTCTGCCACTTTATTTCGACTTCCATTGCCCTAGGAAAGAGCCTGTTTGAAGGCGGGCCCAAGGAGTGCCGACAGCAGTCTCCTCCCTCCACCTTCTTCCTCATTCTCTCCCCAGCTTGCTGAGCCCTTTGCTCCCCTGGCGACTGCCTGGACAGTCAGCAAGGAATTGTCTCCCAGTGCA
FP005438	chr5	69369469	69369720	-	1	train	AAGTACCGACCCGAGTTATGCCCACCCTCCAACTTTGCGACGGAAATGTGTCATCGAAAGCCAGGTAACCAGTGGGCCGCGGCTGGGCCGCCGGGGCAGGCTAGGGGCGCCGCGGTGGGCTTCGGGGCCTTTGGGGCCGTGGGCTTTTGCTGCCGCCCGCCCCTTCCGGCTCCTCCCTGTAGAGCAAAGGGCACGTGAGCGAGGCGCCCGAAGCCGTCGCGGCGGGGACCATGTTGCTTCCGAACATCCTG
FP015360	chr17	19004713	19004964	-	1	train	CCAGGAGCTCTTGGGCTTAGGCCTCTGATATTTTCGGAATTCGGGCACCAGGGGACCGTGGGCAGTGCGTCCGCCCCAGGTCTGTCTGTCTGTCGAGGGGTGAATCAGCTTCCGGCCCCGCCCCGGGCGGCACGTGACCGCAGGTGGCGGCGGCGGGGTAAGCGGGGCGGCCCTGAGTCACCGGCGCGCCCCCGCCCAGCCTCGCGCCGCCGCCGCAGCCGCCGCGTGTGCGCCCCGCTCCGCCAGCGCCC
FP013784	chr15	47717266	47717517	+	1	train	TCAGCAGGGCCGCGGCGCCAGGGTGGGGGCCTCCAGGCGGCGCTGGCCACGGTTCACCCGCGAGGGCGAGCGCGCCCGGCCGGCCGCGCCTCCTGCCGTCCGTGCCCGGGGGTGGGACCCGCGCCGGGAGCTGCCGAGTGCGTGGAGGGAGCCAGGCTCCGGCTCGCCTGATGGATTGACTTGGAGATGGATCGAATTACACCATTTGCCGTGCATCGGTTTGTTTTGCTCCTGCTCTGTCCCGCTGCTGC
FP019790	chrX	155881144	155881395	+	1	train	CCACAAGAGCTGTCGAACGAACGTGAAACACTCAGTGATACTCCAACCGGAACTACTACTCCCAGAATGCAGTACGGCTCCTGGGAAGTGCGGGGGGCTGGGAACGCAGCAGGCCTAGCCGTGTCGCCTGCTGCCATTGGAGGAGCGCTCCCACTCCCAAGAGGCCACGCGTAGACGGGGCGCTTCATGCGGAAGTCAGCGGCGTCCGGTCCCAGCCTCCTCTGGGAGCGGGCAGTTGGCGACCCTGCACT
FP016761	chr19	10416681	10416932	+	1	train	CATTGACGCCGTCAGGAGCCCAGCCGTGCGTCAGCTGTCGCCCTGGCAACGGGGGCGACGTCACCACATTGGTGGTGCGTCAGTCCGCTGACGTCACGCCCCAGGAGAGGCAATAGGCAAGTGGCGGGGGCGGGTTCTAGGCCCGACCGGCAGGGGGAGCCCTGGGGCACTGATGGCAGATGCGTTTGGCTTGGTCTTGTAGGAGGCCCTGGCCCTGCCGACATGGCCACCGCAGTCCCAACGGCGCGCTA
FP012252	chr12	86838915	86839166	-	1	train	TGAAAGAAAGCAATAGTGAAAATGAATTTCATGCCAGCAGACACTTTAAAAACTAGATTTAGTAGCAGTCACATTAGCTTAGATAATGGGAGAATTTTTCTTCATTGAAAATGCTCTCTGTCGCTTAAGAAGAACTTACTGAGGCAGCCTGCCATCTAGCTTTTTCACAGAAAAACAGGCTTGCAGACTGCAGATGAGAGAGTCTGGAGCAGAGACAATTTTAACTTAGCCATTCCTAATTTATCTCAAGC
FP010800	chr11	57568236	57568487	-	1	train	TCCTTCTTGGGGCTTCCCCCACCAACTGTTTCCCAAACCAGCATACAGTAGGCACTTAGTAAATACTAAACCTAGTAAAGGACTGAATGATGGTAGACCGAACCCCCAAAGCAAAAGAGAAAGGGTCCAACAGCGCCTCCTTGCGGTGCTCTGCCCCAAGCCACGGGGAGAAACGTTGCAGCCCGCGCCGAACGCCGGGCAGCACAAAGGATCCCCGACTGCCGGGGAGCGGTGCTCGGAGGGCACAGGTC
FP002435	chr2	74554537	74554788	+	1	train	ACGACGGCGGGTAGGGGCCTGGGGTGCTGCAGCTCCCGGCTAATCCGGGGGTCTCCACCATTCCTTCTCGCTCCTCTCCCTCACCCCACCGCGACCCCCCCAGCGGCTGCCGGCGGGGGCCGGGGAGGGGCGGAAACTCCTCCAGGGAACCCGGCCCCGCCTCCCGCCCCGCCTCCCGCCGCAGGGCCAGGAAGCGCGGAAGGAACCGCCGGGGGCCATGGACGGAGCAGTGATGGAAGGGCCGCTTTTTT
FP001135	chr1	145964489	145964740	+	1	train	GTGAGACCGGAAATAAGTGACCCCAGCTGATTCTGCAACGCCCATCTCTCTCTGCGACCCGCTTCGCAAGCTTCGGTTCCGGGCGTACTCGCTCCCTCCCTCCTTCTCTTCCTAGGCTTCAAGAGGGCGGACTTCCCTCCTCGGCTTCTCCTGCATTGGCCGAGAAAACTCACCAATAGGGAGCTCCGGGAGGCGGGCTTAGAACGCCACCGACTTGAGGAAGCCCAGTACATTTCAAGTTGGTCGCGGCT
FP003182	chr2	224816588	224816839	-	1	train	TCCGACAAGTGTATGGTAACTACTTTTTCTTTTTAGGGATAAGATTCAAGTGCAGATTTTGTTTGTTTGTTTTTATTCAGTTTCTTGGTCTGTAAGTCATAGTCATTTAGTTTGCCTCTCTTTCAGACCTTGTGCCAGTATAAATTTGATTTTCTTCAAGAAGTATGTCAACATGAACACTTTATCCCTTTGTGTCTGCCCATAAGATCAGCAAACATTCCAGGTAATAAAATTCCCAGAATTTTCCTCAG
FP009463	chr9	133429628	133429879	+	1	train	TATCCCTGCGTCCCCTTCCCGCCGACCGCACCGCTCCCGGGCCTAACCTGCATCTGCTCCATCCCACTCAGACCCGTCCCTCCGTCGCCGCTCCCTCTGCTGGCCACCCACCTCTGCGCCGGCAGGAGCCTTAGTCTTGGTCCCAGCCAAGAGCCGGCTCCTGGTGGGGGGCGCGGGCCGAGAACTCCTGTTCCCACTCACAAAAGGCCACGCTTCCAAACGCTTCCATCCTCGTGCCCACTCCTCCGTCC
FP012271	chr12	92702642	92702893	+	1	train	GCAAGTCATTAATAAAATAGATCAACACAAAATGTTTTATGATGTGAAGAAAGTAGAGTCCCAAAGAATGCATCTCACAACCTTAGAGAAAATAAATGAAATAGAAACCTAGAACTGCTTCACTGAACTATGAGTGATGATTTGTTGAATGCCTCAAGAGGAAGTAAAAAGTTACATTTGCATCATTTCTGTATTTGATCAGCCGTTGTTTACCTGACTTGTTCTTAGGCTGCCTTAGATTCTGAGCTGGC
FP012056	chr12	56104365	56104616	+	1	train	CGCCTTCCCGCCGGAGCCTGACCCTTCCCAGAGTGCCCGGCGATTCCGGCGTGCGAGGCCCTTGGAGGGCAAGGCCCCAGGGCCTGGCTTAGGAGCGCGAGAGGCAGGCTGGGAATTGTAGTTCGAAGGCCCTCGAGAGCGGCTAGAGTCTGGCGGCCGAGAGGACTAGTTGTCCCAGCGTGCCCTGCGCCTCAGCCCGCGCGCTCGCAGCTTCTCGCTCTCGCCTGCCTGCCCGCTCCCTTGCTTGCTCG
FP013250	chr14	60165367	60165618	-	1	train	TTATTTTTGGTAGACACAGGGTTTCCTCATGTTGCCCAGGCTGGTCTCGGACACTTGAATTCAAGCAATCTGCCGGCCCCGCCGGGTGTGCCGCATGAGGGCGCGGGCCGCCCAAGGACCGCGAGTGGGGCCGCGAGTGCTGAGCCCGGCGCGCCCGAGCCGCTCCTCCTCCCGGCCTGCTCCCTGGGCGGAGCCGGGGCAGTGCGGCTCTGGGCTGGCCGAAGGGGTGGCGCTGCGATCCCGCAGGGCAG
FP004379	chr3	187291649	187291900	-	1	train	TTTTGTTGTCAGGTTCCTGGGAGTGCAAGAGCAAGTCAAAGGAGAGAGAGAGGAGAGAGGAAAAGCCAGAGGGAGAGAGGGGGAGAGGGGATCTGTTGCAGGCAGGGGAAGGCGTGACCTGAATGGAGAATGCCAGCCAATTCCAGAGACACACAGGGACCTCAGAACAAAGATAAGGCATCACGGACACCACACCGGGCACGAGCTCACAGGCAAGTCAAGCTGGGAGGACCAAGGCCGGGCAGCCGGGA
FP012585	chr12	131894443	131894694	+	1	train	GGGTCCGCTGTTGGCCCCACCCACTCCCAGCCCACCCGGCCCCGCAATGGCCCGGCCCAGCCGCCCACCCCCCCGACAGGCCCCGCCCCATTCCCCGCCCCCCGTCTGTTCGTCACGCCCGGCTCTGCCGCTCCGTCCCGCAGGCCCCGCCCTCATCCCGCCTCCCGGCCCGGCGCGCAGTGCGGCTGCGCGGGCGTCTCAGGCTCTGAGGCCCGGGCGCCGCGGCTCTTTTGTTTCTCCGTTGGGGCCGA
FP008748	chr9	4299918	4300169	-	1	train	GGCCAGCCTAGTGACCCGCGAGCCCGCCCCCACTCTTCCCCCTCCCCCTCCGGCGCTCACACTCAGGAAGCAGCTGGCTACAGGCCGGCCCTCGCGGCCGCACACACATCCACACTCCTCGCCCGGCTGCCTTCCTCCCTCCCTCTTCCCCCTTCCCCTCTCCCTGCAGCCGCGTCCACACTGCGCCGCCCGGGCAGCCCAGAGCGCGGCGCGCACGGGCACCGGCGGCGAGTGCCGCGGCCACTGCCCCT
FP014015	chr15	74995366	74995617	+	1	train	ATCCGGCCGGGGCCCGGGAACTAGCACCTCGGACAGCTCCTGGAAGATGCCGGGGCGGCCGGGTCGTCCTTCTCGCGAGCTCAGGAGGAGCAGCCTTAGGAGCAGAGCGGAAGTCTCGCGAGAGTGACTGCTGCAGGCAGTGGCGGCGCGGGCGCGCGCAGAGACGGCCTCGGGCGCGCGCGCGCGCGCGCGCGCTCCCCGCCCCCAGCCCCGGAGCGGCTCGCGGCCGGCTCCGCGCCGCATCGCTCGGG
FP005481	chr5	75337036	75337287	+	1	train	GTGAGAGATGGTGCGGTGCCTGTTCTTGGCCCTGCAGAGAGCTGTGGGCGGTTGTTAAGGCGACCGTTCGTGACGTAGCGCCGTCAGGCCGAGCAGCCCCCAGGCGATTGGCTAGACAATCGAACGATCCTCTCTTATTGGTCGAAGGCTCGTCCAGCTCCGAGCGTGCGTAAGGTGAGGGCTCCTTCCGCTCCGCGACTGCGTTAACTGGAGCCAGGCTGAGCGTCGGCGCCGGGGTTCGGTGGCCTCTA
FP016286	chr18	26549721	26549972	-	1	train	GTCCCCGGCGGCTCTGGGAAGCTGCCCCTGGGCCGGGAGAGCGGGAGGCGGCGACGCAAGGACGCGCCCGGAGCCTGGGCGCGCCGGGGGCGGGCGTTTGGCTGGAAGTGGGGCAGTGTCTCCTATGGGAAAGCGAGACCCGGCGAGCTCTTTGGCTGCTTTTCCAGTTTCCGAGGTGCCCGCCCCCGCGCGCCTGAGGCAGCGCAGCCCGGAGGGCGAACGAATGGCGGTTGGCAAACTTTGAGTGAGAG
FP009641	chr10	14959190	14959441	+	1	train	CCTTTTTCGCCCTCCATCTCTGCTCAGGGGTGAGCCCTGGCAGCGCTCCTACCTGGGACCTACATTGGCTTAGTAAGATGGGCCGGCGCCCACCCCAGCCCTCGCGCATGCGCATAAAAGCCCACTCCCGGGGGCGGGGCCTGAGCTAGCTGGGCGGGGCCTGAAGGAGCCGGGAGCTCCTGCAGCTTCATCGCGTTACTAAGAGACGCGAGGGTCGCGAAGGGCACAACACTTGGCCTCTCGCTGGCCCT
FP008834	chr9	28719080	28719331	-	1	train	AATTAGTTCCTTCAGGGAAGGAGGTTAATTTAGAGAAGATGTAGGGAGTGTTCAACATGTTCGTTGTGGAAGAGAAAGAGCTAAGAGAGAGGAGCTTAAAGACACAAACGGGTAGAATCAAGGAGTGTGCTCTCAAATGAGAGGAACAGGAGTGACATTAACCTTGAAATGCTCGGAGACTCTACTCCTTCATGACAGTAGGAGGATAATTAACAATAGATACAAATGCAGGTAAGTTGAAAAGTGGGAAG
FP007913	chr7	141014628	141014879	-	1	train	GCAAGGGGCGATGGCTGCGAAGTCTACGGGGGTCTCCAACCTTGTAGAGTCGCCAGGAATAGGGCGAATCCACTTCATTAGTGACCAGCTCGGGCGGTTCACGTGCATCACACAAATAACTTGGCCTTTTTCTGCCTCAGTTGGGGGATTTCTTAAACGTAGAATACCCGCGTTTCCGCTGCCGTAATTTCCTCTCAGGCGCAATTACTCTCTTCCATATTGGTTAACAGTAGAAGGCTCAGTTTCTCTGC
FP011332	chr11	111541165	111541416	+	1	train	TCCCAGTCCAGAGTTATGGGGGTGGGTTGGAGAATGTAGGAGAGGTGGGGCAGTGGGCAAACCTGCCCCAGTGGGTGCCCCTTCGGCCCGCTCCTTTACACGCATCGAATCAGTCCTTGGGACCACCCCCGCGGGGCAGGCTTGCCAGCTGGCCCCCGCCTACCGCCCGCTCCGGGAGGCGGCCGCAGAGACAGCGGGGTAGGGATGCTGGATCCCCGGTGCCGTGGGAACCCTGCCTGTTACTCCTCGTC
FP004407	chr3	194351251	194351502	-	1	train	TGGGATTTTGGCCCCGGCAGGGCTCTGTACATGGGAGGAGTTCAGGGTCATTTTGTAATCTTCAGAGGCCTTGCCAAGGCCCTCTCCCGTCCTGTGACCCTGGTGACATTCTGGGACGCTGGGGCCACGTGAGGGGAGGAACCACTGCCTGGAACTTTGATCAGCGCTGGGGCTGGATTGAGCTGACCACAGGCCACACCAGACTCCTCTCTGCTCCTGAGGAAGACAGGGCAGCCCGGCGCCACCCGCTC
FP012019	chr12	54984711	54984962	-	1	train	TGCCTTAATAGGAAAAGATCTCCTGTTTTTCTGGAAAGACAGAGGCAGCTCTGGGGCAGGAAGGTCTTCCTAGTTGTGAGCACCGTGTCTACGGATGGTTTGTCCTTCGCTCAGTCTCCTGAGCCGGAGCAAAGGATGTGGCTGGGCCAGGAGGCGGGGACTGCATCTACATGTGTGGATGTGTGTGTGTGTGGTGTGTTCTCATTTCCGCTTCCCCTTTTGAGCAGGCAACAGCAGTTGAATGAGTTCAG
FP015598	chr17	42017256	42017507	+	1	train	GGGGGCTGCATCATGGCAGGAACGAGTTTCCCTGAGCCACGGAGCTGCAGGTCGCCTCCTCTACTACCCTGCTACCCGGTTACCTCTTCGCCTCTTGGTCGTCGAGCAGCTCCGGCTCGGTCGCCGCCATTACCACATCGCACTCCGCGGCAGCCGCCATCTTACCGCCGGGACCAGAGAGCTGGGTGGGAGGCCGGCGGTGAAGAGCACTTCCGGTTGGAGCATAGAGCAGCTCCGCGGCGCCGCGCCCA
FP011387	chr11	118152732	118152983	-	1	train	AGCAGAGGCGCAGCCTCGCTCGCCGCTGCGCTCGCTCCGTCTTCCTCCCGCTCCTGCTCCGGCTCCTCCTACTCCTTCCCTCCCGCCGTTGCTCCAGCAGCCGGCTCCCAGCAGCGGGCGAGCGCGCCTCCCCCTCCGCTCCCTCCCTCCCCCTCCTCTCGCTCTCTGCCCGCTAACTTTCCCGAGCCCCGACCGGCGGCGCAGAGCTCCGGGGTAGCTTTGTGGCCGAACGCCGACCTCGGGCGGAGAGC
FP001812	chr1	220786672	220786923	+	1	train	AGCGATCGTGGGGTGGGGGAGGGCACAGCGCCCTGCAGCGCAGGCGACGGAAGGTTGCAGAGGCAGTGGGGCGCCGACCAAGTGGAAGCTGAGCCACCACCTCCCACTCCCCGCGCCGCCCCCCAGAAGGACGCACTGCTCTGATTGGCCCGGAAGGGTTCAGGAGCTGCCCAGCCTTTGGGCTCGGGGCCAAAGGCCGCACCTTCCCCCAGCGGCCCCGGGCGACCAGCGCGCTCCGGCCTTGCCGCCGC
FP003261	chr2	237591038	237591289	-	1	train	TCAGAGGCTCTGGAAATCTTACCTGGAGGAGACTCTTCTGTTACCACCAAGCCAGGAAGTTTCATGGCTTGGGAAGACCAAGAGGAAGTCCACCTCCCCCTCTCCTGCACCTGGAGTCTCCCATGTGCACGGAGGCCACGCCCCCACCCCGCCCTCAGGTGGCATCAGCCCAGGATCTGACCCTGGACTGGCTCACACTCAGTTGCCTCTGGCCAGTGCAGGGCTCAGCCAGGGATGGCTTCTAGCTGACA
FP013695	chr15	40807422	40807673	-	1	train	ACCTAGAGCAGGGAATCCGGACGCTCTCCTGCAGCCAGCGTACGGCAGCGGCGGCAACGGGGGCTGCTGGGAGTCGTAGTTCATTCACGACTGCCGGCCTCCCTCAGAGTCAGTCAAGCCTGACCCAGAACACAATTCCCAGAGGGCTAGGCGCCGCTCGGAGCCTGCAGTCCTCACGCGCGCTTAGACTCTTGGGAGTTGTAGTACGAATCCGTCAGGCCGGAACCATGGCAGTGACCAAGGAGCTCTTA
FP011107	chr11	71479035	71479286	+	1	train	TTGCAAACATTGAAGAAGTGTGATGCTCATAACAGAGCGGACTCTTCTGCTGTTGATTGATTGAGACAGGGTCTCATTCCATCGCCCAGGCTTGAGTGCAGTGGTACCATCACAGCTCACCGCAGCCTGGACCTCTGGGGCTCAAGTGATCCTCCCACCTCAGCCTCCTGAGTAGCTGAGACAACAGACACACACCACCATGCCCAACTAATTTTTGATTTTTTTTAATTTTTTTTTTTTTTTAGAGACAG
FP006647	chr6	53794778	53795029	+	1	train	GTCTGCCTTCCTCAAACGGCCCGATGCGCCCCCGTCTTTGCCGGAGTAAGCAGACCGCCAGCAGCCGGCCCGCAGGTCAGTCGACCCTCTCTGGATGCAGGTCGCCGGGAAAACCCGGAGCGGAGCATCCCTCGGGCCGGGAAACGCCCTCGGCGCGCACCCACTGGCGCGCATGCTCAGTCCGCGCGGCGGCTGCGAGTAGGAAGCTCCGCGCGGCGGCGGGGGCGGCGACGGCGACTGGCGGGTGGGAG
FP009573	chr10	3068295	3068546	+	1	train	ATGCGGGACGCGGCGGGGGTGCGCGGGGTTCCCGCCTGGCGTCTCCAGGGCGGTCCGGCAGCCCCTCGCGTTTCCGCTGGGAGATTTTTAAAAAGGGTTTCCTGCGCTCGCATGTGCTGTCGCCTTCCGGCTCGTGACAAAAACGAAAACCAAAATGGGTCCCAGGCCCTCCCCGCCAGCCTCCGCAGACGGGAAGGATCAGGAAGCGGGGACTCCGGGATGTCAGCGGCGCCGCGGCGCGGGCTCCGGGC
FP003036	chr2	202238513	202238764	-	1	train	ACTGATGCGCAGGACCTGGTCCTTGAAGGAGCTGACAAAACTGCCGTTCACCCGCGCGACCACCGGCAAAGCAGCTCCACAGCCTGCCCCTCCCCTTGGCTGCTCCCCCACCCATTTCCCGCCTTGTCTTTCCTCTCTTCTCTCTCTCCCTCCTCCCTGCGCGAAGCGGAAGTGACGCGAGGCGTAGCGGAAGTTACTGCAGCCGCGGTGTTGTGCTGTGGGGAAGGGAGAAGGATTTGTAAACCCCGGAG
FP004639	chr4	37826517	37826768	+	1	train	AGCCCCGCAATTCCTCCTCGCTCACTCCCACCCCCTGCAGCGCACCCGCAAACGGCAAGGAGGCGTGCCCGCCGCGGCGCGCGTCACTTTCCCCCGGACCGACTGGCTCCGCCGCGGCTCTCCCCGCCCCCGCCTCCGCCTCCACGTCTTGGGGCCGGGCCGGAAGGCAGATCTCACCGCCTGCTTCCCTCTGCAGCGGTAGCACAAGCTCAGCGATGGCGGCTCCAGAAGGCAGCGGTCTAGGCGAGGAC
FP005048	chr4	140523967	140524218	+	1	train	CCGGCGCGGGGCGGGGCTCCTGGCCGCCGTTGCCAGGGAAACCGTCTGGGCGTCCCTAGTATCTGAGCCCTCGCGCCTGGGCCACTCAGCTCGACCTCTCTCCAGCGTTAGCAATAGGTGGAGCGCGGGCCAATCGGAAGAGGCCGGAGGGGCGGGGCGGGCAGTGGCCCTGGGAAGGCGGGTCCGCGGCCTCGCTCCCTGACTTCCGGGTCGCGGTGCTTGAAGGGAGTGTTCCGTCGTTTCCGTTGCCG
FP017839	chr20	916486	916737	-	1	train	GTTAAGTTGCATTTGGTTTTCCAAAGCGGGCAGCATTGGAGCTTGGCGTTTGTGTGTACATGAATGTGTGTGCGTGTCAGCGTGCGTGTGAGTGTGTGCTCGTGAGTGTGTGCATGAGTGTGTGCGCGTGTCTGTGTGCTGTTTTCTTCCCTCCCCTCCAGCAGGGAAGCCAGGCCTCACTGGATAATCAGAGGGCTCAGAGGTGAGGCAGACAGAGCAAGGTTGGTGCACGCTTAGGGTAAGGGCGACGT
FP019314	chrX	54043112	54043363	-	1	train	TTGAGTTGGGCGCCTAGTCTTGTCTGTCTTGTTCGTGAAGGCTTTGAGTAACCTGAAGAGACTTTCAGGGATCCCTTATCTCTCTCTGGTTGGCCCTATTTCAGGTGGTGAGAGGAGGGTCTTGTCTTGAGATACCCCCAGCCTCATTTTTGACGCTGTGGTTTGCGTCATAATAGAGGGAGCCCTGTGAAACAGGGGTCAGCTAGGGGAGGGTTCAGTGAGAGATGAAAGAAGCTTCCCGAAAAGGCTTC
FP012790	chr13	49451384	49451635	+	1	train	ATTTTTTCCAGATAACATAATGTTTTTTTGTAATGGTTCATGCTGTTTTAAAAGATCAAGCTGTTATAACAATATCTTATTTCCTTATATATACATATATATATATATATATATATATATATGTGGTCCTTTACAAATATGCACTATTCTTTATTGTTTTCCTTACAGAATGTTTCATCCATTTGTGGACCAAAAGATGGAGTTGGTTTTTATTTTTAAAAAGATAATGTTAATGATCTGATACCACTACA
FP016668	chr19	6464072	6464323	+	1	train	CTTCCGCGGTCTCCCTCCCCGCATCCCCATTAAGGGACTGGGGTCCCGTTACAGGTGCGCCCACAGCGCGGCCCCGCCCCCTTCTTGACTCCGCCCTCTCAGCGAGGCTCAGGTGCACAAGCCGGAAGTGCGCTCTCCCAGGTGAGTGACTAAACTTTCCGGGTCACGTGACAGGGCGGAGCCGGTGGGGTCGGACCCACAGAACGACCGACGGACCGAGGGTTCGAGGGAGGGACACGGACCAGGAACCT
FP007331	chr7	37848213	37848464	+	1	train	AGAAGAAGAAGAAGAAGAAGAAGAAGAAGAAGGAAGAAGAAGAAGAAGAAGAAGAAGAAGAAGAAGAAGAACAAGAAGAAGAAGAAGAAGAAGAAGAAGAAGAAGAAGAAGAAGAAGAAGAAGAAGAAGAAGAAGAAGAACAAGAAGAACAAGAAGAACCAAACCAGGGTAGATGAAGACAAAGTTAAAAAGAAAGAAAAGGCTGGTGCTGGCAGAGAGCCTTCTGCTCCGATTTGAGGTGTATGCATTTT
FP019305	chrX	53281528	53281779	-	1	train	GGAGCCCAGCCTGGGGTTTGGCCCAGAGGAGGGACTGTGAGCTCTCCAGGCCGCTTCCTCGGAGGAGGAGTTCCCAGCCCTCCTGGAGGAGTAGGGGAAGGAGCAAAGTTTCCAGTTGCAATCTGGCCCGCAGCCCAGCTGGCCTGGCCCCGCCTTCTTGGTTCCTCCTTCTAGGGAAGGGCTGGGGAGGAGGGGGTGGTGACACGGTGGAGACACCGGCTAGGCCAGGGGGCCTGCCCTTGGGACAGGTC
FP004675	chr4	42152300	42152551	-	1	train	AGTTCCCGGGCGTCCTCCAGGTCCTCGCTTTCCCCCTTCCCCCGCTCCCCGTCGGAGGCGCGGAGCCGCCAGGCAGACCCCGCGGGCGGCTGCGGCGCCAGGCCCCGCCCCGCCCCCATTATCATTAGCAGATATTACCCTAAACACTTGCTCTTTACCTCCTTTCTCCCTCCAGGCATTGGCGAGGCAGCCTGTCAATCAGGAGCTCGGGCGGCAGCCCCCCGCGCGGGGGCTCGGCGATGCCAGCCTCA
FP013996	chr15	74202732	74202983	-	1	train	TGCGGGGGTGGGGGTGGCTCCTGGCCACACCCTTGCAGGGCTCAGGCCGCCAATGGTGCTGGGGCCTCCTCCCCCAGGAGGGGGGCTTTGTCAAGGATTATGGTGTATTTCCATCACCGAGGCCCAAAGGGGCCAGAAAGGGGCAGGCTGAGGCTGGGGGAGGGGCTCAAGGGAGCCCAGCTGGGAGGGCCCAGGCAGGCAGTGCGGAGCTGGCCGTGGGCTGCCTACCCTTTCATCTCTGCAACTCCTTC
FP004209	chr3	154121189	154121440	+	1	train	CCGCAACTTCTAGCCCCAAAACGCTGGAGCTCAGAACCCAGCTCAGGTGCGCTCGTCGGTTTCCCAGGGAGACTGGGGCTGATTGTAGGCACGCGGGGGCGGGAAGACAGCTTCTTGCTTCTGGCTGCGATGGAGGCGGATCTGGGCGGGGCCGAAACGGGGGCGGGGCCAAGTGTGGGGGCGTAGTCAGGCGTGGGGGCGGGCCGGCGCCGGCGCCGCGGTCGGCGGCAGCGCTCTCCTAAGCTCTCGCG
FP007029	chr6	150866314	150866565	+	1	train	GACCGCACGCCCGGCTCCACGTGCGCAGCCGGCCGCCGGCTCTGATGCAATCGCGCCGGGCGCGACCCAGACGGTAAAGGGGCGGTGCGGCTGGGGCGGAAACCAGGGGAGGGGGCGAGGAGCAGGAGGAGGGCGGGGCTGCGGCTCGGCGCGCCGGCTGGCCCGGGGTTCGGGAAGGGCAAGCGGAGCTCGGGAGAGGCGGGCTCGGGCCCAGCGCCGCCCGCGCGAAGCTCCCTGGTGTTGTGCGCCCT
FP009797	chr10	49610124	49610375	+	1	train	CTCCAGCCCTGGAGCATCTGGAGAGCGGACCCCTGCCCGGCCACGCCCCGCCCCCGGCCCCCGCCCCGCCGACGTCGCATTAGCATGAGCGACGTAAGTGGCCCGGGCACCACTCGGGGGCTGGGACTCGCCGCGTCACAGCCCCGAGTGGAAGGGAAAAAAAAAGAGGAGGCGGCGGAGGAGGCAAGAGCCGACGCGAGGGGAGGGGAGCGCAGCGGCGGGGCTAACGGGCGGGCAAGCGGGCGGGCGGC
FP014652	chr16	31213824	31214075	+	1	train	GGAGAGTGTGGAGGTGGGAGTGGGGGGAGAAAGGGCAGACCCCCTTCAATTTTCTGTCTTTTCTGTGGTGTGAAAGGCCCTGGCTCCCCGCTGGCTCTGAGCTCTGACTCCAGGCAGGCCGGGGAGAGAACAGTGGATGGGCCAAGACAGGTGGCCCTGTCATGTGATGTGTGCCCAGGGACTACTCAGCCCTCACTTTTAGAACGGTTTCCGGAAGTGATGGGAGGGATTGGGCAGGGCAGCTAAATATA
FP010570	chr11	12377370	12377621	+	1	train	GCAAGGGGCAGTTTTGGGGCATCCTCCCTGCTTGGGGCAACCCAGGGGCGCAAGGCGCGGCCCCGGGAGCGCAGCTCCTTCCAGGGCAGAGGGCGGCGCGAGGGAGGGAGCGAGGGAGGGAGCGAGGGAAGGGAAAGGCGAGCGTGAGCTGCCTCAAATGCTTGGAATAATTCCGCTTCCGTTTGGAAAGCCGCAGCCTCAGTCCCGCCGCCGCCCGCTGCGTCCGCCCAGCGCCAGCTCCGCGTCCCGAC
FP019742	chrX	153793958	153794209	+	1	train	CTCCTTCAAGGACACTAAGGAGGTGACAGCCGCCGTGGCCCACTCTCCCGGCCCGTCCGGAGGGCCCCCCGCCACCCGATGCAGACCCCCTGGGGGAGCCGGTCCAGGGCCAACTTCGGCCAATCCCCTCAGTGACAGCGGAGGCGGCCAATCAACCCCGGCGCGAAGCCCTTTCCCCGCCCCTGGTGGGGCCCCTAGCCAATCGGACTCCAGACTGCTTCGGGTGCGGCTACCCCACCGCTCCCCTGCGA
FP018075	chr20	36092509	36092760	+	1	train	TCCCCTGGTGTTGGCTGCTCGCCGCCGCTGCTTGCTCGTTCGCCCGCCCGCTGCTGCTTGGGGGGCCGGGATCGCCGCCGCCTCCTCCGGGGGAGCCGGCCGGGGCGGGGCGGAGCGGCGAGAGGGGGTGTGGGGGGCGGGCGAGGAGGGGGCTGAGCGCCGTGGGAGGGAGAGCAGGAGCGAGCGCGCGAGAGAGCGGGGCATTTCTGCGCAGCTCTAGCGCGCCTCGGAGCCCGCCGGGGCAGCCGCCG
FP016633	chr19	4831661	4831912	-	1	train	CCTGCAGACAGTAGGTGCTCACACTTGCTGAACGTGACCGCAAGGGTTGGGAAGAGAGCGCATGGGTGCGGGGCGACCCGGGGACTTTCCCGGTGCTGCGGGGCAGGACTCCGAAACCGTGACCTCACGTCCCACACCCACCCGGGACGCGGAAGGGGCGGCGCGCTTCCTTCCCAGGGCGCGGGCCGCGGGGTGGAGCCAGCGCCCTCAGCGCGCTACGGTCCGCGGGCAACTCCGCAGAAGCCCCAGCC
FP006184	chr6	11778752	11779003	-	1	train	GCAAATCTACAGATTATCTATCATTATCTAAATGCAGGCATCTGAAAACCAGCAGTAATCCTGCCTCTGAAGTTTATCAGGAAAGGAGCTTAAAAGAGAACCAAATTCAGCCTGTGTTGGAACTCTCAGTCCCAGAGGGGTGTGGTTTGTAGCTCTCCGGCCTGCTGTTGGACTTAGGCTGTGACCCACAGAAGGACGCCAGAAAGTACTCAAGACATTCACGGTGCCCCGGTCAGCACTCGCCATGACGA
FP002642	chr2	118014019	118014270	-	1	train	TACGGGCGCGATGATCCAGCACAACGCCAGGACCTGCAGCCCAGCCCACCTCTCCCCGGGCTGCTGTGGCGCAGGCGCAGTGGCGCCGCGCTCCGGCCCCAGCGCGCATGCTCTGCCCGGCTGCGGCGCTTCGGGCAGGCGGCGGCGGCGGCGGCGGCGGCGGCAGAGGGAGTTTCCGCTTTGTACTCCACCCCGGTAGCAGCTCCGCGGCAGGGACAGCTTCCTCCGGACGCTTGGCGGGCTTCGCTCTC
FP018971	chr22	46250242	46250493	-	1	train	GCCAGACGCGACGCCCCTCAGCACGGAACCCAGCCTCGCGGACTACCCCGAAGCTCAGAGCCACATCCCGCGCAGAGGACCGCGGCGACCGGAAAAGCAAGGCAGAGGCGGAAAACCACCGCGAGGTCCCGCCTCCCGCTCCGCGCCTGCGTCTGCGGCTGCACCGGGGTTGGCCGGGCCGCGGTGGGGCGGGGCCGCTCAGGCGCCTTGGACGCGGAGCCGTGGGACGAGGGCGGCGGTGAGTTCGAGGT
FP019356	chrX	68433444	68433695	-	1	train	GGCCGGCTCCCTTGCACCAGGAAGAAGTCTTAGCAGCCAGCGGGCCCTGGTCAGGAAACTCTAAGGTACAAGGAAAACAGTTGAGGAAGGAGCCAGAGCGCTCCGGTTTGGTCCTCGGGCTTCGCTGGGGCGGGGCGCAGGCGTTGGCTTTAAGAAAGGGGAGGGGACAGTGCAATCCGGGTTGCCCGCGGAGTTCGGCCAAGGAAGTCTTCCGCTCGCTCCGGAGCGAGGAGCCTGTAGAGAGGCTGTTC
FP019655	chrX	136196844	136197095	+	1	train	ATGCTGAGTAGCCCCGCAGGTATTTATATGTAGCGTTTGGTATTTTTACTGGGTCCTATTCTTCTCCCTTGGATCACCTAAAATGGTTAAGAATTTTCTAGTGGGATTAAGGTTGTGATCTCTGGGGAGGGGTGGGACATCTGCTCTCGGTTATTTTTGTGTTAGCCGCTTCCCCTTAAATGTATGTTCACAAATGAGCCACAGCCTTATCAGCTGGGGTTGAGGGAAGACTGGTCTAGGTGCTGCTCCTG
FP016084	chr17	79027604	79027855	-	1	train	TTCAGATGCCAAGCCCAGCTCTGGAAATCAAATGTAGGCAAGAACAGGCAGCTGGAGGGGTGCTGATTTGCCAAGAGCCACTCATATCACCCCAGCACAAGTGTCACACAGCCGTGACCTTGACAAGGACCCAGAGATAAAGATCCTTCCCACATGGCTCCGAAGCCCCTCCCTTCCTGCTCACACCGCATGCCTCTCCCAGAGAACTAGAGGCTGCAGCGGCAGCAGTTGGAGCATCTCACTGCGGGGCT
FP004302	chr3	179347534	179347785	+	1	train	AGGGGAAGAAGTTGGAAGGGGGAACGGAAAACCACTCTCCTGGTAGCAGTCTCTGCCCACAACAAACCTTGCCTTCTGCCCTTAAGGAAAATGGCGCCTGGCATAGATGCGCTTCCGGCCACGGAGCCAAACAGAACAGAGGAGGCAAGAGGCCGGAAGTGACCGCCCTTTGCCACTCCCCCTGCCTCCTCTCCGCCTTTAACTTCTCGGGAAGATGAGGCAGTTTGGCATCTGTGGCCGAGTTGCTGTTG
FP019041	chrX	2500925	2501176	-	1	train	GGAGTCTGTGGTCCTGGGGGGGGTGTCCTGGCTGGGGTCGGTTGTCCGCTCGGGGTCCCCAGTCCTGCAGGCGGTGCGGGATCCCAGAGGGCCCGGGCGGGGTCTGGGGTTGGGGGTTCCCCGCAGGCCGGGGCGGTTCGGGGCGGGCAGGGGGCGGTCCGGGGCGCGCGCGTTTCCGCCGTCCGTCCGGGCGGGCGGGCGCAGAGTCCCGCGGGCGGCGCGGAAGCGGCGGCGGCGCGGCCGGGGCAGCC
FP000242	chr1	19485482	19485733	-	1	train	CCCGCCTCGCAGGCCCCGCCCCCGCACCCCGTCGGCGCCCGCCTCTCGCGCCGCTCTCCATACTTGGCGATCGCCGCAGCCCCGCGCGCTCATTGGCCGCGAGCTGGCGGCGTGGGGGGCGGGCCCGGGCCGGGCCGGGGCGGGGAAGGAAGGTGGCGGCGGCCCGGCGCGGGGGGAGGGGGGTGCTGACCCGGATGTTCACTCCTGGGCACCCGGGGAAGTGGAAGCGCCGGGCCCTGCTGCGGGGGGGA
FP019458	chrX	101349176	101349427	-	1	train	ATGCAGCCTCCCTGAAGCGCCCTATCTCAGTATTCCAGAAAGCTGCTGTGTAGCATTAGTTTGGCATATTCCTACACTGAGAACTCGACAGGATCCTCCAGGTTTAAGCCCTCTTGTTAACTCCTGCCTCTGAATCAAAGATGCATTAAGCAAAAACAGTGCCTGTCACAGCACGCATGCAATGTTATTTGAATGAATCTGAAAGTTGGGCACAAACTGCCATAGCCTCATTTCCCTACGTCGGGAGAGGA
FP015102	chr17	2303734	2303985	-	1	train	GCCTCCGCGCGCCTCCCCGGCCGCACCTCTGCGCACGCGCAGCGCCCGCCCGCCCCACCCAGGCTCTCGCTCCGGTCCCGCCCCTCTGGAACCGGCGGCGGGCGCTTTTCCTCCCAGCGCGGCCACCGTCGTTCGGAAAGGGAGGGCGGCAGGGGGCGGGGAAAGGCCGGGAGGTGGGCGAGCGCGCGCGCCGCCCGTCTGTGGTGGTTTCCTGGCTGCGCGCGGCGGTGGCGGAGCCGCTACGGCTGTAG
FP014613	chr16	30526695	30526946	-	1	train	CGGTCCTCACCCACCCCGTGGCCGGGCCCCTCCAGCCCAGGGTTAATGCCAACGAGTTTCCACCTCCAGCGGCTCCTAGAGAGGCCGCGGCGCGAGCCGGCGGGGGGACCTGGCCGCGGACGGACACTTCCTGTGCGGGGGAGGGTGGGGGGGTGCTGGGCGGCCGTTGGACAGTGCAGCCCAGCCCGGGGCCGGGGTCCAGAAGGTGGGGGCTCGGGCGGGAGCGGGGGCCGGGGGCCGCCGGGGTAGTC
FP014136	chr15	89263223	89263474	+	1	train	TCAGAATTTTGATTTTGCGAGTCGGGTAGGTTAACAACTGCTATTTCATTGTTTTAAATGGTATTTATTTGATCTTGTTTCTCGGTATTGCAAAATCAAACTTGAATTGGCCCTGTTTTTTTGTCCTCATTAATTTTACAAAGTTATATCTAAATAAGTTGTAAAGAAATAAACTTTGTCATTTTCTTCTACCAGGTGGGATCAGCAATATGTAATCCAACTCACCTCCATGTTCAAGTAAGCATCATCTT
FP010179	chr10	103452678	103452929	-	1	train	GGACCCGGAGAATCCCTAGGCTCCTGAATTCAATCCTCAGCCCTCCAGGGATCCGAAGCAGGTCCCGGGGAGTTAGCTGACTATAGGTCAAAGAGTCAGCATTGGGGATGGTTTGTCCAGTCAATGGACAACTCTGAGGGAGAAGGGCCAGTAGAGGGTGGGGCCCTGGCCCTGAGCATCCTGCAGGGCTCAGCGCGGGCCTGACGACACCCTCCCTTGACCCTCGCGGGGTCTCCTTTGGTAGCTTCTGC
FP019504	chrX	105220435	105220686	-	1	train	GGAACCTTAGGAGGGAGAGAGATCTTCCTCTCTCTTCGGGCGTGTTAAGACAGCGGGGTTGGCCTGTACTTCCTCTGGCCCTGGCTGAAGAGGTGAGGCCTGGTGGGAGGTGTCCTAGGGTAGGACAAGCCGGTCAGGGGGTCATTAGGACGGTCTTGTCAGAGCGGGTAGGGCGGGGACAAGAGGGCGGGAGAAGATGGATGAGGGGAGGGGCTAAGGGGGAGGAAAGGAACCTATTGGCTGCTCCATCC
FP005196	chr4	185678769	185679020	-	1	train	AGATATTTAATGAATGACATTCTAATTGAAGTAAAAATAAATGTACAAGCCTCCCTTTGGCATGTTTGACATACCCAGATTTTAAAACTGAAATAATGTTTGCAAAAACAATGCATATAGTTCTTTGCCATGTTATTTTTACTGGGTTGTTCATTTATTTCACCTTAGTCTTAATTTCAACATTTTTTTTTCTTTCAGGGCGTGATTCTCAGTCACCAGACTCAGGTACAGAAAAAATATTCTGAATAATT
FP007545	chr7	90346514	90346765	+	1	train	ATTATCTCTGTGTTTTAATAGATAAAACGCCTTCTCTCCCCACCCGTGGGGTGCAAAGCACAGCGCATGGGGCTGGTGGCGCCCGAGAGCATCACACAACGCATGCGCCAGTCCGCAGGTGTGGGCGGAGGAGAAATCGCGTCGGCGGCAGGGGATGACGTAAAAAGGCCGCGCTGTACTGCGGCTTGTGCCGCTTCCGCAAGAAGGTTTCCTGGCCTGTTGCAGCCATGGTGCATTGCAGTTGCGTGTTG
FP011485	chr11	126355811	126356062	+	1	train	CGTGCTGCCCCGCCCCGCTCCACCCGTGAGGGTGAGTACGCGGCGGCGGTGCGCGGGGGCCCGCGGGGCGGGGCGGGGCGGGGAGCCGCGGGGTCCTCGGCCGCCTGACCCCAGCCGGCGCCGCGCCTCCCGGAGGGGGTCGGGCCCTGCACGTGGGCGCAGCGCGGGTCGGGGTGGGGCTGCCACAGCCCTGCGGAGCTGCTTTCCGGGGTCCCTTTCCCTGGACCAGATTTTCGCGGGAAGCCGGATCC
FP000388	chr1	27623770	27624021	-	1	train	CTGTATGAGCGTATGAGCATGTGCATGCGCGTGTGTGCACAGGGTGGTGCACCTGGCAGGGGTCCTTGAGTGAGGCATGCCCCATTCTGTAGCAGGGAACCTGGAATGGGCTGTGTGTTCTGCAAGAAATTGGAGCCGGTGGCCACGGCCAAGGAGGATGCTGGCCTGGAAGGGGACTTCAGAAGCTACGGGGCAGCAGACCACTATGGGCCTGACCCCACTAAGGCCCGGCCTGCATCCTCATTTGCCCA
FP017340	chr19	43205587	43205838	-	1	train	AAGTGTGTGTTGAGGTTTGGTGAAAGAATCACTGCTGAAAAATGCAGAGGCCTCCACAATTCCCAGGGACCTGAAACACAGACAAAAGGAAAAACAGGAGGGACAAGGAGGCAGAACTGAGAGAGGAGGGGACAGAGAGGTGTCCTGGGCCTGACCCCGCCCACGAGCTTGAGAAATGCTCCTGCCCCGGGAGGAGGCTCAGCACAGAAGGAGGAAGGACAGCACAGCTGACAGCCGTACTCAGGAAGCTT
FP006657	chr6	56954684	56954935	-	1	train	CTGCTCCTCCCCACTCGCCGCGTCCCCGGCGGGGCTGCGCCCGGGGGCTGCGCACGGACGGGGGCGGGGGCGCCTGGGGAGGGAGGGGAGAGGATCCCAGCCGCCGAGTCGCAGCCTCGTCGCTGTCTGCTGAGTCATGGCAAGCGCTCCCCGCAGCCCTGGATCACTCGCGTAGCGCGGCTGGCGCGGGGTGCGCAGCCATTGGGCTGGGCGCGGGGCCGCGAGCGCCAGGCATGACGCGCTGAGCCGCG
FP003204	chr2	230416738	230416989	+	1	train	GCTTACTTTTAAACATCAGACGCTGTGGTCTCACCTGTCCTGGCAAGGGGCCTCTGCCGGCTGTTCCCATGACTGGCTCAGGGTCTGAGTTCTTATTCCATCAACCTTGATCAAAAGAAGGAAAGGGAAGAAAAAGGCCCAGGGTGACAAACGGCTAAATATTTTCATTTTTACCTCAATTTCCTGTCCCAAGATGGGGGATACCTCTGACCAGAAAAGAGAAGGAAATCCTAACTTTTCTCTAGTAGTGA
FP011801	chr12	31659487	31659738	+	1	train	TCATTAAGAGAAATACCAAGTGGTCACCACAGTCACATACTTCATAACCCATGCCTCATCGGTGCTGGGACGGGAGAGGGAAGCGCCGATCTAAATGACATGGGGTCCAAATCCCAAATAATCTCTATTTGCTCTGCGTTACCTTTAAACTTAGGCTGCGTTGTGTGATTCAAGTGACAGCCTAGTACTGCTTAACAATTAGTCTAGTGGGATACCAGGCCCTGAGAGACCCCTACGATCCCAGGACACTG
FP000532	chr1	37793888	37794139	+	1	train	TTCGCGGGTCACGCGGCCTGACTCAGGCCCCTGCTCCTGTGGCCCCGAAACTCGCCGTTCGCTGGGCCTTGCGTTGCACTCGGCGTGCAGTTCCCCCTCGGCTCGCGGCCACTTGGTCCGCGCCGCCTGCGTCCTGTGTGCCGAGCCCGCCCGCACTGCGGGGACAGGCCGGGCGCCGCCCACGCCGCGCTCTGCCGGGCGCACAGTCTGCCTGGGAAGCGCGCGGCCGGGCGGGCGGCCATGGCGCGGCA
FP013313	chr14	69191297	69191548	+	1	train	AGTAACAAATGCGTCACTTTTCTGAAGAAGACGTTAAGCCCTTCCAATTTTAACAGATCCTTTGTTCAACAGAGGAGAGAGACTGGGTAGGCATCACCTGCCCCAGTAGCTTCCGTACAAAATGGCGTTGCCCGTTCCCACACGCCAGCTGGGATTACGTCCTCCAGGCCTGTCGCAGGAACGGAAGTGGTTACGGCCCTAGTGAATCCGGGTGGGTGCGCGCAGGCGGCCGCACAGGTTCCAGGTCTTTA
FP013206	chr14	54441289	54441540	-	1	train	GTGCGGACAATGTCAGTGCAACGCAGACTCCGGCCATCAGAGGGCGCCGGGGCGGCTGGGCTGGTGCGCAGGCGCGCTGAGAGGGCTGGCGCCGCGGCGGTAGCGGTGGCGGTCGCGGCTGTGGCCGGGGGAAGTGAATGGTTTTACCCAGAGGGCCCTGCGCCGCCTTTCTCCGCTGGCAACGGCGCCGCTCCCCGCTCCTCCTCCCCAGCCATGGCGTTCACGTTCGCGGCCTTCTGCTACATGCTGGC
FP004138	chr3	138434385	138434636	+	1	train	GTTGGTGAGTCATTAACCACCTAGGGCAGCACTTGGTCCACCCGGACTGGCGAGTGCACACCTCCGGCTGACGCCTGCGCCACGTGCTGGGGTCGGAGGCCGAGCTCACCGGCTGGTTGCTCTTTAGAGCGGGCAGGAGCCAGGCGGGCGGCCCGCCCCTCTAGGGCGGAGACTTCCGGAGCTGCGGACTCACTGGAGCCGAGAACTGGGGCGGCGCGGCGCGGCGCGGTGCATTTCCAGGCGCTGCTCTC
FP017145	chr19	35757976	35758227	+	1	train	GGCAGGTTCTCCAATCAAACACGCCCTGTCGGGCAGGCAGTCGTCCCAGCCAATCCTGAAACACCTCCCTCCGGGTACTTCTGTTTCCTCTCTTTCATTGGTCGCCTTGGAGGCCGCTCGCCTCCGGCGCGGGCACAGAGAGGGGCGGGGCTGATAGGGCGTTGCTAAGCGACGGAGATGCGCGCGGGGCCTGTTGGGTGAAGGAGCAGAGCGGCCGGAAGCGCGGAGGGAGCCGCGGGATGGACCGCAGG
FP007424	chr7	64990868	64991119	-	1	train	TATACTCCCTTGTCTCTTACCCCTCATTGTCAAAAATATCCAAACAGCCATAGAGGCTCTTGTGGACAGACGGACTACCACACGACTAATGGCCCTAACTAAGTATTAACCCCTGCCAAGAAAAGAGCTACTTCCTCTTGAAGTAAATGAAGATAGTGATGCTTTCTCTTAAACTTTACTTATAAAAAGCATCAAAGGGGGGAATGAAGCAGGAAATATAAAAGGAAAAACAAGTAAAGGGAAAACAAGTC
FP017549	chr19	49453072	49453323	+	1	train	CATCGCCGTCTCCCCTTCCTCCCCGTTTTAAGCAGTGCGCATGCTCCTAAATACGCTTCCCCGTGGGGGACGTAGGCGGGGCCATCGGGTCCTAAGGAAAAAGTGTGGGCGTGGCTGCTCTCTGTATCTTGTAGCCCCACCCCATTCCCATTAGCCCCGCCCCTTTGGGCTGGAACCGGAGGTGTCGCTCTTCGGACCTCAAGGTTCCCCTTAACACAGAGCGCCCCGCAGTCTTCGCGGAAAGCGTTCGG
FP019206	chrX	46545357	46545608	-	1	train	TTCCTTCTCCCCTTAGAGCGAACACTGTCATAACCACTGAAGCGCGGGCCGAGGAGCGCCACATCGGCCACAGGCTTTGTTGCCCCGCCTTCAAAGCCACTGCGCGTGCGCATTGCTTTGACTTCACGTTCCCAGGGGGCTGTGGGAAGCGGAAATGCTCGGCTGACCCGGAGGGTGGCGCTTCGTTCACACTGGCTGTTGTCATCCGCCCCTGAGGTTTCAGGCTGTCGGCAGTGAAGTAAGCGGGAGGG
FP004244	chr3	159988639	159988890	+	1	train	GGAGGTCGTGGTTAGGGCCCATCCCTACGCAGGACATGCAAAGTGGGAGGCACTCCTCTCTCTACGTCGGCAGGGGGCGCTGCACAGCTGCGGGGCGGGGTAGCTTAGACACGGGGCGTCCGGCTAAGGCCGGGGACCCAGGGTGGTGGGCGGGGTGTCCCGCCCGCCTGTGGACCCCGCGCAGTAACTGCGAACATTTCGCTTTCATTTTGGGCCGAGCTGGAGGCGGCGGGGCCGTCCCGGAACGGCTG
FP016834	chr19	12484741	12484992	-	1	train	AATTCTGCTTAGACGCAAATATCCGTGGCAGGGGAACAGATTCCTGACCCCGGACGAATGCAGGTTGCTTAGAGACGCGACTTCCCTACTAGCTGTCACTCAGGCAACGAAGGCGGGTCCAAGAGTGCAATGTCCAATCAGGGGCGCCGCCGGGAAGGCGGGCGGAGGGGCGCTGCGCAAACGGCGAGGGCGTGGCTACCAGCCGCTGCGCTCCGCTCTGGGGTCTCCTCACCCCAGAAGGCCCTCAGGTG
FP019542	chrX	112840780	112841031	-	1	train	AAACTTCCACGCCCCCCGGCCCCGCCTCTCTTCCGCGCTCCCCTCACTCCTCACTTCCAGGGTCCAGGCAGGCTCCCGCCCTTCTTTCCCCGCCCTCTACTCCGCCTTCCTCCCTCGTCCACCCCTCCCGCCGCCCTGCTCCGGGGCGGTCCTAGCCGCTCACCGCAAGAAGAAAAGCCACCCGTTCGTCCCCCTCCCCTACCTCCCATCCTTTGCCCAGGAGCTGCCTTGGCAGTCACGCCCCTTCCTTC
FP018776	chr22	35065935	35066186	+	1	train	GGCCAGTTCCCCAGCACAGCCCCTCCCTGGGGCCTCCAATACCTGTCCTTCAGATCCCCACTGCCTCCTGCCACTTTTCTACCCTGACACCTGCATCCCCAGCCTTAAGGGGCGTGGAGAGTTTTTCTCTTATCAGCCTGTTTTACTGCCCCCTCCTCCTTGGTTACACAGTAATTTGGAGCAAAATTCCAGGCAAGGACAGATCAACAACTGTCAGCTCCCAGTCAGAGAGAAAGGGCCTCTTCAGTCTG
FP015970	chr17	74736425	74736676	+	1	train	AGGAGCTGCCGGGTGCCTGGGATATCCTGGCAGCTCTGCTCAAAATGATCTACGACTTCATGAATTTATTTGGCTCCTCCTCGGGGCCAGGGTGAGTGTCATGGGTTAATAAGGCCGGCCCCGCCTTCAGGAGCGGTCCACTGGGAGATGTGTGCTGCGCAGCCCTCTTGCGAAAGCTCTCCCCTGGTGGGACATTCTGGGCACAACCAACAGGCCGGGGGAAATGAGAGGTGATCCATACTAAAGGGTCA
FP000448	chr1	32179864	32180115	+	1	train	CCGCGTGGGCCGGGGGCTGTCTGGGGATATGCGCATGCGCGGGCGTGCCTCGCGGCTTGAGGGCGCGCGGGGCGTGGGTGGCTGCGCGCGCGGGGGGCGCACGTGGGGCCTGAGGGGCGGGGGCGGTGCCGGGAGTCCCGCCACGTCAGTCTCCGGCCCTGAGCCAATCCCGCGCCCGGCCTGCCGCGAGGGGGCCGGTTGTGCCGGGAAGTGGCTCCAGGGAGAAGAGGCCTCTTCCCTCACCCGCTGTG
FP001916	chr1	234214018	234214269	+	1	train	GCTTAAGTTCTCAGGGTCTCCCCTCCGCAGGCGCTGGTCCTTTTAAAGGGCTTCTCAGAGAGGTGGTGGCCAGAGGCTGCAGTCAGGACCTGGCTGGGAGGAAGACAGGGATGAGGGCTGCGGGGCTGGGCGTCCCGGGGAGGAGGGCGGAGCAGGTGAGCTGCGGGGTGGGAGCAGGGAGTGCACTTGTACGTGACGTTGGAGTTTGCAGCAACCTCCAAGTAGGAGGCTGTGCGCGCGTGTGTGTGGAG
FP007310	chr7	32891629	32891880	-	1	train	GGCGGGCCCGGCGTCTCGCGGCCCCGGACTGACAAGGCGGCGCGGGCGGGCGGGCGGCGCGGCTCCCGGGGGAGGAGTCTGGCGGTGGTCGGGCCCTGAGGAGCGTGCGGCGGCGCCCGCAGAGGACGCTGCTTTCCGCGCTGGACGGACCAAGCGAGGGAGGGTGCGAGGGGAAACCGGAAGGAAGAGGCGGCGGCGCCAGCCTTCCTCGGCCGGAGGCGGAGGCGAGACCCCAGGCGAGGCCGCGGCGG
FP008901	chr9	35749109	35749360	+	1	train	GTCCCCCAGGATTGGGCGCCAGGTCCCGCCGGCCGGCTCCGGGGCAGCGCCCGCGCCCAGGTGCCAGCCCGTGGGAAGGTGACCCTGGGCGCCGGGATGACCCGAGCCCTTTCCGGTCTGGCCTGCCGGGCGCTTCCGGCCGGAAGGGACTCCGCGGAGGGTGGGGGACCGAAGGGAAGTCCCGCCTCTACCGCCCAGCGGACGCCGCCGCCGCCGCCGCCGCCGCGTACCTAGCCAGGTCCCTGAGGGGC
FP006858	chr6	116461177	116461428	+	1	train	ACAGTGTTATGACATATCTGAAAGAAAAGAACATTGTATTGACGAGTAAATCAATACAGTATATTTGGTCTAATGTCCTGAAGGAGATCAGGCAGAGGAGAGAAGAGTCAGTGTGAACTAGGCTGGTGGCTCTGTCCGGTAGGGAGATGGAAAGGGTGCCGCCCATCATAGAAGTACCAGAACTTGAGCTGGACTTTGCTGATTTAGCTTATGGAAGAGGAACCAGAAATTTGTCCTTGAATAATGTTTCC
FP001460	chr1	161706984	161707235	+	1	train	TCAGACAATGACATGCTTTGACAGGCCCCTGAGCTCTGTGAGTAAATTCTTGAACAGCTGAGAGTGCTACCATTTTGTCATAGCCCTAAGGGTGCAGATGATGCTGAAAGCGTGTTGGCCAAATCTGAATGATGAAAGCCAATTACAAACTAGAAAATGAAAACAGACCCCAGATGCAAGGAGATGAGACAGTTAAATTTACTTCCTCTTTTCTAATCTGAGAGGTTTCATGTTGAAGAAAATCAGTGTTG
FP004077	chr3	129439886	129440137	-	1	train	CAGCAAAAGCAATGGTGGCACTGGAGCCCCTAAGGCGCCAGCGCAACCAGAGCGCGAGCGATTGGTCGGGGCGTGTGGGGGCGGTGCGTCTTCCTCGAGAATGGATTTGATTGGTGGAGCGTGGAAATGGCGGCTGTAGCCGAGGGGGCGGCCGGAAAGCAGCGGCGGCGTCTGGGGCGCTTTCGCAACATTCAGACCTCGGTTGCAGCCCGGTGCCGTGAGCTGAAGAGGTTTCACATCTTACTCCGCCC
FP016705	chr19	7920140	7920391	+	1	train	ACTTCATGAATGAGATAAAGTGCAGGCTGAGGGTCTCCTTCGCCCCGCCCCAGTTCCGGTGCCGATCTCCCCTTCACTTCCATTGGGGCGGTTCCCCGGGCCTCCAAACGCGAAGCCACGCCCCCACGCGCGTCCGCCTCCGCGCGGTTCTTTCTGACCTCGGCGGCCCCCGTAGCTCCGCCCATCGGAGAAGCGACCTTACAGCGCCTGCCTCTTTCTGAGCGGCATGAAGCCACCTCCCAGGCGGCGAG
FP015014	chr16	88736138	88736389	-	1	train	ACAGCTGGCTGACCTTCGTACTGCTGCTCTGGGCCTGCCTCATCTGGACGGTGCGCAGCCGCCACCAACTGGCCATGCTGTGCTCGCCCTGCATCCTGCTGTATGGGATGACGCTGTGCTGCCTACGCTACGTGTGGGCCATGGACCTGCGCCCTGAGCTGCCCACCACCCTGGGCCCCGTCAGCCTGCGCCAGCTGGGGCTGGAGCACACCCGCTACCCCTGTCTGGACCTTGGTGCCATGGTGAGTGTG
FP011307	chr11	107457768	107458019	-	1	train	CACCTGAAAGAATTAAACCCTCCAACACCCTTCGGGTCTTTAAAATCTGGGCTGCGGTAATCCGAGCCCTAAGCGACCTCAGATGTGCGCATGCCCTGTTGGTGGAGCCGTGCTCCTCCTTCCTCGGGTCCGGAAGTCCCGGTTTGTTTGTTAAATTACGACCGTCGTAAACTAGAATTTTCTTGCGGCTTCTTAGCTTTACGATGGCAACAAGTATGGCGGCTGCTAGTGGTAGATTTGAAAGTGCGAAG
FP014278	chr16	1827119	1827370	-	1	train	CTTCAGGAACAGCACGGGCTCGCTCAACACCGCGCTGCGCATCTCCCTGACGTGGTCCGCGTAGTTCCTCCCCACGCAGACGATGTTCTTTCCCCACTCCCAGAAGCGGGACAATGGCCTGGATGCTGCCATGATTCCCATCAAGTGCCCCTGTAGTCACGTGGGCTGGGCCGGTCAGCTGATGCCTACGGCATCCCGGAGAGGACCAACTGCCTCGGAACGCTGTCCCCCGCAGCGACGGCCCGTTCCAC
FP018392	chr21	32771681	32771932	-	1	train	CTCTCTCTCTCTCTCTCTCTCTCTCTCTCTCTCTCTCTCTCTCTCTCTCTCTCTCTCTCTCTCTCTCTCTCTCACTCTCTCTCTTTCTCCAAGTATTGAGAGCGCGTGGGAGCATAGGCGCATGCGCGCTCGTGGGGTGCGCGGTAGCAACAGAGGACTCGACCCGGCTGGAGCTCCGGAGAGCGCGCGTGCGCCGTCACGAGCTCGGCGCTGCCGGGGCCGCGGTGTGGAAGCGAGTATTCGACCGCCGT
FP001994	chr2	1744464	1744715	-	1	train	CGCGCCCCCGAGCGCTCGCCTTGGCCATGCGGGCCGCCCCACCGGGATGAGGGCGCTCAGGCCGGACGCTGGGGCCCCGGGTTCTCGCCCCGCCCCGCCCTCGGGGATTCAGAGGGGCCGGGAGGAGCCTCGCGCATGTGCACAGCTGGCGCCCCCCGCCCCCCGCGCACAGCTGGGACGTGGGCCGCGGCCGGGCGGGCGCAGTCGGGAGCCGGCCGTGGTGGCTCCGTGCGTCCGAGCGTCCGTCCGCG
FP004954	chr4	109433625	109433876	+	1	train	TAAGTCCCGGGGCGTGGGGCGCGTGGGTCAAGGCGGAGCGACCCAGGATCCCCGCGCCCGCGGCGGCCGCCGAGTGGCAGCGCTGGACTCGCGGAGAAGCTTGGGTACCTGAGCCGCGGTCCCGGGTGACACCCTCAGTGACGCCAGGCGCGTTCCTTCCTCTTCCTCTCTCCTCTCCGGCCCCGCCTTCCCTTCCCTCCGCCCACCTCCCTGAAGCGGAGCCGCCGTCGCCACCAGCGCCGTCATGTCGG
FP001786	chr1	212858082	212858333	+	1	train	GAAAATGTCAGCCAGCGCTCGCCGGCCGCTTTCCTCCCCCGCCTCCCGAAGTTCTCGCCCGGTAGCTCCCCGACTGGCTCTCGCGCCGGACAAAGGCGCACGCTGATTGGCCGGAGGGCCGTAGTCATGCAGGACGCGCGACTCTAGGGGCGGGACCAGACAAGGGGTGACTGCCGCGCGGCGCGGGGGAGGAGACCTTCATCTGTTCACGCGGTAGCGCGGATTGCGGTTCGCGGCGCGCGCCACCGGGG
FP006027	chr5	176361713	176361964	-	1	train	TTTCCTCTCGGGGCTGGCGGATTCCACACTCGGAGCTGCGGCTCCACAACTCGCTGGGCGGCGAAGGGCATGGGGAGGGGGCGGGTCTCGCGATCGTGGACAACAACTCCCAGCATGCCCTGTGCTCCGCTGGGCCAAGTCTCGCGCGAGATCCCGCGGTCTCCGGAGGCTTTATCTGCAGTGCTGCCTGCCCGCTGGGTGGTACTGCTACCTAGTGGGTCTTGGGGACCTTCGAAATCGCCGCCGCTCTC
FP017258	chr19	39934548	39934799	-	1	train	CACACACACTGGTCCTTGGATCCATACCCCCTGGCCCAATCTGCCCGACTTCTTTCCCCACCTCAGCCTTCATGGTCCCAGGTGGCCCTTGTCATGAGAGTGGGCCCTTCTGCCCAGGACAGTCTCCAGCAGAGGCGGTGGATCAGCCTTCTATGATCCTTTCTCTTTTCCTACTGCAGCCATGGGTGCCCTATGGAGCTGGTGGATACTCTGGGCTGGAGCAACCCTCCTGTGGGGTAAGTCAGACCAAG
FP012751	chr13	44373629	44373880	+	1	train	CGAACCTTCGGCCCCGCAGCATTCGGCTACAGGACCGCGACACTGCAGGGCCCCGGCTCTGCGCGCTGCGCTCCGGCCGCTCGGCGCGGGAGGACGTGCGGTCGCGCCTGCCTCGGTGGCCGCGCGGGGTAGGTGGCTGCGGGGACCGGAAGGATGGCGAGGAAGTCGGAAAGGGTTTCCTGAGATGAGAGATTACTTCCGTCCGGGCTGCGGCCTCTCTCTGGAGTCGGCTAGCCGGGGCTCGGGGAGCG
FP011011	chr11	66257567	66257818	+	1	train	GCTGGCGGGCCCTGGGAACTGCGCCCCGGGCGGGTCCTCGCACCGCCCCCGCCTCGATGGCCCCGCCCCGTCCCCCTCCCGCTGGAGCCGGCCCGGCCCGCGCTCCTTTAAGGCAGCGAACGGGCCAAGAGAAGCGTGTTTCGCCCCCTCCGACGCCACCGAGGTAGCGGCTTCACCTTTAAGGCGGCGCGGGGGCTGCTGGGAAGGCCGGCGGGATGGAGGCGGCGGGACCGGCTCGCGGGTGCGGGTCC
FP004197	chr3	150703766	150704017	-	1	train	TCGGTGGCGCAGGGCAAGCAGTTGCCCGGGGAGACCAACGCACAACAGAGGCTCGCGCTCGCGAGAGCCGACCGGGGCCGCGGCCAAAGAGCTGGCTGTAATCCCGCCTCCCGCCAGCCATGGCCCACTTGCGCTCGCCTAGCGGCTTCGGAGACCCGGGGAAGAAGGACCAGAAGGAGTCAGAGGAGGAGTTAGAGGAGGAGGAGGAGGAGGAAGAGGTGGAGGAGGAGGAGGAAGAGGTGGAGGAGGAG
FP017355	chr19	43670146	43670397	-	1	train	GCACCCAGGCAATCTGGGGACAGAGCTGTGATCACAACTCCATGAGTCAGGGCCGAGCCAGCCCCTTCACCACCAGCCGGCCGCGCCCCGGGAAGGGAAGTTTGTGGCGGAGGAGGTTCGTACGGGAGGAGGGGGAGGCGCCCACGCATCTGGGGCTGACTCGCTCTTTCGCAAAACGTCTGGGAGGAGTCCCTGGGGCCACAAAACTGCCTCCTTCCTGAGGCCAGAAGGAGAGAAGACGTGCAGGGACC
FP000153	chr1	11099800	11100051	-	1	train	CTTCCCAACCTAATCGCAGGCTTCCCAACCTAATCGCAGGTGCCCCACCCTAAACGCAGGCTTCCCAACCTAATCGCAGGCGCCCCACCCCAATCGCAGGCTCCCCACACTATAATCAGACGCCCCGCCCGCCGCTTCGACCTGGCGCATGCGTGGTGCGCACGCGTCCCGTCTCCTCGGCCGACAAGCTCTCGCGAGACGAGCCGTGCAGGCTGAAAAAATGGCGCCACCCAGTACCCGGGAGCCCAGGG
FP005877	chr5	146878685	146878936	-	1	train	CTGCAAGCTAGTCGCTTGCTGCCCAGGTGCCGATTCGCGGCTGCATGCGCCCCACTGCAGCAAAGAGCAGCCGCAGCCTCTGCCTCGGCCACCACTGCTGCTGGGAAAGAGTCGTGGGGCTGCTGACGCGGTTGGGAGGAGCCTCGCCTTTAATGCACCAGCCGCCTCCAGCCTCCTGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCTGCGAGTGCGCGCGTGTGGGTGTGAGGGTGAGTGCGCTGGCG
FP016781	chr19	10871380	10871631	+	1	train	GGCGGGGCGTAGCCGGTCCCGCCCCACGGGGTGCCCTCTCGTTCGCTCGCTAGCTCCCCGCGCCCAGGACGGCCGGCAGAGGGCGCTGCGCCACGGGGCCGGGCCGCGGCGACGGTGGCGGCGGCGGCGGCGAGGCGGCGCGTGGCCGGCAGGCGGCGCTGCCCGGCTCGGCCTCGGCCTGCACGGCGGCTGCGGCGGCGGTAGCGGCAGCGGCGGCGGCGGCGGCGGCGGCGGCGGCGGCGGCGGCGGCG
FP001506	chr1	169427411	169427662	-	1	train	CTACAAGAGGGAAATTTCACAACCAACACAGCGGCACTTCGCGAGTCTCTATAACGTTAGTGATGCGTGGATTCCTTAGTCGTCCCCTTAGAAACCTCAGCTTCACTGGCCCTTCGCAAGCACCGGGATGGGTGCCGGGAGGGGTCGGTGGTCTGAGGATATTGCATGCGCCTGCGCAGATTTTTGTCGCTCTCGTAGTTAGAGAGGCCCGGATGGAGGACGCAGAGGCACGCTGTTGCCATGGCAGTGTG
FP016269	chr18	23452622	23452873	+	1	train	GAGGACTAAAAAGTTAATGAAAGATTTGAACAGTGCATTGATATACTAAGTGAAATGCAATGTTTAGTTATCTTTTATCTGGTCATTTTATTAGAATCTCTACAAAATACAGAAACAAAAGTTTTATTATGCAGTCTTATTTTTGTTCACTGTTACGTAACACATTTTACACATTTTCCGCATTCCACGCCCCGCCCTTCAACAACAACCACCACAAACTCCGACCGTCCTTTCCAGGCCCTTTCCCTCTA
FP012259	chr12	89524708	89524959	-	1	train	GGCCCCCTCTGCCATCCCGGCCCCCAGCCCCCGGCCCGGCCAAGAGCCTCCTATCCAGCGGTCTGGGCCTGGCGGACGACTTTCGCAAACTAACCACCAGGAGGAGTGAGTGGGAAGGGGAGGGGTTTCTCCTCGGCCCGGGGAGGGTGGGGTGTTGGCTGGAGAGTTTGTGAAAAGTTGTGAGCCGAGCAGGAGGAAGTAGGGAGAGAGGAGTTGGGCTGTGCCGGAGGCCGAGGACCGAGAGGGCTCAG
FP007547	chr7	90403260	90403511	+	1	train	TAGACGGAACATAAGGTGCAAGAGAAATCCGGGGAGGTGGAGGCAGAGAAAGGGAAGGGCAGGAGGTGGTCGCAGGATGTTGCCTGCGGCTGGCGGCCCAGTGGATTCTGGGAATTGTAGTCCCAGCCATCCAGGGCATTGCCGTTCAGGGCCACGGGAAAACCTGACTGCGCTCCCAGAAGCCTCCGGTGTACCTCGCTGGGAACGCACTTCCTGGGACGCTGAGAGGGAGACGCTCCAAGAGGCTCCTC
FP006363	chr6	31736258	31736509	-	1	train	GACCGAGCTGGAGGAGCTGGGTGTGGGGTGCGTTGGGCTGGTGGGGAGGCCTAGTTTGGGTGCAAGTAGGTCTGATTGAGCTTGTGTTGTGCTGAAGGGACAGCCCTGGGTCTAGGGGAGAGAGTCCCTGAGTGTGAGACCCGCCTTCCCCGGTCCCAGCCCCTCCCAGTTCCCCCAGGGACGGCCACTTCCTGGTCCCCGACGCAACCATGGCTGAAGAACAACCGCAGGTCGAATTGTTCGTGAAGGTA
FP009400	chr9	129625982	129626233	+	1	train	GCCTGCTTGACAGAGCAAGATTCTATCTCAAAAAAAAAAAAAAAAAAATTTAAATTATTTTTCAAAAGAAGTTAAGTTAAATCCGGAGCCCGCGCGGGGGAGGGGCGAGCTGAGCGAGGGGGTGGAGCCAGCCATGGGCCCGCCCCTAAACGCCCGGAAGTGACGTTGGCGGACAAAGGCAGCGCGCGCCGCGAGCTGTCGCGTCTGGTCGTGGTCTGGCGGAGCTGCGGTTGGCTTGTGGCGTCTCCGCC
FP018317	chr20	63707207	63707458	+	1	train	GCCTGGGGGCCATTCCCGACTCCTCGTCCCTCTCCCACCCCGTCCCTCTGTAACTTCTCCCAGGTCAGCCGCCACTGTGTCCTGCTCACAGCAATGACTGCGACCTCTCCGCATACACATCGGTTCCGGCCCCTCCCCTGCTCGCGGGACTACCCAGCCGGGTGTTCACAGTGAGCTCAGCCGCGCTCCCGCCCTCCCCCGAGGCTTCGCTCCCACGCTTCACGCGCGCGGAACGGGGAACACACTCGCTG
FP010749	chr11	46591504	46591755	-	1	train	AGCCTCCTGAGTAGCTGGGATTACAGGCGTCGCCACCACGCCCAGCTAATTTATTGTATTTTTAGTAGAGATGGTGTTTTACCATCTCGGCCAGGCTGGTCTTAAACCCCTGACCTCGTGATCCACCTGCCTTGGCCTCCCAAAGTGCTGGGATTACAGGTATGAGCCACCGCGCCCGGCTAAGAGAGTGGTTTTAAGGGTGCAGAGATTCAGTTCCTAGTCCTTGAGTGGAAACTGTACAAGGCCACATT
FP008554	chr8	109334148	109334399	+	1	train	TAGTGTCCGTGAAAAAGTTGAGACTGTTGCATGCTGGGCTATGTAGTTCTCCTGGTACTTAACGCGCGCGCGACGGCAAGGTCTGAACTCTGTTACCCAGAGGCCCTAGCAACCAGGAGGCGGCCCGCGCTGGTTAAATTCTCACATTATAGGCAGGGTGGCGAGACCCCGCCCCGGAAATGCGTGTTCTAGCTTTCTGTGTGCTTAGGTGCCCGAGCTACTGAGGGTCTAAGTCCGGGCAGCCGAAGAGT
FP010028	chr10	92240543	92240794	-	1	train	TGAAGGACGCAGCAGTATTTATTGACGTATGCAAATAAGTGGGGGAGGGTGGTGCTTGGGAAAAAACAGGAACCAGAATCAGTTTCTCATTGCAGGGCGAACGCTCTGAGTTCCGGTGAAGGGAAAAGAGACGTGGCGAAATTGCTGTCTTTTCTGTCTTATAAGCACATCTCTCACTAATACAGAAAAGGAACCAAGACATCTGAAAGGTTGAAGACGCTCTCACAGTATTTTTAGAGACCTTGCTCGAC
FP012846	chr13	77918777	77919028	-	1	train	GACACAAGCCCTTGGCCTCTAGGTGCCTTAATTCCGCGGTTCCCACGCACGCTTAACTAAGACGTGTCTGTATTCCTCCCGTTACGTGAAAGAGTTCGGAGCTTTGCCTGGGACCCCCATCATTCCCTCCCTGGCACACCCCTTCCAGAACGCCCCGCCCCACTGCATATTATTTACCCCTCCTGGCCACGCGGGGGAAGAAAAACAGCTGAGAGGGCATCAGGAAGGAGTTTCGACCCGCGCTGGCGAGT
FP010961	chr11	65181599	65181850	+	1	train	CCTGGTTAAAAATACCCCCTGCCCGGCCCTCCCCCTCCGTTCCCCGCGGTGCCCTTCCGCCGCGCCCCAGCCCCCCCATCCCCCCCCGCGCTGGCCCCTCCCCGGGGACTGCTCTGCCTTTTACTAATTTGTCCTCTCGCGGAATTGGCCCCCGGCCTGCGGAGGGAGGGGCGGGGAGACACCGTGAGTAAGAGATAGTCAGAGACCGTGCCGGAGAGATAAGGGAGAGCCGAGAGGAAGCGGGGTAGGAG
FP006283	chr6	30160840	30161091	-	1	train	TGAGCCCCTCTTCCTTCCTCTCCTCTGCTATTTCCCATCTCTGCTGTTGGCAGGAGAATAGAACCCTGGCTGCCAGAGATGCAAGTGTGTGACGATATGGGTGCTGGTGCATATTTAGTATGTGCCTGTGTCCAGCCATGTGCATGTGTGGGTGTGTGAGTGTGTGACCCAGCCCTTCCCCCGTGGCCAAGCAGAGAGAGTGGCCTTGAGGAAGCCATAGCAGCAGGACCAGCATGGCCTCTGCTGCCTCT
FP009932	chr10	73697812	73698063	-	1	train	CTTCAGGCAGTTGTTGTGTCTCTCAGCGCTTGTTGGGTTCACAACCTATTAAATAAGCCAGCTGGTCTTCACCCTCCCAGACAAGTCAACTCAGGGGAGGCAGCAGGGTGTGGGCCTTGACCCGCAGCCCTAGCCGGGGCCGGGGCCGGGGCCAGGGATGGTGCCCGGGGCCGCGCTGTGAGGTGGGCAGGCGAGGAGCGGGAAGACCATCTCTGCAAGTGCAGCATAGCCTCGGCCTAGGACAGTGGGAG
FP008979	chr9	75028364	75028615	-	1	train	CGCACAGGTAACGGCAGTGCCAGTCCCACCCCACCTACGAGCTCCTCCTCCGCCTCCCACCAAGGGGCCTTGGCCCCGCCCTTGGAAGCGTGGTTGTGACGTCAGCGCGCTGTCTCCGAAGCCCGGCAGCAAATGAGCGCAGACCGCACGGGCCCTCCGCGTACCAAGGCACACGCATGCGCATTGGTTCTGGGAGCCCTGAAGACCTGGCGGAGCCCAGCGCGGATGGAGGCCTGAGGGTGGCGGGGGCG
FP011618	chr12	6928162	6928413	+	1	train	GGGGTTTGCTCCGGGGGCCGGCGGGCGATTGGGGCCAGGCGGGGAAAAGGGGGGATGGGGGCCGCCCTCCGGGGGGGTCGGGGCCGCCGCCGCCGTCGTCGCGGCGGCGACTGAGGCCGAGAAGAGGAGAGGGGGGCGGGGGAGCTGCCGCCGCCGCCCCCCAGAGGCGCCGGAGCCCGGAATCCCGCTCGGAGCCAGCCAGCCGTCCCGAGCTACCAGCAGGTAAGGTCTGCGGCCGCCTGGGCCCCGGG
FP010715	chr11	43680537	43680788	+	1	train	TGGGGTGAGACAGGGCGCTCACTGGCGGAACCCCGGGGGACGCGGTGATGGGAGGAGTGTGGGAGGGGCAGTGGAACTGGAGTCAGCTAATGGCTGACGCACTACGCGCAGAGGGAAAGACGGGTCACCAATAGCGACACGGATATGGCCCCGCGGGCGGGGTTTAGGCCCAAAGTGGTGTCGGAGCAGCGCCTATTAGTGTCATCCTCACCGTCACGGCCGGCGCCTCCTCCTGGATTCATTCACTCGCT
FP013896	chr15	64821251	64821502	-	1	train	CTCCTCTAGCCCAGTGTGTGGCCCTGGCCCAAAGGCCAGGCGTGCGGCAGGGCTGGCTGAACTGCCAGCGGTTGGTCATTGACGAGATCTCAATGGTGGAGGCAGACCTGTTTGACAAACTGGAGGCCGTGGCCAGGTCAGTATGTTCAGAGGGCAGCAGGTGAGAACTGGGACCTGGTGGTCTTCTGTGTGCCCCTAAAGCACCCCACGCTCCTCTCCAGAGCTGTCCGGCAGCAGAACAAGCCATTCGG
FP016438	chr18	63949104	63949355	+	1	train	GAAAAACCCCAAATAGCCAAAGCGCATTATCGCTTCTTGACTGAAAACTCCCGGACAGTTGGGGCTTTTTCCTGCACAGAGATTGGCCACTTCCTGCACATCAGGAGATCCCCGCGGCCTGCGGGACCCGCCCCGCCCTCCTGCCCAGGCGCCCGCTAGGTGGTGCCTGGCTGGGCCCGACTCCGCCCGCCTCCCCATTCACTGGGAACTAACACCCGGCGCCGCTCAGACATCTCTATTCCCGCCTCTCC
FP016031	chr17	76376381	76376632	+	1	train	GTTTGTGACCAGGGTGACAAAGGACTTCACTGTCCTGTAGCACAGATGAGGTTCCACCCTCAGGTGGGTGGAAACAGCCCAGGCCATTAGTGCAAATCCCCTGCCTGGCCCAGGGCCACGTTCAGCCCACTGTCTGTTTGCTTTGGGACCCACCCAAGGTCACATTTTAGCTTTTGAGAAATGTGAAAGTGGAAAACATTGCAAAACAAAAAGCCCCTGGAAAACTGTGCACACACCCATCCCCTCAGCCT
FP007407	chr7	55538613	55538864	-	1	train	ATTATAGACAGTGTTCTGCATCTTGTTTCTCCTTTTTTTTTTTTTTTTTAAGAAATGTTGCTTGGAGATGTTTCCTCATCAGCAGCTCAAACCCACAGCATTCCCCTTAGAAAGCGTTCCTTTATAGGAATGGCCATGACTCATTTTAATCCGCCCTCTCTTGATGAACAGCTAAGTAGTTTGTAGCTGTTTTTGTCATTACACACAATGCTCAGCGAAACTCTTGGGTATCTGTCTTCTGTGCTCTTGCA
FP016291	chr18	31042702	31042953	-	1	train	CGCGAACGCCCGGGTGGCTGCTGGGGAGCAGAGGGGCAGGCTCGGGCAGCTCCTCAGCCGCGCCGGCGATGATAAAACTTGAGAAATAACAAGGAGTGGCCTCTCCGCGTCCTCTGGGTCCAATCCGGTTTCAGAAGTGAAGTTGCTCCCGGGCAGGCAAGCCTGAGAGAGGCCAAATACACTTAAAAACAAGCAGCGGCGAGAAAAGCACCTCCCGCGCCCTGCACCTCGGCAGGTCTCGCTCTCGGCAC
FP012009	chr12	54300964	54301215	-	1	train	CCTGGGCTGAAAAGGAGGGCTGGGAGAGGCTTGGAGCCATGTGTGCCCCCGCTGGCCAGGAGCAAGGTGGGGGCGGGGTGTGGGCAGAAACCACATCGGGCTCAAGTTTCCTCTCTTGGCCGAAAGGGGCAGGAGGAGAGGAGAGCCACGGGGAAAGGGGAGGGTTGGAGGAAGGACTGAGAACTCAGGCCGAGGGCTTCAGACGCTGGTGCGCCTGCTTGGGGCTCCTGTGCTCAGCTCAGCCTGAGCTT
FP000623	chr1	43932986	43933237	+	1	train	CATGGTCCCCAGTGTTAGGATCTTTCCATCCATGTTACTCACCCAGCCCTCAACCCACCTGCCAAATTGCCATAATTAGGCATCTGAAATGCTGCCCTCCAGCTACCTACTTCAAGACTCAGCTCAGATGTTCCTCCTCTGTGACCCCTGCCTTCAGCCCTCAGTACCCACTTCTGCCAACACTGCCCCTGCTGTGCCCCAGTGACTCCCGTGTTTGCAAGTGTGTCCGTACTCCAAGCCCTAAGCAGCAC
FP009955	chr10	75782425	75782676	+	1	train	ATTATTTTTTCTTCTTCCTCTTTAGCAGATGGGATGGAATATTTCTGTTTTGTTTTTTATGGGTGATTTTTCACCGGCTTCTTCCACTTTATATTGAGTCACTTAACATCCTGTGAAGCGGAAAGGAGGCCGGTACTTTCAGCCTGAGGTCGGCCAGGGTGCTGCAATTCCGGACAGAGAGATGTTTTTCCCTTGGGTTGGGAATGGCATGTTCTGAGCTTCTTTTGACTTCTAGTCCTCGCTGCCGGTGC
FP005687	chr5	132830611	132830862	-	1	train	CGCCCTCCCACCACCTCGCTCAGGTTTCTCCGGCCTGGGGCAAGAGGCCTGTGGCCCGCGCGGGAACGCACTGTCCACCCTCAGACCTGGGACTGGGCCGGCGCGCGGACGCTACCAAGAGGCTGCGGCTCCCGCCCCCGCGGCCGGACGTGGCGCCTCCCCTGAGGCCGCGGCCGGAGCCTGGAGGTGGGGTCGGAGTCAGAGCCCGGGGCTCTGATGTCACCGCGCGGCTGCGACGGCCCAGGAGCGCG
FP000683	chr1	46707904	46708155	-	1	train	CTTTTATTAATCCATCTTCTTTCCTACTGTTTGATGATTTTTTCATCTTAGGACCATGAGGGCACTTTATGCTAGTTATGATTGTTCTTATTTTGTCTTTTTAGTGGAATCTAATCAGAAAAGCTCATTCCAAGAAATCCCCAAACTTAATGAAGAACTACTCAGCAAGCAAAAACAACTTGAGAAGATTGAATCTGGAGAGATGGGTTTGAACAAAGTCTGGATAAACATCACAGAAATGAATAAGCAGG
FP002438	chr2	74958442	74958693	+	1	train	TCGGGGCCAGAGGGTCGCCGGTAGCGTCAACCAAGTGCACGACAGCCCACGGGTGGCTAGGCGCGTGGGAAGAGCCGGAGCCCGACCCCCGCCCACCCGGTTTCCTGGCAGCGCACTGCCGCGTAGGGGCGGAGCCGCGCGCCTACACAGGGTGTGGCCTCCAGCCCGCGCGGGAAAGGGCCTGCGCCGCATGCGCGCGCACAGTCGGCGGCCGCGCGCAGCACGCTCAAGGCCGGGATGGCGGCGGCGGC
FP009496	chr9	136410457	136410708	+	1	train	GCTCCGCCCGTGCCGCGGCCCGCGTCGCCCGCCCCCGTCGCCCGCCCCCGTCGCCCGCCGCTCGGCCGCCCCCGCGCCTCCGAGCCTCTCGCCGCTGCTTCCGCTCCGAGCACCGAAAGCGCGTGCCTGAACGCCTTGGGCCGTCGGCGAGGGGGAGGGGAAGCCGTGGGCGGAAGCGGAAGTGACGACTGAAGCGGGGCGGAGACGCAAGATGGCGGCTGTGGTGCTGGCGGCGACGCGGTTGCTGCGGG
FP011904	chr12	49741638	49741889	+	1	train	CTGGGAGTGAGAAATGGAAAGGGAGTTAGCAGTTACTGTCTGAGCAAGAGTGGCAGTAGCTTCCTTCTGCGGACAGATGGCACCCTGAAGGAACGAGGCCCTGCTCGGAGACCAGACTCGGGAAAACAGGGTGTGGAGGGAAGCTAGGGAACTAGTGCTGCGCGTGCGCAACAGTGCTGAAGCGGGGGTGGGGCGGAGGCGAGTCTGCGGGGGTTTTGGGGGGTGTCGAGGCCTCTATTCTGCCCCAGAGC
FP006972	chr6	138988231	138988482	-	1	train	TGAGCTGCCGGCTGTTTCCCACTGCACCCCCCTTTCCGGGGGTCTTGCTGCTGAGATCTCCGCGCAGAGGGCGGCTCGGTCATCCCGCCGCCGGGCGCCGGCCACCAGGGGGCGCTGCGGCCTCCTCCCTCTCCCGGGAGCGGCCAAGCCAGCCCGGTGGGTGTTTCTCTCTCCCTCTCGCGCTCTCTCCTCCCGGCCTCACATCCGGGTCTGGAGCGCGGCCGCCGCTGTCACTGTTGGTGCGGGCGCGT
FP017801	chr19	57935332	57935583	-	1	train	TGAACCCCCTGAGGTAGGGCTGGGACTTCCCTGCAGAAGCGCCTTTGGGAAGCGGCAGCTTTACTACCTTGGAACCTGGGGAACTACATTACCCAGAAAGCTCTACATTGAATGACAGCGCAACAGAACCAGTAAGGGCTCAGAATAAAGGCGTTTCTGTGTGCGCACGTAACGTAGTGGCGGGATTCTCAGCGTTCTCTGGTAGCGACCATTTTGGTTAATGTTGGGTGTGTTTCTGCGGTTTGTGAGGT
FP009410	chr9	129994906	129995157	-	1	train	TTCAGTACTCCAGGCGGTTTTCAAACCAAATTTGGTAATGTTGATTTAAAATGGCTCAACAAAAAAGCACCCCAAGTATAATCATGGAAAATATTAGTACTACTTTGTATGTAGTAATATGTTGTTAAGAGAATTATTCATTGAAATTATGTTAATCACAGGGAAAGACCATAACTTCTCTCTTGGTTTGTTTCAACAGGATCAGTTTGACAACTTAGAAAAACACACACAGTGGGGAATTGATATTCTTG
FP019153	chrX	23743211	23743462	-	1	train	TGCTGCGTGGTTCGTTTACTCAACCGAGAATAGGCCGGTCCTCGCTGTAGCAACACGACCCGCACAACTGATTGAGTGGGAAGCTCCGCAGAGGCGGACTCCAGCCTCCCGCCAATGAACGGGTAGGGTCCGCCCGGCACAGCGCGGCCACGCCCCCTGGACGCCGCGGTCCCGCCCCCGGACACCGCTGTCCGGCTCCCGGGCTGTCCTCAGCAAGGGCGCGGTCTGGTACTCGTGCGTCTTTTATCGCC
FP005772	chr5	140459068	140459319	+	1	train	CAGAGAAGACAGAGATGATTATCACAATTCAGAAAATTAATATCTTTATTATAAGTCTCTTGCAACTAATTTTATCTGTTTTTATACTTTCCTAGGAGCAAATATAAATGCCCAGACAGAAGAAACTCAAGAAACTGCTCTTACTTTGGCTTGCTGTGGAGGATTTTCTGAAGTTGCAGACTTTCTTATTAAGGCAGGGGCTGATATAGAACTTGGCTGCTCCACACCTCTGATGGAGGCATCTCAGGAGG
FP016562	chr19	2702643	2702894	-	1	train	CCCCCGGCCCCGCCCGCGCCCGGAGCCCCGCCCCACTCTCCTGGGCGCAGCCTACCCCGCTGCCTGCCCCGCGCCCCGCCCCCGCTCCGGCCTCCCGCAGCCGGGGCTCGCGATTGGCCTTGCCGGGACGCCAGGGCCCGCCCCCCGAGTGTCGGCCCCGCCCCCGCAGAGGGGAGCGAGCGCTCGCGGGCCGGGGCGGAGGGTTCGGCTGCGCGGCGGGGCTGAGGCGGCCGCGGGGCCCGAGCGCAGGT
FP006322	chr6	31158372	31158623	+	1	train	GGCTCCTCCTCACTGCGGCAACCCGGGAAAACTTGTGAACTAATCAGAAAAAGTGGAAGGCGGGAGATCTTGGGGCGCTGTCCAATGGCGCGGAAGAGAACACATGAGCTGGCCAATCGGGAACGGCACGGGGGCGGGCTCGCTCGGCGCGAAGTTCGGGCCCGGGAATTCCGAAGGAGGGGTAGGCGCTGCCCGCGCGCAGAGGCCGCGCCCCTCCTGGCCCCGGCTTCTTGGCTGTCAAACAGATGCAG
FP009483	chr9	135961142	135961393	-	1	train	ACTTCCGGTGCGGCCGCGACGGCGCGCCCCCGGCGGCTGCTGCGGCGGCGGGAGCGAGGCGACCGCTGAGGCCGCGGAGAGTGACGGCGGCCCGGCCGACGGGAGCCGGGGCGGGGCGGCGGCCCAGCGAAGGAGCGCGCGGGCGGTCTGGCCCCGCCCCCTCCCCGCCCGCCTTCCCGGTGACCTTCAGGGGCCCGGGTGGCGGGCGCAGGCCCCTGCGGCGGCGGCGGGATGTTCGTGCAGGAGGAGAA
FP013097	chr14	31025075	31025326	+	1	train	TATAACTACGTTTCACTTCAAGGGAATATTAAATCTCTTCAGTCGTTTCATAAATACAGCATCGATCGTCTGGTTTGGTTTTTTTATAGATAAAGAAACGGATGGGGGAAAGGAGATCTAGTGGAGAGTACTGTAAAAAGAAAAATTCAGTGCTGAAAGCGTTGATGAGTCACTAAGTAAGGAATAACTACAGAGAATCCAAGTGAGTAAGGCAGCATCTCAGGTGAGAAAGTACTTAAAGTTGACGATGG
FP017199	chr19	37594691	37594942	-	1	train	TCCCACAGTTCGAGCAAGGGAGCCTCAGGGATCGTTCTGTTCTGGAGCGCCGGACCCGATCTTAGCTGCTGGGCCTGCGCAGACACGATCCGGCCGGTTTCCAGTTGTAGACTACATTTCCCGAAGGCCCCCGCGTCGCTGCGCAACGTCGTACACTTACGCTGTTATGTCTTGACGCACGGAGCTTGTCTGCGCCGGCCAGGTGAGTGGGCGGCCTCACCGGGAACCGCTCGCAGCTTTTGGGTTGCTTC
FP002580	chr2	105298483	105298734	-	1	train	CCTACCTAACTCTTTCTTCTTACTTCTAGGCATGTTTGCCACAGTCGCAGGGATATCCCAGCGCGCCCCCGTGCACTGGTCGGAGAATGTGATTGGGGCGGCTGTGTCCTTTCCATACGTCATAGCGCTCGATGACGAATTCATCACAGTCCACAGCATGTTGGATCAGCAACAGAAGCAGACGCTGCCCTTTAAGGAGGGCCATATCCTACAGGACTTTGAAGGTACTCATTCACATAGAAGTCTTAGTG
FP016283	chr18	26226244	26226495	+	1	train	GTGCAGCTTGCCCCTCCCACGTGATAGGTTTCTGGCATCGCCCCTCCTTGCTGGCGATTCTGGGGAAGATTTTAACGCTCCCTGGAGCCGTCCCCAGTGACGTCACGGGCGCGGCTCCGCTCCTTGGCCAATCACGCGGCCGGATCCCGGGGAAACCCGGGGCGGTGGCGGCGGCCTGGGCGCGTGCGCGAGGTCTCCGCGTGGCTATATAAACATGGCTGGCGTGGCCGCGCGGCGGGGCCGTGCCAATC
FP013729	chr15	42495005	42495256	+	1	train	AAAGCTTTCTTTCCTCTACACTCGGGGATGTACGATTAGCACTAAGCCTCTAACAAGGATGAACTACTAGAAATTTCCAGAAGGCGTTATGCTGTTCTCGACATAAGCTGTTCCCCAGAGCCTGGAAAACTCTAGCTCTTTCTCTGCCGGATTTCCGGGGGAACTCCTACTTATCCTTCATGACTCAAGTCACTTCCTCCGGGAACCCTTCCAGACACCCCTAAGAATAAAACGCTTTGTTCATTATCAGC
FP002941	chr2	189763147	189763398	-	1	train	GGCGCGATCTCGGCTCACTGCTACCTCCGCCTCCCGGGTCCCAGTTCAAGCAGTTCTCCTGCCTCAGTCTCCCGATTTTTTTTTTTTTTAATTGTAAAAATAAGCCAGCCCCTTTCTTCCTAGTGAAGTGGGAGAAACGGTTTACACTGTCCGATGAGAAACACTTCCGTTCTTTGGTAACCCTGCTAGGGGGCGCCGCTAGATTCCATCCTATTTCTCCGATGAAAGTATCAGGTACCTCACCCCTAAGT
FP006939	chr6	134177810	134178061	-	1	train	AATTTTTTCCGGACTGATTTGTGAGGATAATCAGTTAAGTTACATTTCTTGTGTGCGTGCTGGGGGTGTAATAAAGCTAGGACACTGATGACTGTAATTTTCTGAGTGATGTCGCAATGGGGAGAATAAATGGCCCAGTGTTGAACTTTGCAGCTTGCTCCCTTCCTGTACTTAAATTGCCTTGATATAAAGTGGGGTTCATAACAGAACAGGGATAGCCGTCTCTGGCTCGTGCTCTCATGTCATCTCAG
FP001472	chr1	163071841	163072092	+	1	train	GCAGGAACTGCTTGCCCCCCCCCCCCCCCCCCAACATACATGGCTGGAACTGAATAGACTTTTACTTTCCCGAGGTGCTTCTACAGTTCCCTCTGCCAGCAGGGGAACAGATGGAAATAGCAATCACCTGCCAGAAGGTGGCGTGCAGCAAGGATGTGCATCTTTTGCCGCTACTGCTTTCTGATTCCTAAAAATTACTCAGAGATCACTCATGTGTTCAGTGATTCAGGTTCTGTTGAAGATACCAAAGA
FP000776	chr1	61081926	61082177	+	1	train	CAAAGAAATTTGCATACATGCAAATGTGCCGCCCGGCCTCCTCCTCGGTTCTCTACGTGCCCACGCGGTGGCCCGGGGGGTGCGGGGCAACCTGGCAAAGTTGCCCAAGTCCTCCACCGAGGTCCGCGGAGGTCTGCAGCGGGGGTGGGGGGATGGGGACGAGGAAAAGTAGTTTTGTTTGCTTAAGCACATCCTGTGGCAGCCTATAGCTGCCCGGGAGTACAGACTGTAACTGCTCCTCACTGCGTTAC
FP003868	chr3	99744734	99744985	+	1	train	ATTTGAACAAAAACTACATATAGTATAGCAGAAAAATAAACTAATAGCATTTTATGTATTTATACATTCCTATTATGCAAGTTCTCCTATGATCCAGAATAATACTTTGATAATGCACTTTTAATTGCCTTGAGTAAAAGTATCCTCTTTTTTCTACTTTAGAAGCTGTTGTGAAGGCAGAGCAGCATCTGCTGAAGAGACAGAAACCAGCCCCAGAGGTGTCACAGGAAGGCACCAGCAAGGACATTGGT
FP011578	chr12	6377406	6377657	-	1	train	TGGCTCAAGGGAGACTGGAGTTTCTAGGGGTCTCTGGGATATGTGGGGCAGTGGGGACAGTGCAGAGACCTTTTCACAGAGCCAAGGAGATAACCCAGCACCCAGAGAGCAGACGAATCCACGGGCTCTGTGTGGGAGTGAGGGAGGCCTTCCCGGCTTTCACATCCAGGTGCACCTGAGCCCCGATCCCCCATGAGTCTGTCTCAGGAAGTAAATGGCAAAAGCGCTAAGTAGATAGCCCCAGAGGAGGA
FP015579	chr17	40341224	40341475	+	1	train	GACAGTGCACCAGGGGCCGGTACTGGTTCCCCAGCTAGGAGACACCTTGGGCGGGGCTTTGCTCGCCGGAAGCACGCAGAGCGTGGGGAGGAGGGCCCCCTCTGCCTGTGTTTGTGCCAACAGCACCCGCGCTGCCGCGTCGGGTTCCGGCGGCCGGAGTCACACATGATGTCACAGACAATGACACAAGCCGGTGTCTCATTCCGACACAGCGTCCGAGCTGCACAATGTCACACCCGGGTGCCAAACAC
FP011894	chr12	49323054	49323305	+	1	train	TCCTTCCTCTTTCCCCGCCTCCCAGCGGCGCTCCACTCTCGGATTGGCTGATTGATCCGAGTCAGTTTTTTTCCTCGCCAGAAAGCGGTTCGACAATTGGTCCTTCTTTTGGCCCCTCCTGCGATGCCCGCGGATTGGACGGCTGAGTCTGGCTACGCGGGCCTCCGCGGGAGCGCGACCGGGCCAATCAAGAGCTTGGCGTATTTTACAAACTGAGAAAGTAGCTCCAGCAGCACCCGAGAGGGTCAGGA
FP010970	chr11	65524897	65525148	+	1	train	CCCCGCCCCTCCCTAGCCCCCCCATCCCCGCCCCTCTCCCGCCCCTCCCTTGTCCCCGCCCCTTCCCGCCCCAACTCAGGTTCTGCCCCACCCCGCCCCAATTCACGTCCTGCTCCGCCCCGACGTAGGCCCCGCCCTGGCCCCCGGGCCCCGCCCCCGTCTGACGCAGGCCCCGCCCCCTCTCCGCCCCGCCCCGGCTCGGGCGGCCGGAGGACCCGGAGCTAAGGCGCCCGAACCCGCGGCGGCGGTGG
FP007128	chr7	767612	767863	+	1	train	ACAGGTGTGGTGTGATATGGAGGGAAAGTGAAGCCAGTGCAGAAGTGTCCGTGCTGCAAGCAGGAACTCGCACTGGGAGGGGAGACACGTGATCAGGGCGGAAGTGTCCGTGCTGCGAGGCGGAGCTGGCGCTGGGAGGGGAGACACGTGATCCGGGCGGAAGTGTCCGTGCTGCCAGCAGGAGCTCCCGCTGGGAGGGCAGACACGTGGTCCGGGTGGAAGTGTCCCTGCTGCGAGCAGGAGCTCACGCT
FP011568	chr12	5043884	5044135	+	1	train	GGGCGGCTGAAGGTTGCATCTGCTGGAAGGAGGCTTTTCGGCTGCTTGGTAACGGGCTGCCAGAAGAGAGAGAGGCAGAGAGCAGGGCAGCGGCTTCTTGACGTCAGGGCCAAGCGAGGGGATCGCGCCAGCAACCCCAGCTCTCCCCAGAGAGGGGCCGGCCGACCGCTGGAGCGGAGCCTGACGCCAGGCGCCCGCGGAGCGTGAGTAGGGGGCGCGGGAGCCGGTCAGCTGGGGCGCAGCATGCCCTC
FP011184	chr11	76782081	76782332	+	1	train	TCTGGGCAGCAGATGATTTTGCTGGCCTGAGGCCTGGGGCAGTGGTCTCCACTCCTACCTCACCACCTCTCTAGTGTCTTCCTTCTTCCCCAAGTAGACCGGCATGCATGCACACACAGGCCCAAGCAACTCCTTTTTTTTTTTTTTTTTTTAATGAATAAGAAAAAGTAATAAGGAAAGAAGCAGCAAGCAGACTGGGTGTGTTAACAAGTCTGGGCCACTGCAGGAGGTACATTGCCCAATTTCTGCAC
FP018301	chr20	63006861	63007112	-	1	train	AGCCCCCACCCCCGGCCTGTGAGCCTGCGGCGCTGGTGGAGCCCGGCTCCAGGAAGAGGAGCTCACCCGGTCCCCCCTGATTGGACAGCAGCGAGGTCCCACCGGGCGCCTCCGCGGCTTAAAACCGAAGCGGGGGCGAGGGGGAGGTAGAGCCCGCCCCCTGCGCGTCGCGGCTGGGAGGGAGAGGAGGTGGGAGAGAAGGAAGAGGCGGAGGAGGCAGCGGGCGCCGAGGCTCTGGGACCCGCAGGCAA
FP017101	chr19	33374134	33374385	+	1	train	TGCGGCGCCGCCCGTCGACCCCCGCGCTGCCCGGGGCGCCCCTCGCGGCGGGCCGGGAGCGCGCGAACGGCCGGGGAATGCCCCCTGCGGAGCCCGGCCCCCTCTCGGTGAAACGGGCGCTGTGACCCGCCAGGCTCGCATTGGCCCCTGGGGATGAAGGGTGTTCCCAGAATAGCAGAAGGTGGACCGAGTCGAGCTTTAGAATCCTGATCTTCTAGAAGGGCTCAGCTTGCGGTGATCGAAAATCTGTT
FP000813	chr1	65992534	65992785	+	1	train	ATGCATGTAAAGGATTTATCAAAAGCCTTTATGAATATTTCATGAGTTGATATATTCAGCTGAATGAATTCAGTGAGTGTCAGTGTGTAGCTTGCAGACAAACCTCATCCACAAGGAGGCTACTGACATTGGAAGCACTTTGGCGCATTTTCAGAGGCAAAGCCAGCCTGATAAAGCTCCTTGTGACAGCCTGACTTGCTATTCTTCGAGTATGCTGCTCTTGCTCTAAGACGCTCATACATTGGAGTCAC
FP012212	chr12	71663785	71664036	-	1	train	AGACCCTGCGGTGGGTGTGACCCTAGGACCGGAAGAGTAAACCCGAAGAAAGACGAAAGACTACAAATCCTAGTGAGTATCGAGTTGGTCTTATTATCGCGTGAACTGGGAGCCTTTGTTTCCTGCGTGTCGCAGGAAGTGACGTTTCGGGTACAGCCGCTACCAGAGTCCCTTTCTCGCGAGGCGGAAGAACCCCGATCGCTGAGGAGCAAGGGGGCGCTAGGAAAGGGAACTGGGTTGCGACGGTCCGG
FP017223	chr19	38617918	38618169	-	1	train	ATCCAGCCCTGGGCCTTGGAGGTTGAGGGTGGTCCCTTGGGTCCCAAGACAAGCTGCCCCTCAGAGGCTCTCAGAGCCCAGGGGACGGGGGGGTGGAGCCCGTTGCACAGTCGTGCAGTGCAGCTGGGGCAGGAAGCGAGAGTGAGGAGGGGGGAGGCCACAGCCCGCGGAGGCAAGGCGGGTGCAGGGCTTCTGGGGACGGAGGGAGGTGCCAGAAGTTGAGCCCTGAGGCCCTGCTGGCCCCTGGGCGC
FP011858	chr12	47773037	47773288	+	1	train	GGGATGGGCCCAGGAATCCGCCTTTTTAGCAGCATTCCCCCAAACGACACGGCAGCTTCGCAAAGCGGCGGTGGCGTCTGCGGTTCCGGGCGCGGGCGGCCTCCGGCCGGGGAGGGCGCTGTGCGGGCGGCGCTGGGGGCGGGCCGGGGGCGGAGCGCGGGGCGCCGGCTGCTCTGGCGGCTCCCGCGGCTCCGGCTGGCGGCTTCGGGCCCTGCACCTGTGACTCTCGGCCGCGCTCGCCCTCGGCCCGC
FP004724	chr4	55948716	55948967	+	1	train	GATGCCAGCTCAGGCACCAGCTGAGGTTTGGCATTTGCTGGCCGCGGGCGCCGACCTCCAGGGGGCGCCGTGGCCTCGCGCTGTCCGGGCCGTTGCATTTCCGGGCACTGGGGCTCCGCCATCGTCGCCAAGCGCGTCCCCGCCGCGAGCCGCTAATCGTCCGCCGCTCCCGTTACCGGGGCAACCGCGGCGCCTCCTCCGTGTCGGCCCCGATCGTCCCTCCGCGCCATTTTCAAACTGCTCTAGCGCCG
FP016414	chr18	58948030	58948281	+	1	train	TGACTGACATGATCTCGCTGCACTCTTCTATTTTAGAATAATATATAAGTGTTCCATGTGCGACACTGTGTTCACCCTGCAAACCTTGCTGTATCGCCACTTTGACCAACACATTGAAAACCAGAAGGTGTCTGTTTTCAAGTGTCCAGACTGTTCTCTTTTATATGCACAGAAGCAACTTATGATGGACCATATCAAGGTGTGTGTGCATCTATCCTCTACTTTAAATGAATTGCAGATCCATTCATTGG
FP014402	chr16	8963194	8963445	-	1	train	GGGCCGCCCCGGGGCCGCCGTCGCCGACGACGCGCGGGAGGAGGAGGAGGAGGCCGCCCCGCCGCCGCCGCCGCCGCCGCCGCCCCGGCTCGCCGCCGCCCGCCCGCCGGGCTCGCAGCCCCGGCCCCCGGCCGCAGGCGAGGCCCAGGCCGCGGCCGACATGAACCACCAGCAGCAGCAGCAGCAGCAGAAAGCGGGCGAGCAGCAGTTGAGCGAGCCCGAGGACATGGAGATGGAAGGTGAGGCCCGAG
FP019285	chrX	51332625	51332876	+	1	train	CCTCGCAGAAGCGGGTGGGCTCTGTGGAGAGTCGCGGGCTCGCGGGCAGCAGCGCCTCCTCCACTCCGGCGGGCCCGGGTTCGGCCGGCCGCCCATCGGACTGGCGGACTGGCTGACTGACACGCCTCGCTTGCCCCCGCCTCCGCCCGTGCCCCGCCCAGCTTCATCTCTCCCTCCGCTCCCCGGGCTCGGGGGCAGACGGCAGACGGAGGCGCCTCTCTCTCCCCGCCCCTCTCCTCGGCCCTTTCTCT
FP008544	chr8	106447857	106448108	+	1	train	ATGTTGGTCTGATACTAGCCCCACCCCCCAACAGCCGGGCTCCTGGCGGAGGTAGTGGGTGGAGCTAACGAGACATCTAGTACGGGGCTCACAGGTAACAGAACTCTGATCAGATCCGCCCCGGCTCCCACACAGCTATAAGGTTGCCTGCCTGCCTGCACAGAAATGACGAAGGACAAAAACAGCCCAGGGTAGGTGGGATCTATCAGCCTCCATCTGTTGTTACTTCCTGTGTTATGAAGAAAACGTTT
FP015039	chr16	89720829	89721080	-	1	train	GCTCCGTCGCGACAACCCACCCGGACTCTGCCGCCCCCGCCACTAACACCCGTGGGGCCCGCAGTACGCGCGGAGGGGGCCCGTCGCCGGAAGTGCGGCCGCGTCCCTTCCTGCGGTCCGCTGCTCGGGCGGCTCCAGCACCAGCGCCGGCTGCGTTCCGGGCCTCCGGTCGCCCGTCCAGCCCCTCGGCTACCGCCGCCGCCTCCCCCGCTCGGCGCCATGGCCGCTGCGGCCGGGGACGGCACGGTGAA
FP004161	chr3	142028575	142028826	-	1	train	TAAGACCTGCACTTTTCAATTTCTTTTGAGATGTCTTTGTTGTAAACAGTATTCATATGTCTTTTGAAAGCCAGTTAACTAAACAGTTTTCTTGAGCATCTTTTTAGTTTTACTGAGAAGTATTTTAAATTGAGCTTTTCTGAGCTCGATTGCTTACGTCTGACACAGTCTCAAGTTTCCACTGAATGGTAACAAAGACTGTAGAATGTTGTTGGTACTGCAGTGAGAGGCATGCTTCCTTAGACCAGGTA
FP001072	chr1	113953597	113953848	+	1	train	ACCTCTGCCTCCCTGGTTCAAGCGATTCTCCTGCCTCAGCCTCCTGAGTAGCTGGGATTACAGGTGCCTGCCACCATGACCAACTAATTTTTGTATTTTTTAGTAGAGACGGGGTTTCACCATGTTGGCCAGGCTGGTTTCGAACTCCTGACTTCAGGTGATCCACCTGCCTCAGCCTCCCAAAGTGTTGGGATTACAGGTGTGAGTCACCGTGCCTGGCCTTCTTCTTATTTTTTAAAAATGTTCCTGCC
FP018719	chr22	30396817	30397068	+	1	train	CTGCCCTTCTGATGCCCACTGGCCTCAGTCCCAGTGGGCACACTTGCTCGCCTCCTGCCTGGGTCTCTGACTTCAGGAGGGCGGAGGGGGCCCGTGTCAGAGCCTTCCGCGCCCCAGCGCATCGCTCCCTACTCCGCCTCTCGGGATCCTTTAAGAGGCGGGGCTTGGCTGCCAGCTCCGCGGCCCGGGCAAAAGGCTGGGACTTTACTCCGGGTGGCGGCGAGGACGAGTCTGTGCTCCATCAGCTGCCG
FP010321	chr10	125823163	125823414	-	1	train	ATGACCCCTTACAGTCACGTCCAGCTTGCGGTCCCATCTTGCTTTTTTCTTCCGCTTTCCACCCTCCATCACTCCTCCCAAAGCTTCCCCTCCCCTTCCCGTAGTGGTTCAACCCATCTAACCTCTGGCCTCATTCCGTGGCCTGGATGGCGGTGCGCATGCGCGAGCGCCTAGCTGCGCGCAGCCACCCACGCGACCCCAGTCTGAGGTGCGGGGTCCTGGGGCCCGGCGCGGGTGGCCGCCGCGGCCCC
FP017322	chr19	42217653	42217904	-	1	train	CAGCCCAAGAGGCGGCGTGCTCTCGCTCTGCGCAAGCGCAGCACTCGCGAACCAAGATGCTTCGGTTGGGGGCAGGTCAAGCGGCCCAGGAAGGGCGGGGCTTGGAGGCCTCGTTCTGCGCATGCGCCGAAGGGGCGTGGCGGTCGCACCTGCGCGAGTTGGGGGAAGGAGGAGGGGCGAGGTGACGCAGGCGTAATAATAGAGAAGGTGCCAGAAAGATCCAAAACAAGTGGCTGCGGCCGTCGCCCAGG
FP002946	chr2	189853776	189854027	+	1	train	AAAGTGCTGGGATCCCAGGCATGAGCCACTGTGCCGGGTCATGATAATGCTTTTCCTAATAAGCAGTTGCATCTACTCAATTTCTCAGTTGAATTTGCTGGGTTTTATTGTACTTTTTAATTACATATTATTTTTTCTTTTATTTATTACATGTATTTCTAGTTAATCCGACATCATTACAATCTGAAATGCCTAAAGGAATCTACTCGTTTGTATCCTGTTTTCTTTCTGAAAATCGATGTTCCTACAGC
FP015516	chr17	37745008	37745259	-	1	train	AGCCCCCCAGCGTGAGTACAATGGACCCTGGCAAAGCCCCGCTCCCGGCCCAGGTCTTCTGCTCTCCAGGTCTGCCCCTCCGGCTCTCCCTCTCTCCGGGTTTCCCCCTCCCCACCATCATTTGCATCCAGCCGAAAGCTGGGCCCTTCCCACTAATTTGCATATCTTATATGGCCTAATGGTGGCGATCATGGCAAGTTAGAAGTTTTCTGACTCCTTTCGGAGGAGCCTCCGGGACCCCGGGGAGTAAC
FP014484	chr16	20770029	20770280	+	1	train	GACTGATGCTAGCTCGTGTCACCAGGAAGATGCTACGTCATGCCAAGTGTTTTCAGCGCCTAGCAATTTTTGGTTCTGTGAGGGCACTGCATAAAGATAATAGAACAGCAACCCCTCAGAATTTCTCCAACTATGAATCCATGAAACAGGACTTCAAACTGGGGATTCCAGAGTATTTCAACTTTGCTAAAGATGTCCTGGACCAATGGACTGATAAGGAAAAGGTATGGGGGGAGGGCCAGTCAGACCAC
FP018106	chr20	38260684	38260935	-	1	train	GCGGACTCGGGTTCCTCGGGTTCCAGCTGGCTGCTGAGGGGGTGGACGGAGGGAAGGGGGGGCGGTGGGGGCGGGGCGCGCGGGCTGGGCACAAAGGGCTCAGGCACAAAGGGGCCGCTGTCCCGCCGCGCCCCGCCCCGCCCCGCCTCTGCTGCCGCCCCCCGTCTGGTGGCCCGACCCGCGCCGCCCGCGGCTCCCTTATGTCTCTCGCCGTCTCCAGCCTCAGTCCCTCTCTGTCTCTCTCTCAGTCT
FP002691	chr2	130342092	130342343	-	1	train	GGGCTAATTGGAGAAACCAGTTGATTGGTTGCTTCCGTGCTGCTTCTCTAAGGGCGCTTGCCCTTAAAGAAGCTGACCCAGAGGCCTTGGTGCTGCCGGCGTGGGGATCCCCAGATGGGAACGCCCCTTGATGGAGGATCTCGTGTGCCTATTGGCTGTCCTGCCAAACTGCGGAGGGTGACAAGGAAGAAGGTGGCTCCAGATCTGGAGGTGTGTCCATGGCGGCGCTTGACCTGCGAGCGGAGCTGGAT
FP002276	chr2	53859768	53860019	-	1	train	GGCGGAGGGGGCGGTGCCCTCGGCGTCTCCGTGACTGCGCCTCTGCGCCCGCGTCTTGCCGCGGCTCCCGGGATGCGCGGAGGCGGTGGCGATGGCGATGATGCCTCTAGTCCTGCATCATCCAGAGCGGCAGGCGGAGCTGGGGTCCGGACTGCGAGATGGAGGAGGGGCGGCGCTGCGGCCACCCGGCAGGTGAGAGGCCGCGGGCCCCTGGAGGAGGACAACCCCACGATGCCGGAGACGGCTCCCGG
FP004842	chr4	86099297	86099548	-	1	train	GCAGAAAAACCTGCTTGCTCCCACATCAGGCATGCCCTGCTGTCCCCTGCACCTAGCCCCGCCTTCCTTCCCCATGGAACAATTCGGCCAAATGATTCTGCCCAAAAGTTCCCTTTGTCCATAATGAAAAGCAAGTTGGAAAGGGGTAAAGATTGGTAAGAGAAGGCAAAGCATGTTTTCCTAACAAGTTTGACAATTTGGAAATAAATATATGTACATAACAATTTTTAACACCTTTAATACTTTTTTCA
FP011255	chr11	93741482	93741733	+	1	train	GCGGAAGAAAAGGGTTGGCTATTTCCGTGGCCCAAGTAATAGTCAGGCCGAAATCTCGCGATACAACTTAGCTTCCGGGAAGAAGGCGAGCGGTGGGAGGAGACGCGACGTGGGGGCGCCATTTTTCTCGCCGCGCAGGGAGGACTGACTAACGTGGGCGGAGCTCTAGCTCGCGTATTCTGAGGAGGCGGGGTTGGCCTAGGCGAAGATCCGGACTCTGGGTGTTTTGCTACCGTGACCGTTTAGGTGAG
FP017645	chr19	52857568	52857819	-	1	train	GCTCTCGCCGCCAAAACCCGGAGAACGGGTTGGGAGGAGGCTGAGAGATGCACAGGTCCCGCTCCGGCCCCGCCCTCTGCGGGTTCTAAAGGGCAAGGTCTCGCCGCTTCGCGCCCCGCCCACACCTCACCCGGGCCCCGCTCGCCTCCTCGTTGGCTCCACCCAGGCCTGGTTTCTGTCCTGCGCGCGCAGATTCGCGCAGACCAGGAAGTGGATCCCGTGGAATGACGGTCACGCCGCGGCGGGCGGTG
FP005346	chr5	42423533	42423784	+	1	train	GCAGCAGTTCTCGAACTGGCCTCCTTGAACGTCCGCTTCGCCTTCGCTTCTGCAACCTGGATCTGGGGGACTGCGGGCCAGGCGCGGCGTGACCCCTGGTGAACGGTGGCCGCCTTTTCCCACCCCTGCCCTCCCATCCTCCCTTCCCGTTTCACCCCGCCCCCTCTCTCCTCCCCAAGCCTGACAGCCCGCGAGCTGCCAAGCAGGGCGCAGCCATGGGAAGAGGAGGAGGGCTAGGGAGCGGCGGCGGC
FP010829	chr11	60841912	60842163	+	1	train	TTGACTGCGCTCGCGCGAATTGCGCATGCGCATGCTGAGCCCCACTCTCCCCAACCCCGGTCCCGCCTCTCCACCCCTCACCAACATGGCCGCCTCAGCAACAGCCCCCCTCCTGTGCGTCACGGACGCGGCGACTCCTGACGTCATAGGAAGGCGCCGGTTTCGGCGGGGGCTGCACGTGCGCAGGGGTGTGGAAACTTACCGGCTGAGCCATGGATACACCGTTAAGGCGCAGCCGACGGCTGGGAGGC
FP007632	chr7	100126549	100126800	+	1	train	TTCAGGAGACCAAGACTGGAGGATCGCTGGAGCCCAGGAGTTCCAGACCAGTCCTGGCAACAGAACTAGACCGTGTCTCCAAAAAAAAAGAGAAAGAAAAAATAGAAACCTTAGCTGGGCTTGGTGGTGCGCATCTGTGGTCCCAGATACTCGGGACGCTGAGGCGGGAGGATCACATGAGCCCAGGAGGTTGAGGCTGCAGTGAGGTATGATCGCGCCACTGCACTCCTGCCTGGGCAACTGAGCATGAA
FP002155	chr2	27890554	27890805	+	1	train	GCCCTCCTGATATGGCGAGACTCCGCGCTCCTCGTCACTCACGGGGCAGGCGCTGAGGAAGGAACTGTCAGGCACCGCGGGGACGCCTTTCCGAGCGCCCGGCTCGGCGCGGCGCGGGGGCGCGCACGCCGCCCAGCTTCGGGGCCGCAGAGGGGGCGGGGCGGGACGTCCGGGGAGTGCGCACGCATCCTGCCCGCGGCGCGCGCGCAGGTCGGTGCGTCTGTCGGGGGCGCGCTCGGGTACCTGTACCC
FP019431	chrX	85244520	85244771	+	1	train	AAGTGAAGAATGCAGCGTCCCCCTGGCAGTTTCTCTCAGCTCTCCCTCCACCCCATATTCTTTTCACTGGCGCCCCAGTCGCCTCGCCCCCGCCCCCTGACCGAGCTAATAGTAACCTTTGGAATGCGCGAGGGATGGCGGCGTGTGCGAGGAAGTGGCACCGAATCGGCATTTTAGGGGATTGGGAGCCGTCAGAGCCTGGGAGTTGGGGGGAGGGGGGGGGCGTGTGCGACAACTGTCGCCTACACTAA
FP000387	chr1	27615533	27615784	-	1	train	CGCTGGGCCTGGCCAAGGACGCCTGGGAGATCAGCCGCAGCTCCATCACGCTGGAGCGCCGGCTGGGCACCGGCTGCTTCGGGGATGTGTGGCTGGGTACGGAGCTCCCGGGGGCCGGGACGAGGGCCTGGGCTCGGGGGAGAGGGTCCTGACAAGACAGCCTCCGAGCAGGCACGTGGAACGGCAGCACTAAGGTGGCGGTGAAGACGCTGAAGCCGGGCACCATGTCCCCGAAGGCCTTCCTGGAGGAG
FP002752	chr2	151828370	151828621	-	1	train	CCTGCGCCTTCCGCAGTCTTGGTTATGCAAATAAGCGAGGACAGGGCGGGCGCCCTTTCCTACGCTCCGCCCCTCTGGCCGCGCGCCTGCGCACTCACGTCCCTCCAGGCCCGCCCCTTCACTTCCCCCGCCCGCGTGGTCGTCGCCACCGCAGCCGCCCGAGAGGAGCTGGGGGAGGACGGTGGCCCGCGAGGCTCGTCGCAGACAACGCGGCGGCGATGTCCGCGAGCCAGGCGAGTGCCGCGGCGGCA
FP010761	chr11	47176815	47177066	-	1	train	ATTGGCCCGGCGCGGCGTTGATTCTTGCGGCCGCGGTTGTCGTCGCCGCCCCCGTCCCGCCGGCTGCCCATTGGTCCAGATCGGACTCCAATCACCCCACCCCTCGGCTGGCCCAGGCACCGCTGAGGCCGGCCTGGTGAGCCCGCCCTCCCCGCCGTGGATTGGCCCGCGGCGGGACCCGTCAGCCGCGGTTGTGTCTGGGAAGGAGAGAAAATGGCGGCGGAGCCGAACAAGACCGAAATCCAGACTCT
FP018406	chr21	33641690	33641941	-	1	train	CCGGGCGATTCCAACTTCGCCCATTTATTGGTAGTCCTTGGGGCGGGGCCTCGCCTCTTTCTCCGCCCCCGAGCGGAACATGACGGGAGATGTAGTTCCGAGACCGGTTGCCATGGCGTTCTGGACAGCCTCGCGAGTCCGACCCCGCCCTTCAGGAGTGACGAGCGGCCTGACCAATGGGAGCAGGGGGCGGGCTCTCTGACGAAGGACTGGAAGGTGGCGGTGGTGAAGGTGCAGGCCGTTGGGGCGGC
FP012944	chr13	113449107	113449358	-	1	train	TCTCTCTGAACAGGGCTCCCTCCTTCCGGGTGAGCCTCTCTGACCGGGGCTCACTCCTTCCGGGTGTGTCTCTCTGAACAGGGTCCCTCCTTCCCGGTGAGCCTCTCTGACTGGGGCTGCCTCCTTCCGGGTGAGCCTCTCTGACTGGGGCTGCCTCCTTCCGGGTGAGCCTCTCTGACTGGGGCTGCCTCCTTCCGGGTGAGCCTCTCTGAACAGGGCACGAGAGCAGACTCTGAACCAGGAAGTAGCCG
FP003572	chr3	45226236	45226487	-	1	train	CACGCACCCCGGGCGATCCGCTCTCCGCCGCGCCCCCGCACCCAGCCCCGCCCCGGCGATTCCTCTCCCAGCCCCGGCGCGTGGAACCCCGCCCTCCCCTGAGTCCCCGAGCCTGGCGAGCGGAGGGCGGGGCGGGGCGGGGCACGCGGGAGGCGGAGCCGAGCCGGGGAATCCTGCTCTGGGATAGCACCCGGCCCCGCAGAGCAGCGCGGCAGCCCAAGGGCCCCGGCGCCGGGGGCGGCGGGGAACCC
FP014917	chr16	71884110	71884361	-	1	train	CGGGCGAGCCCCTCTCGGGCGCCCGGGATTGGGCGCTCCCCGGAGCCCCTCCTCTCCAGAGCCCCTCTCCTCAGGCCCCGCCCACAACGCCAGGCGCGCGCCGGGCCGCGGGGAAGGGGACCTGGCCGGCGCGGGGGAGGGGGAGAGGGAGGCTCGTGCACGCGCCTCACTAGCTGCGGGTGCGCGAGCCGCGCGCTCCCGCCGCCCGCGTCGCCATCTTTTTCCCCCTCCGTCTGTCTGTCTGCTGTTGG
FP018972	chr22	46267819	46268070	+	1	train	GAGCTCCCTGCTCCCAAGGGACCCGTCGCGTTGCAGAGTTGTCCGGTCGGGCCTCTGGGGCGGGCCCGGATCCTGGACCAGGGCCAGTCTGTCCGAAACCCACCAGTGAGCGAGCGGGAGAGCGCGGCCGCCCCGCCCCGCCCCTTTCCGCGACCGCCCCGCCCACTCCCAGGAAGGCCCGGGTGCCCAGAGCTCGCGGTGGACTCCGACCCGGCGCAACATGGCCGCAGCCTCGCCTCTGCGCGACTGCC
FP011504	chr11	130069693	130069944	+	1	train	GCGGGGCCGGGCGGGGGGCGCGCGGTTGGACCGGGCGGGAGGAGCAGGCTCTTCCATCTCCTGATTGGGTCTGGACCGCAAGGGGGCGGGGTCTTGAGGGGTTCTGCGGGCCGGCATTGGGAGCCGCAGAAGGAGGGCGTGGTAATATGAAGTCAGTTCCGGTTGGTGTAAAACCCCCGGGGCGGCGGCGAACTGGCTTTAGATGCTTCTGGGTCGCGGTGTGCTAAGCGAGGAGTCCGAGTGTGTGAGCT
FP017045	chr19	20077772	20078023	+	1	train	TCTCCGGTGGGCCTGGCTCAGCTCAAGGAGGAAGCCCGGCCTGAAAAGGCTGCAGCTGAGGCTGTGACTCTTTCTTCACTCAGCCCAGCATGTGATCACATCTTCTGTCACTCAGGGACTGAGGAGGCGGGGCCTTAAGCATTATCCAATCAGGGACGCTGGGTTGGAAACCGTCCAATCATGCAGGCAGCTGGAGCGAAGAGGAAGGCTTTCGGGTTTGGCGCGGCCATTTGTCTCTTGCTGCAGCTGGT
FP005921	chr5	151087554	151087805	-	1	train	GTCAGGCAAAGCAGGACTCATTAAGCAGGAGATCACTAGGGCTCAGGATGATGAGGTGACTTGTTCAGGATCTCCAGCTGGTCAGTGCAGAATTCCTTGGGCTGCGGTTTCTAAGGCCTTACTTGACATCCAAATGTGACTCGGCATGCACCAAGGAAGGAGTGGGCCCCTTCTTCACTATGGATGGAGAAGCCTCAGAGAGTAAGTGGCAACAGGTAGGATAGGAACGGGGTGGGAGGATGTGGGCTGGG
FP011757	chr12	25195109	25195360	-	1	train	CCCCGGCGGCGGCACTCACGCACTTTCCGGCACGCCGGAGCCGCAAAAGCCTCGACCGCTACGACTCCTCTGTCCGTCTACTGCGAAGCGGGGTGGCGTTGGGGGAGGTGGGCAACCTGTCAGTAAAGGGGGCGGGGCGCCGGCCGGAAGAGCCTCCGCGCGGCTGCGCTCTTTTCCTGGAGCTCAAGTGGGCGGGGCCTGTCGTCCGGGCAACCCGGGAGCGTTTGTCCACACAATTTCTGCTCCGACTC
FP012938	chr13	113209412	113209663	+	1	train	TAGGGGGAGTCCGATCCCTCAGGAGGGATTCCCGGCGCGCGTTCCTAAGCCCTCAGGAGTAGCCGGGCGCGGCGGCGGTTGGGCTCGGGCACGCGGGCGGGGCGGGGCGGGGCGCGCGAGGAGGACGGGGCGGAGGCCGCGCGGACCGGGGCGGGCGGAGCGGAGCTCGGCGGGCGGCGGCGCTCGGGGCGGGGCGCGGCGGTTCCGGCCCAGCCATGGCGGACGAGGCCCCGCGGAAGGGCAGCTTCTCG
FP013660	chr15	37101258	37101509	-	1	train	ACAACAAAGCTGGGCTGTGAGCGCCGGCCACTCCTCCTCGCCCGGCAACTTCTTTCTCCCCCACTCTCTCCGTCTTTTCCTCCTGTGCTTCCCAGCTCTGAGTGAGATCCACTTTGGAGACGCACACTAGTCGCGCGCGCGCACGGGCGCGCGGGCGCGCGCGCGCACACACACACACACACACACACACACACACACACACACACGCCTTTGGCTACATCGGACCCAGATGACTGCCTCCTCACTTCCTC
FP009178	chr9	110048495	110048746	+	1	train	ATTTCTTCGTTTGTGCATCGATTCCGCCGAAGCCACTGAGAGGAGAAGGTGTGGGTAGATGGGGAGGGGAGGGGAGGGAGGGGCGGTCCGTGGGCGCTGGGCTACTGGAGGGGAAGCGAGGAGGCGGGGAAGGGGCGGGCCCCAGGAGCAGGCGGGCGGGGCTCCCCGCCCTCCAGCGCGCCCGGAGGCTACCACTCCCTGCAGATGCGCTGGCCCCAGCCCGGGGCTGCCGCTCGCCTTCCCCCGGAGTC
FP011315	chr11	108222587	108222838	-	1	train	GCTCCGGGGCTCTCACCCACCCTCTTCGCCCTCGTCGTCCTCCCCGCCCTCCTCATCCCCGCCCCTCCAAGTCTGAGGACGGAAGTGACGACAGTTCCGAAGGCGAACGGAGGCGGGTTTCATTTCGGCGCCTTTCTTTCTCTGGCGGAAGCGATTGGCTGCCGTGAAGCGAAAGAGGCGGGACAAATTGCCGCGAGTTCAGTGCCGCCACTGGAACCAGGAGATGCGGCGCAGGAGCTGTCGCTGTGTTT
FP003177	chr2	223945284	223945535	-	1	train	ACTTTTAAACCTTCCTTACTTTCCCCATTTCTCCGCAGGCGCCCCGTGGCATCCTCTCCCAGCCACAGACTTTCAGCTTGCGCGGCATCCCCAGGCAATACGGGCGGTGCGGCGGCAGCACTCGGGGGCTTACGGCGGCGCGGCGCCCAAGAGGCCCCAGCAGCCGAAGGGAAACCGGCGCGTCCCCCGCCCGCCCAGGCGTCAGCTGATGGGCTGCCTGCCGAGGAGGCCGCAGCAGTCGCCGCGCGAAC
FP004523	chr4	2953591	2953842	-	1	train	GCAGGAAAAGATTTCCGTCTGATGAGATTTGGTAGCTTTGCATTTAAAATACTCATTTCTCCTGGCCTCTTTTCTGTGTTGAAACGTTTGGTGCCAACTGAGTGATCACTGTGTGTGTCCTTGGCACTGACATTGGGCCTCTGGGTTGAAGTCCAGTAGTAACGGTATGTGGTGTGTGTCTGTTATTTTTTCCCCAGAGGGAGAGACAAGCTCAACGAGAAGATGCCCTCGAGCTCACGGAGAAGCTAGAC
FP014150	chr15	90265854	90266105	-	1	train	CTGTAAGCATGCCGAACTCTCTAAACTGATGATGAAATACGTCACTAATGAAGGGACCAGTGGTCCCAGGGGAGCTGAACACCTGACGCTGCGCCCCTAACAACGGGTAAGAAGAAACAAGGGCCGCTGCGCAAGCAGCGGCAGGACACCCGGCGCCAAGGGGGCGCGGCCTGGAGCGCCGGGGCACGAGGCCCTGCTTCAAGCTGAGGGCCCGGGGAGAAGCCGGTACCTCTCCACCTCCTGCAGCTCCC
FP016616	chr19	4198447	4198698	+	1	train	CCTCTTGGAACCCCGTGCGCCCCCCGCGCCCCGCGCCCCGGACGCCATGAAGCAGCTGTGTCTGTGCGCAGCCGCCTCCTTCGCGGTAGGGCCCGGGGAGGGGGCGCAGGAGCGGGCGGGGCGCGGGTACCTCCTCTCCCCTCCCTTCCCCGTCCCCGGCTGACCTGGACCACCCCCCCATTCCAGACCCGGGAAAGATGGTCGGCGGCGGGGGGTGGGGGGGAACAGAGGTTGGGGCAGCTTTTGGGGGA
FP018247	chr20	57525078	57525329	-	1	train	CTCCTCATCCACCCCCCCAGGTCCACCAGTGCCCCCTCTGGGGTCCTCCTCATCCGTGCTCCCCCTCCCCCTCCCTACTCCCCTTCCCCCCTGCCCCCACAGTACATCACCCCCTCCCCCAACCCTGCCTGGCTCCGCCCCCTTCACGCCCCCTCTTTTCCGCTCCGCGCCTGCGCACTGCCACCCTCCACTCTCGCGCCAGCCCGGCGGCGGCCGGCTGTGGGCTGCAGCACGCGGTGCACGAGGCAGAG
FP001877	chr1	228207744	228207995	+	1	train	TGGCCGTTGCCCCCTGGCTGGCATCACTGCTGAGGACACAGGGTCCCCCATCCCTGGCTGTGACCCCTGGCCTCCATCATTGCTAAGGACACAGGCATTTGTCCCCCTCTCCCTTGCTGTGGCCCCAAGCATCTGCCCTTTGCCTCCCACAGGGCCGAGGGTGAAACTGAATCCGGGACAGTGACACCCGGGCTATTTTGGGGCTGGGCTGGGCCGGCCTGGCCGGCACAGGGGGTGGGGTGGGGGCGGGC
FP014715	chr16	55480344	55480595	+	1	train	GGAAAGTTTCCTCGAACTTCTCCAAAGGGTCGGAGAAAAGAAGGAGAGAGCTGGCCCGGCAGGAGGGAGGAGGAGTGGGGCAGGCGCTGGAGGGCCCGGCGCGTGGGGCGGGGGCGGACTGCGCTCCGCTCGGGTCGGAGAGCGGCCAGAGAGCCCTCCTTCCTGGCTGGGCTCCCAAACCGCGGTTCAGATGTTGTCTTGTGAGCGTGCGCGCGCCTGGCTGGAGGGGCACTGAGCCTGGCCGCAGTGTT
FP017143	chr19	35745450	35745701	+	1	train	TGGGAAGTGTAGTTCGTTGCCCTCTCCAGGCGCTTACCAAGCGCTCGGAAACGCCCCAGGGCGCTTGCGCGTGGGAGGCGCGCGCCCTCCCCATAGACTCCCCGGTAGGAGTTCTTTTAACAGCGCATGCGTGCAGTGTTGCCTCGCCCAAAGAAGACTACAATCTCCAGGGAAACCTGGGGCGTCTCGCGCAAACGTCCATAACTGAAAGTAGCTAAGGCACCCCAGCCGGAGGAAGTGAGCTCTCCTGG
FP000782	chr1	62272986	62273237	-	1	train	GAGCCACCGGCCTCCTCCTCCTCCCCGCCAGTGGGTAAATACCATCAATATAGGGGCCTGAGACAGGGAGCCATTTCTCCTTCACATGATAAGGCAGAGGGAACAAGGTTTTCAATTATATTAACATGTTTTTAAAAAGATTTTAGCAGCTGGGAGCAGTGGCTCAGGCCTGTAATCCCAGCACTTTGGGAGGCGGAGGTGGGAAGATCTCCTGAGATCAGGAGTTCAAGACCAGCCTGGTCAACATGGCG
FP000565	chr1	40413571	40413822	+	1	train	ACATATTGTGGCTGTAGTAGTAGAATCTCTAACATAAAAACTTGGAATCTTGGGAGAGAGGACTTTACTTACTATTGCTTAAGGAAGAGTCTTTTATCATCGTTCTGAATCACATTGTCTGTGGACACAATCACTAGTGATACTGGCAGAGTGAATTAGGTAATCTTTAAACCAACCTCCCTTTTGGTGTTAAAGTAAAAGGTTTTCTAACCACTGGGAGATTTGTTGGCTCTTGGCTCTTGTACCATCAG
FP006375	chr6	31902022	31902273	-	1	train	CGGGGCTTGGAGGTACAGAGGAGGGGGAGCGAGAAAGAGGGGGGTGTGGAACGTATTTCCGCTCTGGCGGACAAATAATCCCGGCCAAAGAGGAGGCAAGGCCGTCCGGCCCTTTAACCGCGTGGGGGTGCTGGTGAAGAAAGGGGGGTCGGGAAGGGGGGATCCTGCTCCTTTAATTCCCTCCCCTCTTCCTCCTCCCCGAGTCCTAGCCGACGCCGCCGCCGCCGCGCGCGCGGGGCCTGGAACACACG
FP011260	chr11	94128750	94129001	+	1	train	TTTGTCAGAGAGTGGCCCGAGGGGCGCGGAATGCAGCCGCGCCCACCCCGCCCCGCGTCACCGCGTCTTCCGGAAGCTCCACGCCCCTGGGTACTTGGTTTCCCCGCATGGTTCCGGAAGAGCGCGGCGCAGCTGGCTGTGAGCGCAGGGCTATCCCGGCGGCCGCTTCGGCAGCCAGGGCGGCGCGGAGGGGCAGGGCCAGAGGGAAGCGCTTTGTTCCGCGCGTGGTTCCCGCGCCTGGGGGTGCGCGG
FP008196	chr8	27311428	27311679	+	1	train	CCACTTCCGGTGTGCGCGGGAAATCTTGGGAGAGCGGGGTGGGCCTTCTGCCCGCAGTTCCCGCCTCCTCAGGTCCGGGCGGGTCCCTGGCCGGGGTAGCACGGAAGGGTCTCCCAGGCGGCGTAGTAGGGCTTCCGTGTTACTGGAAACCTACTTCCGGCTGCAAATGGGAAAAGGAGCCTCTACCTTAACCAATCCCCGGGAACCTCAGGCCCGCGGATGGGAGAAACCAGAGATGCCAACTTCCTGCT
FP019775	chrX	154547441	154547692	+	1	train	AAAGTGGCCGGCGTGCTTATCATTACCGAGCTTCCGCGGGCCTGCAGAGCCTGGCGGACTCAGACTTCTCTCCGGAGCGGGATGCGGCCCTACCGCGGCCTCACACTTCTCGCCGGCTTCCCGAGTTCTCGGGGGCGGGGCTTGTGTTTTTACTTCCGGATCCCACAGCTATGACACCGGAAGCCGGAAGCGTGGTAGGGAAGGGCGACCGCGAAACTGGGACTTTCTCGGAGCGCCGGGGCCCTACCAGC
FP007798	chr7	123557753	123558004	-	1	train	AGCTTAGCTTTAGGCTTGTATGCCCATCGCGCATGCGTCAATTTTTCTCTTTTAAGATGGGCGGGGCAGAGTCTTTGCTCCTTTGGCGATCCTGAAGGGGTGGAGCTAAGCTGTTTCCAGGGTGACAGAGTGGCGACCTCGGTGGTCGATTGAGCAGGTCTGAGAATTGTTCCCAAAGGGTTGTGCGTCACCGAGTCGTTGGCGCTGTCATGGCGGGTGTGCTGAAGAAGGTGAGACGAATGGAGGTCACT
FP018203	chr20	49278006	49278257	-	1	train	CTGAGGTGACGCCCTCCTCGAGGGCGGAGGCGCCGGCCGTGCCTGGGCCAGGCCCTCGTGCTCTCCACCCTGGGCCCCCGCCTCCTCCTTCTCCTCCTCTGGGCCGCCCGCCCGGCTCCGCCCAGGGTCTGCGGGGAACGGAAACCGAAAGTGCGCGGCGCCGGGCGGGGCCACCGACCTGCCTGGGCCGAAGCCCCAGCACTCGCCGGCGGCAGTGAAAGGACGCGCCGGAGCCGGGTGAGTGGCCCCGC
FP003963	chr3	119557872	119558123	-	1	train	TGGAAGGGCTGTGGTTGGTTCTTGGAACTGGTCCCCTCACAGAATTTCCATTCGCTTTTCCTCAGGGTCACGCCCTGCTTTAGAGGTTGCTTAGGTTAAGCTCTCCAATTAATTTTAGCAGGCAACTCGAGAGTTTTGCAACTCAAAGTAGGCACACCTTCCTTCCTGAATGACTTTTATTTGTTCCTTTTTCAGGCTGTGAAACTAAATCCACAACCTTTGGAGACCCAGGAACACCCTCCAATCTCTGT
FP011321	chr11	109421919	109422170	+	1	train	TGGAGGTAGGGGTTGCTCCTCCTAGTTCCTGGGCTGGAGAAGGGCAGGGAACTGGGAAGTGTGTGTCAGAGGCTCGCTATGCTAGGACAAGGTGTTGGGGTACTAGGGTGGGTGGGGGCCTGAGGGGTGGCTTGGAGCCCTGGCGGTGGCTGCCTGACGTCACCGGCCTCTCTGGCAATAGGCTGCGAGCTCAGCTCCCTGAAGCGGCGGCGAAGGCGGCGGCGGCGGCGGCGGCGGCGGCAGCAGCGCGG
FP001202	chr1	150876659	150876910	-	1	train	ATTAGGGAGACAGCTGGACTTCTTCCTCGCCCTCCCTTCACTGGACTGGCTGGCGCAGTGAGTAGCCCTGCCCTTGGACGATTTGGCATTTTCATTGGTCAATTTTTCTTAGGAGGCTGGCCGTGTGTTGACTCCGCCTACTATATAGGCGGGGTCTCCCCGCCGCAGGGGCTGGGATGCTGGGGGCTGCTGAAGCCGCCATCTTGGATTCCGCGGTAGCGGAGGCGGCGGTCAGGCGCCGCTTCTGGGGA
FP018829	chr22	37906138	37906389	+	1	train	CGCTCCCACGACCGGCCCACAATGAGGCGAGCGGGCAGCGCGGAGTAGGCGGCGGCCGGCCGGGCCGGGCCACGCGGCGGCGCTGCTCGGGCAGGTTAGGGCAGGGCCGCGGGGCGCCCGAGCGAGGACACCGCGGCCCCGCCTCCGCCCCTCCCCTCGCCTGCCGGTCGGCGCCCGAGCTCGGAGCCGCAGCCGCAGCCGGAAACCGGGCCCGCGCGGCGGCCGCCGTCCCGGCCAAGCCGGGGCCCCGA
FP006687	chr6	71288610	71288861	+	1	train	GAAGGGAGGCGCGGCCTGGGGCCGGTGGCGCTCGGCTGGGGGCCGCAGTTGGCTCCCCTGGGGGCTCTGGGCGGCGGCAGCTCCCGCCTGTGGGGAGGAGAAGGAGCCGGATGGGCGGGTTCAGGCGCGCGCGGCGGCGGTCCCGGGGCCGCCGGCTCCTCCCCGCGAGGGTGCCTGTCCGCCGCTCGCCGCGCTGAGGCAGTGCGGGGCGGGCGCGCCTAGGCTGCCGCCCAGCGCCCTCGCCGCGGCCA
FP013967	chr15	70895659	70895910	+	1	train	ATGCTTTCCATCTCTATGGATTTGCCAATGTGGTATTTTGTGTCAGTTTTCTTTCACTTGGCATGATGTTTTCAAGGTTCATCCACATTGTACCATGTGTATCAGTTATTATATGTTTTTTTAATTTATCTTCACTGCTTGGTGTTAAATTTGTCTCCAAATATTTTTTTCTTTAATAATGTTTTTCAGGTTGAATTCAAGCTAAATAAAGACACATCGTCATTCCCCGGTAGACTTTTACAACATGACCT
FP003857	chr3	97764557	97764808	+	1	train	TCGCAGGAAATAGGCAGCTCACCCTGCCACCTGTGCACACATTTTTCTTCAATGTATAGTTCCTCGGCGCGCCTCTGACCTTTTCCCGGAGCTCTTCAATCCCAGAAGCTATAGCACAACGGCTGAATCGCCAGGACCCCCGGGGAGGCGTGGCTTCAGGACCGGAAGAAGCTCCTGTTGCCAAGGGAACGGTGCCTGCCAAGGCGCCTGCTCAGCGACTGATGCACAGACTGCTGCAGAGGCTGCCGGTT
FP013687	chr15	40470796	40471047	+	1	train	GGTCTCAGGAGCTTTTGTCCCCAAGCCCAGAGGACAGGGGTGGCCCTGACTCAGGCGCCCTCTTCCGGGTAGGGGTCCGCGTGCGCCCTCCCAAGCCCGCGCCCCGCAGGGGCGCCCGCCCCTCTCCCCGCCCCCGGCGTGGGCCCGCCCCCGCGCCTGTGTCACTGCGAGGCCGGGGGTGGAGAGCGGCCGGGCGGGACATCCGGCCCGGGTCCCTCGCCGCGCCCGCCGCCCGCCGCCCGCTTCGGCGC
FP013881	chr15	63189405	63189656	+	1	train	CAAGTCCCATGAAGTCCCGGGGCGCGGGCGGTGATGACGAAGCCCTCGCCGATTGGCCGCACGGGGCGGGGCGCGCAGGCCTTCTCCGAGAGGAGGGGCGGGGCGTGGGGCGGGGCGAGCAGCGGCGGATTGGCGGGCACGCCCCCTCGCCCGCGGCCCCCTCCCCGCCTCTCTCCACCGCCTCCTCTGGCTCCCCGGTCAGAGGGCCGGAGCGAGAAGATGGCGAAGACGTACGATTATCTCTTCAAGCT
FP007494	chr7	77122437	77122688	+	1	train	ATCCCAGGAGGCAATGCGCAGGGGAGCACGGGTAAGAGGGCAGGTGTTAGAATCGCTGGAAAGGCGGGCGCCCTTTGTTTCAAGACCATTGGCTATTGGAGGGGCGGTACATGGGAGGCCACTGGCCAATGGGAAGCACGGAAGGGGGCAACGAGGGTAAAGCTGCATGGTCACCTTGGATACCAAGGACGCGACTTCTTGTTTGGAGAGGGTGGAGCTTTGGAGTGAGACCCAGGAGGCCAAATCCCAAA
FP015267	chr17	8288469	8288720	+	1	train	GCTGGCGGGAGCATGCTCGTGACCACCGGGGGCGACATAGACCCTAACCCTAAGTCCAACCCCAAGGCAGACCCCACCGTCCCGCCAGGCGCACTCCTAAGAGCGCGGCTTCCGGAGCAAAGCCGCTGACCCCCGCCCCCAGGGTCTTGGCGCCGATTGGCCAACGAGAGCGCAGGGCGGAGCCAAATCTTAAAGGATCCGGGAGCTAAGCCAGACCCGGGTGGCGGTGGCAGCTGCGAAACCCAGGGAGC
FP007484	chr7	76358938	76359189	-	1	train	GAGTGGGGGCGCAGTGGGAAAGGACGGGGTGCCCGGAAGACGCGCAGTGAGCCCAGCTGGGGAGCGGGGAGTTCTAGCCGCGGCCGGGCTCGGGCGGGGCATGGGCGGGGCTTGCGGACGGGTGGGCGGGGCCTGCCTTCGTCATGCGGCGTGGGCGGGGCCGCAGGGGGTTGTCCGTCAGTGGCACGCACAGCAGCCGCAGCCGCCTCGCGCCCGGTCCCGCGGTCGCAGCTCCAGCCGCCTCCTCCGCG
FP016733	chr19	9538563	9538814	-	1	train	TTTCTGCCTGCAGCCCCCACGCTGAGCCCACCCCAAGGTCCTCCGCAGCCAAGCAGATGGAAAGTAGAATTTTTCTGCCTCGAGGAGGTATTACACGGCAGAAACCCTAGTCCCGAAACCCCGCCTGGCCCGAGGCGGAGAGCGTAAATGACGTCAGCGGTGCGCCGTTCCTTTTGTGACGCCGGCTGTGAGCGCCTGAGAGTCTTTTTGCCTTTCAGAGTTAAGGCCTCACTGGCCTGGGTGAGTTCCGA
FP007508	chr7	80369356	80369607	+	1	train	ATTTTCTAAATAAAACACTGCCACTATACATTGGAGTATGGGATATTATTTCCACTTTAAAGATAAGGAAACTGACTCAAATACACGGAAAACAGGATTTAGTAAACGTGATTTATTTGGAACGTGATCTCAGATCCACAGAGGTGAGAAAAACAGGAAAGGAGAAAGAAAAAGTGCTGATTGAGAGTGTGGTCTCAGCTGGAGACGCTCTTAACTCTGATTGCATGGAGACACTCAGGAGCACAAATTGC
FP001611	chr1	185156991	185157242	+	1	train	GCGTCCAATTGGGTAAGAGGGTTCGGTTTCTACGGTGCGTCTCCGGCCCACCGGCCAACCACCAAACCGAACAAAGACTACAGCACCCAGGATACACTGCGCAGTCTGGGGGCGGGGCCGGAGTTCAAGCCTCGGAGGAAGTAAATGCAAATCTAAGCGAGCGCTGTGGGGCGGTGAGTAGGTAGTGCGGATGCGGGCGCAGAACTACGTTTCCCAGCAGGCATACAGTTGGTGGGGCGGGGTGTGTGTGT
FP018530	chr21	45287842	45288093	-	1	train	CGGGTCTGCAATGGAACCTCTAGCCGCTGCGGTCAGGAGGGCGGAGACTCTGCGCACGCGCAGTTCCCCCTCCCGCCCCTGCTTGCCGGTGATGGCGCATAACGCATGCGCGGGGAGGGCGGAGCTGGGCGTTGCCGTGGCTACTGGGAACGCATTTCACGGGGGCGGGGCGTGGTTCCGGGGCGGGGCGCGGCCGCCGGAAGTGCGTGGCCGCCCGGGGCCATGGCGACACTCAGCTTCGTCTTCCTGCT
FP019809	chrY	20575524	20575775	+	1	train	TACTCCAGGGCCCCGCCCATTTCATCCTTGACTCCACCTTCTCCATGCTGAGTCCCGCCCCGCTTCCTGTTTATTCATCCTGCAGCAAACTCTCCGGTATCCTGATGGAGTCTACTAGGTGTCAGTCATTTGGCCGTGCCTCAGCCGAAGAGAGGCGGGGAAAAGCATCGTAATCAGCTGCGTCGCCTTTTGGTGACGCCAGAGAGTGCGCGTCAGCAGTTTATTAGAGAGCTCTGTAGCCAGCCTCTTCT
FP013563	chr14	103928255	103928506	+	1	train	GAAAGGCTCACCCTGGGCTCGCTCGGGGCGGCGGCTCCCGGCAGGCCTAGCGCGGCGGGGCGGGGCCGGCGGCACGAGGACCAGGCAGCGCGGGCAGCACGCGCCCGGGCCGGAGCCCGCCCACAAGGGCGCGCGGCGCGTTCTGATTGGAAGGCCTCGAGCGGGGGCGGGGCTTCCAGGGTGCGCTTCCGCCGTCGCCTGTTCCCGCCGCGGAGACCCGGCAGTTGGGGGATGCCGACGCCTGGGCCTTG
FP001664	chr1	201828991	201829242	+	1	train	GTTTTCACACAAATCCAGAGATCCTTGCAGAGAAAGGTGCAGTTCTCGTTCAGGCAGCAGGAGGCGCTGTCTCTGGAGGCTTAGATTCATTCCCCGCCCTCCGCCCGCTCCCGCCCGCCTCTGGTGCGCAGGCGCGGCTTCGCGGATTGGCCGCGCGCGGGGGCCGTCATTCGGTGGCGGGTCCCGGCCGCGGGGCTGGCGGGCTGAGGGGAGAAAAGATGGCGGCGGCGGCGGCAGCTGGTGCGGCCTCC
FP002609	chr2	109614163	109614414	+	1	train	GAACTGCGGAGCAGTGATTGCGGCCCTCGCGAGGCCGGAAGTGGGCGGGCCTTGAGGCGAGGCCGCCCTAGGTAGGTGTGGCCGAGACAGCGGGGAGCGGAAGTGGGCGTGGCCTGGTGGCGTGGGCGCGGGGCGGGGGTGCGGGGGGCGTGTCCGCCCGGGCGCGGCCCAGTGAGGCGGTGGCCGAGTCCTCTGGCCTCAGACGCGTAGGCTGGCAGCCCGCTGAGCCCGCCAGACTCCGCCGCCGTCGG
FP008964	chr9	71768841	71769092	-	1	train	GCTCTGCCCCGCCGGCAGTGCGCTCGGGCGCTGTTCCAGCGCGCGGCAGCGAGTGGCGCTGAGCCTCCGCTTTCGCAGCTCGCGGGGAGCGGGGCCGGCGGGAGGGGCCGGGCGGGGCCACCGCCTCCTCTGATGGCAGGCTCCTCCTCTCCCTCCGCCTCCCTCGCTCTCCACCGCCCGCCCGCCCGCCCGCCCAATCCACAGTTCCCGGCCCAGCCGAGCCGGGCTGCTTGTCTGGCCGGCGGAGGCTG
FP006961	chr6	137218654	137218905	-	1	train	ATAATATCGTCTTCTTACTGTACCTTTTTTTATGGTTAGATACGCAGATACTCACTAGTATGCTACGGTTGCCTGCAGTGTTCAGTGCAGTGACAGGCTGTACAGGTTCATGACCTAAGAGCAGTAGGCTGTGTCATCTGGGTTTGTCTGGGTTTCTACTGCCCGAAGAAATTGCCTAAGGGGACATTTCTCCATTTCTCAGAACGGATCTCCATCGTTAAGTGACGCATGACTGTGAGTGTCCTGATCAG
FP009276	chr9	122375093	122375344	+	1	train	TCCTCTCCGCCCACCCTGACACCTTGGGGCACCAGCGGCAGGCCCACTGTTGTCCTTCCTGGGAGAACTTTTCCTGCCCCAGAGCAGTGGTGTTTATGGAAAGAACGTGGCGGGGGAGGCGGGAGGGGGGAAGCTGGATGGGCCGGACTGGGAGGGAGGAGCCTCAGCTCCCGCACAGCCTCTCTTGGCAGGGAGGGTGTGGGCAGCCGTTCCAGCTTCAGCGTCTGGAGCTTGTGGCTCTTCTGCCTGCC
FP002337	chr2	64653862	64654113	-	1	train	CCCACTGCGCCCCGCCGCCCCTTCCTGCTCCTCCTGCGGCCGCGGCCGGCCGCGGCCGGCCCCGGCCTTTATCGCGCCTCCCGGGACCGCCGGGGAGAGGCGGGCCGGGGCGGGGCCGGCGGGGGCGGGGTCGGGCGGCCAGGGGCGGGGCCTGCGGCGCCGGGGCCCGGAGCTCGCGGGAGGCTCGCGGGCCGCACGTCACTCCTGCACGGCGAGTGCTGGAGCACGACGTACCGCTCGCTCGGTCAGGG
FP006388	chr6	32153357	32153608	+	1	train	AGGTGGGGCGCCTGGGGGTTTCGGTAGGGAGCCACCCACAGATAACTCAGACAGCCAGATTCTGGGGGTCGTTCAGGTTGAAAGACTGGTCGAAATTACGCGGGCATGAGTCAGCGCATCCCTACGCGCCCTCCGCCCCTTGAGGGTGGGTCGCTTATAGGGAGGGGAGTAGAGTAGGGCAGGAGAAACTGGGCCAGGCTGCACTTAGCTCAAGGGGCCTCGAGGACTCTCTGCGTCTCTGGAGACAAGGG
FP009058	chr9	92963824	92964075	+	1	train	ATCTGTTTGCAGGGGGCCTCTTCACCTTCTAACTTGTGGCCATTTCATTGTTCCCAACCAAGAGGGTCTGAGGAAGCCTCTGATGGTGTGTTTCTGGGCTCACTCTGGGCTTGTCTGGTCTAGGGCCGGCCCCCTCATTCCCTGCTGGGCCCCTCCTCCCTGAAGATCTACACAGCAGATCTACTCACAAAAGTCCACCCAGCAGAAGCAGGAGGATGGTGGTTGTGGGTCCTCTGTGAGCAGCCCTGTTA
FP003776	chr3	57896027	57896278	+	1	train	GTGGCACTTTTTTATAAGAACTGATCGTGGTTTTTTTTTAAGAGTTTTATTATTTGAGTACAAATCTTACTAAAGCAGCCTTCTTACTAGTGTTTTTATGTAACAGAGCCTTCTCCTTTAATCATCTTTATACAGTGTCTGCATTGTCCTGGGAATTGTACTATATTTGTTGCAGTTCTTTGAATACTTTAGCAAAGCTTGCTTCGTTTTTGTTTCCAGTTGGCAAGTGGACAGCCTTGAACAAAAGCCCC
FP019258	chrX	48911595	48911846	-	1	train	ATCGGGCCGTGTCCCCGGCCTCCCGATCCGACGGAATTTGGAAATCCCGGGGCTATTCATACATTGAGCTTTTAGGAGCGGAGGAGAAAAGCCACCACCCTGACGATCCCGGCTCTCGCTCCACCTTCACTCAGGTGGCCCGGCAGCGGAAGTGACGAACGCGGAAGTGGTTTTTCTGTTGCCGAGGGGACGGGCCGGGCAGATGCCAACATGGCAGCGGTTGGGGCTGGTGGTTCCACCGCGGCGCCCGG
FP007437	chr7	66995507	66995758	-	1	train	ATCCTGTCAGAGCTCTCAGCTCATTGGCGAAAGTAAATACGCCAAGGAAAAGCACCTCCCTTTTTGGGCGTGGAAAGATGGCGTAAAAAGCCACAATACGCAGGCGTCATCGCTCACTTTTCCCCTCCCGGCTTCTGCTCCACCTGACGCCTGCGCAGTAAGTAAGCCTGCCAGACACACTGTGACGGCTGCCTGAAGCTAGTGAGTCGCGGCGCCGCGCACTGGTGGTTGGGTCAGTGCCGCGCGCCGAT
FP006815	chr6	109094439	109094690	-	1	train	GCCGAGACTACGGCTGCCCGAGACTGCCGCCCTCCGACAGCGCCCAAGCCTCCGGCCGCTGCGGCCAGTGGGCCAGGGTTCGGCGAGCCTTCCCTGCCGGCGCCGGCCGGCTGCTGAGGCGCTGGGAGGAGCTGCGGAGGGCGGGGACTGGTGTCGGAGAAGGGGCGGGGTGGGAACGCCGGAGAGAACCCGGTGGCTGCACAGACAAAAAAGCCCCGAATGGCTGGAGGGCGTTCAGCTGTTAACAGCCT
FP015268	chr17	8310040	8310291	+	1	train	TGGAGGAGACAGACCCCAAGTCCACACTTCTCTCTTCAGTCCTACCTCCTGGCCCTCCCCTTTTCTCTCTCCTCTAACACACACACACACACACACACACACACACACACACACACACACGCCAGACACAATGGCTGGGTGGGGCTGAGGAACTGGCTTCCTCCGCAGAGATCTCAATCTGGGCCCTGAGTTCCCACCCCAGAATCACAGGAACACACACTTGGAAACCTCAGCCCTGCATTCCTCGCTCC
FP012681	chr13	29595585	29595836	-	1	train	CCCCGGCCGCAGCCTCTGGGCTGGCAGCCGCCGCCGCGCCGCGCTCCCATTGGTGCCCGGCGGTGACGCGGCCGAGCGGGCCGGGGCTGCCTGGTCCGGGGGCGGGCGTGGGGCGCGGGGCGCGGAGCGCGAGGGGCGGGGGCCGGGCGCACTGCTGATGAAACCTGGCGCCGGAACCCGCCAGCCCTCGGCGCCCATTCAGTCCGCGCAGGCAGGTGTGAGCAGCGGGTCAACTACCTGGCAGGCGCGCA
FP004882	chr4	94253094	94253345	+	1	train	TAAGCATAGATGGGTGGCCTTATCCTGTGTAAAGTCAAACATTACTTTCATTGTAAGCCAATAGTATTTCAAATAGTGTCATATGACCTAATTTTGAATGAATTAAGGGGTGATTTTCCTGAAAAAACCCAAGCTGATTGGCTGGGAATACTGTCACTCATTCCTTTGCATAGAAACTTCGTTCATTTCACTGCAGAAGAAGAAATTAGAGCTTACATTTAGGAAGGAGTTGTTTGCTAGCCTGTTCTGAT
FP002384	chr2	71068449	71068700	+	1	train	GCGGACCGCAGTCGCTCCACCTGGAGGAGACACCAGAAGGAAGACAGCCTGAGGGACGCAGCCATCCCCGGCTCCTACCGGCGCCCCGCCCCGCGCATGCGCACGCGCACAGGGAGTCAGCTGGCTGCGCGGGAGGTCACGGGAAGTGGGGCGGTGCCCAGACAGCTGGAGGGAAGGAGGTGTCAGGCGGGGAGAGACGCAAACGGCGGGACCAGCAGCGACGGTAGCAGCAGCATGGCCGCGATCTATGG
FP012265	chr12	91111466	91111717	-	1	train	TATGGTATTTGAGCTAGTTAAACACATATCTCTCTCCCATTCCATAGGGAATGAGCTGGGCTGTCCTTTCTCCCCACGTTCACCTGCACTTCGTTAGAGAGCAGTGTTCACATGCCACACCACAAGATCCCCACAATGACATAACTCCATTCAGAGACTGGCGTGACTGGGCTGGGTCTCCCCACCCCCCTTCAGCTCTTGTATCACTCAGAATCTGGCAGCCAGTTCCGTCCTGACAGAGTTCACAGCAT
FP012810	chr13	52011386	52011637	-	1	train	TGGGGAGTGGGCGAGGGTCCGAGGCCCACTCTCCCCTCACGCTCTCATCCCCGTGCCCCCAGGTCGGGAGGACGGCGGCGCGCAACTTTGAATCATCCGTGTGAAGAGGGCTGCGGCTTCCCCGGTCCCAAATGAAGGGGCGGTTCCCGGACCCCTGTTTGCTTTAGAGCCGAGCCGCGCCGATGCCCTCACACTCTGCGCCTCCTCTCCCGGGACTTTAACACCCCGCTCTCCTCCACCGACCAGGTGAC
FP009289	chr9	124257381	124257632	+	1	train	TTTACTACTTCCAGAAACCTCTGCACATTGGCTCTTGCTGGTCCCCCGGGGGCCACAGGAAGTGGGAGCTCCAGATGAAGAGTCGCTGGCACTTTGAGGGCAGGGGGCAGGGCCTTCCCCTCATCGGAATACTCTTGGAGCCTACTTCAGTGCCTGCATTCAGCTGGTGCACAATAAGTGCTTGTTACTGTGCCAATTTCAGTATGACTAGGGCTGTGGTAGGAGCAATGCTAGGCGATGGGGAATCCGAA
FP009524	chr9	137054006	137054257	-	1	train	CGGGAGCCGGGGGGCGGTGCTGGTGACTCAGCCGCGCCCGCGTCTCTCACCTGCGCGGGCCGGACGGCGGAAGCTTGGGAGCGGAGGCGAAACCTGTCCGCCCGTGGGCGTGGCCGGTGATGGGCGGGCCCGCCCGGCCTCCCGCCCCTCGTCCCGCCCCGGGGCCCCTACCCCGCGGTCCCGCGGCCCCGCCCGCCTCCGCCTCCGGCTCCCCGCACTCTCCGGGTCCACGCATCGTCCTCCCGCGCGCC
FP011197	chr11	78023217	78023468	-	1	train	CTTAAGGGGTCTGCTCCATTTCTCGGTCTCCAAAGTCAGCCCTTCTCAGGGCCTAACAACTTTTAGAAGCCCCAAATAGACAGACTGTCCATTTCTTTCTGCACTTGGCCCAGTCTCACTGCAAGCCACGCCTGGCTCAGCCCCGTCCCGCCTTGAGCCTCGGTGTTCCCACCTAGGGGCGGGCAGCCAGGGGCACTTCCGCTGGCCCAAGTGATCTGCATGTGGCAGGGCTGCGCAGTGGAGCGGCCAGT
FP019475	chrX	102599005	102599256	+	1	train	TGAAAATTAAGGGCTGGGAGGGAGGACTCGAAGGAACACAAAGTTTGTGGAATCGATTAGTATGTATCCTCCTATCTCAGTCTTGACTCTCTTTGCTGACTTTCCTACCCTGGGCCCCACCCCGAGACCGCCCATTTCAGTGGGCCTCTGTGTTTGGAGAACGAACTTCGAAAGACTGGAACACTTCCGCCTGTCAGGCCAGAAGAGATTTCCTGACACCACAGCTGGAAACCCATGTGCATTTCAGTTCA
FP002150	chr2	27663348	27663599	-	1	train	AAACCAGCTTTCGGGGTCTGACGCGATCCTTGCCTCAGGCCTCTCGAGGTCCAGACAGCCGCCCAGCCCGCTCTGCGACGCAGCAGTGAATAGTGTGGTACCTCCTTGTCTCGGTTCAGGTCCAGACCTCCCCGTCTTCCGGCTGCCCTGAACGTCAGGCGACCTCAGGACCCTGTGATTGGCGCCTGCGCCGGCGGACCGTGACCGAGGAAACCCCTGGAGGGACTTGGGCATTCCTTGGGCTCCGTGCC
FP008023	chr7	152676090	152676341	-	1	train	GTCCCCGCCACCCCGGAGCGATCTCCAAGGGGACGCGGGAGAGCGCCGCGGGGGACGCGGAAGTCTGACGTCACAGGAACTGGGGGCGGGGCGGGGAGGCCCGCACACCCTATTGCGCATGCTCCCGCCTCCCCGGCCGCGGCCTGGCGCAGTGCGCACGCGCGCGGGTGGGCGGGTTTGACTGGCCGTAGAGTCTGCGCAGTTGGTGAATGGCGTTGGTGGCGGGAAAGTTGAGTCTCTCCTGCGCCGAG
FP011279	chr11	100687651	100687902	+	1	train	GCGGCCGCCGCTGGCAGCGCCTGTGCCATGGGGCTGCCCACTCTGGAGTTCAGCGATTCCTACTTGGACAGCCCAGATTTCAGGGAGCGCTTGCAGTGTCACGAGATTGAGCTGGAGCGAACCAACAAGTTCATCAAGGAGCTCATTAAGGACGGCTCTCTGCTCATTGGGGCGTTGAGGAGTAAGTAGGGCTGGCGGGGGAGTGGACACCCGCATCTGGAGAGTCCCCGCGGGGTGCGGGTCCGAAGGGT
FP008346	chr8	58659280	58659531	-	1	train	CGGGGCCGGTCGTGGGGACGCCGGCCGGAGATCTGAGGGACGGGAGGCGGCGTGGGGGCCGGGCACGGAGGCAGGGGCCGGACGCGTGTCCAGGGCCTGGGCTGGGGGCTGTGGCGGAAACCTGAGGTCGGGGTGAGAGGCCGGGGGCCTGGTCTGGGCCGGAGCCTGCCGGGGACAGGAACTGGATGACGGGCCAGGCCAGATTCCACCGCGGAGCCGAGGCATGCCGAGCTATCCGGCAGTGCAGGTCC
FP002561	chr2	101474650	101474901	-	1	train	CGAGATTTTTCAAAACTCCTAGAGACAGTGGTTTTCAGCAACGGCGGGCACATTAGCATCACCCGGGACCCTGAAAAAGCCGGTTGGCCCAGGCGGCCCTCGCCCAGACCAATTCATCAGAACCTCTTGGGGTGGGGTGGGGCGGGGCTGGGATGGGGAGCGGCAGCCGGGGTATCAATATTTTTGAAAACTGCCTGAATAACTCCAACGCTCAAGCAAGTCAAGGACACCCACGGACTCAACACCGCGAC
FP014559	chr16	28984851	28985102	+	1	train	GGTCCTGGATATGGAGGCCACGGCTGCCAGCTGGCAGGTGGCTGTCCCCGTCTTGGGGGGGGCCAGCAGACCCTTGGTGAGTGCCTGGGGTGGCTCCCGGGCCTCCTCCTGCCCCCTCCCCACTGGCGGGTGGGGTGGGAAGGGGGCGGGTGCAGCCGGCTGAGAGCTTGATGATTTCCTGCCCTCGCCCGGCGCTCACCACAGCTTCCTGCCGCAGGCGGGCGGGAGGGCGGGCACGGAGAGGCGGGCGC
FP002427	chr2	74514249	74514500	+	1	train	GACTCAGCGGCCCATGCTCGGGCCCCTCCCGCGCTGAGCCTGGCTCCTAACACCCTGGCCCCCTGTCCCCCTCCCCGCACATCTCCGCCAGCCAGCCTTCCCCCTGCGCTTGGCGCAAACAGGCGTCCCTTCCCCTTTAACTGCTGGGCCCCGCCCCTGCCCTACCCCCAGCCCCCACTACCCACCCGGGCCGATCCGTCAGTCACTGCCCCAGCCGGAGCTGGCCAACCCTCTCCACCCGGGACTTGGGC
FP007832	chr7	129434232	129434483	+	1	train	CCGTCCGCCAGCAATGCCGCAGGACTAAAAAGATCCCCTCAAAAATCTCTTCATTGAGCCCCCACCTCCTCGAGTCCCGCTCCGGCCGGTCGAGCAGCCAATCGCCTCGCGGGGCGGGGTTGCGGCGAGCTGCCGTAACCAATAGAGGTGGAGGGGGCGGGGCCTGGCTCCCGGCGCGCGGCGGTAGGGTCGCCTCCGGCAAAGCGAGCTGAACCCTGAGGGGAGCCGCTGACCAGCAGCATGGAGGACCC
FP006728	chr6	83707816	83708067	-	1	train	GGGGATGGAGGAGGAGAGAGCCGCCGCGCGGCTGGAGGAGGCAAGGCCTGGGTGCTTGGAGGGTAGGGTCTGCGGATTAAAGCCGGACCGTCTCCCCTTGTTTCCTGCAGAAGAGGAGGCGGTAGACGCGACCACAGAAGATGTCGGGCCAAACGCTCACGGATCGGATCGCCGCCGCTCAGTACAGCGTTACAGGCTCTGCTGTAGCAAGAGCGGTCTGCAAAGCCACTACTCATGAAGTAATGGGCCCC
FP018682	chr22	28679761	28680012	-	1	train	ACCGCGCGCTCACCCGCCGAGTCCGACGGGCCCGGCGGGGGTGGGCGAGACACTGGGAACAGCGGCCAGCTCCAGAGGGCGCGAGGCGGGGCGCGCGGGGAGGGGGCGCGGCGAACGCGCGCGTGCGCGGTGCATGTGTGGGCCCGCGGGGGCGCGCGCGCGGTTTGGGGGCAGTGAGGGGCGCCGCGGCGGCGCGCAGCACGGCGGGAACATGGCGCGCGGAACCGGCGCGCGCGCCTAGCTGGCGGGAC
FP019240	chrX	48521628	48521879	+	1	train	TCCAAAACGCCCACTAACCTTGGCGCGTCCCAGTCTCCACCCCAAACCTAACTCAGGTGAAAATGGCGCCACGCGCGGGGGCGGGGCTAGAGGTACAAAGAGAGTGTCTATTGGTTGGGAGCGGCACGGGCCGCCTTCTTCGCTTCACCATTGGCTCGCTCCGTAAGGCAAGAGAACCCACTAGGGGATGAGCCCGAACTAGGGATGTGACAGAGCGCGAGACCCAGCCTAAAGAGAGCCCGGAGCCAGCG
FP014721	chr16	55955924	55956175	-	1	train	CTGAGGTCTCTTTCAGCTCTGATTCTGAGGTCCTCCCAGTCAATGATTTCTTTCTTTCTCTCCACGCTGGAGCTGCCCGTTGGTGGTAAGCAGAGACGTCATCAGCTTGCAAGCTGAGCTTCAGTGTCCCCTGAGAATGCAGCCAGCTGGCAACGGTAGAGGAAAGTGGTGAACTCATTTGACTTTTCTTTATCAGCTGTACCATGTACATGATGTGCAAGAGGACTGGGGCTCTGTGTCAAGGCCAGCTG
FP008618	chr8	132480518	132480769	-	1	train	GGTCCTGCGGCTGACCCGCGCGGGATGTGACCCCCTGACCCCCTGCCTGGCCTCCCCTGCCCCCCAGGGGGCCCGCCTTTGCCTGCTTTTGGGGGGGGGTGGGGAGGGGCGCGCGGATCATGGCATTGGAGTTCCCGGGCTTGCAGCCGCCGCCGCCGCCTCGTTCACGCACCCCGAGCGCCCCTTCTTCCCAGAGCAGCAGCGGAGAAGGCGAAGCGGCCGGCGGGGGCGAGGCAGATGGGGCTCAAGGC
FP018118	chr20	41618326	41618577	-	1	train	CAAATACGAGCACAATGGAGCGAGCCCATCACTCGCCCCTGCAAGCAAGCTTGAAATGGGCTTCTGTCTGAGCGGCAGCCGGGATGACCAATGGGTCTGTGTTTTGGACCGGCGAAGCCCAATGGCGCGACAGGGAAGGCGGGCCCGAGATGGGTTAGGGTGTCCTTGCAGGGTGTCTGAGCTAAACTTCACCAAATAATAGCTGTTTGTATTTTGGCTGCTGCAGGAGCCATTTTAGGTAAGTTCTTCTC
FP018398	chr21	33324194	33324445	+	1	train	TCTATTACTGAGGAATTTAATTAAATAATCATTCGAACCATGGGCTCGAATTGTTTACTGATAAAGCCCGACGACCAAAACAGCCTAGCGACTGCTGTTTGGAGCCCCTCGGCCCAGGGCCCGTGGCTGTTCTCTCCAAGGGACCATCTCGCCCCTCAGCCAAGTCGCCCGGAAAACGAGCGCTCGACCGCCTCTGCCCCGCTCTCGCTCTGCACACAGCAACGGTCTGGTCGCTCAGCCACTTCCTCCTT
FP016383	chr18	54224575	54224826	-	1	train	AGCTCCGCGCGCCCAGTGGGGGCCGGAAAAGCGCGGGAGGGGGCGTGGCCCCGAGAAGGCGGAGACAAGATGGCCGCCCATAGCGCTTGGAGGACCTAAGAGGCGGTGGCCGGGGCCACGCCCCGGGCAGGAGGGCCGCTCTGTGCGCGCCCGCTCTATGATGCTTGCGCGCGTCCCCCGCGCGCCGCGCTGCGGGCGGGGCGGGTCTCCGGGATTCCAAGGGCTCGGTTACGGAAGAAGCGCAGCGCCGG
FP010889	chr11	62771194	62771445	+	1	train	TAAAACAATACACACCTACTCGCAAACGTAGCAAGAGAGGTTGCTTTAAGAAGCGGGAATCTTCAGCGTCACGTGACCGCCGGTCTCTACAGTCTGTTCACACTGGCCAATCAAGAGCCAGGAAGCCTCCTGGAGGGCCGGAAACTTTCCAAGGCGCCCGCCGACTGGCTGTATTGGGGAGGGCGGGGCCGGGGCCCCGGGAGAGGGAATGAGTGTGAGCTCGTGAGTGGGCGCCGCCGCCACCGCCCCCG
FP009183	chr9	110668668	110668919	+	1	train	TCCAGCCCTACACTTGCATTACTGGCCTACGTGGATGCTCACCAGCTGGCTGTGTCCATCATTGCAGCCAAGTCTGAGAATTCCTGATTATACTTAGAGCCTGTGGAGCTGCTGACACAAACAGTCATTAGCAGACAACCCTTTTGCAACAAAGTATGCTTTAAAATGTAAACTGTGGAGCCATTTTCCTTGCGTTGTCCAGAAGGAACTTCGTCCTGCGTGAGCCTGGATTAATCATGAGAGAGCTCGTC
FP008563	chr8	115668950	115669201	-	1	train	CGCTCTCATTTGCGGCGCTCCGACTCGGGCGAAGTTCTCGCGCAGCCGGCTCGCCGCGGCCGGGGCCGGTCCCGGCGGGCGGCGGCGGCGGGTGCTCGGAGCCGGGCGGGGCGGCGGCGGCGGCGCGGGGACGCGCGCTGTCTCTTTAAGGGAGCTCGGTCTGGGGTGGCGCTTTCCTCCGCGAAGGCTCCTTTGATATTAATAGTGTTGGTGTCTTGAAACTGACGTAATGCGCGGAGACTGAGGTCCTG
FP000077	chr1	3611475	3611726	-	1	train	AGGGGCGGGGCTGGAAGCAGGAGCCGGGCTGGGGTGAGGGATACGGGCAAGTACAGGGGCCGTGCCGGGGGCGGGGCGCTGGTGGGGTAGGAGATGGGGAGGGGGAAGGGGCGGGGCCGAGGGCGCAGGGGGCGGGCGAGCCAGGGCGGGTCGGGACTGGGGTCCGGGCTGGGGGCGGGCCGGGGGCGTGGCGGCTTTCCAGGCTCTGAGTCCGGGGCACTGTGCTCAACGCGCCGGGGCGAGTTCCTGCA
FP006214	chr6	18155221	18155472	+	1	train	TCAGTTCCCGCCCCCTCTCCCGGCGCCCCGTCTCCACTCCCGCCCCGTACCTCGTTGCACAGACTCCACCCACTTCCCCAAGCGCCGGGGTTTGCGCAGCGGCGGAAATGGGCAGGGCGGAGCGAGCGCTGCGGCTAAAGCGAAGGCGGGGACCCTACCCATCCCTAGTCCTGTCGGCTCCTCCCACCCCGGGTCACGCCGTGACAGGGGCGGAAGCGGCGGCGGCGGCGGCGGCCGAGAAGAGGCTGGGG
FP019406	chrX	76427495	76427746	+	1	train	TATTTCCATTTTTACTCCCTAATGCATCTACTCTAAGCTCCTTGCTTCACCCTACCAAAGTTTCCCTTCTGCTAGTGCTCTCCTTTGGTGCAGAACCACCGCATCCACTTTCTCAGGCCCAGGGGAGGGGCGGGGCGGGGCGATTTCGCCTTGCCGCAGAGCCAGTCTGCGCCTGCGCGGACTGCGGCAGAGGGCGGGGTGTGAGAAGCAGAGTACAGTGAGACTAGCATTCACTGCTGGCCAGTGCCTGC
FP000056	chr1	1778373	1778624	-	1	train	CGCGATTCGCCCCTCTTCCCCACCGCCCGCCTCTTTCCCGCCACGCCCGCCGCCTCCCGCCCGCGCGGGGTGTCCCGGCGCCGGTGGGACCCGGCAGACGCGCGGGACGCCCCACGTGCGCGGCAGCGCGCATGCGTGGCGGGTGCGCCTGCGCAGTGGGCGGGCGCGGCGCATTGCTGCTCGGCGGCGCCGGCGCCGGGGTCCGGGCGGCCATGGGGCAACAGGCGGCCAGGGCGCCAAGGGCCAGGTAG
FP003509	chr3	38138460	38138711	+	1	train	TCCAGATCTCAAAAGGCAGATTCCTACTTCTTACGCCCCCCACATCACCCGCCTCGAGACCTCAAGGGTAGAGGTGGGCACCCCCGCCTCCGCACTTTTGCTCGGGGCTCCAGATTGTAGGGCAGGGCGGCGCTTCTCGGAAAGCGAAAGCCGGCGGGGCGGGGCGGGTGCCGCAGGAGAAAGAGGAAGCGCTGGCAGACAATGCGACCCGACCGCGCTGAGGCTCCAGGACCGCCCGCCATGGCTGCAGG
FP013819	chr15	52019037	52019288	+	1	train	GCCCCCAGGAGACGCCGGACGAGCCGCGTGCCTTTCCCACCGCCTCCGCGGCGCGCGCAGCCCAGTCACGGGCGCTGGAAATTCCGGCGCGTGCGGGTGCGCTGGGCAGGGCGGGGGCGGGAGAACCCGTGGAGTGAGCCTGTGACGTGAGGGGGCGTGGCACCGCGAAGCCCCGCCCCCTCTTCCTCGCCCTCTCTCGCGGGTCGGGGTTACATGGCGGCGACTGCGGCAAAGCGAGAGCCTCGGAGACG
FP006640	chr6	53064783	53065034	+	1	train	GGGGCCGAGGACCAAGTGTAAGTTCCGGTGCTCAAACCTGAGCGGCCGACTACAAATCCCAGAGTGCCTCGCGGGCGCCGTCTCCACGGCACTTGGGTTTCAGGGCCATGGAATCGCAAGCCCTTTTCGCACTGCATTATGGGATCTGTAGTGAGACATGCCTTGCGTTGGCTCTTTCCTCCTGCTGGGCGCCAAAGCGCGTCTTTTCCTCAATCTCCAGTCTGTCTGTGCTCTCAAAAACTTTAGTCGTT
FP001436	chr1	160862836	160863087	-	1	train	GTTGGAATGGAACAGTTGGGGATACTGGCTGTTTGGCCCCCATGGGGGTCTCTCCATATAGGATTCAGAGAATTGGAAGACTTCTTGGGCTGAAGCAGAGAACGGGATGCAATTTTGAAACTGGCTTTGTGAGTCACTGTGAACTTCAAGCACCCTCCCACTTGAGGTAAGCTTCACCAGTTTGGCTTGTGGGAACTGTCAGACCAGCTCCAGGCGCTGGGGCTTTCTCAGTGGCCTTGTCAGCTCACAGC
FP006460	chr6	34757329	34757580	+	1	train	ATTGAACCTCGTAGCGGCCCGTAGTACTCCTCCTTCCCGCCATCCTATAGATGAAAGGAAGAAAAGGAAAAGCCCCGCCCCTCGCTCGGCTGCTGGAGGCGAGGGCTTCGGAAGTCTTCATGCTAGTCTCGTGGGGTTCCGCGGTGTCGTCGCTGGCTGTGCGCGTCATTTCCGGGCGTCACGTAACGGAGTGGCCAACGGCCTGCAGAGCAACATGCCCAAGTGAGTGGGGCCCCGAAATCTGAGGGTGA
FP008271	chr8	38386931	38387182	+	1	train	TCCCCCAGACCCCGCCGCATGCAGGCCGCGCACACATCAGGCGCGCCGAAGCCGCCGGGCGAGGAAGCCGAGGGCGCGTTTTGCTCCCGTGTGGTGATGGGCAAAGCCGAGGGAGCCCGGGAGCAGCGGGCGTGGGGCGGCGCCAATGCGAGTGCGAGTGGTGTCCGCCGCCCGGGAGCGCCCGGGCCCGAGCGGATTAACCGCCGCCTCAGGTGTCGTCTCCTGCCCGCGAAACACCACCCACCGTTAGT
FP006077	chr5	178941010	178941261	+	1	train	TCTCTGGGAAAGGTAGTTTCCGCGCCTAAAGCGCCGGGGCCGGGCCCTGTTCCCGCAGGCGCAAGGCAGCACTGTCTCCGCCGATTGTCCCTCTGGGAAGTGGAGTCTGGCCGGCGGAACTGCAGCAGCCCTGAGACTTGTGGGAATTCGGCCCAAGGGTTCCCAGGGCAACGCGCAAGCGCAGTTCGGCTCCCGGCTGCAGACTCCAGCTCATTGTGTTCTGACTGCGATGTGGCGCTTGCGATCTCTCG
FP002386	chr2	71130124	71130375	+	1	train	CGCCGTTCCCACGCTCCCTGTCCCATGGCGAAGGTCGCCGGTGCCCGGTATTCACCTACGGCATTCGCGGCTGCAGCCTTCAGCACCCGCGCCATTTTGGAAAGCAACCCGCCACGTCAAGACTAGCCACGGAGGTTGAGAGACGCGCGGCAGAGGGCGGGGCGGGAGGCAGAGGGGGCGGGGACAGAAAGGTCAGGGGCGTGGCGAAGGTCGCTGCTTTCCTGCTACAACGGGCCTACTTTAGCTAAAAC
FP002529	chr2	98444657	98444908	+	1	train	GAAGCAGTCACAGGATAGGGTGGGATCTTGTGGGAACCTAGCAGTAGGAATTGGCGCGAGCAGCTCGGTCACCGGCCCTACTGTCTCAGAGGGCGGACCCAGCTCGGGCTGGTAGTGGGGCGGGGGTCGCAGGGCGCGGGCCGGGCCCTCAGCGCGCATGCGCGACAAGGGGGCGGGGCCAGGGCGGGGCTGCGCCCGGCGTCTAGAGCGGCGGCGGCTGGCTAGGGCTGCGGCGCGCGTGGAGGGTTCGC
FP014059	chr15	78811413	78811664	-	1	train	GACAGGGGGGCGAGGGGGGCGGACATCCGTTGCGGGGCGCTGCCTGCGCCGGGCCGGCCAGCCCTTGAGGCGAGCACGGCCTGCGGGGCCGATGAGGCCGGGGCCGGCCGGGGCGGGGCGGGGCCAGGGGCGGGGGAAGGAGTCCGAGGGGCGGGGCGGGGGCCGGCAGCCGCCCTCCCTCACTCTCCCTCCCTTTCTTTCTTTCTTTCTTTCTCTCCTTCTAGCTCTCTTGTCTTTTCTTTGCCTCTGCA
FP010272	chr10	119207489	119207740	+	1	train	GGCGCGCCTGGGCCGGCGCTTGCGCAGTGCGGCCGCCCGAGCCTGCTAGTACCACACCCCCGGGAGGGACTGAGGGGAGGCAGAAGCATCCGAGGCATTAAAGCATCCGAGGGAGCCGGAGGGGAGGAGAATGGAGTGACAGAGACACGCGGAGGGTGGGGGGTGGGGGGGAGCGTGTTGAGGGAGGGGGGAGGGGGGACACAGAGGGAGGAAGAAGCGGCGGCGGCGGCGGCGGCGGCGGCGGCTCCTCT
FP016509	chr19	1039926	1040177	+	1	train	GATGCAGCAAGAGCCGCGCGGTCCCTTTAAGAAACCCGGCTAGGCGAGGCCCTTCTGTGATCCCGTCTCCTCCCTTGGCCCGCGCAGCTCCGACGGAGCAGGCCAGTGAGTGACGGGCAGGTCGCCCAATAGCAGCGTGCAGAGGCAGGGGCGTGCCCCGGCGCTGCTACCTGCGCGGGCAAGCTCAGCGCACTTGGCTTAAGGGGCGGCGCGCTCCCTGCCTGCTGCTGGGCGGAGGGAAGGCGGCAAGA
FP013833	chr15	55270438	55270689	-	1	train	GGCATTGTATTTGTAAGCAACAATATGAATATAACCTTTAAGGGCCTTGAGAATACAAGGCACTTTCTCAAGAAAACAAATGTTCTACTGTTTTGATTCACGGGGTAGGTGGGCGGAAGAAAGAGCTGAAGTGCAGAGTCTTCTAACACCAGAAGAGGTTAATAAAGCCGCCCTCTGAGTGGCCAGTCGAACTGGAACTTGGTAAGCTTGAAAGGGGCAAAAATCACTTGACTGGTCTGCAGTGCCATATG
FP016538	chr19	1650233	1650484	-	1	train	GACCTAGACTCTGGAATTCCAGCTTCCCTGCGCCTGCTCCCAGGGCTCCCCGTGCCCCCAGGGCTCTGCTGGAACTCAGACCCCAGGGCGGTTGAGGGTTTTTTAGGTGGCTTTTAGCACTCTGTGCTTTGACCACTCACAACTCTTCTCCCCTCCCTGTTTCTCCCTGTCCATCTGTCCCCACATCCACCAAGCAGGGTTTCCAGGCCTGAGGTGCCCGCCCTGGCCCCAGGAGAATGAACCAGCCGCAG
FP006752	chr6	89117953	89118204	-	1	train	CAAATCCCGTCGCACTCCAGGACCCCCCATACCCGGATCCCATAGGGCCTGGGAACGGCCGCGCTCCTGTCTTCGGAATTCCCGCGGCCCCTGGGCCGCGCCGGCGCCGCCCCCGCCCCCGGCCCGTGCGTCCAGGCTGGCGCCGCGCTGCGCCCCCTCTGCGCCGGCCTCTGCGGCTGCGCGCTGGCGGGGCTGAGGCCGAGGGGTGGGGGTCGCGCCGGGGGCCGGCGGAGCTCCTGTGGTGGTAGCAG
FP002286	chr2	55049104	55049355	-	1	train	ACCCAGATATTTTTAGGAGGTGCTTTTGTCTACCCGTCTGGGTTGCTCCTTTGGAAAAGGTGCAGTTTTGCTGCGGGTCTGTTGGCCTGCATTCTGCCCCGGGCTCCCTCCTCACCTCCTGGCTGGCGTGGGTGGCCCCTTTGCGCTTCCCCTCCCCCTCCTCCTGTGCTCGTGGGGAGCTGGCCGAGTGGAAAACGTCCACATTGACCCAACCGCAGTGAAGGGAGAGCGCCTCCAGTGGTGTGGTGCGT
FP004851	chr4	86892312	86892563	-	1	train	AGAGCGCGGCCTGCCCCGGCCCGGTTACCAGAGAGCGGACCCCGCGGGGCGCGTCTCCCGGGCCCGGCGTCCCCTCCCTGGCGGGTGGGGAGGGCCCCACAAAGCCCTTTGTTGAGGCTCCCGTGTGACGCGAGGAGCCCCAGACTCGGCGCCCGCGGCGAGCGGGAAGGAGGCGTGTGCCTGCGGCGCGGCCCCGAGGCGTGTGCGCGGCCATGTGCGCTTAGCGCAGCGCGCTCTTACCTCAGTACCCC
FP014476	chr16	19885438	19885689	-	1	train	GCTGTGCCCTGACCGTCTGCGGAGCGCACGGCCAGTGGCCCGGGCCGCTGAGTGTGTGAGCCCTGAAGGGCAGTGGACATGTGCGTACGTGGGTGATGTGTGCGGTAGGGGTGGTGGCGATCCTGCGTAGGGACCCACGTGAGGGGTGCACCGACTGGACGTGTGTGTTTCGAGGGGCCAAGGAAGCGGGTGTTGGTCCCAGTGTGCGTAAACGGACAACGACATCTGGGAGTTGGGCAGTGAGGAGAGAG
FP011455	chr11	124673688	124673939	-	1	train	GAGCTGGTGCCGCCGGTTCCGGTTGTTTTTCTAGTTGCTGGGTAACCGTTTTTTCTTTAAAAAAAAAAAAAAACTACGGCGGCCGAGAAGGGGCGAGTACAGCCCAGTCCTGAGGTCGCGGGAGGCTGCGAGGACTGCAAAAGGGTGGAGTCGGCCTCGCCCCCGCCCAGGCCCCGCCCCTGCCGGGAACCCACTTTCCCAGTCCTAGGCGGCGGTCAGATCCTTGCAAGCATGGTCGCGCCGGGGCTTGT
FP001612	chr1	185311120	185311371	-	1	train	CAGTCTTTGATTATTCTGAATATCTAACTTAAATTTATCATTGTTATAGATTATGATGCATCATTATCACAGAAGAAATTCGTGTCTATAGCTTTTAAGGACTTGATTACATCATTTTCAAGCCTGATAGTTTTGGAATCACCATTAGAGCTTAAGACACACCTGCCTTCATTTCAACCACCTGTCTTCATACCCTGACGAAGTGCACCTTTTAACACTCCTTTGTCCTTGGATTACTTAAGAGTTCCCAG
FP008100	chr8	17027060	17027311	+	1	train	ATCCGCCGCCTCTTCCATGATCTTCCCGGGCCGAACCACGGGACCGCTACGCTGAAGGTGGCGTCGCGGGTCCCCGGGGCCGCGCGAGTGTAGGGGTCGCTCTCGGCCGGCCGCGAAGCTCGCGGCACCGACTTCTCGCGAGATTTCGGCGACCCCCCCCCCCGCCCCCGCCCCTCCGTTCTCTGCCCCCTCCCAGCTCTGGTGTGGGCGGCCTCCGCTATGGCTGCGCTGCGAAGGCTCTTGTGGCCGCC
FP007693	chr7	103074517	103074768	-	1	train	GGCCTCGCTACGCAATGCCAACCCGAGGCTGAAGAACTACTTCAAGGAGAACTACATTCCTCAGGTCTGCGAGGTACTGGACACGCGTGGCAGCGGACACAGACCGTCGGTGGGGAGTTAGAAGGGGGAGGAGTCAGGAAAGACTTCCCGGAGGAGGGACTAGGTTTCGGGATTGGGGAGGAGTGGAGGGAAGGCGGGGCAGAATGCGAGAAAGTGAAGTCAAATTCGGACTTGTAGGCGATCAGGGCTGA
FP017394	chr19	44954809	44955060	+	1	train	ATTGGAACTCTAAAAACACAGAAAGGATATGAGCTCAGGAGACTTGAACGAAAGAGTTGTCTTAGGAAGAGGGGTCAGAAGACAAGGGTGCTGGGCAAAAGGCTAGTTCTTCGAATCAGAATACGGTGGCCAGAAAGGTTCCTCTGACGAAAGAGGGGCGTGGTTCGAAGCCGTACAGTGGCCGGTAAAGCTCAAGAAACATGGAACGAAAAGGACGCGAAGAATCGGCAGGGAGAAGCGGACAGGATGTC
FP014431	chr16	14632739	14632990	+	1	train	GTACTCGGTCTCTGGGTGAAGAGGAAGGGGCGGGACCGGCGGTATTACGCATGCGCCCACTTCCTCCGGCTGGGAGTGGCCGCTCTAGGCAGCGGGGAGGTCGCGGGGTTGAGGGGGGTTGTGAAAGGAGAGCGGCCTCTCCTCTATGGTCACGGGGCCGGGGCACGCTTCCCCCACTCTGTCTTGTTACTTCCGGTAGCGAAGCCTCTCCCTCTTCCTCTGCTCCCGCGGGGTCTGTGCTGAGAATAATG
FP010030	chr10	92290949	92291200	-	1	train	GTTGCTGCCGGCGGAGGAGCTACCTTTGTGAGGTAAGAGTCGCCGCCTCCGTGCGCGCCAGGGTTACGACGCAGGGCGGCCCAGAGCAGCCTGGGAGATGTAGTCTTCGGACCGTCTCCACTAGAGGCTGCACAGCCCGCCCGGACCAACGCCTCCGCGCCTGCCGCCGTCTGGCCGGCGCCCTCCCGTTTCCTTCCTCCAGGGGGAAAAAATGTCTTTCGGGCCGCTTTCCAGGAAGCAGCAGGATGCGC
FP003550	chr3	43106024	43106275	-	1	train	TTATCAGAACTGCCATAACTGTGTAATCCTAACACTAAGTAAAACTCAGCCTCTCGGCGCAAATTATCACTCCCAACATTCCGCAGGCTGGTGGCGGCGCGGGCTGTCCTCGTACGCAAAACTACACCTCCCAGGATTCCGCGAGCCAAGGTTGTGGCCGGGGCCAGGGCAACCGAACTCGCGGCTCTGCGCAGGCGCACAGGCTCCGCGCCGGGCTCGCCCGCCGGGCAGGCGCACGGCGGACACCTGCG
FP000459	chr1	32362041	32362292	-	1	train	GCGGCGGGTTGCCCTGGCCTGCTGCGCCCCCCTCGGGTGTCCGGGGTCTGCGCGACCCCGTCTCCAGGCGCCATGGCCAGAGGCCTCCAGTCCCCGATTCTGGCCCGCGCTGAGTGCCAGGGGCGGCAGGGGGCACCGTTGGGGGTTCTCCAGTGGGAGGAACGTGCTCTGTGAGGTCAGCGCGCCGCGGCCCGGGTCACAATGCTGCCCTCCTCGACGCCCGGGCCCGGGCACGCCACAGAGACCTGCCC
FP002686	chr2	128318521	128318772	-	1	train	CGGAGTCGGGCGTCGGGCCCGGGGAGCCGGGGCCGGGCGGGGAGCAGCCATGGCACCGCGGAGGGTCGGGGCCCGCGCCGCTGCGGGCGGCCTGGGCGCCGAGTAGCCGGGCCGGGCCGGAGCGCGGGCGGCGGCGGAGGCAGCTGCGCCCGCGCCTCCTGCCCTCCCAGGCCCCGCGCCCCGCGCCCGGGCCCCGGCGATGGTGACACATGCGGCGGCGGCGCGCCGGCGGCAGGACCATGGTTGAGCGC
FP018175	chr20	46376702	46376953	-	1	train	CCACGGTGACTACTAAGCACTTAAAATGTGGCTGGTGCAACTAAGGAACCAAATTTTCAACTTAAATTTGTTTTAACTGATTTAAATGCAAATAGCTGCATATGGCTAGTGGCTGCTATACTGGACAGCACAGCCCTAGAGACAGAGCTACTGAGTAGGGAAATTGAAGGACATAGTGGTGGTACATGTGGAAAATGGAGGAGAGCCTCATTTGGGCAAGAAAGAACCAAGAGGGACAGGATTCATAATGG
FP003488	chr3	33828655	33828906	+	1	train	AATTTAATTCATAGATTTCAATCCACACAAAATCATGTCGTCTTCTCTGTTTACACCTAATGACTAACCTTAATCTCTAAACCATTAATGGGGTGATTCTAATTTCTGTCTTCTTTTCCTTTTTCTTCCTGCATCCCATGTTGTCTGTGGTGGTTTGTGTGGTTGGACTCTCCCCTGGTCAGTATTTTTATTTCCAGGAGGTGTTCCCTGTCTTGGCTGCAAAGCACTGTATCATGCAGGCCAATGCTGAG
FP015868	chr17	61780869	61781120	-	1	train	GGAAGGTACCAGCTCTTTCAAATGAGTTTTAATGTAATCAGTTTTTATATTGTATGTAGCTGGGTCATAGGTTTTAAAAGTTTTGCCACTTAATTAGCACTTTCTTTGCATCTAGATTTGCAGATGATTATAAAATTGCGATTCAACAGACTTACTCCTGGACAAATCAGATTGATATTTCAGACAAAAATGGGTTGTTGGTTCTACCAAAAAATAAGAAACGTTCACGACAGAAAACTGCAGTTCATGTG
FP005626	chr5	119268597	119268848	+	1	train	CTGCTGTCCGGTCCCCAGCTCCTCCTCACCCACTCCGCCCCGCCATGCCCCGCCCCAGCCGCTCGGCTCCTCCTTCCCTGCACGCTCCGCCCCCGACCTCCCGGCTCCTCCCCGGCCTGCCCGCCACACCCTTGGTCCCTCCTGCAAGTTCCGCCCCTGGCCTCTGCCTCCTTTTCTCCCGCCGGCTCTAACCCGCGCTTGGCTAAGGTCCGCGGGAACCCGTGAGCCACCGAGAGAGCAGAGAACTCGGC
FP008591	chr8	123396396	123396647	-	1	train	GCGTTTTCCCCAGGACTCTCCGCTCAGGTCCTCTATACTGTCGGCCTGTGTTCCTCTCCACGTTGGCCCGGCCTCCTTCGCTCTACAAATGGGAGGAGAGTTTCGAGCCTGCTGTTCTACGGTGGTGGCGTGGGCGGGACAGACGGCGGAGGGATTCTGGGCGCGCGCGCTGCGGCCGCTGGCGCGGAGCTTGTGGCGCCAGAATTCGGAGCGCGGAAGAGCCAGAGCTGCGAGCGCCTGGAGCTGGATCT
FP005426	chr5	67004674	67004925	+	1	train	CGCACGGCTCAACTCATGTAATTACTGTTTATAGCTGGCCGAGCCTGACTAGGAGAGGGCAGACCCGAGAGGAAATCAGTTTCCCGGACCTTTGAGAGGAGGCTGTGTGTTAATTAAAGGCTAGGACGGGACGGGTACTTCTCAGACATGCTCCAAGTTGTTCTTGAGATCACAGTTCCCATCACATTTTCTCTGGAGGGAGTGAGTAGATAATTGGGATTTTTTTTTTATTTTTGGCCTTGTCTTTCTTC
FP001604	chr1	184051532	184051783	+	1	train	TTTGCTCAGATTTGAATAAACGCAGAGCGAAACTAGAGAGCGTTCGCGCAGGTTCTAGCTACCTTGGTCAGAAGGAGGGGCTGATGCAACTTTCCTCCCAGTTTCTGAGCCCTCTCGCACTGCAGCTGCGAGCCTCGGGGAAGCGGAACCCACAGGCGCGCGCGCCGCTGCTTCTGGCCGGGCGCGGGTCGTGGTGCACCACGGGAGCGCCGCACCGGCCGGCATGGAGGAGCGCGGCGATTCCGAGCCGA
FP008872	chr9	34651984	34652235	+	1	train	TCCAGACTCTGTCCCAAGCAGCCTCTTCAGTCCGGGACGGGACAGGAAGTGTCCGGCGGAGGTGGGGGTGGAGGTGATTGGCAGTTGGAGTCTCCAATGGGCCAGGAGCCCCCCTATTTCACTCCTCTCCTTCCATTGGCGCCTGGGCGGAGCCACCCCCGCTGCGGGGGCCCAGGGCGGGAGGGAGTAGAGGCCGGGGCAGAGGGCGAGGGCGAGGGCAGAGGGCGCTGGCGGCAGCGGCCGCGGAAGGT
FP008909	chr9	36136534	36136785	+	1	train	GCGGCCGCATCCGGGGAGCCCTCGGCAGTCGCGGAGCTGCTTCCCAGGGGTGAGCTCTCCGGGAGGCCCACGGGGTGGCCCCGGGCGCCTCCGCCCTCAGCTGTCGCTCCTTATAAGGCGGGGGCCGGGCGGCGCCGGAGGAGGGGCCGCGGCATCAGCCCGGTCCTGGGCTCTGGGGCGGCGCTGGGCCGGGCGAGCGCAGTGCAGCGCAGCCGCGGGGAGCGAGGAGCGCGCGGAGCCGGCCATGGGCA
FP015686	chr17	44807782	44808033	-	1	train	AATCCAGTACAGATGCAGGTGTTTTCTTTCTGTGACTTTTTGTGAGTAGTTGTATCTGTAAATCACCTCCCATGGAGACGGGTGTTCCTGTGACTGAGAGCCAGCTGTTGGGTGTTTTAGCTAAACAGCTTGCTCCTAGTTAGCTGCTCAAATCCCCCTGAGAGGTGTATCGGCTGCTTCAGGGAATAGAGCAGATGGGGTGAAGGTTTTTGCAAGGCCTTTAGTGACCTGCTTTCAATTTGCTGCAATTC
FP013305	chr14	67701525	67701776	+	1	train	TGATTCTCCCGCCTCAGCCTCCCGAGTAGCTGGGACTACAGGCATGCGCCACCATACCTGGCTACTTTTTGTATTTTTAGTAGAGATGAGGTTTCACCATTTTGGCTGTGCTGGTCTCGAACTCCTGACCTTGTGATCCGCCCACTTCAGCCTCCCAAAGTGCTGGGATTACAGGCATGAGTCACCACGCCAGGCCTATTATGATTTCTGTAGACTATTTATGACATGTTTGAACTTTTGTTTTGCCCTAC
FP018446	chr21	37420114	37420365	+	1	train	ATAGTTGATTTTGATTATTGAAGACTAAGGAAAATTTAAGTTTGAATGTTAGAAAATGAATTTTGCAAATTTGAAGTAATTTTATTCTTTATGTTTTGGGATTGTAGTTATGTTAGATATTCCTCAGTTGGGGTAATTGTCTTGCATCATTATCTCTTATCATAATCTGTTTTTCTTCACACAGTGTTATAGTTTTGCCGCTGGACTCTTCCCTCCCTTCCCCCACCCCATCAGGATGATATGAGACTTGA
FP014007	chr15	74781925	74782176	+	1	train	TCAACGCAGCCAGGCGGCAAGTGGGGGCAGGGCGTGGGCGCGGACACACCCCCCGCAGTGCCCCCGGCGCCCGGCAGGGGGCGCCGCCCCGCCGGCCCACGCCCCGCCGCGCGCCGGGCAGGCCCCCTCGCGCAGGCGCGCTGCCGCGGGCGGAGGATCCGGGCCGCGCTTCCTCTCGCCAGGCCTGCGAGCTTCCTCCCAGCGGAGCCCTGGGCGAGCCGAGGTTGGCCGCCGCCGCCGCCGAGCCCGCT
FP012508	chr12	121444072	121444323	-	1	train	AGTCCAAAGGTATACAGTTGGTCTGTTCATTCTCTCTTTTTGGCCTACAAGCAAAAGCGTGGCCCTGGCTTTAAGTACGCCTCCAACCTGCCCGGCTCCCTGCTCAAGGAGCAGAAGATGAACCGGGACAACAAGGAAGGGCAGGAACCTGCCAAGCGGAGGAGTGAGTGTGAGGAGGCGCCCCGGCGCAGGTCGGATGAGCACTCGAAGAAGGTGCCGCCGGACGGCCTTCTGCGCAGAAAGTCTGACGA
FP013615	chr15	26772983	26773234	-	1	train	GCGTGGGGGAGGGCAGGGTGCGCGGCGCGGCGGCTCCGCCTCCGACTCTCCGGCCTCCTCCCCCGCCCCGCGCCCGCGCGCGCTCCTCCCTCCCCGCTCCTCCTTCCTCTCCCAGCGCCCGCTCCTCCCCCTCCTCCCCTCCCCCTCCGCCCTCCTCCGCTCCGGGCCAGCGCGGCGGCGGCGGCGGCGGCGGCAGCAGCAGGAGCAGCCCCGGCTGCGGGTCGCGACGGCGGCGGGGCGCCCCCTCCCCC
FP019021	chr22	50603127	50603378	+	1	train	GACTGATGCTCCCCGATTTCCCTTCTCCTAGCACCTGGGCCCCTGGGCTACTGCTGCCCTCTGGCCCAGCCCTGCTATCTCCCTCCGTCCTGCAGGACAGCCTCTCCCTGGGGCGCTCGGAGCAGCCGCACCCCATCTGCTCCTTCCAGGATGACTTCCAGGAGTTTGAGATGATCGATGACAATGAAGAGGAGGACGATGAGGACGAGGAAGAGGAGGAGGAGGAGGAGGAGGGAGATGGGGAAGGCCAG
FP004055	chr3	128052230	128052481	+	1	train	TCCCGGCGGGCCCCGCGCGCCAGGCCCCTTCCCGACAGGCCCCAGGAGCCCCGCTGCATGCCGGGGCTTGAGGTCTCGGCGAAGCGGCGCGGCGCGGCGAAGCGCGGCGGCCGGCGGGCGCGCGTGGCAGGAAGCGGAAGCGATCCGAGGCCCGGCCCCGGCCCCGCCCCGCGCCGCGCCGCGCCGCTTGCCGCCGGGCTAGCACTGACGTGTCTCTCGGCGGAGCTGCTGTGCAGTGGAACGCGCTGGGC
FP010291	chr10	121988962	121989213	+	1	train	CTTCCTGATAAGATCTCAGGAGTTGGGCAAGTGGGCTCAAGCATGGGCCCTAAGGGGCAAAATGTCAGAGTTTAACTAATATATGACCTTCCTCTAGGAATTCTCGACTGGCAAGGGAAAAATGCCTCAAATGAGCACGCGCACAACTTCAGTAAACACACTGTGAGTGCGGCCACTCCCAAGTGCTGGCAGACCACTGCATAAGTGGACAGCCTGCTCCAAGGGAAGGATCAGGAGAGAAGAAACGCAAA
FP016491	chr19	639678	639929	+	1	train	CGCGGAGTCCCGCACGGCCGCGCCCCTGTGCACCTGGCCCCCGCCCCCGAGACGTCCCATTGGCCGGCGCCCTAGCCTGGTCCCGCCCAAGTGGACCCCGCCCCCGCCCCGAGGCACCCCATTGGCCGGCGTCCCCGCCCCAGCGAACCCGGCCCCGCCCCCGAGGCGCCCCATTGGCCCCGCCGCGCGAAGGCAGAGCCGCGGACGCCCGGGAGCGACGAGCGCGCAGCGAACCGGGTGCCGGGTCATGC
FP019386	chrX	71618460	71618711	-	1	train	CCTGCCCCCAGCCAGTCATCCTCTGCCCAGCTTTTCTGGTCCCAAGCTTGTAGCCAGCTCCTCCAGAGAGGTTTCACCCAGTCACGGAAACTCTGTGGCCTGAGGTTTAGGGAGGTCTGGTAGAGGTAACTCCCTGGAAGAGGCTGCTGCTGAGAACTGCCTGAAACTCCCACTTCCTCTGTGACTGCAGGTTTCCAACCACAAGCACCAAAGCAGAGGGGCAGGCAGCACACCACCCAGCAGCCAGAGCA
FP015758	chr17	48626307	48626558	-	1	train	GACAGCGAGAGCGCAGCCCCCTCGCCCGATTGATTTATGTCGGAGCTGACGCTTTATCAGGCAGTCGGAAAAACTTTGACCAATCATTTTGCAAGGAGAGCTGAGACGGGCTGCTCCACTGTACTTTGTTGGCTGAGAAGTTGAGCAGGGGGTGGGGGTGGGAGGGTGGGGGGCTGGGGGGGTCGCGTCCGAAAGCCCTCACACCGGTCCGGGTGCCACCTCTCCCTGCTTGGGCGCCGCCGCGCGAGCGC
FP015581	chr17	40364656	40364907	-	1	train	AGGATGTGGCTCTGGTTACCCCCCTCCCCCATTTCCAGATGGAAATTTAGCTAAAGAGTGAGGGCCACAGAGCCCAGCAAGGGGCGTGGTCCGTGCCTGTTCCCAAGGGCGGAGCCAGCCAGCAGGGTCACCCTGTTCCTTTAAGAGAAGGGGGAGCGCTAGCCCCTCTCAGCCATCCTGTCCTGTCCTCCATCCTGGCCAAATCCGAAAGGGAAAATGAAAGAGGGAGCAGGAGGCGCCGGTCCCAGCCA
FP009257	chr9	120929103	120929354	-	1	train	GAGCTGCGTACACGCGTGTCAGTTAACTGGGCGCACAGTGCCCGTGGCGCCGCAGACCCGGCCCTCAGACTCAGTTTCCCCTCACAGGGATCGCTCGCCCCGCCCCGCGGCGGTCTGGGGAGCAGCAGCCCCGGGCCCCGGCCCTGCCTTGCCGGTTTCCGCCGGTGCGGCTGCGTCCTTCACACTCCACCGGAAGCTCCGGGCCTGGCCGCCGGTAGGCGGTGGCGGAGGCGGCGCCGAGGTTGGCTGCC
FP010779	chr11	47578918	47579169	-	1	train	CCCGTGCTCACCCCTGGTCAGCGCCGAGGCCCCCAAGATCCCGCGCCACCACAGCCTGGCTACCGCCGCCGCCGCCATGTTACTCAGATGCAGACTAGGCAGCGGACGGAAAGACAGCAGGACACGGAGCCCCAAGGGCACGGACCGGAAACGGAAGTGTGCAGCAGATCTTCTTCCGGGCGGACGTGGAGCCGGAAGCGGAGGTTCCGGGCTCCGGGATGAAAGGAGGGAACGCAGGTGAGAAAGCGAGA
FP007681	chr7	102433374	102433625	+	1	train	CCCCCCACTCTCCGAGGAGGCGCCGCCAGCCCCGCCCCTCCCGGCCCGGCGGGTGACGTGGCCGCGGCTCTCCCGCGGGTGGGTCACGTGTTGGCGGCGCCTGGTTGCCTTGGCAGCGGCTGCGGCGGCCGCGGGGGCGGGGTGGAGGCGGGGCCGGGGGACCCCGCGCGACCGGCGGAAGGAGGGAGGGGGCCGCGCTCGGCGCCCCGGCCGGGCCACTGGGCCACAGGCCACGCGGCCACGCAGTCCGA
FP001882	chr1	228644404	228644655	+	1	train	TGCCCCGGGCTGGCACTAGAGGCGGCGGCCTGATCTCGGGTGAGAGGGCCTGAGAGAAACCCAGACACACCCCACCGCCACCAGGAGCAAATCCACTCCCCCACACACAGACACACCCGGGCGCGCTCGCACCCGCGCGCGCGGACACACACACACACACAGACACACACGCACACACGCACGCGCACACGCACGCACACACACACGCGGCTTGAAGGAGAGCAAGGACGAGATGGATGGAGAGATAGAAA
FP019360	chrX	69615913	69616164	+	1	train	GAGGCGAACCCTCACGCCCCCAGGCCCCGCCCCGCGGCTGGAGGCCCGGCTGGAACGGCACGGGGCGGGGCGCCAAGGCTTGGCGGCCCCGTGGTTGGGCGTCCCGGCAGCCGCTTGAGGGATGGGGCGGGGTCGGCCGGGACCTCCTCCTTCATTCCCTCGGCGGGCCGAGCCTCCCCTCTCTCCCGCCCCTCCTCCTCCCTTTCCCACCCCTCGGAGTAGAGCTGCACATGCGGCTGCTCCCTGCTCCG
FP017665	chr19	53867682	53867933	+	1	train	CTGGGGTGGGGCGGGAGCTCCACGCTACTTTTACTCTGGGGGGCGTGGTCTGCTCTTGGCCCCGCCTCGGGGCGGGGCTGGGATAGCCCAGACTCGGGGAAGTCCCTGTCCACCGCCCTAGCCGGAGCTGCGCCCCGCCCCGGCCCCGCCCCCGTCCCGCCCGCGCTGGCTCCCCAGCCCTCCAACCCCACCCAGCCCTCAGACCCTTCCAGCTGCCGCTGTCGTCTTTGCTTCAGCCGCAGTCGCCACTG
FP004641	chr4	37977181	37977432	+	1	train	ACTTCCCGCCAGCAGCGGAGCGCGGGGGTGGGCGGGGTCGGCTGCGCGCCCTCCCCTCCTCTCCCCTCCTCCCCCCGCCCACCCCCTCCCCGCGCTCTCCCCTCCCACTTCCCTTTCTCTGCCTGGCCGCCCCGCGGGCGCGACCTCTCGGGGCAGTGACGAACTGGGTGGAGCCCGCCGCCGCCGCCGCCGCCGCCGGGGAGAGCGATGCCCCGGCCCCGCCGCTCCCCAAGCCCGCCCCCGGCCGCCCG
FP005747	chr5	138543114	138543365	-	1	train	GGCTCATTGTAGGCTGCGGGGCAGAGAGCCAATGAGAAGTCGCGGCGGACGCCCAGCTTCTCCTCACTCGCACCGGCCCCCTCCCTTCCGCCTGCCGCCTTCCACTCCGCCCCTGGCGGAGGAAGTGATGTCACAGGCCCCATGTGAGCGGATTGCAACACATGCAGCTGCCTGGAGAGAGGGAGCCGGTGTCCTACGTCAGAGCCGCCGCCGCCGCGGAGCCGCCGCCGGGGAGGAGCAGCCGCTGCCGC
FP016523	chr19	1354758	1355009	+	1	train	CCGGGAGCGGCCACACATGCCCCGCCAACCGGTCTCCTCAGGCAGCACTCCCGGGAAAAAGGGGTAGACGCGCGGCGGAAGGGGCGGGGCCGGCGCGCGGCCGTGGACGCCGGAGAGGGCGAGGCCGGCGCTCCTTGGGAGCGCGCGCGTCCCATTGGGCAGCGGGCGGAAGGGGGCGGAGCTTGGCGCCGCCGCGAGGCAAGCCCCGCCCCCGGCCCCGCGGGGAGCGGCGGCGGCGGCGGCGGCGGCGG
FP017483	chr19	48170251	48170502	-	1	train	TTTCCCAGTGTGCTCTGCGGGAGGGCTCGCCCCACTTCACCCCTTTTCCCGCCCTCCTCCCATTCGGGAGACTACGACTCCCAGTGTCCTCCGCGCGACGGCGGCGGTGCGGACGGTGCCCAGGTCCCGCCCCTAGGCTCTGCCCCGCCCCCGCCCGCAGACGTCTGCGCGCGAATGCCGTGGCGCGAACTTGGGACTGCAGAGGCGCGCCTGGCGGATCTGAGTGTGTTGCCCGGGCAGCGGCGCGCGGG
FP000121	chr1	7961500	7961751	+	1	train	GGGGCAGGCCGGACTGTGCCATTCGTGGGGGGTACCATGTGGGACCGAGCCGCCTCACCCAGGGCTGTCCAGCTAGAAACTCCCCGGTGCCACCCCCGCCTCAGTCCGAGGTAGACTCGGCCGGACGTGACGCAGCGTGAGGCCAAGGCGGCGTGAGTCTGCGCAGTGTGGGGCTGAGGGAGGCCGGACGGCGCGCGTGCGTGCTGGCGTGCGTTCATTTTCAGCCTGGTGTGGGGTGAGTGGTACCCAAC
FP010624	chr11	20048546	20048797	+	1	train	TAGTGTCTAGGCCCACTTTATGGCTGACCTTTCCCTGAAGTACCGTTAGCCATTTCAGTAGGAATGGCTGTGCTTCCTTCCCCAGGAATTCTCACCAACTCCTCTGCCCTACCCTTCTTTAACAGTCCGGTGCCCCTTGGCACTGGGTGTTCAACCTACACAACCCTTTGTCTCTTCACAGACTCTTTGGTGGGAAGCCTACCAAGCAAGTGCCCATCGCCACAGCTGAAAACATGAAAAATTCGGTGGTC
FP017466	chr19	47232696	47232947	-	1	train	AGAGGGAGGAAAGCTGAGGAGTTCCCAATGTTGCAAATGGGGAGATTTCACGTGAGATATAGATTACCTGCATCTCTTGGGGGAGCTAAGAGTGTGTACTTGGAGGCAGTCAAGTTTGAGAAGTCTGACATCCTTACTCAGCCAGCCCCACACTAGGCACTGGAAGGTGAGTCACTCTGGTGAGGCGATTGCGATTGGGTGAGACCCAGTAAGGATGGAAAGTGTAGAGGAGACAGGAATCCACGGCTTTG
FP010442	chr11	2141195	2141446	-	1	train	CAATTGCATGGAGCAACTTCTCTCATCCCCCAAACCTGTAATCTATTTTTCTGGAGTCTCGAGTTTAGTCATTAATCACGGTTCCCACATTAACGGAGTCCCCGGGGTCCCCTCCTCCAGGACACCCATTCGCTAAGCCCGCAAGGCAGAAAGAACTCTGCCTTGCGTTCCCCAAAATTTGGGCATTGTTCCCGGCTCGCCGGCCACCCACTGCAGCTTCCCCAACCCCGCGCACAGCGGGCACTGGTTTC
FP002746	chr2	149330389	149330640	+	1	train	TTCTCCCGGAGTTGGGCCGCGCGCGCCGGGCCAATGAGCGCGCCCCGCGGAGGGCTCCGCTCCCCCCGCGCGCGCTGATTGGTCCCGCGGCCCTCGAGGCCGGGCCGGCCGCGGCTGCGGGCGGCGGCCAGTGCCCGGCGCGAGTGGGAGTGGCGCGCGCGCCCCTAGCCGCCCCCCGCCTCTCCCCGCTGCGCTCCCTCGCTCCTTCCCTGAGCTCCCGGGCTCCGGCAGCGGGCTGGCGGGGCGCCGCA
FP009181	chr9	110208108	110208359	-	1	train	TGGAGATCTTGGGGAGATCTTTGCTACTGGGTTTCTTTAGGGTGTGGCCGCGGGGCACACATCCAGAGGCACCGACCTTGTGAAGCTCCCGAGGGTGTTCTTTGAACTACTGCAAACCCAACCGGCAGGTTTGCTGAAAAGTGGCCACGTTCCAGTCCAAAATGTGTGAGCAGAGCGCGAGCTTAGCACTCCCCAGCCCCAGTCCCAGCTCTGCCAGGAAAGGGGAGACTTCACCGTGGCACTGGGAATTG
FP003355	chr3	10026236	10026487	+	1	train	CACCGGGGCGCAGTTGCTTCTCTCTGACGTCGCCTCTGGGCCTTCTCTCGCCCCTATGCCCGGCTAGCACAGAACTCTGCTGCAGCGGTGAGCCCAGCTTATTGCCTTCAGCTGGGCTGCCCGGCCTTCCACTTCCGGCGCGGAAGTTGGCGTCACGTCATGGCGCGCCTCGGTGGCGTCAGAGCGGCGTCGGGCCTGGCGGGAAAGTCGAAAACTACGGGCGGCGACGGCTTCTCGGTGAGTAAGTGGAG
FP006152	chr6	6588572	6588823	+	1	train	CCTCTGCTCTCAGCCTTTTACTTCTGACTGTGGTTCCCAGCGAGTCTAAAATTTGAGAAGTGAAACTGAACATCCAACTGCCTTGGTGAGGGGAGAGGAAGAAATTTTACAGGCTTTTTATTGGCCGGTTATTTTTCTGTGTGTCCCATACAGGCCCCCACCATGAAGGGTTTCACAGCCACTCTCTTCCTCTGGACTCTGATTTTTCCCAGCTGCAGTGGAGGCGGCGGTGGGAAAGCCTGGCCCACACA
FP019092	chrX	13734581	13734832	+	1	train	GCCCCGAACCTGTAGCCAGAACGCCGAAGCAGTTCTCGCGATACCCTGGGATGTGCTCTCGCGATACGTAGGCGGAGCTTCCGGCCGGTGCTGGGCAACCAAGCTCATGCGCCGTAGCTCTTCAGCTCGGGAAGGCTATATTTAGCAGGTTTCCGGAAGTTGCCGGACTGGCTGTGAGGCGGTCCTGCCTCGCTGCCTTCAGTCCCTAGTGTCTGGGTCCCCGCCCTCCAGCCGCCTTTGAGTCGTGCCTG
FP002440	chr2	75561278	75561529	-	1	train	TAAAATACTTGTTGATTTTAACAAACAAAACCCCAACCCTGAAAGAGAAAGGCCAAACAGAGGCCTCAGCTGCATTTCTTCCGATAAAAATGAACATTCCTTCGAGATTATGCATCAGGCTTTAACTTCCCACTCAGAAAAAAAAAAAAAAAAAAGTTGATCAAATTTAACAAAGTAAGTGGTTATTTCCTGTCTGAGTTGAGTAGCTGAATACAGGCTCAAAGCAGGCGACGGGACAGGAATGAGAGGAT
FP017452	chr19	46714243	46714494	-	1	train	CCCCGGCCCCCCCCCCCACTTTCCCAGGAGCCCCGAAAAGTCCTCCTTCCAGCTCGCCCCACCCCAGTGCTGGGCCTGGAGCCAGGTAACTGGGACAACAATAGACAGATCCAGGAAGGAAGCTGGGGGGCGGGTGTGTGAGCCTGGGGAGGAGGCACAGGGGAGGGAGTGTTCATTCAGCATCCCCTCCCACCTCCGCCAGGTTCCGGAAAATTCGAGGTGTCCACGCTCCCGGAGCCACTCTCCCTCCC
FP008786	chr9	15510115	15510366	-	1	train	GGTTCTTCGCTTTAACCGCCCTCGGTGCTCTCCCTAGCAGGCGCGTCGGGACGCCCCGAGGCATCCTCCCCCGCCCGCGGGCCCGGTAGCTGGGCCCGCGTCCGCCGCCCGCATCCCCGCGCCGCCGCATCTCCTCGCCGCCTCCCGGGCTTCGGACCCCCGGTCTCGCCCCCGAAACATGACTCGCGATTTCAAACCTGGAGACCTCATCTTCGCCAAGATGAAAGGTTATCCCCATTGGCCAGCTCGAG
FP008707	chr8	144464831	144465082	-	1	train	CTCAGGGATCCGGAAAGTCTAGGACTGAACTTCTCCTAACATCCAGTAATGGGGACCTGGAACCTGGGCTTACTAGAGTGCCGCGCGTAGGGCTCCAGGTCGCTGGCTTCTGCGCTTCCTTCCTCTCCAAAGTTGAGTATCTCCTATCTGTGTCCTCGTACATACTGCCGCCTGAGGTGCCATGGCCCCCAAGCCGGGGGCCGAGTGGAGCACAGCCCTGTCCCATCTGGTGCTGGGAGTGGTGTCTCTGC
FP002805	chr2	163735952	163736203	-	1	train	TGCCCGTCTCTCTCCACTGCCCGCCTCTCGCCACTGCCTCCCGTCTTCTCTCCCTCACGGCAGCCTCCGTCTCTTTCTCCCTCTCTCCTTTGAAAACTGATCTTTGGCTCTTAGTGTGAAAGTGATTTGAATTTGGCTCTGCGGCGCTCTGCGCCTGCGCCCAGCCTCTGGCATGCTGGCTCCTTCCAAGCAGCTGTTTTGGGTTTGAAATTCCAACATGGCAGAGGCTGCAGTCCGTCTTCCCTTCAAAA
FP011839	chr12	45729515	45729766	+	1	train	CGGATCCGGCAGTCGCCTGCCCTTCTAGTTTCGCTCCGATTTCTTTAAACTTTTTTAGAAATTCCCCTCCCTCCTTCCCTCCTCTCCCCCTGGCCTCCTGCTCCCTCCTTCCCCTCCTCCTCCTCCTCCCCCTCCCTCCCTCCCTCCCTGCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCACCGCCGGCCCATGACTGAGCCCCGCCGCCGCCGGCCGAGGAATGGGCTCCGGGCTCTGGTAGG
FP008615	chr8	131040622	131040873	-	1	train	TCCAGCCACATCTGCTAACTTCGCACCCATCGCTGCCGCCGGTCACCGCCGGCCAGGCCCCCTGCAGCCGCGGAGCAGTGGGCGTCCAAAGCCCAGTGCAGCAGCCAGGACCCGCCCGACGCGCAGCAGAAGCACGGCGCCCAGGCGCTTAGGCGTCTCTTGGAGAGCAAAGGCTGCGCCAAAACGCTGAGCCTAGAATCAACCAAGGAGCCTGAGCCCAGGAAGGGGCTGCGTGGCTCACAGCGCTGCGG
FP003899	chr3	108589401	108589652	-	1	train	TCAGGCCGAAACCTCGCGGCCTCTCAGACGAGGGTGGGTTAGCGGGGGCAGCTCCCAACCCCCGTCCTGGACCCACAAATCACCTCGACCCCTGGCCCACCCCGGCCGTCACCGAGAACGGTCCCCTAGGGTGCCTAGGGACTTCCGGAGCCCGACCGGATCCGGAAGCTTCTGAGAGCGAGGGGGTGGGGCCGAAAATCAAAAAAAGCGCGGCGAAAGCTAAAGGCCGGCGCACGCTGGGCGGTGGTGGT
FP013232	chr14	58298374	58298625	+	1	train	AAGCCAATCATAAGTGGAAGATGTCCTACCTGCTGTTTTTCTCACCAATCCATGAAGTTTCACAGCTACATCCAATGAGGACGGCAGGTAGCGAGGTCCTATCCGAAGCTCTTCGGCGTCATGAGCAGCCAATAGGAGTTCGTGTAGAAGCGAGTCTGCTCAACAGCTTGTTATTTGGTGGATTGTGGCAGTAAATCGGGGCGAGTGGGGAACCCGGCGCAGGAACTGCAGCCGCGGCTGGGAGTGGTGCT
FP011810	chr12	33439859	33440110	-	1	train	AGGGCTCCTCGGGCCAACTCTGGGCAACGATGAGCATCCCCGGCCTCGGCAGGGGCGCCGGGCCGCCCGCCCTCCCTCCCTCCCTCCCTCGCGAGTCCCTTCTTCAGAGGACGCGGGGAGCCGGGTCTGCGCGCTCCGCCGCCTGTCACCTAGCGCCCGCGCCGGGCTGCGATTGGACGCGTCGCCGCCAGGCACCCGCCAGAAACACCAGTGCGCTGGGGAGAGGAGGGGGGCGTGCTTGCGAGTCGAGG
FP005634	chr5	122077089	122077340	-	1	train	GTCCGCTGGAAGCACCCGTGCACCTGGTCCCCAGCTATGTGGCTTCTCGACGTGGCTGCCTGGGCGCGGCGGGCCCCGGTCCTCGCAGATCCGATCCCTCCCCCACGCGCCTGCAGTGGCAGCCCTGGAATCCAGTGCAAACCGCGCGTCTGGCCCCTCCTGCTTCCTTTTCACATTGCTTTGCAGTCCCGGGCGTCCCCAGTTCTCTTGCTGTCCTCCGCTCCACTCTGCAGTCCCGGTGGGCGAAGGGT
FP018249	chr20	57620375	57620626	-	1	train	ACATCAAGTCAGGTCCCCAACTCTGTCTTGCTCATCCACCTGGCTTCCACTTTTTTTTTCTTTCAAAAGACGAACAGAGAAGTTTCATTTTCTTTTTCTCCTGAAACCGAATCTGGCCGGCCTGGCTAGGCATCTATTTCCGGGCTGTAAGCAGCTGACACCTGCCCAGTGGAAGCTGGCATCCCTCCCCTTGTGGGTTCAGAGCTGCAAGAAGCACCAGGCTCGGCCACTTCAGAAGCCCCAGCCTCGAC
FP002962	chr2	191683477	191683728	+	1	train	GATTTTTTTTTCAGCACTAAAGAAGAATTTAAATACAAAATAGAATGTTATATATCTTGTAGACAGTCAAATATTTGATGTTTTATGGGACATAATCATTTGGGAAGTTTTTGTTGTAAATGAAGAGTTAGTTTGTTGCTATTAATTTGTTTGACACATAAGTTCATTCCTAAAAGTTAGAGATGTTACATAAAGAAGGGTTGAGGACTTTATTTCAGAAGTCATTTAATTTTTTCTTTATTTTCTTTCAG
FP010576	chr11	13349968	13350219	+	1	train	GAAAAGAAATTATAAAACATGAAAATCGCTTTGAGGTGACCAAGTCCAGAGGCCCCTAACTCCTCCCAAGCTGGATCTGGGGTGTAAGAACTGTGACTTCAGGTAACAAAGTAGTATCCCCTCCCTGCCCCCTGTGAACTCATGATTCTTGGTAGCTTGCATGGCAGTTCTTCCTTTAGGAAGTGTAGTTTCTTCCAGGAACTCTGAAACTTGGGGTATTCTACCTCATGCGGTTGTTTGGCTGGTCAAAA
FP017902	chr20	4815092	4815343	-	1	train	TTACAGCTCATCACACAGACATTTAGGGCAAGGATGGCTCTTTTGTGTACAGTCGGGGCCATGGTGCCACGTAGTTGGGGGGCTGCTGAGGCAGAGTTTGCCCGCAGGTCAGCTCAGGAAGCTGCGGGACCCAGAACAGGGAAACCAGGTCACGTGACTCTATTCAGGGGTCACGGGAGTGGGTCAGGTGGCTTTTCTCTGGGACATTTGCTCTGAGGCAAATGCTGTGAGTTCCTTGTTAGGTGAGTTCC
FP005363	chr5	50403309	50403560	-	1	train	AGTAAGCAATATTGTTGGCATTATCTTATAGCTTTCACTGAAAAACTATGTCATGTATCTTTTATGTGCTATGATAGGAATGGATTTTAATGTCTCCTCATATTAGGTTCCTGTTGGTGTTCAAATGAATAAATATGTGATCAATGGAACATATGCTAACGAAACAAAGCTGAAGATAACACAACTTTTGGAGGAAGATGGGGAATCTTACTGGTGCCGTGCACTATTCCAATTAGGCGAGAGTGAAGAAC
FP003048	chr2	203706438	203706689	+	1	train	ACAAAAAAAAAGTCTTTAAAAATAGAAGTAAAAGTCTAAAGTCATCAAAACAACGTTATATCCTGTGTGAAATGCTGCAGTCAGGATGCCTTGTGGTTTGAGTGCCTTGATCATGTGCCCTAAGGGGATGGTGGCGGTGGTGGTGGCCGTGGATGACGGAGACTCTCAGGCCTTGGCAGGTGCGTCTTTCAGTTCCCCTCACACTTCGGGTTCCTCGGGGAGGAGGGGCTGGAACCCTAGCCCATCGTCAG
FP012214	chr12	71754672	71754923	+	1	train	CACCAACTGAAAAGCAGAGGTGGCTCGCAAGCGGCGGGGCTCTGCCCTAGATTGCCAGTAGGCGCTCGACTCACATCCAGCTCCACCCCGGGAGATGCCCCAGAAGCCCCGCCCACGTCCAGGCACGTTCTTCTGGGGCTAGTCGCAGGGGCCGCCCCGGCACCGGCGCTCACGTGACCCGAGGTGTGACGTAGGAGGAAGGAGACGCCATTAGAGGGAGGCAGAGAGGGATCGTTCTTCGCTTTTCCTCC
FP007201	chr7	12211126	12211377	+	1	train	TCTCACACGATCGACGCCTCTGGGCTCAAACAGGAGGAAGAAGGACGCATGCGTCACGCGCACGCCAGGGGTGTCCCTCCGGGAGCGCTCTGCGCAGGCGCGGACGCAGGTTACAGCAGCGCTTGGCCTCTGCTGATGCCGTCGTTATCCTACCCCTCCCCCGTCCCAGCTCTACGGCGGCCGCGCGCTCCAGGCCGGTCGCTCCACCCCCCGGCTCCCGGGACTGTGGACTCCACGACCCTGTCCTCGGC
FP011172	chr11	75668611	75668862	-	1	train	TGGCCTCCCAGCCCTGCTAGGTGGCCGGCTGCACTAGGACGCCGGCAGGAAGGAGACTCGAGCCCGGTCCGCGGACCCTGCGGCCACACCGCCCGGCCCCTCCTCTGCGCCATGGCTTAAGCCTGTAGCCACCTCTCCTCACGTCGCCTCGCTCGACCGAGGTCTGTGCGCCGCCGCAGCCGCAGGGAGGGGCGCGTCCCAGACACCCTCGCCGGGGACTCGGGAACGCCGTGCGTCTCACCCTTCGGCGC
FP003352	chr3	9933638	9933889	+	1	train	TCCGTTTCAGTGGCGGGGCTGGCAAACGTCATTCCCTAGCCCCGCGGCCCTTTAAAGCCCGGACAGGTGCAGCTCGGTGCCGCCTCTGGTTGGCTGGCGTGGGGTGACGTAATGGCACATGGCCCGTCGCCATTGGCTGGGCGGCAAGCTCCGCCCCCTGGACTGCGGCGCGGGTGGGGGTTGTGCGTTTTACGCAGGCTGTGGCAGCGACGCGGTGAGGAGACGGCCCACGGCGCCCGCGGGCTGGGGCG
FP001028	chr1	110631416	110631667	-	1	train	CTAACTCTGCATGGCTAACCCACCCCTGCTAGTCTGCAGTGCTCAGTTTCTGTTTTAAGAGAAAAACCCGTTCTTGCACTCCCTGGCTGGGGTGACAGGAAACACCTGAAGATTGTTGATGAAGCATCATGCAACCAGGTCACCTAGAACCTGAAACCCTTTGAGCGCTGACCAAGAAAGGAAGTGGTGATGGGGCACATAGAAGAGTGAGCCATCATCTGGTTTCCAGCGCCAAGACTTCTGCAGGAGAG
FP002458	chr2	85410312	85410563	-	1	train	CCTCTGAGGAGACGGCCTGGCATACCCACTGCCCACCCCAGTGACTGCTCTTCTGCTTCAGGCCTGCTGGCCTCCCAGCACTGCCTGCCCCTCCCTGTCGGGGGACATCGCCTCCACACCGGCTGGGGAAGGAGCCCAGGGGTGGGGCTGGTGGGTGGGGCTGGTGGTTGGGGCAGCCAGAGAAGTAAGAGGGAAGTGAGAAGCCGGGTGGGGCAGGCTGGAAGGAAGACGAACCTACGAAGCAGAGGTAG
FP013112	chr14	34462189	34462440	-	1	train	GAGGGGACGACGCGACGCGAGGTAGCGGCCTGGCCAATGAGCGCGGGTCGGGCGGGGCTGGGGGCCGGCGGCCGGGGCGCGCGCGGGTCTGGGCGTGCCGCGGGTGACGGCGGGGCCGGGCGGGCCAATGGGCGCTGGGTGGCGGGCGGGCGGGCCTGGCGGCCGGGGGCTTTAGGACGTGAGCGGGCGGCCGGCCGGACAGACTGACGTGTGAGCTGCATCGCGGGAGGCGCATGGCGGGGATGGCGCTG
FP004597	chr4	17512039	17512290	-	1	train	TGGGAGGAGGGGCTTCGCCGATGTCCCCGTCCCAGGGGCAGGGCCTGGCCGAAGTTACAGTCCCTCCGGGTGGCGGGGCGCCCCGAGCGTGGCAGCGCGCTAGGCGGCAGCAGCGGGCGCGGAGCGGGGCGCGGCCGCCGCGCGTTCCCTCTTGGCGGGGTTGGCCGGCCGGGGCGGGGCGCGGCGCTCCGGCTCGAGGCATTCGGAGCTGCGGGAGCCGGGCTGGCAGGAGCAGGATGGCGGCGGCGGCG
FP001542	chr1	174159319	174159570	+	1	train	GCCACGCAGCGACGTGCACGCGCGTGTGCGCCGCCGCCGCCCGGCAGAGTGGGCGCCTCCGACATCAGGAGCGCGCACGCCGCGCAGAGGGGCCTGGACGAGCGCGCTTCCGACCCGCCGCCACCGCCTCCTCCCCTCCTTCTCGGCCCGCCCCCTCCCTCGCCTCTTCCTGCCAGGCGGCCCTTCTCCCCTCCCCTCTCAGTTCCCTCCGCCCTCCTCGGGCTCCAGCGGTGGCGGAGCGAACGGGACCG
FP014632	chr16	30894004	30894255	-	1	train	GGGCAGGGGGCGCTGAGCGTCCAGGCGCTCCAAGGGGGCGGGCCCGGGTCGGGGCGGGGCCGGCCGGGCTTCCAGGCCTGGGCTCTGGCCGCCCGCGCCACCGGGCCGCTCCGGGGACAGGCCGGGGCGGGGCGCGGCGGCAGGAAACGGGGCGGGGACTTGCGGAGGCGTTGGGGACGAGAGAGGGCGCGGCCAACTCCAGGGGGGACGGCAGGCCGAGAGCGCGGCGCCCGGGCCTGGCGCGGAGCCTG
FP017722	chr19	55258485	55258736	-	1	train	CCGCACGCCCACCTCGCCTGCGGACCCCGGAGGTGCCTACGCGGACTAGTGACGCCCCCCGCCCGTCGGCAACTCCACCCGGCGCTGACCCCCTCAACTCTGCAGGACCCGGCGCGCGAGGACCCCACGACCCGCGGGCGCCTGCGCGGACCCCGCGAGCCCCGGCGCCTGCGCGAGTCCCCGACGCCGCCCCCGCCCCCACTCACCGGGCCCCGCAGCAGGGGCCGGAGCCGCCGGCGGCGGGGCGGGGG
FP009680	chr10	24633414	24633665	-	1	train	TAAGCTAGGATATTCATATAATTGTTTTATAATTATGATACTAATAATTTTCTCACAACTCTGTATTTTATTATTTTAATATATACGACATTAAAAATAAGGACATTATTCAAGAAATATTTCAAAAGTGTTCAAAAGTGACTTTCTGCAGGATCTCTCATTTGTTTTTTAACTTCTCTTTGAAGGTGACCGAATTATAAAAGTCAATGGAGAAAGTGTTATTGGCAAAACCTATTCCCAAGTAATTGCTT
FP010575	chr11	13277589	13277840	+	1	train	CGAGCCGCGCGCGGATTGGCTGGGGGCGGCCGCCGGGACCGGCTCCCTTCGGGCGTTCGGATTGGCTGGCGGGAAGAGGCAGGTATCCGGGCGCTGCGGCTCCTCCATTGGTGGGCGGGGAAGGGGGGTTGGGCACAGCGATTGGTGGGCGGGGGGCCGGGCCTGGGCCGGCGGGGAGCGGATTGGTCGGAAAGTAGGTTAGTGGTGCGACATTTAGGGAAGGCAGAAAGTAGGTCAGGGACGGAGGTGCC
FP012930	chr13	112968247	112968498	+	1	train	ACAGGCCTGAAATACTGTTTCTTTAAGGAAGGGCATGTTACCTATAATACCAAACCACAAAAGGATAGCTGCGGTTTTGGGCGAGGAGAGCTCAGAGAGTTTCTTGCATATGGCCCTGTGATGGCGGCCATGGCCCTGCATAGACACGAGCTGGAATCTGCAGGTGGCAGCCAGGACGCTGCGTGTGTCGAGTGCACAGTGTGGCTTGGTGCCAACCATGGCGAGGGTGGAGAGCCCCGTGCCTGCAGCGC
FP017541	chr19	49361464	49361715	-	1	train	GGGAGGCGGGATCTGACTGTATTGGAAAAAGTTTTCCCTGTTCCCGGCCTCCGGAAAGTCACTGGGAACTTGAGTTGGATCAGAGCCCACGGCGCACGGGGGAGGGGAGGGGCGGGGCCCGGTGCCGGAGGGGAGTCGCCCTGGCCGGCAAGCCCCTCCCCTGATGGGGGTTGTGGGAAGCTGGGGGAGGGGACGCCTGGGCCCTTTGTTCTGGCTCTTCTGATGGCAGGGCGCCCCTGTCCCCCAGCTTG
FP001219	chr1	151190110	151190361	-	1	train	GGGAGATCGCTGGGTACTAGAGCAAGCGGCCTAGAGCATCCGCGCAGGAGTAGGAGAGGCTGAAGAGTAAAGAGGGTTTAGAGGGTGGGACTGGGAGTCTCTCGACACTCTGTTACTTCTGCCTGTAAACGCCCGACTTCCGCCGCTGGTGGCCACCCGCAGGTAGTGATGTCGAGCGTCGAGCTCCCAAAACCGAGCTGGTGAGGGGCTGCAGGTGGCGGCGCAGTCTCGGTAGGCGGTATGAGTTTGGC
FP004748	chr4	68129808	68130059	-	1	train	CTTGGAGCTTCTCTATGCGGATATTGACAATTACTTCACTGTGTGGAGCCAGTTGGCACAAAACTGATCATGTTCTTGTTCTTATCAGGCTAGAAGACTGCATACCTCACTCACTAAACCTGTCAAACTTGTTTCCTTTCTTGTTGCCAGTGACTGTCATTATTTTGCAGCCTCCAAGTCCTAACCTGCTGATACAGATCAGATGGTGACTGAATAGAAGCTGCCCCAGTCCTGGGTTCATGATGTACGCG
FP007103	chr6	166999183	166999434	+	1	train	CCGACCCTAAGTTTCGGCGCTCAGTGGTCCGGCGCTCCCCAAGGCTCGGTGTCCAGCGTCAACCCCGAGGTCTCTATGCCCCGCCTCCCGACGCCAGGGGGCAGGGCCAGCGCGCTGCGCGTCGGGGCGGGGCTTTGGCTGCGTCGGCCGCGTAGCCCGCGCGCGGAGCGTACCCTGCTGCGGCCGTTGGCCGTTAGCGCGGCTTCGGCGGTTGTCTTGGAGAAGCAAGATGGCGGCGACGGCGGCCGCAG
FP000291	chr1	22636477	22636728	+	1	train	GTTGCTGGGGCAGGACGCCCAATGTCCCAGTCTTGCTGAAGTCTGCTTGAAATGTCCCTGGTGAGCTTCTGGCCACTGGGGAAGTTCAGGGGGCAGGTCTGAAGAAGGGGAAGTAGGAAGGGATGTGAAACTTGGCCACAGCCTGGAGCCACTCCTGCTGGGCAGCCCACAGGGTCCCTGGGCGGAGGGCAGGAGCATCCAGTTGGAGTTGACAACAGGAGGCAGGTGAGGCCAGAGTCCCAGAGGGAGGG
FP017928	chr20	10434147	10434398	-	1	train	TAAATTTTATGCGTTCTTTTAAAATTTCAGTTTCAACAATTTGCTGGAATATACGCTGAATAAACTTCGCTCCCCGTTTTTTCCGGTGACGGCCGCTGTTGGCTGGAGCTTCACGGAAAACCCGGAGAAGCGGAGTGGGGGCGGCCCGGCGTCGACCGCAGAGCTGCGCGTGCTCCGTGCCCTCGCGCGACGCGAAGGTTGTCGGGATCCGCGGCAGCAGCGGCTGCTTGAGATCTGTTTCTGGGGCCTCT
FP011778	chr12	27710631	27710882	+	1	train	TTAAGGATTTTTTTTGTTTTCTTTTCATTACTTCTTTCAGTTTGGGGGAATCTTCCTGCACATTTGACTCGGCTCTGAGTGCGCAGACCAACGTGTTTCCTCTTTCTCCAGCCTAGGGCGGTGCCAGCCCAGGGAGCCTGCGCTTAGCGGCCCCAGGCCCGCCCCTGACACTGAACGCCGCTTGTCCCCTCCGGCTTGCCGTCCTCGCAGCCATGGCGGCCGCCGCGCTCCCAGCATGGCTGTCTCTGCAG
FP014353	chr16	3443505	3443756	+	1	train	CCGGAAGCGACCCGCCCTCAGCCCTCTCCGCCATCCTGCCCTTTGATTGGTCGAAGTTCTCCCGAAGGGGAAGTCCTTCTTTTGTGTTGACCAATTGAACACCTTAACCGAGGACCTGTAGCCACTGGGCAGCTGCGAGAGGTAGTTTTCCTCCTTTTCTCTAAGCAACCATTTCCGCTTCCGCTGGCGGGGTCTCCTCCGTGAGCTCCGGGCCTGTTTGCCTGCTGAAGTAGAGTCTTAGGGTGACCCCA
FP003207	chr2	230713498	230713749	+	1	train	CCCCCGTGGTTCTCTGGCGGGGCTGGCGGCGCGTGGTGGTCCGGGCCGGAGTCGGGCGGGCGGGGAAGTGCGGGCCGGGTCAACTGTCACCGGCTGCACCCGGGCTTCAGCGCCTCGCTCCCGCCCCTGGTAAACTTCCTGTTGTCGGGAGGCAGATACCAGCTTTGTGACCAATTCAGGAAAATGCCTAACTGATTTCTGGTTAGGAATTGGGCGGTCGGAGCGTTCTGGAGTTCAATGGAACCACCCCA
FP019528	chrX	109733183	109733434	-	1	train	AGCGGGGGCGGGCGCGTGGGTGGGCCGAGCCGTCTGCAGCCAGCGATTCGGCTGGCTCTGCCACACCACCGCGCGCCCCCGCTCCGCCCGCCCCTCCGGGCGCGTCTTTTCCGGGCTCGCGCTGAGTCCCGCCTCCGCCGGCTGTCCGGGTGCGCGCGCGCCGCTGCGGCTTTTTCTCTGGCCTCCGCCGCGCGCTCCTCCTCGTCCCAGCGCTAGCGGGCACGCGGTTCCTTTTTGCGAGCTTTCCGAGT
FP002127	chr2	27262996	27263247	-	1	train	TGGAGATGAGGAGTTCTCCAGAACTCGGGTGGCGGGGCGCTCCCGGGCTCTGGCGCGGGGAAATGGGGCTTTGGCAGTGGGGGCGGGGCTTAAGGGTGGGGAGAGGCCGGGGAGCCTGGAGGCGGAGCTTGGGGAGCTCGCCCCACGCGGGGCGGGGCTCCGGCGGCAGCGGGATCTGCAGTTCGGACTCCGCGCGCCACAGTCGCTGCAGTTCGCTCCACTCTGGTGGCCCCGCCGCCCTGCGGGGATCC
FP004068	chr3	129121981	129122232	-	1	train	GGGGCGGGCTGAGGTGGGCCTGTGGGCGGAGTCGGGGCGGTGTGCTGAGGTGGGCCTGAGGGCGGAGTCGAGGTCGGGCTGAAGGCGGAGTCGGGGAGGGCTGAGGTGGGCCTGAAGGCAGAGTCGAGGTCGGAGCTGGGGACGGAGTCGAGTTCGGGCTGAGGGCGGAGTCGGGGCGGGCTGAGGTGGTCCTGTGGGCGGAGTCGAGGTCGGGCTGAGGGCAGAGTCGGGTCGGAGTCGGGGCGCGCTGA
FP003174	chr2	222671457	222671708	+	1	train	CGTCCCTCGTGTCCCACGCGGCGCACTCTCGCCAACGCCCGGCGCGCCAGCCAATGCGCCCGGGCCGCTGTAGCTGCTCCGGGGCCAGGGCGCTGGGCCAGCTCTGGCGCCCGGCGCTTGTGATTGGCTGCCCCCAGCCTCAGCCCAGCGCGCCCTCCCACCCAGGGCGCAGTTGGGTCCCCGCTCGCTGCCTAGCCTCTGCCTTTTCCTCTCCGCCCAGCCAGTGCCCAGCGCGGGGCCCGGATCCGGCC
FP017777	chr19	57435124	57435375	+	1	train	TCCGTAAGGTCCGCGCGCCCAGCATTGTGAGGCGAGGCAGGCAGCAGCGTGGGAACTACATTACCCAGAAGACACTGCGGGCGACACAGGCAGCAACGTGAGAACTACATTACCCAGAAGACACTGCGGGGGCAGGGCAGCGAGTGTAGCCATTGGTCTAGCAGAGAGACGACTAATGAGGTCTCAATTGTGTGGGCGGGACTTCTGGCGGCGCCCTCATGGTTGCGTTAGCATGGCTACCTAGGGATCTG
FP009943	chr10	73997915	73998166	+	1	train	CTCAGCCCCAATCCCAGGCCCCTCCCTTAGGAATGGGGGAGGGGCGGGGTTTGGCGGCGCGGTGGCTGGGGCGGAGCATCTCGAAAAGGGACCAGTAGGAGTGGCGGCCCAGGTCTGAGCTGCGCTGCCATTGGGCGCGGCGACGAGAGGGGGCGGGGAGCCCGTGAGGCTGGTTACGCCGAGGGAAGCCCCGACTCCGTAGTCGCTGCACAGTCTGTCTCTTCGCCGGTTCCCGGCCCCGTGGATCCTAC
FP013458	chr14	92121768	92122019	+	1	train	GCAAATCATACTACTTTTTCCTCCTCAGGGTTGACTCCTCTCATTTTTTTTTTTTTTAGCTTCCAAACATTACCAATAATTTCTGAATAAAATTTCGCTCCAGACAGACGAGGCGTGCAGGCGTGACCGCCGTCGCCTTCCGACAGCACCGTGCCCACCCCCCGGAGATGGTCTCGCCCTGGTTCCTCCTCGTCTCCGCCGCTAGTCTCCAGCTCCAAAATGGCGGCTGCCACTGTGGGGCTTCTGCCGGC
FP015949	chr17	69244647	69244898	-	1	train	TATAACTAGCTATTATTTTATATAACTAACTATAATATGACAAATCCTTTGCTAGTAGTACTAACAACGTTTTATAGGAGCACAATTAATTTTACTTAGGATAAGTGTTGTTATTATTGTTTTTATTGTTGTTCTGTTAGTTACTCAAAACTTCATTCTAATTGTGCCCTGAGTTTGTTAAAATACCATACTGTATTTTTGTGTAACATGTAAATAGGCATTAATTTTTGAGAAATAGAAATGTTTATCCT
FP017835	chr20	462465	462716	-	1	train	GCTGGTCTCGAGCACACTCGCTCAAGAGCGCCACACTCTACACCGTAGAGTCTACAGCGCAGACGAAGCCAACTCCTCAGAGCGCTCACCGCGCTCTTTCCGCAGCCTCCCCCGCCGCCGGCCGGGCGTAGCGCAGGGTCACGTGCCGGCGCGGGGCGGGGCCAGCGGGGCGCCTGCGCGCCTGCGCGCCGCTGGCCGACGGAGGGGAGCCTGCCGATGCCGAGCGGGTGCTACGTCCCGCGGTCGGAGCC
FP015075	chr17	1485477	1485728	-	1	train	CAGGCCGGCCGGGCAGGATGCGCTACCGGGCGTCGGTGAGCCTGGGGGCGGCGGGGCAGGGCGGGGGGTCCCGGGCGTCCCTCCCCACCTCGCGGCGTCGCCTCGGCCTCGCCAGGAAGCGGGCGCGGCCCAGGAATTCCGGGTGGAAAAACGTCCCGGGAAGTTGCAGGGGGAGGAGGCCGAGGCGGGGAGGGCGGCGCGGGACGCTCGGACCCGGGGCCGGATGAGGAGCCGCCGGGGACGCCTGAGAA
FP002623	chr2	112275396	112275647	+	1	train	GTCTCTCGCCGCCCGGCCCCGCCACTAGGCTGCCGCCTCCGCCGCTGCACAGAGAAGGGCGAAGACCCGCCCCTGCAGGTGATTGACGGGCACTCTCCTCCAATGGTCAACGCGCTCGGCCCCGGCGGGCTGGGCAAGGGCCGCGGCTCGCGGAGGAATCTCTCTTGCCCTTTAGTCCCGGGCAAGGTGTTGCGGCCGGCGCCATTTTCTCGAGCCGCCTGTTTCGGGTGCCGCCATGTTGGTGAGGAGAA
FP000292	chr1	22653062	22653313	+	1	train	ACAGTAAATCCAGTGGGTTGCAGAAATAGGACCTGAAACTGCCTGAGGGCAGCAGGGTGGGGGGACAGTGAGGGGACGGGAAAGGGGCCAGCCTGCTGGTCCATGGGAGGGGACCGTCAGGGGAAAGCCCTTCCCGCCTCTGGGGAAGGGAACTTCCGCTTCGGACCGAGGGCAGTAGGCTCTCGGCTCCTGGTCCCACTGCTGCTCAGCCCAGTGGCCTCACAGGACACCAGCTTCCCAGGTGAGGACGG
FP000596	chr1	42825430	42825681	+	1	train	GGTCAGTCATACTCAGTGGTTCCCTCATTTTTGGGAGCCTCCTCTTTGCTCTGTTCTTTTATCAGGGGCATACAGACAGGGTGTGGCTTGCTGGTTTACAGGGACAGCGAGAAGGGAGTTCTGTGGGGGCAGAGAGAGCCCTGGCCTCTCATATTTCATAAAATATGAACTTTTCCCGGCCCACATCCCTAGGCCTTCCTGATGCGCTTGCCTGCTCCCTGGTCTCTCTGCATGGGGAAGGAGTGTTCCCA
FP011265	chr11	94973505	94973756	-	1	train	AAAATGAGGTTTAGGGTCTACTCTAAGGCACTGATTTCAAACGCCCGCCCCCTGGGCCGCTTCGGCCCCGCCCCTTAGTGCCGTTACGCCGACTTCCACTTCGGCCACTGCGGATTGGTCGTTGCTTCCGCCCCCGAAGCGCTTGGATTTTGCTTCCGGGTCGTAGGCGCTAGCTCTGGGCGCAGAGGTTTCTGGGAGCCAAGAGTGGTAATGGCGTCTGTATGATCTTCGGAGCCTGCTGCATCGGACCT
FP017674	chr19	54100752	54101003	-	1	train	AATAAATGAACACCACAGGTTATGACTGAACCCCCTGCTAATTTTTCCACAGTGCCATAGGGCTATGACACAGTCACCCACAGGCCCCCACCTCGATACTCTCTTCCGTAAATGAGGATCTGGGTCTGGTTTTCTGATGTTGCCTCATTTCCTGGGAGGGGAGAGGGTGCGACCAAGCCCTGGCTCCAGCTCTAGCGGGTATCTGCCCACCATGGCCCTGGTGCTGATCCTCCAGCTGCTGACCCTCTGTG
FP017405	chr19	45340549	45340800	+	1	train	GCTTGGCCAAGGCCCCTGCGCTACAATCACCCTGATTCGAGGTGTCCTGGAGGTTACCAGTTCCGCGTGCGAAGTTCTGCCATTCCCGCGGGGCTGGAGCCCCGAGGCTGAACTTCCCACGGGCGGTGGGCGTGGCCGGCTCACCTGCGGGGGCGTGGCTCCCGGCTGCCCCGCCCCTCGGAGGAGCCGCCCGAGGTCCCAGACGCCCGGCGCAGCGGGAGCGGCGGGGCGTGCCTGGCCTGCGGGACGCG
FP000506	chr1	36149435	36149686	-	1	train	CTGCCCAGCGCAAGCTGGGCTCTACCAGGCAGGCGCGCGCGGAGCACTCCGGGAGTTGTAGTTGACTGTGTTAAGAGGCGGTGCTGCCAAAGGGGCTGTTGGGTGTTGTAGTCTGACGTTTGTTGGGCGGGAGTTGCCCACGTCGGCCGGACCACTAGTCCCGTGGTGCCCCGCGGGAGGCCGCGCAGGCGCAGTGAGTCAGTGCCGGTCGGTCTTGTGGGCTGAGGGGCAGCGGCTTAGGCTCCGGCGTC
FP005247	chr5	6745655	6745906	+	1	train	ATATGGCACTGAACTGAGAGTGGTGCTCATCTGTGTGACAAGCAGGATGGAAGCTTGCTATAAATATTGTGATATAATGCTAATGACCTTTACACAGCTAAAATCAATCCTTTCACTTTTCCGGTTTTATGTGATACTGCCATACTAGTCAGTCTAACACTGACCCCTGTTGGTTGTGCTTCAGCATAACGAAATTAGGATGACGAGAATCTGAAATTACATCTACCATCCAGGTGACTAAGTTATGCAGA
FP007166	chr7	5282717	5282968	+	1	train	GAGGAGAAGAGAGGAATGGAGAGGGAGGGAAAGAGAGGAGGAATGGGGAGGAGGGGGAGGGAGGAGAGGAGGGTGAAGGGAGGAGGGAAGAGGGTGAGGAAGGGGGAGGGGAGGGGGAGGGAGGAGAGAGGAGCGAGAGGGAAGAGGGAGGAGGAGGGGGGAGGAGGAGGGAGGGGCGGGGCCGGGCGCGGGGGGCGGGCAGAGGCGGGGCGTGGTGGCCTGGGCTGTGCTCGTCGCCGCGCTCGTGGACC
FP004971	chr4	112636849	112637100	-	1	train	GGAGGCAGTGCGCGCTCCACCTACGTCACAGGCGAGGTGGTTCCAGGAGGGCTGGGCATCGGAGAGCGCGGCCGGCTGAGAGACTCGAAGCCAAGGAAGAATTTGATTGGCGATTTGAAAGCGGCGGTTTTTGGCTTAGGCGGAGGTGGGCGGGCCAAGTTTTGGGTCCTCATCTGGTGAGGTCCCGCGTTAATGACGTCACGTTGTTTCGCGGGAGAAATTCGAAAGTGGATTTTGTGCAATTTGACAGG
FP010356	chr10	133276817	133277068	-	1	train	TCTGGCCGGACTTGGGACAGGCTGTGCCTGAGTTTCCTCACCTGTGCAAGGGAGGATGCTGGATTGTGGGGAGAGGGGAAACGGACCCCGCCCCCAGGTGCCGCGCGCCCCGCCCCTCCCACCGGCCGAGGGGCCCATTGGCTGCGGGGCGCCGGGGCGGGGCGCGCGGAAAAGAGCCTCGGGCCAGGAGCGCAGGAACCAGACCGTGTCCCGCGGGGCTGTCACCTCCGCCTCTGCTCCCCGACCCGGCC
FP004651	chr4	39458498	39458749	-	1	train	CAGATGCGAATCCTGTTGTGGGCTTTTTGGCCTATTCCCGCCCCTCAGTCTTGCCGGGATGGCACCGCCCGCATAGGACTTCCAGGGTTGGGCTGAGTGGGAGTTCGACTGCTGGGCCTCGTAATTCTCGCTTTGGGGCTGCTCCTTCCAGGCTGGGACACACTGGGGCCCGCTGTCGGTCTCCCGTCCTCCGACATCTTGTCTGGAACTTCCGCCTGGCAGTCTCCAGTAGGAGTGGAGCTCTGTGCGGC
FP009121	chr9	98709037	98709288	-	1	train	GGAAGGTGGCCCTCCCCGCGGCCGCCGCAGCGCCAGCGCCTCCCCCTCCCCCGGCGCGCACGGCTCGTCCCCGTCCCTGGAGGCGGCCCGAGCCGCCCCTGAGCAGCCTCGCCTTCGCCTCCCGCGTTTCCTGCCGTCCGCCCTCCCCCGGCCGAGCTCCAGGGGCTGCCGCCTAGCAGCTCCCGGCGGGAGAGCGGTTCAGAGCGCGCACGGGGGCGGGCGGAGGCGGCCCGGTGCGGGGCGGCCGCGCT
FP014465	chr16	19168052	19168303	+	1	train	CTGGATGGGAAGGACCCTCCCTGCTGCAGACTTCATGGCAGGCTGCACTGTGTCCCCTCGGCTCCACGGCTGCCCCGGGGGCGCTGCTTTCGGTAGGTGCCAGCCCCCGTGCCAGCAGGATGGGCTGACCTGTGTACCGCCCCCTCCCACCCCCTTCCCCATTCCCATCCCCTCCCCCGCCTCCTGCGCGCTCCGGCCCCGGCGTCTCCGGCCCCGCCTGCGCTGGGGTCGCTGCAGTCCCCGCAGCTGCC
FP009358	chr9	128128389	128128640	-	1	train	AGGTCCCCGCTGGGGACTGACACTGCGGGCTCGGTCGCCGCAAGGACTGGAAGGAAGGTGGAAGATGCGGGGCGAAAGCGGAGCGAGAAGAGTGGGACGCGGCGGAGCAGCAGAGCCGGGACTCAGAGCTCCCCCCTCGCCCCGCCCCTCTCCCCGCCCACTCTGCGAGTCCCGCCCCCTCTCGGGCGCCGGGCGGGGCCATCCCGGGGCTGTCCGCGGAGACGCCTATGCGGTGGAGGCTCCGGGCTTCA
FP004036	chr3	126101460	126101711	-	1	train	CCCAACAAGAGGTGGAAGCCAGCCTGGTGTCTCCATAGGGTCAGTCCTGGCTTCCCCGTAAACATCCTTCGTGACAGAAGAGCAGGCAAAGCCTCGAGCACAGGCCCATCCAACTCAAACAGAGCCAGGAAGTGCCTCTCCCTGCTTTTCAGACACAAACAGAGCTTGGTTTTAGCCAGGGCGCCCACTGCAGTCACGCTGGAGACCTCTGGTGCAGATGGGCGAGACTGGACCCAGCTTTCCAGCTTTCT
FP006017	chr5	173989003	173989254	+	1	train	TGGTCCGGCCCTAAACCCGTCCCCCTCATCGCGGTGCCATGGTCCGGCCCTAACCCTTTGTCCCCCCCATCCCCACAGGGTGGTCTGGTCCTAACCCCCCCCAATCCTCACGCGGTGGTCTGGCCCTAACTCCCTCCCCCCAATCCCCGCAGGGTGGTCTGGCCCTAACCGCTCCCGCATCCCTCGCCTGCACAGTGGGCAGTCTGGCGCCTGTGGCGTCGTGTTTGCTGAGGGCCCGGCTGCCGTTGACT
FP007490	chr7	76480366	76480617	+	1	train	TCTGCAGGGCTACTGATGTGCATTGGTAACTGCTCATGTCTGTCCTGTTCACAGATCTGCCGGAGGCGCTGGGCAATGACCCCGGGACTCCAGGCCAGAGGGGTCTGAAGCTGTTTGGGAAAGCAGCGGGACTCCTTGGGAAGATGGCCATGGCCCCAAGCCCTTCCCTGGTGCAGGTGTACACCAGCCCCGCGGCTGTGGCCGTGTGGGAATGGCAGGACGGGCTGGGCACCTGGCACCCCTACAGTGCC
FP016223	chr18	9474808	9475059	+	1	train	GGGGCTGCTCGGTAACTGCTGAGCCGGGGGCGCGCTGAAGCCCTGCAGAAGCCCACACCCGCTTCAGGAATTCCGTGGATGTTCCCGACAGAGGTTTATTCCCGCCTATCCCTACAGGCCCACGGCGCTCTTTCCTCCAATCCCACACCCAGTCGAAGGCGGGGCAGCCGCCCCACCCAGCGGCCAGCCGGGCGGCCCCCAGTCTGGTTTAACTGGTTGGAACGACTAAAGCACGCTGGCGCAAGGAAAGC
FP011586	chr12	6606389	6606640	-	1	train	GTGGGGGCGCCTGCCGGGGAGCCGGCTTCCGGGCTGCCCCCGAGGAGTGTGGAACTAAGTTCAGCCTGAGCGGTTCGGGTGCCCTGCTAGGGTGGCTGGGGTGGGGGGCGGAAAGAGCGGGTGGGTGTGGGGGAGGGGGACAGTGCGCGTCACTGACTGTTCTGTGTTTCTCCCCCCTCCCCCGCACAGTGACTCGGGCCAGTGTAGAGGTCCTCAGGCCGCCGGCAGGAGCAGCTGGGCCAATTCCCTGG
FP016500	chr19	890955	891206	-	1	train	CCACATGCCCTGGGTGCTCAGCCAACATAGACGTCCCACCACACCTCCCTCCCGCAGGAGCTGGTGACTGCCCTCATGTGTGATTTGCGGCGGCCAGCGGCAGGTGGGATGATGGACTTGGCCTACGTCTGTGAGTGGGAGAAATGGTCCAAGAGCACCCACTGCCCATCGGTGCCCCTGGCCTGCGCCTGGTCCTGCCGAAATCTCATCGCCTTCACCATGGACCTGCGCAGCGATGACCAGGGTGCGTC
FP016685	chr19	7533963	7534214	+	1	train	AGGGTGTTGAATAAAAGGGAAAATAAATGTGTCGTTTTCATTTTTAGCGGGAGGAGCAGTCCTTGCGTTAAGCGGTGTGAGGCCCTTTAAGGCGCGGCCACACTCAGCATGGCGGCCTCAGTCGGCCTTCCAAGCATGGCGCGGGGAGGAGGGGTGGGAGGGTCGGGAGGGACTGCGGTGCGACTAGGAGTGAATAATTTAAAGGGGCCGTGCCTGCGGAGCCGGGCGGAACGCTAGCGGTGTTGGCGCGG
FP009908	chr10	72273723	72273974	+	1	train	TTCGACTGCGAGCTTTCTGGGGCTCAATGGAGGCGGGGCCCGGCCGCTGTCACCGGGCAGGAGAGAACGTTGCTTACGTGCGCCCGGAGTCCATTGGCCAAGGCGGGCCACACTCCCGGGTCTGGATTGGGTCGTGGCGCAGAGAAGGCGTGGCCTCGCCGCGCTAGTCCTTATAGGCTGCTCCGCGCTGGTGCTAGGGCGCAGCAGGCCAAGGGGGAGGTGCGAGCGTGGACCTGGGACGGGTCTGGGCG
FP000017	chr1	1280241	1280492	+	1	train	ATCACGAGGTCAGGAGATCGAGACCATCCTGGCTAACACGGTGAAACCCCGTCTCTACTAAAAATACAGAAAATTAGCCAGACGTGGTGGCGGGCGCCTGTAGTCCCAGGTACTCAGGAGGCTGAGGCAGGAGAATGGCGTGAACCCGGGGGGCGGAGCCTGCAGTGAGCCGAGATCGCGCCACTGCCCTCCAGCCTGGGCGACAGCGAGACTCCATCTCAATAAATAAAAAAAAAAAAGAGTTGTTATCA
FP017868	chr20	2840544	2840795	+	1	train	GGTGGCCGCCACAGGGCGCAGGAGCCAGGGCTGGGCCCACTTGGCGGGCGCGAAGCCCAGGTACGGGCTGGGGTCTCGGGGCGGCAGCCAAGTGAGGCTGCCCACAGTGGTGATGGCCGCGGCGGCGGCCTCGCCACGTGACCCGGACGGGCGGAAGCGGAAGTGCCGGCCTGGAGCCCGCTCTAGGTGGGTGTCCCCTCGGTGCTTCCCAGCTGCCGTCTGCACCAGCCATGGACTGCTACACGGCGAAC
FP018612	chr22	20966975	20967226	+	1	train	GGCTGGAGTCGGGGGTAGGGGGAGCTTATGGCTGCTGGGCACAGGCGTGGCAGGGCTGTGTGCCTGGCTTCCCTTTCCCCTAACCCCATCAGGCCCGGGCTGGCCGCCCACTTCTGTTCACCACAGAGCAGGGTTTCCCTGGGGACCAGTGGACTGGCCCAGCCCCACACCGCCCCCCAGCCCCTCCTGGCCACAGCTTGAGCTGGGAGGAGGGTGGAGGCTGGGCCGCCTCCTGGACCCCTGCCCTCCCT
FP002398	chr2	73214044	73214295	+	1	train	ACTCTGATACAGAGGTACAGACACAAAAAGGCACCGATGTAAAAGCACAAAGGCAGGCCCAGTTACACTCCCGTCAACAAACCACGGAACCTAAATTCAGACTTGGTGGAGCCCAAAACCTCGCAGCCTCAGGGGCACTCTCGCCGGGGGCGGGACCAGAAGGGGGTGTGGCCTCTCAGGTCGAGGCGGGGTTAAGGGTCATAAGGCGGAGGCGCGCCCAAGATGGCGGCCTCCATGTGCGACGTGTTCTC
FP013047	chr14	24095074	24095325	-	1	train	CACACAGTCTTGAGGACAGCCTCTCTCTCCCTGGAGCCTGGCTCACGTTATCTCTTGTGAAGCCCCCACACCCTCTTGCAGCCCTCTAAGTGGGACGTTTCCTTCAGAGACCACCCACAAGGGCTGCCCAAGTGGAACGGGCTGTGCGACAGATGTGGGTCACTTCCGACTCTGGGACAAAAGCTGAGAAGAGAGGCCTCAGAGAATAAATAGGAGACTAAGAGGACAGGCAGACAAACAGGACGAGCCTG
FP004006	chr3	122793746	122793997	-	1	train	ACTAGAAAAGGAGAAACAGAAACGAAGCTAAGACTACAATAAAGAACGCTAAGGTCCGGGTTAGGTAAAAAGCGCCACAACTCAGACCACGTGTTTGTATTTCCACACCCGGGGTATCAAAACAAAGCTGCGTGGTCCCGCCCAGCATCCGTCACGTGACCTCCACGTGAGAGCCGGCGTTTCCGTAGGAGCCGGGCGGGAGTCGCCGGGGCTCCTTCCTGTGGTGCAGCTTCGGGTCTCGGAGTTTGGCC
FP005498	chr5	78294641	78294892	-	1	train	CGGGAGCCGGGCAGTTAGCCACCCGCTGCCTCCCCTTCCGCAGTCCGGCTCGCAGCCGCTCAGCTACCTCTAAAGCCCGGAGCAGGAAAGTCCGCGCAGGCCTGGATGGCCCGGGTGGTAATCTAGCAGGCCTGCACACTGCGCATGCGCAGGGGGTGGACTGCCAGGTCGGCTCAGGGAGCCGTGACGAGTCCGGAAGCGCCTGCGCGCGCTCCTCCGTACGAGAACTAGTTTTGTTCCGTGCCCTCTGG
FP002349	chr2	68053008	68053259	-	1	train	ATCTGTTAGGGTTACTGGGCAGCATCCCCGCAACCCAGGTACTCAGTCTCAGCCACACCTTACTGAGTGGTGAAGAGAAGCAAGTTTTCTCATGCCCTCTCCGTCTGTCTCTACCTCTGCAGCCAGTGTTTTAGAAGGAGTATGTGGAAGGGCAAGGAGTACCTATAGAACCCCGGAGGAGGGTGAGGAGCAGAGCTGGGTGAGTTCTTTTCTTCTTTGGAAATTTTAACATGTACAACTTTAGGCATGTT
FP018120	chr20	43507512	43507763	+	1	train	TCCTAGGTTGAGGGGGCATGAGGCGCGCACAACCAGGGCCTCGAAGGCAGGCAAGAGGGCGCGGGCGCCACAGAAGCCTGGGCCTGCGCCTGGGCCGGGCAGATGTGCCCGGCGCTCGGGCCCCGCGCTCTGCGTCCTGCAGGGAGGCGGACCCGCCCCCCGAACGCGCAGGCGCGCGGCGCCCGGCTCGGACCGTAGCTAGGCGCTGGGCGGCCACCGGCTGGCCAGGCAGGTAAGCAACCAGCGGTGCG
FP016045	chr17	76737477	76737728	+	1	train	GCGCACAGCCCGGCGGGCGGGCCAAAAAGCGCGGAGTCACGGCTGGAGGGAGGGGGAGCGGAATTAGCGGGCAGTTGGAAAGCCCGCGAAACGCTTTTTCCGCCTGGGAGGCCGGACGATCGCGATTGGGCAGGAGGAAGAGGAGGTGCTCCCCATCTGGGCCCCCACCTTTTTTTCATGCCAATACTCCAAGTTCACGCCCCTTTTTTGCTCAGCCGTCAGCCCCGTCTCCGTCTGAAGAGTGCTTCTGC
FP018356	chr21	25735602	25735853	-	1	train	ACCCGTCCGGCACGGGTCCCGGCGGTGCCCGGTCCCCCCGGGCTCCAGGCCGCTCGCCCCCCGCCCCTCCCCTTCGCTCCCCGCACAGCTCTCCCGCCCCTCCCCACCCGGCTCCCTCAAGTCTCCACGAACGGAAATGGGAGTCACAGCGCCGGGGACCAGCGGAAACAAAACACAGAGGCGGCCGCGGGGGGGCCCTTGTGGGGGAAAAGGCCCCGGAACGAGGCCGGCCCGCGAGGGCCCGCGCTCCC
FP017057	chr19	21767538	21767789	-	1	train	AGCTCAGCTGAGGGAGGAAGCCCTGTCAGAAGAGGGGGCAGTCTAGGCTGTCACTCTTTCTTCATTGAACCCGCATCTGATCACATCTTCTGTCACTCAAAGCCTGACAGGGCGGGGATTTAAGCATTATCCAACCAGGGACGCTGGGCTGGAAACCGTCCAATCAGGCATGCAGCTGGAGCGGACTGGACGGCTTCCGGGTTTGGCGGGGTCTTTGTCTCTCGCTGTAGCCGGAGCTCCAGGTTTTGCTC
FP017561	chr19	49705759	49706010	+	1	train	AAATACTAAGTAAATAAAAGTCCTAGGGTGCTGATGATTACTAGGGTACTTTGAACAACTGACGACACCTAAGAAGTATATAATAAATGGTCACTATTTTTATGTGTCTGGCCTTCCTGCCCCCATGCTTCTGCCAACGCCACCCCTAGACTACATCCGCCACCTCCATGACAGCCAACACGTGGCTGTCTTCCACCGGGGCCGATTCTTCCGCATGGGGACCCACTCCCGAAACAGCCTGCTTTCCCCGA
FP018589	chr22	19950494	19950745	+	1	train	GACTTGTTTTGCCCTGATGGATGACTACATGTTTCCAGTGAGCTCAGCTGGTTGCTTTTGTGGGTCCCTGAGGCACTGCGAGCCCCTGATGGGGCAGGAAGCCTAAAGTTCCAGGGACCGTGGGGAGTTGAGGGGCTGGAGGGGGAAGAGAGAGGCCTGCAGCGGGGATCCACAGGTGGAGGAGGCCAGACTTGAAGGGGGTGTCTCCTTCAAGAGAAAGTAGGGAAGTATGGAGGTGATAGGAAGGGAGA
FP007252	chr7	24757384	24757635	-	1	train	ACACGAAGGAGGGGAAGCGGCTCTCTCTGGGGCTTCTGGGAGGTCTGGCCCGGGCCCCTCCCGCAGCCTCCGGCGGCCAGTCCCGCGGCTCTGGGGCGCCCGGACCGAGCAAAGGTCCCGGCGGGCGGGCCCTCTCTCCGCCCTCCCCAAGCGCCCGCCCCCCGCCGGCCCAGCGCAGGCTCCTCCGACTCCGCCAGCGCAGGGGGCGCGGCTGCCGGGGACCCAGCAGCGGGCGCGCGGGTAAGTGCGCG
FP017157	chr19	36032816	36033067	-	1	train	TTTGCCGTTGTATCTGCTCATCTCGCTCCTTTCCGTAGCCATCTCTTCCTCTCCCAGTGCCCGGAGGCTCTCCCCTTTCAGTCGCCGCCTCTGGCCCCTCCATCAATGCGAGCGCCTCCCGGTCCCTCCTCCTGGTCCCTCCTCCCCCGCCGCCGGTAGGGGGGCAGCAGGCGCATGCGCAGTCGCGCCCCTCCCTCTCCGCCCCCACCCCCTGTCGGCGTCTGGGCCTCGTCCCCTTCTCTCTGTCTCCC
FP018436	chr21	36990160	36990411	-	1	train	AGAAGCGCGCCTCCCAGGCAGCGTGGGCGAGAGAGGACAGGTGGCCGAGAGGCAGAAGCTGGCTAGCGGCGAAAGGCCCGCGTTCCCGAGCGCGCGGCACCGCGGCCAGGGCAGGAGGAGGCCAGGCGGCCCCGCGGCCCTGCGCAGAGAGCCCAGTTGCACCACCAGGCGGGCCACGAGAGGGCGCAGCGCGGCGCGGCAGGGATTCGCGGGCGACCACCCGGCGCAGGAGCGGCCGCGTTTCGGCCTCA
FP010885	chr11	62727425	62727676	-	1	train	GTGGTGGGAGGGGCATGGCATCGGGGAGCTCCGGGCGGCTGCGCAGAACTCCCCGGCTGGAAGTACTGAAAGACCGGCGCGCACACGCTTCCGTGCTACGAGGCACTTGAGCTGCAGCCAGCCTTGCCCGTCTTGGACTCCACTGCGGCTGCGCGATAAAAGCACCGGCGGCGATTGGCCCGGGGGCGCCAGAGGGCGGGAGGAAGCGGGAGGAACTCCGTCGTCCTCCGCGCGCGCCTGCCCGCGGCTGG
FP002296	chr2	55519517	55519768	+	1	train	TTCTTTTTTAACCAATCAGAAACGTTTTTCAAACGTTTGCCCTCCCCGCTAAGTGCGCAGGCTCCTGGAAAGTTGTCTTCACTTCCGGGCCGCGGTCTAGGGCGGCTACGTGTGTTGCCATAGCGACCATTTTGCATTAACTGGTTGGTAGCTTCTATCCTGGGGGCTGAGCGACTGCGGGCCAGCTCTTCCCCTACTCCCTCTCGGCTCCTTGTGGCCCAAAGGCCTAACCGGGGTCCGGCGGTCTGGCC
FP005221	chr5	443079	443330	-	1	train	CCAAGGACCCGCCGGCCGTCAGGACCTGCCACCGACTCACCCTCTACGGCTACGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCCCCGCCGCCCCCTCCGCCTTCGCTGCCTCCGCCTCCGGGGTCCCGGCCGGAAGTGGACAGCGCGCGGGCACTTCCGCTTCCGGCAACGGCGCCGGAAGCACGGGTCCGGGCCCTCCCGCGGGGCGGGTACTCGTCTCCCGGCCGAGCGAC
FP003241	chr2	233549095	233549346	-	1	train	TAAATAACAAACAGTGAAGTTTCTTATTAATTTCAGTAGTTGAAATTATTTTTTCTTTTGATTATTATCAGCATTTTATGAAAACTATATCAATATTTTTGTTCTTTTTTCTTTTAGAGTGAATTGGATGACTTAGAATATATATATGACCTCTTCTCAGTTATTATACACAAAGGTGGCTGCTACGGAGGCCATTACCATGTATATATTAAAGATGTTGATCATTTGGGAAACTGGCAGTTTCAAGTACG
FP007055	chr6	155148949	155149200	+	1	train	TAGTGGAATTTCGAGCATCCCTATCTTAGTCAGGTTTCCTAGGAGACAGACTTTGATAGTGATTTAGATGTAAAAAGTTTTACTGGGAATGCTCTCAGGGACAACGCCAGCCTGGAGTGAGGGAGCAGGATCTGGGAGGGACAAGTTGCATTGTGATGCAGCTTGTCCCACAGCGAGCTGTGGAGCTGGGAGGGCCATTCAGAGCTCTGCATGGAGGCAGGGTGGCCGGCCTTCGTCCCCTCAGGCCAGCA
FP009658	chr10	18651533	18651784	-	1	train	TTAACCTGAGGCAGGTGAAAGAGCCTCACGCAGGGTTAGGAACCTCCGGGGGCGTGGCGTAGCGTAACGCAGTGTTGGGGAAACTTGCGGCTGTGGATTGGGGGACCGGAAAGGGAGGGGTGGGCCGCAGGCGCCGGACGTGACGTAAGCCGGAAGCGACTTTCCGCCGAGAAATAGGGGGCGCGTGTTTGGAAATTGATAGAAAAGATAAAGGGACCGAGCTGCTGTCAGCCTGGCTTACTGATCTGCGT
FP006423	chr6	33073598	33073849	-	1	train	CTAACTCTGAAAATGAACTGTGAACTGGAGCTCTCTTGACCACGCTGGTACCTAAAATTCTCCCATCTCTTCCCCAGCACCTTCCAGCGTCCTCTTTACCCAGCAACAGAGAATGTCAGCTCTATGATTTCTCTGATAGGTGAATCCCAGCCATGCTGATTCCTCTCCACCCATTTCCAGTGCTAGAGGCCCACAGTTTCAGTCTCATCTGCCTCCACTCGGCCTCAGTTCCTCATCACTGTTCCTGTGCT
FP018584	chr22	19525286	19525537	-	1	train	GACCTTTCGAGAAATGGCTGGGCCATTGTGCAGAAGAATGCCCGGAAATCCCGCGCCTCCCTCCTCCAGCAAGGATGGGGGCTCTTCCTCCTGGCCAGGAAACTCCAAGTTGGCTTCCGGAGGGTGGCCTGGGGGCTGGGGTGCCAGGGACACCATCGCCACTGGTGGGAGGGCAGGGCACAGCCCCTCCGTGTCCCTTTGTCTCTCCTGTCTGAAGGCCAGAGCAGGCTGCTAGGCCTGGGGCCACCACT
FP006861	chr6	116528845	116529096	+	1	train	CAATAGGCTCTTACAATTAGGTTTTGAGTTAGCCTTTGAAAAACAGGACGAGCAAAGAAAGATTTTATTTTTAGGCTTATCTGATAAGTTTCCTTTGTTGTTGTAGTTGTTGAGGGATTACAGGGATTAGTAATCTTAGAAAAATGTTGAGAATGAGGCATATACGGTGTTTTACCCTTGGGTCCTCCCCTGCCTACACTGTGCTGAGAACTGCTCTCTCTGACCGGGTCTCCTGTTTCTACCTGGTTTGT
FP011189	chr11	77174242	77174493	+	1	train	AGCTAGAAAGTTGTCTGAATAACCGCAGATCTACAATACCTGCAACGTGAAAGGTCCTCCCTGGGGCTGTGCAGAGCACCCCTCACCTGCACAGCCATAACTTGGCCTGGACGAGCCTCTCTAGCTCCCTCAAGATTCCCAGCAAAGTCCTGCGCTTTCAGGCTTCCAAACCTTTGCACAGGCTCACCCCCTCCAGCCTTGGAATACTCCCCCTCCTGTCAGGGTTAGGGTGATAAAGACGCCTCCTCCAT
FP002497	chr2	95025507	95025758	+	1	train	AACGCAGGAGTCAGACCCACAGTCCCCAGCTCTGGACGCCCGCAGCGGGGCCTCGAAGAGGTTCAGGGCGGTGCCCGCGGCGCTCGGGCCGGGTCTCCCGGGGCGTGGGGCGGGGGGCGGGGTTGGGCGGCGGCCGGGGCTCCTCCCTCTTCTGCCCCGGGCTCCCCTGCTCTTAACCCGCGCGCGGGGGCGCCCAGGCCACTGGGCTCCGCGGAGCCAGCGAGAGGTCTGCGCGGAGTCTGAGCGGCGCT
FP018345	chr21	15065899	15066150	-	1	train	TGAGATGTCTGGGGAGCGTTGAGGATACGATTTTTCAAACAGCAAATACAAGGATAAGGGGAAATTATGGTTTCGTATGACTCTCTCCTCCTTCCACTTTCTCGTTTCACCAATCTGTGGCCCCCCCAATTTAGAGGCGGGTTCAGTGAGATTAGGCTGCAGGAAACCGAGAGGCTCCACCGCGCTGCCGCCCCGGCCTCGTGATGTAATTGTTATTTCTTATTCGGAGACCGAACCGAGGAGAGCTGCTG
FP008452	chr8	90791574	90791825	+	1	train	GCCGTGCCGGGGGCAGAGCCCGCTCCCCTCCGCGCGTTGCTACGTGCGGGGATGACATGGATCCAGACGCCGTCAACTACGGGAAGGAGGAGGCGAGAGTTGGAGCGCGTGGGAGCGCAGAGGCCGGAGGAGGGAGGGGGGGACACCGAGCGCGGAGAGCGCGGAGAGCGCGGAGGGAGGCGCGCGCGGGAGCGAACACCCTCCCGGATCCAGAGCCCGGCGGCGGCGAAGCAGCAGCTGCGGCCGCGCCC
FP002240	chr2	44168674	44168925	+	1	train	AAACCCAGAGGCGGGGCCGGGTGGGCGGGGTAAACGAAGTACGGAGGTGCCGAGGGAGCAGCACGGGGAGGGGGAGCGGTGATTGACAAGACTCCCAGCAAACCGGGGGCCGTGAGAGGGCGCGCGGCATCCCAGTGGGTGGCAGCTGGAGGCGGGGCACCCTGAGGGGCGGGGAAGGACTAGGGTGCGGGGAGGGGGTTGCAAAAGGAGCCGAGCGGCTTCTGCTCAATGGCGGAAAAGCCGCCGGTGCT
FP003189	chr2	227328478	227328729	+	1	train	AATGAATAGTTGTAGGACCCCATCTGTGGAACTAAGATTGGTGGTAAGCTTAATTTTTTTCTTTCTTTATTTTCTTACTGGTCTCTTGATGTCAGGTTGTGCATCTTTGTGACTGTTACTATTTGGAGAATTTCCAGTAATGAATCCCTAATAGTGGTTTCATCTAGTTTCTCAATTAGCCCCTGTTTTTCTTCCCCAGGGACAAAAGTGGCTCTCAATCCAGCACATGCACATTGAAGCAAGTTAAAGGA
FP018837	chr22	38272723	38272974	-	1	train	GACTCCGGGGCTGCGGCGCCGCCCGCCCCGCCCGCAGAGTCCGGCTGCCGCGCATCGTCCGCAGACGCCGCCACCGCCATGGGCTCCTGAGGTATCCCGCCTGGCCGGCCTCGCCCGGGAGCCCCAAGGGGGAGCTGCGACTCCCGAGCGCGGGGGGCTGCTTGTTTCGGAGAGGCCCGCGCGAGGGACGCCCGGCTTCGGCCGCCGGCTCCCGGGCGGGGAGGGCACTACGGAGGCCGGTGCGGCCGCGG
FP006139	chr6	3456393	3456644	-	1	train	GCCTGGCCCGGCGCGCTGTGCCCGGGGCGCCCGCGGAGCCTCCGCCGCGCTCTATGCGCCTCTGCGGGAGCCGCGGGCCCGGGCCATGGCCATAGACCGGCGGCGCGAGGCGGCGGGCGGCGGGCCTGGGCGGCAGCCGGCCCCGGCCGAGGAGAACGGCTCCCTGCCGCCCGGGGACGCGGCGGCCTCGGCGCCCCTCGGGGGACGCGCGGGCCCCGGCGGCGGCGCGGAGATCCAGCCGCTGCCCCCAC
FP018952	chr22	45009802	45010053	-	1	train	GAGGGTCCCGGCACCGCCCCCCGCCGCCCCGGCCCGCTCCATCCGGGCACTGCCGAATTAGCATCGTGCCGAGGCACAACTTTGCCGAGGCCCAACGAGATCCAGGCGCGCGCGCCGGAGGAAATATAGTCCCTGCCCGCGGGAGGCTCGCGCGGGAGGAGGCGGCGGCGGCGGCGGCTGGGCGGGCGCGCGCGGGGCTGCGCGGGGCTCCCGGCGCCGGGGGGGCCATGCTCCGGGCCGCGCCGGCAGCC
FP008981	chr9	75088356	75088607	+	1	train	TCCGGCTGGTTCCGCCGCCGTGCGCGCCGCCCGCGCTCCTGCTGCGGCGAAGCTGCCAGCCCGGGAGCCAATTAGCGCTCGGCTAGGCGGGGTCCACTTCCTCGTCGGCCCGGCGCGCGCTCTGGACTGCTCCGGCGGCCGCGGGGCGGGGCGGAGCACTCGGCGGAGCCGCTCTGCCTGCGTCCGCTCTTCCCGCAGCCAAGGGTGGGCGCCGGTCCTAGGAGGCGCACGGTTGTAAGCCAGACAAAAAG
FP007846	chr7	130668672	130668923	-	1	train	GCGGGCGTGGGCTCCCGGGTCGCAAGCCGGGAGGGTCGAGCCGAAGTCTCCGGCGGGCCGCAAGAGGAGAAGGGGGCGTGGCTGCGTCCCGGGGGCCGAGGGCGGCCGGGGAGGGCGGGCCTTGGCCCCGAGTGAAGGCGCGGCGGCCCAACCGGCGTGAGAGCGCGGGGCCGGCCTTCCTGCAGCCTCTTCCGCTCGCCGGCTGCGGCGCCTGGGACGGTTGCGGTGGGTCTGGGCGCTGGGAAGTCGTC
FP016273	chr18	23662855	23663106	-	1	train	GGAATGTGGAGCGTATGCACAATGTGTGTGGAGTTTGGAGTGTGTGTGTGGACGAGTGAGTGTGGAAGTGTGGACGTGTGTGGAAAGTGCACAGTGCGTGCGGAGTTTGGGGTGTTTGTGCACGCGCGTGTGTGGCAGGTCCTGGAGCGCGTCCACGCGACACCCCATGTGTAGTGTGCGCGCGCGCGCGCCCCCGCGCCGCCCCTTCGGCCCGGCCCTGTGTGCGGCGGCTGCTGCCGGGCCGGGCGGCG
FP012606	chr12	133037191	133037442	+	1	train	TTTGGAGTCACTCCGTCTCCAAGCGACAGGCACTTCCGGACGCGGGGCACTGTGGGAAGTTGAGTTCGCGGGGCTCCTTCCTTCGCGCATGCGCACATGCACTCTTGGTAATGCTTGCAGGTGGCCCGTTCTTCCTCTTTGTCCTTCGCTGCCACGTGAGAGGGGAGTGTACGCGTGTGCGCGCGTGCGAGTGTGCGCGTGTGCAGTCCCCGACTCGTCCCGGCCCGTTCGGAGCCGCCCGGGAGCGCACA
FP003663	chr3	49723900	49724151	-	1	train	CCGCCTTTGGATCCGCCAATGGCCAGCTGACCCGCGGGTCCCGGCCCGGGTCTCCATGGCAGCCCGGCGTACTCGGGCGCTCATTGGCTACGTCTGTCGCAGCGCTGCGCTCCAACGTTCCGACGCCGGCGGCCGTGCCACGTACGCGGCAGCCGATTGGCCCGCGGTGGTCGCGTCACTGCCCGCGCCGGGTCCGGCCTGAGTTCGGGGCCAGCAGCCGTCTACCCGGTGTCGCGTTCTGTGTTGTGGCG
FP007657	chr7	100656373	100656624	-	1	train	CTTAGCGACGGTCTCTAAGTCCCTGCTAGGTGCCCCCTGCCCCATCTCCTCCTGGCCGCTTAATAGAGTGTATCTTCCCCTTTAAAGCAGCTGCCCCCGGTCGGTCATTGTCTCGCCTGGCAGCCAATCGGCGGCGCCGTTGGAGCGGGGGATCCGGCCTCCCAATTGGCCGCTGAGAGTCCCGCCCCGCCAGGGATCCCGGGAGCTGTCCGGCCGCCTCGGTGCTGATCCCGCCACCGCCCACGGGCCGC
FP005864	chr5	143404419	143404670	-	1	train	AGAAATCCAGCTCGCTGGAGGTTTTGCATTTGGCGTGCAACTTCCTTCGAGTGTGAGCACATTGGGCGGGAGGGGTGGGGGTTGAACTTGGCAGGCGGCGCCTCCTTCTGCCGCCGCCGCCGCCTCGCAGACTCGGGGAAGAGGGTGGGGGACGGTCGGGGCGCGGGGGAGGGTGGGTTCTGCTTTGCAACTTCTCTCCCAGTGCGAGAGCGCGGCGGCGGCAGCTGAAGACCCGGCCGCCCAGATGATGC
FP010918	chr11	64226080	64226331	+	1	train	GAGGCCGAGCCCCGCACCCCAGATCGCTGGTGCGCCCCGCAGGGTGGTCCGGGAGGCAGGGCCGACGTGCCGACGGACCGGGCGGAAGCGTCGGGGCGGCGGGGACAAACCTCCAGGATCCTCGACCGCGGCGGACCCGCCAATGAAAAGCCGCGTGGAAAGGGGGCGGGCACTTCCGCTTCGGGGAAGGGGCGGAGCCTGAGGGACCCGGCGGCTGGTGAGCGCCCGCTGGAGGCTGGAGCTTCCGGGCC
FP000743	chr1	54053361	54053612	-	1	train	CCTCTGGGCCACAGCGAACCACATTCCCCAGAATGCACTGCGAAGAAATGCGGGCGGAAGGCGCGCTGAGGGCGGCTGTAGTTTTCCGAGACCAACTTTCCCGACAGCACCGTGCGCCTCCCGCCTACGGAGAACTACATGCCCCAGCCTGCCCCGCGAAGGGAAGAAGTCAGGAGGCCCCGCTTCGCGCTAACGCTTGCGATGGTTGAATTCCCCTCCTCACGCCAGCCTAGGAGAAGAAGTTCGTAGTC
FP004297	chr3	179148156	179148407	+	1	train	CTTCTGCCGGAGGAGGGGGGGGGCCGAGGGGGTGGGGAAGAGTTCGTTGTTTGTTTACACGATGTGAGCGGAAAAAGAGACCAATAAAGTTTATTCTGGAAACAAAAGGAAAAAAAAACAGGGGCGACGGAGAAAGGAGTCGGGGGCGGGGGCGTGTGGCGGGGGCTAGCGAGGAGAGGGAGCGAGAAGTAGAAAGCGGCAGTTCCGGTGCCGCCGCTGCGGCCGCTGAGGTGTCGGGCTGCTGCTGCCGC
FP010855	chr11	61813270	61813521	-	1	train	GAAAAGTTCTCCAAGTCTGCACTCGACCCAGGAAGTCCATCTGGCTTCACCTCTCACTTCAACTTGGGTACAGCCTTCTGGCGGGCAGGAGGATGGCCTTTGGTGCGAACACTGCCGGAGTCCAGGGGGCTGGCTCCCTCACCTTTCATCTTCTCCCGGCACTTGCAGGATCCCTTTGTGGCCTTCCACATCAACAAGGGCCTTGTGAAGAAGTATATGAACTCTCTCCTGATTGGAGAACTGTCTCCAGA
FP005351	chr5	43192024	43192275	+	1	train	GCGCCGCGCGTCCCCTCCCTCTCGCCCCTCCCTCCTGGCTGCCATAGAAACCAGCTTACAAGGTCCAGGCCGGCGTCCGGGTGCACTACAGGCCCCCCGTCCCTCCTTCCCCACGCCAGCCCCTCGCTCCGGATTGGTCCTTCAGGAAACGCTGTGGGGCGTCTCGGTCCCCGCTGCGGAGCCGGCCTAGGCCGGAGGGCGGGGTTTGCCCTGGGCCGCTGCCGGTCAGGTCGGCCGCCCCTGACAGCTCC
FP004320	chr3	183253052	183253303	+	1	train	ACTGACGAGCGGCAGGGAAACACTTCCTGGAATTCTCATCTACAGACAAGAACAAACTGGGGCGGGGCCCATCACCTTCACCTACGCGCCGGGAGGGTGGCGGCTGGCGGGCGGGGCCGGGCTCGGGCCGTGACGCCGAGAGTGCGGGGCGCGCGGCTGGGAGCCTCGCGCCCCCGCCCGGGCCCGCCCCCATCCCGCCCGCATACAGCCCGCATCCCGCCGGGGAAGCGAGCCCAGTCCAGCGCTGCCCG
FP017748	chr19	55675066	55675317	+	1	train	GGCGGAAGTGACGTAGAGGCGCCTAGCAACGGGGCCCAGCGCGCCGGAAGTGATGCCTTCCAGGTTGGCTTGGCTGCGGAAGCGACTCCTGCCAGGGCGGGGTGCGGCACGGGAGGGCGGGGAGCGCGGCAATTTCCGCTTCCGGTCCGTCGCCTCCTTCTGTTGCTTCCCGTCTCCTCGGCGGCTCCCCTCCCCCGCCCGGCTCTCCGCGCCCCTTCTGGGCGGCGGGGCGGCGGAGCCGTCGGCGTGCG
FP018772	chr22	32475110	32475361	+	1	train	ACCTGGGGGTACAGGTACGCTGGGGCCGGGGCTGGGCGGCCCGCGGGGAGTGGGGAGTGCTTGGGGTGGGTGCAGGGCGGGTTGGCGTCATCTGTGGGCTGCAGGCGCGGGCGTGGCCGGGCGATAGGCCAAGTGCGGGGACGCCGGGGGGGCCTTCACGGGAGGCCCGGGCTCTTCCGGGCGTCGCGGAGCCGGAGGGTGCAGGCGACGGGAAGCGCGGGTGGTCGGCTGGGGTCCGGCTCCTGGAGAAC
FP016463	chr18	76495158	76495409	-	1	train	CCTCCCCGTGCCGCGCAGCCCCCCGAGCCCGGCTCGCGCCGGGGCCCCCGCGGGGCGGACGCGTCCGGGGCGCGCGCGGGCGGCGCGAGGAGGGGGCCGGGCGCGTGGCGGGCCCCGCGCGCCCTCGCGCGTCCCCCGCGCCCGGGCGCAGCTGGCGCGGTCGGTCTAGTGCAGCGCTCGGCTCCGCGCCGCGGGCGCTCGGCTTCACCTTCAGATGCGCGGGGCGTGCGCGTCCTCCTCCCCAGGCCCGC
FP005934	chr5	154445796	154446047	+	1	train	CTTTCCTCTTCCACTTTCTCTACTTTGAACCCATCCCAGCAGAGTCCAGGAAGTAGCCTGGCGCCGGAGACTGGCGGCGGCGACGCCCAATGGGACGCGCCAGCGGCGGCGGCAGCCCCGCCCAGAGAGCAAATGAGACCAATGGTCGCGTGAAGGCGGGACTTCCGCTGTCCCGCGAAGGGCGGGGGGAGCGAACTGTTGTGGTGCGGAGCGTTCGGCGGGCGGCGGCCGGGCGGCCCAGGGGCTGCCGC
FP003782	chr3	58491920	58492171	+	1	train	CCGCCCGCTCTCTGGCTTCCAGCCCGCGCTGCCGTTAGGGGGCGCCCGCGCTCGTGGCGCCGCGGCTCTGCTGGGAAGGCGCCGCCCGCGCCGGGCAGCTGGAGCCGCAACTCCGGGCGCGGGCTCAGTCGTCCCCTCTGCCGCCGCCGCCGCCGCCGTCGCCGGGCGCGGGCTCGCTTGTCCCCGCGCTCGCGCTCTCCGGCCGCGGGCATCTCCCGGCCCGGCCGCAGCAGCCGCCGCCGCCGCGCAGT
FP002414	chr2	74391796	74392047	-	1	train	TGTGCCTGATTGATGTAATCCCGGCTGCGGTCGGTGCGCAGTTTTCAGTCAGGGCTGGACTGCGGGAAAAGAGGGTTCACTCGACCCAGAACGCCTCTGGGCCTCTGGGCCGAAGGGTTCTGGGCGGTCCTGGAAGGGTCTGGCGGCGAGTCGGGCAGCCCACGAGGGCGGGGAGGGCGGGGCCGGCCCCGCAGGGTCTTGGGCGTCGGTCGTCGGTGGGGTCGGAGCTGGGCGCGGAGCCCCTCACAGTG
FP015237	chr17	7614823	7615074	-	1	train	AAGGCTAATCTTGTTCTGCGCAAGCGCTAGCAAGCGACCAGGAGGGGGAGTGAGAGAGTGAAAGGGAATGTGGAAGCCCCGATGCGCTAGCTGGCGCTAGCTGGGGCTCTAACCTTGACGCCTGCGCGGTGGCCCTCGGGGCCCCGCGCGCAGCGGGCGGGTGCCCGGTGCGCCTGCGCAGTAGGCGGCGGTGGCAGGGGGAGGTGGAGGCTGTGGAGCGGCAGCGGCAGCAGCGGGTCCCGGGACTGAGG
FP014724	chr16	56451263	56451514	-	1	train	CCTACCCAATGGGAAGCACCAAAGTAAGAGAAGGCGGAACCTATACCCGGTTTACCAATGGGAGCTCCAGCTATTGCAGCCATCCCACCAGCAATTGGCCCACTGGAGCTCGGTCTGGCAGGCCACCGCCCCTCTACTTCCTGCCGCCAGAGGCTCCGGGTTCACTTCCGGCGTGCCTACGCCTCCTCTTGCGCTGTCCTGTTAATGGCGGGCAGTAGCCGCTGAGGGGATTGCAGATAACCGCTTCCCGC
FP017608	chr19	51366292	51366543	-	1	train	GAGATCCGCTTCCCATTGGTTTGCATTGCTTGCCACACCTCCCTTCCTTCCGGCCTCTCGATTGGTCCTCTGTCGGGGAGGCGGGCTCTGCAGCGGTTGACTGGCTGACCGTGCTGCGCGCAGGCGCAGAGAGGGGCGCGGGGGGCGGGGGTGGTGGGGCTTCTGGACTGAGCCGCTGAGGGTGCGGGCTGACCCTGTAAGTGGCTGCGGCGGGAAGATGGCGGAGCTGCGCGTGCTCGTAGCTGTCAAGA
FP005545	chr5	93741329	93741580	-	1	train	CGGGTGCCATTCCACCCATGGCCGGACACAGGCCCTCAAACCACTTCTGCCCCCTTCCAGGCAGTGGTGGGGGCGGCCCCAGAGGGCCGATGCCCCTGCGGGTTGACACTCTGACCTGGTTGAGCACCCAGGCGGCCCCTGGCAGGGTGATGGTCTGGCCGGCAGTCAGGCCAGGGATCTGCCCAGGCCCTGACGTGTGGAGGATTCCCCTGGGTCCCCTGCCACACGAATTCCGGGGCTGGATAGCACCC
FP010777	chr11	47523944	47524195	-	1	train	ACCTCCACCCCTTTAAGAAATGATTTAGTGCACATCTTGAAAGAGTGCAGTAGGGGTTAAGTGTGTTGACTTCGCTGTCTGCCAGGAAAGGAACTGTGGGGAATGCCCTGTTGCTTTGGGCCCCACAGCAACCTATGCTGTTTGCGCTGTTTGAGTGCTTTGTAAAGTTTGGCCTTACAGAGGCCTTGTAATGCCTTTTGGTGTCATGTCTGTTTAAGGAATAGCAGAAGACATGACATGTAGACAAATTT
FP015193	chr17	6831655	6831906	-	1	train	CCGGAGCATCTGGCCCTACCCTTCCCACACGCCACACTCCTGGGTCTCTGGCTCTGGGGTCCGCAGAATTTGCGCCTGCGCTAAGCCTTTGGCCAGGCTGCTCAGTGGCGGAACCAGCCCTCTTTGCCCTGGCAACGCGTCACTTGTTGTCATGGAGACCGCGTTGTTTACCTCTCTGTTCTGCAAGAGCTGGAAGCTCCAGTTGGGGTAAGGAGGCTCCCATCCCGAGCTTGCGTAGTGGTTCTGTGTGG
FP001467	chr1	162412085	162412336	-	1	train	CACAGTTCCAGGAAGCTTGACTTCAGCATTGCCTCCCTAGCCATGAGCATCACTGATTCCATTGTGTGTTTAAATGTGCTTTACCTACTTCTGTGAGTCACATGTGGGCCATTTAGAGGGTGAAAAGTGTTGGTCATTTTATCACCCCGCCCTCCTCATTTCCTGAATCTTGTGAAGCTGGCTGAGTCCCACAGACCGCTAGAGGAAGAGGTAGCTCCACAGGAGGTACAGCTGCTTACACATCTCTCCTC
FP000749	chr1	54764472	54764723	-	1	train	TTACTCAAGGCTGCCCTGGCCAATCAACGATCTTGGGGAAAGCCGGGTACTCGCTCTTTGGCTACACTTCCGGAAAACCTCTCACGCGCGCCTAGCGTGGGGCGGGGCCAGAGCGCCCCTGCCCTCGGCCCCGCCTCCGTGAGCGCGGCTTTGCAATTGGCGAAAAGGGCAGGAGGCGGGGCTTAGGAGCGGGCTTGCTCGAGGCGCAAGCGCGCTGGCCCGGCACGGCGGTGGTCTTGCGGGAGGCGTGG
FP003356	chr3	10115480	10115731	+	1	train	GCGCGTGCACTACAACCTCGCCGCCGGCGTCCAGGCTTCTGTGTTCTATAGCGTGGGAGTAGAGAGCAAACGCCGGCGAGGACGTGACGTTGCACAGGCCTATCATAGTGCCCGAGAATTCGTGGGTGCTCAAGAGGATGGGTGTGGCCTGGCAGCGCAGGCGCACTAGAGGCCTGTAGGGTCGGGGCGCCTGCGCAGTCGCTCTTCCTCAGGCGGCGGCCATGGCGGGACAGGAGGATCCGGTGCAGCGG
FP001345	chr1	155941387	155941638	+	1	train	TAGCGGGGTAACTCTCCCTTCTCCCCTCCCTCTCTGCCATTTAGAGCCCCTCTTACAGGCGGGCGCATGCACATATACCCTGGCATTCAGGCTGTGCCTCGCCCTGCCCCACCTACCACCAATCTTGACCAACAGGAAGGTGGTGGGTTGTCCTTTCCACACCCCTCCCTCTGAGGTGTGGGCGTGGGCCAGGGCTCACCAGAGGCCCCAGAGAAGCACTTAATTCTACAGCCTCCTTCCTAGAGCCTTCA
FP001216	chr1	151156448	151156699	+	1	train	GAGTGGAGGATGCAAATCCCTTTCCCCCCTTCAAAGAATTGAGACTATTTCCAACAGAATTCCACTCTTTTTGACCTTTGTCTTATTTCTAGGTTGCTCTTGGGCCATAGGCCTTCTTGGTGACAGAAGTGATGAACCGGAGGTTAAGCAGAGAGTAGGTAGGCGGGGCTCAAGGGGTGGGGCCAAGCCAAAGGGCTCTCACACTAAGTGAAGCTTCTCCATTCTGTAAGCTTTCCGGGAACATCCAAGGC
FP016382	chr18	52339996	52340247	+	1	train	GATACGCAGACCAGGCAAATGCAGCTCCTGGAAGCCTCCTCGGTCCGCTCTTTTTTCCCTAAGCCATAGTCTGTCTGCCCCCACCCCCTTCCCGTGGAATCTTTGCCTGATCCAGTTCGTTTCCCTCCGGTCTCCACATTTTCCCTTCCAGCCCTCCACTTCTCCGGATCAATGTGTAGTACGGTTCCAACTCCCAGCTCGCACACCGCTGGCGGACACCCCAGTAACAAGTGAGAGCGCTCCACCCCGCA
FP009428	chr9	131270518	131270769	-	1	train	GAGGAATGTCCCTGTTGACCACGTGGCCTCTTCCTGAAGCCGGGGTCTGAGCGTCAGTTGTGCAAAGAACAATTGAGGAGGAGGCTTCCTCTCTGTAACCGGAAGGCGGCCTCCCCCATCTGCTGGGCCCTGGCGGAGGCAAGGGCGGGGTGGGTGGGAGAGGGAAGTCCTTGTTTCAAGGCTTTGGGGTGGGAGGAAGTAGGGGGGATGACCCAGCCTAAAGAGAGCTCCCCCAGGACCAGCCCTGGCCA
FP009073	chr9	93662701	93662952	+	1	train	GAACGAATGAATGAATAAATGGATGGATGGATGGGTGGGTGAATGGGTACATCGATGGGTGGGTGAATGGATGCGTGGATGGGTAGATGGATGGAGGCTTGAGCTGCAAGCACATGGGCCAGAAGAAGACAGGGAATCTGTGGCTCAAGAGAGTTCCACCAGCCCTCTATGTCTGCCTCTGGGTCACAGCAGCCCTTTTTTCTAGGCCACAAAGAGTGTCCTGAGTGTGCCCAACAAAGATGTGGTTCACA
FP018610	chr22	20858878	20859129	+	1	train	CGCTGCAGGGGAGAGAGGGGATGGACAGTAGGCTGCGGTTGCGCGCGCCGGATGTGGGCCCCCCCCAGCCGACCTGCGCGATGGTCGGCTCCGAGGGCGGGGCCACCTGCGCGCGACGCGCGGAAGGAGTTCGCGCGACGACCGCGGGGTCGGCGGGCGGGGCGAGGCCCTGGACGGCGGCGGCAGTGGGGCTCCTCCTTCTGTTTCCCAGACCGAGAGCCGCGCCGGCACCATGTCAGCTTACCCTAAAA
FP000987	chr1	108661387	108661638	-	1	train	GGCCCCCGTGGACTGCTCTGGGATGAAAGCGGAGGCTGACTGGGACTCCAGCCGCCGGCCGCCTCGCGTGGAGCCTTGCTGCTCCTCTTGCCGCCTTCCGCTGGCCAGGACCGTTTTTAGGCCTTTGCCACGCCTCGGGAGCGCCACGAACTCAACAGCCGCCCAAGCCGCCTTCCGGCTCGCGGTCTGCGGACGCGCTCGGAGTTGGGGGCCTTCGCCAACAGCTGCCGTACCGTGCCGCGTCGCAGCCG
FP001746	chr1	207089384	207089635	+	1	train	CTCCAGGGAGGGTAAATCAATATAACCTCTTCTCTAGAGAGGAGGTGGTTAGGTTGGTCTTAAGCAGTGTTAGAAGATCTATTTTTTTTCAAACCAGGTGTCTGAGCTGGGTGAATTCCAGCCTGGGGAGAGGACTTTGATCACCAGATGTTTTTTTGGTGTGCGTGCTGTCTTATGGTTGCGTGGCGAGTTTCTGCTTCAGATGGTACGTATGCTTTCCTTCAACTCAGCTTACAGGCATTTGGTGTCCT
FP000761	chr1	56854596	56854847	+	1	train	GGTATTAGTTAAATATCTTCCTTTCCTTAACCACACGGATATTGTTTTTCAAAAGGTCTAAAGGGCAGTGGCCTTTGGTGCCTGGAACACTTTTAGAGTATTTAGGTAATCAATACCCTGGTTACTATTTAACCATATTTGAAAGAGAATCAAGTGAGTTCTGTGCAAATCAATGTGTATCTGGGTGAGTTTCCAACATCAGATAGATCTTACAGGTCCCAGCCTGTAGACATCTTTTACTCCAATTTCCT
FP017144	chr19	35748375	35748626	+	1	train	TGGGAAGCTGGAACTACACCTCCCAGGAAGCTAGGGGGTAGATGGTTTGTTTCCGGAAGCAAAGCCCCAGGTGGGTGGGACCTAAAGCTGGGAGACTACGGAGGCAGCTCTCAATTGGTCAGGATGCGAGATTGACGGCTGCAATAACTAATAGGAACAAGCTACTGCCGAAGGGGCCCGCCCACAGAAGGGTGGTGGCCACGGTCCAGGCTGGACACAACCAAAGGCGGAGGACCCGTGGCCCACGAAGC
FP000817	chr1	66694287	66694538	+	1	train	CGTAATCATAATTCCAGTGTTTTCAGTTTTGTTTCCTTTTTCCACTAAAATCATTCCTGTGTTTCAATCAGTAAAGTGGGCTTCTTGATTTCATTTGGGATTTGTATTTGTGTTTTTGTTTTCCATTCGTTTATGTTTCTTTGGTTCGTAGTGTCAGAAGACGATGTTTTTTATGACAAACTGCCCTCGTTTGAAAGGCGCTGTGAAACGCCTGCAGGTATGGTGCTAGCCAAGTGATCTCTAGAGACCTA
FP015194	chr17	7035814	7036065	+	1	train	TCCCCGCGTCACGTGGGGCCCTACCTAGTCAGCCTCCTAACGCCCCTCCTTACGCATGCGCCCATTCACTGCTGGTCCCCAACAATGCCTAAATCCCGCCCTGCCCTTCTCGTTCCGCCCCTGCCCGGGAGCCCCGCGTCCTCATTGGCGAGCTCCAGGGTGGCCCGGCCCGGACACCCCAGTGATAAAATAGATCATCTACACGGAAACTGGCGCGCTCCAGGGGTGGGGCCCAAACTCAGTTCCACCCT
FP004622	chr4	25377071	25377322	+	1	train	CCCATCCTACCCCCACCTCGATCCGCCCCTCGCTCCTCCCTGCCCCGCCCACCGCGCCCCGTCCCTGTCCGCCCCTTCTGGCTCTCAGCACCCCCAAGACCGGTGCGCGCCCCGCCCCTCCTCCTCCTCTCGCGAGGCAGTCCCGACGCCGGAAGTGCCTGGAGCGCGCGACAGCGGCGGGGCGGGGCGGCCTGGAGGCTGTGGCGCGCGGCCGGCAGAGGGAGGGGAGAGGCCACTGGGGCCGTGTTAGT
FP005319	chr5	37213573	37213824	-	1	train	GCAATTTTCTTTTTGTATAATGTTGGGAACTTAAATATTATTACTTTTGGTAGTTGATCACAATCTTCTTAGGTCATTGCTGTATTCCTTAGGTGCTGCAAAGTCTCATTTTGAGTGTGGAATGGTGGGCGGTGTTCATCCTGAGGCAGCAGTGAGAGTCGTCCAGTCCATGGCTCGTTTCATGGCTGCCTATTTCACCAATCAGCAGCTTTGCATTTTGCCCCCTCATCATGTGAATGTTCTTCCCCCAC
FP004645	chr4	38929057	38929308	+	1	train	GCATCTGCTGCCTCACTGCCATGCTCACGAATGGAAGACATTCATCTTTGCCCAGCGGCTGGCGGGCAGGCTGCTACAACCTCCACTTAGTCTACCCCAGCTGCAGGCTGTAATGTTGGGGGAGCTGCCACATCCTGAAGGATGAAAAATAAATCACGCTTCTTCTGTCGTGTTCTATTTCTAGGCTTAGAAGAAAAGGGAGAAGAATTTGCTCGCATGCTTACAGAGCTTCTCTTTGAATTACATGTGGC
FP016121	chr17	81683810	81684061	+	1	train	ACCGGCCACTCTCATTGGATAGTTCTCACCTTCGGACTTCTCATTGGTTCTTAGGGCTCATTGTTCCAAGGGCGCGTCCAATTAGCGCGCAGCGTTAGCGGGAGAGCGGAAGACCCCGCCTCCTCGCGGCTCTAGGGCCGGCGCGGTGACGTGCGCGTGCGCGCACTGGAGGGAAAAGGCGGAAGCGGAAGTCGGGGGGCGCGCCAGCTCGTAGCAGGGGAGCGCCCGCGGCGTCGGGTTTGGGCTGGAGG
FP010868	chr11	62574859	62575110	-	1	train	TTCGACATCTCAAGTGAAGACATGGCCCCTGAAGGGCAATAAAGCTGCTAGTTTATTAATACAGTCTCCCGTTTCCTTTCATTTCCATCTGGTCATAATCCCCCTTCCCCTCCCCCAGAACTTCCTGCAGCTGCTCCCGTCTGACCCCGAACATGGGTCTCTCCCTGTTGCTCTCCACCCCACCAGCTTCTCCTTCCTACAGTTCGCACAGCTCGAGGGGAGTGCGGGGGCAGCACCGCGCGTGGCACCTG
FP019455	chrX	101219763	101220014	+	1	train	CCCTCTGCTTCTCAGGCTCTTCCGCTTTCCTCCTCTGCTTCTAATGAATTGACATCTCCAAGAGCCGCCACATCTCTGGTAGTTTTACAAAACTGTCCTTGCTCTTCCTGCGTTTGCCCAGTCAAGGGTATTTTTGCCTGAGGTCGGAATGATGCTGTTAGTCCTGGTGATATGATTGCATAGCAACACAGAGCAGCTCTGAGAGGGGAGGAGGAGGAGAAGAAGGAAGACAGGGAAGTGGGAGAGACAGA
FP005094	chr4	151408975	151409226	+	1	train	GCACTGCCTAAGTCCCCGGCGCACCTGGCGGCGGCGTGTCCCAGCGGCAGAGGCGGGGCCGGCGGCGGCGGGGGCGGGGCCGGGGCGCGCTCCCCGAGGCCTCCAGGCAGGTTGGGCCCCACCCCCGCCCCTGCCCCGCCCCGGCCCCCGCCCCCGCCCCCGCCCCCCCCGCGCTCACTAGGGAGAGGCTGGGGGGAGGCAGTGACCCACGCGCCTCAGCCCGCGGCTGACGCACCTTCGAAAAGTTGCGC
FP015302	chr17	13017937	13018188	-	1	train	CAGAACTAGGGGCGGGGCCGCTTGAGACGCTCTAGTATTCCTCTACTCTATGGCCACTGTCAATTGACAAGTCCCGAGCGGTAAAGCTCCTTTCTATTGGATGAGCAGCCTCGCGTAGGCGGGAAGCTCGGTGCACGGCGCGCTGATTGGCTGGATCCGCCATGCGGAGCGGCTAGGTGGTGCACGGGAAACGCGGGCGTAGGTGACCGGCGGCTTTCTCAGTTTTGGTGGAGACGGGCGCATGTGGGCGC
FP006745	chr6	87472776	87473027	+	1	train	CACCTAGGACGCGGCCGGGAAGGCCTCCCTTCGATCCGCGTTCTCCCCATTTTGGGGTGTAGCTTGGATTCCTGAGCCCAGTTTTAGTGCGTGGCCGACCGGGAGGCCGGCGAGGCGCTCTAGGCCGAGCGGCTTCGTCTCTATGACCACAAGGGGCGGTCCCCGGTGTCCTGCGCGGGGGCGCGGAGGGGGCGGGCGTCAGTTCCGCGGGGGGCTGTCGGGGAACCATGGCTGCCCCGAGAGGTGAGAAC
FP018191	chr20	47355762	47356013	-	1	train	TCTTCTCTTATTCTACAGAGCTTCTGCACTTGAAACTACATTGTTCAGCCTGTTTTCCTGACTTAGGTTGCTACAGAGTGTTCTGCAAGCCAGCCTACAGCCAGCAAATGTTTCTGAAAGTGTTTTCTAGGCTTTGGAGGAAGTTTTTTGGAGTAGGGATGGGGACTGGGGGGTGGGGGGAGAGATCTTGGACAACATCCTGCAAAAAAAAAAAAAAAAAAAAATCTGCAAGGATTCTGAATCCCTTAGCT
FP014344	chr16	3235075	3235326	-	1	train	TCCCTCCGGCCCTGAGAGCTCCTCTGGCCTGTCTCAAGTCTTAACGTCTCAAGCGCAGACTGCCGGCTCCGAACGGGGAGACCAGGCTTCTGCACCGGAAACAAGGCACCGGTTGTGACGTCACAGCCGCAGAGCGCCCGACTTCCCAGAAGGCACCGAGTCCCTGCCGTTCTCCTCAACTGGCGGCGGCGCGAACGAATAGTCGCCGGCGACCTGTGAGGGCACTCGGAAGGGCGAGGGGAGGGCTCGAC
FP002616	chr2	110798845	110799096	+	1	train	TATGCATGAAGAAATTTATCATTACATTTTGACATTATATGCTCTCATGCTTGGTGCACAGTTGTAGAACTGAATGAGTTTGAGGATCACAGAAATGTTATACTATTCCAGGACATGAGTAAAGCTATAGGAAGGAGAGCTTAATAATGATCTCGATGCCTTCCTTAGGGCCCCACTGTTTCATCGTTCCTGTCCGGGATGAAAACGGAAGCTTGTACCCAGGAGTCACAGCTATTGATATGATGTACAAG
FP008369	chr8	66667395	66667646	+	1	train	GGAAGAGGACGGCAACAGGAAAAGGGGCGAAAGAAAGATGGTGACTCGCGTTGCCTGCCGGTAGTTGTAGTTTTACTGGGCTCTCCTTCGTCGTCTTCCCCACTCTGCAGCCCAAGGCAAAGAGTCTGAGAGACCACATTTCCTAGGATGCCGTGCGGTGCGTCTAGCTGCACTTCCTCCTTAGGCGGAGGGGAGGATTCAGGGAAGCTTGGTTAAAAACAGTTATGGCAGTGGGAGTCGAAGCGAGGGTC
FP013224	chr14	57390385	57390636	+	1	train	TCTGCAACTTCCCGGCCCCACAGCCCCACCCACTAGCACGCGGCCTTCCCCGCAGCCCGCAGGAGACACCTTTGTCCCCGCCCCTCCACAGGTCACCTCCCTCCACGCCCCTCTCTCTTGGCCCGGGCAGCCGGCAGGCAGGGAAGTGTCGTAAAGCCAGGCCCAGGAAACTTTACCCGGCCTAACAGCTGAGGCGCTTTACGGCGACGGCGGCTGAGTGAGAACCTTGGCGGCTGTGGAGGCTGCCGCGG
FP015431	chr17	28950415	28950666	-	1	train	CGCCAGCCCACTCCATCTCGTTGCACCGGCAGGCCTCCCCACCCCCTTCTTGTTGGCCCCTTTGTGCTTCCCCCAGCACGTCCGGGTACCGGGCTTTGGGCCTCTCCCGAGCGGGTCTTTGGAGTCCCCCACTGTCCATACAAGGAAGTGGCAAGAAGGGAACCGCGCACCCTCTTCCCATCCCCCTACCCTTGATCCTCCCCATTTGGAAGAAGCTGTTCTCGTTGTTGTGCTGCAGGAACGCGGTTAGG
FP003856	chr3	96814393	96814644	+	1	train	GCCTCCCCTGCTTTTGGCTCCGCAAGTGCCCACTTGAGTCGGGAGAGGTCCTCGGGCCGCCCCAACTGCCCGCCCCACCTGGCAGTCCGTCCCCGCCCCCGTCCTCGCGGTGAGGCGCTCCCCGCCCCCTCGCTCCCCTCCCCCAAACCACAGCCCGAGCTCGCTCTTGCGCGCGCGCGCTCTCTCCGGCCCAAGTGAATAGTCCTCGCGCAAGCGGGACACTGTGGTGGATGCAATTCCCCTCGCCTCCA
FP010086	chr10	97484050	97484301	-	1	train	CTGCTTCTCATCTAGGAGGGCTCTTGGCAAATATAGTGGGCCCCCAGGTACAAAGCCTAGGGCAGGCAGGAAACATTATAATTAGTGTTATTCAACCCTTTGGTAGGTTTTTTTTTTATTTTTCATATTTATTTATTTTATTTCCTGCAGATGTGAAATCTGGCAACTATACAGTGTTACAAGTTGTGGAAGCCCTTGGGTAAGAGCCTCTGCTATAGAATGTTCTTCTCCAACCCTGACTGTGTATTCTG
FP009180	chr9	110125300	110125551	+	1	train	CTCCCTGCTCAGCCTTGAGCGTGACAGGAGCAGGCCGGGCCTGCTCTGGCTGGGAATGCCGCTGCCCTCCAGCTCCGGCCAGCCAGGGAGGGGCCAGGCCTCTCCGAAGGAGGTTAAAGTCCTTCTTTGTGTGAATGGCCTTTTAGGGAGCAGAGGGAGGGCCCACGGTGACATCACAGTCTTCCGTCAGGAGGCTCAGGAGGAATTTGTCCAGCTTGAGCCAGGGTGCACGCAGGAATCTGTCTGGAAAA
FP004875	chr4	89894690	89894941	+	1	train	AATAATTGCCTGTGCTTTCCCATTTACCCTTAGCTTTGTCAGAGACCCTTCAGCACTTCCATAGAGCCATCCCACTAGAGGTCACAGATCACAAAAACCCTACAGAAACAGGAAACCGAACTTCCTGAGTACGCTACAAAACACAGAAACCTGTTTCCTCTACACATCTCAAACTGGCAAAACTCAGTCTTAGCAGATTCAGTGTGGAAGCAGCTATCAAAAAGGCCATAAGGATTTTGTCCCCAAATTTC
FP001443	chr1	161089507	161089758	-	1	train	CTTCCACATACTCGCAGGTAAAGCCTGGAGCTGAGGATGGGACCCCACCGCCCCCACCCAGGTCCTTCCCTCCTTCCCTCCCTCCCTGGCGCCTATTAGCTAGCCCAGGCCATGGGGGCGGTGGCTGCTGAGTCTCCGTGCCAGGCCCAGCCCCCAGAGACGCACCTGTTCTGACCTGCTGAGCAGGTTCCCAGGTTTCTGCCGTCGTTGTTGGCCACAGCGTGGGAAGCAGCTCTGGGGGAGCTCGGAGC
FP004074	chr3	129278644	129278895	+	1	train	GCTTGGGATCGATATGTCTCCCAAGGCAGTCACTGTCTCCTGGGGAAACGGGAGAAAATTCATCCCAAACCGGAGGGGAGAAGACGCCGCCTCCGGGGCCCGAGGGCACCGCGGGAACCAAGGGCCCTTGGGGGCGGGGCCTGGCGCCGGGAGCGCGGGCTGAGGCGGGGCTGACGTGAGGGGCGGTGACGCGGGTGGCCAAAGCGGAGCGGAGCGGGGGTGAGGAGAGTCGAGGGAGGTGACGCGCGCTG
FP006164	chr6	8064338	8064589	-	1	train	ATCCGCCCTGATCCAGCATGGCGGCAAGAGCTTCTGCCAGTTCACAGCCTGACTACACGGCAGGCGCCTTTCCGGAACCGCCCGCTGGGTAGATTAAATTTCTCCGGCGCCCTGTAATGACACACACACAAAAAAAACCCGGGCGCGCCTTGTAAGAGGGCGGCCGGGTTTGCGCAGGCGCAGCGTGGCCGCGGGTGGGCGGAACTGGTCGGGATGAGTGGCGGAGGGACAGAGACCCCTGTGGGTTGTGA
FP009644	chr10	15097270	15097521	-	1	train	AAGCAGCAGAGACCCTGCGTGCGCGCGGGCTTCGGCCTCCAGATTCCGCAGAGTCCCGGGCCCGATCCCCCCGGGGCTGGCCTCCTGCCCTACGTGGGCCCCGCGCGCCCACCTGGAACTCACGATGGCGAATCCGTGGTCACCTGAAGGCAACAGCACCGGCGCCCAGTGCGCCTGCGCCGCGCGTGCGCAGAGCCCTCGCCTCCCGGCCTCGCCTCGCAGACGCGACGGGCCGGGGAGGAAGCGCTCGG
FP003951	chr3	114758986	114759237	-	1	train	AAATGGCCAGTGTGAGGAATTATTAATTAAAGCTAACAAATCACCCTGACAAGCTGTGCCCTGGCATGGCTGAAGGTTGATTGGCGAAAGTCTTCTCCTTTGGAGAAAAGACTTTTTGTGCTCGTGACGCAGTCAGGGACGGATTATTGAATATGCAAGTTAGAAGAAAAATCACAGGACAAGCAAATTGAGTATTTACTGGCAAATAATTGCACTCTTAAGCTGTGGCTCTCTTCCTAAATTCTTAACAT
FP005981	chr5	163505409	163505660	+	1	train	GAATTCACTGCCTAAGGTCAGGGCCTTTCTTTTGTGTGTCGCTTTAAGCATCGGCGCGTGGGCTGGGGGCAGACCGCGCGTACCCGCCCTCTTTCTGGGGCGTCGGCGGAGCGTGGCCAATCAACGGGCGCGGCTATGGCAGCGGAAGCCGGAAGCGGCGAGCGGGGTCGTTCTGGGCCTAGGGGAGGCGGGCCGAGGGCGTCTGAGCTGAGGCCCGCGTCGATCCTGGGTTGGAGGAGGTGGCGGCCGCT
FP004604	chr4	20700257	20700508	+	1	train	CTCCACCACGTGCCCCTTTGTCGCCAGAGACAGCGCCTGCAGTGCGTGAGCTGCCCCCTGAACTCCTGGCGCCTGGTTGCCTGGGCAACCCTCTTCTGGGGCGCCTCCGCGACCTCTTACCATTGGTTTAGACGACGCAGCGTGCGTCTGACGTCATTGCGCGGCGCGACCAGGTTCAGGGGCGGGCCGCGCGGAGCCTCATTTCCCCAAACGCAGGCGCTCGGTGGCGGTAGCCGCGGTTGTTGGCCGAC
FP000264	chr1	20588885	20589136	+	1	train	GAGTGGTTCCTAAGGGAGAGTGTGAAGCACACGTAGGCACTGTCTTACACCACACCTGCTGAGTCCAAACCATGGGAGGCTCCTCTCCTAGACCCTGCATCCTGAAAGCTGCGTACCTGAGAGCCTGCGGTCTGGCTGCAGGGACACACCCAAGGGGAGGAGCTGCAATCGTGTCTGGGGCCCCAGCCCAGGCTGGCCGGAGCTCCTGTTTCCCGCTGCTCTGCTGCCTGCCCGGGGTACCAACATGGCCC
FP007946	chr7	144269161	144269412	-	1	train	TGGGTGCCCGCGTTCCCCAGCTCCCCCCGCAGCCCGCTCCACAGTGGTCCGCTCCGGTTGGTTGTCACGTGCGCATTCGGGTTCCAGACCCAAGGCTGCGTGTTCTCCACCGCTTGTTGTGGCCAGTGTTACTGCGGTGACCGCCAGAGCAGCCTCGACGCTATGGAGGAGCCTGGTGCTACCCCTCAGCCCTACCTGGGGCTGGTCCTGGAGGAGCTACGCAGAGTTGTGGCAGCACTACCTGAGAGTAT
FP007099	chr6	166542151	166542402	-	1	train	CAGCCGATTCAGGAACAGACTTGATCTTCCTAAGGCATTGCATGCAAATCCTCATCAAGCTCTAGATAAAATTACCTGCTTCATAAAACAGAAAAGGCACATATTTCAAAATGAAATGTGAGTAACATTTTCATGCTAGTTGCCCCTTAGGGTGGTTGTCATGAAGGGATCCAGAAACCAGGGCTGTCCAGAGAGGGTCGTTATCAGCAGTGCCCGGGAGTATGGTGCTGGGGCTTCATCTGGCTGTTTCC
FP005563	chr5	96741077	96741328	+	1	train	GGAAAATAATACTCAATGAGTAGCCACTCTCTTCAAAACATACACTTTGATCTAAGTTGTTTCATTAACACACCTCATTTGGTAGCAAGGGAATTGGAAAAGTGCATTCTTCATTTGCTAGGTTTATGATCCATTCTTCCACTTGAGTGCTTCCCGTAAGTTACCCATCACTGTGCCTGTTTCTTTAGAAACCCATAGGGCCAGATGATGCTATAGACGCCTTGTCATCTGACTTCACCTGTGGGTCGCCT
FP004600	chr4	17810678	17810929	-	1	train	GGGGGAAATCCCGCGAGACTGCCGTAAGCGCTTTGGGAGAAGAGAGCGGGAAGCGATGGGTCCACCCACCTCGCACCCCGCGCTTTGGGCGCATGCGTCTAGGTTCAACCTTGTCCAGCCGTTTTCACGGAGGGAAGTGGCGTTTCTAAGGGGGAGGGGCCTTGACTCGGCCGGAAGTAGCGTCGGCGCGGGAGCTGAGCGGAGTGGCCTTGCCACCGGAGGGGAAGGGCTGCTGGCGTTAGGCCTGGCTG
FP008852	chr9	33524055	33524306	+	1	train	GGGGCGGGAGTCACGGCCAGGCGGGAAGGGATGCGCGCCTGAGATTCCGGAACGTCCCGCGCCAGCCCAGGAGAACCCGCAAGCCAGCGGCGCCTGAGCCCGAGCTGCAGTCACCCTCTGCGGGCCACGCGAGCTCGAGGGCGCCTTCCGGAGTCCGGGCTGGCGCTGAGCTGTAGGCGCGCGCCTAACCGCTTTACTGGGCTCACTCTATCGAGAGGTCGGAGGCTGCGAGTGTCGCTGCTGAAGGCTGT
FP014096	chr15	82647582	82647833	-	1	train	CTGGGCGGGGCCGCGTCCCGGCCGCAGTAGCTAACGGTCCGCGCGCAGCCGGGTCCGGCGCCGCGGTACCCCCATCCCCCGCCCCGTGCGGGCGAGTCCCGGGCGGGGCGCGGTCACGTGCCCGCGGGCGTGGGGCCTCGTGGAGGGGTGGAGGGCGAGGGGGGCAGGGAGCCTCCTCGTGACTCAGGGCCTTTGGGCTTAAAGGCGCCGCAGCCCTGCGGGGGCGGCCGGTGGGGCCTCCGAGCCGCACT
FP014820	chr16	67347146	67347397	+	1	train	TCACATGGCCCTGGGAAAGTAATTGTAGGGAAACTAAGTAAACTCTTTCAGACAATGGTATGACTTTCAGCCCTAACATTAGGCCTTTGAATTGTTTACTGTCCAAGCTCTGCTTTCCTGAAGGGTAGTTACAACTGTGAGGCAGTGGTCGCTATGGCTTCCTGCAGTCTCCCTGAGTTCCTGCCTGCTCCAGTGACAACATAATAGCATTATTTTGTGACCTGTCTCTAACACACCAAGATGAGACTGGT
FP004146	chr3	139389592	139389843	-	1	train	CCACGTCACGGCCCTTAGTGGTCTAGGCCACACCCCTCCTTCCCGACAGCCTCAAGTCCACTACCAGCGGTCTGGAAGGCCACGTTGCCTAGGAAACTGGGGCGGGGCTTGGGAAACGACGGCCCTGCGCGCAGTTTGCGCAGGCGCTCCGGCCCTGGCCTCTGACTATCGCGAGAATCGGCTCCCGGAAGTTCCACGTCAGTCAGTCTGACGGTCAGTGGATCGGTGGGTTTATCTCAAGGCCTGAGTAG
FP016885	chr19	13953332	13953583	-	1	train	ACATGGAGGTCTCAGTCTGACGGAGGAAACAGCCTGGCCAGCCAGGCCCAGGCCGACAGGGGAGACACAGTCCCTGCCCAAGGAGCTTCCAAGCTAAGGGCGGAACCACAGCCAAGCCCAGGGAGCTCCCAGGCTAAGGGCGGAGACTGTCCCAGCCCAGGGAGCTCCCAGTCAAAAGGGGGAGACACAGCACTGCCCTTACAAAGCTACCAGCCTCACGGAGAAGGCGCAGTCCCTGTCCACAGAGACAC
FP009125	chr9	98796488	98796739	-	1	train	GTCTGCGGCGGGCGCTCCTCTGAGCGTCCCTGGACTCCCGAATACGGGCACTGGAGGTGCGCAGCCCCGGCCCTAGCGCGCTCCCGGCACACAGCAGGCGCGCGGCCCTAGCCCCGGCGCGCTCCCGGCACACAGTAGGCGCGCGGCGCGGGCGGGTGGCCGGAAGGGGGCGCCTGCGGGCGGTATCCGGGGGCGGAGCCGACGCGGCCGGCGGGGCGGACGGGAGCGGGCCGCGCGTGGCGGCGGCGATG
FP003904	chr3	108822566	108822817	+	1	train	CTACCTTTATTTATTTATTTTTCAATTAGACTCAGGTATTGATAAAAATTCAAATGTCAGATTACAAAGGTGTGTGGGATTTTTCTTCCCACGTTACACAATTTAAGTCGACTGTTTTCAGATCAAAACTCAAGACAACTCCTTCACCACATTTCCTGTTTGTAACTGAAACAAAGTACACACAAAAGATTTTAAGAAACAGAAGAGAAAAGAATCCGAGGCACAGATAAAGATAAGTTTTACTGTCATGC
FP009136	chr9	100352874	100353125	-	1	train	GACTTATGCGCCACGCTTTGTCCGGGGACCTCGCGCTGTCCTCCCTCCGGTTCCCACAGAGCAGGGGGCTTCCGGCGGGGTACCCTCCTCTGTGGGCGGGGCCTACCGCTCCCGCTCCCGCCTCCCCGGTTGCGCGCGGCTGCGGGCTGCGATCACGTGAGCACAGCAGGGAGGGGGAGGGGCCCTGATTTCCGGGCGGCGGAAGGAGACGCGGCCGCGTGAGGACGAGGCTATTTGAAAACACGCTCCGG
FP007621	chr7	100082520	100082771	-	1	train	TGCGCATGCTCCCTACAGGGGAAAGTCCCTACTGGAGAGCGCTCAGCTTGAATGAGCTCAATTACAATGCGAATGCTGGGGCTTACTGTGTTGACTGCGGTCACCACGGTTGCCGCGTCTCCAGGACACGGTCACTTCCTTGACTATCCCCGCCTCGCTTCCACCTGTAGCTTCCTTTTTTTTTTTTTTGAGACAGTCTCACTCTGTGGCCAGCCTGGAGTACAGTGGTGCGATGTCGGCTCACTGCAACC
FP001282	chr1	154272439	154272690	+	1	train	GGCGGTCATAGGAAACTGAAAGTGCGTCACGCCCACTCCCGTCACGGAAACTGGTCTCTGAAAGGTGGGGTAAAGATAAAGCCTTTCAAATTTGAGGAAGACCTTCGGTCCCGCCTCCATTTCACGTCCGGCTTACCGTCGTTTACGACAGTGTCAGGATCGCGGGCTTGCTTTCCGGTAGCGTGGGCTGACGCCTCGCTCAATTTCTCACAGGGCTGCGCAGGTTTCCCCCGTCTGCGAATGGACCACTG
FP009659	chr10	18659230	18659481	+	1	train	CCGGAAGTGAAACCGAAGAAAACGCCACGTCGAAGGCGTGGGTTGCCGTAAGTGCGCCGTCGTCACGCGCCTAGCTTCCGGCGATGGGTGGTCCTGGGATCCGGAGGGGAGGCGGGGTGGAGGCGGGGCTTGGGGCTGTGGAGAGGCTGTGGTAGGTAGGTGGGTACAGACCGAGGGGACTACGGGTCGGCGTTGGGCTCAGTGGGCTCGAAACAAAGGGCTGTCCGGTGGGGATTCGTCGCGGCGCCTTC
FP018030	chr20	33443712	33443963	-	1	train	CCCGCCCCCTGACGCCCCGCCCAGCACCACGCCTCCCGGGCTTTCGCCTTACTTAGTGGCCCGCCCCGCCGCCCTTCTGACCCACCCCGCTAGGTAGTCCCACACGGCGGCCCTCGGCTCCCGGGCCCCGCCCCGGCCTCCTGCATTTGCCCGAGGCCCCCTCCCCAGCACTGGCCACACCCCAGGGTAGCCCCGCCCCCAGAGCTCCTCCCCCAGCGGCCGGGGTAGGGTGGCGGCTGGCCCAGCCGGGC
FP006950	chr6	136037858	136038109	+	1	train	GAAAAGCAAATGCTAGAACCTCGATGGCTTTTTTTTTTCTCCCTGCTGTCGTTCCAGCTGTAGCCTGGGATTAATTAAGTGCTGTGAACTCAGTGCCAGAGAGAGGAGGGGAAAAAAAATGCATTCTATCTCTAAGGAAGGAGTACAGTAACAGTTTCTCACCAGAGAGAAACCTAGGGAAGTGAATCGCTCTCGTCGGCAGTGTTTGTGGAGGGCCTGAAGAGACAGGGAGGTTGTGCCAGGCTGGAGGA
FP003587	chr3	46693655	46693906	-	1	train	CCAGGGGCCTCCGGATCCCGGGGGACGCCAGAGGGAGGCCTCAGGAGAGGCTTGGGGCGGCGGGACGGATCAATACCGCGCGCCAGCTCCTCGGTGGCCCGCCCGTGGCCCCCGCAACTCCAGAGCAACACCCGGAGGCGGGGCGGGGCGGGGCTGGGCGAAGGGGGCGGGGCTGGCAGGGGGCGGGACTGGCAGGGGGCAGGGTGGGGGTGCGAGCGAGGAGCCGCACAGGCAGTCCCGGCGCGGGGCAA
FP002661	chr2	120797184	120797435	+	1	train	AATGAAGTTGGTTGCTTTAGGAGCGAGCGGCGGTGTGAATCTGGGTCGGCGTTTCCCGGCTGGGTTTGGGCTCAGTGTTGGTGAGTGTCTTTGTCTTCTCTTTTAGGATTGCCACCCAGGACGATGAGCGGCTGAGATGGAGACGTCTGCCTCAGCCACTGCCTCCGAGAAGCAAGAAGCCAAAAGTGGGATCCTGGAGGCCGCTGGCTTCCCCGACCCGGGTAAAAAGGCCTCTCCTTTGGTGGTGGCTG
FP018724	chr22	30572815	30573066	-	1	train	CAGGCAGAGCCCTTCCTGGAGGAAGCTCCTTGCGGACAGAGAATGCCTGGTCCCCATGCATCCCCTTCCCTTAATGGGTCCGATGCCCTCTTGTCGCCCAGCAAGGGAGGAGTGTGTCCCCCACAGCCCCTCCCACTGTGTTCTGAGGGACTGGGTGTGCCATCCAGTGAAAGAGGCAATGATTAACCTGAGAGAATGATGCACAAATGGAATTCTGGGCCCGGCTGCAGGCTCCTGAAGGGCTCCTTGCT
FP019186	chrX	40177037	40177288	-	1	train	GAAATCGGGTCACACTAGACAAGGCTGCATGTGCGGGGAGAAGGCGGGGGGGCTCCTTTGCACGCGGCTGCTCGCGTTTCCCCAAGGCCTCCAAATAGTCTAAAAACTCACCTCGGAGGAGGAGAAAGAGCGTGAAGTGGCTCTTTCGATCTGAGACACCCCCTCCCGGAGAACTCGCCCCCTCCCCCCCAGCCCAGAGCGGAGTGAAATCTTAGAAGCCGTGGCTGCTGGGTTTGAACGGAATGCTGGAG
FP002552	chr2	100417596	100417847	-	1	train	GATGGAGGCCCAGCGGGACTTGTTCCGGCCGCAGGTCTCGGTCAGAAAGAACGCTTTTGCAGCGGCAGAAAGGGAGACTCGGGATCCGGGCCTGGGGTCCCGCGGCCGCGCCCCTCCCCACTCCCCGCGACCCGCTGGAACGCGCACGGCCGGGGACGGCGGCGGGGACGCGCGCGGGCAGGCGGCGCGCACGGCCGGATAGGCGCGAGGGGGCCGCGTGAGGCGGTGCCGGCGTTCTGGCCCCCAAAGCC
FP009632	chr10	13586764	13587015	+	1	train	AGGTGCAGAGAGTACTACAAAGACCCGGGTTTCCGTCCACGTTCACGTAAACCTCCCTACGAGGGGAAACTACAGGCGCGCTGCCGGAGGGCGCCGCCAGTGGGCACTCACGCACAGCTGCTTTGGCCACCACAGGCGCCCCGGGCCACTCTCTTCCCTCATCTCTCTCATCCCAGCTTCCGCCGGAAGCGGCTCCTGTCAGTTGTTCTCAGGTGTTTGGGCTTGTTGTTCCGTATACTCAGTGGGTTCGC
FP013929	chr15	66386711	66386962	+	1	train	CGCCACGGCGTCCGCGACGGACCCCGCCCCCAGGCAGTGCGCCGCGCTCCCGTGACGTATTTCCGCGTCATCTGCCGCCGAGGCTTGCCCCCATTGGTTGTCTGCGAGGCAATAGGGGCGGAGCCGAGTGGGAGTGTGGAAAGCGCCGCATCCCGGGTGGGAGGCGAGGCTTCCCCTTCCCCGCCCCTCCCCCGGCCTCCAGTCCCTCCCAGGGCCGCTTCGCAGAGCGGCTAGGAGCACGGCGGCGGCGG
FP000651	chr1	45012053	45012304	+	1	train	CGTCGCTACAGCAAACTTTACGGTGAAAAAAGGTAGGGGTCCTACGGGCAGCAGCCAGGGCAGCCCTGGAGCTGTCGCTGGAGTCCGATCATGTGATCTTCAACATGGCGACGCTCTTGGTTCCCTACAGAAAGGGGCGGAGCCTGGACTGGGGGGCAGGCTCAGATTCAGGTTAAATTGTGGATTGAGCTCGCAGTTACAGACAGCTGACCATGGAAGCGAATGGGTTGGGGTGAGTTCTCCAGAGCACG
FP006020	chr5	175444094	175444345	-	1	train	ACTCACCGGGGCGTCCTCCGCCCACCTCGCCCGCAGAACCATCCAGGGGGACCACGTGCGGCTGTCGCCCGCAACTCTGCCTGTCAAGCGAGGACCGCCCCCAGGGCAGGGGAGGGGACGCGCGGGCGGGGTGGGCTGTGCCCCGCGGGAACCCCGCCGGCCTGTGCGCTTGCTGGTGCCAGCTCGGCTCGCTGCCTCGCATTGCCACAGGCTCCTGAGAGGTCGCGGGCAGTGCCTGCGGGGAGGCGCGG
FP017306	chr19	41376332	41376583	+	1	train	AAACAGGACAATAAACGATTATGCCTCTTAGGGTTGTGCTGAGGAATACATGTGTATGTAAGTTTATTTGCAAAGCACTTAGAACAGGGCGCACGGTCCTTGTTGCTGTTATTATTATTGGCTATTATTGCGCAAAGGGGAAGGGAGGCGGGGCCAAGCCAGGAAGGCCCTGGAGAATCCGGGGTGCCCCCTCCTCCGGCAGAGCTTGGGCCTGGGCTCACGAGGAAGGGGCTGCAGTTCTCCAAGGATTC
FP005235	chr5	1882503	1882754	-	1	train	CTCCCAGCGCCTAGAAGCCTGCAGCTCCGGAGCAGTGGCCGCGCCACGCCGGCCCCAGCGCGCAGAACCCTGCAGGCCCCGCCCGTCCGCCCCGGGCCGCGCCCGCCATGTCCTACCCGCAGTTTGGATACCCCTACTCCTCGGCTCCCCAGGTAAGCGGAGCCCCGCCCCGCCCAGGCCACCGCAGGTGCCGGTAGGGCGGATGGGGCGGGGACGGGGGGTGGGGAGTGTCCGACCTACGACTGCCTGCG
FP004397	chr3	191328860	191329111	+	1	train	CTCCCAGCTGATTGTGTACACACTCTACTCCAAACTGGCCCAGAATTTTATCAGCTGCTCTTCATTCTGTACGTTGTATTTGGGGTTTTTATGGCCATCTACCATCTCTCCTCTAATTCCGGGTGCAAAAAGAGTGTATGAGAGAGAGGCAAGCATGCAGTAAACATCCTGTGTTAATCTCAATCCCAGAACCAGACCACAGAATGCTGACTACACCGCTCACGGGAAAGACTATGTTTTAGGTGACCCGT
FP004717	chr4	54657756	54658007	+	1	train	CTCCCGGCGGGCACGGCCCCCCGGCATTAACACGTCGAAAGAGCAGGGGCCAGACGCCGCCGGGAAGAAGCGAGACCCGGGCGGGCGCGAGGGAGGGGAGGCGAGGAGGGGCGTGGCCGGCGCGCAGAGGGAGGGCGCTGGGAGGAGGGGCTGCTGCTCGCCGCTCGCGGCTCTGGGGGCTCGGCTTTGCCGCGCTCGCTGCACTTGGGCGAGAGCTGGAACGTGGACCAGAGCTCGGATCCCATCGCAGC
FP001349	chr1	155989448	155989699	-	1	train	GTGCACTATTTGGCTGAGCAGGGGCAAAGGTTGGCATCAGCAACCAGGCAAGGAGGGAGAATTCAGCTTGCTCTTGCTGTTCGCCTTCCCCTGAACCCTGGCTTTCTTCTGGAGGCAGTACTGCCCTTTGACCAGCAGGGGCCACTCTTGCCCACAGCCAACTCTGCAGCCCTGGCAAAGCAGAGCCCCAAAGTAGGACCAGTTTGGGTCTGGGCTGTGCTTATGCCAACAGGAAGGGCTGGGCTGGGGGC
FP015269	chr17	8311792	8312043	+	1	train	CCTCCACAACACACTCCACGGACACACGGTTCAGTGGGCAGACACGGAGCGGCTGTATCCCGTATTGAGTTTTATGGATTCTCTGGAACCCCCACACCAGCCTTCCCTCCTGACTTCAAAATCCTGCCTTCCAGTGGCCCCAGTTTACCCTCCCTGTGGGGCACTAAGGGTCTTTCTCTTTCCCTCCCTCCTCCTCAGGGGATGACTCCAGCAGAGCACCTCACTCCTTTGAAGAGCACAGAGGAAGATGT
FP007953	chr7	148883219	148883470	-	1	train	CTGACGTTTCCCCACGACGCACCCCGAAATCCCCCTGAGCTCCGGCGGTCGCGGGCTGCCCTCGCCGCCTGGTCTGGCTTTATGCTAAGTTTGAGGGAAGAGTCGAGCTGCTCTGCTCTCTATTGATTGTGTTTCTGGAGGGCGTCCTGTTGAATTCCCACTTCATTGTGTACATCCCCTTCCGTTCCCCCCAAAAATCTGTGCCACAGGGTTACTTTTTGAAAGCGGGAGGAATCGAGAAGCACGATCTT
FP009696	chr10	27242049	27242300	-	1	train	ATGTGGCATTTCAAAGAGGCACGAAATGCAACCGGCTAGGAAGGAGGGATTCCGCCGCACGGCTGCGGCGTCCGCGGGACAGAGACGCCTAACCGGAAGAGCTAGCAGGAACGCCGGACGCCGCTGGGACGTGTCCGGAAGCGCGGCGGAACGGCGGACTACATTTCCCATAAGCCCGCGGGGCAGGGCGAGGGGCGGGCGTCGGGGACATTACCGGACAGGCCTTGTCCGGCAATACCAAGGTAAATGCC
FP012703	chr13	33817948	33818199	+	1	train	GAGAAAATAAATAAAGCAGTGAACTTCAGAACTAGAGGGCATCGACGCGGCAGCTGATTTTGTCTGCACACGGCCCCCGTAGCCTTTCCGTCCAAAATCATTTACGCGGCCGCGACCGGTAGTGACGTCACGAGATTTGGAGCTCGCGGGAAAACTTGTCTCTGCGTTGTGGGGAGGACGCGCGCTCGCGCGGGATTTTCAAGCGTAGGCCCCCGGGAACTCGAGCTGCCATGAGCCTCTGGGTGGACAAG
FP014053	chr15	78438047	78438298	+	1	train	CCGCCCCCGCTCGCGAGAACAGCGGCGACGGCGCGAGAAATCGCTTTCTGGTTAGCTCCGCCCCTTCCCTTTCTTTGTTTTCCTGTCCGACGATCTCGCGGGAGTTAGGCGACAAATCCCGCGAGCGCAGACCCGGGGCTGGCTCTGCTGCTCTCGCGATATTTGCGCGAGCCTGCTTCCTTCTTTCCTCCCTTGCCAGTCCGCCTGTCTTCCTCCCCGTCTTCCCTGCCCGGCCTCCCCCTTCTTCCCCC
FP002867	chr2	173965813	173966064	-	1	train	TTGCTTTAAATGCCTGCCTTCAGGGTACGTGAAAGTGGGAGACCGGTTGGAAAACTGGGGCTAAGGGGGCTGGGAGCGAGATGGCTGTGACTGAATCAGGAAGCTGGATGTGAGAGAGAAAAATCCTCCTCCAGCCTCGGAGGGGGTGTCTCCCCTAATACCAGCTCCCTCCCCCCGGGCTCGTGGGGCGGGTCTGGTTTGCTCAGTAGCGGCAGCGGCCGCAGCAGCAGCCCCGGGGCTGAGCAGCGGTG
FP012419	chr12	111368977	111369228	-	1	train	TGATTTCGAGGGCGGGCTCACGGGGGCGTGCCGGTTCGGGGCCTGCCAATCGGGAGGTGGGAAGAACGCGGGGGCGGGCCCGGCGCGAGCAGGCTGGGAAGGGTTAAAGGGGCGCGCCCGTGGGTCGCGGACTGTCCGGTTGCTCCCGCCTCCTTCCGTCGGATCCGCGCTCTCCCGCGGCTGCGCCGGCCCGGCCGCCCAGGAGACAAAGCGCTGCCGCCGCCGTCGCCGCCACCGGGCCAGTCCCGAAC
FP012716	chr13	36673819	36674070	+	1	train	CCTCCTCGTCCTCCGAGATTTTCCAATGGTGGCGCAAGCTCCTCCCTCGCCCGAGCTGGGCGCCGTCCTAGAGCCAAGCGCCCGCTAGAGCCGAGACTAGGGGGCGGTGAGGAGTGGGACCCGCTCGCGGCCATTGGTTGTGCGGCCAGGAGGGGCGGAGGGACGACGCGCGGGCTGTGCGGGGCCGGCAGCCCGGATCCACTCCCGCTGCGCCTGCTGTCCGCCGTGCGCCCAGACTGCGCGCCGCGCCG
FP016909	chr19	15125655	15125906	-	1	train	GGGTGTGAGGTCTGCAGTGCGAGCCAATGAAAGGCAAGGGGAGGCGGGGTCTATGAGGGAGGCGGCCAGTAGAGGCAAGGCTCTTTCGGGTGGCCCCGGAGCGGGGCTCCGACTGTGCGCATGCGCATCGGGCCGGACTACGGGGCGCCGCTGAGCCAAGTGGGCCACCCGGGCACGGCCACGCTCCCGGGTCACGTGACACGGAGGGGGCCGAATTCTGCTGGAGGCAGCGCCATATCCTGGAGGTGAAG
FP013861	chr15	58933628	58933879	-	1	train	CGGACGTTGCTGCGGTAGGACGTGCCCGACCTCCCAGTCCACACAAGGGAGCGCACCGCCCAAGTGGCCGCGGCCCGCTCGACCGAGCGCGCCGCTGCCGGTCCCTGCGCCAACGCGCACGCGCGGCCGTCATCCGGGTGCTGGCGGCTGCTGCTGCAGGTTGCGCTAGGGTGGCGGACCGGGGGAGGGGCCAGCGGCTCATTGTGCGCAGCGCTGCGCGGCGCCGGCGGCCGCCGAGGCCTGGGTGGAAG
FP012961	chr13	114314330	114314581	+	1	train	GGGCGGGCACGCAGCCCTTCCGAAGGCCCGCGCGAGCCGCTAGTTTTGCCCACGCACTTTTGGCACAGCCGCGCCACGCGATCGGCGATCTGATTGGCCCGCGCGGGAGGGCGCGCGGCGCCAAACTTGCTTCCCGTCAGCCCCCGCCCGTCCCGCGGGAGCGCGCACGCTCGCGCACCCGGATCCCGGCTCCTGCATCCAGTCGCCATTCGGGAGGCCGCTGCGCTGCAGGGCCTCGCGGAGCCGCCCGC
FP010013	chr10	89414367	89414618	+	1	train	AACAAACAGTTACACGGGCAGCATGCGAACTCAGCGTACTCAGGGAGGCAGATAAAAGCCGAGACCAAAGTGAAGGAGCCCAGCTGCAAATTCAACTCCGAAAGACCCTGGACTCCGCAAACGTCTGGCCGCGGCTCTTGAGCTCTTCTATTAAGTTTCGGTTTCTGAAGCTTACGTTTAGTTTCGATTTCTTAAGTTTCAGTTTCTGAGCGCTCGGCATCTGATTCAATCTCCAGTTTCCTGTTCTTGCT
FP009688	chr10	26438140	26438391	+	1	train	GCGCCCTTCCTCCACCCCCACCCGAGAGACAGCTGAACTCCGGCCGGGACGCGCGTGTTGCCAGTCCAGCCCTGCACCGCGTCCCCTGAGGGCGGGCTGCAGGCGGCCGGGAAGCCTTGCACAACCGGCCCAAAAGAGGAAGCCCAGAAAGTGCTGAAGTAAACACTTTGGGAGACCGTTGCAACATAAAGCGGCCTCTCAGTCTTTGGTGGAACCATCACTAGGCCCCAATCCCTTAGTCCCTCTTGCGT
FP016768	chr19	10517899	10518150	-	1	train	TCAGCGATAAGGGTGGGAGAGAATCGGTCGGGGGAACACGGAATCCAGGGGGTGGGACAGGAGAGACCCTCCTTATCTGAGGAGCGGCAAGTTCCCATCACAATCCTGCCCGGGCCCGCCCCACCCCCACTCGCCCTGAGCCGGGGGCGGGAGCGGGAAATTGGGAGGGGCCTGTGCCCGGGAGCAGCGGGAACCTATCTGCTGGTGGGAGAGGACTCAGGCTAAGGTGGCCCCCACTGAAGACTCCTGCT
FP019651	chrX	136146546	136146797	+	1	train	TTCAGGGTTCAGCTTAAATGTCACCTCCTCCTTTCCTAATCACACTATTTGTACTTTTCCTGCACAGGCTTTATCCAAATTTGATTATTTTCTCACTTCTTTATTGTCTGTCTCCCCCACTAGATTACAAGCTCCCAGGGCAGGGTGCGTACTGTCTGCAGACAACGCTGTTATCACCAGAGCCCGGCACTGAGCTTGGCACATGGCGAGGGCTCAGTAAACTGAATGCTGAGTGAATGAGACGCCACGCG
FP013755	chr15	43693707	43693958	+	1	train	GTCCCGCTAATTACCTGGCGGGTGCTGCCCACCCCTGCCCTCGCGCACCTAGCGCGGTGGCAGGCGGGAAGGCGGGGCCTGGGGGAGCCCCACCCCTGGAGACTGCGGCTGGGGCCTCCCTCTCCTCCGCCCGCCCGCCTGCCACTAGCTCATTGCGCCTCTCCTGCAGTCTGATTGGCACCGGCTCCCATTCCGGCTCCAGCCTCCAATCCGACCCCCATTTCGGCTGCAGCCTCGGACCTAGCTCCGGC
FP012577	chr12	125104815	125105066	+	1	train	TGCTATAAAGAAATACCTGAGACTGGGTAATTTATAAGAAAAGAGGTTTAATTGGCTTATGGTTCTGCAGGCTGTGCGAGAGGCATGGTGCCAGCATCTGCTTCTGGGGAGGCTGTATGGAGCTTGTGATCATGGCGGAAGGGAAGTGGAGCGGGCACTTTACATGGGAAAGCAGGAGTGGAGGGGGTCGGGGGGCGGGGTGCCACACACTTTTCAATGACCAGCTCTTGTGTGAACTCAGAGTGAGAACC
FP000990	chr1	108746484	108746735	+	1	train	CTCGGGCGACAGCAGCCTCCCGCTAGGGGGCTCCCGACTTCCCACGGAGGTTCCACCTGTACTGCGCAGCCTCCGTCCGGGAGTGGGCGGAAATTCCCTTGCTGACGCGCGTGACGTCACCGGAGGGCGCGGTCTGCTAGAGGGGGTGGGGCATTCCCCTGCAGCAAGGGGCGGGGCCACCCCAACGCCGCTTCTGCGGCCAAAGTAGGTTGGGAGTGGAAGGTGGTGGCTGCTGCTCCGCAGTGTCGGGA
FP011243	chr11	89490439	89490690	-	1	train	TTGAAAATGCTTTCTGCAGTTACTGATTTAGGTTCCATTGAAAGCACTGTGTCGAAGAATTTACCTGTGGTTTCTATTCATCAGGTAATAATTATTTTCAATTTTTGTTTTCTGTCTCTCATGTCTGATTGGTTTAGTTCATCTGGCTCTCCATGAATGTCCTGCTTTTCTGGAAAACCTTCTTGCTGTATAACCAAGGGCCAGAGTATCACTACCTCCACCAGATGTTGGGGGTAAGTAAGGTGAACTTG
FP017937	chr20	13784855	13785106	-	1	train	CGCCGACATAAGCGCCAGAGCCCTGCCGGCCGCAGCATCTCCAGCTGCGACCCCAATTGCCGGCGCTTTTTGTGCGCATGCGCCAGCGCGGCTTTTGTGTCTACTCCAAAGATCTTCTCGAGGGAACGTGCAAGTAGTTTGGCCTAGTTCGCTTGCCATTACGGCGGAGCGTCACGCGGAGGATTGTGGGTGTGAGCCCCACGTGAGGCTTGGTAGGACTGCGGACGGTGAGTGGGGATGGACTGGAGTTG
FP013399	chr14	77320835	77321086	+	1	train	CGGGCAGCGTGGTCGCGGCCCGGGCCGCTAGGAGGCGGCAGGAGGCGCAGAGCATGTCGGGACCGGGAGGGCCGGCCGGCCCGGGGCGGGGGGGACTGGGCAGGAGGGGTCGGGCTCAGGGAACTGCAACTCCCAGCAGCCCCCAGGTACGGGGCGTGGCGGGAGAGCTGGGCGAGCCAGAGGCGACCGGAAGGATCTTTCTAGTCCAGCCCCTCGCTTTACCCGGACGAAAGACACGGGCCTGATTCGTC
FP013490	chr14	95133371	95133622	-	1	train	TTATAGGAAGAGTTTGAATGGCTCATAGGAATGATATTCTGATGTTTCTCTTGTTTCTGTGCTTTCTTTGTTTTTAACCCTGCATGATTGTGTAATGGTATTCTTTTTCCCTTTTGTAGTAAGCTGTGCTAGAACAAAAATGCAATGAAAGAAACACTGGATGAATGAAAAGCCCTGCTTTGCAACCCCTCAGCATGGCAGGCCTGCAGCTCATGACCCCTGCTTCCTCACCAATGGGTCCTTTCTTTGGA
FP003843	chr3	87227111	87227362	+	1	train	CCAACCCGCGCACAGAAGCCGTGGCCCCACGCTCTCCTATCCCCGCCAGCTCCAGACTCGCCCCGCTGCCAACTCCCGGTGCAGGAGGAACTGGCAGCCGCAGACGTGAGGAAAGCGGCCGCGGCTTCAAACTCCGTAGTGCGCAGGCGCCACACAACGCGCAGGCGCCGCCTAGAAGTGACTTCTCCAAAAAGTGTGTTAGTTCCCGGTCACCTGAGCTCCGGGTGACGCGGCTGCGGTAGCTGCGGATA
FP013134	chr14	38207750	38208001	+	1	train	TGGTGGTAAGTCCCAGGGTTCGTTTAAAATCCTGATAACGGAACATACATTCTTTCTTACGGGAAAACCGTTTTGATTCTTAAATGAAGTCAGTGAGCTTCAGGCTTGCCTACATTGATATCTCCTAATGGTTTGGGCACGTGACCCAGAGCCAGCTCACAAATCAAGCCTCAGAAAGAGCTGACATCCTAGCTCTTCCCGGAAAAACTCGAATGTCGCCCTGCCGTTCCTGGGGTTGGTGACAGGTCTGG
FP009342	chr9	127854607	127854858	-	1	train	GGGAGACAAGCCTAGAGCCTGGGCCCTCCCACCCCACTGCCTCCCCCCATCCCAGGGCCCCCCACCCAGTGACAAAGCCCGTGGCACTTCCTCTACCCGGTTGGCAGGCGGCCTGGCCCAGCCCCTTCTCTAAGGAAGCGCATTTCCTGCCTCCCTGGGCCGGCCGGGCTGGATGAGCCAGGAGCTCCCTGCTGCCGGTCATACCACAGCCTTCATCTGCGCCCTGGGGCCAGGACTGCTGCTGTCACTGC
FP017100	chr19	33373540	33373791	+	1	train	TGCTCCCGCCACACCCGCCGAGCCAGCCGCGCGGAACCCACTCCCCCACCGTGGGCCCGCCTCCCGCCGCCAGCCTGCCCACGATTGGTCGACTGGGCTGCCCGTCGCGAGAGAAGGCGGTGCCTCCGGCAGGCCGGCGCTCCCATTGGCCGGGATGGCGGCGGCGGCGCGCGCGGCCCCGGCGAGCAGGGGAAGCCGGTGGCCGCGGCTGCGGAACGGGCGGAGGCTGCCGGTTTCGTAACCGTCGCTCC
FP013892	chr15	64387642	64387893	+	1	train	GCACTGACAGAGCTCGACTCCGGAGGGCACAAGGAAGTAGACAACAAAGCAGGAAGAAGCAGGAGGCGCACAGATGGGCGGGATCTGCACTGGAGGCGGTGCGACCTTGCGGGGCGGTGCAAGCTGGCGGAACCCGCGTGAGAGAGGTTGGTGTTGCGAAGGGAACTAGTCCGGTGCAGGACGTGGGGCTTTTGCAGCTCAGCTGGTTCCGGCTGGGGAAGATGGCGGTGGCTGGGGCGGTGTCCGGGGAG
FP007541	chr7	90211685	90211936	+	1	train	GCCCCCTCCGAGCTCCCCGACTCCTCCCCGCGCTCCACGGCTCTTCCCGACTCCAGTCAGCGTTCCTCGGGCCCTCGGCGCCACGAGCTGTCCGGGCACGCAGCCCCTAGCGGCGCGTCGCTGCCAAGCCGGCCTCCGCGCGCCTCCCTCCTTCCTTCTCCCCTGGCTGTTCGCGATCCAGCTTGGGTAGGCGGGGAAGCAGCTGGAGTGCGACCGCCGCGGCAGCCACCCTGCAACCGCCAGTCGGAGGT
FP010819	chr11	59754917	59755168	+	1	train	TCCCCCCACTTTCACTGCCAGCAGCCCTGAGGCTGCGCGGGATCCCGCTACTCGGAATGCCTTCCTCCTGGCGGGATATTTTTAGAACACTCGACTGCCACCCAATGTAGCCCACATCACATTGTCTTATTTGTTTTCGGGAGCTTGTCCCCGCCTAGCAAGGAGTCGGCTAAGAACTGGATCCTAGCGAGGAGCCCGGCACAGACAGCGAATGACCGCAGCCAGACAGTCGCTCTTGCTCTTCCTCGGCC
FP013769	chr15	44536970	44537221	+	1	train	CTCTCCCTCAGCCCCGCAGCGCCACCGACCGCGTTCCCCGCCCACTTCTTACCCGCGCGCGTCGCCGCCGCCGCCTGAGGGGGCGTGGCCTCGGCTCGGCGCACAGTCAGCCACGGTCCCATCCTGCTCCGCGCCGGTCAACGAGAGCAAACCCAGTGACTCACCTCCGCCGTGCTAACTCCTCGCTAGCTCTCCCTCTCACACACGCTCACACCCGGCTCGAGATGGCGGCGGCGGCGGCGGCGGCGGGG
FP010782	chr11	47594360	47594611	-	1	train	GCTGCCTCTGGCCTCAGTTTCCCCAACTGCTCAGATTAATGATGCCTGCTCTCTAGACCCAGAGGACGAAGCTCTAAGGAGGTCACAGATGAGGAAGGGTCCAGTGCTCGGCGCAGACTCCGTGCGGAGCCCCTAGGCCCCGCAGCTCGCCCCTCCCTCTCCCCTACTCCCCTCCTCTCCCCCTCCCGTCTCTCCCCCGCCTCTTCGCTCTCGCTCGGCTCCCTCTCTAGCTGACCTTCCCTTTCCCTCAC
FP011340	chr11	111937142	111937393	+	1	train	GTGCGCGTGCGTGCGGGCGTGCAGCGCGCGCCCTCGCCAGCCCGCCTGGCCGTGCGGCTTTCCCGCAGGAAAGCGGGGCTGGGGGCAGCCCGGCAGCGCCGCTCAACCTAGTGCGCGCCCAGTTGTTTCCATAGGAACCGCGACCGCGCCGGGCCCCTCCAGCGGAGGCCCCCGTGCGAGCATGCCCAGTGCAAGCCGCTAGTTTGGCTCCAGTCTAGGTTTCCAGTAAGTGGCATGCGGGACTCCGGAGG
FP017392	chr19	44891062	44891313	+	1	train	GGAGGGGCGCCGTGGCTACCCTGCGAGTGAGAACCAATACAAAAGGACATTTCAGGGAAAGTGGGCGGGACTTTATGCACAAGTCCAATGGGAAGACCGAGTCTTGACGCTGGTGGGCGGGCCTCAGGGCACACTAAACCAATGGGCTAGGTGGGGCGGGGCGACGGTGGTGGCGGCGGCGGCAGCGGGTTCGGTTGCGCGTGGCGCACGGGGTGGGAGCGGAGCCCAGGCCGGGAGCAGGCGCCGCCGCC
FP006830	chr6	110476562	110476813	-	1	train	GGGCGCCGGGACCACAGCGCGCCGGGAAGGAGGCCGAGGCGGCAGGAAAAAAGCCGAAGATACTTGGGGGGACCGAGGGGCCAAGCGACGGAGGGAGGAACAGAATACAGCCTCGCGCTGGTCCCGAGCACTGGGACGCGCGGGGAGAGCAGGAGGCCGGGCGGGGAGGTTCGGGGCGGGGCGCGCTACCCGCAGTCCCCGGAGCTCGGCTAACTCGGCGCCCAGTGCACGGCCGCACCATGGGGTCCCGC
FP000924	chr1	92168720	92168971	+	1	train	AAATAACATGTACCCAACGATCTTAAAAATCTAAATGATGACATCATTGATCTAGCTTAGGAGTAAGTAGATTCTGAAAGGAATAATGACTGGTTGCTATGACAATGTGGCTCTCACCAGCTGCTGATTTGTTGCTTGAACACAGGGTTTTACGTGCAAGATCCAGGCTCTGCGTGATAAGCTGTGGATCTTCCTGGTTCAGTCTTTCTATGCTGTTCGTCACACAGAAAGCTGGAAGCTGATGAGCACAG
FP014222	chr16	482321	482572	+	1	train	CTGGGATTACAGGCGTGAGTCACCGCGCCCAGCCTGAATTGTGTTTTATAAGTAATGTCTCTGCTTAGAGACAGTTGGGACTTCAGGCGTCAGACCCCTCCCTCCACACTGCCCTCTCCAGGGCCTTGCAGCAGTGAGGTGTGGGGCGTTCGCAGTTCCCCTCCCAGCTTGTGCCCGCTGACTGCTGGCGCACTCTCTCCTAGGCCAACGAGGTGACGGACAGCGCGTACATGGGCTCCGAGAGCACCTAC
FP014857	chr16	68086598	68086849	+	1	train	ACTGTGGTTCATTAAGTTCAGTTTCCCTGTTACATATGCATACGAACTTCAAAAAGGAAATGGATAAGAGTAATCCCTATTTTTTCTTGCCTGGTATGAGGTGAATTTCTTAATGTTCATGTTTGATGTTCAGTAGACTGTTCTGTAGAGTTGCATTGAAAAAACATACCTGAACGTGAGGCATGAGGATTCTCTAGGCGGTACTTATTTTATCGGTCCTTTTAGTCTTCATATGCTAGATGTCTTTTTCA
FP003761	chr3	56683186	56683437	-	1	train	CCACGGCTGCGGCGTCGAGGTCCTCGCGGGGCGTTGCGGTCGCCCCTTTTGGCGGGCTCGGGCGCCCCGAAACTCCGCCCCCCGGCGCGGTATTTCCGCACTCGCGCAGGCGGGTGCCGCGCTCCCATTGGTGGAGTGGCCCGCGGTGGCCCCGCCCCCTCCCGGCCCCGACGCGCGCACGTTGAGTGGCGGGGGAAGGCAGAAGAACTGCCCGAGGGAGGAGCGGCTCCGAGGACCGGGCAGCGCATTTG
FP013886	chr15	64093849	64094100	-	1	train	CACTTCTACCAGGCGGACACTACAAGTCCCAGAATACAATTCGCTGCCTTCCGAAGAAAATAGCTCACGCATCCCAGCATGCAATGTGCTGACGAGCCGCGAAGATTGTTTTTGTCCCGCCGAAATCGAGCAAAGCACGCTGGAACTTGTAGTCCTTGAGGCCCCTTCCCTAGGTCCTTCGAGCTACTCCGTCTGGCCCCGCCTTTTCTCTGCTCTCCTGAACCTTTAGGCTTGTCTCGGCCCATTTGAAG
FP015330	chr17	17206282	17206533	-	1	train	CTGGGCGTGGCCTGGCACGTGGCCTGGCACGTGGCCGTTGGCGCTGCGTGACGGGGGCGTGGCCTGGTGCGTGGCAATTGGCAGGACGTGATAGGGCGTGGCCTGGCGCGTGGCCATGGGGGGCTGGGCGTGGCCTGGCACATGGCCGCTGGCGGCGCGTGACGGGGGCGTGGCCTGGCGGTCTAGGGCTGGGGGCGCGCAGACTCCGCTGCGGCGGCGTGGCTGTGGGTCCCGGATTAGCGGCGGCATGG
FP005149	chr4	169010204	169010455	-	1	train	GAGCAGCGCGCCTGCGCCAGTTGCTCTCATGCGTCAAGACTACAGGTCCCAGCAAGCATCGAGGCCCTCTCCAGGCCATTCTTCCGCCGGCGGGGAGCTCTCGTCGGCGTATTTGTTAGGTGTGGCGCGGAAGAGTCCGTGCGTGATGACGTCGACGCGGCGACGTCGAGCTCTTCCTCCTTTTCACGGCGTCTTGCATTACTATTGTGCGGCTGCAGGAGGTGTCGAGCGGCGTTATTTTTTTTTGCGGT
FP013071	chr14	24307876	24308127	-	1	train	TGTCATCACCAAAGACACACATACAAGCTCCAATGGCTTTTGCCAGGCAATTCTTCCTCCAGGACCCCATCTGGCCCCTCCCTCATCCCTCCCCTTGGACTTTGCCCTTCTTACTGGCCAGGCAGGGGGGCCAGAGTCCAGGCTTGACTCATTCCCACCTTGTCCTGGGCTGAGATCCCAGGTTTGTAACAGAAAACACCACTAAAGCCCCAGCACAGGAGAGAACCACCCAGCCCAGAAGTTCCAGGGAA
FP012842	chr13	77535505	77535756	+	1	train	TATATATATGTTTTTTTTTTTTTTGCATAAGGCCATAAGTTAAACAAAGAATTCAAACAAAGACAGTAGGTATATGAGTGAGAATGCTTTAATTGAACTGGTATTTGTACAAGGTGACACCCTGTTGCATACCACACACCTCCTCCCTGAGTGACTCAGCCGCTGAGCAAGAAACCTCTGAACTGTTCACTAATACAGTCAGGTAGAGGTTGAGACTCCACTGAATAAACTCTAGGTTCCCATTTCTTTCA
FP014492	chr16	22513410	22513661	+	1	train	TGTTTTTTTGTAGGTTTCAGGCACAGAACTGTATATCCAATAATAGTGAAATGGATCCCACTAATTATGACAGAAATGATGATACATTTAAATGACTTGGATGTTTTATAGGTATGATCTCGTGAAATCTTGAGAGAAACTGAATGACGAATGAAACTATTGTTCCTGTTTCACACAGAAGAAAACTGAGGTTAAAAGGGGTAAAGTAATTTTGCATGGCATGAAGTAGAAATTCAAAGTACAGGAATTTG
FP002985	chr2	197705168	197705419	+	1	train	AGCTGTTTTTCTGGCAAGCCGCCGTCCATTGGCGGCCTCTGGCGCCGCGGCTTTCGGCGTGCGGACGGCGCAGGCGCGGGCGGGGCGGGGCCGGGCGGGGAGGGCGGTGAGGGCCGGCGCTCGGGGCCGCGTTTCATTGGCTTTCCGGCCGGAAGCTGCGGCGCGACCCGGCTGCGCATGCGCCTCTCACACGTGCTGTCAGAACGCCGCCTCCTCCGCTTGCGGCCGGTCTGCACCATGCTGCGAACGTC
FP011005	chr11	66011807	66012058	+	1	train	CCCGGGCCAGGTGTGTCCTGGAGGGCAGGGAAGCGTCTTGGCACGCGGGTGCGCGCCGCCCCCTCGGCCTCCTGGGCTCCCTGAACCTCGCAGGACCCCGGCAACTTCGAGCCCCGCCCCAGCTCCAGGCCGCGGGGGCGCATCGCGGGCGTCGGGCGGGGCGGCCCAGCGGGTAAAAGCTGCGCGGCCGCAAGCTCGGCACTCACGGCTCTGAGGGCTCCGACGGCACTGACGGCCATGGCGCGTTCGAA
FP011768	chr12	26917530	26917781	-	1	train	TTTTGATACATGTATACAATGTGTAATGTCTCTGACATATTTTTAAAGGTGATTCGCATCTAGGTGGCGGCAGTCGAGAAGGCTCGTTTAAAGAAACAATAACATTAAAGTGGTGTACACCAAGGACAAATAACATTGGTAGGTATTTAAGAATAACTGAAAAAGACGAAATCAAAAGGTTCTTAAGGTATTATATGGGGGTCAATTTATTCTGTTTTGCATTTTCGTTTTTCATTGTTTCAGGACTCAAT
FP002188	chr2	33587229	33587480	-	1	train	GTCTTTATCTTGTGGGTCTTTTACATTTGCATTTCAAAACATCCTCCACCTTTGCCTCTGTCAGGAATCTTCTTTTGATGGCAAGATATGATGGGAATTGGGAAGATGAGGATGTTTTACTTTTAGAGTTTTTATTCATTTATTTATGTTTTTGTGTCAGTTAAAGGAAACATTAGCAAAAGTTCCACCTAATCATGTGGGAAAGCCTTTACTGAAGAAGCCAATGGGACCAGCCCACTGGGTGAGTAGTA
FP011465	chr11	125111575	125111826	-	1	train	GAGGTCGGTCCAAAGGCTGACAGGCACCGACCAATCGGAGGGCGTAACGAAGGGTGACGAGCCAACCCACCGCGCTCGGCGTTGCCGGGAGCCGCGCTAGGCGTAAGCTAATGAGGAAGAAGTTGGCCGTTGAAAGAGCAACGCTGATTAGTGAATCTTAAATCACCTAAATGAAAGCCCACTTGTGATTCGCCCTAAGTAAACCAGAGCCCCGTGGGCTGGAACGCGCCGGAATCTGAGGTGTGAGTAGA
FP011721	chr12	16346914	16347165	+	1	train	TCATATAAACCCATGAAAGATAGATCCATAAATATTGTGGGGACTATGGGGACTAGGAAGCTTGGAAGTATCCCTAATCTAGATAGTGATGGTAGGCTAAATATCTACGGCTTCCGTAAACGTTCCTCTAAACCTCCCTGTAAAAAACACACTAAGCTGATTGTGTAATGAGCTGATCCAGTACAATTGCCTCCAATTTCATTCTGGACCCTGAACAGGAGGGGACATCGTGACAAAGCAAATTGTCTGGT
FP013853	chr15	57591707	57591958	+	1	train	ATGGGCGTCCAGGCTGTCCAGGGCTCCTTCTTGCCCTGGGACTGACCCTGGGAGGGCGCAGCGCCCTGGGAGAGGATGGTGCTCAACTTTAACCGGGTCGGCGCCCTCGGGAGAAAATGCAGCCTGACGGGTCGGGTGAGCGCGCTCGGCCCCGCCCCGGCCCGGCCCGGCCCCGCCCCCGGCCCCACCCCCGGGCCTTCGCGGTGCAGCTGAGGCTGCAAGTAGCCGGCGCCGTCCCGCGTCGCCCCCGC
FP002453	chr2	85327925	85328176	-	1	train	AAACCAGGAAGAAGAGGCTCGCCTCCCACTCGGCGACAGTAAGCGAAGCAGCCGAAGGCGAGCGCCGACATCAGCAGCTGCCCCCTAAATCCCGCCCTTCGTCTTGGCGGCAGCGGGAGACTGAGAGACGCGCGCAGCAGGGGCGGGACTGGAGAGGGGCCCCGCGCGCGGATCTCGCGAGAGCATTAGAGGGCGGAAGCGCTATCCGAGCAGGATGCGGTTCGTGGTTGCCTTGGTCCTCCTGAACGTCG
FP013130	chr14	37172522	37172773	-	1	train	GCTTTGTACTTGGCGCTCCAGGCCTGGGTCACTTCACTTGAGTTATCAGAGGCAACCGTCTAGTTTGACAAACTACCGGAGGCTCCTGATAGAGAAGGGCGTGGGCGCCAATCTAGGAGACAAAGCCTGAGCCGCTGCGGTGAGCCAACCCAAACCCATCCGCGGGTCTTCTTTCCAGAGACATTTTCGGAGCCGAGGGCGCCTCTGCGAGAGCGCTGCGAACAGGTTCCGACGCGCCGGAGATCGCGCGA
FP011954	chr12	52905025	52905276	-	1	train	GGTTTCTGGGGGTGGTGCTGGAGTGGGCTCCAGGGTTGGAACGGGCCCTTGGGTGGCGGCAGCTCTCTGCTGCCCCCACCTGAGTCCTGCCCGGAGGTGGCAGGTGACGGGTTAGGCCCAGCCCCCTCTGGGCCTAGCCACTCAGGTACGAGGCCTTTCCCCCCCATCCCCCGGGGCTGGGATCTCTTTTATAAAAGGCCATTCCTGAGAGCTCTCCTCACCAAGAAGCAGCTTCTCCGCTCCTTCTAGGA
FP006264	chr6	28281360	28281611	+	1	train	CAATCGAAATGATGTACATAAATTTGGGGCGTGCTTGCACCGTCCGAATGGGCCTGGGGTCCGGCGGCGGAGAGTGACCTGGGCGGGGCAGGAGGTGGCAGCGGGGGTCCTCCCGGCTCACCAGAGAGACGAGCGGCCGTGCTCCTAGAGAGGCAGGGAACCCGCCAGACTCCGCCACTCCGCTGCGGGCGCGGCGCAGGGAGAAGCTTTTGTACCCGCCCAGCTGCTGGAGGCGCCGGCAGCGCCCGCCA
FP004984	chr4	118836071	118836322	-	1	train	AGAAAAGACGGGAAAAAGAAAACCAAAAAACAAAAAACCGGAAAAAGGGAAAAAGCGAAAAGGGGGAGGTCGCTCGGCCGCTCGCAGGCTCAGCCCGGTGGCCCCGCCCCGCGGCGGCCGGTGGCTGTGCGCGTGCGCAGGCGGGAGCGCGGCGGCTGGTCCAAGCCCGGCGCTGGGGGCGCGCAGCCGAGAAAGGGTTTGACGTGGCAGTTCCCAGCCCAGCTGCAACTCCGAGCGTGAGTCCAGGCTAA
FP014316	chr16	2537820	2538071	+	1	train	CGCCAGGCCCCAACCCGGAAATGCAGCTGGAGCGGAGGCGGAGCCCACTAAGGCCGCGGCGGAGCGACGATGGGCGCGGCCAATGGGCGCGGGCGTCGGCTGCGGCGCGACGGAAGTCCTGCCCGGCGCCGCGCGGGGGCGGGGCGGCGCCGGGGGCGGGGGGCGGCGGGCGACGGGGCGGGCGCAGGATGAGGGCGGCCATTGCTGGGGCTCCGCTTCGGGGAGGAGGACGCTGAGGAGGCGCCGAGCCG
FP006217	chr6	18264473	18264724	-	1	train	AGCCGGGGAGAGCCCCGCGCGCGCATCCCTCCGCCCGCGGCCGAGGCGACCCCTCGCGGTCCCGCCCGCCCGAGCTTCCCGCGGAGGCACCCGGCCGCCCCGCCTCCCCCGCCCCGCGCCCGCCAACTCCCAGCTCGCGGCCCTGATTGGTCGCCGCATTCCCGCTCTCCTTCCCGAACCGCCATTTTGAAAATCTTGTTGATTCTGGGGAGCCGAGCGCGCGGCGCGAGCGTCACGCCAGACAGCGGCCC
FP018734	chr22	30881480	30881731	+	1	train	TCCTCCTGCTCACAGCTGAGCACGAGTCTCCTGGCCCCAGGTAGAGTTGTCACACAGCCTCAGAGAAATGAGACCACGTTCAGGCCTGGGCTGGGTCAGCCAGGCCCTCCCTGGGCCCCTGGGAACCTCCCCCTGCCAGTCCCTCTGCCGGGCCCCAAGCCTAATCCAGCTTATCAAGCACACAGCACACAGCCCAGCTCAGCATTCTGAGAACAGCAGGGCCACGCCAGGCGCCTGCCTCTCCCCACAGC
FP008843	chr9	33026320	33026571	+	1	train	GAATTATCGCAGGAATTGTTACCTTTTTGGTGAAATTTAGGGAAGGTGGTGTTCTTATAATCGGATTTTGCAAAGAGATTGCTCGGCCTTAGCTCTCTTTTTTATTAAAAGCTACCAGTTGAAAGCCGGTTAAAGTTGCATTTTCTTATTTCAGGCAGTAGAAGATGGTGAAAGAAACAACTTACTACGATGTTTTGGGGGTCAAACCCAATGCTACTCAGGAAGAATTGAAAAAGGCTTATAGGAAACTG
FP009269	chr9	122159724	122159975	-	1	train	CATGGCGCCTGTTTTTAGCTTCCAGCTGTTAGGATCCGGAGTCTCAGTGGATACGGCATCACTATGACAACCTGGAGGGGTGGGCGGAGCCGAATCGAAGCCCCACCCCGCTGGAGGCTGAAAAGTTCTTGGTATTGCGCACGCTCCCTCGCTCGTGGTTGGGCAGGGCAATACGCCTGCGTGTTGCCGGATGCGCATGCGCAGGCGCCGTGTGGCACTCGGCGGTCGAAAGGGGAGTTCAAGGAGACGGG
FP019367	chrX	70454976	70455227	+	1	train	GGGAAGGCGCGCCCTAGCCTGCGGGCCAGTGGAGTGGCCATTGGCCGGCTGAGGGCGGCCTCCAGCCGCATGCCCCGCCCCCTGACCCAGGAGGAGGGCGGCAGGCGCCCCTCCTTCCCCCCATCGCGCGCCTCAGGCGTCTCCTCCTCCTCCCTCCTTCCCCCTCCTCTCTCCTCCCCTCCTCCCGCGCGCCCCGCTTTGTGTCCGTGGTCTCCCGCGCGGGACGGAGGGACTGGCCGGAGCTGGCGCTT
FP000141	chr1	10032937	10033188	+	1	train	CCTGACGGGGGTCACGTGATCCCTTTCAAAGATGGCCGCCCTGTTGTTTTGATGAATAATACTTGGTGGGGCGAGGGGGAAAGAGTAGGGGTGGAGGGGTAGGAGGATTTACTCTTCCAGCGAGAGCTACGCGCATCCCATCCTCCCCCTCCCCCCTACCCGGGCTCCGGCGTGGAGGCGGGGCGTGGCCGGCCTGCTTTGGGAGGGGAGGGGCTTCCCTTACAGTGCTGGGCTCTGCCAGGACGGCTGTG
FP012164	chr12	64451937	64452188	+	1	train	GAGGGCAACAGTGGACGGGGTGTCTCCAGACATCCCCCTCATTACCGTGGCCGCGGAAGCCGACTCGGCAGTTGCCGCCGCGGCTGTGGTGACTACCAGACGGGCCATAGGCGTGCGCACGCGCACCCGCACCGGCGCGCCGGCCGTCGGTCACGTGGCCTCCGGCCAGGGCTTGCGAAGCCGGAAGTGTCCTGAGTCTCGAGGAGGCCGCGGGAGCCCGCCGGCGGTGGCGCGGCGGAGACCCGGCTGGG
FP014499	chr16	22206109	22206360	+	1	train	GGGAGCCAAGACTTAGCGCTAACCATTGCACCTGGGGCTGGGAGCCAGCGCCGGAGCCAGGAGGCCCGCTAGGCGGCTGCAGGCGCTGTGATGGCCACCTGGGGGCGGCCACGTGAGCGCCACGCCGTGCGCCCGCCAGGCCAGCCCCGCCCCTGCCCGCCCGCTTCTGCTCAACCTAGACCAGCCCCAGCTTCAGCCTCAGCTCCCCTCCTTCCTGGATCGAGCGCCCGCACTCCCGGCCCTGCAGCCAC
FP017713	chr19	55069460	55069711	-	1	train	CAGGTGCGGCTGCTTCAGGTAGGGTGGTCAGGGAAGGAGGCGGCATCTGAGCTTTGACCTCAATTTCGAGAATGAGCCAGGAATGCAGAGAGGATTTCAGGCAGAGGAAACCTTAGGGAGGCTGGCGGGTTAAATCTGCAGGGGTGTGGCTGGCCGGAGGAAGTGAGGTGTGAGACAGTCACGTTGATGGTGACAGCAGCAGTGACCGTAGCAGCCTAGAGGTGGCCGAATGCTTACTCTAATGGGCTGGC
FP006611	chr6	46652741	46652992	-	1	train	CCCAGAATAGAAAAACACAGCGGAGCCCAGGAGGTTACTGGGCAATGTAGTCCGCCGGGCTCCGCTACGTCTCTGATTGGCCGGCCGCGGGTCCCTCTGCGGCTCCGCCCCCGGCCTCCATGGCAACGCGGCTGGTTCTCGCCCGTCAGTCCTAGCCCGGCCCTGCCCCTCCTTGCATTTTTTCCGCGCTGGCTGAGATTCAAAGAGAAGTGGAGGTGGGAGGGAGCGACAATGGAAAAATCACCTGAAAA
FP001871	chr1	228082894	228083145	+	1	train	GGCCGGGCCGGGGGCGGACGCAGACGGGCCGGGCTGAGGCTGGGGCCCGGCCGGAGTCGGGGCTGGGCGGACGGGCGGGTCGGTGAGCTCCTCGCACCCCTCACAGGTTCCCGAAGTCGCTCGCGGCCGCTTGTCCTCCTCTGCCTGTCCCTGCCCCCGCCCGTCGCCCGGAAGTCCGCTTGGACGCCGGGCTCTTCTCCAGGAAACCTGGGCTTCCTGCTTCCCTCGCCTCTGCCTTTCTCGTTTCCCGA
FP003764	chr3	57060463	57060714	+	1	train	CCACCCCCACAGTAGGAGTCAGAATCAGGGACTGGAAGGGATGGGGTTCCTGGAGGAAGCTCACAGCAGCAGCAGCCTCTGAGTCACAACTCAGCCCCTCCCTGGCATATCCCTTTGTTTGCTGGGAGTTGAGCTGTGACTCACAAAGGCTCTGGTCCCGGGCTGGGCCCATGTGACCCAGGGTCCTGACTGCCCTGCCCACAGTCACTGAGAAAATTCTTTGTGCAGGCTCACCTTCCCCTCATCTCCCC
FP016416	chr18	59273344	59273595	-	1	train	AAGGGGAATATTCCTAGTGCCGGCCCAGGGGTGGCCACTAGAGGACCCTGTGGCTGGGCCGGGTGAGGACACGCCCCTCTGTGGAGGGAGGGGCCGAGAGAAGGGGCTGGGTCCAGGGCGAGAGGGGAGGAGCCGAGGGCCAGTTGGCTCCCCGCCTTTAGCGCCGAGAGCCCCAGCTACCCCGAGCCCGAACTTCCGACTTCTGGGACTAAGGCGGCAGCGGGCTGAGCGCTCGGCCACCCCCAGCGTGC
FP005863	chr5	143404005	143404256	-	1	train	GTAGCCCCTTTCGAAGTGACACACTTCACGCAACTCGGCCCGGCGGCGGCGGCGCGGGCCACTCACGCAGCTCAGCCGCGGGAGGCGCCCCGGCTCTTGTGGCCCGCCCGCTGTCACCCGCAGGGGCACTGGCGGCGCTTGCCGCCAAGGGGCAGAGCGAGCTCCCGAGTGGGTCTGGAGCCGCGGAGCTGGGCGGGGGCGGGAAGGAGGTAGCGAGAAAAGAAACTGGAGAAACTCGGTGGCCCTCTTAA
FP001932	chr1	236142352	236142603	+	1	train	GGGGCGAAGGCGCGCGGTGCCCGGGGGAGGGCGGGCCGGGGCGCGCGGGCCGGGGCGGGGGCGCGCGGGCCGGGGCGGGGGCGCGCGGGGCCGGTCGGCGCGCGGGGGCGGCGGGCGCGGCGCTGCCAATCGCAGACAAAGGCCGTCCCAGTGAATCATGTGGTGCCGGGGGAGGAAGTGCGGCTTGTTTTCTTTCCTCCAGTCTCGGGGCTGCAGGCTGAGCGCGATGCGCGGAGACCCCCGCGGGGGCG
FP012834	chr13	75549590	75549841	+	1	train	GCTTATTTTTTTCTCCTCGGCAGCATCTTAATTTAAAATATGACACTTGACCTACGGCCCTGCACGGAGCGGTTAAGAGGGTGAGAGGCCCGTCAAACTCTTTTTGGTGTTTAGGCGCTCCTCCTCCGGGCGGTGTGTTGGGAGGGCCCAGGTCAAGGCGCGCGTGGGCGGAAGCGGCGGCGGCGGCGAAGGCGGCGGCTGTCAGAGCTGGAGGGCCGGGCACCGCGGCCATGGAGGGTCAACGCTGGCTG
FP008569	chr8	117520512	117520763	+	1	train	AACGCTGAGCCAAGACTGGGAAACGAACTCTGGGAACTCACCCCAGGCTCCCCAAGAACATCGCCCCTCTGGCTGGAGCGCAATTGGTGATTGGCTACTTAACCCGTCCGTCCTTTCCCGCCCAGGGGTCCAATCCAATCCAGCCCGGCTCCGCTCGGAGACAGTTCGCCGAGTGGGCGGTGTCTATGACGTTTTCTGACGTGTTACGTCACAGTGGGCGGAAGTCGCGGCCGCTGTTTTGAAATCGGGCC
FP008669	chr8	143635857	143636108	+	1	train	CGCAACCCTCCCAACCGCGTAGCAACGCCGCTCCTGCCGGACACCCCTGGCCCCTCCGGGGGTCTTAGTCCCCGGGCCCGGAAGTCCGCACCACTGAGAGGGGAGCCGATCCCTGGCGCTCCTAGAACGGCGCAGGAAGTTTCCCAGGCGGGGCCTCTCGCGACTTCCGGTCGCGGCGGGCTGGCGGCGGTGCAGGCTTTGTCGGCTGATCTGTGGGGCCCGCGCCGGCGGGGTCCAGTCAGCGGCTGCAG
FP019333	chrX	55717548	55717799	+	1	train	GGGTGGGGCCAGGCCCACCCACATTAGAGAGGCTGAAGGCTCCCAGCGCCCTACCCCGCCCGAGGGGCGGAGCCTGAGCCAAGGCCACGTGATGATGACAGACGGCACTCTGGCTTTCCTGGAGCTGTCTCTATGGTATTCTTCCCAGCCCACCCGTCCCTTTGGTAGCGGCAGTCACGTGACAGACTCCGGGGTAAGGCAGATGCCCACGTGATCCTGGCCTGCAGTTGGGTGGCTGCGGTGAGATACCT
FP014227	chr16	589461	589712	+	1	train	GGCGCGCCTGCATCCTGCCGCCCGTCCGCGCGTTGAAGGGGCGGATACAAACAACGTGGACTTCCGAGCCCCTGATTGGCGAGCGTGTAGGAAAGGGGCGGGGTTAGCAGAGCCGTGATGGACATGCAAGCGACCCAATGGCGCCGGCGACGGGGCGGGCGAGGACAACGGCGTTGTGGGCCGGGGGCGGGGCGGCCGGCGGCTCTGGGATTTCTCTGGGAGGCAGCCGCAGGGAAGGGAATGATCTTGGT
FP009284	chr9	123268535	123268786	-	1	train	TTGGCGCAGGCGGCGGCACAGCAAGGCGGCGGCGGGCGCTGGCCGGCGCTCTCAGGTGCGGCAGGACGCGCGTGGAGGGGGCGCGCGGCGAACGAGGGGGCGGGCTTTCCGTTCCCCAGCAGCCGCGGGGCCCGCCCCCTGGCCTGGCCAATGGGCGCGCTCGACGGTGTCACGTGCTCGCTGCCGCCGCTGCCGCCGCCGAAGCGGAGACCGGAGCCGCGAGCGCCACCAGGGCAGCAGCCGCCGCAGCC
FP001074	chr1	113979240	113979491	+	1	train	TCTCTTTCCCCTCCAGAGGACCCTACAGCCTAGGCGGGAGGTGGTTAAGGCTTCTGGCTGCTGTGCAATGGGGCCATCTGTGTTTGATCAATCCTGGCGGAAAGGAGGGGGTGGGGGTTGTAAAGAGAACTGAAAGCATTCCAGAGTAGTGAGAGAGACCCAGAGATCAGGAGAGAAGGCACCGCCCCCACCCCGCCTCCAAAGCTAACCCTCGGGCTTGAGGGGAAGAGGCTGACTGTACGTTCCTTCTA
FP010883	chr11	62707497	62707748	+	1	train	GGGGGAGTCGGCCTTGGCCTCAGCTCTGTTGGGGAGGTCTCTAGCCCAGAGTCAGGATGCCTGCAATGGGGGAGGGGCAGGCAGTGAACTAGCGACAGTGGGGGTGTGCGCAGGGATGCAGGCAGCTAGGCTCCCCCACTGGCCAGGCAGTTGGTATGCTCCAGGATCTGAGCAGCTCCTTCTAGCATCCTTCATCCTTCAGGTACCAGCCATCCAGACAGTGCTTGAGCTGCAGAAACTGAGACCAGACC
FP002827	chr2	169069359	169069610	+	1	train	ATCATAAAATGAATGTTATTGATCATTTTAAAGTCTCTTCTAAAAGTAGCTAAATTTATACTGCTGATTCCTTTTGAAACATGCGACCTACAGCTTCTTGGCTTTTATGAGCTATTCAAGAGATATTTAGTCATCACGTTGTGTCACAATGGGAGTGACTCACAGAGCAAGGAGAGAACCTGAGGATTCCTCACACATGTAGTACTCAGAGCTCTACGGAAACCCAGGCACCTCGACCTCAAGAGGATCAG
FP002768	chr2	156435092	156435343	+	1	train	TGGGATTTAATAAATATATGCATTCCTCTGGGTCAACTACAGGAATTCTGCGGTATTTAAGGACATTAAATGAGTGACTTAATACTTTATTTAAAAAAATACCACTGCTTTTTGGTTCACACTAATTTTCTTCCTTCTTTTGGAAAGGCATTTCCGTCCATCTCCTGGATATTTTCCAATACCACGATAGAAAGGTTATTAGTACAGCACCCAGAGGTCTTCGGAGCAAAGCTGTTAAAGAAGTGGGGTCC
FP002353	chr2	68252446	68252697	-	1	train	CGCGCGCTCAGCGTTCCCCAACCAGAGCTCGCCAGAGCGCCGCGGCACTCGCCGCCCAGCGGGGCGCGCAGGTTCCCGGATGTGCGGCGCTCGCGGAAGCCCCGCCCCCGCCCCGCGCGCTGCAGCACTCCGCTTTCCCCTCCCTCTCCGCCCTCCCCTTTTTTCGTGCCTTGAGGTTGCGGGTCAGCGCGAGCCGCTGCAGTGAGTCCGTCACGGCTCCGGCGCGAGCGCGAGGCTGCAGCCCCCGAGTT
FP011413	chr11	119030826	119031077	-	1	train	CTTAAGAATTGGACATGCCGTGCCCAGCCCCTAAGGGGCGCTATGCTCGTTCGTCCAAGGAGCAGCGTGACCAGCGGGAGGGAGTAAGTCGACGTCCCCGGCATGTGGGGCGGGGAACTCCACTGGCCGGCAAGCGAAGCAGCAGGCGAAGACCACGCCCCCGCCGGCCGCGCTTGCGCAGCCTTCGCTAGCCCCGCCCCGTCCTATTCGGGCTCCCGCCTCTGTTCAGGTAGGAGGCGGTATGCGGGGGA
FP012551	chr12	123270232	123270483	-	1	train	AGGAGTGGCCCCGTTGGTCACCGGTGCCACGCTGGCTCTCCTGCTCGAGTTTTGGGCCTTGTTTGCAAACGTTGACCTCGTAACCCCCCAAATCGGAAGTGGGACCCATAAAAACGATACCTAAAAAAAATCTTGGACACTGAGGTGTAGGGGCAGGAAAGGGTTAAGCGTCAGGGTGCGGAGGGAGAGTTGAATTGCTCACTTGTGGGCTCGGAGTCCCAAAGAGCAAACAGCCACTGTTTTTTTCTCAA
FP013813	chr15	51681302	51681553	+	1	train	CTAGGACTGCAGCACAGAAAATACACCAGCTGGCCGGTCGCCCCTCCTTTGTTCCATTCCCGGGGGATTGGAGTAGCGTTGGAGTCACCGACGCCATCCCCTCCCGCCTCTGGCGTGCATGGAGCATGCGCTTCCTTCCTCACTTCCTCTGCAGGAGGGAGCGAGAGTAAAGCTACGCCCTGGCGCGCAGTCTCCGCGTCACAGGAACTTCAGCACCCACAGGGCGGACAGCGCTCCCCTCTACCTGGAGA
FP000862	chr1	77933082	77933333	+	1	train	TCTGGGAGGCAGAGGTTGTAGTGAGCTGAGATCACATCATTGCACTCCAGCCTGGGCAATAAGAGTGAAACTCCATCTCAAAAAACAAAAACAAAAACTAAAAACAAACAAAAAAAAGAAATCCCAGTTTCTTCAAGAAATAGTCCCTTTTTAGTATGTGTAATTCTGGCCAGAGTGATAAAATAATTATTTTAAATAGGTAGTAGATGATGACTCCCCAGAGATGTATAAGACAATCTCTCAAGAATTTC
FP010618	chr11	19241569	19241820	-	1	train	AAAAAAAGTTGCTGAACTTTTCCCCCAACTCTGCCGTAGAGGCGGGAGTGGAGGGCGGTGCCTGCACCGATTCGCCGGCGGCTACGGTCGGGAGGCTCGGATTGGCGCTCGGGGGCCGGGGCCGGGGCGAGCGGGCGTGGGGGAGGGGAGCGGCCCTCCCCGCCCGCGGTATCGGCTCGCGCCGGGAGCGGGTTAATTTCAAATCGGGGGCTTGGCTGCTCCTGGACGGTCACGCTCCCTCTGCCCGCCAG
FP013025	chr14	23286066	23286317	-	1	train	ATCTTCCCTGCCCCACGCATCCAATAGATGACCCCAGCTGAGGCCCACGCAGCCAATACCTGACTTCAGCCGAGACCCACGCAGCCAATGGCCAGCCCAGTGCGCTTCGCCCGTGCCCGCCCGCCTGCGCCTACCAATCAGGACTCGGGCCACCTTGGCACCGCCTCTGGACGCCCACCCCACCCATCCGCCCCGCCTCCAACTTGCTCGAGCAGGGCTGGGTAGACCGGCGCGCTCCCGGGGACGGTGGG
FP014708	chr16	53434290	53434541	+	1	train	TGCGGGGAAGGTGCTTTTATTTCACCCCTGGTGAAACTAGGGGAGCTAATTTTTTTAAACATGATTTTTGGCCCCCTTGAACCGCCGGCCTGGACTACGTTTCCCAGCAGCCCGTGCTCAAGACTACGGGTGCCTGCAGGCGGTCAGCGTCGTTTGCGGCGGCGCAGGCGCGGTGCGGGCGGCGGACGGGCGGGCGCTTCGCCGTTTGAATGGCTGCGGGCCCGGGCCCTCACCTCACCTGAGGTCCGGCC
FP010240	chr10	114174746	114174997	-	1	train	CAGCTATTTTCATTCATGAGATATGTGTAAGACCTAAACCAAAGAACATCGTTTCCCCGGACTGAGGAGGTTTGCGAAACCTCCAGAGACTGAACTGCTGACCTTCCTTCTCTGGGCCGGGCGAGGCAACTGTACCCAATACACTTTTTGTTGCCATGCTGCGTGCCAGGTTACTCAGCCGGGAGAGGACACTTCCTACCATCAATGCGCTCAGCTTGAAGCCAGAGACGAAGGCACCGACAGTTTATTAT
FP010688	chr11	34511740	34511991	-	1	train	GAAGCCCAATCTTGGCCACTTTTTTCTATATTTTGCACCCTATGGCCTAGTTCTGCCCAGTGATGATTTGGCCCGTAAACAGCCAATGTGTAGATGCTTAATTGGGCCAATTTTTGGTCACATGCCCAGAGTGAAGTTGATGATCACCACCAGAGTCAGGAAGGAATTTTCCTCCTCTGGCAAACTGGCCAAGGCTGAGTGGTTTGCTCCTTCCCCTCTCTCTGGGAGGCTGAGCAGGGGTGCCGGGTTGC
FP013165	chr14	49767952	49768203	+	1	train	CGGGCCCCATAGATCTGCGGTGCCCAGGCCGTAGGCGGAGGCGCGGGCAGAGGCGGCGGCGGCGGCGGCGGCGGCTGGAGGACCCGGCGCTGGGGGCGCTGAGCGGCAGCCTCGCTTCCGGAGCTCGGCGGGCGGCGTCAGCGGAACGGGCGGCTTCCGGCCGCGGCGTAAACAAGGCCGTGGACTGCAGGAGGCGGGGCAGACGGGCTGCAATAGGGAGCCGGCCCGACGCGGACCGCTTCCCTGCAGTG
FP010025	chr10	91909431	91909682	-	1	train	ATTATTTTAAACATTGGAAGCAAAATACAAGATTTGTATTTCATACTACAGGACCGCTTCCATCATCCATCTGGAGAGACAGAGAGAGAGAGGGAGAGAGAGAGAGAAGCGCGGCGGTGATAAGTCTCCCAGTGTGAGAGAAATCGGGGAGGGGCGAAGAAGGGTAGGCCTCTTCTGTTCCATTGATACCACGTTCATTCACAGGGAGCCGGCTAAAACAGACTACTCCCCCGTGCAGTCGTTCCCTCTAC
FP019085	chrX	12866871	12867122	+	1	train	ACAGAGTTGTTTGGATTTAGACAAGACGTTGCCCCAATAGTGGTGATAGAAATAAGAGGAACCCCGTGCTTTTGCAAAGCCCATATCTGGGGTGGCTTAAATAATCATGCTCCTCCCCATCCCCCGACCTGATCTTTGTAGTTGGAAACTCCAGGGCTGGCTGCCTGTAGTCTTTGTGACTACACTTCCTGCCTCCCATCACTTCATCTCAGAAGACTCCAGATATAGGATCACTCCATGCCATCAAGAAA
FP004743	chr4	65669452	65669703	-	1	train	CCCCCAAGCGGCGGCGGCGACACCCCCATCACCCCAGCGTCCCTGGCCGGCTGCTACTCTGCACCTCGACGGGCTCCCCTCTGGACGTGCCTTCTCCTGTGCGCCGCACTCCGGACCCTCCTGGCCAGCCCCAGCAACGAAGGTAGGGAGATGGGGGTGGGGACCGTGGCGACCTGGGGCGGAGGGGCTAGGCGGGGCCTGGGAGTTCAGGTCTCAGCGCGCTGGACGCTGCTGCTGCAGACTAAATCGCA
FP008328	chr8	53843220	53843471	-	1	train	CCGCCTCCCACCTGCCCTGGCTGGCTGGGGCCCGACCCTTGCGTCGCTCCGGGGCCCAAGTTACGGACTCGGGAATCCAGTTGTCCGCACCGAACAAGCCCCGCCGCAGCGCCCGCCCCTCCTCTTGCTGCTTTGGAGTCTCCCGGCTTCCGCCTTCCGCAGCAGTCACGTGCCTCCGATCACGTGACCGGCGCCTCTGTCATTCTACTGCGGCCGCCCTGGCTTCCTTCTACCTGTGCGGCCCTCAACGT
FP013773	chr15	44711312	44711563	+	1	train	GTACAGACAGCAAACTCACCCAGTCTAGTGCATGCCTTCTTAAACATCACGAGACTCTAAGAAAAGGAAACTGAAAACGGGAAAGTCCCTCTCTCTAACCTGGCACTGCGTCGCTGGCTTGGAGACAGGTGACGGTCCCTGCGGGCCTTGTCCTGATTGGCTGGGCACGCGTTTAATATAAGTGGAGGCGTCGCGCTGGCGGGCATTCCTGAAGCTGACAGCATTCGGGCCGAGATGTCTCGCTCCGTGGC
FP015991	chr17	75227877	75228128	+	1	train	CGATCTGCCTGTCTTGGCCTCCCACAGTGCTAGGATTACAGGCGTAAGCCACCATGCCCGGCCTGAAAGTCAAGTTTTGTGTCAATATTTTCACAGGCCTGTAAACATTGTGAGAAGTCCCTGCATGGAAGAATTAGTGAGAATAACTTAAAAACACACAAACACACCATCGTTTGGCTTTGTTGGTGTCTCTGCCCATAGTTGAGTATTCTAAATTTCCTGTGTTATAACAGTGAGTGAGTTTCACCAGG
FP003486	chr3	33718162	33718413	-	1	train	GCAAAGAACGGCCCGCCTCCCAGGGGGCTCGGCCCAACTCGGACCCCAAGTCTCCCTAGAGGTCCTATCGCTCCCAGCGGTTTCCGCAGCCACCTCCACCACCTCCGCAGCAAAACGCTAGCCGGACTGGAGGGCCCTCGCCGGCGTCGTGCTGACGTCACGCGCGTGCTGACGTCGCCCGCGGCCGCGGCCTCTGAAGCGGGCTGGGGATCGGGGGGCGCCGAGTTTGACTAGTTTGGGGGCGGCTGGGC
FP009078	chr9	94278666	94278917	+	1	train	TTCCTCTACTCCCTACTGAATTGGGAGGCTGGACAGAGTGTGAAGGGTGTTCTTCCCAAGTACCTCTCTCACTTCTCATCTCTTTCCCCAGATTTGCCTCTGCAACTTGACTCTCCTCTAGGAAGAGTACTCCAGAGAGCAGGGATATGTCACCAGGACTCCTGACAACCAGGAAGGAGGTAAGTCCTGAAATTTCTTCCTAGCTATTGCAGGAAGGTTTCATAATCTTTTCTTGTCCTGAAGGCTGCCTT
FP000846	chr1	75786055	75786306	+	1	train	AAGCATATGGTCGCAGGACTTTGGACGCAGGCAGTGAGGACACCTATGAGCGATCAGTGGGGTCCTACGGCAGCCGTGGAACTGCTCTGCCTTGCAGGCGAAAGTGTGGTCAGCTATATCGGAGCGCGATGGGGGCCTGAGAGGCGCATCTGCGCAGGCGCCCGGCTCCTAAGTCTACCCAGGAACTGACCCTGCTCTCTCCTTTCCCTGTTAGACATGGTAAGTGTGAGTTTAGCGCTGCTGTCCGGATG
FP003650	chr3	49199299	49199550	+	1	train	TGTCGCGGCGAGGGCGGGGCCCTGGGGACTGGACGGAATCCCAGTTGGTCAGAGGAGACCCTGGGGGCGGGGCCGCGCGAGTTGCCGTTAGGACAGTTAAAACCGTTAGCTGCCTGTCGTGGCGGGGCGGGGCCCTGGGACAGGACGGAATCCCTATTGGTCAGAAGAGGACCTGGGGGCGGGGCCGCGCGTGCCGTTGCAGAGGCAGCGGGACGCGGCCACCTCGAAGCCACGTCAGGGCAGCCCCAGGT
FP000788	chr1	62688335	62688586	-	1	train	AGCGAAGCGGCTGGGGCTGGCGCCTCGCTTCCTCAGCGCTCCATTCTTCCCCTCGGCTCCCGCCGGCCGCAGCCGCCTTCCGCAGCCGGGGTTCCCGCCGGGATTGACGCGCTGGGGGAGGAGCGGTTTCTCGTTGCGCGCCTCTAAGGAACATTACGGCAGGGCTCGTTCCTGGCTCCGGCCGCCAGCCCCAGCCTCCCAGGTCCGGAGCCCGGACTGGCGGAGGCCGCGAGGGAGGGAGCACGAGCGAG
FP002818	chr2	166073468	166073719	-	1	train	AACATCTCTTAGTCCACTCTTTAAAATATCTGTATTCCTTTTATTTTAGGAATTTCATATGCAGAATAAATGGTAATTAAAATGTGCAGGATGACAAGATGGAGCAAACAGTGCTTGTACCACCAGGACCTGACAGCTTCAACTTCTTCACCAGAGAATCTCTTGCGGCTATTGAAAGACGCATTGCAGAAGAAAAGGCAAAGAATCCCAAACCAGACAAAAAAGATGACGACGAAAATGGCCCAAAGCCA
FP015505	chr17	37205811	37206062	-	1	train	CTTTACCTGATTTAATCTCTCTGTGGCTATTGGGCAGTGGTTTCAATTTGCTCATTCCTACTGCTGGGCAAACCAAAATATCCTGTGGGGTGTATCTCTTTTTCAGGTATTCTACCAAATATTATAACTTCTATCTGCATCTGTATTGGCCAGCCTCATAATTTCTTTGACTTGTGTTCTTGAGTTACAGGTTCCCAGGGATGAACCAATTCACATTCTCAATGTGGCTATCAAGACTGACTGTGATATTG
FP008751	chr9	4666415	4666666	-	1	train	ATAGCAGGAGCTGATGGAAAGAAGATCGGGATTGTTAGCATTTCTCTATCCGTCTAGCACGCCAGACAGGGCCAGTCTCTCCGGCTGGGGGCACTGCGTGGGCAACAGCGACCGTGAGGGCGCGCGTCTGCGCTGGGGCCTCGCGCGCGTTCCACGATGAAGACTGCGCGTTTGAGTGTCCCTGGCAACGCCACGCTTCCGCCTTCTGGAAGCCTGCACTCCAGCTTCGAGTGTGAGAGACCCGTCCCGGT
FP003789	chr3	62217257	62217508	+	1	train	CAGAGTGTGCTGCACCTACTTGACTCTGTCTTCTTACCTGTGAGCAGGGGATACATCAGGTCCTGTCTGCGTCACAGGCCTGTGCCTGAAACAAAGGAGGTGGTCAACATCAAATGCATTAGAGGAAATCCAGTGAAACGCATCGTAACACACTTCTTCCTGACTCGGGAGTCTTGCCTCTTTAAAAATCTGCTCCCATAGCCAATTCTTTCTTCCCTACTCAGCCTACCTTCTCTCACCTGGAAACTCAA
FP009398	chr9	129111135	129111386	+	1	train	CCTTGGGTTAGGGTCAGTACCACCCACAAAGATGACAGGTGGTACTCCGGGACAGGAGACTGTCCCGGAAAAACATGGCTGAGCACAACCCAAACTTGACGCCCCGCACAATCGTGGCAGTCGCGGCGCCCGACGTTCGGGCGGCCGTGAGCGGTCCTAGCGCTTGGCGGCCGTTGGCGCGCATGCGCCCCGCGCGCCCCGCACTGACATGGCCGTCGCCCGGTTCCGCGCGTCCGCCGCGCGCCGGCCGT
FP014959	chr16	81077187	81077438	-	1	train	CTAGCAGGCACAAGGGCTCCCCAACGGGACTCGCCGACCCGAGAGCCCGGGTGGGCCTCAAGCCCCGCCATCTGAGACCCTCCGTCGCTGGCCCTTCCGGCGGCAGCGCCCGGAGCGGATAGGAGATGCCACGAACCGCCTCGCCAGTGCTAGGCTTTGTTGGGCTACGTCACTTCCGCCGCGGTCCCGCCCCCAGCGTGGTCGTAACCCAAGGCAACGGCCCATCCGGCAGCGACCTGAGTAGCTCTTGC
FP001529	chr1	172444018	172444269	-	1	train	CACATCCCATCTTTCAATAGGGGATTGCTGGGGATTTGCCTTCCTAGGCTCAGAGCATTCGGGTTTGGGCCAGTCTCCAGGAGAGGAGACCGGAGGTCCTTCGAGCAGCCAGGAGGGGGCAGCAGTCACGTCGCGATGGTTCCAGCCGGGGAAGGGTGCCCTCGCTAAGGAGATTGCGGCGGACCCGGAAGTGCTTGGCCACAGTCGCAGCCCCGGCGCCCCGAAGCGGGAAAAAGGCTGGGTGCCGCCGT
FP011879	chr12	48818324	48818575	+	1	train	GTCTCCGCCCGGGTCGCGTCTCCCTCGCGGTCTCAGCGCCCCCTCCCTCCCCCGCCGCAGCCCTTTGTTTCCCGGCGGAGTAGTTCCTGGGTTGCCGAGCGTCTCTGTCTCCGCATCTCTCCTCGCCCCGCCTCCTCCCTGCCTGGATCCGCCCTGCCCGCAGCCTCTCCTCCCCTTCCTCCCCGCCGCCACGGCCCCGTAGGTGCTCGGGGACCCACCTTCCACCTAGCACGGGTTCGTTCCCCTCTCCC
FP017086	chr19	32438951	32439202	+	1	train	TGATGATGATGATGATGATGATAAGGACAATTATTGAGAACCAGTGGCCAGAATCTTAAGTATACTTGTCCTTAATGTAATATATCTTTCGTTGAGGAAGTCTTCGAGGAAATTATTGTAAAAACTATCACTTTGGCCATTTGTGTTTTCTTTTGTGCAGAATAGATACCACAAGAGTTGAGTTTACCATCCCACTGAGGGAGAACTGGGCGCTGCCATTCTTTGCAATTCAGATAGCAGCAATTACATAT
FP005895	chr5	149730109	149730360	+	1	train	CGCGGCGCCAGACACGGCGCAGGAAAGTGGGTGAGCGACCCCCGGCTCCCGCGGGCGCCGCGCGGCCCCGCCCCCGCAGCTAGCGGCCCTGCGGCAGCCGGGGGCTCGAGCTCCGCCCTCCGCCTCCCGCCGGCCTCACTCCCTCCTCCCTCCTCCCTTGCTCGCTCGCTGGCTCCCTCCCCCCGGGCCGGCTCGGCGTTGACTCCGCCGCACGCTGCAGCCGCGGCTGGAAGATGGCGGGGAACGACTGC
FP014797	chr16	67109742	67109993	+	1	train	GAAGTACTGGGCTCCTTGCTCCTAGAGCCCGAACAGTTCCCAACGTTACAGCCCTGCCCGACTCAGCCTCCGCAGAGACGGCGGCGGGACCCGCAGTTTGCGCGTGCGCGGCTCCGCCCCAGGCTCCTCCCTCGCGCTGGCACTCCCGCCTTCGCCCGGCCCTCCCCGCGCTTTACGGCCCGGCACGCCACTTTTACTGCAGTCGCGCCCGCCGCCGTCGTTGCCCCCGCTGCCGCGGCTGCTGCAGGTGA
FP019204	chrX	45200805	45201056	-	1	train	GAGTAACTCTGGAAACAGCGTTATCAGCATCCTCCACTGACTTCCTCACCTCCTCCCCAGCCTCATTAGCCCCAAGCTCCTCAGCATCCTCAAAGAGAAACCTGGGAGGCTGGGATGGGGTCAGCACCCAGAAGCCAGCCCCCTCTGACAGCTTCCTCTTTGGCCAAGCCCTGCCTCTGTACAGCCTCGAGTGGACAGCCAGAGGCTGCAGCTGGAGCCCAGAGCCCAAGATGGAGCCCCAGCTGGGGCCT
FP008773	chr9	12693184	12693435	+	1	train	CTTCTGGCCTTTTCTTAAAACTTTAAGCATCACAAGGAAATCAGTTGGAAGGGAATCATGTGCTGATCAAGTCCTTAAAGGGCAGAAATATTCACTGAAGTGAAAAGGATTAGTAAAGGGTGGAAAAAAAGACCAGCCCCCCGCCTAGTTTGGGTGAGCAGATTTGGGATTAATTATCAGGCAGCAATCCACATGCACTTAACAGTTCTGACGTGAGAGGACAAGAAACACAAGCAAATATAAAACATTCA
FP010850	chr11	61752435	61752686	+	1	train	CGGCCCCGCGCCCCGCAAGCCTAGGACTGCCCAGCGCTGAGCGCCGCCTGCAGGAGTCGGGAGCGTGGTGCCCGCGGCCGGGCTGGGCGCGGCGCGGGCCGGCAGGGGGCGCTGGGCGCGGCGGGCGGGGCGCGTGGGGCGGGCAGGGGCCGGGCCAGGGACGCGCGGGGGGCAAGCGCGGCGGCGGACCGGGCGGGACCGTAGCCGGAGCCCAGCCGGGACTGTCGCGCGGGCCGCGCCGGCGATGCCGC
FP002881	chr2	176107387	176107638	+	1	train	AGCCAGCATGTACCTGCCGGGCTGCGCCTACTATGTGGCCCCGTCTGACTTCGCTAGCAAGCCTTCGTTCCTTTCCCAACCGTCGTCCTGCCAGATGACTTTCCCCTACTCTTCCAACCTGGCTCCGCACGTCCAGCCCGTGCGCGAAGTGGCCTTCCGCGACTACGGCCTGGAGCGCGCCAAGTGGCCGTACCGCGGCGGCGGCGGCGGCGGCAGCGCGGGGGGCGGCAGCAGCGGGGGCGGCCCCGGCG
FP012796	chr13	49792634	49792885	-	1	train	CGGCTCCATTTCTGGCACAAAACTTGCAGCACCGAGGGGTTGTGGAGAGCCCTTGCAGGGGAAGAGGGCAGGGTCATCCCGAGAACCAACGGGCACGTATAGCCCGGCGAACGCCCAAGCCGGTCACCGCCCCCGGTCACGTGTCGCCAGCCTCCGCGGCCGCGCGCCGCTCTCAGCACCGTTCCCGCCCCACCCGGCCCGGCAGTCGGCCCGCGCCTCCCCCGGCGCTACTGCCACCTCGCGCTCGGAGG
FP017634	chr19	52336092	52336343	+	1	train	CCGGCAGGATCCCAGTTTGCAGAGGGCTGAGCTGCTCCAGCCTCACTGCCAGCAAAACCCGGAAAACAGAGGCACTGGAGGCGTGGCCTAGGGAAGCCCCGCCCCGTCCCGTCCCGGTCCGCTCTCTATGCTGCGCGCGCGCAGTTTTTTGCAGACCCGGAAGCGGATCGCGTGGGTAGAAGGTCACACCGCAGCGCGTCAGTTTCCCTTTGTTTAGATTCAATCTGGGCTTCCCAGCTCCCCCGCGCTTC
FP011102	chr11	70420212	70420463	+	1	train	AGGTTATTTCCTGCCACTCTCCAAGGAGGGCCTCTTCATGGATGTTATCTAAAGTATTACCAAAGATCCGGAAGGGAAGAGTAGATTGAGTGAATCATATGCGTTTAACTGTATTGAAAACATACTTTCCACCTGTGACCTGCACTACCTGTTAATGATGGGTTCTGGCCTTTCATTGTGCATGTAGACTATGTGAAAGGGTTTGGAGGAAAATTTGGTGTGCAGACAGACAGACAAGACAAATGTGCCCT
FP000526	chr1	37556739	37556990	+	1	train	TCGAACTGCCGTTTTCCTCGCCTCCAAGAGTACAGGGACGAACCGGTGAGCAGAAGGCCTAAGAAGTCAGGCACAAGAGGTTTCTGCCCAGGAGGCACAGAATGAAACTTGCCGGCTGTTGGCCCCGCCCCACTTCGAAGCTCCGCCCAGGCCAAGAAGCGCGGGGGCGGGAGTGAGGGGGCGGTTTCCATGGTGACGGCAAACAAGGCCCACACTGGACAGGGCAGCTGCTGGGTTGCTACTCTCGCCTC
FP007988	chr7	150801157	150801408	-	1	train	GCCGCACCCCTGGCCCCTCACGAGGGGAATGTGTGTAGCAGAGGGGGTGCTGACCATGCTGGAACTGCGGCGACTACAGAGCCTGCGGGAACCTCCCCTTTCGCCCAAGATCTGCTCTGTCCCCCTCATCCTCCTCCCAGGGCCCTGGCGTCTGGGTCAAGCAGCGCCCCACACCTCGACCCCTCACCCCCTCCTCCCGGGCTCTTCCTGCGGCCTCCCCTCCACAGTCCGCAGGCTCTGGGACAGGACCG
FP017580	chr19	50384142	50384393	+	1	train	GCTTCGGGCAGACAGGTCATGGGGAGGCGGAGTTAAGGGAATTTTTCAGCAAGGGGGCGAGGCCACTTCGGAAGCTGAGAGAGGGGGCGGGGCCTGCCCTGCAGTCGAACAAGCGGGGCGTGGCCTTGCCCGCACTTGGGCAGGCGGGGGCGTGGCCCGTCTCTGAGCGCCGCGGCTCTGGGCTTGCGCGCGCGGGAGTCAGGGGTCACGGCGGCGTAGGCTGTGGCGGGAAACGCTGTTTGAAGCGGGTG
FP002543	chr2	99180981	99181232	-	1	train	TCAGCCTGAAATTTTCCTCCGAAGGGAAGCAGAGCAGAGGAAGAACTACCAAGTGCTACACTCAAAGCCTGCCGTCGCAGTGAGCGCGACCTCCAAACTGAGGCATTTTTGTTCCGGCGAAATCCCTCCCACTCAGGAAAGTCCCTAGAAAGAGAGCGCAGGCGCCTGGGTATCACATGACCACTTCCCGGAAGCGCAGCAGACCCGCTCAACTTCATCCTGGGTTGAGGCGGAGGAGAACTTCCAGAATT
FP008455	chr8	91070143	91070394	+	1	train	CAGTCTGGTTCCTCCCCTGCCGCCCCGCCCATCACGGCAGGGTCACGGTAGCGCGCACGCGCAGCACCCCATTTAAGTTTCTCGTCTTTGCAGTGGCTTTGCTTAGATCCGGTGCCGCCTTGAAGGCGGGGCTGGGTCCCAGCCGTAGCCAATGGAGCCCCGGGTGAGGGTTGAGGGGTGGAAGGTGCCTACTAGCCGGTGCAGGTTTCTTCTAGCGCGTGTGCTGGGGTACCTGGTCGTCATGGAGGCGG
FP019312	chrX	54014323	54014574	-	1	train	TTCCTTGGTAGACTTTCTAACCTTGTGGAGACACCGAAGATTGTTCGAAAGCTGTCATGGGTCGAAAACTTGTGGCCAGAGGAATGTGTCTTTGAGAGACCCAATGTACAGAAGTACTGCCTCATGAGTGTGCGAGATAGCTATACAGACTTTCACATTGACTTTGGTGGCACCTCTGTCTGGTACCATGTACTCAAGGTAGGAATGCGCATGGCTTCTGGAGGCAGCCAGGCCTTGGGCCTCCCTGTACG
FP018514	chr21	44339351	44339602	-	1	train	TAGGCCTGCGCTCAGAGTCAGGCCTAGGCACTCGGCCATGATGCGCGTCGGACCGAGCCCACCCCGCTCGGTTCCTGGGAGCAGCAGTTCAGCCCTCACTGGGCCGACTCGGGGACCGCCACCTGCACCCACCCTCTGCTCGCTGCGCGCTTCCGGGGTCGCGTTAGAATCGCGTCAGAGAAGGGCGCCCCAGGCCGCGCATGAGCAGGAGCAGAGCCAACGTGTCAGCGGCGCCCAAGCGGCCCCAGCGG
FP016245	chr18	12658352	12658603	+	1	train	GCGACGCACGGTCCGGAGGGGATGCGCGGCCGGGTAGGGCGTGCAGGGTTGTGGGGCGTGGGGGACACGCGGGGCCGGGCGCGCGGTCGGGGGGCGCAGGGGCAAGAGGGTGTCGCATTTATCAGGGAAGTCCCCGGCCGACTCTGGCGGGGAACTGGGCGCGCCAACCCGGGTCACCGCCCTGCACAGCGGGAGAAGTCACCCGCATTGACGTTCCCTAGGCGCTGGATCGCCGAAGGTCTGGTTGAAAA
FP015116	chr17	3557761	3558012	-	1	train	AGAGGGAGGAGTTGGACCCACAACTTCGATCTTGGCACCTGCGGCCTCCCCTGCCTTTTCTCCGGTGGGGATGAGGCTGCTGTGTGGGTGTGGGGGTGACTCACACGGGCCACGTGGGTGGGCGGGTGCTGCAGCTGTGGCTGGTGGGCGTGGCCTGCCTGGCGCCGAGGGCAGGTGGCTCAGCCAGTTCTGCCTCTGACGCCTCATTCCAGCCATCCCTCTGCCTGCAATGAGAGCTTCCCGCCGCCTCA
FP013257	chr14	60981044	60981295	+	1	train	GATGTGGGTCGCGGGTGGATGGGCGGGTCTTCTATGACATCATCACTGTTCGCCGCGAAGAGGGCGCGCGTCATCAGATCAAGTCGACTCGCTCCTCTCCTTCCAGGCCCTGGTGAAGTACGGAATGCCGGAAGGGCCGGGCTCAAAGCTCCGCCTCTGGCGCGACCGACGACTGGAGCGCAGGGCAGGGGTAGAGGCTCGTAGATGGAACTGGTAGTCAGCTGGAGAGCAGCATGGAGGCGTCCTGGGGG
FP009295	chr9	124503327	124503578	-	1	train	GACCCACCCCGGGACGCTGCGCGGGGCGCTCGGTGGGTGCCGGCGGCAGCGCCTGGGCACAGAGAGGGGATTACGCGACGGGCCGGCGACAGCGGCTGGGGCGGCGGCCGCGGGGACCCCAGGCTGCCGGTCTCCGCCGGCCCTCCCTGACCCGCTGTCCCTCCGCAGGCGGACGCCGCGGGCATGGACTATTCGTACGACGAGGACCTGGACGAGCTGTGCCCCGTGTGCGGGGACAAGGTGTCCGGCTA
FP013547	chr14	103099946	103100197	+	1	train	GCGGAGACCCTTTTTGGGGGATGGTCTCAGGAAAGGCTCCTGATGGAAGCAGCTGTCTATATCTCCGTGCCCAGCTGCGGCCTGGCACACTGAGGGCTCTCGCCAAGAGAGTGGCCTGGTGATGCTTCCTTTTGACAGCCTGAGGGCTGCCTCCTCTGGAGAGCCTTCCTTGACCAAGCCCCTGGGTGTGGCCAGGCCCCACAGGGCCACAACAGGTTTCTGCATCTGCTCCTATGGCCACTCCTTCTTGC
FP004858	chr4	87529347	87529598	-	1	train	CAACAGCATGGAGTTTCTTTTCAAGGTCTTAAAATAAAAATCTCCAAAAAAGGGATGCTGCTTATAAAACCTTTCAGAGTTGTTTGAAAAAGAAAAAAAATGCATAAAGAGCCAAGTGCTTATATTCTGGCCAAGTTATGAGGCTCTGAGAACAAGAGCTTGAGGGGAAGACTGTTAACCCCATCCACGCCACCAGAATTAGCTCTTTCCCTTTTGGTTTGCAAGCACTGCCTGTAAAGCCCTCGCATGAG
FP014016	chr15	75023408	75023659	+	1	train	GAGTCCCAGCAATACAAAGAACAGGGCCAGGGTTTGCTCCAGGATTTCCCCAGTGGACTACAACTCCCGGAAGGCAGAGAAGTGGACTACAGCTCCCAGAAGGCTGCGCGACATACAGCCAGCCCGGTGCTCTCGTGAGGCGTGCCAGTCAGATTCCGCGGGAGAGCGGCAGAGATACCGCGATATTTGGGAGCGGCCCCGAGACGCGCCTGGCGCGGGTGAGCAGTGGAAGGGGGCTGGGAAGCTGGGTT
FP015187	chr17	6640631	6640882	-	1	train	CGCTAGCTGCACTCCCAAGCCCAGGCTTAGATCTACCCTAATCTAGTCCCAGCCCCGACCTCATGGACACTGGCTGCAAAACTTACCGCCTTGCAACACCTGGATCCTTAAATAGTTCAGGGCCTCAAAGGCTCCCAAGCCGCCCTGCGCCTGTTGCTATGGCGACTGGGCAGGCGGCCGGAAGCGCGGCCTTGTTGGGGGCCCCCGTTGCCCGCCTCCCAGTAGCCGCGCCTCACCCTAGGCCAGGTGGG
FP003014	chr2	201129639	201129890	+	1	train	GACCACCCAGAAGGAAAGAGCCCATACTTTCAATCTTAGGCATAAGTTAGCTTGATAAGATTTTCAGAAAAATTCCCTTTTAACCACAGAACTCCCCCACTGGAAAGGATTCTGAAAGAAATGAAGTCAGCCCTCAGAAATGAAGTTGACTGCCTGCTGGCTTTCTGTTGACTGGCCCGGAGCTGTACTGCAAGACCCTTGTGAGCTTCCCTAGTCTAAGAGTAGGATGTCTGCTGAAGTCATCCATCAGG
FP001518	chr1	170074614	170074865	-	1	train	TCAAGGGGGCGGTGACTACGGAGCCACTCCTTATACACGGACGCTCAAAACTGCACCAAAACTTCCCTGCTCGGCTCCCCGGTGCGTCAAACTCTGAGGCCACCGCCTCAGACTGAGCCTGCGCATACAAGAGGTGCAGAGCAAGCGCATGCGTCGTGACGGCCCGGCTTAGGCGACTCTGGGCGGGTCTGGGCCGCTCCAGTGTTTTGGGGCACAGAAGCTGTGGGAGGAGCTGGAGGCTTCACCGTGGT
FP003696	chr3	50617453	50617704	+	1	train	TCCAAGGCTGGGGTGTGTTGGAAAAGTCTGGGCGGGACTCACTCTTCCCCTTTCCCCCAGGTGCCACTAGAAGCGCCAGGCTGGGGCCGCCTCTGAGCGCCCCGCGGGGGCCATGGATGGTGAAACAGCAGAGGAGCAGGGGGGCCCTGTGCCCCCGCCAGTTGCACCCGGCGGACCCGGCTTGGGCGGTGCTCCGGGGGGGCGGCGGGAGCCCAAGAAGTACGCAGTGACCGACGACTACCAGTTGTCCA
FP002407	chr2	73828798	73829049	+	1	train	CCTTTACCAAGATGGCCGCTGTGTCGTTCGCTCCTCATTGGTCGGATTTTCCGCAGGTGCCGATTCTAGCCCCACCTACTAGCTCTCCGTTCCAAGTCCTAAATCACTGGGAATGTAATAGACAGACGAACGGGAGTGGGCGGAGCGGGCGCCGGATGTGACGTTTCCGGAACCTCCGGGTGTCATCCGCGGGGAAAGGTGGGGAAGGGTCCCGGGAACGTGGTGGGGCAGGGCCTCCGAGCGTGGTTGGA
FP004144	chr3	138947086	138947337	-	1	train	CCGGCAGATTTCAAGGGCGCGTGAGCCTGGCTGTCGGCTGGGCCCCTGAGGCTCGCTGGGCGGGGGCAGGCCGGTCCAGGCTGTGCGGGGCGTTTACAAAAAGTGACTTGGAGATGAACTCGCCCGTGCGCGGCTGGCCGCCCCGCTATAGGGGCGAAGGCGCCTGACGCAAGCGGAACTCGGTGGAGCCCATACGAATCAGAACAGAGCGAGGCTCCTGGCGCACTAGGGACTCCAGGAGGCAGCTCCGC
FP016366	chr18	49813831	49814082	-	1	train	CCCTGCCCCCAGTACAGTACCCAAAACCGGTCCGGGTCCAGCTCGGCGGCCCAAGCCTCTGCCCAGCGGACTGCGCACCCGCCCGCCCCAGCCAATCCCGCCGCCGCCGCCGCGCCCCGCCCACAGGACGTTCGGTCCCGCCCCCAGCTGGCGGCCGCGGCCGCCCGCGCGCCAAGTTCCTCAGCCCTTGGCTCCTGCCCAGTGTTTAGGGTGTTGGCGGAGACAAAGGGGAAGAGTCATCGCCTGTCGGG
FP012507	chr12	121399928	121400179	+	1	train	CTCCCAGCCAGGAGGAACGCCAGAGGCCCCGCCCAATCCCTGTCGCTTCATTGGCTCTGGTGACCGGCCTCCCGAAAGTCGGCTTCCTATTGGTCGTCGTGGCCGCCGCTCTGCGCTTTTACCGGGGCACGCGGCGAGCGTGACCACGCCCCTTACGTCCGGACGCTCGGTCGCCTCCCGGGGCGCGGTAATCACCGCCCAGAGGGAAGGAGGTCGGCAGTGTGAGGAGCTGCTATGGTGCTGAGTTTCCT
FP010382	chr11	392397	392648	+	1	train	AGGTGGGCCTGGGGTGGACCTGGGTGCAGGGCGGGACCCAGCCCATGCCAGGCACATACCTCCAGGCCCGCTCTGCGCCGCACAAACTTGTCTCCCAGGATTTTTCCACACGAAACCTGTCTCCCAGGGAATCAGCCAGTCCGATGAGCTGGGCGTTCCCACCCCGCCCTCCGAGGCCCCCACAGTCCTGTCCCCGCCGCAGAGGCTGCCCCGCACGCGCCGGGCCAGGGATGGAGTCCTGGACACCTCGG
FP012150	chr12	62466625	62466876	+	1	train	AACCCTGTCTAGCCTACTTGATTCCGGGGGTGGGGCTGTGGGAATCGCTGCCCACCCGGAATCCCGCCTGCCCCGAGCATGGCCAAGCCCCAGCCCCGCCCCGCCGACGGCCTGCCGCCGGGGTTGGCCGCGGCGGTGGCGGAGGCCACGGTACCCGGAAGTGCGGCTGACGTGTCCCGGCAGCGCTCGGAATTGTGGGCGACTCGGCTAATGGCGTCGGCGAGTCTTAGGGGCCTGGGGAGCTGGCGCTG
FP001015	chr1	109687643	109687894	+	1	train	GGGCGCCCTGACTTCGCTCCCGGAACCCTCGGGCCTGGGAGGCGGGAGGAAGTCTTACTGAGTGCAGCCCCAGGCGCCCTCTCCCGGGCCTCCAGAATGGCGCCTTTCGGGTTGTGGCGGGCCGAGGGGCGGGGTCGCAGCAAGGCCCCGCCTGTCCCCTCTCCGGAGCTCTTATACTCTGAGCCCTGCTCGGTTTAGGCCTGTCTGCGGAATCCGCACCAACCAGCACCATGCCCATGATACTGGGGTAC
FP004755	chr4	70704588	70704839	+	1	train	GCCCAGCCGGAGGGAAGCGGAAGGTGGGGAGTGGGAAGAGGAGGACGCTGCCCCCTAGGGGAGCCGAGTCACAGGCTGGGGGCCGGGGCGGGAGGGGCTGGAGGAGGCGGCCGGTAGCGAGGCCGCGCCGTCGGGCGGGGCGGTCCCTCCAGCTGGCTTGGACCCTGGCGGGGCGGGCTGTGGGCCGAGCGGCGGAGCCCAGGCGCCAGCCCGTGGCCGAGAAGAAAGGTGGCGGTGGCGGCGGCGGCGCA
FP004799	chr4	77756747	77756998	-	1	train	CTTGCAGTTATAAGTTTTAAAGGTTTTATTTTTGTCTTCTGCTTTAGGCAATCCTTTATCACAGGATATTCTCAACTTATACCAGGACCCAGATGGAACCCGAAAGCTACTGAACTTCATGCTTGACAATCTCGCAGGTAACTGTTAGTGACAAAGCTGTAAAATAAATTCTACAGATATTATGAACTAGCTAATATATGTTCCTCAAGAGCATCTGACTGTTTTTTATAATTGCCTATGGGTTTGGAGTT
FP009268	chr9	121768242	121768493	+	1	train	TCGCTGGACTTACCATGAGCAAGGAGGGCCCTGGTGACCTGGGTCAGAGCATATTCAGCTGACTTGGGGAGTGAATGGAGTGGGAGCCTGCCATAGTGGGGAAAGCGGGTGGTGGTGTCCCAGTGGAGGGAGTTCCTGGGCAGGGCCTGCCTGGGCCCAGTAGTGCTCACCCAACTGCCCTCTCTCCAGATTTGGCAGCAAGGAGGAATACATGTCCTTCATGAACCAGTTCCTAGAGCATGAGTGGACCA
FP003984	chr3	121660865	121661116	-	1	train	CTCAGAGTCAAAGCAGCATTTTCTATGACTGCTCTATGACCCCCATTGTCAAAACCTTGCAAAAGACCAGGCACAGAAACATTTAATCCTTTTAAAATTTAAGTGTCTAAAGCAGAAGGAACTGACTCAAATTTCAGGAAGTAGCTGTGAGGATGGGGTGGCAGGAAATGACGAAAACCTGAATACGGAACTGAAGCAGCAGCTCAGTTTCTCACTCCGAAGTGGCAGCAGCCAGAGAGGGAGTCGGTGTG
FP009408	chr9	129835248	129835499	+	1	train	CCGCGCCTCTGCGCAGCGGCCCAGGCTGCTTCCGGCGCGCGGCAGAGCGGTCAGAGCGCCTGCCCCGTCTGGCCCCGCCCCCGGCACGCACAGTCCGCGTGGCCACGCCCCCTCTCCTGGCGTCAGCCTCTTCCGATTGGACGGCAGAGGGAAGGAGGTGGGGCGTCGCCAGACGGCCCCACAACCCTGCGCGTCGCCTCAGAGGGGGCGCGCTTGACTGACAGGCGGCGGCGGCGCAGTTGCGAGTGCAG
FP015257	chr17	8038858	8039109	+	1	train	AACTCTAGGCTCTCTGGCCTGGCACCACCACTGAAAGGCCCCGTTCCCCAAACCTCAGGGTATTCTTAGTCCCTCCTGTTCCCAGCGCTGCCAATCCCCCGCCCACCCCCACTTTAGTTGCGTGTTCCAGCCTCTCCGCCCCGCCCCTCCCCGCCCTGAAACGGACGTGCTTTTTAGAGGAGTCCACTGGGCTTGGAGTCAGTGGCAATAACCAGGGGCAATAACCAGGCGTGTCCCAGGGGGGAGCCCCG
FP014629	chr16	30762009	30762260	-	1	train	AGGCGCCAAAGGCAGCCCCGGATGTGAGGAAACTTCTTTATCTTCTGCTTCTATTGGTTACGCTTGGTCCCGCCACTCGGGTCCTCTGGCTTCTTCCCAAACACCTCCTCTTTCTCTCGGTGATTGGCCAAGAGGCAGTTCCCATAGTAACTGACACCGCGCTCCCACTTCCCAGCCTGGAGGACAGCACCGGCCCTCGTATTAGCAACCTGGAAGAGAGGGCGCCCAGGTGGGCACTCGCAACTTCTCAC
FP001805	chr1	220046454	220046705	-	1	train	GACGAAGTGTCGTAAAGTCCCGTCCACTTCAGCGTCTTTCCCACCGCAGCACGAAAGGCCACGCAGGCCGTTCTAACCCTGGCGGCCAAGAGGAGGGGTAGCAGGCTCTGGAACCTGCGGGCGGGGACGAAGACGGCGCGCGACGATGTCGCACCGCGGCGTGCGCGGTGTCGGCAGTAGCTGCGGCGCAGGGGCGGAGCGAAGGCTGCGGCGGCGTCGGGTACGCGCACACGTTGCATCTTCTTCCTTTC
FP010202	chr10	109996172	109996423	+	1	train	CGCTATTAAAAATAAAAAATATTAACTAGACTGGTTTTTACAACTTGATCTCTATTTATACTGTATATAATCTGCAAGTGATTAGATTTCATTTTTATTACTCTTGTTAAACTACAACATGCTTCGTTATCTCTCCCTGGGTGTGGCTTACACCCAGAACTCCTGCTGGCTTTGTTCAGTATGCTTTCTGTTGTTACTTTAGTTTCAGTAATCATTGTGAATGAGACTGTGAGAAACTTGGCTTGTTCAGA
FP001095	chr1	116570975	116571226	-	1	train	AGCTGGGGGACTGGACGGCGCCCCGGGGACACAGGAAGGAGGCGCGCAGAGCCGAGGCCAGAGGCGCGCCCGGGGAGTGGAAGCGCGAAGACAAACGCGGCGCCGCGGAGGGTGGGGGAGGAAGGGCCGGGGCGGGCCGCCGGCTGCCAGCCCAGGGCGGGGCGGAGCCCTACTTCTGGCCGACCGCGTAGGCGGTGCTTGAACTTAGGGCTGCTTGTGGCTGGGCACTCGCGCAGAGGCCGGCCCGACGA
FP016019	chr17	75878530	75878781	-	1	train	CACGCGGACTCCCCTGGGGGTCGCGCGGGCTCGGAGGCCTCCCTGCGTTGTCTGGGCGGGGACTGGGAGTCCGCGGGCCCGCAGGCGGGCGGTGCTGCGGGGCGGGCCGAGCTGGGGGCGGGCGGGGGCCGGAGCCCTGGCCTAGGTCCCAGGGGGAGGTGCCGCGGCGCGAACGGCCCGGGTCCCTCCCAGGCCGGAGCACAATCGGCGGCGCCCTGGGCGGCCGCGGAGTCATGGACGGCAGTGGACCC
FP012646	chr13	24270584	24270835	+	1	train	TGTATATAGGTTTTGCCTTAGTTTAGAATTCTCTCCTGAAGTCCCCCGCAATGTCACTGTAGCCACTAACCCTTGGTCACACGGAGTGTGGAGCCAGCTGTCAGTTTCTGGGCAGGCTCTGGCCGGGAACGCATACAACGTCAAAGCAGTGAATAAGCGATTTCTGCAGTTCCCGTAGCTGCCAGCCTGTGCAGTCCTCTGTGATGCTTGGGGACCGGCTCCTCGGTCACACCCCAGTCCTGCTCTGAAGG
FP012626	chr13	20525806	20526057	-	1	train	AGCGCGTGGTCTTCCCGCTTTCCACCCCCCGACGTGCCCCCGCGCTGGTCCTGCCCCTGCCCACGAGGGTCGGCGCGCCGTGTGTCTCCCGGTGTCTCGGGCCCGCGGCGCGCACCCCACTCAGGGACCAAGCGGCCCTCCCCGCCCTCCCGGTCCGCGCGCGCAGCCAATGGGCGCGGGACCCGCCCTCCCCGGAGCCCAGAGCTCGCAGCTCCGCCGGCGCCTGGTCCCAGCGCCCGCGGCGCCGCGTC
FP019214	chrX	47190646	47190897	+	1	train	TGCACCAGGCACTGGCCAGACAGAACTACCACCCCCAGCATGCCGCGCGCATTTGGGCCGTACCACAACCTGTAGGGTCATCTGGGTCTGAAACGGGCACCAGCGACGCGGGAGCAGGGACTGACGCCCCTCTAGCAAAGCATCTGTGGTACATCCCAGACCCGGGGCTCTCCAAGGCCCCGCGCTTCCGAGCTCCGCGCAAACTCTGGCTTCTCTTGTACGACAGAGGTGGTTTGCTCTTCCGTTGCCCC
FP016325	chr18	36129289	36129540	-	1	train	GCTAGCGAGTAGGGCCTCGAGGAAATGACCCCCAACTGCTTACAATGCGCTAGGGCCGCGCCACGCCCCCGCGGCGTCGGCGAACACACGGAAAAGCGGATCCGTCTGCGGAGCCGGGTTCCGGGGTCGCACTGCGCATGCCCTGCAGTCCTGGGCGAAGGGGGCGGTGGTTCCCCGCGGCGCTGCGCGCGGCGGTAATTAGTGATTGTCTTCCAGCTTCGCGAAGGCTAGGGGCGCGGCTGCCGGGTGGC
FP004255	chr3	161221110	161221361	+	1	train	GAAAATCGAGACGATAACTGAAAAGCAGTCACTGACGGCAAATGAGCCCCTGGGACTTCTCTCCGCGAGCTTTCCCAGCCACCAGGCCGCTCCCATCACCCTTCCGCCCGTCGGTGCCGCCCCTTTCCTCGTCCCCACCCCGTCCCTTTTAATACGTCACTCGGGCCGCGGGATTTCGAGCATTTCGCTCGCGAGATCTTCTCTGTGGCGGAGACAGCCAGGTTGGCAGCTGACGGGACAGCCGGGGTCTA
FP016376	chr18	50560502	50560753	+	1	train	GTTTGGGATGGGAGGTTGCCGCTGGGCTCCTCGCGTTGTGTTTAGGGGAGGAGGACGCAGGGGCCGGGCGCCGCTAGGGGACCCCACCCCCGGGGACAGTCCGGAGCGCTTGGGGTCGCCGAGGGGCAGTTCACACTGCGAGTTCAGATTCGGATCGCAGTCCCGATTATCCTCCCCTCCAGCCTCTCCCTTTCTCGTTGAAGGGTTAATACAGCGTCCTCTCCCCTCGCCACCCGACAGAGGCGCCTACA
FP011395	chr11	118344148	118344399	+	1	train	AGTCTCTGAGTGGGAATCCAGCACTCTCTCCCTCTTCTTCCCCACCACCTTCACCCTCCTTAACGGAAAAACAAAAGGCATCTGCACCTGCAGCCCTGCTGAGGCCCCTGCTGCTCACACTTGCAGCAGAGGGTGGAGGCTCTGGGTTCTTGCCTTCTCTCAAAGGCCCCAGCCCCAACAGTGATGGGTGGAGCCAGTCTAGCTGCTGCACAGGCTGGCTGGCTGGCTGGCTGCTAAGGGCTGCTCCACGC
FP001784	chr1	212791693	212791944	+	1	train	TCTCTGTGCCAGCCGCGAGCTCCTTGTCCCATGGAGGGTCAAGGACCACCAACTCAGGAGACCCCGCCATTTTTCGTCGGAACTGTGGGCGGGGCACTCTGGGAGCGGAAAAGCGGGTTCACACTGGTGTAACCAGCATGCATCCGGCTGGAAACTCAGGACGCTGCAGCTGAAACGTTCCGGCTCCGGCTCTGCTGGCCGGTCTAAAGCGGCAGCCGCCGGGGCGCAATGCGAGCGGCTGGCGTAGGCTT
FP019439	chrX	93673525	93673776	-	1	train	CAGCGAGCGGCGTCTGGGAAGGTGGGCGGAGCCAGGAGTAGTGGCGTTTGGCCGTTCGTTGGGCGTACAGTTTGTCAATTAAGGTGGACCAGCAAATGAGGAGCGAACTAAAGGCACACTGGGAACGAAATTAACGGGAGGTCTGACTGCAAGGGGAGGGGGCTCGCGATCTAAAACGAGAAGAGATCTCGGGGTCTCATACTGCGCCATTCGGCTGCGGTACATCTCGGCACTCTAGCTGCAGCCGGGAG
FP006955	chr6	136550870	136551121	-	1	train	CCTGAGCAAGCCAGAAGTCACAAGGTCGTTTACTAAACGGAAACTAAGAAAAAGAAAAGCCACATCAGCCGGCACAAGCCTCTCTAACTTGTGTGCCCTGAGGCTCATCAGATGCGAAGTCCTGGCGACAAGATAGGGTCGCCCGCAGTGGCTTCCTAAAGCGCAGATCCAGGCCGGCTCCCAGCGTGGAATCCGACCCATGTACTTCGCCTCTCCACGTGGCATCAGATGCTCTGCCGGCTGTCACCGTG
FP003931	chr3	113212103	113212354	+	1	train	GCAGGCCGAGGCAGCGCCAGCGAGTGGCTGCCGGGGGACGTGCCGGCGGCGGGAGCCGGGTCCCAGCGCTCGGCCGGGCGCCCGGGGCTGGGAGCTGCGGCCGAGGCTGGGCGCGCCGGGCGCCGGGCGAAGAAGTTGGGGCGAGGCGGGCAGCGGGGGTCCCGGGGGCTGCAGCCGCTACGGGGCCCAGGGGCGCCGCATCTCGTTCCCTGCCGCAGCGGTCTGGCCGGGCGATGGTGGTTTCGCCTCAG
FP016001	chr17	75456440	75456691	+	1	train	CTCTTCACTCTCCCGGCCGCATCACTCGGCACCTAATTCCGCCAGGCTCCCAGAGGGGCGGTTGCAGCTACAGCCTAGCACGCCGCGTAGCCAATAAGAAGCCAGAATTCCCCCTATTCTCAAATCCTGACCAATCGGTGGCCTGAACTTTAGCTCTCTGGCCGGCGCCTCTGTTGCCCGGAAGGGAGGCAAGCCCCCTTAGAGTTGCCAGTAACGGACATGGCTGCGGCCCCCGGAGGAGGGGACGTGAA
FP010652	chr11	30016925	30017176	-	1	train	TCCCAGATCCTGGGAGTCGGGGGGCCCAGCCGGGGTGCGGGGAGTCGCGGCGTCGTGACTGCACCCGCCCCGCCGCGCCGGGAGGCGCCCCTGGCGACCGTCGCGAACGCCACACCCTCGGTTCGGCCCTGGCCCCGCCCCCGCCCAGAGCGCGTTCGCGGCCAGGGGAGGCGTGTGCTGCTCCCGCAGCCAGCGAGAGCGAGAGAGGGATGGATGTTGGAGGAGGAATTCGGGCTTAACAAGTGATCGCT
FP006900	chr6	125956589	125956840	+	1	train	ACTATGGGCTTTGAAGAACGCCTGTCCAGAACACCCAACAAGCTCCATTATGTCAAACGCTCCCGCGCCTCTGTGATGTCACCACGACGCGCTGGGCGCTCTGCGACTTGGCAGACGTCGTGCGTCATTACCAACGAGGCGCAGGGGTCAGGACGACTCTCGGCAGCGCCATTGCGCGCCCTCTAGTGGCAGCCGGTTTTGAGGCCGGCCTCCGGCTTTGAAGTTCCTCACCGCGTCTCCTTCCCTCTCCC
FP003410	chr3	15427475	15427726	-	1	train	CCGAGCTTCTGGCGCTGGCGTCTGGGAGGACTGTGCTTCCGGGCCACCGCTCTGGAAGACAAGGAGAGCCGAGCCGGCACCCACCCGGGTCCAGAAGCAAGTTCTCTCCTCTGCGTGGGGGTGAGAGAGGAATCTGAGGAGGCAAGGCCGTGGGACTTCCGGCGTCGGGTAGGAACTTCCGGGTTCCTAGCGGAAGTGGCGGGGGCGGTGCCATGGCAAAAGAACTCCCGGATAATCCGAATTCGTGGTCC
FP019500	chrX	103832169	103832420	-	1	train	ACCAGGGTTGGACGATGAGCATGAAAGAAGAGTGGGCGAGCCAGGGAGCGCTTGCCACATGGAGGGCGGGAGGCGGCTACACCGGGTCCCCACCGCCGAGAAAGGGGCACGAGGAGGCGGCGCGGGCCCCGCCCCCTCCCCTGGCGGGCGCGGGCGCGGGCTCGAGACGCTGCGCGGCGGCAGCGGCGGGCGCGAGCGGCAGCTGTCAGGCCACCGAGGTCCAAGCCGCACTTGCTGCCCCATTGAGGACG
FP014772	chr16	58629775	58630026	-	1	train	CTTCCTCGACTTCTGCAAGTGATAGAGTGTGATTGGTTTAAATCCGCGGGCACCTGTCGGTGATTGGGTAGAAGCCACAGGCGGGGCGCGGCCATTGGCTGGCTCTGGGGCCATCCCTCCTCGGGCGGGCAGCGAATGATGAAATGGGAACGGGGAGGGGAAAGGCCGCGACGTGAGCGCGCGACCTCCGGCGCCATTTTGTAGAGAAACAAGCGGAGTTAACCGAAGAGGGGGTCGAGGAGAGCCGGAGT
FP018393	chr21	32771949	32772200	+	1	train	GAGAGAGAGAGAGAGAGAGAGAGCGAGAGAGAGAGAGAGAGCCCCACACCTCTCGCGAGAGCGTCGGCTCTTTGCATCGCTCTCTGTCGGAGTGGAGTACTATTAGCATGGAACTCTGGTGTTGTTTGACAGGTCACAAAGAGGATCATAGAGATGAGAAGAAAGAGTAGGCTCTGTTGTCCCTCAGACGGCGCTGCTGCAGTGCGTCTTCGCACTTACGCGGAGCGGTAATGTGAGGAAGCCTCCCGCCA
FP008422	chr8	79766604	79766855	-	1	train	TTTTGCTTTAGTCCATGCTTTACGATTTACAAAGCATGTATCTTTGTATATATGTTTGCATATTTTAAATTGTTCCCCAAAGGAACTTGTGCCAAATCAATCTTTTGCCATTCTCATCTTTAGGGATCTGCTAAGCTAGAAAAAGCCGAGATCCTGCAGATGACCGTGGATCACCTGAAAATGCTGCATACGGCAGGAGGGAAAGGTACATCTCCTAGACCTAGTGGGGGCTCTGAATGGTTGGCAAAGAG
FP018039	chr20	33993661	33993912	+	1	train	GCCAGGCTGAGAGGGTGGGGAGAGCCCATGCGCCTGGGAAGCCGGGCTGTTTCTGGGCGGGGCTGCAGACTCCTCGTGCTCGTGTCACCGTGTCCTTCCCAGGAGACAGAGAAGGAGGAGAGCCCGGAACCAGGGCCGGAGCGGCGGGCGGAGCCGAGCTGCGGGGAGCCGCTCCGGGTGCCCCCACCCCCGCGCGCCTCAGTGGTGCCGGCCGAGGGCAGGGCTCGCGGTTGCGGGGCTCGCGCCGCTGT
FP002803	chr2	162343931	162344182	+	1	train	GCCAGTAAGAGTCGAGCAGGGACTGGAGGGCTCCCGATGGGGTGTAGCCAATCAGGGCAGAGATGGGTGGGGCTAGGCTGCGGGAAGGGGCGGACTGCAGCTGAGTTGGCTCCAAAGTGTGGGACTCAGCCAATCGGAACCGTGCAGGGCGGGGCTGGCCTGCGGAAGGGGGCGGGTTCGGAGGAGTGAACTGTGCGGTTAGTGCGCCTTTCAGCCTCACCTGCAGCTGCGCCTCCTTGCACCTGCGCCTG
FP011009	chr11	66230359	66230610	+	1	train	TCAGTCACCCGTGAACAGTGGCGATTTTCCTTCGGCTGAGGCAGGAGCTTTAGGACCACTTGCTTTCAGGAAAAGAAAATGAGTAACTCTCTCTTCAGGATGGTGATGGGGCCTGGCCAGTCCAAGGAGGCTCCTGGGTGGTCACTGAGTGGGAGGTCATCTTCACTGTTTCCTCTCCTCCATGGTAGGAAATCACTGACCAGGACATGTTTGGAGATGCCAGCACGAGTCTGGTTGTGCCGGAGAAAGTC
FP017444	chr19	46471512	46471763	-	1	train	GCCTCCGTGGAGGCGCCACCGGAGTGTATCCAGACCCGCTTGCTCTGCGACCTGAACAGGCCACTCTTGCACCCGGCTCCCTAGGTCCAATTGCAGCGCCGGCGCGCCGGCCGGGCTCAGCCCTAGCAACCCACGCCCGGTTGCCATGGAGACGTCGGCACCTGCGCAGCCCAAGACCCCCGAGCGGCGGTGCGGGAGCCAGTGCGCAGGCGCGCGCTGCGACGCCAACCTCGGCTTCTCGCCGCTAACGG
FP001209	chr1	151036412	151036663	+	1	train	GTCTGGAAAGGAGAACAAATTGCCTTCCTAGACTCAACCGAGACTGGCCCCCACTAGAAAATGCACTCTCCTCCCCAAAACCCGCCCCCTCACAGGAGGCAGGGTTTTTCTTTTGGGCAAACCTGCTCCTGGGGAGAGGAGTGGCAAAACCTGTTTTGAAGATGAGGTTACCTCCCTTCCACACCTCCCTCCTTGGAGGCAAGAGCTACAACAGCTGAGACAGAAAAGAGGTAAGGAAGTGTTGGGGGCTG
FP011968	chr12	53173292	53173543	-	1	validation	GGCTCCTGCTGGGTAGGTCCGGGAAGGGTTGACAGGCTGAGTAGGGAATGACTCTGCTCTGCCACAGATCCTGATGGCTGACTCAGAAGCACTCCCCTCCCTTGCTGGGGACCCAGTGGCTGTGGAAGCCTTGCTCCGGGCCGTGTTTGGGGTTGTTGTGGATGAGGCCATTCAGAAAGGAACCAGTGTCTCCCAGAAGGTGAGTCCTTTCTTCCTCCTCCTTCCCATCAAGCTTTCTTTCTTTTCTTTTC
FP002236	chr2	43776625	43776876	+	1	validation	GACTAATCCTCAGATGTTTTACCAGATTATTGTTCCTCAGAAAATAATGTTCATTTGTTTCATAAATTATACTGAAATAAAATTTCTCTGAGCTTGAAATAAATATTTGAAAGAATTCTTTTTCCCTCAATTTTGTTTTTTCTGCCTCTCATGTAGTGAAACTCTCTGGGAAATTGCAAAAGCTGAAGTGGAAAAAAGGGGAATTAATGGAAGTGAAGGTGATGGAGCTGAAATTGCAGAAAAATTTGTTT
FP009662	chr10	21497206	21497457	-	1	validation	TCCCAAGCCCGAACTGCGGGCTTTCCCGAGCCTGGGACGCAACTCCCCGTCTCCTGGAGAGGAACGCGGAATTTGGCAGGGCGCCCGCCTGGCCGCCTGGCATCCTGAGCTGGCCGCCAGCTCTCGGGGGGTGGGGACTGGGTGTGTCCTGGGGAACCCGCTGCCGAGAGAGCAGCTAGCGCAGCTCGCGACGAGCCATTGAGGACCAGGGTCGCGCCGAGAGGCCGGAGGGGGCTCCAGGCAGCCTGGAG
FP010175	chr10	103367775	103368026	+	1	validation	AGGCAAATGTAATAGGTGGTCACCAATGCCCACTTCCTGCCTTTCTTTCGCTCGGTTTTGAAAGTTGCTCTCTTTCCTATTCCGTTTCTCCTTCCTTTTCGCTCCTCCTACTGCCGTAAATGGCCTACTAGAGTTGTCGTGAAATGCTCCTCCCCGCCCTCCATGTGGCGAAAATGCAATCACGCTTGACGGCGCGAGGTGGCTCAGCCGCAAGATGGCGGCGCTGGCGGAGGAGCAGACGGAGGTGGCGG
FP015973	chr17	74748427	74748678	+	1	validation	CCACTTCCTGAGCGCTAGTCTTCGCCCGCCGCGGGGCGCCGCGCCGAGCGCAGGCCCCGCCCCGCGCGTTCCCAATGGCCGGCGCCGTTCACCCGGCCGGAGCGCCCAGGCCTGCAGCCCCCTATTGGCCCGCGGAGGTCCCCACCCTCAGCGCGGCCCCGCCCCCGGGGTAAGGAGCCGGGGCGGACTCTGGGACGCTCAGACGCCGCGCGGGGCGGGGATTGGTCTGTGGTCCTCTCTCGGCTCCTCGC
FP009956	chr10	77853367	77853618	-	1	validation	CCTGAGGCTGGTGATGCCCTGGGGTGGGGCGGGGTGTGAGGGGCTGGCTGGGGCTCACTGAGCATGTACCGGGTGTTTCAGCACACTCCACAGCCGGCTCCTGAGTGACCAGACTCGGCTGAAGGATGACGTGGACATGCTGAGGCGGGAGAATGGGCAGCTGCTGCGGGAGCGAAACCTGCTGCAGCAGTCATGGGAGGACATGAAGCGGCTCCACGAGGAGGACCAGAAGGAGATCGGTGACCTCCGTG
FP016480	chr18	80147729	80147980	+	1	validation	TGCTTTTGATGTTTCAACAGAATATTGTTTCTAAGATTATGTATCTTGCACGTTTGGCACTTGAGGGGGTGCTCATCTTTTGATTCCGCTATTGAGGTAGAGGCGGGACGGGGGAAGCAGCTTTCAGTAAGACCCGCCCGCTCTTGCCCTGTCAGTTTACCATTGCCATGGCAACACCCGGAAGTTGCCGCCCCCTTTCCATGGCAACACCCTGATGGCCGAAAGTTACTACCCTTATTCCAGAAATCTCT
FP018527	chr21	44939838	44940089	+	1	validation	CCACCCGCGTCCTCCGGTCTGTTTCTCCCAGGTGCGTCGGTGAAAGGCGGAGCGCCGCCACCCTACACCCGCGACCCTGCGCCCGCCCGAGCCGCTTCCGGGTACTTCCGGCGGCCGCGGGGCTCCCGGCAGCCGCGGCTCCTGTTTCCGCCGGGCGGCGAGAACGCAGGACCAGCTTCCGGGAGGCGCTCCGCACGTTTGCCGTGCTCCGCCGGGAAGATGGGGAAAGTGAGGGGGTTGCGCGCCCGAGT
FP016413	chr18	58864172	58864423	+	1	validation	GTTTTCTTCCCCCGAGGTGAAGGGATTTGGACTTCTCCCCCCAGGTGCTTGAAAGAAGAGAAGGTTGGAGGAGGAGGAGGAGACTGGGGGGCTGCTGGTGGTGGTGACAGGGCTGGGAGAGAGAGGAGAGGGACGGAAGGGAGGAGGGGGCTGGGTGGGGTGGGGGTGACAGCAGAGAGGAGCCTGGTCAACAAGCCAGCACCACTGTGCTAGGAGGCCAGGCCGGCTCTGCAGCCGACGCATCCCGTGTG
FP019587	chrX	120985730	120985981	-	1	validation	TCCAGCCTGGGCAACAGAGCAACACTATGAAAAGAAAAGAAAAAGCTCTGGATTCTGGATTCTCAGTTCTAGAGCTTAAGGGCCTTGATCATTCATTGGACATAGAACATGTCAATCTGAAGTTTTCAGCAAATGAGCTTAGGGCTCACTGAGAGCCCCTCGTTGACCCTTCCGGTCCCGCCCCCTTTCGCCTGCCAACCAGAATCTTTCCCAACTTGTCTAAGTCCTCTCAGGCCAGCCTTGGTGGGAGG
FP001379	chr1	156728355	156728606	-	1	validation	GCCGCCACTGCCGCTCCTCACACACTCGGAATCACTGGCCTCACTACCTGGGGTCGGGGCGGTGCGGGTGTGCGCGCGCCGACCTCTGGCGCGTTCTAACCCGCGCGCCTGCCTTCCCGGGCTTCGTCGGCGCGCACCGATTTCCGCATCCGGCTCCGGACGCTTCCGGTCTTAGCTGGACCCGTCTGGGAGGTAGGTTTGTGAGCGTGAGAGATCGATCTGTACCGCGGGGATCCGAAGTATGCTTATCC
FP006767	chr6	97282503	97282754	-	1	validation	GGATTTCGAGGTCCCTGCCACAGCCAGTTGGTTGTGCATTTTGTCTTAAGTTCTGACAGCCTTGATGGACGGAGGGCGAGGGGGAATTGGGAATTTGGCCTCTGGGTTTGTTCGTTTGTTTTTCCTTCCTGCTGGCGGACGTCTTGAGAACTTTCTGAAGTAATCCCCACTCTCTCGGCCCCACCCCCCCCCCCCCAACAGGAATCTCTTCACACCTTCTCTTTGGAGCCCTTAATGATACGACGAACCCC
FP009907	chr10	72217061	72217312	-	1	validation	TTAGGAGAGACGGGGTTTCACCGTGTTAGCCAGGATGGTCTCGATCTCCTGACCTCGTGATCCGCCCGCCTCGGCCTCCCAAAGTGCTAGGATTACAGACGTGAGCCACCGCGCCCGGCCTACACTTTCATTTTAAGGTTCAATGATCTTAAGAAACACCCTGCCACCTTCGAAAAGTCTTTCTACCCTCAATGCCGAGCACAGAACCTGGGGCAGTGTTTCCACTGTGCTTCCAGTCCCGAGCATTTCGG
FP011559	chr12	4273596	4273847	+	1	validation	GCTGCTGTTCTCCTTAATAACGAGAGGGGAAAAGGAGGGAGGGAGGGAGAGATTGAAAGGAGGAGGGGAGGACCGGGAGGGGAGGAAAGGGGAGGAGGAACCAGAGCGGGGAGCGCGGGGAGAGGGAGGAGAGCTAACTGCCCAGCCAGCTTGCGTCACCGCTTCAGAGCGGAGAAGAGCGAGCAGGGGAGAGCGAGACCAGTTTTAAGGGGAGGACCGGTGCGAGTGAGGCAGCCCCGAGGCTCTGCTCG
FP015463	chr17	31487836	31488087	+	1	validation	GGAGGCGGGGCCGGCCGCGCGGCCATTGGTCGGCGCCCTGGAGCCACCTGCCCCTACCTTGGGGGCGGGGTTACCTGGGCCCCGCCCCGCGGCTCGGGTTCCGGGGCCGCGTCCCTGTCCTCCGCCCCCGCCCCCGCCCCGCCCCGGCGCGAGGCCCCGCCCCCGAGTCCCGGGGCCCAGGCGCCTCGTGATGTCACCGCAGCTTGATACAAAGAGCCCTTCGGCCCACGCGGGTCTCCCGGCAACGGGCG
FP019173	chrX	33211498	33211749	-	1	validation	ATTCTGGTTTGGATATTAATCAGAACACAGTTGAGCATTGTTTGAATTCACAGAGCTTGCCATGCTGGAAGCACAACCTTATATGTAGTGACCATGGACAGTCCTATTATGGGAAACCAACTTGAGAGAGAAGGCGGGTCACTTGCTTGTGCGCAGGTCCTGGAATTTGAAATATCCGGGGGCCTCTACAGAATCCTGGCATCAGTTACTGTGTTGACTCACTCAGTGTTGGGATCACTCACTTTCCCCCT
FP015225	chr17	7435634	7435885	+	1	validation	TCCGGTGAACCAGTTCATGGGGGCGGGGACCTGCCTATGAGAAAAGGGCCAGGATAGAAGAGGTATTTATTTGTTCATTTTGAGGACTAAATTTAGAACCCATCATCTACAGACCCATTTGTAGGTTGTGTCTGGGGCACTCGCTCGGCCTCAGGAAGAGGGAAGAACGGGGACGACTTTGTGGCAGCAAAGCCACCTCCCTTCCGTGTTTTCCGCAGCCCAAAGCGATAGAACCGCATGGCTTCCGCAGT
FP004997	chr4	121696573	121696824	-	1	validation	GAGGAGAGCGTGGTCGCGGGGCACTGGATTCGCGCGGACGCTCGGCCGAGAGCTGTCCCGGTAGCTGCGAGAGGGCGGGTCGGCCCCGTGGCGGGGCCCTCCGGGCTGTCTGAGCGCCGCCGGGTCCCCGCGGACCTGCGCTTGGGGAGGGCACGAGTTGCAAATGGCGCGCTAAGCCCGAGGTTTCTTCTCTTTTGCAGTCCTGCTTCACCTTCCCCTGACCTGAGTAGTCGCCATGGCACAGGTAAGGC
FP005548	chr5	94618471	94618722	+	1	validation	CAGGAGAAAACCCAGTAAAATCACTTACCAGTCTTCCACCTCCGGTGCTGATCTAGCAAGAGCCTGACCCGCTGGCCAGACTCAAGCTCTGCACCAGGTATTACCACCCTGGCGACGGCTCCGGAGCGTCACTGACAACCACCCGGAAATGGTACTATCGCTTCCGGGGCTGCCTAGCGCGCGGCGGGAAATTGTTTCCCAGAGCCCGGATTCGTGAAGCAGTTGAGTGCTGCAGCGGCAGTCGTCGCCCC
FP010294	chr10	122163232	122163483	+	1	validation	CTATTTACCTGTTCCTCCAGCCATCCCCTTAGTCCTGGGCCGGGAGAGACCATTTCCTCCTTCCTGGAGGGATGAGCCAACAGGGCTGAGCCCCCTACTCTCTCGGTGGTCAGCCCGCCCGACCACCACGCCCCTCCTTTCTAGGGCCCCAGCTCCCGGGCCGCGGCTCGTTTGCATTTCAAAGCTACACAATGCGGGCGGTCCCGGCGAGGCGGGAGGGGCCGGGGCAGAGGGAGGGTCTTCGCCTCCCC
FP013815	chr15	51751423	51751674	-	1	validation	GTCAGTCCGTCAGCCCCGGTGGAGCGGGGCCGGGAGAACAGGGGCGGCGCCGTGGCCGGGGCGAGCGCGGCGATCAACAGGCGGCTCCCGGCCGGGCGCGGCGCCGGGGCGCGCGGCTGCAGCGGAGCAGGCTGGGAAGCGTAGTTCCCGGCTGAGCCTCGCCGCGGCCCCGCAGTCCCCCAGCCCCGCCCTTCCCCGACAGGTGTCCTCTCCCGGAATCGGGGTCTGAACCCCGCCACAGCTCTGCAGGG
FP019610	chrX	130165695	130165946	-	1	validation	TTGCGAGCTTGCGGCCAGCTAGTGCATGCGCAAATCCTCTCGCCGTGCGGACCAATCGCAATGCGGCAGCGAGTGCTACGCCTGCGCAGTAGGCCTCCGGTCGCCGTTCCCCTTCCCCGGCTCTAGCAGGCCGGCTTCTCTGTCCAATGCCCACCCGGAGCTGGGAGGAGGAGTCTGCGTAATGTGCGTGTGAAGAGACTGGGGGAGCTGGCCGGGGCTCACGGTGTTTGACCCGTCGGTCGTGCGTGAGA
FP010978	chr11	65595771	65596022	-	1	validation	CTCACCGCAGCCGCCTTCCTCCACCCATCCCCACCATCCCAGCTGCTGTGAAATGGGCAAATGTCTTGCCTCCAGGACCTGGGAAGGGTGATGCAGCTCACCTCCCTCCTGTTCACACCCCAGCCACCCCAGCCATCCCAGCCGGGTCCCAGCCACCCACACAGCACCTCCTCCTGGCCCCGCCCTGCCCAGCCCCTCCCAGGTGCCTGAGGTGGAGCCGCCAGTTCCCACAGCACCCCAGCGGCCTGCCA
FP003503	chr3	37987793	37988044	+	1	validation	GGTCCAGATACACACACACACACACACACTGGGTCAGGCGCAGGCCACAGAAACATGTCAGCAAACGTTTTATTAAGCAACTGCAGTGCGCCAGGCACCGTTCTAGACACAGGTGGACAACAGCAAGGCCCTGCCCACAGTGACTCACAGTCCAGCTGGGACACAGACACTAACCAAAGATCCCACTTAGCAAAGAGACCAGCTTCTTGGAGCACCACGATGAGGAAAGTGGTGATGTCTGAGGAGGGAGA
FP008746	chr9	4145133	4145384	-	1	validation	TAACACACACAATTGAACCACCTTTGGGTGTAGTTTTTCATTTACCAGCGTGTTCTCCAGAGGTACCCTGTATTGATGTATTTTCCCAGTAGTGGGACGTTGTAGTATTGGCTTAAAAACCTTACCTAACAAGGAGTAAGTAGGCTGATCCTGGATCAGTTTATTTTCTCTGCCATCCCAAAGACTTGCCCACTGACCTCATCCATAGTGTGGTGTTTGGTCTATGAGCAGTTGATTCACCAGTTTTCAGA
FP004911	chr4	102324360	102324611	-	1	validation	TGTATGAATGAGTAAATAAATGAATGAATCTCTCACTTGTCTTCACAGCATTTTTGGAGCCAACTGCAAAAATGGTTTAAGTTCATAGCTGCAAACTAAACAAAATTATGAAAAAGCAAAAGAAATGAAGTGCAGAAAGCGAAGTATACCTAAAATTCTCACTCTGGTGAGAAAAGTTGGCTGATGTTTGAACTGTAGGATTTTTTTTTTTCTTTCTTGAGCCGGAGTCTTGGTTTGTCACCAGGCTGGAG
FP013880	chr15	63157424	63157675	-	1	validation	CGCGGCGCTCGCCAACAAGCCCGTTAGACCCACGGGACTTGGCACCCTGGAACGCTCACCATCCCCGGGTCTTCCCTGCGCCCTGGACCCGAGTACCCTGCTGCGGCGTCCCCCTACCACCGAGAGGAGGGCCTAGAGCAGACAAGCCCGCCCCGCCGCTCCTCCCAGCAGCTTCTATCCCGGAAGTTGATGCCGAGCGCAGATCGCTTGCAGCTTGCTAGCTGTGTGGGCTGGGAGGTCTGGTAGGGCTG
FP008596	chr8	124372649	124372900	-	1	validation	GTAGGTTTAACGCCAGCACAGGGTGGCGCACCCTACCTCCTTCCACCTTCACGCACCAGCGCGGGAGGGGACGCGTGTGCACGCTCCTCCCCTAGGAGGGAGGGCGGCCCGAGAGCCTGCAGACTGCGCCTGCGCAGCCTCGCTCACACACCTCCCCGCCCCCCCGCACCTCCTTCCCCTTTCGCCCCGCCCCGTTCCCACTTCTGCTGCCGCTAGGGGTAGCGGCGGCGGCGGCGGCTGCGGCTCGGGAG
FP006029	chr5	176388547	176388798	-	1	validation	GAAAATCGGGACTCCGACCTCAGCCTCCCGGTGAAGGTCATGAAAGGGGCGGGGAAACGAATAAATTGAGCCTTGTACGCAGGCGCAAATGCTCGTTGCATCCTGGGAGTCGTAGTGCTCAGCACGGTAGTGCTACAAAAGGACTACATTTCCCCAAATGCCCGCAAAGCCTTGTGCACGCCTTCCGGAAGGAGTTTGTTACACGAGGTCTGAGAGACAGAGGCAGCGTGTTTGAGCTGCTGGTGCGGTGG
FP017656	chr19	53520813	53521064	+	1	validation	CTTAGGGTCGCGTAGAGTTGCATGCTGGGAAATCCTGACAGCCGCTTCCAACTCTGTGGATTTCCGGGTTATGGACTACATTTCCCAGAGGCCTCTGGGTCACGGCTCCATTTGCGCCGTGTAGAGTTGCATGCTGGGAAACACTGACGGGCGCCCTTCCAGCTGCGTCGAAGCTTCTCTGGTTTCGCATTACATTTCCCAGAGGGCCTTGGGGCACAGTCCAATGAGGAGCGAGCCGGGTGGCCGGCTGA
FP011895	chr12	49337137	49337388	-	1	validation	AAGATACAGCGGGGGCCGCGAAACCCTGAAGAACCCGCTCAGAGCCGGGAAGGAACCTAGGGGCAGTGGGGGAGTGGAGGGGAGGCGGGGAGGACGTGGGTGGGGGCCGCTGCCTGCCCCGCCCCCGCACTTCCCACTCTGGGCACCAGGGGTCGCTCCGCCTCTGGCCGCTTATAGTGGCTCCCGCCGCGCGGCGCCTCATACCACCCGCGCCCGGGAGGGAGGGGAGGAAGGTTAGGGAGGCGGAGAGG
FP005278	chr5	24644885	24645136	-	1	validation	CAAGAAGCAGTCCTGTTCCTGAGATTGTATTACGGACACAGGCGGATTAGATTGGGCTGATTACCTTAGCCTTGAAACAGAGAGTCTTTCTGCCCTGCGGGGGAGGGGACGCCTTTCAAGCAAAGCCCTCCTCTCGGATGCGCCCTCCCCTCGCATTCACTTGCAAATCGCTGTGTGCTCGCAGAAGCCAGCACTTCCACAGTCTCTAACCTTCCCATGGCGAGTGCTGCAGCTCTATCAGCAGAACCTTT
FP015045	chr16	89873444	89873695	+	1	validation	GGAGCCCGCCCAGACGCCGCCGCTGAAGCGGGAAAGTTCCGGAGAGCCGCGACGCGCAGGCGCCGCAGCCAGGTCCTGCCCCCACCCCTCGCGCCAAGAGTGCGCAGGCGCGCAGCCAGCTCCCCCGCCCCCGACCCCGCGCGAAGAGTGCGCAGGCGCGCCGACAGCCGAGTTTTCTGCGCTTCCTTCTCCCTCTCTCCAGACGTCGTGGTCGTTCGGTCCTATGTCGCGCCGGGCCCTCCGGAGGCTGA
FP011832	chr12	43758708	43758959	-	1	validation	GCCGCGAAGGGGCGGGGCCTGCCGGGTTGGCCCCGCCCCACTTCCCCGATGGGCTTCATCTTGGCGCCTCCAGGCGGCTCCACAAAGTGAGAAACTCCACGTATTGCGTAGGGCGCGTCTTTACGTGGGCGCCTGCCCCTTTAATTACGCCCCTGTCGGAGGCTGAAACACTTAAGGCTTCCGGGCATGCGCACAGCGTTGTGCAAATGAATGCCTTCCACTGAACCGAGGTAGACGGGGTCGAGTTTCTT
FP015059	chr17	676369	676620	-	1	validation	CAGGAGCCAAGAAAGAAGGTCAAGCGTAACTCACCATCTGCTTATTTCCATGTTAGAGGCAGTCACAAGTTTAAACTAGACTCTGCGTCTCAAGAGGAAAATGCCATTCATCACCACTGTAACTGCTCTGGCTAGTACCCAGGTCACTCTTGCTTTTGATTGCTAGTAGTTAGAAATCGTGTAAGGAAAATTCCATCCTCAGTTTGAGGCTGTGCATTAGAAAAATTGCGGAATTACTGCCAACAGAGCAA
FP011656	chr12	9869781	9870032	-	1	validation	TGAGGGGTAATACAAAATCACTGGGAAACTGATTAGAGCAGATTTGAAGACAAAATTGTGGCCTGGTAACTTAGCAAAAGCATAATGGTGTCATCTTGTATGCTGATGATAACGTTCTGGGCACTTTGTTACTTTCGTTTTCTTTTTTCCCTCTGTGTGTCTTTGTGTGTGTTTGTGTGTGTGCTCACACCCACACGTGTGACTTTGCACGCATTTGTCTTTGAGAGAGAGATAATAGTTTTTGTTTTTCT
FP011350	chr11	113314382	113314633	+	1	validation	GACTTTGGCCTTCAACTTGGCGCGCTCCGGAGCTAGAGCTCTGCGGTAGCGAATCTGATCGCGTGTCCATGCCCGCCCCCTGTCCCGCCTCCTGGCTTTTGACCCCGTCCTTGGTCCCGCCCCGGGCCCTGGCTCCTGGCCCCGCCCTTGGTCCCGCCCCGGGCCCCTGGCTCCTGGCCCCGCCCCAGCCCAGGCAGGTCACTGCGCCATTTCCTGTCCAAAGCTGGGCGAATCAGGTCGGTGCCGCGGGG
FP004616	chr4	24980123	24980374	-	1	validation	TTATCAGAAGACGCCTTTTAAATCAAGTTTTCTTTGTCAGTTTTCACGGACAGGACTTTCAGGTGCAGCCGACCATTGGGAGCTCCCTCTTGTGGCCATATCAGGAATGGAAATGTATCTCTTCCCACCACCCGACTTAAACCTCCAAAAGCAAAACTGAGGCCAACTGACTCAAGCTGGTGAATCCGCGATTGCGTACTAGAGAAGGGGGCCTTAGGGAAGTCTCAAAATGCTACGCTTAAATTTTCCTG
FP018624	chr22	21867594	21867845	-	1	validation	GAATGCACGTGACCGCGCCTAGTGGCGCCCCCTCAGTCAACGCCGTCGCAGTGCCCCTCCCGAAGCCCGCCGCTTCCGCCGGCCCGCGCGGCCCCGCCCCCAGCCGCGAGGAACCAATCCGCATGCACACTTCTTGTCCCCGCCCCGGCACCCTCCTCGGCGCGCGCCCCTCCCTCCGCCCGCCCGCCGGCCCGCCCGTCAGTCTGGCAGGCAGGCAGGCAATCGGTCCGAGTGGCTGTCGGCTCTTCAGC
FP007206	chr7	13989334	13989585	-	1	validation	CTTTCGCCTAGCGTGGCCTTCAGGTGGGTCTTTCCAACAACTTTTTATGTCTCTCCAGATAATTGCATGGTTGTGGGTCGCAAAAACTATCCTGATGAAAGAGGTGTCTCCGTCTCTTTAGTCCCGTTAGGTGCAAAGCAAGTCTCGTTGATCGCCATTGCTAGTTTTGCACACGTTTGCGAATCAGAGCTGCCCGGGGTACACCGACCGCGCAGGGAAACATCGAGAGTGTAAATAAATACATCGCCTCT
FP018981	chr22	46773921	46774172	+	1	validation	ACCTTAGAAGTCTGTGCCCTTACCCATCACAGCCTCATCCTTATCTCCTCCATTCCCTAGGGGACTTAACAGGTGTTGAAATTATTACAGAGAAAGCTGACTCACTCACCAGGAATCTGATCCTGCTGTGGGCTGGCTTGGGTGGAGGTGTTCCTCCCGCCCCCGCACCCATCCTCCTGTTTGAACTCAGGCTGCTGCCTGCTGGGCCTGCCTGCCCTTGGAGCCCTGCTGAGCTCAGCCTGAGGCCTGGC
FP004904	chr4	99930437	99930688	-	1	validation	TTTCTTCTCAGGATCTTGTGTATGTACGTTGATGTAAACAGGCATAGTGTACTTTATCTTTCTACAGCACTATTGGAAATAATTATGAAAAATGGAAGCACGGCTGGAAATAGCCCTCATTGCCGAAAACCATCAGGTAGTGGCGATCAAAGCAAGCCTAATTGCACAAAGGACAGCACATCTGGTAGTGGTGAAGGTGGAAAAGGCTATACCAAAGACCAAGTAGATGGAGTTCTCAGGTAGGAATAAAT
FP018241	chr20	56412247	56412498	+	1	validation	CTTCACTGCTTTCATTTTACTCTTATCGTGCTTTCCAGAAAGTTTGCCTGCTGGGAGAGTCTTTTTGATCGTTTCCCATGTGTTGTCAGATAGCTCCATAGAATTCAGTTTCTGAGAACCAGCCAGAAGCATGCAGTGACATTGCACAATCTGCCTCTGAAGCTGGAGATACTAGCTGCAGAGCTCAGGGGAGCTGCTCCACATCACCGACATGAAGGGAACAGGCATCATGGACTGTGCGCCCAAGGTGA
FP015689	chr17	44899352	44899603	-	1	validation	GGAGCAGCCCACGTCTTGGGCTACTCGTCTTGGGGACCCGCTACTCCGCCTTGCCAGTCCGCGGCGGGCATACCTCGGAACTGGTCCTCAGGCAGAGGGTTGTCCTCCCGGGCAGAAAGGCGACAGCGGCGTCTGCGGGGTCCTTCCTGGCGGCGGGCGCAGGCGTTTCCTCGGCGTGGGGCGGAAGCACGATCTCCGGCAGCGGCCTGGGAACTCTTAGCTGAGCAGGCGAGAGGTAAGTTGAAGCGGGT
FP005991	chr5	169583587	169583838	+	1	validation	CACGTCTGCGCCTGCGCCTGCGCCAAGCCCACGGCGCGCCACTTGCACATGCGCAGTAGCTGTAGGAGGGGGTGGGCGCTCCATTGGGCGTTTCTCTTGGTTTTTCCTTTCCGCGCGGCTTGGGCGGACGTCTCGTGAGACGTGGGACTTCTCGCGGGAACTGCATTCAAATATCCCAGGCGCTTACTCGAGAGCTAGCTGAGCGAATGGGCCGGCGACTGTGGAGTTAGCGTCCTCAATGTGGACGCCCT
FP016773	chr19	10625352	10625603	+	1	validation	GGGTGAAGCCGCAGGGTTCCCGGGTGGGGGCGGGGAAGGCTAAATGCGGCCGGCCGGTGAGTGGCGGGAGCAGCTGCAGCCCCGCCCGCGCCCTCCCGGGTCCCTTAGTCTGGGCAGCTGCCCAGCTCGGGCCGGTCTGACCGGTTTGGGCCGCCCCGCCTGGCGCTGTGCTGGGAGGAGCCGCCGCCAGTCGCGCGGTCAGTGCCTCCCTCCAGACTCGGGAGGGTCGAGGGGGCGCGGGAGAGAGCGCG
FP016865	chr19	12995410	12995661	+	1	validation	CCCCTCCTCCCGCCCGCGCCGCCCGCCCGCTCCCTCTCCCCGGAGTGCGCCGCTTCCAAACTTTGTCTAAACTTTCACTTTCACAGCGCGGCGGCTGCGGCGGCGGCGGCGGCGGGCGAGGGTGACCGGCCGAGCGGCGGCGGCATGGAGTAGACGCGCGGCGGCAGCGGCGGCGGCGGCGGACGCGAGAGGCAGCGGCGAGCGCGGCGGCGGCGGCGGCAGCGGCGGCCCCGGAGCCGGCGGGGCCGAGC
FP011817	chr12	39625983	39626234	+	1	validation	ACTAGATAAAATTATGATGGAGGTGTTTTGAAACACTAATCCTGTGGCCCCTGTCAAAAATTTAGGCTTTCATTTATTTAGGCCTGCCTTGCCTCAGCGATTAAAACGATGTCAGCCAGGTCGGCTGCTTTGCCGCAGCGTCCCCTTGCGGCCCTATCAGGACCGGAAGGAACACGCAGAAGCCGCAGAGGCTGTTGGGAGGTTGGGCCTCAGGCGGTGGACCAGAAGGAGCTTACTCGGGCCCTGACGTT
FP002771	chr2	157325351	157325602	-	1	validation	ATGGGGATAAACCACCTGAAAACGGTCAACAAACAATCACTAAAATCAGTGAGGAATTGACTGATGTGGACAGCCCCCTGCCACACTACAGGGTAGAACCCAGTCTGGAAGGTGCACTCACCAAAGGAAGTCAGGAGGAAAGAAGAAAATTACAAGGGAACATGCTGCTCAACTCATCCATGGAGGACAAAATGCTAAAAGGTAAAGTGTTTCCCTTCTCCTTAATTTTCTTGTTTTGACTTATAAATTTA
FP000767	chr1	58700011	58700262	-	1	validation	GGCCACTTCCCAAGGTCTCTACATGTCAACAAGTTAGAAAGGACCAAGCCCCAGGCCTCTGGGGGCGCCTTTGCTCTGGAAACCCCGCCTCCCGGCACGGGAGGGGCCGGACGACGCTTCAATCCCGCCCCGTGACGCGTTAGGCCCTGCCCCCGGGCCATCTTCGTCTCGCGGGATCTCTCGGGAGGACGGACGGGGTCAGGTCCCATCATGGCGGCTGAAGAGGCGGATGTGGATATCGAAGGGGACGT
FP005577	chr5	103129321	103129572	+	1	validation	ATCAACTCAAGAAAGCAGTAACTTCACTGTCTTTGTATTTTGAATTGCAACAACAACTTTGATATCAACAATGAAGCAATGATATCTAAGAACAAAAGAGTATTTGCCAACAGTCATCATAATATCAAGTGATTGTATAAGCAGAAACAAGCTGTCACAGACCTGTGCGTCAGCTAATATATGGAGAATGCTTTCTTCTGATACTATTTACTTAGAGGCAGTTTTAATATAAATCATTTCAATTATATCTA
FP011649	chr12	8697795	8698046	+	1	validation	CAGTGCTCGGCGCGCTGGCCCGGCCCCCAGCTCCTCCGGCCGCCGCCCCCCGCCCGCCGCGCCGCGGCGCCAGGAGGGCGGGGCCGGGGCGCGCGCCGCTCCTCAGCAAGCGGGCGGGCGGCGTTCGGTCTGAGGGAGCGGGGCGGTCTCCGCCGGCGTCGCGCGCGCTGTGTGTGAGCGGGGTCGCGCGCGCGCGCGGCAGGCGAGTGAGGGAACGAGGAGCGGCCGGGTGTGAGTGTGTGGGAGTGAGA
FP009616	chr10	12042758	12043009	-	1	validation	GACGATCAGGACTGTTTTTAATCGGGCAGTCGCGCGGATGGCCTTTTCCCTCTCGCCTCCTTCCGCCCCGCCCCCACTCTCAGCCCGGCCGCGCTGGTGAGTGGCGGGCGGGAGGGCTGGCGGGCGGAGGGAGGAGGGGGAGCTGAGGGAAGGCGGGCCCTCGGCTGCGAGATAGGTGGGGGGAGGGGAGGGGAGAGCCCGAGCGCTGGAGTTGGTGCTGGGAAACCCGGGGCTAATGTTGACAACAGGCT
FP007668	chr7	101200982	101201233	-	1	validation	CCTGGGACCCAGTGGGTGGGGCTAGCCAGATCACTCTTGTCCACTCCAACATCAATTCCTGAGGCACTCGGGTGTCACCTCTCCCTTGCACACCCCTTACGCCTCCCAGCCCCACCCAGCAGCCAGCCCAGACTGGTGACTGACAGGAAGTTCAAAGATCAGGCTGAGAAAAAACCCAGAGACATCTGGGGCTCTGGATCCAAAGATCTCCCCACTCACACACCTACGGACACACGCTACTCTGGGAGGTG
FP015819	chr17	57085045	57085296	+	1	validation	TCCGGGCGGCGGCGGGATGTTAGCCCAGCGCTGTAGACCGCGCCCCCTCTGGCGCAGCTGGGAGAGGCGCGGCGGAGGCACTGCCGGGAGCGGCCGGCGGCGCGCACGCGCAGGTGAGCGGGGCGGGCTCGGGAGCGCGCGGGCACAAGGGCGAGGGGCGGGGCCTGCGTCGCGCTGCGGGCACGTCGCCGCGCGTCTTCAGTATTTAATGTCTCTGTGTTCCACCCGCCTGGGCTAGCACGTGGGGGAGC
FP001836	chr1	224330089	224330340	-	1	validation	AACTCCCTGAGGTTTTCAAGTCTTCTCCCTCTCCTCGGAATTTTTTCCCCTTCATTCGCATTACTTTTATTCTGACGGAAACGGCGCGGGCTAGGCCCCAGGAGGGGCGCATTTCAGGTGCAACTGGCCCGGCGATCGCGGCGGGCAACCCCTACGAGGCCCCCGGTTGACTAGGAGCTGGCGGTCCGAGCTGTGGCTTGGAAGACCGACGCGATGAAGCCCAGACCTGCAGGGTTCGTGGATAATAAACT
FP007930	chr7	143289152	143289403	+	1	validation	CGAGACTGAAGATTAAGGTCTTCGCAAAGGGAACCTGCAGAAGGACACAGAATTAAGACGTAGGAACTCCGTGGTGCCCTTCGCCCCTCCTCCCCTTCCAGTTGTGTCCTGAAATTCTAAATGATTGGAAACTTTTATAATGTTCACGTTTAAGACTCTCATACTACAGAGTAGAGGAAGGAGCGGGTGTGCAGAGGGGGTGATGCTTCAGATTAAAACTAAAGAGTTGATTGAAAATTCAAGTCTGGAAC
FP017592	chr19	50854720	50854971	+	1	validation	CATCCAGGGTGATCTAGTAATTGCAGAACAGCAAGTGCTAGCTCTCCCTCCCCTTCCACAGCTCTGGGTGTGGGAGGGGGTTGTCCAGCCTCCAGCAGCATGGGGAGGGCCTTGGTCAGCCTCTGGGTGCCAGCAGGGCAGGGGCGGAGTCCTGGGGAATGAAGGTTTTATAGGGCTCCTGGGGGAGGCTCCCCAGCCCCAAGCTTACCACCTGCACCCGGAGAGCTGTGTCACCATGTGGGTCCCGGTTG
FP003260	chr2	237517948	237518199	+	1	validation	GGATAGATGGGCCACTGATTGATTGAGTGGATGGATGGATAAATCGATTGATGGGTGGGTGCATGGATGAAGGAGGGAGGGGTGGATGGGTGGGTGAGTGGATGAGCCACTGATTGATTGGGTAGATGGATGTATAGATGGATTGATGATGAGTGGGTGGATGAAGGAGGGATGGAGGGATGGATGGATGGATGGGTGGGTAGGTGAATACATGGATGGATGAGCCACTGATTGAGTGGGTGGATGGGTGG
FP007208	chr7	13991326	13991577	-	1	validation	ACGTGACCAAGAAGGGCCGGAATTTTGTGAATGGAGACGAAAGGTGGCTGTTGGGGCGGGCGGGACTCCGCCCCCTTCCCTAAAGCGGCCAATCTCCTAGCGGAGCGCTGAGGGGTCAGCAATAAACAACAATGGGCTGGGGATTTACGGCTCGTTATTGGCTGCGGGAAGAACCTCGCCGGGACCTAACCCGAAGCTCCGCCCCCTCGCGGTTACCCTGGATACCCGTCAGGCTCGGGCTGCAGAGAAGG
FP011228	chr11	86003365	86003616	-	1	validation	CAAAAGTAAGATTGTCATTCTTATTAAGTAGCTAAAACTCCACTGTTAATTAGTACATGAAGAACATACCTGTTTATTTTTGTTGCTCTCTAATTAGATGATACTTGATTTAATTTGACCAGTGGCCTAATTATCATGAAATTCTTAATATTTGTACAATTGCAGGCCCCTAGCAGTCTTCTTGATGCTTTGGAACAACATTTAGCTTCCTTGGAAGGAAAGAAAATCAAAGATTCTACAGCTGCAAGCAG
FP013636	chr15	32614944	32615195	+	1	validation	CATTTCTTCCGCACTCTCCTCTCACGACGGGTCTTCTTTGTTGTACTTAATTTCCTACGCAATAAGATTTCAGCATGACCATCAGTCCCCCAAAGACTAATTCCCACAGAGCCGAAGTTCCCACCAAGGGCCGAGGGTTAAGGTTACTAAAATCAGCGTTTCTGAATCCTGTCTCAAGTTGTCTCATCTGGGCTTCCGTAAGAACGGTTTCTTCATAAGAGGGCCTTCAGCGACAGCCAAGCTCGGAAAAG
FP015230	chr17	7484198	7484449	-	1	validation	GAGGTTACGGCAGTTTGTCTCTCCCCCTTCCGGGAGCCGCCTTCTTCTCCAACCGTCCCGGCCGCGCTCTCGGCGCTTCTGAGCAGCGAACTCGCTGAACGACGCTTCTTATAGATTCGCCCTCGCGTCCCCGCCCCTTCCTTTCCCGCCCTCCCTTGCGCTACGGGGCCGCCTGCGCAGGCCCGCAAGGGGTGCGAGCCAGAGTCGTGGGTAAACCCCCCGCTCGCTGGTTGGGCGACCTTTTGAAGTGA
FP005518	chr5	81750997	81751248	-	1	validation	AGCCGCCACCGCGCGCTTCGGAAGGCCAGAGGGAGGGGGAGGCCTGTCAGTCTCGCGCGTTGCCTGGGCGAAGGGGGCGGAGCTTTGGCGTGGGGCGGCCAATAGTGGGGGTGGCTGCGTGGGTCGCCATGGGGACGGGGCTGTTCCCGGGGAGGCTGTGATGGGTTGACAGGTGCGTGACAGTGGGAGCTGCTCTCGGCACAAGCATGTACGGCAAAGGCAAGAGTAACAGCAGCGCCGTCCCGTCCGAC
FP000375	chr1	26890223	26890474	-	1	validation	TCTAATTCGAGTCTGAAGGTTAGAGAAATTCCGAGAGCGATTTTCCCGCAAAGACCCCACATTCGACCCACTACCCGCGAGAGAAATGCGGCTGCGCCGCCCGCCCCTCGGCGTCTAGCTTCGTCCCCGCCGTGAGGGCGGGACTTCCTCTCGTTGGCTCGCCGTTTCCGACGCTGTCCGGAAGTCGAGTTAGTCTAGTTAGTATCGGCCTGTTATCTCCTTTTGCGCGACACGGTCTCAGCTGTTCCGCC
FP007692	chr7	102912799	102913050	+	1	validation	GACTCTGCTTTAGACTTGTGCTCAGACTGTCTCCACAAAAGTGGTCCGTTTCCTCATCCAATGAGCTCTGCCTGTTTGTCTGTGTGACACTCCAGTGGAAGGAAGCCAAGAGCCTATAATAGCGATCCTGGGGAAAATTTCAGCAGCTGGGGCCATGTAATTTAAAACCTCTGAAAAGTGTGCTGCGGTCCGTGCACAGCATTAGTATAACGTGAGGGCTGAATGCAGCCCATTCTCTGGAGAACTTCCTC
FP015769	chr17	49230731	49230982	-	1	validation	TCGGAAGGGCAGCGCTCCATCCCCACCCTGCTACCCTTTCCCCGAGAGCCTAATCCCACCTTAGCTGGCGCTCCCGGAGCCTCCGGGGCAGGAGGGAGGCGTGGCCTCGGGCGGCCCGCCCCTTTGATGTGCGCCGGCACCGCTGCGATTGGACAGTCGCTTGTGACGTTGGGGACTGCGGTGGGCTCCGCTGCTGCAGCAGCCGCAGCGCCGGCCGCGGCTCCGGCTCCGGCTCCGGCTCCCGGGCATTT
FP017775	chr19	57363365	57363616	+	1	validation	CGTGCGTGCGTGCGTGCGTGCGTCCGTCCTCGTGCTCGCGCATCGTAGGAGGGCGGGACTTCCGGCGTCCTCTTGCCGTGGTTGATTTGATTTTCTCTGGTGTTTTCACTAGTTCCGGCCTTTGGCGCTCTATGACGTCACCGAAGTGACGGAGCGGAAAAGCGCGAGAAGCGGCTTGGTTCCTTGTACGCAGAGGCGGTAGTGACACAGGCACAACTGACAGTGGCAGAAGCTCAGCTGACAAGGACTGG
FP001653	chr1	201171505	201171756	-	1	validation	GTGTTCTTTCCGAACACTCTTCGGAGTCTCAGTTTTATCCTCAAAAGGCAGAGAGGCGCAGGTTTTAGCTTAACCCACTGCGCAGGTGAGAAAACTGAGGTCCAGGGTGACGTCATGATCTAGGGCGAGCGGCAGGTGGGAGATTAGAACCCGAGAGGCTGCCGCGGCCTCTGCTCCCTCCCTGCTAGGCTCTGTCCCACAATGCACCCGAGAGCAGGAGCTGAAAGCCTCTAACACCCACAGATCCCTCT
FP018399	chr21	33324779	33325030	+	1	validation	CGCGCGCGCACAGGGGTGCTGCAATTAGGATGGGGCAATGGGAGCTTGGAGAAGGGGTGCTAGCTAGGAGGAAAGGCGCGTGCGTGGAGGAACGGCGCGTGCGCGGAGGGGCGGTGTGTGTGTCAGAAGAGGCGGCGCGTGCGTAGAGGGGCGGTGAGAGCTAAGAGGGGCAGCGCGTGTGCAGAGGGGCGGTGTGACTTAGGACGGGGCGATGGCGGCTGAGAGGAGCTGCGCGTGCGCGAACATGTAAC
FP012658	chr13	26557482	26557733	+	1	validation	AAGGGTGAGAGCCACCGCCTCCGCAATCCCCCGCCCCTCGGCACCGTCGCTCAGCGCGCTCGCCGGCTCCCTATTGGCCGAGGCACTCGCCAATCCTGGTGAAGCGGCCCCGTGAGTTGGCCCTGGTTGGTGGAGCCGGCACTGAGCCCCCCTCCTGCTCCGGCCGAGCCCCCCCGCCGGGAGGGGGCGTGGCCGCGGCCGTGGACGGGCCGGAGGCGGCGTCGCTCGCGCCGCTGCGTGCGGTGTGGTGC
FP011149	chr11	73876783	73877034	-	1	validation	AAGGCGCCGCCATCTCGACCCCGCCTCCACCAGCCCACCTCTTCCGCCCCTTCCCGGAAGTGAGTCCCCGAGAAGCGTGCGGCGAGGACAGAGGTGGGCTCAACACCAAGAGGCCACCAATGAAAACGTGACTTTTTCACATTCCCCGCCCCTTCCGTGTGCACGCTCCCCACGAGGGGCTGGTTCCTGAAAGGATGTACAGAGGATCCCCAACCGCCTGCGAAACCCAAGCCGCCGCGTAGGAGCGTGCG
FP011435	chr11	120239875	120240126	+	1	validation	CGCCAGGGCTTCCTAAAAAGGGCGAGCAGGACCAGGAGTTGGCGCCACTCCGGGCGCGGTGAGGAGGTGCTGCCAGGCTGGGTTGGAGCAGGCAGGTGTTCCGCGGGATCGAGACCCGCCCCGGCAGCACCCCCGGCCCCCTCCCCAGGTCCCCAGCTCCATGTATGCAAATCACTGCTTTGCATGGGCCTGGGGCCAGGCGCGGGGCTCCTGCCAGGGGGCGTGGCCGGCGGCCCCCGCCCCGCGCCGCG
FP008389	chr8	70608299	70608550	-	1	validation	CGCGAAGCAGGGAAGCGCCGCTCACATTCCTGCACGCTCGCCCGGCCGCGGGCCAGCGGCCTCCGGTCCCCAGGTGGCGCTGTGGGCTCGCGGAGGCGGCCGCGGCACCAGGGAGCGTCGTCTCCCTGGTGCGCATGCTCGCCCCCGCTGCGGGCTAGCTGTTGTGTTTTTTTTTTTCCCCCGGGCGGCCCGGCGGCTGCGTACTGGCTGTGGGATGGGAAGTGAAGCCCCAGCGAGCGGCTGCAGCGGGG
FP007160	chr7	4682094	4682345	+	1	validation	TCGCCCCCAGCCCCGCGCTGGCGCGCGCCACGTGGGCTACGCTGGGCGGGACCGGCGGGCGGGCGAGGCGAGGCGCACGGTGGGCTGGGAGGCTCCGCTCCGCGCCGGCGCGATCTGGCCCGGGTTTCCGCGCCCGCCCAGCGCCCCGACCTCCCCGCCCCCCGCGCCGCCGCTGCAGCCGCCGCCGCCGGAGGCCGCTCGGAGCCGCGCGAACATGGCCGAAGTCGGCGAGGACAGCGGCGCCCGCGCCC
FP004160	chr3	142000251	142000502	-	1	validation	TGTTGAAACCTAATCCCTACTGTGGTGGTATTAAGAGGCGGGGCCTTTTGGGAAGTGATGAAGTCATGAGGGCTTCACCCTTATGAGTGGAATTAGTGCCCTTATAAAAGAGGTTGAAGGGAGCACCCTGGACCCTTTCTGCTATTAGGATGCAACCAGAAGGCATCATCTTTGAAGCAGAGAATAAACCCTCACCAGGCACTGAATCTGCCGGCACTTTCATTTTGGACCTCTCAGCAACCTCCAGAACA
FP001203	chr1	150926162	150926413	+	1	validation	GGAGACTCGGTAATATACTGGCCACGCCTACTCTGCCCTGCCAGTCTCTTCTCACGTGTGCCCCTGGAAGGCTCTGCCTTAACGCATGCGTAGTTACCTTAGTTTTCGTCGCCTCTGAGGGGCGCCCCGCGGCTTTGGATTTGACCCCGTCAGGCTACCGTCGCGGTGACCGGAACGGCACTAAAGGTTTGCTTCCGGGCGTTTCTTTTGCTTCCCCTTCCCTCTTTCACGCTTCCTCCCCTCCCCCTCCT
FP014375	chr16	4538722	4538973	-	1	validation	TCTTTCCCTATTCTCCAGCCTCGTGTACATTGGGCAAACAGGCCCCGACCATCCCACGTGCTCAGCCCCAGCTCTAGGCCAAATCGCCTCCGCTCCCACTGACGTCATCACGGCGCATCACGTGCCTCCCGAGACCCCACCCCCCACCGTGACGTCCCGGGCCTCGGCCCTGGTCCAAAGTCTACCCGCCTCCTTGTGACAGAAGTGCGACTGCCAGCTGCCGAGGCGTTCGGTCCTGCTGTTGCGGCCGC
FP000993	chr1	108963402	108963653	-	1	validation	GAGTTCCAGGCGAGATTTCCTTCGCAGAAAGAGTTCCGCAGATGAAAAGTAAAAGAAAATCCGCGGCGAAAATAACCCGCGCGCAGAGGCCGGTCCCGCTACTAGCCCCGCCCAATCAACGAGGCCAATCAGCGGCCCGCAACGGGGGAGGGACACGCAGCCGTCAGCCGAACAATTCGATGACGAGGCCCAGGAAGCACGCTGAAACCCTGGGCGGCGGCAAGCTGTGCGACCTCTTCTGCGGCCGGCCT
FP004813	chr4	81215171	81215422	-	1	validation	TTGCCTCACTATGGAGGGTATGGGGTATAAAGTGCAGACTGCGGCTGTGTCGGGAGTAAAGGTTCGGGCTAGTGTGCGTGCGGGCGCGGGTGACCCCACCTTTAGGGAAGCCGGCGCCCAACTATCCAGGCAGTAGCCCCGGCTGACCCCCTCCCTCCTCCTTCCCTCCTCTCTTCCCTCCCTCCCTGCGTGTCTCCTCTGCACCGGCCCCCGCCGCGGAGCCAGGCGGCAGCACCGACGCCGCCGCAGCT
FP000235	chr1	19251471	19251722	-	1	validation	GCACTGCAGCCTGCCTCTCCCTGCCAAGCCGCCGCCTGCGCTTTCTCCCGCGCCTGTAGGGCAACTTCCCGTGTGCCGAGGCAGTCCCTGAATCCCGAGCGTCCCGTCCGACCGAGGGCCTCGCGCGCTATCCCAGCTCTGTGGTCCCGCGGAGGCGGAGCCCGCGGCGCGGTGCATGCCGGGACGGCGGTGGGTGGTGCATGCGCTCGCATCATGGCGGCTGAGTGGGCTTCTCGTTTCTGGCTTTGGGC
FP005558	chr5	96661856	96662107	+	1	validation	CGCACGCACACACACACAGACCAATTTGTTTTTGTGATCCGCTTCCTTATCTAGACTGTCCAGGCTTGGGCCAGATTTGAGTTGGTAAACTCTCGCAGCTAAAGCGGAGCAGAATGTAGAAAGAGTATCTACTTCTCCGGATTGTTGGGAGGAGGGTCGAAGCGTAGACTGGCCACGCCCGGGTCCCGCTCTCTCGTTGCACAACTGCAAGCTAGATCTGGGCATCGAAGGAGGAACGCTGCAACCTCCAA
FP005504	chr5	79277773	79278024	+	1	validation	AGTCAATGCTTTTCATTCCCTCTTGCTTCTGTGTTTTAGGGTTCCGAGGTCCTGGCATGATAGGGAGAGTTCAAAGCAAGTATTTATAGGTTATTAGAATTGATTTTCTTAGTTGTGAAACTGTCTTGTTCCACAGCTCTTGGATAAGCACAAGAATACAGAGAGCATGGTGGAGCTTCTGGACTTGTATCAGATGGAGGATGAAGCCTACAGCAGCCTTGCAGAAGCTACAACCGAACTCTATCAGTATT
FP017887	chr20	3781397	3781648	-	1	validation	TTTGGCTGGTCCCAAAATGGATGAGTGGCTGGTCACTAGGTGGCGTGGGCACAGAAGTGATGCTTTCTTGGGAAGAAGACAAGGGCAGCTAACCGCCTTTTCAAAGCTCGGGCCTGAGAGGAGGTAGCGGGGAGGGGATAAAACTACAACTCCCAGAAGTCTTTGTACCCAGGAGAGTCGGGAATGGTTTCCATGGTTTCAGAAAACAATATGGCCGCTCCCAGCTGGGACGTGAGTCTCTGGATTAGGCA
FP016257	chr18	21111627	21111878	-	1	validation	GAGTGCGCGCGCGCGCGCGGGGTCCCGGCTGGTTCCCCTTCCGAGCGTCCGCGCCCCGCATGCGCAGTCTGCCCCGGCGGTCTCCGTTTGTTTGAACAGGAAGGCGGACATATTAGTCCCTCTCAGCCCCCCTCGCCCCACCCCCCAGGCATTCGCCGCCGCGACTCGCCCTTTCCCCGGCTGGGACCGCAGCCCCTCCCAGAAGCTCCCCCATCAGCAGCCGCCGGGACCCAACTATCGTCTTCCTCTTC
FP005608	chr5	113294887	113295138	-	1	validation	GTTTCTACTCCCCCTCCTGGACTCGGGTCCTTAATTACTATTGCAGAGAACTGGAAGGATGCCTGCTCTGCTCTCCCAGCACGGTCTCCTCTGCTCGGCCTCCAGGCTACTCGCCCCCACCCTTTTCTCCCTCTAGCCCCGCCTCCTCCGCGCACCCCGTGGCCAGCAGCTAGACAGCTGTGTTCAATCAGAAACCACTTACAACTCCAGACGATTCGAAGGGGAAACTTCGGCGTGAAGTGCAGCTCCGC
FP008189	chr8	26383182	26383433	+	1	validation	ACTGCGAGGAAAATGAGCAGTCTCTGCCCCCGCCGGCCGGCCTCAACAGTGAGTGCGGGGCCGAGGCTCTGTGAAGGGGATGGGGGAGGAGGAGCAGCCCCGGCCGCCGCCACCGGCGCGGCGCGGGAGGCGGGAGGAGAAGGCAGCTCATTGGCTCAGGCATGGGATGTCCAGGTGACTGACAGCTCCCGTCCCCTGTCAAGAGGAGGGGCGCCTGCCTTGCTCCGGGCCTCTCTGTTGCCTCCTGGGGT
FP009392	chr9	128920595	128920846	+	1	validation	TTCACTGAGTCTCTACTACAAGCCTTCTGTGGGGGCTCCCAGGGGAATGGCTGGCCCAGTCCGAGGGGACCTCAGTGTTCTTGGCACATGGTAGGCATCTGTCTTTGTTGGGCAGTTGCATCAGAAGGGTTAAGGACAGCTGGGAACACATCCTGCCTCTAGTGAACCTCGTGGTTCTGTCATCTGCCTGCCCCTCACCCAGCCTAACCCCTCTGAACCAGGAGCCTGAGCTGCACTTACTGCTCCCCCCT
FP007387	chr7	47582060	47582311	-	1	validation	GGGCGCACCTCGGCAGGGCGGGGGACCCCGGTGAAGGGCGCCTGGAGGCGCGCTGCATTGTTAGGGTGATGAGGCCAGGTGACCCGCCAGGCTCGGGGCGCGGGCGCGGGGGCCGGCCGTGGCGATTGCCCGCGCCGCCTCGAGGGGGCCCTGCCGCGGGCGCCGGCAGCCAGCCAGGAGGACGGGCCCGCCCGCGCGCTACTCGGAGCCCAGCCTCGTTGCGGCCATCGCCCTGCCGGACCGCGCCTCAG
FP015522	chr17	38604664	38604915	-	1	validation	AGAGGCCAGGGGGCAGGGGAGGGCGCCCCCAGCTGGGGAGGGCATCTAGGGTGAGTGAGAAGATAGGGAAGGCAGGCGGCTGAGGGGTGGAGGGTGGGGCTGGGAAGGGGACGCACCGTGGCCCCCCCTCCCCTCCCACGCACCCCCCTGCTACGTCAGAAGCTCCCGGGCCAGCTCCGAGTCCTGTCGGTGGGGGCAGCAGGATGCTGGTGGAAGCAGAGGGGACAGGCCGGGTGCCTCCGCAGCTCAGC
FP018486	chr21	42315388	42315639	-	1	validation	GAGGACCCGGAACCAGAACTGGAATCCGCCCTTACCGCTTGCTGCCAAAACAGTGGGGGCTGAACTGACCTCTCCCCTTTGGGAGAGAAAAACTGTCTGGGAGCTTGACAAAGGCATGCAGGAGAGAACAGGAGCAGCCACAGCCAGGAGGGAGAGCCTTCCCCAAGCAAACAATCCAGAGCAGCTGTGCAAACAACGGTGCATAAATGAGGCCTCCTGGACCATGAAGCGAGTCCTGAGCTGCGTCCCGG
FP017130	chr19	35248535	35248786	+	1	validation	ATCTCGGTGCGCTTGGTTTGGCCGGAGCAGATGGGGGCCGGAAGGGACCTGTGGTCCGCAGGCGCCCTCCCAGCGGGCCAGTCACTTGGTTCGGGCCCTGGGGGACGGAGCGCACCTGGGTCAGCCCACTTCCGGGGAGGGAGGCAGAGGAACCCCTCCCCGCCGCTCACCCCTAAGCCCAGCCCTCGGCTCCCACCCTTGTGTACCTGGGCCGAACCATTCACCGGAGCGCGCAGCGGGTGGAGTGTGGC
FP000828	chr1	67833387	67833638	-	1	validation	CAAAGACGACCAGACGTCCACCGAGCTTCCACAGGCCAGGCCTCGCCCCTCCTCCTGCGGGGCCACAACGCGGGCTCCCCGGCCCGGGACAGCACAGCCAGGGCGTCCGGGCGGGAGGCAGGGCCCGGCGCGGAGGAAGGAGCCCGGGGCCGCCGCCGCCCCGCCCACCCCAGCGAGGCGGAGCCGGCGCCCGGAGGAGCAAGAGGAGGAGGAGGAGGAGAGGTCGGAGCCGTCTCCAGGAGCCCTTAGAG
FP016559	chr19	2328530	2328781	-	1	validation	GCTCACCTGGGCCGCGAGGAGCAGAAAGGCCGCCAAAAGCCGCGCCAGCGCAGCCGCCACCGCTGCCGCCATGTCGGCCGGTGCCCGCCGCAACCAACGGCAGATGTTTCCCAGCAACGCCCCGAGACCCCGGGCCACGCATGCGCCGCGCATGCGCGTGGGGGAGGTCAAAGGGCGCGGGGCGGTGCCTGCGAAGAGCCACACGGCGCGACAAGATGGCGGTGAGCGCGCGGCGGAATCTGGAGCCGGGG
FP010090	chr10	97584122	97584373	+	1	validation	CTGGCCCCCTTCCTGGTGCCTGCTGTCCTGTGGGGCTGTGAGTGGCTAGCACAGGCTTGTCACGTGGCTGCCTTGGCAGGAGGAGCCTGGTAAGGGAGCGGGGCAGGGGATGGACAGCCTGTCACAGGCAGCTTCTTGCCCAGCCTTGCCACACCTCCCACAGGGCAGGACCTGCTCAGCCAATCAGCTCCACCAACCTCAGAACTGCCCGTGGAAACCTGGCTGGGCTTCTGGCTTCAGAGCTGGGAATT
FP012335	chr12	102480441	102480692	-	1	validation	CAAAAAAGTCCTTACTCAATAACTTTGCCAGAAGAGGGAGAGAGAGAGAAGGCAAATGTTCCCCCAGCTGTTTCCTGTCTACAGTGTCTGTGTTTTGTAGATAAATGTGAGGATTTTCTCTAAATCCCTCTTCTGTTTGCTAAATCTCACTGTCACTGCTAAATTCAGAGCAGATAGAGCCTGCGCAATGGAATAAAGTCCTCAAAATTGAAATGTGACATTGCTCTCAACATCTCCCATCTCTCTGGATT
FP016160	chr17	82418146	82418397	+	1	validation	CTGGGAGCAAAGCCCCCTTCTCTTTCTCCTGCCCCTCTGGGATAGGTGGGCCTCTGCCGTCGGCTACCGCGGCTCCCGCCCGCACCTGCCCTGCTCCCAGCATGGGCTGCGGTAGACCCCGCCCCCCGCGGCAGACCACGCCCACCTGCCCCCCCCCACCCCCCCACCCGGGTCGGGCCAGGCCCCGCCCCATCAGCCCCAGTCCCGCCCACTCCATGGCCCTGTCCGCCGCCGCAGCGCGCGCCCTTCCC
FP016697	chr19	7747395	7747646	-	1	validation	TTGGATGACAGATCCCTACCCAACTTCCTGTTTCTCTTTCTGTGGGAGACTAGATTTAGGAAGTAAAGATCACAGGGTGGGAAATAAAAGCTGTGGCCCCCAGGAGTTCTGGACACTGGGGGAGAGTGGGGTGACATGAGTGACTCCAAGGAACCAAGACTGCAGCAGCTGGGCCTCCTGGGTGAGGCTGGGTTGGGACGCTGGGATTCTGGGAAGGGGGAAGGGATGGCCAGCCATGGCCTCAGCCTGCC
FP018730	chr22	30693664	30693915	+	1	validation	AAAAAAAAATCACTGATGGCTGGGCACAGTGGCTCACGCCTGTAATCCTAGCACTTTGGGAGGCCGAGGCGGGAGGATCCTGGCCTGAGGTCAGGAGTTCAAGACCAGCCTGACCAACATGGTGAAACCCCGTTTCTACTAAAAATACAAAATTTAGCTGGGCATGGTGCGGGCCGTGGAATTCCAGCTACTAGGGAGGCCAAGGCAGGAGAATCGCTTGAACCTGGGAAGCGGAGGCTGCAGTGAACCAA
FP009363	chr9	128204171	128204422	-	1	validation	CTGTCTTAGTCATCTCTGCCCTCTGCCGCCTCTCCCTCCAACACGATTCCTCTGGCCAGGCCAAGGGGAGGGCGACACTGACAGGCGCCCCCCACCCAGGGGCCGTGGCGAAGCAAGGGGCCGGCTGCTCAGAAAAGGATAAGAAGTGGTTTCTCCTCCCCTCTTCCCTTCCTCATCCTGCAGCCCGCGCCTCCCCCCTCGCCGCGCTGCGCACGGATGGCGGCGGGAGCCGCAGAGGTGTGTGTGTGCTG
14155	chrX	147946729	147946980	+	0	train	TAATTATTAAGTAGTTAGGGTTCCTTCCTGACTATGGATTCATTTGGGACCATCACCCAAGATTAGTGAGAGATACTTTTTAAGACAAAGTTTATGATGGAATATTTCTTGGAATTCATAGCAGCTCAAAGTAAGTGTTAACTATTGGTGGGTTCTTAAGGACCAGCATGTCCAAGGGATAGATAGATTGGTAAATTACTGTATGTACCTGTCTTTTACAAAGCTATAAGATATTTAAGTACCCGTTTTGT
10965	chr16	2127870	2128121	-	0	train	CCCTGCCAGCCCCTCCCACCTCTCCCTCCCTCCAGCCCCTCCCACCTCTCCCTCCCTGCCAGCCCCTCCCACCTCTCCCTCCCTGCCAGCCCCTCCCACCTCTCCCTCCCTGCCAGCCCCTCCCACCTCTCCCTCCCTGCCAGCCCCTCCCACCTCTCCCTCCCTGCCAGCCCCTCCCACCTCTCCCTCCCTGCCAGCCCCTCCCACCTCTCCCTCCCTGGCTCATCCCTGCTGTGTCCCTTCTCTCTAGT
14398	chrX	149488433	149488684	-	0	train	TGCAGTCAGGCGGTCCACAGCTATTTCTGTAGCATTTGCCATGTGTCAGACCCTGTGCCAGGCCTGGGGGCCCCCTCTGCCTGGTGCAGGGCAGCTCACAAAAGCTGGCAGAGGCCGAAGGCTGCGTGCCAGTATTCAGAATGCCACAGAGCGCCTGGCTGCTGTACCTTCAGAGCCTCCACAGGCACCCCAGTCAGAGTTCCGGGAGTGGATTCCTGAGGTCGCACTGCCAGGTTCCCTCACGTTGCCAT
7890	chr11	119088222	119088473	+	0	train	AGGGGACTGTGACCTGGGGACTTTTTCTGCAGGAAGAAAACAGCCCAAAGATGAGAGTGATTCGCGTGGGTACCCGCAAGAGCCAGGTGGGTGCAGGAGCCGGGGTGGAGGAGGTTTGTCAGAACAGTTATGATGCTCACAGCATCACAAATTGGGGGACTCAGAGGGTTAGTTCCTAGTATGAAGGAGATGGGGTGGCTGGGCGTTAAGTTCCCCGGGAAATGGCAGATTACATTCTATGGCAAGATCAT
1007	chr1	173915077	173915328	-	0	train	ATCTATCAGAGTCCTTCCCCAAACAGTTTCTGTAGATGGCTCCCCCTACCACCCTGACTCTTCACTGGGCACTAAAGCCGATTTTTTAGGCATGCACATTCCATGTCACAAACAGGAAGCTTCTCATTCTTTTTTCTCCCAGCGTGGGGAATTGAGCACATAATACTCCAAATAACCATCAGATGATTCTAATTCCAACATGACCACGTCCAGGCAACTGAACTGTCCCCTGGCAAGAAGTCTAGGACTGA
9603	chr15	90887566	90887817	+	0	train	CAGGGTTTTATTACATTCATTGAGCACTGTTCTGGGCTCTGGATTATACCAGAGAACGATGGTAGACAAAAACATCTGTCCTCAGGGATCTTTCGTGTTAGTGGAGTGAGGATGTGAGGAGCACTAAGAGCCATGGAGAAAAATAAAGCAAGAGAAGTGGATCGGGACCTGGGAGCACGGAGGCAAGGGAGGAGGTGACAGTTGTCCATAGAGTGATCTGGGAAAGCCTCTTGAGAGGTGACATTCAAAGA
1365	chr1	225837385	225837636	+	0	train	GCCTGGGAAGGGGCTGCCTTCTTTTGATATGCAGAGGCCCCAAGTCAGTCCAGAGGAGACAAGAATCCATTCACCTGCCCTTGGCTATGATGATGTTTTCAGCATTGCACTTAAAAAAAAATCTTCCACTTTATTTTATTTTAATTTTTTGAGATGGAGTCTTGCTCTGTTGCCCAGGCTGGAGTGTGGTGGTGTGATCTCGGCTCAGTGCAACCTCTGCTTCCCAGGTTCAAGCGATTCTCTTGCCTCAG
5160	chr7	45916551	45916802	-	0	train	CATCAGACATAAGGTATCTGAGGAGCAAATTACAGGTCCCACTTTTGGTAGTTGTGCAGCATCGTAAGATTTTTAAAGCACACATTCTAGAGTAAAAACTGTGACTCTGTTGCTCTGGTCCTTCCTGATCCCCAGGGTCCCTGCCGTAGAGAAATGGAAGACACACTGAATCACCTGAAGTTCCTCAATGTGCTGAGTCCCAGGGGTGTACACATTCCCAACTGTGACAAGAAGGGATTTTATAAGAAAAA
13035	chrX	106035819	106036070	-	0	train	CATACTTAGGCATGTTACATACATAATTTTACCATGTATTATGCACTCACTGTGTACTAAACACCATCTGACATGTGTGGTATACACACACTGTCTCAATCAATACTCATAAGTGTCCAAGTTTACACAGTTAATAAGTTTTAGAACTGATATTACCAACAAGGTCTGTGTGAGTCTAAAGCTGGTGCTTGTAGAAACAAGAATCATGTGACTATTTCCTAAAGGTACTGTGCTCAAATGATTTCTCCTGG
12100	chr18	46089915	46090166	-	0	train	TTAATCAGAGAGGGTTGTTTGTTAGTTGGAAAAGTACTCAAGAATAAACCAGGTGAAAAGGACAATACTGCTTCCCCCCCCACCCCCCGACTATTTTCTTATGTAGTATTGGTGTATGGGGAGGAGTGCCCATTCAGCTTCATTGATGTTCAATTGTGTCTTTTGTGCAGGGTATGTCCTTGAACTTGGAACCTGACAATGTTGGTGTTGTCGTGTTTGGAAATGATAAACTAATTAAGGAAGGAGATATA
5078	chr6	42178921	42179172	+	0	train	AGCGGAGGGGTCACCATGGATGTGGGGTCACCAGGGGTGGAAGGTCACTAAAGGAGAGGGTGAGGAAGGGAGGAGAGGCCCAAAGGCCCCCGTGCTGGTCACTTCCTCCACCTGCCTCTGCCCCAGCCACAAAGTTGGCTTTTAGGGGCCCCTGGACCAGAATCTGGGCTCTGGGTTCCTCTGCTTGCTGCACCCGCAGCAGGGGCTCTGACTTCTCCTCACGTGGGCTCTGTCCCTGCCCCTGGCAAGAA
2526	chr4	71759099	71759350	-	0	train	CCTCAAAACATTAAAAATAGAATTAACATGTAATCCATCAATTCCACTTCTGGGTATATATCTTAAAAATTCAAAGTGGGATCTCAAAGAGATTTGCACTTCCATGTTCACTGCAGAATTATTCACAATAACCAAGAGGTGGAAGCAACCCATATGTCCATCAACAGATGAATGGATGAAGACAATGTGGTATATACATACAATGGAATATTATGTAGCCTGAAAAAAAAAGAAAATCCTGTCACGTGCTA
9919	chr16	2091789	2092040	-	0	train	AGAGCGAGGGCCCCGGGCGTCTACGCCAAGGACAAGGGAGTAGTTCTCCAGGAGTGCCGCGGCCTCCTGACCAGCCTGGCTCCGGGGTGCCGGAAGGGCTGGGGTGCGGCACCCACGCCACCCCTCTCCGGCAGGGCATGGTCCTGGGGCTCCTGTGCCGTGTATGACAGCGGGGGCTACGTGCAGGAGCTGGGCCTGAGCCTGGAGGAGAGCCGCGACCGGCTGCGCTTCCTGCAGCTGCACAACTGGCT
163	chr1	109606533	109606784	-	0	train	GTCTCTTTCAATTTACTGCCTCTTGATGGAACTTTTAGCACCCACTAAATCTCATTCAAAAATCTACAAAAGCTTGCAGTCAGGGCAAACCCAAGGAAAACTTATCTAACTTGATTTAAATGCGCTGGAACTCCAGCCTCAAATGGGCTTGTCCCTCAGTCCATCCCCCATTTCCTCTAGTTAATGACATATGAAGGTTTGAAGGAAAGAATCCTAAATTTAACAATATGGCTTCTAGTCCTGATTTCTAT
1896	chr2	162145332	162145583	-	0	train	AAGGAATTCATTGCTTGGCTGGTGAAAGGCCGAGGAAGGCGAGAGTAAGTCTGTACATTCTTATTTGACATTTTTTGCCTTGATGCAGAAAATTTAAGACTACAGTTATCTATATATGGATCTGGATTACAGAAGCAATTAGTAGTCTTGCAAAGTAAGGAAATAATTCCTATTGATGAAAAACAGTATATAAAAGTTAAACCCATTTTGTTTTTGGTACTAAGTATTAATAATAGAGCCAAACAGGTTAC
11534	chr17	39668518	39668769	+	0	train	ACTCGGCCCCGGGCCAGGCGGCGGTGGCTTCGGCCTACCAGCGCTTCGAGCCGCGCGCCTACCTCCGCAACAACTACGCGCCCCCTCGCGGGGACCTGTGCAACCCGAACGGCGTCGGGCCGTGGAAGCTGCGCTGCTTGGCGCAGACCTTCGCCACCGGTGAGCGGGGGAAACTGAGGCACGAGGGACAAGAGGTCGTCGGGGAGTGAAAGCAGGCGCAGGGAAATAAAAAGAAGGAAAGGGAGACAGAC
853	chr1	173904615	173904866	-	0	train	CAAACACTCTAGTCTGCTAATAAGCTAATAATTTAGTGCTGGAATGAGCATGAAATAGGTAATATGGGGAGATAGCGGGTAAGGAAGGGAGGAACAAAGGAAGGGGAAGGAAGAGTGAGAAGGAAGGAGAAGACATCATCAACCAGCTCCACAAAACCCAGGGAGCCGGTTAATCATGTGCTTTCATTAAGAGCAGAAACAGAGTTTTAGTGATATTCTGGGTCCTGAGGCAAAATTTTCTGAAGGTGTTT
8869	chr14	102082577	102082828	-	0	train	CTAACACGGTGAAACTCAGTCTCTACTAAAAATAGAAAAAAATAAACCAGGCGTGGTGGCACGGCCTGTAATCCTAGCCACTTGGGAGGCTGAGGCAGGAGAATCGCCTGAACCCAGGAGGCGGAGGTTGCAGTGAGCCAAGATCGCACCACTGCACTCCAGCCTGGGTGATGGAGCGAGACTCTATCTCAAAAAAAAAATTGTGCATGTAAAACATGAAATTATAACCTGTGCTCTTTGGATACCTAATG
524	chr1	119510210	119510461	+	0	train	AGAGGAACCTGAAGACATTGATGTTGTAACCACTGCCTCCAGCTAGACAGCTGCGATGTTAGTTCATTGGTGGTTAACTCATTTTCAGCATTGCTACCTGCTTTTGATGATATCTGATAATAGTTTTCCAGGGAAATTCATCATTGAAGCCCACCCACAATTCCTGAGTAATGACTCCACTCTCCTCCCAGTTGCCCAAGCAGGAAACTGTAGGGGTGATTTTATCTTCTCCTTTTCCTATCAGCTCAGAT
8884	chr14	102083822	102084073	-	0	train	ATCTTCAGGTTATCTTTGATTTTGGGAGTTTACATAGTACCAGTTTTGTCCTTGGAATGACTCAGTGCATTTGGTTTATATTTTTTTCAGACTTCATTAGAGGGGTGGTAGACTCGGAGGATCTCCCTCTAAACATATCCCGTGAGATGTTGCAACAAAGCAAAATTTTGAAAGTTATCAGGAAGAATTTGGTCAAAAAATGCTTAGAACTCTTTACTGAACTGGCGGAAGATAAAGAGAACTACAAGAAA
9816	chr16	173332	173583	+	0	train	AGCGGCGGGCCGGGAGCGATCTGGGTCGAGGGGCGAGATGGCGCCTTCCTCTCAGGGCAGAGGATCACGCGGGTTGCGGGAGGTGTAGCGCAGGCGGCGGCTGCGGGCCTGGGCCGCACTGACCCTCTTCTCTGCACAGCTCCTAAGCCACTGCCTGCTGGTGACCCTGGCCGCCCACCTCCCCGCCGAGTTCACCCCTGCGGTGCACGCCTCCCTGGACAAGTTCCTGGCTTCTGTGAGCACCGTGCTGA
5251	chr7	76303338	76303589	+	0	train	GCAACATAGCGAGACGCGCCCCCCCGCCCCGACCCCGCGCCATTACAAAAAAAAAGCAAACAAAAATTTTTTTAAAGATCATCGATGAAGAGAGAAAATGCGCTTTTCTACAGAGTCCCCTTCCCACCCACAGCCCCATCCCCAGATAAGCGGGGAGTTCCCTGGCGCGGTGCCAGTTTCTAGCCGCTGAGTGGGCGTGTGCGCGGCTCCAAGTGCGCCTGCGTACTGCTCACTCCCCAGCTCCGCGCCCT
9511	chr15	50263361	50263612	-	0	train	AGTCAGGATGGTCTTGATCTCCTGACCTCGTGATCCGCCTGCCTCAGCCTCCCAAAGTGCTGGGATTACAGGCATGAGCCACTGTGCCCGGCCAAATCCATCTCCTTCTTTACCAAGTCTCTTGGCCCGGACCTGAGAGGGATGGGCACAAATCGGTCAGGCCCAGTCAGGGGATTTCAGCCTGATTTCTCTATGTCACTTCTAGGGAGAGAGATGGTGGATTACATCTGCCAGTACCTGAGCACTGTGCG
235	chr1	109608932	109609183	-	0	train	AGAACAAAGGTGATGGGTCCCTGTCTTCTATCTTGTCTATTTCATGGTCAATCTGGCCTTTACATTTTCAGTCTCTCTGCAGCAGAGGACCCACGTTTTGTGGCGGTAAACTTTCAGACTAAGGATGGAAGCCAGAAACTCCAGAGAGGAAATACCTGGGACCCCTGCCCCGCTTTAGACACACCCCAGGGCTGGGTATAGTGCCACCCTCTGCATCTGGGGAGCATACTCAAAAATTCAACAGTATGTTT
9862	chr16	180843	181094	+	0	train	CTGCGAGTGGACCCGGCCAGCTTCCAGGTGAGCGGCTGCCGTGCTGGGCCCCTGTCCCCGGGAGGGCCCCGGCGGGGTGGGTGCGGGGGGCGTGCGGGGCGGGTGCAGGCGAGTGAGCCTTGAGCGCTCGCCGCAGCTCCTGGGCCACTGCCTGCTGGTAACCCTCGCCCGGCACTACCCCGGAGACTTCAGCCCCGCGCTGCAGGCGTCGCTGGACAAGTTCCTGAGCCACGTTATCTCGGCGCTGGTTT
1825	chr2	112834167	112834418	-	0	train	AAAAAGTACTCTCACAGGATTTGCAGAATGCCTATGAGACAGTGTTATGAAAAAGGAAAAAAAAGAACAGTGTAGAAAAATTGAATACTTGCTGAGTGAGCATAGGTGAATGGAAAATGTTATGGTCATCTGCATGAAAAAGCAAATCATAGTGTGACAGCATTAGGGATACAAAAAGATATAGAGAAGGTATACATGTATGGTGTAGGTGGGGCATGTACAAAAAAGATGAACAAAGTAGAAATGGGATT
5913	chr9	125237030	125237281	-	0	train	TATGAAATTACTTTTAGCTCGATAAAACCAAAAGTGTCACTTTATGCTTCAGACTGAAATGCGGGGATCTAGATGTGCTAATGCTTGTCAGTAACAACTAACAAGTTTTTCTGTATGTAACTTCTAGGTGAAAGACCCCTGACAAAAGACAATCATCTTCTGGGTACATTTGATCTGACTGGAATTCCTCCTGCTCCTCGTGGGGTCCCACAGATTGAAGTCACCTTTGAGATAGATGTGAATGGTATTCT
4116	chr6	29670645	29670896	+	0	train	GGGAAGGTGCTATTCATCTTCCACTAATCACATATTTGTTTCTTTTTGTTTTCAGGGCAATTCCTTGAAGAGCTACGTAAGTTCTCTTCTCTCTGTTATAAGCAGAGAATAAAAAGCCAGGAAAGGGAGACAGAAGCAACAAGAGGAAGAGGCGGGCTATTGAGGGATCACATTCCCAGAGGAAAGGAGGAGCTGGAGAGCCTGGGTGGAGGGAAGACTCCTCCTGGGAGGTAGAGGGCAAAGAAGCCAGC
3985	chr6	29662542	29662793	+	0	train	CCACCTTGCTGCAGCACTTGTCAATCCAGGGACCACCCACCTCACCGGCTCCCCACTCATTACCACCCTCCCCTACTCAATTACTGAGGTAAATCCTAGGCAGCATGATCATTTCTTTTTTTTCTTTTTATTTATTTTGAGACAGGATCTGTCTCTGTCACCCAGGCTGGAGTGTAGTGGCATATCTCTGCTCACTGCAGCCTCTGCCTCCCGGGCAGAAGCCATCCTCCCACCTCAGCCTACATAGTAGC
1015	chr1	173916102	173916353	-	0	train	GACCACGTCCGTGAATCTGCACTGGGTGCCTGTCTTTCTCTCCCAGGAGAAGATGGGAAGATCCAGTACCCACACACAGACCCCCTTGTGTACACGCAGGAACCATAAACCAGCTGGAGGCAGCCCCTGCCCCACCCTGTCTTATCTACAAAAAATATTACAAGAGACTTTATCTCTTGATTTGCTTCATCGAGTGTCCCAACTACCTCATTTTTTTAAAATGTGAAATTAGCTTCATTTACCTTCATTGA
12658	chr21	39348162	39348413	-	0	train	AGCGAGCCCCGTAACCGTTCGTTTTCCGCGGGTCGTCCCGGGTGAGGACGCTCAGTGCTGCTTTTGCCTTTCAGAAACCTCCTGCAAAAGTGGAAGCGAAGCCGAAAAAGGCAGCAGCGAAGGTAAGCCTCGAAACGCGCATTGGGATGCAGCGGGGCCTTAGGCTACACTGCTTCTTAATGCGGGGCTTCCATTTTGATTAGCTATTGGAGCTTTATTTATACTTTAATAATTACGGTAAATAATTTTTC
62	chr1	67687109	67687360	+	0	train	GTAGGTAGGGAAGTTATGTTTTCAGGGGTAAATGTGCTACTTTTGTCTTCTAAATTTTGCTCTTTTTTGACTGGTCTAGTCAAGTGACAGCCCGATTATTTTGCTACTCCTTAAAAGTACTATTCTGTCTCTTGGAGTATGGTTGATGGCAATTCCAGTTAACTGCTGTGCAGCTCTCATCTCATTGTGCACACAGCATGGAAATCTTTCTCAAAACTGTTTCACTCAGGTCAGGGTAACAAGTTTGGTAG
11323	chr16	28935788	28936039	+	0	train	GCACAATCTGAGCTCACTGTAACCTCTGCCTCCCAGGCTTAAGTGATTCTTGTGCTTCAGCCTCCCGAGTACCTGGGACTACAAGTGTATGCCACCACACCCGGCCAATTTTTTATATTTTTAGTAGAGACGAGGTTTCACCATGTTGGCCAGACTGGTCTCGAACTCTTGACCTCAAATGATCCGCCCACCTCGGCCTCCCAAAGTGCTGGGATTACAGGCACGAGCCACCGCGCCCGTCCGCCTCGCAA
2046	chr3	49027959	49028210	-	0	train	AATTGCACTGAGGTGGGGTGGGGTGGGAGTAGGGGTTTATTCTAATTTAGTATTCTTTCTTCCCACCATGGGGTTCAGTTACTGAGAAGACCCTGAGATTCTGTTTCTTAAAGCAGCAGCAATAGACCAGGTGTACAGTGCCTCCAGCCTACCCATGTCTCTAAGATGTGTTGGTGTGATTTGGTCTTGTGGCACTGCCAAAGGGATCGATAAGCAGAGACCCCATGCTTCAGATCAAGAGCCTGATGAAA
4325	chr6	32040141	32040392	+	0	train	TGGCACTGAGACCACAGCAAACACCCTCTCCTGGGCCGTGGTTTTTTTGCTTCACCACCCTGAGGTGCGTCCTGGGGACAAGCAAAAGGCTCCTTCCCAGCAACCTGGCCAGGGCGGTGGGCACCCTCACTCAGCTCTGAGCACTGTGCGGCTGGGGCTGTGCTTGCCTCACCGGCACTCAGGCTCACTGGGTTGCTGAGGGAGCGGCTGGAGGCTGGGCAGCTGTGGGCTGCTGGGGCAGGACTCCACCC
6979	chr11	5226066	5226317	-	0	train	ATAACAAAAGGAAATATCTCTGAGATACATTAAGTAACTTAAAAAAAAACTTTACACAGTCTGCCTAGTACATTACTATTTGGAATATATGTGTGCTTATTTGCATATTCATAATCTCCCTACTTTATTTTCTTTTATTTTTAATTGATACATAATCATTATACATATTTATGGGTTAAAGTGTAATGTTTTAATATGTGTACACATATTGACCAAATCAGGGTAATTTTGCATTTGTAATTTTAAAAAAT
13060	chrX	136490060	136490311	+	0	train	GTCACTTCTCACCCTAAAATTTGGTAACTTTGACATTTATTTGGACCTTTGCCTCTGATTATGTGTTTCTAGATACAAGGCAGTTGTGAAGCCACTTGAGCGACAGCCCTCCAATGCCATCCTGAAGACTTGTGTAAAAGCTGGCTGCGTCTGGATCGTGTCTATGATATTTGCTCTACCTGAGGCTATATTTTCAAATGTATACACTTTTCGAGATCCCAATAAAAATATGACATTTGAATCATGTACCT
11127	chr16	18398821	18399072	-	0	train	TCAGCCTCCCGAGTAGCTGGGATTACAGGCGTGCACCACCATGCCTGGCTAATTTTGTATTTTTAGTAGAGACAGGGTTTCTCCATATTGGTCAGGCTGGTCTTGAACTCCTGACCTCAGATGATCCGCCCGCCTCGGCCTCCCAAAGTGCTGGGATTACAGGCATGAGCCACCACGCCCAGCCCTGACCCATGTTTTGAACCAAATTCCAGCCACCCTTTTATCTGCAAGCATTTTGGAGGGCATCGCAA
3795	chr5	132543217	132543468	-	0	train	TTCTGCATTTGAGTTTGCTAGCTCTTGGAGCTGCCTACGTGTATGCCATCCCCACAGAAATTCCCACAAGTGCATTGGTGAAAGAGACCTTGGCACTGCTTTCTACTCATCGAACTCTGCTGATAGCCAATGAGGTAATTTTCTTTATGATTCCTACAGTCTGTAAAGTGCATAGGTAATCATTTGTGATGGTTCCTTTACTATATATAGAGATCTGTTATAAATAATAAGATTCTGAGCACATTAGTACA
9536	chr15	50264092	50264343	-	0	train	ATCCATGGTTGTGCTTTAGGGATTCTGTGAGCCTCAGAAATTATACACAAAGTTCTGTGTATATGTAAGCCTACAGTTTTAATCAGATTCTTGAATTAAGTCCTCATATGAGTCCAAAAAAGATGAATAATAACTGATTCAATGGGGATGGTGAAAAAATAGATTTTTTTGTCCTTTATCGGAACTGGGCATGGAATCTTTCTATCTGTGCTAACAGATATAACTGGTGGGTGTGTGTGTAACAATTTTTG
12990	chrX	106034286	106034537	-	0	train	GTGAGTTTGGTGCCATGTATTGACTTACATAGTAATGGTTATCAATACTCAGGGAAGAAGCAGAGTCCAATATAATACCATCATGAGAGAGAGAAGGAGAGAATCATAAGCTTGATATGGTGATTGCCATGTGTTCCCTTCCTCTTTTCCCACAGATGGGTTGACTTGTTTGTTCCAAAGTTTTCCATTTCTGCCACATATGACCTTGGAGCCACACTTTTGAAGATGGGCATTCAGCATGCCTATTCTGA
13116	chrX	147913163	147913414	+	0	train	TTTCCTTTGAATTTGTGGTGTTGCAGTGGACTGAATTGTTGAGGCTTTATATAGGCATTCATGGGTTTACTGTGCTTTTTAAAGTTACACCATTGCAGATCAACTAACACCTTTCAGTTTTAAAAGGAAGATTTACAAATTTGATGTAGCAGTAGTGCGTTTGTTGGTATGTAGGTGCTGTATAAATTCATCTATAAATTCTCATTTCCTTTTGAATGTCTATAACCTCTTTCAATAATATCCCACCTTAC
10476	chr16	2111322	2111573	-	0	train	CGGTGCGGCGGCCCAGGCGGATGTGCGCGTCTTTGAGGAGCTCCGCGGACTCAGCGTGGACATGAGCCTGGCCGTGGAGCAGGGCGCCCCCGTGGTGGTCAGCGCCGCGGTGCAGACGGGCGACAACATCACGTGGACCTTCGACATGGGGGACGGCACCGTGCTGTCGGGCCCGGAGGCAACAGTGGAGCATGTGTACCTGCGGGCACAGAACTGCACAGTGACCGTGGGTGCGGCCAGCCCCGCCGGCC
1264	chr1	192810618	192810869	+	0	train	AAGAAAATCAGCCTAAACAAATTAAAGTGGCAGTTGCTTATAGTTAAGGTTGAGTCAGTTTTTCCATTGCATACAATGTTTTCAAGAGGCTTAGTTCCCAGAGAATTTATGGCTCCCCTATAAATATTTACTTTGCATTGACAGCAAAGTACTTATATTTTTGCAGCAGAGTGCCAACATAAGCCTTTTGCCTACTGGTATCTTAGTCTTTAAAAAGCTAAATTTTGAGAATTACAGGTTTGAGCGAAATC
14970	chrX	153866541	153866792	-	0	train	GGCAGGAGCAGATTGTCAGCGACCCCTTCCTGGTGGTGTCCAACACGTCCACCTTCGTGCCCTATGAGATCAAAGTCCAGGCCGTCAACAGCCAGGGCAAGGGACCAGAGCCCCAGGTCACTATCGGCTACTCTGGAGAGGACTGTGAGTATCCGGCAGGGCCCCTCGCCCACGGTTGAGCTCGCCTGCTTCCACCCCAGCCCATGTCCACACCTCTCAGCATTGGTGTGCGCCTTTCCTAAGGATGAGGA
2893	chr4	71776623	71776874	-	0	train	AACTTCAGTAGTCATTAAGACGTGTATTTTGTTAGGAAACAGTGAAAGAAAATCACCCATGTACAGTGTTTTCTGGTTTAAAATGTACTGTTTTTTAATTGGCAAATTATAATTATATACATTTCTGGGGCACAATGGAATAGTTTGATATCTATATATAATGGGGAATGATAAAATCATCAATGAACTCCTCTACCATCTCACTGCTTATCATTTTTTCTGAGGCATTTGAAATTTACTCAAGTTATTTT
863	chr1	173905086	173905337	-	0	train	ATGAATGTCAATGACCTTTAAGACAATAGCAAGAGTAGAGGTATTGAGGTCAGAACAAGGGATTTTACAAGAGTGCTGTATTAATGGTTTTGGAAGTTAAGATGACACTGCTCACACCCTCTTTCACATGGATTTTTGGAAGAAAGAACACTTAGGAAGACTGCAAGGGAAATTGAGTCCTCAGGGTTTTAACTCTCATTGAATATCCTCTGGTAAGGACTCCAGTTAGAAGTGGTCAACTCAGACCTCCT
4178	chr6	30490649	30490900	+	0	train	AGCTGTCACCTGAGGTACAGGAGATCCTATACCACAGAGTGACTCTCTTAAAGGGCCAGACCTCTCTCAGGGGCAATTAAGGAATCTAGTCTCGCTGGAGATTCCATCCTTCAGATGAACTGATGAGCAGTTCTCTTTGACTCCCAGTATTAGGAATCACGGGGGAGTTTCTCTCGTGCCTGATTCTCAGCCCCACACCAAGAGTTTTTGGAGGTCTGACTCCAGCTTTTCTCAGTCACTCAGCATCCACA
6720	chr11	4385310	4385561	-	0	train	CAAACAAAAATATGAGGCTGGCACCTACCCCCAGACTCCCCTCCACCTTCAGGTGCCTCCATGCCAAGTTGGGATTTTCCTGGACTATGAGGCTGGCATGGTCTCCTTCTACAACATCACTGACCATGGCTCCCTCATCTACTCCTTCTCTGAATGTGCCTTTACAGGACCTCTGCGGCCCTTCTTCAGTCCTGGTTTCAATGATGGAGGAAAAAACACAGCCCCTCTAACCCTCTGTCCACTGAATATTG
16239	chrX	155998670	155998921	+	0	train	CCCACAACATGAACACCGTCCAGAAGCAAGGGCAGCCTCTGCAGGTGGGGCGGGGGTTGGAAAACATTTATCAAACGGTTGAGTTGGGTGCAGGGGATGCAACATGATCAAACAGGGTCTTGCCTCCAGGAGCCTCATGTAGGGACAACGCACAGTGATGACCTTCAACTGCAGTGGGGCAGCAAGGCTGTGGGAGGGGTGTTTTGGGCAGAGCAGGCGACGTGGGTACCTCTTCACCAAGACAGCAGGAA
12254	chr18	46096393	46096644	-	0	train	TTTTGCCCCTTGTAGGAAGCTGTTTATCTTAGAATTCCTGCAACTCCATTAAAATACATCTAAAGTCAAAATAATAAAAATTGGGTACCTTTAAAGGGAAAGACACTTACATTTTGTTAACAGTTGCTTTTTTTTTTTTTTTTTGAGACGGAGTTTTGCTTTTGTCGCCCAGGCTGGAGTGCAGTGGCGCGATCTTGGCTCACCACAACCTCTGCCTCCCAGGTTCAAGCGATTCTCCTGCCTCAGCTTCC
15024	chrX	153868628	153868879	-	0	train	GTGCAAGGCCTTCGGAGCGCCTGTGCCCAGTGTTCAGTGGTGAGTGTCTCGTCCTGGTAGTGGTGAGTGTCGTGTCCCAGTGGCCAGGGAGCCAGGGAGGGCAGGGAGCCCAGGCCAGCCAGTCAGAGCCAGGCCCGCCCTGCTCCCTCCAGGCTGGACGAGGATGGGACAACAGTGCTTCAGGACGAACGCTTCTTCCCCTATGCCAATGGGACCCTGGGCATTCGAGACCTCCAGGCCAATGACACCGG
9339	chr15	50255249	50255500	-	0	train	CTTCTTTACATAAATAGCTTTCTTTTTATGGAGTAACTCCCAGGCACTTAATATCTGTGGGATCTGGTCCACACCACTCTCCATGACAGTGTCTGCCAGGCATTTTTAGCAGGATTCTATCAGCTGTAGGTCACTTGAAGATTTCAAGAAGCAAAGACATCAAAGACTGGGTGAATGAATGATAAGGAACCATAAAGACTCCTGTGAAGTTCCTGAGGTGGCCCCGAAGATTTCCACAATCATGGCCAGAG
1369	chr1	225838172	225838423	+	0	train	TATACATCTACCTTGTCAGTTTAGATGACTGTACTGGACTCCAGTATACTGTCAAACTATACTTGATTAATCCTGTATTGCTGGATACGTGGGGCTTTCTCCCTACCCTCCAGATTTTAAATTATTGAACAAGTATTTATGGAGGCCTGCTGTGAGCCAGGAGCTGTCCTGAGCCCTGGAAACCCAGCAGTGGCTGTACAGACCTGGCCCAGCTGTCAGGGGGCACCTCTAAGGAAACCGGGAGGCAATAA
9206	chr15	50251851	50252102	-	0	train	CTTTTTTGTTTCTTACCCAGCAGCAGGGCAGGGTGGAGAATTTCCCTGGTGGGGAAAGAGCTTTCTTGCTGAAATCGCTTTTGTTTTTTTGTCCCTGACAGAGGCGCTGTGGGGCAGCTCTGTGGTAGTGATGGGCTGCAGGACTTGTGTGAGGTTCTGGAAGCCTCCAAGCATTCCTGACTTTCCCAACTGACTGCGCCTCAACTCTCTAACTCACTCTCTCTCTCTTTTCTTTTGTTTTCTTTCTTTCT
2175	chr4	69958777	69959028	-	0	train	TGTACACATTTTCAAAATTTCCTCCTTTTAATCTTTACAGATGGTAACATATCTTTATATATGATATATCTTTAGCAGAAAGTTGAGAAGGTTAAACATGAGGACCAGCAGCAAGGAGAGGTAATTTGTTAATGATAAGTATATGTTTAAAATTATTATAAAGTATAATACATACAAAAATATTTATAATGTGTATGTTGATTCTAAAGAATGATAATAAAATAAATGCCATATACCCACCAACCACTTTA
4379	chr6	32182557	32182808	-	0	train	GTGAGCACGTTCTGGAAGTCTGACCCTTAGGGAAAGAGGGAGTCAAGCCCATGGCCACTGGGATCACTCACAAGTGTAACTCTCCACCTCAAAACCCTTCCAACTCCCAGAGCCTGTGCCTCTGGAGGAGGTCCAATTGGTGGTGGAGCCAGAAGGTGGAGCAGTAGCTCCTGGTGGAACCGTAACCCTGACCTGTGAAGTCCCTGCCCAGCCCTCTCCTCAAATCCACTGGATGAAGGATGTGAGTGACC
12895	chr22	20785312	20785563	+	0	train	TCCTGCCTTGGCCTTCCAAAGTGCTGGGATTAACAGGCGTGAGCCGCTGTGCCTGGCCCATTTGACTTTTAATTGAGATCTTACTTGGTGCAAGGTATGAGCTAGGTAAAAGAGTGAAGAAGATCAAGCCTTCCTGCCCATCCAGCTGGGATTGCACCTTAAATCTCTTTATCCCCTGCAAAGTGCCAGACTAACTCCACAGGCACTACTGTTGCTATCCGCCCCCTTAGGGATTGAGTAAGTTGAGGCAA
9359	chr15	50256146	50256397	-	0	train	AAAAATTGCCATGTTGCTCTGAGATGAGAACCAGGAAGTTCACTAGGTCCTCTTGCTAGCTAAGTGACTCCCTCTCTCACTGCAGCCTTGCAGCAAGATTAAAATCTGAACCTCGGAAGAAAGACAAATTGCAGCATGTTGTGGACATTTTGGATTGTCAGCAAATAGTTACAACTTCAGATATCATATTTGGGAGAACATGCTAAATGTCTGCGGTACATGATCTTGCTTTTCATCTGATTATCTCAGGA
3309	chr4	121821901	121822152	-	0	train	AGATCTTAAATGCTTATTTTAATCTGATCCCATAAATGTGTTCTTGGTTGGGAGAATAGATTTCAGTTTTATTTAACTCACTTATATATCACTTAGTGCTATTTGAAGATGCAGTCTTAAATATTGTGATTACATAACGTCTCTTCCAGTTTTATAATTTTGTTTCTATAAAACCAGTCTGGATTAGCATCTGATACCAGAAAGTGACACTTTTCCTCCTACAAGTAAATAGTTGTACCATGGCTGTTTAC
1661	chr2	79159025	79159276	-	0	train	AAGAGGAGGAAGGCTCCTGTGTGTCATGTGAGGTAATGACGTGGTGTCTAATGAACCTGCCTGCAGTTCTTGCATCATCTCTCCTTCCTTCAGGTTAACTTGCAGTGGGAGGCTCCATGGTGGTCCACTAACAGTGGAATGAGATGGCTTCCATTTAGTCAGTGGACTCTAATATACACTGGTGGGAAAGTGGACTCTAATATACACTGGAGGGTCAGTAATGAGATGTGGGGAGGGACAATGATTGGAGG
1212	chr1	192809265	192809516	+	0	train	CGTTAGGAAACTAGCCTGAGCCTATGCAGGGAAAAAAAATCGAAAAGGTCAATTTGTTAAGTAAGGTTAAATCTGGGTGATGCTCGGGTACAGTTTAAGAACCGAGGGAGACAGTTGATATGAGGGCGGTGGTTGATGCGCTAAGAAATTGCGGGTTGGCTTTTTGTCCTCCTGCATTCAAAATGACATCAGAATCCTGCGGCTGAAGCGCGTCCCCAGCATTCATACGTTGCATGATGAGTTCTCATCAG
8733	chr14	75281099	75281350	+	0	train	GTTCCCAGCATCATCCAGGCCCAGTGGCTCTGAGACAGCCCGCTCCGTGCCAGACATGGACCTATCTGGGTCCTTCTATGCAGCAGACTGGGAGCCTCTGCACAGTGGCTCCCTGGGGATGGGGCCCATGGCCACAGAGCTGGAGCCCCTGTGCACTCCGGTGGTCACCTGTACTCCCAGCTGCACTGCTTACACGTCTTCCTTCGTCTTCACCTACCCCGAGGCTGACTCCTTCCCCAGCTGTGCAGCTG
2377	chr4	71753263	71753514	-	0	train	TCCTCCCCCCAACTTTCTTTTTTTTTTTTTGCCTCCTTTGAGTCTCTTCTTGCCTCATTTGCACCCCCCGGCAGCTCAAACCTACTTGTTTCAACAATGCTTTTATGAACTAACTGGTCCCCCTTACCCTTTTGTCCCTTCCCAGCATCCACTCTTGCCAAGCCCAACTCTAGATTAATTTGAGTCTCCTTACCTCCCATTGCCCCCCGCTGGGCTGGCAAGCATTTCCTGAAGAAAGTCATAAAACCAGG
430	chr1	109660421	109660672	+	0	train	ACAGTGGTGTTTAGCTCAGGTACCAGGTGGGGAGGTTTAGACTTTCTGCTTTAAAAGGAATGATTAGAGCCTGGTCTGGCGTTTCTTTTGCTGGTCCAACACACCTTGACCACTTTCATCCAGGTTTTGCCAGGTCCTTGGGTGAGATCTGGGCTCTCTTCCAGGCTGCACAGACATTTTCAGAGGTCCCCTCTGTGTGTGCAAACCTAGGCAAGCCAGGTGCCTCCCTGTGAAACAGGAGAATGTTGTGT
13538	chrX	147927279	147927530	+	0	train	TGACAGATCGGGACCAATGAGATACCATTGCTAGGATGACCTCACTGTCTTACACACCGATTCGTTGAGGTAATTTCTCTGGCTAGAGTAGTTGAGAAAGATGGAAGCAAGAGGGAAAGGTATGTTAAGTTGTATTCAGTCACTGTATTGTGTGATCATTAACTTTGGTTGTAAAAGGGAGACAAGTTCGAGCTTTTACTCTACTTGTCAGCCCTAGACTTTGTACCTCTCTTGGCCTCATGTCTTTATCC
2774	chr4	71771513	71771764	-	0	train	TGTCACAATATCCAGCCTTGTAAACTAATCAATACATTTCAGAAAATGTACAAGATGTGAAGGTTCTCAGACATTACAGAGGTTTGTTGCAAGTGGTTTTTTTGGTTCCACTAACAATTAGCTTTATCAACTCTACTGCCTTGTCCTAGTTGCCAGAATAACATCAGATTCAGGATTCAGGTATGTGGGTTATTTTCCACCTTCTGTGTGGTTTTGAGGGTGTTTGAGTAGGTCTCTTGTTAAAAAGTGCT
16185	chrX	154328736	154328987	+	0	train	CGGCAGGGGCCCTCTGAGCCTGCACAAGCGGTGCCTGATCTCTCGAAGGACTCCTGGGACCCAGCACACAGTTGCTCTCACACCTGAGGTTGATGACAGGGAAAGGGTGCAGGGGGGGATCCACCAGGGCAGGAACCATGGGCCAGGTGGAGTTGAAAGAAGTCCTGCCCCAGCCTCCCCTTGGCCCCACAAGGGGCATACTGAGAAGTGCCTCCCACGCAGGCTGTGTTCCTGCCTAAGGGATGCCTGCT
14463	chrX	149491220	149491471	-	0	train	CAGGGGCTGGTCTGCGAAGACACGGCTCTCTTCTCACCCCTCTTTGGGGTGGAAGATTAATTTTTGTCCTTAGCATTTGTGAACCAGGTTGGGAATGAGAGTCAGCCCAGGAGGGGCCGGTGGCTCATTTACTTCAGGGCATGATCTGGCTGTTCCCAAAGTGCCTCTTGGGTTCAGGGACTGTGAAGGACGTGCTGCTCTCTGGTATCTCCTTTGCTCTTCTCCTGCCTGCTGACAGTTTGTTAGAAATG
9098	chr15	50247401	50247652	-	0	train	CCATGTTCTTCATGGAAGAAGCGGGACTTAACTAAGAACTCTGGCCACTGGCATTGTTCTCCCACTTCTACAAGAGTCCACATGCAAGACAAAGCCATTGATAAAAGTAGTGGGGGAAGCTGAAGACACTGTCCTTTCTGCCTATGTGTGTCTTTGTCCTTCCACCTTTCCTCTAGGGAATTCCTTCTTGGTTAAACAAACACAAACCAAAATGAAGTGGTGGAAGCGGTAACTATGATTTTTTTTAACAT
10038	chr16	2095126	2095377	-	0	train	GCAGTCGCATGATCTCGGCTCACTGCAAAGTCTGCCTCCCACGTTCAAGCAGTTATCTGCCTCAGCCTCCCAAGTAGCTAAGATTACAGGCGCCCGCCGCCACAGCCGGCTAATTTTTTGTGTGTGTGTTTTAGTAGAGAGGAGGTTTCACCATCTTAGCCAGGCTGGTCTTGAACTCCTGACCTCGTGATCCACCCATCTCAGCCTCCCAAAATGCTGAGATTACAGGCGTGAGCCACCACGCCTGACCA
11789	chr18	31595258	31595509	+	0	train	AGTATACAGACCTTCGAGGGTTGTTTTGGTTTTGGTTTTTGCTTTTGGCATTCCAGGAAATGCACAGTTTTACTCAGTGTACCACAGAAATGTCCTAAGGAAGGTGATGAATGACCAAAGGTTCCCTTTCCTATTATACAAGAAAAAATTCACAACACTCTGAGAAGCAAATTTCTTTTTGACTTTGATGAAAATCCACTTAGTAACATGACTTGAACTTACATGAAACTACTCATAGTCTATTCATTCCA
8309	chr12	57233593	57233844	+	0	train	TGCAGGTTCTGAAGAATGCTCGGGCCATGGCAGATGCCCTGCTAGAGCGAGGCTACTCACTGGTATCAGGTAAGCCAGCAGGTGATGGGTGAGGGCCTCTGTAGCTTCAGGCAGAGGCCCAGGACTCACCACTCCCCATTTCTTACCCACCTTAGGTGGTACTGACAACCACCTGGTGCTGGTGGACCTGCGGCCCAAGGGCCTGGATGGAGCTCGGGCTGAGCGGGTGCTAGAGCTTGTATCCATCACTG
2345	chr4	71751963	71752214	-	0	train	ACCTAGTAACAGCCTATTTAAATGAAAAAGAGAAGAATAAGAATACAGTGAGAACATCAGAGTTCATTACCTTCACATAGTGAAGGTAAATTGTATTTCATATATCTGTATGTGTGTTTATGTAGATACATACATATGCATATATAGGCAGGTAGATTGATAAAAATAAATACTTTAGAAAAATGGGGTCATATCATACCTTCTATTAATATGTTAATTATAAAAATATTATTTAACTTTGTTTTAATATG
1129	chr1	186677265	186677516	-	0	train	TAAAAACCTTAAAGGAATTTTCCATTTACTTCACTGGTCTAGTAAAATTATACACACACACAGACATGCACACACATATATAAACATTCACACACATACATATGTACAGGTATTGTTATTTGTAATTTGACCCTTGTATTTTTTAGTTTAAAATGTTAGTACTGCAAAATGTTATGTCCTCAAAAACACATTGTACCATGATTATGCCGCTTTCAATATTGTAAAGTGAGGTTTTTGCCGCATTATTATTT
6616	chr10	47351588	47351839	+	0	train	CCATTGTCCGCACATGCAGGGCTCTGTGCACAGTGCGTGACAATGGCTTTTAGATTTGTTCTCACGTTTAAGTTTTGACCGGTCAAGTCCTTTCCTCTTTCTCAACCTGTTCCATCCACTCTCTGTGACCCTGGGGTTGCTGAACACCTCTGTAGAACATTCATATTAGGTTGGTGCAAAAGTACTTTCAATGGCAAAACCCGCAATTACTTTTGCACCCACCTCACAGGAAGCCAGTTTGAAAGCCAACC
12135	chr18	46090995	46091246	-	0	train	TCTAATGTAAGTCTACTTTGTATTGGGTTGAGCCTCATCTGTCAGGGTATGAGAAATAAGTTGTAGGAGTTAGGGAAAGCAGAGGATGACATCTGGATAAACTAGAGAGAGTGGAACTGTCTAGTAGGCATGCAGATATGGGAAAAGATAGTCTTGATACTCTTGGTAATGTCTACTACGGAAAAAGAGCCACTTGATGTGGAAGAATAATTACAGAGCCCAAAGTTATCATTAGCTGTAAGACATTGGCC
12510	chr19	44930604	44930855	+	0	train	GCCTCCTGAGTAGCTGGGATTACAGGCGTGCACCACCACGGGTGGCTAATTTTTGTAATTTAGTAAAGACGGGGTTTTGCCATGTTGGCCAGGCTGGTCTCGAACTCCTGACCTCCAGTGATCTGCCTGCCTCGGCTTCCCAAAGTGTTGGGATTACAGGCGTGAGCCAAATGCCCAGCCAAGGGTAAAGTGTTTAGACTTCAACGTGCTTTGGTCCACCTGGGAAACTGAGGCACAGAAGTTGGCCCACC
16260	chrX	156000311	156000562	+	0	train	GAGTGTTCAGGAAAGGTGAGGGCAGGGCAGCCCACAGCTTCACAGTGGCCCAGGGAAGCAGGGCAGGCAGGCTATGAGGCCTAGTAGGCATCTGGGCCAGACTTTGACACTGAGGCCATGGAATGGGTGGGGCTCTGAGAACAGACCAAAGTGATCATGGGCTGAAGGCTATGTCCACAGATCCAAGGCGGGATAGGCTGTGCTGGGCAGTGATGTCAGCCAGGCTCCCCAGCGGGACTGGGGGTGTCAGG
13681	chrX	147931656	147931907	+	0	train	GACAGTGGTTATCCACAAGTTTGGGATGTCAGTGTAACTTAGTTTCGTCATCACTGGATATTTACAAGTGCTCATCATAATTGTGGATCCAGATCACAAGGTCTTACGGACTCTGGTCTCATGTTAGCTTTCAGATGCTTACTAAGTTTTTAATTTTTGACATTTTTTGATTATTCAGTTTAACTACTGTACCAACAATTTTTGATTCCTTAGGTTTTGAGGGAGGTGTCATTTTAATGTCATTTCAACAG
1268	chr1	192811241	192811492	+	0	train	AAAAAGTCCCTCCACGTTGTAGCTTTCAGTTATGTTAAAGTTCTCCTGTGACTTAGCTAGTAAAGCTAATCACACATAATTTTTATTTTTTGTTTTCAAATACTAAATTTTAATCTTTAACTCTGAATACCAAATAAACAACTTTTTTGTTTTATTTCAGATAAACATAGATTTTCAAACCAAAACTCTGATTGCCCAGAATATACAAGAAGCTACAAGTGGCTGCTTTACAACTGCCCAGAAAAGGGTAT
15344	chrX	154299993	154300244	+	0	train	ATCTACTACTTTTAGTATTTTTTTTTTTTTTTTTTGTCTTGAGACGGAGTCTTGCTCTGTCGCCCAGGCTGGAGGGCAGTGGCGCAATGTTGGCTCACTGCAACCTCTGCTGTCCGGGTTCAAGCGATTCTCGTGCCTTAGCCTCCCGAGTAGCTGGGACTACAGGCACACGCCACCACTGCCAGCTAATTTTTGTATTTTTAGTAGAGACGGGGTTTCACCATGTTGGCCAGGCTGATCTCGGTCTCTTG
6684	chr10	47354725	47354976	+	0	train	CAGGCCCCGAACCCAGCCAGGCCATCCCAGGCCATCAGTTGGACGAGGCTCCTTAACACATCACTAGCCCCAGTGGGGAGACAGGGGCCCAACGAGGTCACACAGTGGGACAAATCTAAATGGCCTGGGAGAAATACAGGCTCCCCTGCTCCCAATGCAGCTCCACAAGCCTGAATATTTCAATACACTAAGTGCCCTGTTAAGACACCTGAACTAAGACAGCTGGTAGGTGTCATAAGATTGTCACAGAA
11029	chr16	2129744	2129995	-	0	train	TGGGAATGTGGGGCACCCGAGCTCCCACTGCAGAGGCGACTGTGGAGACAGAGAGCACCTGCAGGTCATCCATGCAGTATCGGCTTGCATCCAGATCATACAGGGAACACTATGATTCAACAACAGACAGGGACCCCGTTTAAACATGGACAAGGGGTCACTCACGCCTGGAATCCCAGCAGTTTGGGAGGCCAGGGTGGGTGGATCGCTTGAGCCCAGGAGTTTGACACCAGCCTGGGCAACAGGGTGAG
13718	chrX	147932503	147932754	+	0	train	ATGGTGCTAATATTCAGCAAGCTAGAAAAGTACCTGGGGTCACTGCTATTGATCTAGATGAAGATACCTGCACATTTCATATTTATGGAGAGGTAAATATTTTACTGCATAGTTTTTTTTTCCCCAAACAAGTATTTCAGCTGGCTAATCTTTTGTCTTAAAATGTTTCCCCTTTTATTAGGATCAGGATGCAGTGAAAAAAGCTAGAAGCTTTCTCGAATTTGCTGAAGATGTAATACAAGTTCCAAGGA
3125	chr4	73998094	73998345	-	0	train	TCCCCAGCTGGTCCTGCCGCTGCTGTGTTGAGAGAGCTGCGTTGCGTTTGTTTACAGACCACGCAAGGAGTTCATCCCAAAATGATCAGTAATCTGCAAGTGTTCGCCATAGGCCCACAGTGCTCCAAGGTGGAAGTGGTGTAAGTTCTGTGCTGCTGTGTCCGCTGTGACCTTGGCAAGAGAGAAATCCCGCAGCCTGGGTCTTCAACCTTGGTATCTCATGAGTGTATCTTCTTTTTCTTTCCTTCAGA
1942	chr2	162147414	162147665	-	0	train	CTAAGGAAGATCTTTCTAAACTACCTATTGAAATACTCTAGATGCCTGCCTTACTGTTTTATATGGTCTTGTATTTTGTAGTGAAGATGCTTCTCAAGTGAGTCTACTCTTGAGGAGAGATTTATGTTGTACCAATCACTGTTCTTCACAGATCATTCTCAGCTTCCCAGGCAGACCCACTCAGTGATCCTGATCAGATGAACGAGGACAAGCGCCATTCACAGGGCACATTCACCAGTGACTACAGCAAG
12257	chr18	46097208	46097459	-	0	train	TTTGAGTAATTTTTAGTGATGGAGAATTCAAGTAAAAGAGAACAAGGTTGGAAAATGGCTGTAAGGATTAATACCCTTCGTTATTTGTTTCCCTTGAAATTAACATGATTTCAGTGTAAACGCATTGACACTGACACTTTTTTTTTGTTCTAAAAGTAATCTCAATTTGAAAGGTGAAATAAACATGTTGGCTGTTTTGCTGCCTTAACAACCGCTGGTAGCATCTTGGGTTTCTGTTAACAAAAAAGCAT
12564	chr19	50876791	50877042	+	0	train	CAGATGCCTGGGTCTGAGGGAAGTGGGGCCAAAGAACCAGGTGGGGTCCGGCCACAGCCCAGTTTTTCTCTGACCCATAGTCTTGCGCCCCAGGAGTCTTCAGTGTGTGAGCCTCCATCTCCTGTCCAATGACATGTGTGCTAGAGCTTACTCTGAGAAGGTGACAGAGTTCATGTTGTGTGCTGGGCTCTGGACAGGTGGTAAAGACACTTGTGGGGTGAGTCATCCCTACTCCCAACATCTGGAGGGGA
7822	chr11	118341347	118341598	-	0	train	TGAAGCATTTGATGTGCATGTGTCTCCCTGAGGATACATGCATATACTTCCAACCTGCTTCAGAAGTGTGAGCAACAGCTGTTTCAGGGTTTTCTAGAAGATGCTCTTGTATCCAAATTTGGGACTTAGTGTGAAGCCCAGGATGAATGAATGTAGCCTCCACTTTTCCCTGTGTTTGGAGGCATAGACTGTAGTGCCTCTGTGGCCTGTGGAGATGTGCTCAGGCTGTCAGAGGAACTCACTGCATCTGG
12576	chr19	50876962	50877213	+	0	train	CATGTTGTGTGCTGGGCTCTGGACAGGTGGTAAAGACACTTGTGGGGTGAGTCATCCCTACTCCCAACATCTGGAGGGGAAAGGTGAGTGAAGACCCTAATTCTGGGCTGCAATCTGAAAGCTAACCAGACATCTGCCTCCCCTGCTCCCCAGCTATAGCCACGCCCCCTCCCCATGCCTCATCTGCCGCCCTCCTTCCCCCTTCCCTGACTCCCTCAACACAAGAGGTGATTCTCACAGCATAATTCACC
14188	chrX	147947681	147947932	+	0	train	CCTGGGCGACAGAGCGAGACTGTCTGGGAAAAAAAAAAAAAAAGATACAAATCAAAGTACTGAATCCTTGGTAACGAGACATTTAAAACACATGCACATACCCACTACTTAAACATACTTTGAAATTACAACCATTTGGGGATGTTTTTAGCATTTGTGCTTGAAGTAGATCAATATTTGTAGTTGTTTTAGTTCCATTTGTCACTGTTAACTTTCATTTGTACCTCTGGAATTAGCAGTGCTGTATTCAG
3415	chr4	122454600	122454851	-	0	train	GGGTGGTAGAATTCATGGAAATCTAAGTTTGAAACCAAAAGTAATGATAAACTCTATTCATTTGTTCATTTAACCCTCATTGCACATTTACAAAAGATTTTAGAAACTAATAAAAATATTTGATTCCAAGGATGCTATGTTAATGCTATAATGAGAAAGAAATGAAATCTAATTCTGGCTCTACCTACTTATGTGGTCAAATTCTGAGATTTAGTGTGCTTATTTATAAAGTGGAGATGATACTTCACTGC
14098	chrX	147945033	147945284	+	0	train	GTGGAGGAGGCTTCAAAGGTATGGAGATCTTCATTAAGAAATCAAAGTGAATTGTAACAGCTGTCTTGAAGTTCCATGAGAAATCCTATTGATGCAATGAACTGTTACCAAGATCCCATCTCTCCCGTTTTGTGCTGATACCATAGGAAAGGATCAGCCTTCCACTTGTGTAGAAAGAAAGAATATTAGGCAGCCTTCCTTATGGTTCATACAGATATATGAATAAATGCTGATAACTAATTCAGCATCTT
13434	chrX	147923958	147924209	+	0	train	CCTGTTACTGATAGCAGATTGTTTTAGGTGCTTTGTTTTTAGGTGCTTCCTAAACCTTCCATAAAGCATAAAGGTGTTTTCATATTTCACATTTTTTAATAAGGAATTTGAATTTTTATCTGTATACACTTAAAATCATTATTTTTTCCTTGGAACCATTGAGTTATATATTATTATATGCTTTAAAATCAAATTTAGCAATGAATGTATAACATCTATTAGGTGTGTATAATAACTCAGAGGAGGGTTCA
3432	chr4	122454970	122455221	-	0	train	ATCCAAGCCCAGAAAATAATAGGATTTAAGGGGACACAGATGCAATCCCATTGACTCAAATTCTATTAATTCAAGAGAAATCTGCTTCTAACTACCCTTCTGAAAGATGTAAAGGAGACAGCTTACAGATGTTACTCTAGTTTAATCAGAGCCACATAATGCAACTCCAGCAACATAAAGATACTAGATGCTGTTTTCTGAAGAAAATTTCTCCACATTGTTCATGCCAAAAACTTAAACCCGAATTTGTA
10567	chr16	2114317	2114568	-	0	train	TCCCCAGTGGCTGGGCTGCGGGTCATCTACCCTGCCCCCCGCGACGGCCGCCTCTACGTGCCCACCAACGGCTCAGCCTTGGTGCTCCAGGTGGACTCTGGTGCCAACGCCACGGCCACGGCTCGCTGGCCTGGGGGCAGTGTCAGCGCCCGCTTTGAGAATGTCTGCCCTGCCCTGGTGGCCACCTTCGTGCCCGGCTGCCCCTGGGAGACCAACGATACCCTGTTCTCAGTGGTAGCACTGCCGTGGCT
1367	chr1	225838100	225838351	+	0	train	AATGGATATGACTTGATCATTTGCTTTTACCTCCCAGCAATATCTTCTGATCAGTTTTCTATGTTAAATAAATATACATCTACCTTGTCAGTTTAGATGACTGTACTGGACTCCAGTATACTGTCAAACTATACTTGATTAATCCTGTATTGCTGGATACGTGGGGCTTTCTCCCTACCCTCCAGATTTTAAATTATTGAACAAGTATTTATGGAGGCCTGCTGTGAGCCAGGAGCTGTCCTGAGCCCTGG
1156	chr1	186678334	186678585	-	0	train	TTAAAGATGTATTTCAAGTGGCCATTAGACTATAAAGTGTAGTTGTTTAAAAATAGATTTTTTTTATTTTGGAGTTACATTCAACCTCAGGTGCCACTTTCCACATTTTACAATAAAAATAATGGTTGATTTACTTAACAAATGAGAATAAATAAAACATTTTTTTCTTTGAAAATTTCAGCCAGATCACATTTGATTGACAGTCCACCAACTTACAATGCTGACTATGGCTACAAAAGCTGGGAAGCCTT
6814	chr11	5225608	5225859	-	0	train	ATATTGCTAATAGCAGCTACAATCCAGCTACCATTCTGCTTTTATTTTATGGTTGGGATAAGGCTGGATTATTCTGAGTCCAAGCTAGGCCCTTTTGCTAATCATGTTCATACCTCTTATCTTCCTCCCACAGCTCCTGGGCAACGTGCTGGTCTGTGTGCTGGCCCATCACTTTGGCAAAGAATTCACCCCACCAGTGCAGGCTGCCTATCAGAAAGTGGTGGCTGGTGTGGCTAATGCCCTGGCCCACA
3100	chr4	73997901	73998152	-	0	train	AGCCTGGGTCTTCAACCTTGGTATCTCATGAGTGTATCTTCTTTTTCTTTCCTTCAGAGCCTCCCTGAAGAACGGGAAGGAAATTTGTCTTGATCCAGAAGCCCCTTTTCTAAAGAAAGTCATCCAGAAAATTTTGGACGGGTACTTGTCACTTTGATCTTTGTGGTTTCTAAATCTGATCTAGGGAGACCATAGACTTCACAAGGTCTTTATTCTCTGTACGATTTAAGTAACACTTTTCATGTTTAGAA
2361	chr4	71752649	71752900	-	0	train	ATGAAAGAAGACTGGACTTCCAATTCAGCAGCGATTTGTATGTTTATTTTTATGATCTCGAAGAGGCATGTTTCACTTTCTGATCTCAAATTGACTATTCTATACCACAGGTATAGAATTTTCTTGAGACAGGCAAGTATTTCTATTTTCATTTTTATTGTAAAAGATCTGAAATGGCTATTATTTTGCATTAGAAATTTGTATAAAATAAATACATGTAGTAAGACCTTACATTTAAATGGTTTTTCAGA
136	chr1	109605509	109605760	-	0	train	CAGTGAGCTTGGTCTCAAATTAGACATCTAAGTATCACTTGGACATCACAAAGCTCATAAGAGGAATTGAGTGCAAAGAGATAAGGGACCATCAACTAGGCAAAGCAAAGGAGTTACACTTAGTACTCTCCCAAATTGCCTAAGGAAGGAGATGAAAATGACAGAACAGAGAAAATAACATATGATATGAATCTTCATTGCAACATAATAGAAGGGTTGAGCTAGTAACCCCACTTAGGAGGCTAAAAATG
12667	chr21	39348395	39348646	-	0	train	CGGGGCCGGGCGGGCCCGCGAGTCCTGGGACTGCGGCCCGCCTCTATTCGTGCGTCTCCGTCTCCGCAGGTCAGCTCCGCCGAAGGCGCCGCCAAGGAAGAGGTGAGTGCGGGCCTTCTGCGGGGGGTGGTGGGTTTCCCGTGAGCCGCTGGCCTGCCTTCTCTTCTCGCTGACTCTCCTTTTTCTTTCTCCAAGCCCAAGAGGAGATCGGCGCGGTTGTCAGCTGTAAGTAAAGCGAGCCCCGTAACCGT
2593	chr4	71763394	71763645	-	0	train	AGTAATATCAATAAGTTCCTTTCACCAGTGTTCAGTTTCCTATTCATTAAAAAATCATGAAATTAATAGTCCCTATTTTTGCCATTTATACAGTATTCAATGAGTCTTGACCATATAATGAGATTCTTTCACTTGTTTTCTAGAGACTCCAGCTTAAACATTTATCACTTCTCACCACTCTGTCAAATAGAGTCTGCTCACAATATGCTGCTTATGGGGAGAAGAAATCAAGGCTCAGGTAAAGATTAAGT
1731	chr2	96144008	96144259	-	0	train	TGCCCCAACCACTTTGAGGGCCTTTTCCGCTACAAGAGTATCCCTGTGGAGGACAACCAGATGGTGGAGATCAGTGCCTGGTTCCAGGAGGCCATAGGCTTCATTGGTAAGGGGGCACCTCTGCCCAGAAATCCCGAGGGGCTCCAGAGGAGAGGACTGGGGGCACCTGTCTCCTGGGCTGTGCACATTCCGTCTGACACCACTCCCCCATCTCCCTTGCAGACTGGGTGAAGAACAGCGGAGGCCGGGTG
7213	chr11	5226740	5226991	-	0	train	GCCGTTACTGCCCTGTGGGGCAAGGTGAACGTGGATGAAGTTGGTGGTGAGGCCCTGGGCAGGTTGGTATCAAGGTTACAAGACAGGTTTAAGGAGACCAATAGAAACTGGGCATGTGGAGACAGAGAAGACTCTTGGGTTTCTGATAGGCACTGACTCTCTCTGCCTATTGGTCTATTTTCCCACCCTTAGGCTGCTGGTGGTCTACCCTTGGACCCAGAGGTTCTTTGAGTCCTTTGGGGATCTGTCCA
6395	chr10	47306808	47307059	+	0	train	TGAGAATAAAGCCACATGTTGGAAGGAGAAGAACGCCAGTGTGACGGGGGCAGAGGGGTGGGGTTGGGCTGTGGATAGGAGATAAAGTTGCAGAGGTAGGCAGGCCAGATCAAGTAAGGCCTTGGTAAGGGGTTGAGTTTTATTTGGGGCAATGGAGGGCTCCGAGCTTGGGGGGTGACCTGTGATGGAATGCACCCAGCACACCGGGGGCCTCGCTTGGACAGTGCTCTTTAAAACTGTGACATGAGGAG
12	chr1	67685819	67686070	+	0	train	GGGACTTCTCACGGGACGCCCGGTCCTTGGGCGTGCAGGGGTCATGGGGGGTGACGGGGCCGCGGGAGCGCCGGGTTTTCGTAGAGCCCAGGTGCGCGGTGGTGCTTGCATTCGAGAGGGAGGGGCGTGGTACCGGACGAGGGGGGCGGCGATGGCCCCGAGGGCACCGGGGCTGACGGGACCCCTCGCCCTTGCCCGCGTGTAGGATGGATAAGGTGGGGGATGCCCTGGAGGAAGTGCTCAGCAAAGCC
7636	chr11	69773904	69774155	-	0	train	CCCGCAATGACCTGCGCCCCGCCCCCAGGCCTGCTGGAGCTCTCGCCCGTGGAGCGGGGCGTGGTGAGCATCTTCGGCGTGGCCAGCCGGTTCTTCGTGGCCATGAGCAGCAAGGGCAAGCTCTATGGCTCGGTGAGTACCGCAGGGGTCTGGCTAGGCACCTAGTTGGGAACAGCGGACATGGCTAGCAGGCTCGTGGCTTCTCCAGCCCCACCTGTGCCTGGGTCTTGGAGGGGTGGCAGGGTCACCAG
14905	chrX	153864865	153865116	-	0	train	CACCGGCTACGTGCTCTCCTACCACCCCCGTGCGTGCGCCGCCCCAGCAGGGAAGGGAGGTGGAGGGGCCACGGGGAGGGGGCAGAGCTGCAGCCACAGCCAACCCCTGTCTGTCCCCACAGTGGATGAGGGGGGCAAGGGGCAACTGTCCTTCAACCTTCGGGACCCCGAACTTCGGACACACAACCTGACCGATCTCAGCCCCCACCTGCGGTACCGCTTCCAGCTTCAGGCCACCACCAAAGAGGGCC
4053	chr6	29667281	29667532	+	0	train	TCCTCCCTCCTCATTCAGATGGGAAGTGGCTTTAGATAAACAAAGTGGCAACGCAGTGGGCTGGAGCAGCTCTGTGAACTGAGAATCCAAGAAAAGGGGCGAAGAGCAGCTGGGATGTATTGGATGCTTGTGCTGGCTTGGAGCATTGCTCACATTCTTTATTCGCTATTGTATCTAGACTATAGCTAGAGAAAGAGCCGCAACCATTGGCTTTAAATCCAGTGCTCTTCCTACTCTCCTGAGGTTGTTTC
5008	chr6	37171957	37172208	+	0	train	TAGGAACTATATTATTACTGGTGGCTTTTTTTTTTCTTTAGTGTTAAGGGGAGAGAGAGTCAGGAATGAATGTTGTGAAATAAGATCTGTCGCTGGTTTGAAAATTAGTTGGGTGTCTCCGCAGAGAGGATGAAAACCTATCCTAGGGAGGGGCTTGGAGCGGGTTCTTTCAGAAAAGAAGGAATGGAGAGCCTGAGATCAAAGCTGCGGAGGGTGGGTCATCATCTGAGCGGCTTAACCTAACAAACGAC
6886	chr11	5225868	5226119	-	0	train	TGTACACATATTGACCAAATCAGGGTAATTTTGCATTTGTAATTTTAAAAAATGCTTTCTTCTTTTAATATACTTTTTTGTTTATCTTATTTCTAATACTTTCCCTAATCTCTTTCTTTCAGGGCAATAATGATACAATGTATCATGCCTCTTTGCACCATTCTAAAGAATAACAGTGATAATTTCTGGGTTAAGGCAATAGCAATATCTCTGCATATAAATATTTCTGCATATAAATTGTAACTGATGTA
12388	chr19	41879016	41879267	+	0	train	ACAAGGTCCCAGCATCATTGATGGTGAGCCTGGGGGAAGACGCCCACTTCCAATGCCCGCACAATAGCAGCAACAACGCCAACGTCACCTGGTGGCGCGTCCTCCATGGCAACTACACGTGGCCCCCTGAGTTCTTGGGCCCGGGCGAGGACCCCAATGGTACGCTGATCATCCAGAATGTGAACAAGAGCCATGGGGGCATATACGTGTGCCGGGTCCAGGAGGGCAACGAGTCATACCAGCAGTCCTGC
12194	chr18	46094289	46094540	-	0	train	CATGCACCACCATGCCCGGCTAATTTTTTGTATTTTAGTAGAGACAAGGTTTCACCATGTTGGCCAGGATGGTCTCGATCTCCTGACCTTGTGATATGCCTGCCTCGGCCTCCCAAAGTGCTGGGATTAAAGGCGTGAGCCACTGCACCGGTCCACTGATAGGTTTTATTTTTTCAAAGGCAGTAGCTACTAGATATTTGGCAATCTGTGAACTTGCACATTACAGATGCAGTAGAGAAGTGGGGAGGCAA
1020	chr1	173916226	173916477	-	0	train	GCTGCTTCTCTCCGGCTTTGCACCTCTGTTCTTGAAAGGGCTGCAGAACTGGACTCAGACCACGCAAGAAGGCAAGTCCCCCTCAGCTGCCCCAGCTTCCAGCCAGCCCCAGGCTTGCCCAACGGACCACGTCCGTGAATCTGCACTGGGTGCCTGTCTTTCTCTCCCAGGAGAAGATGGGAAGATCCAGTACCCACACACAGACCCCCTTGTGTACACGCAGGAACCATAAACCAGCTGGAGGCAGCCCC
15405	chrX	154301941	154302192	+	0	train	TTATTTTAGGCAGATAGTCAACTTAAAAATCATTCAGCGTGGGCACTCCTGAAACTGTCAGGATCATGTTGGTTTCCTCGCTGTTTGCCGGCTATTGGCATTTGTGTCATCACACACAGTCCGATGAGGTGACAACAGAAAAGCATATCTCACTTCTATCGTAGGTTTCCATTCACTTTCGGGTTCTCAATCGCCCTGTGTTTCCTCAGTGATACTCCAGACTTTCCAGATGTTCTGGTTGACCCATTCCT
6384	chr10	47306592	47306843	+	0	train	TAGGAAGATAAGAGCAGTAAATCAGTATACAGCAGGATGTCAGGAAGTGCTCTGAAGGCCAATGAAGCAGGGGCAGACAGAGTGATGAGGCTGTTCTCCAGCCACGGGAGCCCAGGGGACCCATTGTGGGAGGTTCCATTTGAGCAGATACCTGAAGAGGTTGAGTGGTACTGGATGGGGAACCATGAAGGTGGAAGGAAGCGTGAGGCACAGTCCTGAGAATAAAGCCACATGTTGGAAGGAGAAGAACG
1008	chr1	173915751	173916002	-	0	train	TGATTGACTCTGGGGTGAACTGATTGACTCTGGGGTTTGACTAAATGAGGAGGAGAGAGGGAGGAATCCAGGGTGATTCTCAGGTTTCTGTACGGGATTCACTGAGCCCACTCACAGGAGCAGGCCTGTGGGGGAGAATTAATTACCAGTTCAGTTTGGTCCTGTTTCCCTGAAGAACTTGTAGGAGTTCCTGGTGGAACTGTCCAGCAAATAGTCAGTCTGGAGCTCAGTGGAAGGGTTAGGGCTGGAGC
2635	chr4	71764659	71764910	-	0	train	AGGGGCCAGATTATACACTGAATCTTTATGTTAGTCTTGGAAGGTCACAAAAATGAACAAAACTGGGGGAAAATACATTTATTAAAGTAAGAAATATATACATATTCAAAAACCAGGAGCAATGTCATTCAAAGTGGTCACAAATTGTAAAAATTAGTAGGATTGCTCATTCCTGTCAATTTTCAAAGAGGATTTTGCCCAGAATTTCAGTCCCTTTTAGATCCATAAGGACATCTATTCCCAGAAATAGA
4947	chr6	33072723	33072974	-	0	train	TAGGAGACGAAGTGCAAAGAGTGTTTCTGTATCCTCCCTCTCTTCTAGGACCCTAGGGCTCTTCCTGGGTCTTTGTGGGTGGTCACAAGCTTTCCTCTCTCAAGACAGCAGGGTTGCATGGTCTTGATAGCCTTGTGATTCGGGTTCTGAGAGATTCAGGACTGCAAGGGAGGCCTAGACTTTTGATAGCTGCAAGGACTCAGCCAGAGATGGACCGTAGTGAATGCTCCTTTTTCCTGTAGCTGAAATCA
16376	chrX	156005125	156005376	+	0	train	TGTGTGCACGTGAATGTGGTGAGTGTGTCTGTGTGTTAACACAAGTGTGTTCAAGAGTGTGTTATATGAGCATATAATGCATGTGTGTATTCTCGAGGGCTGAGGGACCCAGCCCCACCTTCACCACCTGCTAACTGTCCCCACCCCCACAGCAGGCCCAGCACAGGGATCACATTGTCGGGGTGACCTGGCTTATACTTGAAGCCTTTGAGCTGGACCCTGGCTTTATCCATGAGGCCAGGCTGCGTGTC
15894	chrX	154319463	154319714	+	0	train	GGGGCAGTGGCTCACGCCCGTAATCCCAGAACTTTGGGAGGCCGGCCGAGGTGGGTGGATCACCTGTCAGTTCGAGACCAGCCTGACCAATATGGTGAAACCTCGTCCCTACTAAAAATACAAAAATTAGCCAGACGTGGTGGCGGGTGCCTGTAATCCCAGCTACTCGGGAGGCAGAGACAGGAGAATCGCTTGAACCCAGGAGGTGGAGGTTGCAGTGAGCCAAGATTGCACCAGTGCACCCCAGCCTG
12331	chr19	13099925	13100176	-	0	train	CCACAGCCATGAGACATAACAAGAGTCTCACACAGTCATACAAGACACAGGACACAGACAGTCATAATGAGAGGACCTCTCAGACCCATGAGTACACCCAGCCAGCCGTACTTGGGCACAATCAGAATGAGGGCCCCACGGACAGCTTCCCAGACCAAACAAACACAAGGAAATCTTTCTTTAGGGAATCTCAGTCATTGACATAAAGGTGCCCATAGTCACAGATACAGCAGGCCCTTGTCCGTAGGCCG
2431	chr4	71754772	71755023	-	0	train	TGAATGCTGTGATGTTGAAGACTCAACTACCTGTTTTAATGCTAAGGTATATTTGTTGGATTTTCTTTATCAAGCACATAGATCAAGGGTTGGAAAATTATAGTTCATGGGCCAAAACTTGCTCACTGCCTACTTTTTTATGGCCTACAAACTAAGAACAGAAGAATGGTTTTACATTTTTACATTGTTGGGAAAAAAGCAAAAGAACATCAGTATCTCATGACCCATGAAAATTATTCAAAATTCAAATT
11376	chr16	28937174	28937425	+	0	train	GAGGGGGTTGGAGCGGTCTGTGGCCCGAATAGTGGACTGGGCCCTGGAGGAGAGGGGGCATGACTCGGTTCCCCATCCCCATCCCCAAACCCCCAGGCCCAGAAGAAGAGGAAGGGGAGGGCTATGAGGAACCTGACAGTGAGGAGGACTCCGAGTTCTATGAGAACGACTCCAACCTTGGGCAGGACCAGCTCTCCCAGGGTAAGGCTGCCCTCCCCCGTGGCCCCCCACCTCTGCGGTGGCCTGTGGAC
14802	chrX	149504243	149504494	-	0	train	CAAAGCCTAACCCTGCCACCCAGGACTCAGGCTTCCTCCTCGAGCCCCACTCCCACCCTTGCTGAGGCACAGCGCCCTCCCTGGCTAGGCTGTTAAGGTGCAGGGTCCAGCCTTGGGCCTCTTAGTAACCTAGCACCTACCATGAGGGAGGGTTCAGTGTCAGTGCAGGTTACCTCACCAAAGCCCCTCCCTCCTGTGTAGATGCTCTGAACGTTCTTCTCATCATCGTGGATGACCTGCGCCCCTCCCTG
16159	chrX	154327865	154328116	+	0	train	TTACCATTAAACCTCTGGATGTCGCCACCATCGTCTCCAGTGCAAAAGCCACAGAGGGCCGGATCATTACAGTGGAGGATCACTACCCGCAAGGTGTGTGTGGGCATTGAGAATGCTTTGGAAGGTTTTGGTGTTTTTGTTTCCTTGATTGGCCACATGGCATGCTCTCAGGCATCACCTATAGTCTACCTCTCACCCAGCATGGCTTTGTTATTTGATGTTCACTAAGCGTGGGGCCATACTCATACACA
2550	chr4	71760714	71760965	-	0	train	TTTCATGCTGCTGATAAAGACATACCTGAGACTGGGCAATTTACAAAGAAAGAGGTTTAATTGGACTCACAGTTTCACATGGCTAGGGAGGCCTCACAATCATGGTGGAAGGCAAGGAGGAGCAAATCATATCTTACATGGATGGTCGCAGGCAAAGAGAGAGCTTGTGCAGAGAAACTCCTGTTTTCAAAACTATCAGATCTTGTGAGACTCATTCACTGTCACGGGAACAGCTCAGGAAAGACCCACCC
12443	chr19	44928134	44928385	+	0	train	CTTGGGCTGGGCGCCGTGGCTCACACCTATAATCCTAACACTTTGGGAGGCCTAGGCGGGCGGATTGCCTGAGATCAGGAGTTCAAGACCAGCCTGGCCAACATGGTGAAACCCTGTCTCTACTAAAAGTACAAAAAATTAGCCTGACATGGGGGTGTGCACCTGTAATCCCAGCTACTCGGGAGGCTGAGGCAGGGGAATTGCTTGCACCAGTAAGGTGGGGGTTACAGTGAGCCAAGATTGCACCACTG
15190	chrX	153905854	153906105	+	0	train	TATCTGCAGATGGTGGGCATGTATGCCTCCTCCTACATGATCCTGGCCATGACGCTGGACCGCCACCGTGCCATCTGCCGTCCCATGCTGGCGTACCGCCATGGAAGTGGGGCTCACTGGAACCGGCCGGTGCTAGTGGCTTGGGCCTTCTCGCTCCTTCTCAGCCTGCCCCAGCTCTTCATCTTCGCCCAGCGCAACGTGGAAGGTGGCAGCGGGGTCACTGACTGCTGGGCCTGCTTTGCGGAGCCCTG
10708	chr16	2118624	2118875	-	0	train	TGGTCCCCAACACCTGCCCCTGCCCTGCAGAAACCTGAGTGGGAACCCGTTTGAGTGTGACTGTGGCCTGGCGTGGCTGCCGCGATGGGCGGAGGAGCAGCAGGTGCGGGTGGTGCAGCCCGAGGCAGCCACGTGTGCTGGGCCTGGCTCCCTGGCTGGCCAGCCTCTGCTTGGCATCCCCTTGCTGGACAGTGGCTGTGGTGAGTGCCGGTGGGTGGGGCCAGCTCTGTCCTTCCCAGCCAGGTGGGACC
3554	chr4	154606648	154606899	-	0	train	CTGGAGATGCCTTTGATGGCTTTGATTTTGGCGATGATCCTAGTGACAAGTTTTTCACATCCCATAATGGCATGCAGTTCAGTACCTGGGACAATGACAATGATAAGTTTGAAGGCAACTGTGCTGAACAGGATGGATCTGGTTGGTGGATGAACAAGTGTCACGCTGGCCATCTCAATGGAGTTTATTACCAAGGTATGTTTTCCTTTCTTAGATTCCAAGTTAATGTATAGTGTATACTATTTTCATAA
1850	chr2	112835898	112836149	-	0	train	AGGTCTCCTCTTTCAAGAGTAGAGTGTTATCTGTGCTTGGAGACCAGATTTTTCCCCTAAATTGCCTCTTTCAGTGGCAAACAGGGTGCCAAGTAAATCTGATTTAAAGACTACTTTCCCATTACAAGTCCCTCCAGCCTTGGGACCTGGAGGCTATCCAGATGTGTTGTTGCAAGGGCTTCCTGCAGAGGCAAATGGGGAGAAAAGACTCCAAGCCCACAATACAAGGAATCCCTTTGCAAAGTGTGGCT
8449	chr14	24631607	24631858	-	0	train	GGTAAGACTATGCACCTGCCTGGATTGGCTCTTGGGAGAAAGATGTTTGGGGAATATCTGAGACCTGGAGACTCAAGTAGTGGGGGACTCCTTCACCCACTAGACTGTGATATTTCTCTCTGGAAAGAGAAGAGGGGACTAGACTGAGCTGGGGAGAAATTAGGGCCTCTGCAAACTTACCAGGAGGCTTATGGTGGATGGTGCTTCTTTGGAAGGATGAATTTGCAACACTCCACCCACTCCAGGTCACA
6738	chr11	4386008	4386259	-	0	train	AGGAGTGAGTCCTGGAACCTGAAGGACCTGGATATTACCTCTCCAGAACTCAGGAGTGTGTGCCATGTGCCAGGGCTGAAGAAGATGCTGAGGACATGTGCAGGTGAGGCAAGTTCTAGTTTTGCGGGGGATAATGGGGTGCAGAGTAGATCCCAGGGTCAGGGAGCCTGGATGGCAACTTGGAGGAGAGATGGCAGGTCAGAGCAGGGGGAACAGAGATGGAGGTAAGGAAGATGGTTTCTTCAGAGGTC
10120	chr16	2097644	2097895	-	0	train	CCTCACGTTCTCAGGCCTCCACGCTGAGGTGAGGACTCTACTGGGGGTCCTGGGCTGGGCTGGGGGTCCTGCCGCCTTGGCGCAGCTTGGACTCAAGACACTGTGCACCTCTCAGCAGGCCTTTGTTGGACAGATGAAGAGTGACTTGTTTCTGGATGATTCTAAGAGGTGGGTTCCCTAGAGAAACCTCGAGCCCTGGTGCAGGTCACTGTGTCTGGGGTGCCGGGGGTGTGCGGGCTGCGTGTCCTTGC
1148	chr1	186678141	186678392	-	0	train	TGATTGACAGTCCACCAACTTACAATGCTGACTATGGCTACAAAAGCTGGGAAGCCTTCTCTAACCTCTCCTATTATACTAGAGCCCTTCCTCCTGTGCCTGATGATTGCCCGACTCCCTTGGGTGTCAAAGGTGAGTAAGAAGAATCCATTAGAGATGTATTAACTATAAGACGGGCTGCATTGCTGCCAAAAAAAAAAATTGACCTTAGACTACCATTTATTTATTAACAAAAGCAGTTTTTACTTTTA
15264	chrX	154297560	154297811	+	0	train	TATAGTTTTGTTGTAGTATTTGCTTTTGGTATCAAAAGTGTAATGCTGGCTTCATAGGATGAGTTTCAAAGTGTTCCACCTTAAATTTTTTGGAGAGGTCGGGCGTGGTGGCTCCAACCTGTAATCCCAGGACTTTGAGAGGCCAAAGCAGGCGGATCGTGTGAGCCTAGAAAGTTCGAGATCAGCCTGGGCAAGATGGCAATCCCCTGTGCAGAATGTTACAAAAATTAGCTGGGCGTGGTGGCACGTGC
9102	chr15	50247513	50247764	-	0	train	GCATTTATTGAAGGCCTATTTTGCTTAATTTATTTCTATTCAATTCAACAATTACAAGGTATTGTGCTAAGAAAGGGACACAGAGGTAAATGAGACAGGTGCAGCCTCTCTCCCATGTTCTTCATGGAAGAAGCGGGACTTAACTAAGAACTCTGGCCACTGGCATTGTTCTCCCACTTCTACAAGAGTCCACATGCAAGACAAAGCCATTGATAAAAGTAGTGGGGGAAGCTGAAGACACTGTCCTTTCT
10984	chr16	2128463	2128714	-	0	train	AAGGTCTAGGAAGAGTCCGCACCCTCTCCCCGCGGTGGCCACGCCGGGCTCCGCGCTGAGCCCTCTGTGTTCTTGTCTCTCCATACCTCATCACGGCACCGCAGGGTTGCAGCCACTCCTGGTCTCATTTTACACACCAGGAAATTGAGGCTCTTTGAGAAGCCGTGGTGATGATTTCATCAGCATGCTCTGGGGCAGACCCCTGCAGCCGCACAGGGTGCCTGGGGCCCACACTAGTGCCCTGGTTTATA
336	chr1	109612504	109612755	-	0	train	TGGGTGAGTGAGATGGGAAGATGAGCCAGAGAAGGCAGGGGTCCTTCCTACTTTCCTGAAGGGTTGGTGGGTTCTACCTCACCCCATGGGAAGGAAGGGTGGCAGGTCATTTTTCCTCTCCTCTACACAGTCTGGCTAGGGGATCAGGAGATCTAGAGCTGAGTTAATATGGGGCCTAAACAGCCACCCCAAGGGGATCCAGAATGCCAAGGCTATTCCAGAGTTTTTCTACTCTTGAGCGAGGAATAGTG
13460	chrX	147924644	147924895	+	0	train	GGGACTATAGGCACATACCACTGCACCTAATTTTTTTTTTTTTTTAATAATTTGTTGTAAAGATCAGGTCTTACCTTGTTGCCCAGGCTGCTCTTGAAGTCCTGGCCTGAAGCAGTGCTCCCACCTCAGCCTCCCAAAGCTCTGGGATTATAGGCTTGAGCCACCGCATCCTAATATTTTATATTTTTATGGATATAAAAAATAATTTGGTATCTTTCAGAGTTGTTTAATATCATTTTAAATTTAAAAAC
1581	chr2	79086194	79086445	-	0	train	TGCCTTCGTGGCCTCACTGATTAAGGAGAGTAGCACTGATGACAGCAATGTCTGGATTGGCCTCCATGACCCAAAAAAGGTCAGTCTGCAGCCACCTCTATCTCCTTATAAACATTTTTGAGAGGTAAGAGGGACGTTTAAGGTCTGGCACCGCAATCACCAACTTTTATCTTTTTGTTTGTTTAAATAAAAGCAACCTCTTTATAGATCCTATAATGTATGAGTTGTGAAGTTCAGTGTAGGTAGTTAGA
2687	chr4	71767258	71767509	-	0	train	CAAGTGCTCCAGAATAATTTAGTGTTGGTTTCAAATTTGGAAGGAGGATTCAAAAGTTGCCCTGTATGCACATAACTGAATTCTAGGTTTCTCCCCTATAGCACTTATGCAAAAATTAAAATGCTGACATGTTCAGACAGTAGCTGAAAAGGGTTAAATCTAAGGAGTTTCAGCAATTTATTAATGCAGTCGTCAGGCCTGATGCAAATGACATTTAGATGCATTTTGATAAAGGCTTTTACAGACAGATT
11967	chr18	46085620	46085871	-	0	train	AACCTCCACCTCCTGGATTCACGCGATTCTCCTGCCTCAGCCTCCTTAGTAGCTCGGATTACAGGTGTGAGCCACCACGCCCGACTAATTTTTGTATTTTTAGTAGAGACAGGGTTTCACCATGTTGGCCAGGCTGGTCTCGAACTCCTGACCTCAGGTAATCCACCCTGCCTCAGCCTCCCAAAGTGCTGGGATTACAGGTGTGAGCCCCCATGCCCGGCCCTGTATTCTGTTTTTTAAAGAGAGACAGT
5414	chr7	99970233	99970484	-	0	train	GATCACCTAAGGTCAGGAGTTCGAGACCAGCCCGGCCAACATGGTGAAACCCTGTCTCTACTAATAATACAAAAATAGCCTGGCATGGTGGCACACGTCTGTGGTCCCAGCTACTCAGGAGGCTGAGGCAGGAGAATTGCTTGAACCCAGGAGGCAGAGGTTACAGTGAGCCAAAATCCTACCATTGCACTACAGCCTGGGTGACAAGAGTGAAACGTTGTCTAAAAACAAAAAACAAAAAACAAAAAAAG
6544	chr10	47349178	47349429	+	0	train	GCAGCCAGACCAGGGGCGTGGCCGAGGACATCGCGCACATCCTTAAGCAGATGCGCAGGGCCATCGTGGTGGGCGAGCGGACTGGGGGAGGGGCCCTGGACCTCCGGAAGCTGAGGATAGGCGAGTCTGACTTCTTCTTCACGGTGCCCGTGTCCAGGTCCCTGGGGCCCCTTGGTGGAGGCAGCCAGACGTGGGAGGGCAGCGGGGTGCTGCCCTGTGTGGGGACTCCGGCCGAGCAGGCCCTGGAGAAA
8266	chr12	57231841	57232092	+	0	train	CCTGGCCGTCTACACAGCCCTTCTGCAACCTCACGACCGGATCATGGGGCTGGACCTGCCCGATGGGGGCCAGTGAGTATGGATGGGCTGGCTGATGGTCTTGGCGGCAGGATTGGTGTGGGAAAGGAGTTATTTATTGAATACCTACTGTGGACCATACAGATGGAACAGGCCTTGCCCTGTCCTGCATGTCACAGTGGATGAGGAAGATAAGATCCCAGTTATAGTGCCTACCACAGAGTGGACAGAGC
15531	chrX	154305982	154306233	+	0	train	CAAAGTACACTGTGTCCCGAAGCAAATTAAAACCTGGCTAGGGGCCGGGTGCAGTGGCTCACGCCTGTACTCCCAGCACTTTGGGAGACTGAGGTGGGTGGATCACTGGAGGCCAGGAGTCCGAGACCAGCCTGGTCAACATAGCAAAACTCCGTCTCCACTTAAAAAAAAAAAAAATTAGCTAGGCATGGTAGCACATGCCTCTAATCCCAGCTTTTCGGGAGGCTGAGGCACAAGAATTGTTTGAAGCT
8483	chr14	24633437	24633688	-	0	train	CTCAGAACACTGTTAATGTGTTTGCTCAGTCCCATTCTCCAACTCTGCTTTTCTTCCCTGGCCTTTGGTGGCTCCCCTCTTTCCAAGGATGAGGCACTACGGCAGGCCCCAGCTTCCCTGCTTTCTAGAATTCCACCAGCACTGCTCTACCAGCCCTCATCCAGAGGCTAACTGGAGCCAGTCCATCATGCAGCCATGAACATTTACTGGGCACCCACTACATGTCAGGCTCTAGGAAACAGGATATGACA
3700	chr4	154611280	154611531	-	0	train	CATCCCCCATTTATCTTACAACATAAAATCAATCTCATAGGAATTTGGGTGTTGAAAATAAAATCCTCTTTATAAAAATGCTGACAAATTGGTGGTTAAAAAAATTAGCAAGCAGAGGCATAGTAAGGATTTTGGCTCCTAAAGTAAATTATATTGAATGTGGAGCAGGAAGAAACATGTCTTGAGAGACTAAGTGTGGCAAATATTGCAAAGCTCATATTGATCATTGCAGAATGAACCTGCATAGTCTC
13665	chrX	147931190	147931441	+	0	train	AGCCCAGAGCTGCCTCCTTACTTTGATTGTGAGTATTATGAAAATATACTCATATGCCTCTTAAACATCGAATTATTCTTAAATGATCTATCATACTTTTGGCTAACCATTTATCAGTGAATTATCAATGAAAAAGAACGAACCAGATTAATGAACCAGGACTCAAAAAACATGACTTCTTGAAAAGTCCTAGAATTATTTTATAGATTTTATTACTCTAGTCTAAAAAAAATGGGTATTGCACTTTTAAA
11056	chr16	2130741	2130992	-	0	train	CATCAGCTCAGGGCCCCCTGCTCTAAAGGCCACTTCTGGTGCTGGTTGCCACTCACCCTGGCTGGGGGTCACCTGGGTCTGCTGCTGTCTCGCAAATGCTGGGGTCCAGGACTGGGCACATCGAGGGACTTGGTAGGTGCTTGGTTCACTGATGTAAAATATAGGAGCACCCGGGGCCTTGCCCTTTCCCACCTGCATCCCTGAATGACAGGAGAGTGTGGGAGAGTGTAGGGACAGCAGGCGCAGACCCC
16307	chrX	156003077	156003328	+	0	train	GGACTGGGTTGTCCGATGTCAAGCCTCTAGGGAAAGGTTTGGCCCAAACTGTGCTGGGGCATGTCCTCTAGGGGTCAGCCTGGACCTCAGTCTCTAGTCTCCCTACTTTTACCTCCCTACCTTCATTCCCTGGACCGACTGTAGTCTCCCTTCCTTCACTCTCTTGACGCCTCTCCAGATCTGACTTGCCCGTGTACCACAGGTCAGAGCCCATCACTTCCCAGGCCTCCCAGTGCTTCCCTGGACAGATT
5845	chr7	150857865	150858116	+	0	train	GACTTTATCTTCTACCCCAACGGGGTGATGGAGGCCAAGATGCATGCCACTGGCTACGTCCACGCCACCTTCTACACCCCCGAGGGGCTGCGCCACGGCACTCGCCTGCACACCCACCTGATTGGCAACATACACACTCACTTGGTGCACTACCGCGTAGACCTGGATGTGGCAGGTAGGACTCAAAGCGAGACTCTCCCGTTCAAACATCTGCATCCAGCCAATAACTTAAACTCCCAGGAGACGGCACT
212	chr1	109608349	109608600	-	0	train	TGCTTAAGCAATCTTCTAGCCAGTCTTCTCTCTGGTTGGGAGAAACCTCACCCAACCCAAAATTTCAGGCATTGAAAGCTGGAGACCCAGACTGAATTCAGCCTGTGGATCTGTTTTGTTAGGCTTCAGCAATGTTTTGAATTTAATGCTATGGGGAGATCTGCCACAGTTGTCATGACTTTCTATTGCTTTACACTGGCTCACTACAGCCTACAAGGCTGAAGATGCAAAATTCAAAAGACGTGTTCTGC
14420	chrX	149489064	149489315	-	0	train	GTACTGTTAACGTTCCCTGGTTTTGTACACTTAATGTCATAGGCAATCTCTGTGTCATTTCTCAGAAGCCTACCTTTTCCCTTAGAAATGTCTGTAATATTTTATTATATATGGAGGTGCCATAATTTTTCACATATTCCTGATTATCTGCCATCTGCTTCTGAGCCCTCGGTGCCCAGTTTCCTCTTCTCCACGAACACACTCTGTTGTGAGGCAGTTGCCGTAGATTACAAATAGCCCATCTCAGAGTC
2541	chr4	71760530	71760781	-	0	train	TTTCAAAACTATCAGATCTTGTGAGACTCATTCACTGTCACGGGAACAGCTCAGGAAAGACCCACCCCCATGATTCAATCACCTCCCACCAGGTTTCTCCCACCACCTGTGGGAATTGTGTAAGTTACAATTGAAGATGAGATTTGTGTGGGGACATAGCCAAACCATATCAACCTTATATTTGCAACCTTCACTTTGATTTTTGTATGACTACTGTTAACTTGTATTTTAATTTTGATAGTTTCTATGCT
2630	chr4	71764493	71764744	-	0	train	TCATTCCTGTCAATTTTCAAAGAGGATTTTGCCCAGAATTTCAGTCCCTTTTAGATCCATAAGGACATCTATTCCCAGAAATAGAAATTATTTCTTCCAAAAAATAAGTAATCTATACATAATAATTATCTAAAGTAATAAACTGAGATAGATAATGAAAAATTAATCTAGCAATTACATTCATTTTAATTAATTCAATTAGCAGCTTAATTAACAAGCAGATACCATGGGTGATGCACCTGACCAGACAT
11066	chr16	2130975	2131226	-	0	train	ATGTCCGATGATGTCTAGGAGCTTCCCTTCCTCTCTTTTTCCTTGTGCAATTTGTTGAAGAAACTGGCTCCTGCAGCCTGGATTTCTCGCTGTGTCTTGGGGGTGCCACCTCCATGGTGTCACCTCCGTGGTGCTGTGAGTGTGTGCTTTGTGTTTCTTGTAAATTGGTCGTTGGAGCCGACATCCCATTGTCCCAGAGGTTGTCCTGGCTGGCACTGGCCTAGGTGTAGATGTCATCAGCTCAGGGCCCC
14468	chrX	149491316	149491567	-	0	train	ATCTTTTCTCTTGACTGCAAGTGAGAACGGGTGAGTCTCATGATTGTCCTCACCCTGGCAGTGATGAGAAGAAGCTGGTGGGTCCAGCTGATAAGTCAGGGGCTGGTCTGCGAAGACACGGCTCTCTTCTCACCCCTCTTTGGGGTGGAAGATTAATTTTTGTCCTTAGCATTTGTGAACCAGGTTGGGAATGAGAGTCAGCCCAGGAGGGGCCGGTGGCTCATTTACTTCAGGGCATGATCTGGCTGTTC
10789	chr16	2121693	2121944	-	0	train	AGGGATGGAGGCCGCCCCGGCTTGGGGCTGGCTGCCGGGTGGTCATTGCTGGGAAGAGCAAGTCTAGGCGGAGGCACCTGCTGGGTCACTCGTGGGGAGGGTGACACCTGGGGAAGTAGAGGCCCGTGGCAGGAGGTGAGGCCTCGGGGTCCTGGGGAGCAGGGGGGTGGTGTGCAGACCTGCGGAGCCATAGTCCTGTGCCAGGAGCACTACTGGGAGTGCGTGGGACCAGGAGGGGTGCCCAGGGTGGG
10396	chr16	2108159	2108410	-	0	train	CCCGAGCGCCTGGTGCCCATCATTGAGGGTGGCTCATACCGCGTGTGGTCAGACACACGGGACCTGGTGCTGGATGGGAGCGAGTCCTACGACCCCAACCTGGAGGACGGCGACCAGACGCCGCTCAGTTTCCACTGGGCCTGTGTGGCTTCGACACAGGTCAGTGCGTGGCAGGGCCGTCCTCCATGCCCCTCACCCGTCCACACCCATGAGCCCAGAGAACACCCAGCTTGCCACCAGGGCTGGCCCGT
8117	chr12	14883672	14883923	-	0	train	CGAGGAGTAATGACAAATGGTAAAGCACAGAGCTGGACGCCAAGTCAGCTGGGAGACCACAGGCGCCACGTTAAGCTGAGTGCTGTTTTGGTTTTTTTGTGTTTTTCTTTCTTGTTTTTTTTTTTGAGACAGTGTCTCACTCTGTCGCCCAGGCTAGAGTGCAGTGGTGTGATCTCGGCTCGCCGCAACCTCCACCTCCCAGGTTCAGGCAATTCTCATGCCTCAGCCTCCTGAGTAGCTGGGATTACAGG
3168	chr4	87979340	87979591	+	0	train	TTTTTATTTTTTTTAGTAGAAACGGGGTTTCACTGTGTTAGCCAGGATGTTCTCGATCTCCTGACCTCGTGATCCGCCTGCCTCGGCCTCCCAAAGTGCTGGGATTACAGGGGTGAGCCACCGTGCCTAGCCATTTCATTTTAATTAACTTAAATTTAAATAGCTCCATGTGGTTAGAGGATACTGAATTAGCACAGTCTTAGAGAGTTCCTTCTTGTTCCATGGACTGGACACAATGAAGATTAACAGTA
6372	chr10	47306169	47306420	+	0	train	GAACACTCAGCATAGGACCAAGCACATACACTTGGAGCTCAGAATGTCATAATTAATAAACAATTTTCTTCCAATTTAAAGATGAGAACACTGAGGCTCAAGGGGAAGTCTGGCTAAACACCTGGGTGCCCACTGTCTCCATTTGGATTTGCATCATCAGCTGCATCTTTGTAAAGAATGAGGATTCCTTGACTTCTTAAGGCTGTGGTTACTGCACTCTAGGGAAGCATGCCAGGAACCCCCAGTGTTAA
1105	chr1	186676581	186676832	-	0	train	GTATGCTTCCTTTGACTATTAAGACTTAGTTATTACCGCTTATACCCATATTTTAAAATCCCTAAAAATGTGTTCCTTAACTTTTTAACTGATGTTTATTTATTTATTTATTTTTTTAGATAATTGATGGAGAGATGTATCCTCCCACAGTCAAAGATACTCAGGCAGAGATGATCTACCCTCCTCAAGTCCCTGAGCATCTACGGTTTGCTGTGGGGCAGGAGGTCTTTGGTCTGGTGCCTGGTCTGATG
45	chr1	67686612	67686863	+	0	train	GCAGGGTAGAGCCCCGGAAGGACGGGAGTCAGGGCTGGGTTGCCTGATTGTGGATCTGTGGTAGGTGGGGGTCAGGAGGGTGGCTGCCTTTGTCCGACTAGAGTGTGGCTGGACTTTCAGCCGAGATGTGCTAGTTTCATCACCAGGATTTTCTGTGGTACAGAACATGTCTAAGCATGCTGGGGACTGCCAGCAGCGGAAGAGATCCCTGTGAGTCAGCAGTCAGCCCAGCTACTCCCTACCTACATCTG
1871	chr2	162144380	162144631	-	0	train	CAGTGTTTTAAAGGGATACCAAAAATTCTGCAATAGTAAACCAGTGAAAGAGAAAAATCTAATATAGATGAAGCTTTAACCTCTTAATACTGCATTTTGCAAGGCTGTTCCTGCAAGCTCTGGTTTTATAGGATATGATATATTTAGTTGAATTACAGACTAATAATCTCAACAATAGTTTCTGTATTGTCAATATACTAAAATCTTCAAAACAGCCTAGAAGATTGAAAAGGGCATGAAATTATGCAGGC
13560	chrX	147928053	147928304	+	0	train	GAGCACTAATTATTGCTGAATTAGAACAGAAATATAGGAAAACTGATTTTTACAAGGAGCTTCAAAGCAATCTCAGGTAGTTTCTGATTATGTATCTCTGCCTACCTCGGGGTACATAGACAGGGTTACAATTTGGTTGAGGATATATGACATGTGGTTTTTAAAGACACCTAGGGGCATTTTAAGAAAATTTCCTCGATATCTGAAAATCTGTAGATTTCAAAATTATGTTAATCATGAAATATTCTGTG
12509	chr19	44930596	44930847	+	0	train	CTGCCTCAGCCTCCTGAGTAGCTGGGATTACAGGCGTGCACCACCACGGGTGGCTAATTTTTGTAATTTAGTAAAGACGGGGTTTTGCCATGTTGGCCAGGCTGGTCTCGAACTCCTGACCTCCAGTGATCTGCCTGCCTCGGCTTCCCAAAGTGTTGGGATTACAGGCGTGAGCCAAATGCCCAGCCAAGGGTAAAGTGTTTAGACTTCAACGTGCTTTGGTCCACCTGGGAAACTGAGGCACAGAAGTT
7040	chr11	5226191	5226442	-	0	train	TCAGGATCGTTTTAGTTTCTTTTATTTGCTGTTCATAACAATTGTTTTCTTTTGTTTAATTCTTGCTTTCTTTTTTTTTCTTCTCCGCAATTTTTACTATTATACTTAATGCCTTAACATTGTGTATAACAAAAGGAAATATCTCTGAGATACATTAAGTAACTTAAAAAAAAACTTTACACAGTCTGCCTAGTACATTACTATTTGGAATATATGTGTGCTTATTTGCATATTCATAATCTCCCTACTTT
8484	chr14	24633444	24633695	-	0	train	CACTGTCCTCAGAACACTGTTAATGTGTTTGCTCAGTCCCATTCTCCAACTCTGCTTTTCTTCCCTGGCCTTTGGTGGCTCCCCTCTTTCCAAGGATGAGGCACTACGGCAGGCCCCAGCTTCCCTGCTTTCTAGAATTCCACCAGCACTGCTCTACCAGCCCTCATCCAGAGGCTAACTGGAGCCAGTCCATCATGCAGCCATGAACATTTACTGGGCACCCACTACATGTCAGGCTCTAGGAAACAGGA
16378	chrX	156005158	156005409	+	0	train	TGTTAACACAAGTGTGTTCAAGAGTGTGTTATATGAGCATATAATGCATGTGTGTATTCTCGAGGGCTGAGGGACCCAGCCCCACCTTCACCACCTGCTAACTGTCCCCACCCCCACAGCAGGCCCAGCACAGGGATCACATTGTCGGGGTGACCTGGCTTATACTTGAAGCCTTTGAGCTGGACCCTGGCTTTATCCATGAGGCCAGGCTGCGTGTCCAGATGGCCACACTGGAGGATGATGTGGTAGAG
13495	chrX	147925920	147926171	+	0	train	GAATAGTATGTTGTTTGTTTACAACTGTGTCCCCATTGTAAGCAAAATGGATTATGAAAATTAATTTTACACAGGAAAGAAATCATGCTTTATTACAAAATAGTATACTAGAATTTCTTTAAGTAGCAGTGAATCTTCTTGGTATATTTTTAAAAACCTACAAGCTTTAGTTTATACATATTGGTAAAATCTCTTTTTCACAGGTTATACCGTGAACTACCTGCTTATCTCCCGTTGAGCTCTTTGACCAA
604	chr1	119513006	119513257	+	0	train	ACAGCTCAGGGGAGGCTGCAAGGTCCTCCCACTGCAGGTGTTACCAGCAGAGGACACACTTCTCTCCCCAGCCCTCCACCAGGCTCCACAGGAAATGCCAGGGCAGGGTTTAAAAGAAGGTTTATCACCTCCACTTTACATAACCAGGTAAACAGAGTCACAGCCAGAATTGGAAGCAGCTTTCCCAGGGAATGCACAATCAGGAAGAGTGTGGGTTTCCAGCATCTCCCCACAACCCACTGCTTGCTCAG
15861	chrX	154317036	154317287	+	0	train	TGGGATTACAGGTGTGAGCCCCCGAGCCTGGCTCATTAAGCTGTTTTCATTTTGAATAGAAAACAATGCGATCCGTAATTTTCTGGATTAATGCTCCTGTCAAAAATTTTGCTCCCCACCTCACTGATTGCCTCAGCACGCAGCCATGTAGTCTACTCCGTCCCTGCTGATATGTTCTCACAGTTGGAAAAAAGTTGTAAAGAAAAGGCGAGCAGGTGGTATTTAGTGCCATGTTGAAAGAGGTGGCTACA
15149	chrX	153874780	153875031	-	0	train	ATATTCGTGAATGTGATTTTGACTTCCTTACATGGGTGACTGTGTGAGTCACTCTGTTACTTACTGGCCAGGCTTGTGGAGGTCGGGAGGTATTTGACCACGACATTTGATTGCATTGGGTCATGTGTATGTGTGAGTGGGGCTGAATGTAAGTACACACTTGCGTGTGGGGGGTGTTACCGTGATGGTGTTTTCTCCTGTAAGTAGCTGTCAGGCCGTGTGAGGGGCATGTCACAGGGTATCATGTGTGA
1658	chr2	79158518	79158769	-	0	train	GTGAAGAACCCCAGAGGGAACTGCCCTCTGCACGGATCCGCTGTCCCAAAGGCTCCAAGGCCTATGGCTCCCACTGCTATGCCTTGTTTTTGTCACCAAAATCCTGGACAGATGCAGATGTGAGTGGTTAGATGTGGTGTTGGAGGTGACCGGTCTCAGGGGGAGGAGGGTCTCCATTCAGGAGAGTTCCTTGGGAATGAGGATGAACACGTTTATCTTTCACACAGTCCTCCTCCCACCTACCTTTGCCC
14596	chrX	149495747	149495998	-	0	train	GATGGGCAACTCCCGAGCAGCCCAGAATGCTTGGTTTGCTGGTACTTGCAGGTTCCCTCTGTGGGAAAGCCCATCCTGGCTAGCTGTCACTCTCAGGGGGTGGCTTGGTGAACAGGCCCTTGGTGCAGAGGGAACTGGGACACCATCATGGCCAGTGCCACCCAGCTGTGATGTCCGGCATTCCTTTGCCATGATTTAGGGGCGGAGCAACCACACAATTAGAATCCTCACACTGACTCATCAGTCACCCT
4913	chr6	33071643	33071894	-	0	train	CTCCTTAGATGACTTTCAGGTCTTCTAAGTGCAATAGCCCCACAGTAAACTCAGTATCTTCTCCCGGTCAGGCTGTCTTCCCTGAGAGAAGTGGCTTTTGCCCTGTTTTCTGAATGCCTACATTGAAGCCATCTGTTCCCCAGGAAGCCTTCCCTGATGTGCTGTTTGGTCGCATCTTGTGTATACCTACGTATCTGCACTTATCCTTCTGAACCTGCTGTTGTCCTGTCACTTGTGTTTCCTTCTGTGAC
6137	chr9	130700157	130700408	+	0	train	GATTACCGGCGTGCGTCATGCCTGGCTAATTTTTGTATTTTTAGTAGAGACAGGTTTCACCATATTGGCCAGGCTGGTCTCAAACTCCTGACCTCAGGTGATCTGCCCACCTCGGCCTCCCAAAGTGCTGGGATTACGGGCATGAGCCACCATGCCTGGCCTCTGTCTTTATTTTATTTATTTATTTATTTATTTATTTATTTATTTATTTATTTTTTTGAGACGGAGTTTCACTCTTGTTGCCCAGGCTG
3166	chr4	87978911	87979162	+	0	train	TCAGCAGATCAGATGATACTTACTCAGAGCAATTTCCACTCCTTTGCAGTAGCATATTATCAGTATTTTCCAGATAAATAACTTGGCTAAAGAAAAATCCATTTCATTTACATCTTTGGCACCTTACAGCAATAGAACTTTTGTGCAATGATTTTAATATTATATTTCTACATTGGCTGATAAGATACATATGGCTATTGAGCACTCAAAATGTGGGCTAGTGCAACTGAGGAACTGAATTTTTATCTTCT
15797	chrX	154314516	154314767	+	0	train	ATTCATAAAACTCAACATTTTTTTTTTGAGATGGAGTCTCGCTCAGTCACCCAGGCTGGAGTGCAGTGGTGCAATCTCGGCTCACTGCAACCTCTGCCCCCGGATTCAAGCGATTCTCCTGCCTCAGCCTCCCAAGTAGCTAGGATTACAGGCTCCCGCCATCATGCCCAGCTAATTTTTGTATTTTTAGTAGAGACGGGGTTTCACCTTGTTAGCCAGGCTGGTCTCGAACTCCTGACCTCAGGTGGTCC
11049	chr16	2130328	2130579	-	0	train	GCGAGGACTAGGGATTGTCACCAAGGCCTCCATGAGCCCTCAGCAGAAGGAGGGCCACCCTCGAGGGCTCCGTTATCACTGGAGCCCGCGTTCAACCAACACGCAGATGATTCTCCAAGGACAGAGATGGATGATGGGGAGGGGGCTGGCCTGGAAGGACCCCCAGTGCAGGTGACATTGAAGCCAGGTTTCAAAGCTCCCACAGGGAGCTGCCCAGAGAGAGTCCCCAAGGGGCAAGGTGACTCGGGGGC
9465	chr15	50261441	50261692	-	0	train	GGCAGGGGTGGGAAAATATTCAAACTCAGAAAGTGATGGCAATTTTGACTTTTTTTTTTTTTTTTTTTTGAAGCAATAGAGACAAGGTCTTGCTATGTTGCCCAAGCTGGTCTCCAACTCCTGGGCTCAAGTGATCCTCCTGCCTCGGCCTTCCAAAGTGTTGGGATGAGAGGCCTAAGCCACTACGCCTGGCCATGAGGGCAATTTCATACAAAGCCACTCATCTCTTTGACCATTCAGCCTATTTCACA
7246	chr11	5253965	5254216	-	0	train	ACAGCAGGGTGTGAGCTGTTTGAAGATACTGGGGTTGGGAGTGAAGAAACTGCAGAGGACTAACTGGGCTGAGACCCAGTGGCAATGTTTTAGGGCCTAAGGAGTGCCTCTGAAAATCTAGATGGACAACTTTGACTTTGAGAAAAGAGAGGTGGAAATGAGGAAAATGACTTTTCTTTATTAGATTTCGGTAGAAAGAACTTTCACCTTTCCCCTATTTTTGTTATTCGTTTTAAAACATCTATCTGGAG
10276	chr16	2103414	2103665	-	0	train	GGCGCCCCAGGGCCTGGCTGCCACTTCTCCATCCCCGAGGCTTTCAGCGGGGCCCTGGCCAACCTCAGTGACGTGGTGCAGCTCATCTTTCTGGTGGACTCCAATCCCTTTCCCTTTGGCTATATCAGCAACTACACCGTCTCCACCAAGGTGGCCTCGATGGCATTCCAGACACAGGCCGGCGCCCAGATCCCCATCGAGCGGCTGGCCTCAGAGCGCGCCATCACCGTGAAGGTGCCCAACAACTCGGA
1882	chr2	162144701	162144952	-	0	train	GGGCTGTCATATAGATATCTACTAAATAATCTAAGTTGAAAAACAACCAAGACCATCAATTACTTGCTTAGATCTTAACACAGCCAAACAGACCCCTGAACCATCTCATTTTCTTCCGATTTTTTTTGGAGAGATGAAATATGAGAGACGGAGAATTTATGTTCAACTCTGATTTTTAAATTAGATTTAAAACAAGCTTACTGAAATTTAAAGAGATCTCAAGATGAAAGAGAACTAGAATAATGGTTGGT
4744	chr6	32949506	32949757	-	0	train	AAGGATTTGGGCCTACTTTTGTCTCAGCTGTCGATGGACTCAGCTTCCAGGCCTTTTCTTACTTAAACTTCACACCAGAACCTTCTGACATTTTCTCCTGCATTGTGACTCACGAAATTGACCGCTACACAGCAATTGCCTATTGGGGTGAGGCTTTCTCCCTGGAATTCTGGTCCTTTTGGGGGCAAAAAGGGATAGATCCATGGGAGGAGGCTTCTTTCTCCACTGGTACCTTGTTTAGTCCATTCCTA
3666	chr4	154609943	154610194	-	0	train	TTTGCAGGAAATATATAATTCAAATAATCAAAAGATTGTTAACCTGAAAGAGAAGGTAGCCCAGCTTGAAGCACAGTGCCAGGAACCTTGCAAAGACACGGTGCAAATCCATGATATCACTGGGAAAGGTAACTGATGAAGGTTATATTGGGATTAGGTTCATCAAAGTAAGTAATGTAAAGGAGAAAGTATGTACTGGAAAGTATAGGAATAGTTTAGAAAGTGGCTACCCATTAAGTCTAAGAATTTCA
7945	chr11	119090813	119091064	+	0	train	CATGTTAGTCAGGCTGGTCTCAAACTCCTGACCTCAGGTGATCCACCAGCCTCGGCCTCCCAAAGGGCTGGGATTACAGGCTTGAGCCCCGCACCCGGTCAGTACTTCCATTTTTATATGCTACTATATTGTCTTGACTTTTACAATGAATATGTAGTACATTTCATAAAACTAAATTTAAAAATAGTATGTGCTAAGTGCTCCAATAAGTGAAGTTGGGAATTTTCTGGAAACTTCTAGTTGGAACATCT
13788	chrX	147934888	147935139	+	0	train	CTCTATTTTTCACAAACTCTCTTTGGTATGTGCACTAATTTATCTTTTCATTAAAATTGGATTCTTCACTCACTAAGAATGAATTGGTATGACTATTCAGTCAGCTATAGTTAACACACAGTACTTAGGATAATTAAGATATTATCTTAATAAAAACATCCCCAAATCTCATTTTTCATTTTGTTAACTCACTTAGATCTTTATTAAAGTCAACATTAATTCATGAAAATGTAAGTGTTGCAGTTTAAAAT
13883	chrX	147937560	147937811	+	0	train	CACCCATTTTTCTCAACCTAACAGTACAAAAGTCCAGAGGGTAAGAATTACTTGTCACTTTGAATTACAATACAAGTAATTTGTCTCAGATGTCACAATTGGTATTTTGGATGTTTTCTCTGGTTAGACTTGTAGGCTACTTATGTTCTATTTTTTTTCCAAATGTAATTGCCAGCTTATGAGATACTACAAATTAGTGTGATACATACTTTTCAATTTATTCACAGTAACAACTCTTATGCCATAAACAC
12236	chr18	46095870	46096121	-	0	train	AAGTCAGGAGTTCGAGATCAACCTGGCCAACATGGCGAAACCCTGTCTCTACTAAAAATAAAAAAATTAGCCGGGCATGGTGGTGGTCGCCTGTAGTCCCAGCTACTCAGGAGGCCGAGGCATGAGAATCGCTTGAACCTGGGAGGTGAAGGTTGCAATGAGCCAAGATCGTGCCCCTGCACTCCAGCTTGGGTGACAGAATGAGACTCCGTCTCAGAAGGAAAAAAAAAATGTGTGTGTATATATATATT
5872	chr7	150860558	150860809	+	0	train	CACCAGAACGACCCCTGGCACCCGCCCGTGGTCTTTGAGCAGTTTCTTCACAACAACGAGAACATTGAAAATGAGGTACTGCCCTGTCCCCAGCCCTGCCCGGTGCTGGCCCTGCCTCCTTCCAGCTCAGCCCAGGACCATCCTCATCACCATCAGGGAGTCCCAACCACCCTTTGCCCAGATCTGTCCCCAGTCGCAGGAGCTGTGCCTTGCTGTGTGGACGGCAAGTTCAGAGGTCACAACAGAGCTGC
6660	chr10	47352565	47352816	+	0	train	GACTGAGGGCAGGGCAGGGCCTGGAGGGCAGTGTCTCTGTCAATGAAGTCTCCTTGCCTGTCAATCTCACCAAGACCTGCCTCCCTCCAGCAGCCTTAGAGAGGGAGGAGGAGGTGCATCCACGTGCGAGTAGCCTGTGCTAGGCTTGCAGAATCCCCAGTTTCCAAATCAACATCTCCTTCCTTTCCAGTATAGCCAAGGTTCACGATTTGGAGTCAGATGTGGATTCAGATTCTGGCTCCACCACTTAC
3749	chr5	132542405	132542656	-	0	train	TATGACTATATGATGGTGTTGTATGCATTTGAATATGTCCTGGTCATATTAAAATGTAAAATATATAGTTTTATTAGTCTAAATAGAATAAAACTACCAGCTAGAACTGTAGAAACACATTGATATGAGTTTAATGTATAATGCATTACACTTCCAAAACATTTTTTTCCAGTTACATAATTAAGTTATATCCTTTATAAAACTCCTCAGTAATCATATAAGCTTCATCTACTTTTTGAAAATTTTATCTT
10128	chr16	2097901	2098152	-	0	train	GCGAGGGGCTGCCATCACGGACGGTGCAGATGTCCCATATATCCAGCATTCTAGGACATTCTGTCAGATGGCACCGGGCTCTGTCCTGTCTGCTGAGGAGGTGGCTTCTCATCCCTGTCCTGAGCAGGTCTGAGCTGCCGCCCGCTGACCACTGCCCTCGTCCTGCAGGTGGCTGGGAGCCCGAGCCCCACACCTGCCGGGCAGCAGGTGCTGGACATCGACAGCTGCCTGGACTCGTCCGTGCTGGACAG
9756	chr15	90894261	90894512	+	0	train	CAACACAGTGAAACTCCATCTGTACAAAAAATACAAAAATAGACTGGGCACGGTGGCTCACACCTGTAATCCCAGCACTTTGGGAGGCCGAGGCAGGTGGATCACCTGTGGTCAGGAGTTTGAGACCAGCCAGACCAACATGGTGAAACCCCATCTCTACTAAAAATACAAAAATTAGCCAGGCATGGTGGCACGTGCCTGTAATCCCAGCTACTTGGGAGGCTGAGGTGGGAGAATTGCTTGAACCCAGG
5542	chr7	99972605	99972856	-	0	train	CGTGAGCCACTGAGCCCGGACGAAATGTTAATTTGTTTTTTTTGAGACGGAGTCTCACTCTGTCATCCAAGCTGGAGTGCAGTGGCATGATCTTGGCTTGTTGCAACCTCTGCCTCTCTGGTTCAAGTGATTTTCCTGCCTCAGCCTCCAGCATGACTGGGATTACAGGCCCGCACCACCATGCCCAGCTAATTTTTGTATTTTTTAATAGAGATGGGGTTTCACCATGTTGGCCAGGCTGGTCTTCAACT
8364	chr14	24573807	24574058	-	0	train	ACATTCTCCTGGAGAAGGGAGAGGTACCTTGACTCAGATTGGGCTGGAGACAGTAATTAAGGCAGAGCTGAAGTCCAGCGACCGAAAAGATCCAGAGGCTTGGCTCCTGTACCCCACCGATCTTCCATCTCACACACACCCAGCAATTGAAGGGGCCCACCCACCCCTGCCTTCCCTGAGAGCCCGGAGCTCAGGGAAGCAGGAGCAGGGAGGCCTGTCTCAGTCTCCCTTCTCCTCTCTACCTACAGGGG
8183	chr12	49295541	49295792	+	0	train	GGTACGCTTTCTGGAGCAGCAGAACGCGGCCCTGCGCGGGGAGCTGAGCCAAGCCCGGGGCCAGGAGCCGGCGCGCGCCGACCAGCTGTGCCAGCAGGAGCTGCGCGAGCTGCGGCGAGAGCTGGAGCTGTTGGGCCGCGAGCGTGACCGGGTGCAGGTGGAGCGCGACGGGCTGGCGGAGGACCTGGCGGCGCTCAAGCAGAGGTCAGGGGGCAGGGCTGGGCCGCTGCCGTCGAGGCGAGGTCGAAGCG
6479	chr10	47311215	47311466	+	0	train	GGGCAGGAATCCACAGGGAATAGGCCTTCACTGTGCACAGGGTCTGCAGGACACACAGCACTGGGCTGTTCTGTCTGTTGCACCAGCCTAAGGATTTCCAGTTCTCACCTTCGCTCCAGACTGGGCTGTGGCTGAGTCCATCCCTCTCCCTGCAGCCTGCTCTGCAGCAGCCTGGCTGCTTGTGATGTCAGCCCTCGGCACTCAGCACACAGTTTCTTCCTGCTTAGTTACTCTGCCCAGGGCAGGTCATG
15065	chrX	153872426	153872677	-	0	train	ACGGAACAGTCTCCACGGCGCCTGGTTGTCTTCCCCACAGATGACATCAGCCTCAAGTGTGAGGCCAGTGGCAAGCCCGAAGTGCAGTGAGTGATCTCTGCCGTCTCAGCCCCTGTTGCCTGCCACTGACCCCGGCCTTGTGCCTCCCTGGACTGCCTAACTTCTAAGCCCAGAAGCCCCAAACCTTGCTAATCACCCCAAGTCCTCGCCATCTCCCAGTCCATCCCCTGGGGGCCCTGGCTGCTTGCACC
11941	chr18	31598366	31598617	+	0	train	CCTGCTCTAAGGACTGTGCATGGTTCCAAAGGCTTAGCTTGCCAGCATATTTGAGCTTTTTCCTTCTGTTCAAACTGTTCCAAAATATAAAAGAATAAAATTAATTAAGTTGGCACTGGACTTCCGGTGGTCAGTCATGTGTGTCATCTGTCACGTTTTTCGGGCTCTGGTGGAAATGGATCTGTCTGTCTTCTCTCATAGGTGGTATTCACAGCCAACGACTCCGGCCCCCGCCGCTACACCATTGCCGC
9981	chr16	2093512	2093763	-	0	train	CCCTCTGAAGCCACCCCCTCCCCAGGTCTTGCTGGAAGCCCTGTACTTCTCACTGGTGGCCAAGCGGCTGCACCCGGATGAAGATGACACCCTGGTAGAGAGCCCGGCTGTGACGCCTGTGAGCGCACGTGTGCCCCGCGTACGGCCACCCCACGGCTTTGCACTCTTCCTGGCCAAGGAAGAAGCCCGCAAGGTCAAGAGGCTACATGGCATGCTGCGGGTGAGCCTGGGTGCGGCCTGTGCCCCTGCCA
4552	chr6	32189248	32189499	-	0	train	AGAGGGAACCCCCGGTGGCCCGGCTCCCCACTCCTAACCTTTTGCCGACCCCTGCAGTCTCCTGGAACAGCCCCATCCCCGGGAGCCCCCTCTGGCTCCCAGACTAAGAAACTGTTCTTGGGCTACGTTATCTTCTCCCCTAACTCTCCACCCAGCCCCCTCATTCTCTCCAGATGTGGAGACCTCCACACCCTCTCCAGAGCCCCTAAAGCTCCTCTCCACTGCTCAGCCAGACACTAGGTGCATCAAAG
9498	chr15	50262934	50263185	-	0	train	GGTGGGTGGTGGCCTGGAGAAAGACCTGCTTTGGGTCAGCTTGGGATGAGAATGCATCCTTCAGCCACTTGGCCACCAACACCCCAGCCCATCTGCTAGGCATGTGGAGGGCAGGACTTTGACTGGTATTAGGTTTCTTTTTCTTTTTTTTTGCCCAGTTGTGCCATGGCATGAGGTTTCTGATGCCAGGGCAGGGCTGGCCAGAACACCTGCATTCCAGAGCACAGCCTGGCAGAAGTGTGGAAGTCTAA
906	chr1	173910705	173910956	-	0	train	CCTGGGAAAATGGAGAAGCCAATTGAATAGCACAGGTGAGTAGGTTTATTTTCTGTTCTCCTCAGGAAAATGCAGAGCAATCCAGAGCGGCCATCAACAAATGGGTGTCCAATAAGACCGAAGGCCGAATCACCGATGTCATTCCCTCGGAAGCCATCAATGAGCTCACTGTTCTGGTGCTGGTTAACACCATTTACTTCAAGGTACTCAGAATGGCCCTGGAGAGACCCCAGGGACTTCCTCTTGCTCTT
278	chr1	109610483	109610734	-	0	train	CATGAGACAGCCCAAACTAAATAGTACTGAAGTTAAGAAACTGCTCTAAATCCAGGTTGAATGGCCTGAGCTCAAGCCTGCCAGAAATTGAGGGCAGCAGTCATCCCTATGTATTCTCCCCTAACAAGACCCCCAAGCAAGCAGTGGCTCTGACTTCTCCCAGGCCATCTCCTGGAAGGCTGAGGAGAACTGGTGGAAATCGAAAGCATAAGCATTTTTCCTTCCAGGTGCTGGGGAGTCAGGAAAGAGCA
2637	chr4	71764717	71764968	-	0	train	ACATCTACTACATCAAGTAAACACGTAACCTTAAAATTTTTATGGTTTAACATGCACAAGGGGCCAGATTATACACTGAATCTTTATGTTAGTCTTGGAAGGTCACAAAAATGAACAAAACTGGGGGAAAATACATTTATTAAAGTAAGAAATATATACATATTCAAAAACCAGGAGCAATGTCATTCAAAGTGGTCACAAATTGTAAAAATTAGTAGGATTGCTCATTCCTGTCAATTTTCAAAGAGGAT
2857	chr4	71774759	71775010	-	0	train	ATAAGAATGACTTTTTTCCAGGACTGATAGTCAAAGATAAAGATTTTTAAACAGGCATGAATAATAATGAATCTTTCGATACCATAAAAAGAATATCACTAAATTTCAATAGAGTAATTCTCTAACTCATGGACAAAATGTACTCAAAAAGTTATATGTTATTTAGAAGTTAAAGTACAATTTTTCATAGAAATAATTTAAATATGAGATGAAGTTCTTAAAATGATGCATAAAAATAGGTAGTAAACATG
8107	chr12	14883392	14883643	-	0	train	TATTTTTAGTAGAGATGGGGTTTCACCATGTTGTCCAGGCTGGTCTCGAACTCCTGGCCTCAAGTGATCCACCCACCACAGCCTCCCAAAGTGCCGGGATTACAGGCATGAGCCACCACACCCAGCCAGCTGATTGCTGTTGAATAGCTGGATTTATAAAGACTGAGCATAGGAGGAAATGGCACATCACTCTCATTTTTAATTTATTCATTATTTTTATAGTGTTTAAACTGTTCATGTATCGGCAATCT
8007	chr11	123058274	123058525	-	0	train	GGTAATTTAAGGCTTTTACTTAGAGTTAATTTCTTTCCTAGGCCGTTTGAGCAAGGAAGACATTGAACGTATGGTCCAGGAAGCTGAGAAGTACAAAGCTGAAGATGAGAAGCAGAGGGACAAGGTGTCATCCAAGAATTCACTTGAGTCCTATGCCTTCAACATGAAAGCAACTGTTGAAGATGAGAAACTTCAAGGCAAGATTAACGATGAGGACAAACAGAAGATTCTGGACAAGTGTAATGAAATTA
15549	chrX	154306592	154306843	+	0	train	CTAGGCTTTAAACATGTCTATAAACTTGAACTTTTTTTACATCTTTTTTTGTTCTTTTTTTGGGGAGGTAGAGTCTCGCTCTTGCCCAGGCTTGAGTGCAGTGGCTCGGTCTCAGCTCAATGCACCCCTGACCTCCTGGCCTCAAGCGATCTTCCCATTTCAGCCTCTTGAGTAGATGGCATTACAGGCATGTGCCACTACGACCGGCTAATTTTTGTATTTTTTGTAGAGACATGGTCTCACTATGTTGC
15647	chrX	154309494	154309745	+	0	train	CTAGAGTCTCGGGGGGCAGCCACCTATCTCCGTGATGTGGAGAGCCCTGAAGGGCCAGTCCCCATCACATCCCGTGGTGACCCCTCTGGAAGGCCTTCCCTGTGCCTTTTCCCCACCTCGGGCGGCTGCCCCACTTTCTCTCCCACAGGTCTGTTGAAAGGTTTCAGCTGCTGTTGCCTTTCCATTCTTATCCTAGGGTTGATAGCTTTTTTAAAACTTCCAGCACTTTAGTGGGGTCTGGGATGTGTGGT
1070	chr1	186675386	186675637	-	0	train	ATTTTAGGCTTGAATACTAGTGTTATTTTTGAAATGTAAAAAGGCAAATTAGTTCTAGGCTGGTGTCCCATTGAATTTTAAGCAGAGCTCCTGTTGAAATGTAGGTAAGCATCTTTCCAGCAAATAAAAATTGTCTCCGCTGGGAGTTTCAGTTTTACCTGATTTGTACCTAAGGCAAGCTGAATACAAACAGTAAATATGCCTAAAATTCTTGTTTTACAACTAATTTTACTTTCCACAGGTTGCTGGTG
3538	chr4	154606406	154606657	-	0	train	TTTTCATAAAAAATAATAAATAGATATGAAGAAATGAAGAATAATTTATAAAGATAGTAGGGATTTTATCATGTTCTTTATTTCAACTAAGTTCTTTGAAACTGGAAGTGGATAATACCAAGTTCATGCCTAAAATTAGCCCTTCTAAAGAAATCCACCTGCTGCAAAATATCCAGTAGTTTGGCATTATATGTGAAACTATCACCATCATAGCTGGCACTGTGGGTTGTGGGATCTCCTTTAGACATACA
7388	chr11	14968988	14969239	-	0	train	CATGAAGGACTGAACAGCATGTGGAATGCCAGAAAAGATCATCCTTCCCCATCCAGCCCTTCCCTGCATTGCCCTAGCCCACTGCACCTCTGAGCTCTCCTAATGAAGGATAGATAAGTGAGCTGCCCTCCTGCCTGCCCCTCCTGCTCCCAGGGTCCCCTGCCTGGTCTAACCTTCTAAGTGACTGCCCATGGGGACAGATTCTGGTGCATGGTACTGTCTGGTATGTGTTTTCCCTGCAGCCTGGACAG
15795	chrX	154314428	154314679	+	0	train	GAGATGTAGGAATAGAATTCAGTGTTCTAGTGTTAGGTTAAAGGAAGGAGGGAAGTGTGATCATCATCATACACACTAAACAAAAGCAATTCATAAAACTCAACATTTTTTTTTTGAGATGGAGTCTCGCTCAGTCACCCAGGCTGGAGTGCAGTGGTGCAATCTCGGCTCACTGCAACCTCTGCCCCCGGATTCAAGCGATTCTCCTGCCTCAGCCTCCCAAGTAGCTAGGATTACAGGCTCCCGCCATC
13946	chrX	147939348	147939599	+	0	train	TCCATAGAGAGAAGTTCAAACTTTCCTTGTGATTCAGCCTTTTAAAATTTTCAGAGTACATTATTGAAATGTGTATGGGCTTGATAAAACTTAATGCTTGTCAATGTACCTATTTTTTTGTCTCACAGATTATTCAGTGCATTTTTATCCTGATGTTTTTGTTTGTATTTTGTGTGTACTCTATTTACTTGGTTAGCAGAAATGATTTACTTTTTACTTTTGTACCCTTGGTATTTCATTTTGATTTTTTT
3873	chr5	132679384	132679635	+	0	train	CAGATGGTCCTGCATTCTAGTCCCCACTGTGCCTTTTCCTCATGGGATGACTTTATTCAGGTACCCTTTCGGCAAAATCCTCCAAGAGAAAGGAAACTGGGAGGTTCTGGGGAGAAGGCTGCTGCGTTTGCAATTGGGAGAGGTTGTTGACAGAGGTTTATGTCTGTGGCAAGCAGCCTTCCTTCAGTGGAATACTTGAAGACAGGTCTGTAGTTGAGCAAACTCACCTCCATTTGTCCTCCTGGAAAGAA
2229	chr4	71747294	71747545	-	0	train	TCCACAAAACAGTTGATACCCATAAGTATACCTTCCTTCTTGAAACAATTTACTTAGCACCTGTAGTATTATTTTCTACTTTTTGTCTTTCTATGATTCCTCTCTTCTTGTTCTATTTCATATTGTCCTCTTCCTTTACCTGTCTCTTAAATTTGTTTCCAGAGTTTTTTCCTTGTTCTTCATTTCCATGGCACTACATTTTCTCAACTGTTTTCATCAATGCTTATGCTTTTAATCTCCAAATCTCTACC
14732	chrX	149501884	149502135	-	0	train	GACCTTTTTCTGTTAGCATTGGGCCTGCTTCAAGATGACAGTAGCCACCACTGTGTGCCAGGTATGTACCGTTTACCATGCTGGCCACTCCTTTAATCCTCCCAACAGCAAGCAAGGTGGGTTTTTATCATCATCATCATTTCCCAAGGAAGACATGAAGGCTCATGGGAGTCAAGTAGCTTGCCAGAGGTTGCAAGGATGCTAAGAGGCAGCATCAGGATTTGGACCTGTGGTTGTTGGCTTCCTGTGTG
5253	chr7	76303385	76303636	+	0	train	AAAAAAAAGCAAACAAAAATTTTTTTAAAGATCATCGATGAAGAGAGAAAATGCGCTTTTCTACAGAGTCCCCTTCCCACCCACAGCCCCATCCCCAGATAAGCGGGGAGTTCCCTGGCGCGGTGCCAGTTTCTAGCCGCTGAGTGGGCGTGTGCGCGGCTCCAAGTGCGCCTGCGTACTGCTCACTCCCCAGCTCCGCGCCCTGCTCCGTTCCTCCCAAAACTCTGAATCGAAGAACTTTCCGGAAGTTT
9226	chr15	50252404	50252655	-	0	train	CCTTCCAAGTGGATGATGGTGCATTTTGACTGTACTGGGTTCTGGTGAGTGTAGCAGCCCAGCTCCGAGCACGCAGGAACGCCTTGCCTCCTCTGGAGAGACCTCAGCCACTAATGCCCATTTGGAAACCCACAGGGTCAAGGACAAGTACAAGCTGCAGCAGACCTTCAGTGTGAATCCCATCTACCTCAGGCATGCCAACTCAGGCGTGGCCACCGACTTCATGGTGAGTGGCCAGGGACGGGCAGCCT
13235	chrX	147916992	147917243	+	0	train	GAATTACAGGCATGAGACACCGGGTCCAGCTTCCAGCAGGTTTTCTTTAGGAGGGATATTTTACAATGCTGTAAGTTTTTCCTAACGAGAATTATCATAGCACTACATGTTCTGTCTTCAGTAAGTGATACAAGCTTACTGATGATGTTGTAGTTATGTTCATTGGTGGTCGGGTGTACATTGAAACTTTAACACATAATAGCCTCTCCTGTGAGCAGTGGTTCCTGTTGGTAAGATACTTTACTAAGGGA
8499	chr14	24633724	24633975	-	0	train	AAGTGTGATGCTGGATAAGCTATCAGCAGGAATGGCAGAGCAGCAGGCCATTCTCAAGAAGAGCCAGTGGGTACTATCCCTTCCCCAGAGCCCACCTTTGTCACCTGGAGAGTAGGACTTTCCTAGAAGTAAATGGCAGAGGATGGGAAACTAGAAAAGAGAAATATTAAATTATTCTAGAGTAGGCCTGGCTTCTGTTTCTGGGATAAGACAGGTGCTTCTCTCACTGTCTACTTAGGAGAGAAACCCAG
5991	chr9	130695566	130695817	+	0	train	GATCTGTGTGAAAGCTTTGAAAACCAGGTGAGAACAAAAGGTGTGTATTCCCTTTCTTGCAGCATCAGGTTGTACAACCAGGCTTTTCCTGCCCCCTCCTTATCCCCCACCCCACCCCTGCCTGCCCTGTCATGCTGTAAGTCTGAGGTTTTGACCATGACTGTAGGTGGGGTGGTGCAGATCAGTGCAGGCTTCAGAGGTTTCCCCATGTTATGTCACCACTCCAAGCCTTGGTTACCTCCTCTGCAAAG
10972	chr16	2128211	2128462	-	0	train	ACAGACAGAGGTGGCAGTGGCGCTTCCGAGTCGGGCTGCGATGTGCTTGCACTCCCCGAGGGGCTGAGGGGCCCTGCGCCCAGGTGCAGCTGCTTGGGTGCTGCCAGCCCCTCCCACCTCTCCCTCCCTGCCAGCCCCTCCCACCTCTCCCTCCCTGCCAGCCCCTCCCACCTCTCCCTCCCTGCCAGCCCCTCCCACCTCTCCCTCCCTGCCAGCCCCTCCCACCTCTCCCTCCCTGCCAGCCCCTCCCA
13157	chrX	147914613	147914864	+	0	train	CTATCTTCTGGTGTTTATTCTTAAAAACTTAAAAGTTAGATTTAGCGATCACCAGAGCCACTACTTTTATGCTTAGGTATTTGTTTGACTTAGAAAAAATTGGTCACGTGTACCACTTTATAGTGCCCTGCAGGTGTTAAGATATGAAGGCACTTTGACTTACACCTCATAAAATCTTTACAAAGTATTTTCTAAATGAATAATGATGAAATAAAGTCTTTATTCTAGGTGCATCTGCCCCACATAATTTG
7045	chr11	5226204	5226455	-	0	train	CAGTGTGGAAGTCTCAGGATCGTTTTAGTTTCTTTTATTTGCTGTTCATAACAATTGTTTTCTTTTGTTTAATTCTTGCTTTCTTTTTTTTTCTTCTCCGCAATTTTTACTATTATACTTAATGCCTTAACATTGTGTATAACAAAAGGAAATATCTCTGAGATACATTAAGTAACTTAAAAAAAAACTTTACACAGTCTGCCTAGTACATTACTATTTGGAATATATGTGTGCTTATTTGCATATTCATA
10389	chr16	2107914	2108165	-	0	train	GCCCGTCCTCAGTGCCTGGTGGGCCCCGTCCCAGCATGGGGAGGGGGTCTCCCGCGCTGTCTCCTGGGCCGGGCTCTGCTTTAAAACTGGATGGGGCTCTCAGGCCACGTCGCCCCTTGTTCTCGGCCTGCAGAGGGAGGCTGGCGGGTGTGCGCTGAACTTTGGGCCCCGCGGGAGCAGCACGGTCACCATTCCACGGGAGCGGCTGGCGGCTGGCGTGGAGTACACCTTCAGCCTGACCGTGTGGAAGG
4818	chr6	33068782	33069033	-	0	train	TCCTCAAGCACTGGGGTATGCAACTGCTTTTCTCTCCATAATCTCCTGGCATCCTCTATTCCAAAGACCTGGTGTCCTCTGCACCAGCTTTCCGCACTGGCTGGGTCTCAGTCCTCTCCTCGTCCTAACATCCAATTAACTGGTCCATAACCTTCAATTCCCACAACCATCCCAGGCCATCACCACCCTCACTGCACCTCCTGACCCTATCTCTTCATTCTTCCCCCAGAGGCCCAAGAGCCAATCCAGAT
4823	chr6	33068873	33069124	-	0	train	AGCTTCCACAAGTTCCATTACCTGACCTTTGTGCCCTCAGCAGAGGACTTCTATGACTGCAGGGTGGAGCACTGGGGCTTGGACCAGCCGCTCCTCAAGCACTGGGGTATGCAACTGCTTTTCTCTCCATAATCTCCTGGCATCCTCTATTCCAAAGACCTGGTGTCCTCTGCACCAGCTTTCCGCACTGGCTGGGTCTCAGTCCTCTCCTCGTCCTAACATCCAATTAACTGGTCCATAACCTTCAATTC
3558	chr4	154606860	154607111	-	0	train	CACTCAATGAGCAAATTTCAGCCTTAAGAAACAAAGTCAAAAATTCCAAGGAAGCATCCTACGAAAGAGGGAACTTCTGAGATCCCTGAGGAGGGTCAGCATGTGATGGTTGTATTTCCTTCTTCTCAGTACTGCAGACTATGCCATGTTCAAGGTGGGACCTGAAGCTGACAAGTACCGCCTAACATATGCCTACTTCGCTGGTGGGGATGCTGGAGATGCCTTTGATGGCTTTGATTTTGGCGATGATC
10035	chr16	2094928	2095179	-	0	train	TCTCAGCCTCCCAAAATGCTGAGATTACAGGCGTGAGCCACCACGCCTGACCAAGTTGAGGCTAGGTCATTTTTTAATTTTTTGTAAAGACAGGGTCTCACTGTCTCCAACTCCTGAGCTCAAGTGATCCTCCTGCCTCAGCCTCCTGAAGTGCTGGGATTACAGGCTTGAGACACTGCGCCCAGCCAAGAGTGTCTTTTATCCTCCGAGAGACAGCAAAACAGGAAGCATTCAGTGCAGTGTGACCCTGG
5311	chr7	99967768	99968019	-	0	train	ATTTATTCCTTCCCTGGGATAATATAATTTGTGGTCCAAAAAGAACATCATCAAAATTTCAGGCAGAATGGGCCAGGAAGGCCATTCTTTCTTGATGAGTGTCCCCAAATCATCTCCAATTAACAGACAAGGAGCTTGAGGTTAGGGAGGTGAGGGTAACACTGTCTGTAAGAGGCAGAGCTGGGACTCAAATTCCAGATTTCAGATTCCAAATCCCATCGTTTTTTATCTCTACAATGATGCCTCCCATC
2103	chr4	39457552	39457803	-	0	train	AGTTGAAAACCATGTGCTTTGATAAGGGTTAATTCTAGAATCTTAAAGTAATTCGGTAAGTGGTCTGTTGTCTTTTAAGCCTGCATATTTCTGGAAAGAATACATTTTTTGTTCTGTTACAGCTCCGGGTTGACAAATGGTGGGGTAACAGAAAGGAACTGGCTACCGTTCGGACTATTTGTAGTCATGTACAGAACATGATCAAGGGTGTTACACTGGTAAGCAGATGTATCAGACTTCCTTGTTTTGGA
18	chr1	67685999	67686250	+	0	train	ACCCCTCGCCCTTGCCCGCGTGTAGGATGGATAAGGTGGGGGATGCCCTGGAGGAAGTGCTCAGCAAAGCCCTGAGTCAGCGCACGATCACTGTCGGGGTGTACGAAGCGGCCAAGCTGCTCAACGTGTAAGTGGGGCCCTTGCGCGTCCCCCATGGCACCCCTTCCCGCCCCAGCCCGGGAGGTCGCCTTGGCTGGGCGCCCCTCGCCCGGCCGCGCCACTTCCTGTCGCTTTTCTGCCTGTCTCGGAAG
14207	chrX	147948380	147948631	+	0	train	CTGTTTGTTTCTCACTGTTTTATTCTACTTAGAAGAAGACATGATAGGATTGTGAGTTTTTCTCTAACTTTGGATGCAGTGAGATGACCAGTGTGTTCCAGTTAAAGAAGAAGAGTGTTTTAAAATCATAAACCAAATAAAGAATCCTACCTTACATTAATTTCTTGCACTTCTTCTGTTCTCATCTCCATTTCTCTTTTTAACATGTGGAATTTACACTTCAGGTTTAAATTTCTTGTCAGGCCAATTAC
7958	chr11	119091151	119091402	+	0	train	TATCTGTAAAGTGGGGATAATAATACTACCTTCCTCACAGGGTTGTTGTGAAGATGAAATGAGCTGACATATGGAAAGTACTTTTAGAGCAGTGTCTGGCATGTAGTAAGTATGATGTAACTGTTAGCTGTTAACATTAAGCTGAGAGCTGGAAGATGACTGAAAGTCAGCCAGCTAGAGAGGGAAAGACAGACTCAGGCAGAGGGAACCGCACGAGGCCCCAGATTGCCCGACACTGTGGTCCTTAGCAA
4717	chr6	32940486	32940737	-	0	train	CTGGGGACTCTCCCTTCCCCTGCTCCTGTTTCAGGGTAAGGGTGTTCCGTTTTGTAATTGCATTTGACACCCCAGATAGTTTGTCTCCCTGGTATACATTCCTATAGCACTTTGTACTTTGTAGCAATTTTAATGTAATTAATCTGTATAATTATCTGTGCAGTGTATATTCCCTGCTGGAATATAGGCAAGGACAATGTTCATCTTATTTATTGCTGCCTCCTCAGCTCCTAGCACAGTGCCTTGCATGC
9234	chr15	50252803	50253054	-	0	train	AGGATGTTCTAGGCTGAGGCCAAGAGGGTGCTGGCCCTTTACCCAGTGGAAGAAAAAGGCAAGGGTGAGGCTTTGTAGACTTTCCCATTCGGAGGTAATGATGCCTCCTTCAGCCCCACTTCTTCAAACTGACTCCACCTGCTCCCATCTCCACACCCCTCCGTTTCTCATTTAGCCAAAGCACATCATCCTTGCAGCCCACCACTCAGCTTGGCCAGGTGCGAGACATCTTTCCCCACTTCATACCTCCC
9753	chr15	90894243	90894494	+	0	train	GTTCAAGATCAGCTTGGACAACACAGTGAAACTCCATCTGTACAAAAAATACAAAAATAGACTGGGCACGGTGGCTCACACCTGTAATCCCAGCACTTTGGGAGGCCGAGGCAGGTGGATCACCTGTGGTCAGGAGTTTGAGACCAGCCAGACCAACATGGTGAAACCCCATCTCTACTAAAAATACAAAAATTAGCCAGGCATGGTGGCACGTGCCTGTAATCCCAGCTACTTGGGAGGCTGAGGTGGGA
10174	chr16	2099327	2099578	-	0	train	GCCGCCACTTTCCAGTGCTGCAGCCAGAGGGAAAGGCGTCCACCAAAGGCTGCTCGGGAAGGGTCAACACACTTGAGCAGCCTTAGCTAGACTGACCAGGGAGAAAGAGAGAAGACTCAGAAGCCAGAATGGTGAAAGAACGAGGGCACTTTGCTAAGCAGACGCCACGGACGACTGCACAGCAGCACGCCAGATAACTCAGAAGAAGCAAGCACGCGGCTGTGCACGCTTCCGAAATGCACTCCAGAAGA
15688	chrX	154311129	154311380	+	0	train	GACGGCCGGGACGTGGAGGCACTGTGCCAGGTATTCTGGCAGGCTTCTCAGGTGAAGCACAAGCCCACTGCTGTGGTGGCCAAGACCTTCAAGGGCCGGGGCACCCCAAGTAAGCAAGCACTTTCCTCCTGCTCCTGGTTGTTAAAAGCATCTGTCCGCTCAGCAGCAGAGGCGGGAGAGGCTTAGAGGGGCCTAGGCAGATGACTCATTTAGTGAATAGCAGTTCTGCCTGGAGTATATGTTGGAGGCTC
15545	chrX	154306445	154306696	+	0	train	TGAACCATAACATCAAATGGTTTGGCAAAAAATAAATAGAATTTACATGTGTACGCGCATGTGTGGAGAGAGGGATGAAGTAAATGCCTTCCATTGCTTTCAGTTGGTTATTGGAGGTAAGCAGTATACAGGAAGGAAATCATTGTGCTAGGCTTTAAACATGTCTATAAACTTGAACTTTTTTTACATCTTTTTTTGTTCTTTTTTTGGGGAGGTAGAGTCTCGCTCTTGCCCAGGCTTGAGTGCAGTGG
6281	chr10	47302279	47302530	+	0	train	AGATGAAATGAGGTCATATGTGCAAGCTCAGAGCCCAGCACTGGCCAGATTAAGTGTTGATGTACGGGATTCCCCTGGGAGCACAAATAATCGTGCTAAGTATTGTGTATTCCTGTGGGTTTAAACTTGACCTTGCTTCTCTATTAACTCAATCCTCAAAGCCTTGTAGTTGAGGAAGCAAGGCCAGCAGGGTCACTGCTGCGACACGGGTGAGCCTGCCCACTGAGCCCATCTACTTGCTGCAGCCAGGC
5670	chr7	99975084	99975335	-	0	train	TGGCTTCTCAGTTGTGTGAGTTTAAAATTCATGACATTTACAAATTGTCAGAAAAGGTGTTATATGTTTGTTATATAACAATCACTTTGGAATGTTAATCTGATTCTGTGCCAAAATCTGAATTACTCAGGGTTCTCCAGAGAAACAGAACTAATAGGTGGTACACATATACATATATATGTACGTACACATACATACATACACTGTATACACATGGATACACACACACATAGGAAGAGATTTACATATAT
5468	chr7	99971423	99971674	-	0	train	GGATCAGCAGGATGGAAATAGTCCCAATCCCAGGGGAAGAACAGGAGACACAGCAGAAACACAGACATGTCCACATCCCACCCACCCCACAGCACAGGTGCTCCCCGCTTCCCCATCAATTGCCCCATCCTCATCCCAGGCCTCAGGTCACACAGGAAGTGATGGCAGAGTCACTTCCTATCCAGGCACCTATGACCTCTCACCTCCACACCCCACCCATCGGAGGCTGATACCCCCGTGAGAAGGCATCA
8860	chr14	102081756	102082007	-	0	train	TGCTGCTTGGAGGTATTAAAGTATGTTTTTTTTAGGGATAAGTAAGGTCTTACAAGAGCAAAGAAATGAAATTGAGACTCATATGTCCTGTAATACTGTCTTGAAAGCAGATAGAAACCAAGAGTATTACCCTAATAGCTGGCTTTAAGAAATCTTTGTAATATGAGGATTTTATTTTGGAAACAGGTATTGATGAAGATGACCCTACTGCTGATGATACCAGTGCTGCTGTAACTGAAGAAATGCCACCC
15856	chrX	154316838	154317089	+	0	train	AGCTCACTGCAACCTCTGCCTCCCAGGTTCAAGCGATTCCCTTGCCTCAGCCTCGCAGGTAGCTGGGATTACAGACATGCGCCACCATGCCTGGCTAATTTTTGTGTTTTTAGTGGAGACAGGGTTTCACCATGTTGGTCAGGCTGGTCTCCAACTCCTGACCTCAGGTGATCTGCCTGCCTCAGCCTCCCAAAGTGCTGGGATTACAGGTGTGAGCCCCCGAGCCTGGCTCATTAAGCTGTTTTCATTTT
15407	chrX	154301979	154302230	+	0	train	GTGGGCACTCCTGAAACTGTCAGGATCATGTTGGTTTCCTCGCTGTTTGCCGGCTATTGGCATTTGTGTCATCACACACAGTCCGATGAGGTGACAACAGAAAAGCATATCTCACTTCTATCGTAGGTTTCCATTCACTTTCGGGTTCTCAATCGCCCTGTGTTTCCTCAGTGATACTCCAGACTTTCCAGATGTTCTGGTTGACCCATTCCTCTTTCTTTTTTTTTTGCAAAACAAAGCAAAAAGCAAGA
2070	chr3	129170653	129170904	-	0	train	CCTGATATCCTGTGGCTTATGGTACCTGGGCAGTTTTCACAACTGGACTTTTTTAATATATAAAAGTAAGAGTGTTATAATTTGAAACTTCCAGAGACTTCATAGAAAGCTCTGTAATATACATAAATCTTTTATCATGTAACCAGAAATCTTTGCCTGTTTGTGACATGTAAGTGTATAATTTGATAAATGTTGTTGTGTACATATCTGTGAAACCTTAGGGGTTAATTGCATGAAAACAAAGATCAGGC
8130	chr12	14884052	14884303	-	0	train	TAACATAATGAATCATGGATAAATATTGATATAATGAATCTTTTTTTTTTAATTTCAGAATCACATGAAAGCATGGAATCTTATGAACTTAGTAAGTGAATATTTAACTTCTTTATTCAAATCCCTTGCATTAAAGAACCTCTTCTTATTTTTAAATAAACAAGATGGAAAGATATATAACAGGGAGGGAAAAGGGGGCCTCTTTTGGAAAACTAAAGTAAATTTTTAAATCTAATGACTATAAAAATTGC
14331	chrX	149486401	149486652	-	0	train	CATCCCCAAGATGGGGAGGGCTGTGGTACAGTCTACAAGAGCCTGCTCCAGAGCTTAGAGTAAGGGGCTGCCCCCGTAACAGACGAGGAGAATCCTTTCCCTCGATGCGCACTTTTGGTAGGGACTCAGCCTTCTTTGTGGGGGGCACATTTTCCATTTTGACAGCTCATCAATCTAGTGAACCACTAGATCTGTAAAATCTACCAGGTGATTCTACTCTGCCCTCCCCAGCAGTGAAGGGCAAGTTTACT
5204	chr7	45919008	45919259	-	0	train	GTAGTATAGGTTGATGTGAGTTATAAGATTATAAAAAGATCTAAGTGACTTCTAGAATCTATTTGACAAAAAAAGGTAAATTTTCGACAGTCAAAAGTCACAATTATCTGTTGCTTAAATAGAACTGTTTTGTCTTCATGCCCTAGTCTGCAGCCCAGGCATTAAGAAGAAACCAAGGAAATTTAAGAAATTACTCAAGGTTCTTAGAAAAGAAGTATAAATACGTTTATTTACATGTTCTTAGAGTATTT
10317	chr16	2105131	2105382	-	0	train	CCTGAGGGTCCACACTGTGGATGACATCCAGCAGATCGCTGCTGCGCTGGCCCAGTGCATGGTAGGATGGCCCCACCTGCTCACCCTGCCCCGCATGCCTGCCAGGGCACTGGGTTCAGCCCCCCAGGGCAGACGGGCAGCTTGGCCGAGGAGCTGAGCCTCCAGCCTGGGCTCCTTCCTGCCATGGCGTTCCTCGGTCTCTGACCTGCTTCAGTAGCCTCAGCCGTTCTGTCCTGTGTGAACGCAGGGTG
1391	chr1	225840428	225840679	+	0	train	AGGCAGTGAGGGAGGGCCCAGCAACCATGTCTGCCACCAGGGTCCTGTTAAAGGACAATACTTTTTCAAAAAGGTAAATTTATAAAGGAACCAGTAAAATTTTAAAAAGAAACAATTCAAAGACAAATAAGAACACTGAAATTAATTTGCTAACAGACACAAAACTTGTCAACATCCACAGAGTAATATCAATTAAATCTCCATGGGACAAAATTCATTTGCCTCTATGAACAAGAGTCATTCGCATTACC
10492	chr16	2112423	2112674	-	0	train	TCTGTTGGGAGGTAACTGGGTGCACAGGAGCCCTGAGGCTGCACGGGAGCCGGGAGAGGCCTCAGCACAGCCGGGTGGGCCCTGAATGGAGGCCCGGGGCGTGACTGCAGAGTGGAGCCTCGGCTGGGTCCCAAGCACCCCCTGCCCCGCCACCGCCCACCCCTGTCCCGGTTCACTCACTGCGTCCCACCGCCCCGGCAGGTGGACCTTTGGGGATGGGGAGCAGGCCCTCCACCAGTTCCAGCCTCCGT
14823	chrX	149504744	149504995	-	0	train	TCCCTCCCTTCCTTCCTCCTTCCTTCTTTCCTTCCTTCTTTGTTTATATCCATTCTTTTTACCCTTCCTCTCTCCTACCATTCCTTCTTTCATCCATCATTCGTTCCCTCCCTCCATTTTTCACTCCTTCCTACCGTCCCTTCATCCCTCCCTTCCCTCTTTCCATCTATTCATCCATCCATCTCCTATCCCTCCGTGCTTCCCTTCCTCTTTCCCCCCATCTTTCCCATCTCTATCCATCCATCTTTCCC
10291	chr16	2104071	2104322	-	0	train	CCCTATTACCATCCCTTTTCTCCATCTCTCTCCCCTTTTCTCCATTTCCCCCCCCGTCCTCCCCGTCCTTTTGTCCATTCCCCTCATCTTCCTCATCCCCCTCATCCCCCTTCCCCTCCCTTATCCCCCTTCCCCTCCCTTTCCCCCTGCTCCTCTTCTTCTCCCTTCTCTTTTCTCTACCCTTTTCCTTCCTTTTTCCTCCCTCTCCCCATCATCCCCCTCATCTTCGTCCTCATCCCCATCACCTTCCC
10616	chr16	2115549	2115800	-	0	train	GCCGGCCTGGGTCAGTCGGCTGGCCGGAGACGGACGCAGCACTGGGCTGGGAGTGCTGCCCAGGTGGGGAGACCTGTCCTCACAGCAAGGCCAGGATTGCTGGTGCAGGCAGTTGGGCATCTCTGACGGTGGCCTGTGGGCAAATCAGGGCCCCAACACCCTCCCCTCCTCACAGGGACCCCGGAGAACGGCAGCGAGCCTGAGAGCAGGTCCCCGGACAACAGGACCCAGCTGGCCCCCGCGTGCATGCC
16314	chrX	156003369	156003620	+	0	train	TGTCAGATCCTCCCCAACCCCCGAGCTCAGCTCTGGCCTGAAGTACTTACCGTGGGCTCCTGATGGTCACTGTCTCCAGGGCCAAGGTCTAGAACCTTCACCTGCCTCACCAACAACATTCTCAGGATCGATTGCCACTGGTCTGCCCCAGAGCTGGGACAGGGCTCCAGCCCCTGGCTCCTCTTCACCAGGTGAGCATGGAGGGCCATGCCCACCTGGACAGGGATGAGGGTGAGTTCCCCAGGATTGAA
9638	chr15	90888902	90889153	+	0	train	GAGTAGCCAAGTAGCTGGGACTACAGGCATGTGCCACCATGCCTGGCTAATTTTTGTATTTGCTTTTTCAGTAGAGATGGGGTTTCACCACGTTAGCCAGGCTGGTCTCGAACTGACCTCAGGCAATCCACCCGCCTCGACCTCCCAGTGTTGGTATTATAGGCGTGAGCCACTGTGCCTGGCCCACTGGATCCTTATTACAACTGCCAGTGTCCCTCTTATATATATCAGGAAATAGAAGATTAGGGAGA
4005	chr6	29663675	29663926	+	0	train	CTGATAATGTCTGCTTTTCTCTCTTTTCTAATTATTTGTGAAAGGAAAAATGTGGGGGGTTGGGAGAAAAAAACCCTTAAGTACATACTCGCTAAATCACATTGCTACAGGTAACTTCCATTAAGAACTTGAAAGTAAAGGTAGCTGCATTTTCCCCTAGGGAACACAATGATAGACAGGAGCCTTAGTCTACAGCTTGAAGGATTGTAATTATACCTAAGCAACCCTCCTGGACCAGTTTAATGTTATTA
8389	chr14	24574672	24574923	-	0	train	CATCCCCTCGATTCTGGTTAGCTGCAGTCTTGCCCTCCCCGTGCTGTCTGCCTACCCTGCAGAGCTGGTGGACCATAGCTCCTGCAGCCCAGACCTACCTCTTGCTTTTGCAGCAATATAAATGTCACCCTGGGCGCCCACAATATCCAGAGACGGGAAAACACCCAGCAACACATCACTGCGCGCAGAGCCATCCGCCACCCTCAATATAATCAGCGGACCATCCAGAATGACATCATGTTATTGCAGGT
9835	chr16	180603	180854	+	0	train	GCCCCCAGGGGCCCTCCCTCCCCAAGCCCCCCGGACGCGCCTCACCCACGTTCCTCTCGCAGGACCTTCCTGGCTTTCCCCGCCACGAAGACCTACTTCTCCCACCTGGACCTGAGCCCCGGCTCCTCACAAGTCAGAGCCCACGGCCAGAAGGTGGCGGACGCGCTGAGCCTCGCCGTGGAGCGCCTGGACGACCTACCCCACGCGCTGTCCGCGCTGAGCCACCTGCACGCGTGCCAGCTGCGAGTGGA
1738	chr2	96144168	96144419	-	0	train	CCGGGGCTGGGCTGGAAAGGCCTCACCGCCCCTCTGTTCCCTCCAGGGTGGCCCTGTGGAGATCTTGCCCTACCTGTTCCTGGGCAGCTGCAGTCACTCGTCAGACCTGCAGGGGCTGCAGGCCTGTGGCATCACAGCCGTCCTCAACGTGTCCGCCAGCTGCCCCAACCACTTTGAGGGCCTTTTCCGCTACAAGAGTATCCCTGTGGAGGACAACCAGATGGTGGAGATCAGTGCCTGGTTCCAGGAGG
12869	chr22	20784613	20784864	+	0	train	CCTCCCAGTACTTGGAACCTAGGAGGCACTCAAAAAAAGATTGGCTCAACTCTTCCCTGCCCAGGAAATTCCAAGGTCCTCTTAGCCTACCGAGGACACATCATTCATGATTTCCTCTATTATTATTCGTTACTTTGTAGTTAAAACTGCAGGTGTTAAGTACTTATTGAGATTATTATTGGGTCATGGCAGAAAGAATGGAGAGGTCTTATTTCTGTCTTACTGGATACTGGCTAGGCCCATATGAAGAA
13500	chrX	147926037	147926288	+	0	train	TTTAAGTAGCAGTGAATCTTCTTGGTATATTTTTAAAAACCTACAAGCTTTAGTTTATACATATTGGTAAAATCTCTTTTTCACAGGTTATACCGTGAACTACCTGCTTATCTCCCGTTGAGCTCTTTGACCAAAATGTGCTATGGAGATCTAACATGTGAACAGAAATGTTGTGTTTGTGATTTCTTTCTCCATTAGAAATTGGTTTTGTATTTAAGGCGCTTACAGTGCCTCATATAGTTTGTGTGGTA
5585	chr7	99973705	99973956	-	0	train	TGTGCTATCAAATAGTAGGTCTTATTCATTCTTCTTTTTTTTTTTTTTTTTGTGACAGAGTTGCCCAGGCTGGAATGCAGTGGTGCAATCTTGGCTCACTGCAACCTCTGCCTCCCGGGCTTAAGCGATTCTCCTGCCTCAGCCTTCTGAGTCGCTGGGACTACAGGTGTGTGCCACCACGCCCGGCTAATTTATGTATTTTTAGTAGAGATGGGGTTTCACCATGTTGGCCAGGCTGGTTTCGAACTCCT
3960	chr6	29661078	29661329	+	0	train	ATGATTAAGATGTGGACAAGGTGAAGCCGATGGAGGGGGAGCTTTGAAAGTTACTTGCTATTTAATTGAGGAACTAAACTGCTTTGAGAGCCTGGGGGTCAGATCCTCTGCCTTTTCCTCCTCCCCACCTGCAGTGCAAACATCAGACAATTGATCACTATTGTATCTTGGAGGTGGGAGTGACCATTGCAGTGCTGGGACCAGAAGATGGCATTGTATGTGGAACAACAAAGCACTATTTCTAGAGACTG
6412	chr10	47307124	47307375	+	0	train	CAACTTTGTTATCTGTCTCCCCTCTAGGCTGAACTCCTCCTGGGCAACTTTGTTATGACCCTGGTCAGCCCAACTTAGCCTGGAGTAGGAGTTTTGCAAATCTTTATCAGGAAAACAAAGAGAAGAAGAGGAAGAAAGAAAGGAAAAGAGGGTGGATAGAAAAGGGGAAAGAGGGGAAGAAAAGAGAGGAGGGAGGAAGGCAGAGAAGAAAGAAGAGAGAAAAGACATGAATCACAGAGCACCAGCACGCA
5958	chr9	130694347	130694598	+	0	train	CATGCACCCATTTATACACATTTTGAATCTAAGGGGTGTAGTGCAGTGTATTGTGGTTAAGAGGGAAGGCTCTTGAGTCAGACAGTTGGGGTTCCGGTCCTGCTACCATCACATCTTATTAGCGGCATTGCCTCTCCACGGTCAGTTTCCTCATCTGTAAGATAGTGATAATAACAGTATGATACGGTAATGAGGATTAAATGAGACAATTATGTAAGAAAAGAAGTAAAACCAGCCATTATATCATCATA
4234	chr6	31161551	31161802	+	0	train	CGGAGTTCTACTTCATGTTCCAACAAGTACGAGTCAAGCCTCAGGACTTTGCTGCCATTACCATCCCACGGTCTAGGGGAGAAGCCCGGGTTGGGGCTGGTTTCCGGCCTATGCTGCCCTCCCAGGGGGCTCCACAGCGGCCTCTCAGCACCTTCTCCCCTGCCCCCAAGGCCACACTGATCCTAAACTCCATAGGCAGCCTCAGCAAGCTCCGGCCCCAGCCCCTCACCTTCTCCCCTAGTTGGGGTGGA
11697	chr18	31593100	31593351	+	0	train	TCACATCATCTGCTAAAGAATTTACAAGTAGATTGAAAAACGTAGGCAGAGGTCAAGTATGCCCTCTGAAGGATGCCCTCTTTTTGTTTTGCTTAGCTAGGAAGTGACCAGGAACCTGAGCATCATTTAGGGGCAGACAGTAGAGAAAAGAAGGAATCAGAACTCCTCTCCTCTAGCTGTGGTTTGCAACCCTTTTGGGTCACAGAACACTTTATGTAGGTGATGAAAAGTAAACATTCTATGCCCAGAAA
8174	chr12	14885338	14885589	-	0	train	TCTCCGCAGAAACTGATCTGTTCTATTAAGTCTTTTTTATATCCTAAATATCCAGAGTCTTATGCAACTTAACAGGCAAACCCGTTCAGTGGTAAGTCTCTGTATATCTAGAAACTCATATTTCAGAAAGAAGATACCAAATTCCCAGCCCCCTGCATCCTCATTTTTAAGGATATTTATTTAGACTTTGGTATCAATGGGTTAAGGGTATTGTTTAAACCACTTGCCTTTGAGAAAATCCATTTTTATGT
5087	chr7	44062888	44063139	-	0	train	TTTTCACCACTGGGCTATTTCTGTAGGCTGCTTGGTCTAACTCAGTTACTCCTTGACCTTTGGCAACATTTCTGTGGCCTCGTTCTCAGGGCTGGGAAGGAATTGGTGCCAGGGGAACTGGCTCTGTGGACCATAAAGGTCACATAGTGTCTGCTGTGTAAACAGGCTGGGGACAGAGGGGCTAAGGACACCTATTCCTTCCGGCATAGGGATGTCAGACCAGGCGATCATGGAGCTGAACCTGCCCACGG
12581	chr19	50877443	50877694	+	0	train	GGACATCCTGCAGAAGGTAGGAGTGAGCAAACACCCGCTGCAGGGGAGGGGAGAGCCCTGCGGCACCTGGGGGAGCAGAGGGAGCAGCACCTGCCCAGGCCTGGGAGGAGGGGCCGGGAGGGCGTGAGGAGGAGCGAGGGGGCTGCATGGCTGGAGTGAGGGATCAGGGGCAGGGCGCGAGATGGCCTCACACAGGGAAGAGAGGGCCCCTCCTGCAGGGCCTCACCTGGGCCACAGGAGGACACTGCTTT
10852	chr16	2123314	2123565	-	0	train	GGGGGGGCTGGCGCGAGGCTGCCTGGCTAGGCCTTGGCGTTCCCCCAGAACGGCGATGGCAAAAGCAGATGGAGACGTGAAAAAGTACGGGAGCAAGCGAGGTGAGGACTCCACGGGGACCCCTGTGCTGTTCCCTGTCCCTGAAGCCCACACCTGAGTCCTGCCCAGGGCAGATGCTTCCACACCCAGGGGGCACCTGAGTCCTACCCAGGGCAGACGCTTCCACACCCTGGGGGCTGGGGGACTGCACC
8619	chr14	74893823	74894074	+	0	train	GTCAGGTGCAGACATAAAGCCTCAAGGAAGTATGGTCAGGATTGTGGTTCCTCTGCCTGAGAATACTTCCTAAAAATATGATAGCCTCGAAAGTTTGTTGCTCATGAGGTTCTGTTCCACAAATAGCTATTCTTATGCAAATCTTTATCAACAGAAAGTGCCCTTTTTCTTATGATTCTTATGTTGGCTAGACTACACTACACTACATTCTTTTATAAATGGGTTTTGGTTTTTAAATCACTGTTTCTGAG
15523	chrX	154305794	154306045	+	0	train	GACATCCCACCCCTGTCAGTTAGCTTCCTCTTCTATGTTTCTGCACATCTGACCTGAAACTGATGGGCACTTCTCATGAGATTGGGCAAAATACCAGCCACTTCATGAAGTATCATGGGTGATAGGAAAGAAAAAGAACCTTAAAAACAAGGTCAGTTTGAGAATGTGTGGCTGGTTTGATGCCATCTCAAAGTACACTGTGTCCCGAAGCAAATTAAAACCTGGCTAGGGGCCGGGTGCAGTGGCTCACG
3243	chr4	121818617	121818868	-	0	train	TATTTGCCATCAGTTATTGCTGGAGCTGCCTTTCATTTAGCACTCTACACAGTCACGGGACAAAGCTGGGTATGTATTACGGTCTTCACACACCTATCTTGTGACTGAAATGCCTGTGCCAGATTAAATAATAATTGTCCTACAAAACTGAGGGTTGCCTAGATGTTCTTACTTGGAAAAACTTCAAATATATATAGTTGACCTTTGAAAAACACAGGTTTGAACTGTGTGAGTCCACTTACACATGGATT
13970	chrX	147940052	147940303	+	0	train	TGGGAGGCTGAGGCAGGAGAATGGCGTGAACCCGGGAGGTGGAGCTTGCAGTGAGCCGAGATCCCGCCACTGCACTCCAGCCTGGGCGACAGAGCGAGACTCCGTCTCAAAAAAAAAAAAAAAAAAACAAAAAAGAAAAAAAAAAAAGAAAAAATTTTGATCAGGATAAAGCCTTTCATAAATGAATTAGAGCATCTTATACTACTCATTTTGTTTATACATCTCATGGTAAAGCTTGTATAATTTTGAAT
14923	chrX	153865273	153865524	-	0	train	GAGGTCTCCTGTTCGCTTGTGCAGGTGACGTACTGGAGGGAGGGCAGTCAGAGGAAGCACAGCAAGAGACATATCCACAAAGACCATGTGGTGGTGCCCGCCAACACCACCAGTGTCATCCTCAGTGGCTTGCGGCCCTATAGCTCCTACCACCTGGAGGTGCAGGCCTTTAACGGGCGAGGATCGGGGCCCGCCAGCGAGTTCACCTTCAGCACCCCAGAGGGAGGTGAGTCCTGCACCCCACGCCTCAT
11259	chr16	28934039	28934290	+	0	train	TGATCCACCCGCCTCGGCCTCCCAAAGTGCTGGGATTACAGACATGAGCCACAGGGCCGGGCCAAGCCTAATTTTGTATTTTTAGTAGAGATGGGGTTTCTCCCTGTTGGACCAGGCTGGTCTTGAACTCCTGACTTCAGGTGATCTGCCTGCCTTGGCCTCCCAAAGTACTGGGATTACAGGCATAAGCCACCGCACCTGGCCTAGACTTCAAGTCTTTCTTCCCTCGCTTCCAAGACACTACTTTTCTG
3456	chr4	122455624	122455875	-	0	train	ATTTATGGTTTCATTTAAAAATGTAAAACTCTAAAATATTTGATTATGTCATTTTAGTATGTAAAATACCAAAATCTATTTCCAAGGAGCCCACTTTTAAAAATCTTTTCTTGTTTTAGGAAAGGTTTCTAAGTGAGAGGCAGCATAACACTAATAGCACAGAGTCTGGGGCCAGATATCTGAAGTGAAATCTCAGCTCTGCCATGTCCTAGCTTTCATGATCTTTGGCAAATTACCTACTCTGTTTGTGA
11700	chr18	31593113	31593364	+	0	train	TAAAGAATTTACAAGTAGATTGAAAAACGTAGGCAGAGGTCAAGTATGCCCTCTGAAGGATGCCCTCTTTTTGTTTTGCTTAGCTAGGAAGTGACCAGGAACCTGAGCATCATTTAGGGGCAGACAGTAGAGAAAAGAAGGAATCAGAACTCCTCTCCTCTAGCTGTGGTTTGCAACCCTTTTGGGTCACAGAACACTTTATGTAGGTGATGAAAAGTAAACATTCTATGCCCAGAAAAAATGCACAGATA
8942	chr14	102085998	102086249	-	0	train	TTTCTGAGAGAGCTCATTTCAAATTCATCAGATGTAAGTCACTTATTAACCCAGAATCGGATTTTGGTTTCAGTGTGAACTTCTTGGGGGTGCTGTATGCTTAAATTAATATTTTTTGTTAACAGGCATTGGACAAAATCCGGTATGAAAGCTTGACAGATCCCAGTAAATTAGACTCTGGGAAAGAGCTGCATATTAACCTTATACCGAACAAACAAGATCGAACTCTCACTATTGTGGATACTGGAATT
3262	chr4	121819442	121819693	-	0	train	CAGTTGGTGTGGATTTTGGATTCATTCCATATTCTGCTAATGGGATTTCAGAACTGGGATAAGGAAGCTTGCTTCTCTTTTAGGACTGATTACTTAAAACTTTTTTCATTGTAGAAAGTTTGAAGAAATATACCCCCCAGAAGTAGCAGAGTTTGTGTACATTACAGATGATACCTACACCAAGAAACAAGTTCTGAGAATGGAGCATCTAGTTTTGAAAGTCCTTACTTTTGACTTAGCTGCTCCAACAG
9053	chr15	50246275	50246526	-	0	train	CACCCGTGTCCTTCACCTGAACTACCCTCTCCTCTCCAAATGTTGTGAATGGATTAAATTCTGCTTCCACTAGAAGCTTTTCTGCCTATTCCATCTGTTCTGGTTTCTGCTTCATTAGCATTCCTGTGATCACCTACTCAAACACCTTTACAAAACTGTGCTCAAAGTTCTTCCAATGTCCCACAATAGAAGAAATGGAATTAAAATAAAGTTGTTTGATATGATTTGATTTTATCTCTTCAACTTGACGG
12472	chr19	44929138	44929389	+	0	train	GAGAATCACTTGAACCTGGGAGGCGGAGGTTGCAGTGAGCTGAGATAATGCCATTGCACTCCAGCCTGGGCAATAAGAGCGAATCCACGTCTCAAAAAAAAAAAAAAAAATTGAAAAAAAAAAAGATGGTCTTGTGGGGTAATGAAGGACACAAGCTTGGTGGGACCTGAGTCCCCAGGCTGGCATAGAGCCCCTTACTCCCTGTGTGATCTTAAGAGAGAGGCATTACTGTGAGCCTCAGTTTCCTTTCC
4355	chr6	32181221	32181472	-	0	train	GTGGGAGGATCAGGGCTGGGAACTCTAGCCCTGGCCCTGGGGATCCTGGGAGGCCTGGGGACAGCCGCCCTGCTCATTGGGGTCATCTTGTGGCAAAGGCGGCAACGCCGAGGAGAGGAGAGGTGAGTGGAGAAAGCCAGACCCCTCAGACCTAGGGCTTCCAGGCAGCAAGCGAAGAGGGGTCGGGGGGTGGAACGACAACGTGCCGCATTCCCCCCAATCTTTCTCCTCAGGAAGGCCCCAGAAAACCA
6304	chr10	47303410	47303661	+	0	train	CCAATTCTGCGATCTGGGCAAGGGACTGAACTTGTCCATGACTCAGTTTTTTTCACTTCTAAAATGGAGCTAATAGTGCTCCCCTCCTTATAGGATTTTTTTTAATGGAAACTGAGATATGTCAAGTGCTTACACCATTCCTGACACAGGTAAGTGTCAATGGTGTATTAGCTCTTGCTGTTGATATACCAGCCACCCCTTCCTTCCTTCCCTCATTGCTCCCTGAAGTGAATTTCGTTCCTTATTAAATC
7921	chr11	119089686	119089937	+	0	train	GACCTGGTTGTTCACTCCTTGAAGGACCTGCCCACTGTGCTTCCTCCTGGCTTCACCATCGGAGCCATCTGCAAGTAAGAGTCTTGCAAGTAAGGGGCTTGGGCAGGGGTAGGCATCATGTGAACCTTTGCCTTTCCCTTTGGGGCCTGACCCTCTGCTTCAGGGTTATCTCCTCTGCCCTGAGGAGTGTTGACTGGTGGCAGAAAACTCAAGAAATACCAGTGAGTTGGCAATCGAGAGAGAATAGAGGT
8278	chr12	57232285	57232536	+	0	train	GCTCAACGTGAGTGCTCTAGGGTGTGGGGAGGGGCTCTTGGCCCTGGTGGTGGTCCTCCCCTGGAGAAGCTGAGGGCCTGGAGCGCCGGGCCGTCCTTAGGGTTAAGGAGGAGAGTGAGCTGCCCTGCTTCCTTCTCAGGGCTTTAGCTGTTTGTGTGTCTGTCCAGCCCAAAACTGGCCTCATTGACTACAACCAGCTGGCACTGACTGCTCGACTTTTCCGGCCACGGCTCATCATAGCTGGCACCAGC
942	chr1	173911823	173912074	-	0	train	CCACCCATGTTAACTAGGCAGCCCACCAAACCCACCACCATTTTTTTTTGACTTCTATAGGTATTTAAGTTTGACACCATATCTGAGAAAACATCTGATCAGATCCACTTCTTCTTTGCCAAACTGAACTGCCGACTCTATCGAAAAGCCAACAAATCCTCCAAGTTAGTATCAGCCAATCGCCTTTTTGGAGACAAATCCCTTACCTTCAATGAGACCTACCAGGACATCAGTGAGTTGGTATATGGAGC
5736	chr7	101131944	101132195	+	0	train	CTACTTCAACGGCCAGTGGAAGACTCCCTTCCCCGACTCCAGCACCCACCGCCGCCTCTTCCACAAATCAGACGGCAGCACTGTCTCTGTGCCCATGATGGCTCAGACCAACAAGTTCAACTATAGTAAGTCCAAGAGCCCCTTCCCCACAGCCCACAGCAACTGCATCTCATTCCTGGGGTCTCCCAAGGAATACCCAAAATGTCACCCTCTGAGGGAGGAAGACCACAGGGAATGCTCCCCTTTAAGGG
6035	chr9	130696758	130697009	+	0	train	AACAGGTTTTGCTACTGGACTTGTCCTGGAGCGTGAGAGAGAGAGAGGAATCAAGGAACTCTCCGAGGTTTTTGGCCTGAGCAACTGAGGGGATGGAGTCACCGTTGACTGAGATGGGGACAACTCATCCCCTATCTCGGTTTGCTATGGAGGACTGGAGGGGGCTGCGGGAAAGCTTAGTTTTGTGCTTGATACATTTGAAAATGCCTGTTAGTCATCTAGATGGCAGAGTTGGGTAGCTAGCAGTTGAG
3272	chr4	121820153	121820404	-	0	train	AGGGCAGTGGAGGAGGATGAAACTGTTCTAATCTAGAACACTAATTTTCCCCCAACTGGATTCCTTGAACATGATATAGTACACTAGGTTACAAATTTTAAAACATTTTCCTTCCAATATTACTTTCTGGCATATAAGCAGTGTCTCTTTTATTAAGAAGTAAAGGCTGGGTGCGGTGGCTCGCGCCTGTAATCCCAGCACTTTGGGAGGCCGAGGTGGGCGGATCACTTGAGGTCAGGAGTTTGAGACCA
11020	chr16	2129599	2129850	-	0	train	ATGGACAAGGGGTCACTCACGCCTGGAATCCCAGCAGTTTGGGAGGCCAGGGTGGGTGGATCGCTTGAGCCCAGGAGTTTGACACCAGCCTGGGCAACAGGGTGAGACCCCGGTCTCTAAAAAATAAAAGAACATTGGCCGGGCGTGGTGGTATGCATCTGTGGTCCCAGCTATTCAGGAGACTGAGGTGGGACATCACTTGAGCCGAGGAGGTCAAGGCTGCAGTGAGCTGTGATCACACCACTGCACTC
15125	chrX	153874088	153874339	-	0	train	GTGGAATGCAAGCCGTGCATTCTCTACAGAAAGATGGGAGTGGGGAGAGAGAACTAGGCCAAAAGGCCATGGCAGCTCCAAGGCTGCCCTGGTCACTGACCAGCACTTGTTCTTGACCCTGATACGACACAGTGGGCAGTGGCAGGCAGCCGGCCCTCCCTGTTGGAACCTGTCTTCCATCTCTCCCTCCCCAGCCACTCCAGCCCCCTTTGTCTCTTTTCCTGCTTCCTCCACCTAGTTGGCTTTCTGCA
14035	chrX	147942264	147942515	+	0	train	TACTATAGTTATGAATCTGATTCCCCGTAGCCAGTAAATGAGATGTTTATGAGTGGATTGGAAGGACAGTGGTTAGAAATGAGGTTTCATCTGGTTTTAACAGAACCCAACATACTGGAAGTGCTTTTTGTAAGGGTCTGTCATTGGACCCCAGGTGTATATTTCTATCAACTAAATGTATACTGTTTCTCGTACTTGTTGTAGCAGAGTTGTAGTCTTTAGGAAGTTTTCAAGGCATGATGTAAGACCTA
13347	chrX	147920681	147920932	+	0	train	AATAATAGAGTGGGGTAAGTTTATTGTAGGGATAAGCATAGGATGCTATGAGAATGCAGAGGACAGGAATTTAACCTAGACTTCTCAGTTCTCCCGTTGAAGTTGACATCTGAACTGAAATCTGGAAGACCAGTAAGAGATAGCTATGTAAAAAGAGGGGAAGGACAATAGGAAAGAAGGAATGGAGAGAGGCCCAGAAACTACAGAGTATGGCACAAGTAGTTTAGCATTGTTGGGTCACAAATTCTAAG
2888	chr4	71776465	71776716	-	0	train	TAATGGGGAATGATAAAATCATCAATGAACTCCTCTACCATCTCACTGCTTATCATTTTTTCTGAGGCATTTGAAATTTACTCAAGTTATTTTGAAATATAAAATACATGATTAACTATAGTCACCCTGACGCAAAATAGATCTCAAAACATTCCTCCTGTCTAACTGAAATTTTGCACTATTTTCCCAGCAGCTCCTCATTCCCTCCCTCCTCTTCTCCCTGCAGCCTCTAGTAGCCATCATTTTACTCT
3088	chr4	73997812	73998063	-	0	train	TTGATCCAGAAGCCCCTTTTCTAAAGAAAGTCATCCAGAAAATTTTGGACGGGTACTTGTCACTTTGATCTTTGTGGTTTCTAAATCTGATCTAGGGAGACCATAGACTTCACAAGGTCTTTATTCTCTGTACGATTTAAGTAACACTTTTCATGTTTAGAATTAAAAGGTTGTTGAATTGGGAAAGTTTTTCTGGATTGTCCTGGGAAAATATACCAATCTTACATGTAATTACTTGAGCAATTACACAC
8871	chr14	102082630	102082881	-	0	train	TTTGGGAGGCTGAGGCAGCGGATCACAAGGTCAGGAGATCGAGACCATCCTGGCTAACACGGTGAAACTCAGTCTCTACTAAAAATAGAAAAAAATAAACCAGGCGTGGTGGCACGGCCTGTAATCCTAGCCACTTGGGAGGCTGAGGCAGGAGAATCGCCTGAACCCAGGAGGCGGAGGTTGCAGTGAGCCAAGATCGCACCACTGCACTCCAGCCTGGGTGATGGAGCGAGACTCTATCTCAAAAAAAA
3409	chr4	122453893	122454144	-	0	train	TAATCAGCCTACATCTGTAATAGGCATTTAGATGCAGAAAGTCTAACATTTTGCAAAGCCAAATTAAGCTAAAACCAGTGAGTCAACTATCACTTAACGCTAGTCATAGGTACTTGAGCCCTAGTTTTTCCAGTTTTATAATGTAAACTCTACTGGTCCATCTTTACAGTGACATTGAGAACAGAGAGAATGGTAAAAACTACATACTGCTACTCCAAATAAAATAAATTGGAAATTAATTTCTGATTCTG
1603	chr2	79087169	79087420	-	0	train	TGGCTGATTTTATCCCATTCAAAAACACCCTCACCTCATTCATGGGTTTGAGACAGAATTTAATAGGACCACTTATAGGTGACCATTGTGGTTGAGTTTATCTGATTGAATCTATATGCGATGGCAGTTTGGGGGATGTTTTTATGTAGTCATTGCTAGGATGGAGAGCTAAGGCAAACGTGTGCAGGGAAACCGAGAGAAACTTGAGAAAGGAGGAAGCCTGGGTCTTTAAAGGCAGAAGCCTCAGCCTC
15074	chrX	153872575	153872826	-	0	train	TTCGTCCCCCTGTGGCCATGGGGCCTCCAGGGGCTCTGGACCCTGCTTCCCTCACGATGGGACAGGTGCTGAGGCTATGACACCAGCCAGGCAGCACCCTCACCACCGCTCTCTCCACCCTGTCTTCAGTGATGGAGCCACCTGTCATCACGGAACAGTCTCCACGGCGCCTGGTTGTCTTCCCCACAGATGACATCAGCCTCAAGTGTGAGGCCAGTGGCAAGCCCGAAGTGCAGTGAGTGATCTCTGCC
7345	chr11	6393454	6393705	+	0	train	TCTGGGCACAGAAGTTTTATTTTCCTGGCATTCCCAACAAGTGTTCCCTGGGGATTCAGCTCATGGTCACTGTTGAAAGCCTTCATTCAGTCCCCCTTTCTCTAGCCAGGGCTGCCTGGACCCCTGGATGCCCTGATTACCATCCTTAATTCTCCCTACTAGGTGCATATAATTGGCCACATTCCCCCAGGGCACTGTCTGAAGAGCTGGAGCTGGAATTATTACCGAATTGTAGCCAGGTAGGACGGAGA
7964	chr11	119091634	119091885	+	0	train	CGTTCAGTAGAAACTGTACTTAGTACCCATACAGCCATTCTGTTTTTTACTTTCAGTACAGTATTCATTACATGAGATATTCACTTTATTGTAAAACAGGCTTGGTGTCAGATGATTTTGTCCAACTATAATAGGCTAATCTTAAGTGTTCTGAGCACATGTAAGGTAGGCTAGGTGTATTAAATGCATTTTCAGCTTGTTTTCAACTTAACAATGGGTTTATCAGGATGTAACCCTATTGTAAGTCAAGG
4434	chr6	32186730	32186981	-	0	train	TCCCCACAAGAGTTGAGAGTTGTGGTTCATCCTCTACCATCACGGGCTCTATTACACTCTTCCCTCTCTGCCCCCACAAGGCTCTGGCGGCTCTTTCAATCTCTCAGGATCTGGAGACATGTTTCTGGGGATGCCTGGGCTCAACGGAGATTCCTATTCTGCTTCCCAGGTCAGATGCCCATCTCCTCTCGAATAGGGCTTTCCCCAACTCCATTTCCTCTACTTTAGGATACAAGACCTCTTTCCTCTGA
15334	chrX	154299760	154300011	+	0	train	CTGCAAAATACATTGTTTTTTATGGCTGAGCAGTATTCCATGTTGTATATACACCACCTTCTTTATCCACTCATTGGTTGATGGACACTTAGGTTGGTTCCTTATCTTTGCAATTGTGAATTGTGCTGCCTATAAACATGTGTGTGCATGTACCTTTTTCATATAATGACTTTTTTTCCTTTGGGTAGATAGCCAGTAGTGGAATTTCTGGATTGAATGGTAGATAGATGGTAATCTACTACTTTTAGTAT
5530	chr7	99972485	99972736	-	0	train	GTTCAAGTGATTTTCCTGCCTCAGCCTCCAGCATGACTGGGATTACAGGCCCGCACCACCATGCCCAGCTAATTTTTGTATTTTTTAATAGAGATGGGGTTTCACCATGTTGGCCAGGCTGGTCTTCAACTCCTGATCTCAAGTAATCTGCCTGCCTTGGCCTCCCAAAGTCCTGGGATTACAGGCATGAGCCACGGAGCCCAGCCTAGAAATGTTAATTTCTAACGCATGTCAGATTCCATGCACACTGG
880	chr1	173908112	173908363	-	0	train	CGCCTGCCACCACGCCCGGCTAATTTTTGCATTTTTAGTAGAAACGGGGTTTCACTATGTTGGCCAGGCTGGTCTTGAACTCCTGACCTCAGGCGATCTACCTGCCTTGGCCTCCCAAAGTGCTGGGATTACAGGCGTGAGCCACTGTGCTATTGGGCTGTCTTTAAGCTAGTTTTGAAAACTAAAAATGTTGCCAGACTGGAAAGAAAGATGTTCCTTCTGGATGGAGTGAGTTTTTTCTGTAAGAACAG
9401	chr15	50258455	50258706	-	0	train	TGGGCCACATTAGTGACAAGAACACCAGTGAACAAGTCTGATTTTCATGGAGTTATGTCACTTATTTCTGCTTAAAAGTGTCAGCCCATCTCTAGAGTGACACTTCACCTTGCCAGAGATGGTCTCTGGAGAAAGAAAGCCTGCTGGAAATGCCTCCTGGTGCCAACTCTGTGCATGCTCTGCCTCCAGGTGGTACATTGGCAGAGCCCCCATATGCACGCCTACTACCCAGCCCTCACCTCTTGGCCCTC
15641	chrX	154309214	154309465	+	0	train	GGGCGGGAGGGGCTGTTCGGATGGGACGCTGCTACAGAGCTGACAGGAGCAGCCTGCACTCAGTGCGTGAGTCCACCTGATACCATGTCCTGCAGCTGCGTCTGGGCTCACTGGGTCCCTTCTCTTACAGAGACTGTCGTTTGTGGATGTGGCAACAGGATGGCTCGGACAAGGACTGGGAGTTGCATGTGGAATGGCATATACTGGCAAGTACTTCGACAGGGCCAGGTGAGGTTCTTCCCCAGAAGCCA
2258	chr4	71748634	71748885	-	0	train	ACTGCAAGTCACTTATATCCAGGGATCAGTCTCTACTTGTCATTCCAAATACACTTGGCACAACTACTGGAAGACTGGGAAAAGGGGCTTCTGAACCATACTTAGTGTTTCATTACATGGGTATCTACCCCTATCCTCACCCAATCTTGAAATGACCAGGAAATAGAACAATCAATGTCATTTTCTTTTGGCTTTACCATGGCACTCTTAGAAGTGGTTCATTTGTTTATGCCAAAAAAGACCCATTTGCC
7685	chr11	116832018	116832269	+	0	train	GACCCCTGGGCTAGGGGTTTGCCTTGGGAGGCCCCACCTGACCCAATTCAAGCCCGTGAGTGCTTCTGCTTTGTTCTAAGACCTGGGGCCAGTGTGAGCAGAAGTGTGTCCTTCCTCTCCCATCCTGCCCCTGCCCATCAGTACTCTCCTCTCCCCTACTCCCTTCTCCACCTCACCCTGACTGGCATTAGCTGGCATAGCAGAGGTGTTCATAAACATTCTTAGTCCCCAGAACCGGCTTTGGGGTAGGT
13398	chrX	147922641	147922892	+	0	train	GGTAAGATTGTAAAGTACATATAAACTCCTTAGTCAAAGTGTAGATGTGGCTATGATCTTAGGATTTTACTAAACTCTGATGGATGGTTAACAGTTATCATTTTTTTGGCTCTTATATACCAAGAAAATTAATAATATATCAAAAGCAGGCTGCAAATCTATAGAGACAGAAAGTAGATTAGTGATTGCTTGTGCTGGGGCTGGTGGGGAGAACATAGCTAAAGAGTACAGGATTTTTGTGTGTGTGGCAA
12679	chr22	19964012	19964263	+	0	train	CCAGAACCCTAAAGAAAACTGATGAATGCTTGTATGGGTGTGTAAAGATGGCCTCCTGTCTGTGTGGGCGTGGGCACTGACAGGCGCTGTTGTATAGGTGTGTAGGGATGGCCTCCTGTCTGTGAGGACGTGGGCACTGACAGGCGCTGTTCCAGGTCACCCTTGTGGTTGGAGCGTCCCAGGACATCATCCCCCAGCTGAAGAAGAAGTATGATGTGGACACACTGGACATGGTCTTCCTCGACCACTGG
15682	chrX	154310957	154311208	+	0	train	TGGGACACAGTGGTGCATTGCCCGCCGAGCACTGCATAAACATCTATCAGAGGCGCTGCGAAGCCTTTGGGTAACTGTATTCTCTTGTGCTTGATTTCCATTCTGTCCTGCCCCCTTCATCTCTTGTAACCCAGCCTGCTCCTCTGTTGGCAGGTGGAACACTTATGTGGTGGACGGCCGGGACGTGGAGGCACTGTGCCAGGTATTCTGGCAGGCTTCTCAGGTGAAGCACAAGCCCACTGCTGTGGTGG
137	chr1	109605519	109605770	-	0	train	AATTTACTGACAGTGAGCTTGGTCTCAAATTAGACATCTAAGTATCACTTGGACATCACAAAGCTCATAAGAGGAATTGAGTGCAAAGAGATAAGGGACCATCAACTAGGCAAAGCAAAGGAGTTACACTTAGTACTCTCCCAAATTGCCTAAGGAAGGAGATGAAAATGACAGAACAGAGAAAATAACATATGATATGAATCTTCATTGCAACATAATAGAAGGGTTGAGCTAGTAACCCCACTTAGGAG
13141	chrX	147914044	147914295	+	0	train	TTGCTCCAGTGATTTTGCTTGCACACTGACTGGAATATAAGAAATGCCTTCTATTTTTGCTATTAATTCCCTCCTTTTTTGTTTTGTTTTGTAACGAAGTTGTTTAACTTGAAGGTGAATGAAGAATAGGTTGGTTGCCCCTTAGTTCCCTGAGGAGAAATGTTAATACTTGAACAAGTGTGTGTCAGACAAATTGCTGTTATGTTTATTTAATTAAGTTTGATTTCTAAGAAAATCTCAAATGGTCTGCA
4654	chr6	32938619	32938870	-	0	train	CTGACCTGCTGGGATCCAGAGGAGAATAAGATGGCCCCTTGCGAATTTGGGGTGCTGAATAGCTTGGCGAATGTCCTCTCACAGCACCTCAACCAAAAAGACACCCTGATGCAGCGCTTGCGCAATGGGCTTCAGAATTGTGCCACACACACCCAGCCCTTCTGGGGATCACTGACCAACAGGACACGTGAGGAGAGAGGGGTGCAGAGGGGCTACCAGGAAGTGCAGTTAGGAGGGCAGGCCAGGGAGGA
6313	chr10	47303762	47304013	+	0	train	CTACCAATGTGATACTGAGTTGGTTTTCTTCATCTCTAAATTGAAAATAATACTACATACTCACCTGTCAAAAATGCCAAAATTCAAGTTGATTTCAGTTCCCCAAGTATTCCCTAAGCACTTCGTTTGGTAGCAGATGCTGTAAGCTCAGGGAGCACAAAGATGCCCAGAAAAGCTCATAAACTAGCAGGGCAGATCAAAGTAAGCAGCAGAATAGGGCAGCAGGCCTGGGGGGTGGCATATAAGAAGGA
11260	chr16	28934051	28934302	+	0	train	CTCGGCCTCCCAAAGTGCTGGGATTACAGACATGAGCCACAGGGCCGGGCCAAGCCTAATTTTGTATTTTTAGTAGAGATGGGGTTTCTCCCTGTTGGACCAGGCTGGTCTTGAACTCCTGACTTCAGGTGATCTGCCTGCCTTGGCCTCCCAAAGTACTGGGATTACAGGCATAAGCCACCGCACCTGGCCTAGACTTCAAGTCTTTCTTCCCTCGCTTCCAAGACACTACTTTTCTGGGTCTTCACCTA
16447	chrX	156009461	156009712	+	0	train	AGTCTCTGCAGCTGCTGAAAGGCCCTGAGGCACATGCTGTCAGGAGCTGGCTCTGTCCTGGGCAGATATCACCATCTGTACCTCGGTTCAGGCTGCCGTGGGCACCAGGCCCTGTGCTGGGGGAGTGCTGAGGAGCCTGAAGGGACTCAGGGTCCCGTGATGAGGCTGGGCTGGCACATGGAGGAAAGACAGAATGTCCAAGACACAGGCGCTGCTTGGCCTCTGGGTGTGGACCTCAGGAGGGCTTCCTG
7397	chr11	14969234	14969485	-	0	train	AAGGTGAGGCCCTTGCAGGGGATGGGGATGGGTAGAGCAGTGTCTGAGGTAGGTTTGAGCCTTTAAATGTGTGGCATCTGTGGAGATGTGCATGTTGTCAGGAGGCCAGGGAGAGCACCCTTCCCCAGATCCACATCCCTGTGTACCTTGAGCCTGAGCAGAGACCAGCCCCTGGCCTGGCCCCAGCACTGCTCAGGCAGAGGCATGTGTTGCCCCTGCATCTGCCCTGAGAACCCCTCTGTCAAGCATGA
10180	chr16	2099727	2099978	-	0	train	GGCCGCCTCGTAGCCGTTTCACTCGCATCCAGAGGGCCACCTGCTGCGTTCTCCTCATCTGCCTCTTCCTGGGCGCCAACGCCGTGTGGTACGGGGCTGTTGGCGACTCTGCCTACAGGTGGGTGCCGTAGGGGTCGGGGCAGCCTCTTCCTGCCCAGCCCTTCCTGCCCCTCAGCCTCACCTGTGTGGCCTCCTCTCCTCCACACAGCACGGGGCATGTGTCCAGGCTGAGCCCGCTGAGCGTCGACACA
2314	chr4	71750141	71750392	-	0	train	TTCTAGAGGTGCATTCAGCCTCCATTTAAATTTCTATAGTCAATTTATCTACTCCTAAGAGTAGAAATGGAAATGGCCTATTTCATTGCCAGGTAGCTGCATTATGAGAACATTCTTCTATAAATTAAGCTGTGATATTCCCTTTGTAATTTCTCCTGCAATCACCCTGGCTTTCTCTATCAACTAGAAAATGTCTATTCTCTGTTCGCAATAGCAGCTCTTTTATACTTGAAGTGACTTTTTAAAAATTA
5911	chr9	125237005	125237256	-	0	train	AACCAAAAGTGTCACTTTATGCTTCAGACTGAAATGCGGGGATCTAGATGTGCTAATGCTTGTCAGTAACAACTAACAAGTTTTTCTGTATGTAACTTCTAGGTGAAAGACCCCTGACAAAAGACAATCATCTTCTGGGTACATTTGATCTGACTGGAATTCCTCCTGCTCCTCGTGGGGTCCCACAGATTGAAGTCACCTTTGAGATAGATGTGAATGGTATTCTTCGAGTGACAGCTGAAGACAAGGGT
5289	chr7	99967308	99967559	-	0	train	AAGCCCACTGAGATGTGGGAAAACATGGAGAAGCACACGGAGCATTCACAACTTATTGCCGTCAGAGTCAATACATGGGTGAGGTGGGGATTGGGCAAGAGGGAAAGCGTCAGCCTTCCCTGATATTCTGGAAAGTCTCCCGGGGCTGGGGGTGGGCAGGTACAGAGCTTCGAGCTCTGCTGATCGCTGACATCCAGGGGTGGGGGTAGGAAGAGACCTGGGCCGGGAGAAGTCCACCTCAAGCCTGCAGT
8804	chr14	94566792	94567043	+	0	train	CCATTCCCCAGGCATGTCAGGTTTTGAATGGTCAAAATGGGACATCTTGATGGGCTCATAGGGCAGGATCTGGAGCGACTGTTTCTGTAGCTCAGAACACCAGATGGCATTCCCTGGCTGGAGGACTAGCTCTGTGGCCTGCAGATGTCCTGTACCTTCTTTTCATCTTCCCTTCAGCCCTGTGGGAGAAACCATTCATTTCCTCAAGGACCACTCCCAAAGACTTCTATGTTGATGAGAACACAACAGTC
7208	chr11	5226733	5226984	-	0	train	CTGCCCTGTGGGGCAAGGTGAACGTGGATGAAGTTGGTGGTGAGGCCCTGGGCAGGTTGGTATCAAGGTTACAAGACAGGTTTAAGGAGACCAATAGAAACTGGGCATGTGGAGACAGAGAAGACTCTTGGGTTTCTGATAGGCACTGACTCTCTCTGCCTATTGGTCTATTTTCCCACCCTTAGGCTGCTGGTGGTCTACCCTTGGACCCAGAGGTTCTTTGAGTCCTTTGGGGATCTGTCCACTCCTGA
14335	chrX	149486831	149487082	-	0	train	ACATGGAGAATGGGCCAAATACAGCAATTTTGATGTTGCTACCCATGTTCCCCTGATATTCTATGTTCCTGGAAGGACGGCTTCACTTCCGGAGGCAGGCGAGAAGCTTTTCCCTTACCTCGACCCTTTTGATTCCGCCTCACAGTTGATGGAGCCAGGTATAAAATATGCTGAAATGATATTGCTTGACAGTAAGATCACCTTTAGTTTATATGTGAACCACTTTATTGAATCATAGGCTTTGGGGGTTA
10552	chr16	2113901	2114152	-	0	train	TCCACCACCAGCCCCCAGGCAGGTGCCTGCAGACAGGGTGCTCACACAGGGCGTGAGGCCTGGCTTCCCAGTGAGGGCAGCAGCCCAGTTACTGGGGACGTCGGCCCCGGGCAGGTCCTGCTGGCTGGCTCCTCGGGCTACCTGGTGGGCTTTAAATTCCTGGAAAGTCACGGCTCTGACAGTGGCTCCGCTAACTCATTCCACTGTCTCATTTCACAAAATGAATTTAAAACTCTGCTCCCTGACCTCAC
16142	chrX	154327316	154327567	+	0	train	TGGAGTCCAATCTTTGCATGCCATAGTGAGCTCTCAAACCAGACTCCTGATGGGTATGTGGCGTATCCATGCTCCAGGACAGCTGGAGCAGTGTAATTTCCTATCAAAAAGTATGTGTCCTGGCTTGTAAATGCATCTGATTTTTTTAGTAGTGAGTTATTAAGGAGTGAAATTGTTAGAATTGGGGGATAGCGGCATAGCAAAGTGCCTTCCACTTTGACATGCATTCTGTGTGTAGTAACCACGGCATT
8100	chr12	14883278	14883529	-	0	train	ACCACACCCAGCCAGCTGATTGCTGTTGAATAGCTGGATTTATAAAGACTGAGCATAGGAGGAAATGGCACATCACTCTCATTTTTAATTTATTCATTATTTTTATAGTGTTTAAACTGTTCATGTATCGGCAATCTAGTTATGCTTCATAAATCCTCAGGACAGAGAATTTCTCCTCAAAAGGAATTTAAAATCTACCAAGTAGAAATACAGAAATTAAGAAAGGCAAAGTGATCGTCCAAACTCAAAAC
12304	chr19	2250805	2251056	+	0	train	GGGCCCCCAGCCCCTGAGCCAGCCGCGTGCCCACCCACCGCAGACTCCCGGCTGAGTACCGCCCGGCTGCAGGCACTGCTGTTCGGCGACGACCACCGCTGCTTCACACGGATGACCCCGGCCCTGCTCCTGCTGCCGCGGTCCGAGCCCGCGCCGCTGCCTGCGCACGGCCAGCTGGACACCGTGCCCTTCCCGCCGCCCAGGTGCGCGCAGGCACCGGGACACGGGGCAGGAGCGGGCGGGGGCGGCGT
1230	chr1	192809707	192809958	+	0	train	TCTTTGTACAGTCTCTGGCGTGGTCCAGAACCTCCTGCTCTAAAGAGAGAAGCGTGGGCCGGCTCCAGACAGTTCCATGTCTGTCCTTTTCATTAAAGTGCAAAACGTCTCGGAATTGTAATTAACCTTGCAAACAAACTGATGCCCTTTGTGAGCCAGAAATAGTGTCTGCCTTTTGAACTAAATTCATTAACAATTCTTTAAAATACCCTAGTGATTATAGGTAGCCCTGCCCTTAGTTGTAAAACTAG
2563	chr4	71761455	71761706	-	0	train	AGCTGGAATGCAGGGCACCAAATACCTAGACTGCACACAGCACGGGAACCCTGAGCCTGGTCCAAGAAACCACTTTTTCCTCCTAGGTCTCCAGGCCTGTGATGGGATGGGCTGCTGTGAAGACTTCTGACATGTCTTGGATACATTTTCCCCCATTGTATTGGGGATTAACATTCAGCTCCTTGTTACTTATGCAAATTTCTATAGCCAGCTTGAATTTCTCCTCAGAAAAAGGGCTTTTCTTTTCTCTC
3320	chr4	121822563	121822814	-	0	train	CCTGTGACAAATGGGAACATCCCTTCTCTTTTGAATACTGAAACTCTTCTTTGTCCCAGAAAGTTTAATTCCTGATAGAGTATTTGGGAGAAAAAGCAAAGGCCAACACCCATAAGAGAAAGAATGCAAGACTAGTAGGCTCAAAGCCAGTTATTAATTTTTTTTTAGGTTGCACCCCTTAAGGATCTTCCTGTAAATGATGAGCATGTCACCGTTCCTCCTTGGAAAGCAAACAGTAAACAGCCTGCGTT
2914	chr4	71777270	71777521	-	0	train	CTTAGCAAATATATCTAAGGAATTATATCCTGGTGGCAATTGGAATGTAATATCTAGTATATATCCTATAGAAAACATTGGCATTCCATTTTCCATCTACATGATACTGCAAAGCTCATTATTAATTTATTTGTTTAAAAGGCTCTCTTTAGCCTTACTAAGACAAATGAAGTGCTATAAAAGCACTTAAAGCAATGTATATCACAATGGTTAACACAAAATAGCATTGAGTACTTGATAGTGAAATTTCC
16188	chrX	154328769	154329020	+	0	train	CCTGATCTCTCGAAGGACTCCTGGGACCCAGCACACAGTTGCTCTCACACCTGAGGTTGATGACAGGGAAAGGGTGCAGGGGGGGATCCACCAGGGCAGGAACCATGGGCCAGGTGGAGTTGAAAGAAGTCCTGCCCCAGCCTCCCCTTGGCCCCACAAGGGGCATACTGAGAAGTGCCTCCCACGCAGGCTGTGTTCCTGCCTAAGGGATGCCTGCTAGAGACTTGGCACCTCAAGTGTGTACTGGGGGC
5257	chr7	76303448	76303699	+	0	train	CAGAGTCCCCTTCCCACCCACAGCCCCATCCCCAGATAAGCGGGGAGTTCCCTGGCGCGGTGCCAGTTTCTAGCCGCTGAGTGGGCGTGTGCGCGGCTCCAAGTGCGCCTGCGTACTGCTCACTCCCCAGCTCCGCGCCCTGCTCCGTTCCTCCCAAAACTCTGAATCGAAGAACTTTCCGGAAGTTTCTGAGAGCCCAGACCGGCGGGCACGCCCCCATCCCCAACCCCCTCTGTTAATCCCTACCAGCC
3734	chr5	132542101	132542352	-	0	train	GAACTCATGTAAATACCACAAAACAAAGCCTAACTTTGTGGACCAAAATTGTTTTAATAATTATTTTTTAATTGATGAATTAAAAAGTATATATATTTATTGTGTACAATATGATGTTTTGAAGTATGTATACATTGCAGAATGGACAATGGACCAAATTTTTATACCTTGTCTTGATTATTTGCATTTTAAAAATTTTCCTCATTTAGCACCAACTGTGCACTGAAGAAATCTTTCAGGGAATAGGCACA
495	chr1	119509273	119509524	+	0	train	TAAGCAGTAGTTTGTCTTACAGCTCCTTTAGGCCTTCCTCTAGCCCAACTTACACCATAAAAGCAACTCCATTTAGTCTACATAAAACACTTGCCAAGTCTACAGTGCCTTGGAGAAGAAGGGAGCCATGACTTCATCAACAAAGTACCTTTTCCTAGTCCCTCTAGATCCCTGGGAACTCACTGGGGAAGAGTCCTTTTCTCTCAGCTTTGGACTTCTTCCACTTTGACACTCCTTTGAGCCAGTCTGTC
4219	chr6	30491890	30492141	+	0	train	GGTTCAAGCAATTCTCCTGCCTCAGCCTCCCTAGTAGCTGGGACTACACATGCGTGCCACCACACCTGGCTAATTTTTTTTTTTGTATTTTTAGTGGAGATGGGGTTTCACTATGTTGGCCAGGCTGGTCTCGAACTCCTGACTTTGTGATCTGCCTGCCTCGGCCTCCCAAAGTGCTGGGATTACAGTCGTGAGCCACCGCACCCAGCCGCACCTACTCTTTTGTAAAGCACCTGTGACAATGAAGGACA
13128	chrX	147913646	147913897	+	0	train	CTTTGAGCACAAAGATAAAGCCTTTTGCTGTAAAAGGAGGCAAAAGGTAACCCCGCGTTTATGTTCTTAACAGTCTCATGAATATGAAATTGTTTCAGTTGACTCTGCAGTCAAAATTTTAATTTCATTGATTTTATTGATCCATAATTTCTTCTGGTGAGTTTGCGTAGAATCGTTCACGGTCCTAGATTAGTGGTTTTGGTCACTAGATTTCTGGCACTAATAACTATAATACATATACATATATATGT
6268	chr10	47301921	47302172	+	0	train	CCTGGGGTCAGATCTGTCACTGGCTGTGTGACTCAAGGTAACATCTGTGGCCTCTCATAACCATGGATTGGGGAGCACCTGCCACAAAGAGGCCCCATGCCCTCCTCCTCCACTTAGCAGACTCACCAGGTCAGTGGCCCTGGGCTCCAGGCCAATTGGATCTAACTCCATCCTCACTCATAACTTTCTTGCACCATGTGGGGTCCATTTCTTGAGGAACGTTATTAAATTAAACCATTTAAGAATCATCT
12459	chr19	44928881	44929132	+	0	train	AACAAAACAAAACAAAAAGCATACAAGCCAGCCCCGGTGCGATAGCTCATGCCTGTAATCCCAGCACTTTGGGAGGCTGAGTCGGGCAGATTACCTGAGATCGGGGGTTCGAGGCCAAGTTGGGCAGATCACCTGAGGTTGGGAGTTTGAGACCAGCCTGACCAACATGGAGAAACTCCGTCTCTATTAAAAATACAAAATTAGTCAGGCATGGTGGTGCATGCCTCTATTCCCAGCTACTTGGGAGGCTG
14122	chrX	147945844	147946095	+	0	train	GACAGCCACACTATGATTTTATTATTTATTATTTATTTTTATTTTTTTGAGACGGGAGTTTCGCTTTTGTTGCCCAGGCTGGAGTGCAATGGCGCAATCTCGGCTCAGAGCAACCTCTGCCTCCCGGGTTCAAGTGATTCTCCTGCCTCAGCCTCCTGAGTAGCTGGAATTACAGGCATCCACTACCATGCCCAGCTAATTTTTGTATTTTTAGTAGAGACGGGGTTTCACCATGTTGGTCAGGCTGGTCT
9399	chr15	50258341	50258592	-	0	train	AGAGATGGTCTCTGGAGAAAGAAAGCCTGCTGGAAATGCCTCCTGGTGCCAACTCTGTGCATGCTCTGCCTCCAGGTGGTACATTGGCAGAGCCCCCATATGCACGCCTACTACCCAGCCCTCACCTCTTGGCCCTCCCTGCTAGGAGACATGCTGGCTGATGCCATCAACTGCTTGGGATTCACCTGGGTGAGTAGCAACGGCTGTAACTACTCGCAATGGGGAACGTAGCAGGGAGTGGTATCCTAGGG
5210	chr7	45919138	45919389	-	0	train	TTATGTCATTTGCCAAGTGTCGTACATTATTGTGCATTTTGGGGTATTCAAAAAGTGATCTTAGAAATACTGATACACATCGTCATTCTTGGGCTTTAGCAATCATCATGATTACCACCTTAGTAGCACTGTAGTATAGGTTGATGTGAGTTATAAGATTATAAAAAGATCTAAGTGACTTCTAGAATCTATTTGACAAAAAAAGGTAAATTTTCGACAGTCAAAAGTCACAATTATCTGTTGCTTAAATA
10087	chr16	2096680	2096931	-	0	train	CATGGGGCTTTGTAGGAGCAGAAAGGCTCCTGTGTGAGGCTGGCCGGGGCCACGTTTTTATCTTGGTCTCAGAGCAGTGAGAAATTATGGGCGGGTTTTTAAATACCCCATTTTTGGCCGGGCGCGGTGGCTCACACGTGTAATCCCAGCACTTTGGGAGGCCGAGGTGGGCAGATGACCTGAGGTCAGCAGTTCGAGACCAGCCTGGCCAACATGGCGAAACCCCGTCTCTACTAAAAATACAAAAAATT
8006	chr11	123058249	123058500	-	0	train	TTAATTTCTTTCCTAGGCCGTTTGAGCAAGGAAGACATTGAACGTATGGTCCAGGAAGCTGAGAAGTACAAAGCTGAAGATGAGAAGCAGAGGGACAAGGTGTCATCCAAGAATTCACTTGAGTCCTATGCCTTCAACATGAAAGCAACTGTTGAAGATGAGAAACTTCAAGGCAAGATTAACGATGAGGACAAACAGAAGATTCTGGACAAGTGTAATGAAATTATCAACTGGCTTGATAAGAATCAGGT
12370	chr19	41878315	41878566	+	0	train	CCCAGGCAGGGAGAGGCCAGGGAGCCAAGAGTTTGAACCCAGTGCCACTCCTGACTGCCTGGTGATGCTGGCAACCCGCCTGCCCTCCCAGAGCCTCAGCCATCCCTCCTGTAAAATGGGGCTAAGGAGAGAACCTACTTCTAGGGTTCTGTGAATGATTACACAAGAAAAAGCGCCAGGTGCTGGGCCTGGCTGAGGCTGGGGTGCAAAAATGGACCGGGAAGGCTGCGGGAGGAGGGGACGCCTGCACT
7690	chr11	116832097	116832348	+	0	train	GACCTGGGGCCAGTGTGAGCAGAAGTGTGTCCTTCCTCTCCCATCCTGCCCCTGCCCATCAGTACTCTCCTCTCCCCTACTCCCTTCTCCACCTCACCCTGACTGGCATTAGCTGGCATAGCAGAGGTGTTCATAAACATTCTTAGTCCCCAGAACCGGCTTTGGGGTAGGTGTTATTTTCTCACTTTGCAGATGAGAAAATTGAGGCTCAGAGCGATTAGGTGACCTGCCCCAGATCACACAACTAATCA
6030	chr9	130696657	130696908	+	0	train	CTGTGCATAGGACCCCGTCAACTGTGAATAGAAATAAGTTTACTTCTTCCTTTCCAATTTGGATGCCTTTTATTTCCTGCATATGTTTTAAGGGTAGAGCCAACAGGTTTTGCTACTGGACTTGTCCTGGAGCGTGAGAGAGAGAGAGGAATCAAGGAACTCTCCGAGGTTTTTGGCCTGAGCAACTGAGGGGATGGAGTCACCGTTGACTGAGATGGGGACAACTCATCCCCTATCTCGGTTTGCTATGG
14493	chrX	149492196	149492447	-	0	train	ACTCCACACTGTTCCGTGCCCTGCAGCCCCTGCTTGGAGCATCTGAGCCCCATTTGTCTGACTGCAGGTGCTTCTTGTCAAATTCTAACTCCTCTGTGAAGCCTTCCCTGAATCTCCCAGGCAGATTTGAGGGCTTATTCCCCTGTGTCACCCCTGGGCCTCTGTGGGGGATCGGTCAGAGTGGTGGGAAAAACTATAGGGAAAGGATGCAAACCTTCTGAAAGGTCAGAAGGTTCTGCAGAGCCCCAGGG
7901	chr11	119088676	119088927	+	0	train	GCCTCGTACCCTGGCCTGCAGTTTGAAATCAGTGAGTTTTCTGGAAAGGAGTGGAAGCTAATGGGAAGCCCAGTACCCCGAGAGGAGAGAACACAACATTTCTGGCTTTGCCTATAGCTAAAGCCCGTCCCGCTGCCCCGAGATTCCTTCTGGGCTGCTCCCAGTTCTGAAGGTGCTTTCCTCTGAATACCTCCAGCTCTGACTACCTGGATTAGCCTGGCATTTAACATCTTGAGCTTTGGGTCTTTTTA
4980	chr6	37171162	37171413	+	0	train	GAAGAAGGTGAGCTCGGGTTTCTCCGGCGTCATTAGGCTCCTGGACTGGTTCGAGAGGCCCGACAGTTTCGTCCTGATCCTGGAGAGGCCCGAGCCGGTGCAAGATCTCTTCGACTTCATCACGGAAAGGGGAGCCCTGCAAGAGGAGCTGGCCCGCAGCTTCTTCTGGCAGGTGCTGGAGGCCGTGCGGCACTGCCACAACTGCGGGGTGCTCCACCGCGACATCAAGGACGAAAACATCCTTATCGACC
12974	chr22	43162624	43162875	+	0	train	ACAGATGAGGAAACTGAGGCTGCGATGGGGGAGGGGCTTGGCCAGGTCACTCAAGGGTGGAGTGGGGGTGAGTGAGGCTCCTGACTCCCAAATCCAGTGGGAGTTGGGCAGTGGGACAGGCACTTGGGTGAACGCGGTGCCTCAGGCCTCCCCATCCTCCGTCCCCCAATCTCTGCAGGCCTTGGTGGATCTCCTGCTGGTCAGTGGGGCGGCGGCAGCCACTACCGTGGCCTGGTACCAGGTGAGCCCGC
3460	chr4	122455955	122456206	-	0	train	TAGAATTACAAGAATCCCAAACTCACCAGGATGCTCACATTTAAGTTTTACATGCCCAAGAAGGTAAGTACAATATTTTATGTTCAATTTCTGTTTTAATAAAATTCAAAGTAATATGAAAATTTGCACAGATGGGACTAATAGCAGCTCATCTGAGGTAAAGAGTAACTTTAATTTGTTTTTTTGAAAACCCAAGTTTGATAATGAAGCCTCTATTAAAACAGTTTTACCTATATTTTTAATATATATTT
8240	chr12	49298053	49298304	+	0	train	AATGCCAGGACCCCACCATTTCCCATATTATTTCTGAGCCCCAGCCTGGCTGTCGCTAATCTTACTTAGGGTGGGTAATTCTTGGGGAGAGACAGAGAAGGAGACAGAGAACGTGGATAGATAACCAGGAGCCTCTGCTGGTTACCCCAGGGAGTGGGGCTCTATGATCCAAAGGAGTAGGATGGAGGGTGGCGCTGAATGGCTTGTGCCCATCATATGCCCTGTCCCCAGCAGGTGGTGACAGAGTCCCA
2696	chr4	71767508	71767759	-	0	train	ATAGGTACACCAAAATCTCAGAAATCACCCCTAAAGAACTTAGCCATGTAACCAAAAACAAAATAGTTTTGTTCCCCCAAAACTATTGAAATTAAATATATATGTATATATACATTTTATATATTATATAATTTAATATATATATATATATCAGCTCATGAACCTCTTGAAAGAATTCTAAAATATTGTGTATAGCTTAAAAACCTGATCATAGCTTTATCCTCTTTTATCATCATGCCAGGACTGGTACC
13391	chrX	147922468	147922719	+	0	train	CTGATCTTTTCTGTTCTCTTTTGCTAGATGAAGAAATAAAAACAGTCATAGGCCTAGGAATATTAACTGTATGAAAGCATCAATAGTATAGATGTTAACATTTTATTGGAGAACACAAGTCTCTTGACTAAATGTTTTGAATGCTAATAAAGGCTATTTTCAGGGTAGCTGTTGGTAAGATTGTAAAGTACATATAAACTCCTTAGTCAAAGTGTAGATGTGGCTATGATCTTAGGATTTTACTAAACTCT
10054	chr16	2095505	2095756	-	0	train	CGGCTTCCTCAACCCTTGGCCGTCCCTTGCTCGGACAGTGCTTCGGGCTGACCAGGTCGGAGGCTTGGGTTTGTCCTGGACCCCTCTGCGTCCTTCCTCACTGCAGCCTCCAGCGCGTCCCGTGGCTCCTTTCCCAACGCAGAGCACGGCCTTCCCTGCGCCTGAGCCTGCACCCTCCGTCCTGGCGGCGCCTCTGCCCTGGCATTCCCTGCCACTCCATGCCTCCCTATTGGCCATTCTCCGTCTCTGCC
2399	chr4	71753899	71754150	-	0	train	GATTATACTAAAAACCTAGACTTCACCACTATGCCATACATCCATGTAACAAAAATGCACTTGTACCCCCTAAATCATTTTATTTTTTTAAAAGTAGTTGGAATCTAACATCTGTATAAGAAGAAGCTAAGTTTTTCTGGAGGAAACATGTCATGTGTAGCAAGTGGGTGGCCAATTAAAACAGCCATAAAAATATAGATGACGAAATCTATCTGCAGGAACTTGAGTCAAAATGGCAAAATGAGCACATA
14411	chrX	149488914	149489165	-	0	train	CCATCTGCTTCTGAGCCCTCGGTGCCCAGTTTCCTCTTCTCCACGAACACACTCTGTTGTGAGGCAGTTGCCGTAGATTACAAATAGCCCATCTCAGAGTCCCCTCCTGGGACATCCTCCATCAGAACCGCCTGGGAGCCAGTAAAACATGCTGGTTTCTGCAACTGGTCTCAGGAGCTCGCAGTTCTGGGCATGGGAAAGTGCACTTTAACAGGTGTCTCAGGTGGTCTGGCCACCTCTCCCTGAGCTCA
4500	chr6	32188155	32188406	-	0	train	AAAGGGGGCGGCTCAGCAGCAGCAGCTGCAGCCGCTGCAGCCTCTGGTGGTGGTGTGTCCCCTGACAACTCCATCGAACACTCGGACTATCGCAGCAAACTTGCCCAGATCCGTCACATATACCACTCGGAGCTGGAGAAGTATGAGCAGGTAAGGAGAGGAGGCTTGGGTGGGTGGAGGGAAGGGCTCTTGCAGGGGAATCCCATGGTCAAAGGGCTCCTCCTCACCAGCCCACTGGCCCCCACTACAGG
7161	chr11	5226632	5226883	-	0	train	TGGGCATGTGGAGACAGAGAAGACTCTTGGGTTTCTGATAGGCACTGACTCTCTCTGCCTATTGGTCTATTTTCCCACCCTTAGGCTGCTGGTGGTCTACCCTTGGACCCAGAGGTTCTTTGAGTCCTTTGGGGATCTGTCCACTCCTGATGCTGTTATGGGCAACCCTAAGGTGAAGGCTCATGGCAAGAAAGTGCTCGGTGCCTTTAGTGATGGCCTGGCTCACCTGGACAACCTCAAGGGCACCTTTG
4888	chr6	33070858	33071109	-	0	train	GGGCTGAAAACTCACAGGGAAATCTGTGAGTTGGGAGGTGAGAGCAGAAGAGTCCCGTAGTTCCTTCTCACTCTGATGCATTTATCATTCTAAACCCAGACTTTCACATACACATTCATCGTTTTCTTTCATGATAATAGTTGCTTTTATCCTCTTATCTTTGCTAATTCTTACAAACTAATAAAGACTAAGAAACAAAATAAATTAAATCCTACAGGTGTTCCAAACTCAGCAATAATTTCTAGTTGGCC
13210	chrX	147916157	147916408	+	0	train	TAGGAAATTTGTCTTTGTTATATTGGGAGCTCATAAAACTGAAGTATTCAAAAGTTAGAATACATACACACAAGAAAAATTAGTAACTAATTTAATAATGTTTTCTTTGCACATGTCTCTGTTGTCTTTTGGTCAGAGTGAAGCTAAATGTGTTTTTCACATAATTTGTAGCCTATATGAAGTCCTGGACATGTGGTATGGTTGGAAGGACTGTTGATGAGGTTTATTGTCTCTCTTTATTCTTTTATGTT
4713	chr6	32940365	32940616	-	0	train	TAGCAATTTTAATGTAATTAATCTGTATAATTATCTGTGCAGTGTATATTCCCTGCTGGAATATAGGCAAGGACAATGTTCATCTTATTTATTGCTGCCTCCTCAGCTCCTAGCACAGTGCCTTGCATGCAGCAAGTGCTTCATAAATATGTGCGAAGTGAATATTTAATATTTCCAGCACAATACAAGGCTGACTCTTTCTCTTGACCCTTTTTCTCTCTCAATAATTTGCCTTACTGAAGGTCTGTGTT
13870	chrX	147937184	147937435	+	0	train	GTGAGGATGCACCTTTCTGTGCAATAGTACTCTTAAGATACTGTCTCATCAAAGAGCCGTAATTTTCACATATTGTCTAGCTTTCTTTTTGAAAGATTCTTTTAGCTAAAGTTAAATCTTTTTTTTCTGCGTACAATTTGTATTCGGTAATTTTATAAACCAAAACTGTTTATAATAACTTACATTTTTTAAAAATATGATGATTGATAACACTAATAGAGCTAAATAAAGTCTTAAATTGGTCCTTTTTT
10841	chr16	2123058	2123309	-	0	train	CCTGTCTGGGCCCCAGCTTCATTCCACTGCCCTGGGCCCTGGGAGCTCGGCCGAGCGGGGTCCCCAAGACCTTGCTGCATTTCTGGGCCTTGGGCTGGGGTGAGGGCCGGGAGAAGGAGCCAGCCTGGAGCCTGGCACGCAGGGAGTGCATGGCCAGAACCGGTGACAGGCAGGGCTGCCTGCTGGCGTGGAAGAAGTGTCCATGGCACCCCCAGGCCTGGTTCACAGTGGGATGGGCGGGGAGCCGGGGG
5298	chr7	99967520	99967771	-	0	train	ATCTGGGTGGTGGAGAGAAGGGAGGCGTGTAAAATGTCAGCCCCAGAAGGACAAGAGCAAGCCAGTGTGAGCGGAATTGATGGCTGCAAGCTGAGACTTGGATTGGAGACGTAGTGAGACTCAGGATTGTGCAGTGCTGCAGGGAAGTGGTTGCTGGATAGAGGCATGGGCTGAACCAAGCAGCTGGACTGAGACTGGGGGACAGAACTCCAAAGCCCACTGAGATGTGGGAAAACATGGAGAAGCACACG
7030	chr11	5226169	5226420	-	0	train	TATTTGCTGTTCATAACAATTGTTTTCTTTTGTTTAATTCTTGCTTTCTTTTTTTTTCTTCTCCGCAATTTTTACTATTATACTTAATGCCTTAACATTGTGTATAACAAAAGGAAATATCTCTGAGATACATTAAGTAACTTAAAAAAAAACTTTACACAGTCTGCCTAGTACATTACTATTTGGAATATATGTGTGCTTATTTGCATATTCATAATCTCCCTACTTTATTTTCTTTTATTTTTAATTGA
13251	chrX	147917344	147917595	+	0	train	TAGGCATTACATACTTTCATAAATATGATTATTGTAATTACCTCTTTGGCCCAGTTGCTAGTAAATTAGGGACCCCTTAATGATTTATTTCCTGTTTATTCACCCTGATGAAGAACTTGTATCTCTTTTAAACTGTACTTTATCGCCTTTCTCAAATTCCAAGATTCTCATCACATTTTTTTTCTTCCCAAACTCTAAATAACCTTTTAATATTAAGTATCTTTGTGGAAACATTGTTTTCTTTTTCTATC
12549	chr19	50876351	50876602	+	0	train	ACCCCTGTGCTTTTCTCACTGTTTCTTTTTCTTCCCTTTGGAGTCTCCCTTATCCTCCCCTGCCCCATCTACCTTTCCCCATTTTCTCTCTCCTCATGCATCCACCCCCTTCCTCCCCAGGAATAGCCAGGTCTGGCTGGGTCGGCACAACCTGTTTGAGCCTGAAGACACAGGCCAGAGGGTCCCTGTCAGCCACAGCTTCCCACACCCGCTCTACAATATGAGCCTTCTGAAGCATCAAAGCCTTAGAC
12834	chr22	20783496	20783747	+	0	train	AAGCTACTGGGGAGGGAGGCTGAGGCAGGAGGATTCCTTGAGCCTGGGAGTGTGAGGCTGCAGTGAGCTATGATGGCATCGCCGCACTCCAGCCTGCATGACACAGTGAGACCTGGTCTCAAAAACCAAATAATAATAACAGTAATAAAAGCTGGAAAGAGCTCAAAGTTACTCATTTGACAGATGTGACAGATGAAGAAATAGAAGCGAGTTAGGTGCCTTACCATGGTCAAACAACTAGTTCGTATCAG
11371	chr16	28937034	28937285	+	0	train	GTTGGGCCGCAGGCCTGGGGGGCACTGCCCCGTCTTATGGAAACCCGAGCAGCGACGTCCAGGCGGATGGAGCCTTGGGGTCCCGGAGCCCGCCGGGAGTGGGTGAATGACTGGGAGAGGGAAGGGTCGTTCCCCACATGGAGGGGGTTGGAGCGGTCTGTGGCCCGAATAGTGGACTGGGCCCTGGAGGAGAGGGGGCATGACTCGGTTCCCCATCCCCATCCCCAAACCCCCAGGCCCAGAAGAAGAGG
10519	chr16	2113147	2113398	-	0	train	GTGCCCCGCGTGGGACTCTCCAGCCCGACGGGAGGTGTGTCCAGGAGGCGACAGGCTAAGGGCAGAGTCCTCCACAGAGCCCAGGCTGACACCATTCCCCCCGCAGAGGTACAGCCCCGTGGTGGAGGCCGGCTCGGACATGGTCTTCCGGTGGACCATCAACGACAAGCAGTCCCTGACCTTCCAGAACGTGGTCTTCAATGTCATTTATCAGAGCGCGGCGGTCTTCAAGCTCTCAGTAGGTGGGCGGG
11	chr1	67685799	67686050	+	0	train	CCTGTGGCAGGGGCACTCTCGGGACTTCTCACGGGACGCCCGGTCCTTGGGCGTGCAGGGGTCATGGGGGGTGACGGGGCCGCGGGAGCGCCGGGTTTTCGTAGAGCCCAGGTGCGCGGTGGTGCTTGCATTCGAGAGGGAGGGGCGTGGTACCGGACGAGGGGGGCGGCGATGGCCCCGAGGGCACCGGGGCTGACGGGACCCCTCGCCCTTGCCCGCGTGTAGGATGGATAAGGTGGGGGATGCCCTGG
14581	chrX	149495212	149495463	-	0	train	ACCAGGTGATTCTGATTCATCAGAAGTTTGAGAGCCTCTGCTGTAGAGTATTGTTTAAAGCAAACACAGAACAAAACCACTGCATACAGATATCCCTTGCTATCCAAACTTACTTGGCCCCTGAGTTTGCACAACAAGAGTTTGGTTGGTGAGAGGTGTACATTCCTCAAATGTATTTGATTCCTGTTTCACTGAGAGTTTGGAAAGCAATGCATCCATCTGTTTGTTTTCTAATTTTGTGTTTCTGGGCT
16356	chrX	156004394	156004645	+	0	train	CTCACATGGATTCACTCTGTTCCAGTTAAGCTGGACCCGCCCTCTGACTTGCAGAGCAACATCAGTTCTGGCCACTGCATCCTGACCTGGAGCATCAGTCCTGCCTTGGAGCCAATGACCACACTTCTCAGCTATGAGCTGGCCTTCAAGAAGCAGGAAGAGGCCTGGGAGGTAACACTTTGGCTGGCTTTCCCTGGGGGCCTCTCTCCTGGGAACAGCAGTCCAGGGTAGACTCCCCACTCTACATAGGG
9278	chr15	50254023	50254274	-	0	train	GGCTCACTCCTCTGTGGAAAAGGCTGGTTTGATTTCCCTTGTGAAGATGAAATTTCTGCCTGTGGATGACAACTTCTCACTCCGAGGGGAAGCTCTTCAGAAGGCCATCGAGGAAGACAAGCAGCGGGGCTTGGTGCCCGTCTTTGTAAGTCAGGATATATGCCTAAGCTTAGAATGATAGAACTTGGATGTGAGGAAGGAATTTTGGCAGAATTCAGGTAAGTAACATGCTGCACACAAAAGCAGAAAGG
4853	chr6	33069823	33070074	-	0	train	ATAACTGCTCATCTGCTCTGTATTATTTCCTTTATGGTGTTGCTCCTTCTTCTTCCCCATATGTCCTTCCTTTGACCTCTTACCTTCTTCCTTTTTATATTCATAAGTCTTTATTCATTCTCTAGCTTTGACCACTTGCATATTCAAACTGACATTTTGTCGTGTTTTTCTCTACTGTCTTTATGCAGCGGACCATGTGTCAACTTATGCCGCGTTTGTACAGACGCATAGACCAACAGGGGAGTTTATGT
13746	chrX	147933649	147933900	+	0	train	ATGTTTTTAATTTGTAAATAAAATTGGGACATACTGCCTATATGTTTTTTATCTTTTAAACTTGAGATCATGGCCATTTTATATAAAGTAATACATGTTCTTTGGAAACTTTAAAGAAGAAAGCAAGGACTTCTCATACATTCTGTTTACGTAGTTTCAATGCTAATTTAGCCTGCTTTTTTGTCAGCATAGACTCATAAACATCTTCCTTGTCATTAACAAACACTAGAAATGATAAATGTATCTCCCAA
5649	chr7	99974781	99975032	-	0	train	TATTGGGAGGAGTAACAAGTCCTAAATCTTCAGAGCCGGCCAGCAGGCTGGAGACCCAGGGAAGAGTTGATGTCTTAGTCTTGATTCCAAGGGCAGACTGTAGGCAGAATTCTTTCCTCTTTAGGGGACATCTGAGGCTTTTTCTCTTAAGGCCTTCAACTGATTGGATGAAGCCCACCACTATGGAGAGTAATCCACTTTACTCAAGGTCTACTGATTTTTTTGTAAATTAAAAAAAAAACTGTGGGTGC
11873	chr18	31597319	31597570	+	0	train	AGGTGTGAGCCACCGTGCCCGGCTACTTCATGGATTTTTGATTACAGATTATGCCTCTTACAATTTTTAAGAAGAATCAAGTGGGCTGAAGGTCAATGTCACCATAAGACAAAAGACATTTTTATTAGTTGATTCTAGGGAATTGGCCTTAAGGGGAGCCCTTTCTTCCTAAGAGATTCTTAGGTGATTCTCACTTCCTCTTGCCCCAGTATTATTTTTGTTTTTGGTATGGCTCACTCAGATCCTTTTTT
10681	chr16	16317649	16317900	+	0	train	CCTGGAGCTCGTGTGCCCGTCCTCGGTGCAGAGTGACGAGAGCCTCGACCTCAGCATCCAGAACCGCGGTGGTTCAGGCCTGGAGGCCGCCTACAGCATCGTGGCCCTGGGCGAGGAGCCGGCCCGAGGTGAGTGTCTGCTGCCCACTCCCCTTCCTCCCCAGGGCCATCCAGATGGGGCAGAGCCTGGTACCCCCGTCTTGGGCCCACACTGACCGTTGACACCCTCGTTCCCACCGGTCTCCAGCGGTG
13170	chrX	147915196	147915447	+	0	train	GCAGCTAGCAGCTAAATATGATAAAGTGTACAATCAAAAGGATATTTTTAATGAAGATATTAGTGGTCTAACATGTCATTTCAGATACATAGCTGAAATGTAGTAAAATCAGTTTTACTACAAATAAACTTGCATAAGGTTTATAAATTTATAAGTTTATAAATCAACTTGGGTAAAGTGTAAATAAACTTGCACTCGTGGTTTCTCTGAAGTCTCCTGAGCTAACTTTGCATAAAGGTGTTATTCTGTAC
13446	chrX	147924180	147924431	+	0	train	GTGTGTATAATAACTCAGAGGAGGGTTCAGTTAATTTAGGCCCTAATCAGATTTCCACAAATTCTGACTTAATATTTGCCCGCTTATATAACAGCTCTTCTTTAACAAAAACAAGTACTTTTCTCAATAGAATTTTACTAAGAAAGCTCTTTAGTAAAACATCGACATTATACATACAACATATCTCAGTATCTGCTGATGAAGAACACCAAAAAGAACCCAGATGTGACTGCTCCGGAAGTTGAATCCTC
10013	chr16	2094367	2094618	-	0	train	AAACCTGCATGGGACAGACACACTGGCGTCTCTAGATTGTAGAGATGCTTGTTGGATGGTTGAGACCCAATCATAGTTTGCAGGGTTGAAGGGGGGCTCATTGCACCCTGAGAGACTGTGCACTGCTGTAAGGGCAGCTGGTCAGGCTGTGGGCGATGGGTTTATCAGCAGCAAGCGGGCGGGAGAGGGACGCAGGCGGACGCCTGACTTCGGTGCCTGGAGTGGCTCTTGGTTCCCTGGCTCCCAGCACC
11382	chr16	28937754	28938005	+	0	train	TCTTCCCTTTCCAGACTTCCTGAGCCCTCATGGGTCAGCCTGGGACCCCAGCCGGGAAGCAACCTCCCTGGGTGAGAGATGCTTTCAATCAGACTGCCTTGCCCAGCTTGGGTGACCTGGCCTCAGCTCTGACACCAGATCCAACTTTGACCTGACCCTGACCCCAAACCCGAACCCAATCCTGTGACTCCTCTCACCTCAACACTGAGCCCCATCCCCCATCCTGAGCCCCATCCCCCATCCTGACCCCC
10607	chr16	2115269	2115520	-	0	train	TCTGCTTGCCGCTGGACGCCTCCTGCCACCCCCAGGCCTGCGCCAATGGCTGCACGTCAGGGCCAGGGCTACCCGGGGCCCCCTATGCGCTATGGAGAGAGTTCCTCTTCTCCGTTCCCGCGGGGCCCCCCGCGCAGTACTCGGTGTGTGGCCCTGACCTGGGTCTGTTCCCTGCATCTCCTCAGGCCACCTTCCTGTCTGCTGCCCAGGGTCTGGGTCTGTGCACCAGACACACCCAGCCTGCAGGCCCC
12959	chr22	43160582	43160833	+	0	train	CAGGCCTCCCCCTGGACCTCACCCTGCAGTCACAGACTGACTCAGGGTCTCACATCTGCCTCAAAGCCTTTGCATGTCCTGTTTCCTCCCCTGGAAAACTCCTATTCATCATTTAAGGCCCAGGCCAAATGTCCCATCCTCTATGCAGTGCCCTGGCTAGGTTAAGAGCCTCCTCTCTACCTATTCACACCATGTGCTTCCCTTTCATAGCAGTGACAGTGATAATAATGGCAGGTATTTACTGAGTGTCC
8946	chr14	102086084	102086335	-	0	train	AGGAGGTTGAGACGTTCGCCTTTCAGGCAGAAATTGCCCAGTTGATGTCATTGATCATCAATACTTTCTACTCGAACAAAGAGATCTTTCTGAGAGAGCTCATTTCAAATTCATCAGATGTAAGTCACTTATTAACCCAGAATCGGATTTTGGTTTCAGTGTGAACTTCTTGGGGGTGCTGTATGCTTAAATTAATATTTTTTGTTAACAGGCATTGGACAAAATCCGGTATGAAAGCTTGACAGATCCCA
11962	chr18	46085421	46085672	-	0	train	GGTGTGAGCCCCCATGCCCGGCCCTGTATTCTGTTTTTTAAAGAGAGACAGTGTCTTGCAGTGTCACCCAGGCTGGAGTGCAGTGGTGCCATCATAGCTCACTGCAGGCTTGAACTCCTGGGCTCAAGCAGTCCTGCAGCCTTAGCTTCCTGAGTAGCTAAGACTACGGGCATGCACTACCATGCCTGGCTAATTTTTTTGTATTTTGTAGAGACAGGGTCTCACTATGTTGCCCAGGCTGGTCTTGAACT
2597	chr4	71763605	71763856	-	0	train	TGGTAGGGTCCTGCTGTACCTCTGCAAGCCCAACTGTATGCTTTTTGAAAGAGGTATGTCCCATTTTACTTATTATATGTTTACGTTTTTTTCTTCTAATAGCATTCCTTTTAGATAATGGTAGAAATTGAGATTTTGCAATCAGTAAAAGTTTCTAAAATGCCACCCTTGCATCTTCTTAATTGGAAAATAAATGTAATACCTCTGGAAAAGTAATATCAATAAGTTCCTTTCACCAGTGTTCAGTTTCC
14720	chrX	149501405	149501656	-	0	train	CAGTCCTCACCTGACTCACAATAACCTCTGTGCTTGGAGGATCCCCCACTTGGCACTTCGATGCTTTTGAAAGCTGGAAGTCCTTCCTTGGTTTCCAGTGCTGTTATACTGTCTTCACAAATAAACTATAAACTACTAAGCTTAAGGGTGGGTTTCTGCTTCATGAATGAGCACTTGAACTTGGAATTCTAAAACAAGGTTTTGAGTCCTGGCTCCCTTATCTAATATCTGGTTAGCATAGGTAAGTCACA
13062	chrX	136490088	136490339	+	0	train	TTTGACATTTATTTGGACCTTTGCCTCTGATTATGTGTTTCTAGATACAAGGCAGTTGTGAAGCCACTTGAGCGACAGCCCTCCAATGCCATCCTGAAGACTTGTGTAAAAGCTGGCTGCGTCTGGATCGTGTCTATGATATTTGCTCTACCTGAGGCTATATTTTCAAATGTATACACTTTTCGAGATCCCAATAAAAATATGACATTTGAATCATGTACCTCTTATCCTGTCTCTAAGAAGCTCTTGCA
5372	chr7	99969079	99969330	-	0	train	TTCAAGCGATTCTCCTGCCTCAGCCTCATGAGTAGCTGGGATTACAGGCATGTGCCAGCACACCCAGCAAATTTTTGTATTTTTAGTAGAGATGAGGTCTTACCATGTTGGCCAGGCTGGTCTCAAACTCCTGACCTCAGGTGATCCTTTGGCCTCAGCCTCCCTAACTGCTGGGATTACAGGCATGAGCCACTGCGTCCAGCCTAATTTTATATTTTTGGTAGAGATGGGGTTTCACCATATTGGCCAGG
14697	chrX	149499795	149500046	-	0	train	GCGGGTGCTGGGCTGTCCTGCCCAAGGGTGGGGTGAGCAAGGCAGGGCTGTGAAAAGGCAACACTCTTTGAGATGGAACAGCAGCTGACACAGCCCCTCGGTCTTTGTGGTATACAAATCATGGTCAGAACTAACTTTGGTCTCACGGCCAGCATGTCTACTTTAGGAAGCAAAGCAGGAGGTTTCATTCTGTGCCTAGTGTAGCTGAAGTTCTGGAAATTCCATGGACTGTCACCTTATCAAGTTGATTG
12341	chr19	41877386	41877637	+	0	train	TGTGGCCAAAGGGCAGGAACTGGCGGGAGGTGGGGGAAGCTGTGGAGGCTGCAGAGAGGGCACAGGCAGAGGGAAGGGGGCTCAGGGAAAGGGGAAGAGGAGGCAGAGGATAGGGGACCCAGGGAAGATGCCTATAGAAATCGTATCTGTGCCAAGATGGGCCAAGGTGGGGCTGGAGGGAGCCCAGCGAAGGAGAAGGGGCGTCCACAGTCTCACACAGGGAGGCAGGAGCAAGAGTCACCTCCCCCACC
13124	chrX	147913542	147913793	+	0	train	TAGGAAAATAAACAATTGACTTTATTCTGTGTTTACCAATTTTATGAAGACATTTGGAGATCAGTATATTTCATAAATGAGTAAAGTATGTAAACTGTTCCATACTTTGAGCACAAAGATAAAGCCTTTTGCTGTAAAAGGAGGCAAAAGGTAACCCCGCGTTTATGTTCTTAACAGTCTCATGAATATGAAATTGTTTCAGTTGACTCTGCAGTCAAAATTTTAATTTCATTGATTTTATTGATCCATAA
204	chr1	109608202	109608453	-	0	train	GCTATGGGGAGATCTGCCACAGTTGTCATGACTTTCTATTGCTTTACACTGGCTCACTACAGCCTACAAGGCTGAAGATGCAAAATTCAAAAGACGTGTTCTGCCTAGTAGCCCTCAGCCCTGTTCTCTTAATTGAGGTGTACACTAAGCCTCTCCTTCCAGAACCCTAAGATAGCCCTGCAGTTCTGTCATACACTCTCGCAGGAGTAATGTGATTCATGATGGCTCTTTCAGGGCAGAATTTTCTATCT
11563	chr17	41583573	41583824	-	0	train	GTGACCAGTATGAGAAGATGGCAGAGAAGAACCGCAAGGATGCCGAGGAATGGTTCTTCACCAAGGTGGGTGTCATTTGAGGTGGAAGGAACCCAGACCACCTGCCTTCTGGGGCCTTCTGGTGTGAATGGCATTCTCTTTTTTGCAGACAGAGGAGCTGAACCGCGAGGTGGCCACCAACAGCGAGCTGGTGCAGAGCGGCAAGAGCGAGATCTCGGAGCTCCGGCGCACCATGCAGAACCTGGAGATTG
2281	chr4	71749261	71749512	-	0	train	TCTAAATGTCTTTTAGATTCTATAAGAAATATCACTTTTGAAACCTTTCCCTGTCCCTTTGTTATCTGTCTATACAGATATGCACTGTCCTGCCTGACACCATACAAGACAAACTGAGCTTTAAGCGCAAATTCATGCTTCCTAAAAGCCCCTGGTTTTTCATTTCTTTGAGTTGGTTTTCAATATAACATATTGCCATTCAAATCATTACAATGATGAGAAGACAGGTTTGTAGCTCTCTCACTTCTTGC
4366	chr6	32181518	32181769	-	0	train	TGCCAGCCTGGGCAACAAAAGTGAAACTCCATCTCAAAAAAAAAAAGAAAGGGAAAGACTCCACTGGGGCTCCCACTAAATAACCCTCTCTCAACCCGAAGTCTTCCTTTCTGACTGGATCCAACTTTGTCTTCCAGAACCAGGCGAGGAGGGGCCAACTGCAGGTGAGGGGTTTGATAAAGTCAGGGAAGCAGAAGATAGCCCCCAACACATGTGACTGGGGGGATGGTCAACAAGAAAGGAATGGTGAG
16097	chrX	154325862	154326113	+	0	train	TGGTCTCAGCTACTCAGGAGGCTGAGGAAGGAGGATAGAGGAGCGCAGGAATTCAAGGCAGCAGTGAGCTATGATGGATCACACCACTGTACTCCAGCCTGGGTGACAAGAGTAAGACCCTATCTCTAAAAACAAGACAAAAAATAATAGAAATCACTTGAAATGAATAATCTGTTAGGTTCAGATTTTACAGATCTAAGGCCAGGAATAAAAGGGAACATGATCTCTGGTATTCTCACAAAAGCCCTGAG
12697	chr22	20779515	20779766	+	0	train	CTGGATTCCAGAGGGGGAGGAGGACGACGACTATCTGGACCTGGAGAAGATATTCAGTGAAGACGACGACTACATCGACATCGTCGACAGTCTGTCAGTTTCCCCGACAGACTCTGATGTGAGTGCTGGGAACATCCTCCAGCTTTTTCATGGCAAGAGCCGGATCCAGCGTCTTAACATCCTCAACGCCAAGTTCGCTTTCAACCTCTACCGAGTGCTGAAAGACCAGGTCAACACTTTCGATAACATCT
2073	chr4	39454726	39454977	-	0	train	TTGATGCTGTGCAAATGCCCTCTCCCCTTTTAGGTGTTGCTTGTTCAGTATCTCAAGCCCAGAAAGATGAATTAATCCTTGAAGGAAATGACATTGAGCTTGTTTCAAATTCAGGTTTGTATGTTTACTATGTCTAACTATGCCTACAAGATTTTGTCTAAATCTTGTTTAAAATACAGTATTTTGCTAATTTTTATTTGATTAGCTTTTTAAATTGTGAAAATAAAAAGGTCACATTCTGAAATTTTAAG
12111	chr18	46090339	46090590	-	0	train	TTTTTGCTTCAGACATGATTTAAAGATCTAAAATATAGTTTTAATGATGAGAGATGATTTGGTGCTGTTATGTTATGAAAATTTGTATTTGATTGGTTACAGTTAGGTAGCTTTTAATTTCACTTTCTACTAACTCGAAGAGTACAGGTTTTGATTAATTTTTGAAACTGTTTATGGAGCTTCTCACACATAAATAAGATGATAATCAAATTTGTCTTAAGATTTCCATTTGGCCTTCATCACTTTATTTC
7269	chr11	5254274	5254525	-	0	train	CAATCTCACAGGCTCCTGGTTGTCTACCCATGGACCCAGAGGTTCTTTGACAGCTTTGGCAACCTGTCCTCTGCCTCTGCCATCATGGGCAACCCCAAAGTCAAGGCACATGGCAAGAAGGTGCTGACTTCCTTGGGAGATGCCATAAAGCACCTGGATGATCTCAAGGGCACCTTTGCCCAGCTGAGTGAACTGCACTGTGACAAGCTGCATGTGGATCCTGAGAACTTCAAGGTGAGTCCAGGAGATGT
3117	chr4	73998034	73998285	-	0	train	ACGCAAGGAGTTCATCCCAAAATGATCAGTAATCTGCAAGTGTTCGCCATAGGCCCACAGTGCTCCAAGGTGGAAGTGGTGTAAGTTCTGTGCTGCTGTGTCCGCTGTGACCTTGGCAAGAGAGAAATCCCGCAGCCTGGGTCTTCAACCTTGGTATCTCATGAGTGTATCTTCTTTTTCTTTCCTTCAGAGCCTCCCTGAAGAACGGGAAGGAAATTTGTCTTGATCCAGAAGCCCCTTTTCTAAAGAAA
14652	chrX	149497812	149498063	-	0	train	TGTGTTGTTTGTTGAGGGAGGGATTTGCACAGGGAAGGTGGCTACATCCTGCCATCGCCAGGCACCATGGTTGCCTGATGGGCACTAGTGTCCTCAGTGGAGTAAAGATGGGATTTAGAGGTCAAGGCCAAGAACATGTAAGAATCTTGTAAGAAATGCTTGGCTTTCCGCTTCACTCCACTGGAGGGTTTGATTTCCTTTCCTTGAACTTATTGAGCAGATGTTGGGTTGGATGGTGAGATCACCACAGA
667	chr1	159303564	159303815	+	0	train	TGGTTGAACATGGCAGACAGTGTTTCTACCTCAAAAGAGATTGCAGTCCTCATTTACAGATACTGAATTGAAATTAACAGAAGTAGAGTGAGTCAGCTCAAATCACATAGTGAATTGGTTTCTTTGTTTTTAAATCTCCTGCATATGTGTCCTGTCTTTCTCCCTGTGTTGGGCGTTCCCTGGGGCACCAATACTAATTTCTCCTTCCCCTAGAAATCAAAACAGGGTCTTATCACCAACAGAATAAGGAC
15011	chrX	153868235	153868486	-	0	train	GACTTCCCCACGCACGCATTACCCTCAGATGCAACTCAGATCACTCAGGGGCCCCGCAGCACAATCGAGAAGAAAGGTTCCAGGGTGACCTTCACGTGCCAGGCCTCCTTTGACCCCTCCTTGCAGCCCAGCATCACCTGGCGTGGGGACGGTCGAGACCTCCAGGAGCTTGGGGACAGTGACAAGTGAGGACAGTGACGGTGAAAGGGGGCAGAGTGGGAAAAGCTGGAAGTCCAGACCTCTTGGCCTCG
11243	chr16	28933594	28933845	+	0	train	AGGGCTACTCCCAGCCTCACCCCAAACCCCAACTTCCACACAGAACACTGACTCCAAGTCTTTCTTTTTTTTGACAGAGTCTCGCTCTGTTGCCTAGGCTGGAGTGCAGTGGTGCCATCTTGTCTTGGCTCACTGCAACCTCCGCCTCCCAGGTTCAAGTGATTCCCCTGCCTCAGCCTCCTGAGTAGCTGGGATTACAGGTGCCCACCACCACGCCTGGCTAATTTTTTTTTTTTTTTTGAGACGGAGTC
8322	chr14	20455791	20456042	+	0	train	ATTTTTGCAGGGGTTTGTGAAGAAGTCGCAGGAACCGTAGGCTTTCGTTGGGTCTATAGTTAACGCCGGATCGCAGTTGGAAACCACCAGCTTTTTGTCAGTATATATTACTCATTTTATAGAGCCAGAGGCCAAGAAGAGTAAGACGGCCGCAAAGAAAAATGACAAAGAGGCAGCAGGAGAGGGCCCAGCCCTGTATGAGGACCCCCCAGATCAGAAAACCTCACCCAGTGGCAAACCTGCCACACTCA
5048	chr6	37173279	37173530	+	0	train	TGTCACCCTTGGCTTAGAATTGTAGGTGAGTGATTTACACTTGAGCTGGCCTCATAAATCACATGGTTTGCACTTGAGCTTTCCTTGGGAGGTCAGAGGAAGGCATGTGTGAGCATATTAAGAAGAAAAGACAATCTGGCTTCTCCAAAAACTTTTTTAAAGGTACCAACAGAAACCTGATAATTCCTGGCTGTTTTGCCAGGGAGTAAAAAGTTAAAAGCTCTTTTAGCATCTTCTTTAAGGCAGCAGCT
7082	chr11	5226294	5226545	-	0	train	CCCCTTCTTTTCTATGGTTAAGTTCATGTCATAGGAAGGGGATAAGTAACAGGGTACAGTTTAGAATGGGAAACAGACGAATGATTGCATCAGTGTGGAAGTCTCAGGATCGTTTTAGTTTCTTTTATTTGCTGTTCATAACAATTGTTTTCTTTTGTTTAATTCTTGCTTTCTTTTTTTTTCTTCTCCGCAATTTTTACTATTATACTTAATGCCTTAACATTGTGTATAACAAAAGGAAATATCTCTGA
15014	chrX	153868285	153868536	-	0	train	TGGCAGTGCCTCTGGGAGGGAGGGTCTGGGCCTCTGGAGGACAACAGAGTGACTTCCCCACGCACGCATTACCCTCAGATGCAACTCAGATCACTCAGGGGCCCCGCAGCACAATCGAGAAGAAAGGTTCCAGGGTGACCTTCACGTGCCAGGCCTCCTTTGACCCCTCCTTGCAGCCCAGCATCACCTGGCGTGGGGACGGTCGAGACCTCCAGGAGCTTGGGGACAGTGACAAGTGAGGACAGTGACGG
2356	chr4	71752246	71752497	-	0	train	ACTTAACATGGCAGAATCTTTTAAGAACGTATGCACTCCAATCTACTCATTTCTTTCCTGTTATTGAGATGCCATTATGTGACAGGCTTTTCCTGGTGTTATTGTAACTTGGCTGTCTTTGCAATGAAAGTAAGAAACATAACTGATTTCATGCTATGCTCATTTAAAAGCAAGTTGAGGTAGTTGTAAAACTTAGGAAATTTTATACTTTTTTTAAGAAACAAAAGAGTCTGTGAACACAGGCAACAAAG
6207	chr9	130702333	130702584	+	0	train	TCAGTGGGATGGAGGGGGCTCAGTCTTTGCTGTGTTTTTGTGGCCAGTGAAGTTGGTTGTTTTTAGCTATGTTACTGGTGTAGGCTGAGTCACTTTGACTTTCCATCACGGTATGTTCATGAAGCCCATATTATCTTCTCTTCTAAGGAGATGAATTCAGTGAATGGTTTGGTGTTTGTTGGTGGAGGTGGCAAAGTTGCAGGGTAATAGCCGAAGAGCCAGTGAAGAAGCCATTCTTTTTCTGTTTTTTT
10135	chr16	2097984	2098235	-	0	train	CCATCTAAAAAAGAAAATATGAAATTTAAAACTCTGTTCCTTAGCTGCACCAGTCTGCTGTCAAGTGTTCAGTGGCACACGTCGCGAGGGGCTGCCATCACGGACGGTGCAGATGTCCCATATATCCAGCATTCTAGGACATTCTGTCAGATGGCACCGGGCTCTGTCCTGTCTGCTGAGGAGGTGGCTTCTCATCCCTGTCCTGAGCAGGTCTGAGCTGCCGCCCGCTGACCACTGCCCTCGTCCTGCAG
5510	chr7	99972177	99972428	-	0	train	ACTCTTTCATATTATCCTGTGGCCTTCAAAGTCGTCACCTCTAGGGATGAGAAACAAAAGGGACAAGCCAGCTGGTAGGGTCTTGGACAAGAAGAAAGACATCACTTCTGCTCACATTCTCTTTTGACAAAACTCAGTCACATGGTCCCAATATATCTTCGAGGTGGCTGAGTAATGTTATCTTCCTATGTGTCAAGCAGAGGAAATAATGTAGTGAAGACACAGGATGGTCTCTGAAATATCATCTCAGG
211	chr1	109608327	109608578	-	0	train	GTCTTCTCTCTGGTTGGGAGAAACCTCACCCAACCCAAAATTTCAGGCATTGAAAGCTGGAGACCCAGACTGAATTCAGCCTGTGGATCTGTTTTGTTAGGCTTCAGCAATGTTTTGAATTTAATGCTATGGGGAGATCTGCCACAGTTGTCATGACTTTCTATTGCTTTACACTGGCTCACTACAGCCTACAAGGCTGAAGATGCAAAATTCAAAAGACGTGTTCTGCCTAGTAGCCCTCAGCCCTGTTC
12862	chr22	20784491	20784742	+	0	train	GAGACAGCTTGGTGCTTGCTTTGTGGCTTCGAGTCCCAGCTTCATCATCCCTAAAATGGGTATAATTCCATTACTTCCCCGGGTCACTTGAGAAAATAACAGAATCAGCGATGCTGAGCGCCCCTCCCAGTACTTGGAACCTAGGAGGCACTCAAAAAAAGATTGGCTCAACTCTTCCCTGCCCAGGAAATTCCAAGGTCCTCTTAGCCTACCGAGGACACATCATTCATGATTTCCTCTATTATTATTCG
12321	chr19	2251202	2251453	+	0	train	CGCCGCGCCTGGCCCTGGATCCGGACGCGCTGGCCGGCTTCCCGCAGGGCCTAGTCAACCTGTCGGACCCCGCGGCGCTGGAGCGCCTACTCGACGGCGAGGAGCCGCTGCTGCTGCTGCTGAGGCCCACTGCGGCCACCACCGGGGATCCTGCGCCCCTGCACGACCCCACGTCGGCGCCGTGGGCCACGGCCCTGGCGCGCCGCGTGGCTGCTGAACTGCAAGCGGCGGCTGCCGAGCTGCGAAGCCTC
12496	chr19	44929951	44930202	+	0	train	GCCAGGTGCAGTGGCTCATGCCTGTAATCCCAGCACTTTGGCAGGCTGAGGCGGGCAGATCACCTGAGGTCAGGAGTTTGAGACCAGCCTGACCAACAAGGAGAAACCCTGTCTCTTCTAAAAAAAAAATACAAAAAATTAGCCAGGCATGGTGGTGCATGCCTGTAATCCCAGCTACTCAGGATGATGAGGCAGGAGAATTGCTTGAACCTGGGAGCTGGAGGTTGCGGTGAGCTGAGATCGCGCCATTG
1894	chr2	162145309	162145560	-	0	train	GAAAGGCCGAGGAAGGCGAGAGTAAGTCTGTACATTCTTATTTGACATTTTTTGCCTTGATGCAGAAAATTTAAGACTACAGTTATCTATATATGGATCTGGATTACAGAAGCAATTAGTAGTCTTGCAAAGTAAGGAAATAATTCCTATTGATGAAAAACAGTATATAAAAGTTAAACCCATTTTGTTTTTGGTACTAAGTATTAATAATAGAGCCAAACAGGTTACATTTGTATCCCCTTATAGTTGCA
12078	chr18	46089333	46089584	-	0	train	TTGGTGACCGACAGACTGGGTAAAGACTCAAAAGCATATTAAAAGTTCTAACTAAAATTTGCTCAATGGAAGGATTATATTTTAGTCTTGGAATGGTAAATGGTAATGGGGACATTAGAATCTAATACGAGGATTAATAGTAAAATTTCAGCTGGGAAGTGAAGGGGGCCAGCCCCTCCACACCTGTGGGTATTTCTCATCAGGTGGGACGAGCTAGCTCAGTCGGTAGAGCATGGGACTCTTAATCCCAG
1012	chr1	173915946	173916197	-	0	train	ATTACAAGAGACTTTATCTCTTGATTTGCTTCATCGAGTGTCCCAACTACCTCATTTTTTTAAAATGTGAAATTAGCTTCATTTACCTTCATTGAATCCATGTTGGCGACTATTAAAAATTCCAGGCAATAAAAAGGGATGAGAGCCTGAACTAAAGCAGTGGCAATAACTGGTGAAAGAGTAAAAAAACAGAACTGATTGACTCTGGGGTGAACTGATTGACTCTGGGGTTTGACTAAATGAGGAGGAGA
14068	chrX	147943758	147944009	+	0	train	CCCTAGGCACAGTCAAGAAGTAGGCGAAGTTTCTCTTTGCCTTCCCAGCTTTTCTGTCCGTGCTTCTTTCTTCTCTCCCAGTACCTTTCAGTGAACCTGATCCCCACCTGCCAGTGCCTACTGTCTTTCTTGCTCAACAGTGTATCTCCTTTGTAACTTGCTAATGATGGTATAAGGTATAATCCATTTCACGCATATTTGCATTTCAAAAAGGAACTCTTTAAGATGGAAAAGCCTTCAAGATCCACAGG
14388	chrX	149488198	149488449	-	0	train	TCCCTCACGTTGCCATCCTGCAGGCTTCTCGATCCTTGACCTTTTAGATTCCCGCCACACTGAATCTAAAAGGGACAATGTGACCTCTAAGCTGGTGGCCTAGGAGCCTGCCGATAACCTACCCATTCATCAGCCCTCGTGGGTCAAGGCTGCCTCGTCCTCCCCGAGAGAGTGGAGAGGTCGAGCAGTGGGGATGCCCCTCAGGCCCCGGGGCTTACTGACTGGGGCGGGTGTCAGGGGGAACTCCCTCT
14994	chrX	153867441	153867692	-	0	train	GGACTCACGGTTCCTCATGGGAATCTGGAGATGCCAGTTGGCCTGGGTAATCAGGGACCCAGACTGTACCCAACCCCTCCACAGCCCTTCCCCCAAAGCCACATGCTGATCACTCCATTGTCGGTTCATTTCTTGGCAGAATATGACATTGAATTTGAGGACAAGGAAATGGCGCCTGAAAAATGGTACAGTCTGGGCAAGGTTCCAGGGAACCAGACCTCTACCACCCTCAAGCTGTCGCCCTATGTCCA
9516	chr15	50263530	50263781	-	0	train	TGAGTCCAGGCTGGAGTGCAGTGGCACGATCTCGGCTCACTGCAACCTCCGTCTCCCAGGTTCAAGCGATTCTCCTGCCTCAGCCTCCCAAGTAGCTGGGATTACATGCGCCTGCCACCACGCCCGGCTAATTTTTTGTATTTTTAGTAGAGAGGGGTTTGACCATGTTAGTCAGGATGGTCTTGATCTCCTGACCTCGTGATCCGCCTGCCTCAGCCTCCCAAAGTGCTGGGATTACAGGCATGAGCCAC
12039	chr18	46087624	46087875	-	0	train	GGAGTTTCACTCTTGTTGCCCAGGCTGGAGTGCGATGGCACGATCTCAGCTCACTGCAACCTCTGCCTCCTGGGTTCAAGCGATTCTCCTGTCTCAGCCTCCCGAGTAGCTGGGATTACAGGTGCATGCCACCACGTCCGGCTAATTTTTGTATTTTTAGTAGAGACGGAGTTTCCTCATATTGGTCAGGCTGGTCTCGAACTCCTGACCTCAGGTGATCACCCGCCTAGGGCCTCCCAAAGTGCTGGGAT
9097	chr15	50247398	50247649	-	0	train	TGTTCTTCATGGAAGAAGCGGGACTTAACTAAGAACTCTGGCCACTGGCATTGTTCTCCCACTTCTACAAGAGTCCACATGCAAGACAAAGCCATTGATAAAAGTAGTGGGGGAAGCTGAAGACACTGTCCTTTCTGCCTATGTGTGTCTTTGTCCTTCCACCTTTCCTCTAGGGAATTCCTTCTTGGTTAAACAAACACAAACCAAAATGAAGTGGTGGAAGCGGTAACTATGATTTTTTTTAACATTTA
7969	chr11	119091722	119091973	+	0	train	TTGTAAAACAGGCTTGGTGTCAGATGATTTTGTCCAACTATAATAGGCTAATCTTAAGTGTTCTGAGCACATGTAAGGTAGGCTAGGTGTATTAAATGCATTTTCAGCTTGTTTTCAACTTAACAATGGGTTTATCAGGATGTAACCCTATTGTAAGTCAAGGACCATCTGTCTTCACTTCTTGACCACCCCACCTCTAACACCGTAGGCTGGGAAGATTGTGAATCAGAGGCCAGACTCTAGGCTTTCAT
10898	chr16	2125289	2125540	-	0	train	GCCTGCCTGTGGGTCCCGGGAGGACCTGAGGCTGCCCATGTCACCCCCGGCATCTCATCCTGGGGACAGTTCAGCCGTGGGAGGGATCTGTAAGGACAGAATGCCGCTGAGCCTGGGGCTCCCCAGCTAGTCTCACACCCCGTGTCTGGGACCCAGAGACCCTCGTGCAGGGCTCTGTTGCTTGGGGCCTGGCAGCCTCGTCCTGTATCAGAGGCTGCCACCCCCACCCCTCGTGGGGCCAGGGTTGTGGC
10875	chr16	2124440	2124691	-	0	train	GCTCCGTCCGGCCCCCAGACCCCACTCAGCATCTGGTCTGGGGAGTGGGCGCCTGGGGCACTCAGCTCTGAGTGTGAGACTCTGAGGCAGGTCTGGTTTGTCTGGGGCCATTCCCTCTGCTGTGGATTGGGAGGGCCCCGGGAGCTGCCCCACACCCAGGGAAGTTCTCCTCAGTCCCACTGTTGCATTCCCCGACCCCGGCTCCCCCGGCCCAGGAGCGCCTGTGGGGCAGAAGGCCCAGCCCCAAGACT
13177	chrX	147915528	147915779	+	0	train	TTGTGGCTGCATCATTTTTCCCCTTTTGAACTGTGCATTTTCTAACCCCATACTTAAATATTCTCATAACCTCCAAATTATTAATTAGATGCAACATTCAGTGGTATATTACTGGAGTTTCTGATTTCTGCCCACTATAGGAATGTGCTTCCTGAGAAGATTGGGATCGTGATTATAATAATAGTTAACAGGGGATGAGTACTTTCTAGGTGCCAGGCACTGTTCTCTCTGATACTTTATTTGATGTATTG
12539	chr19	50875912	50876163	+	0	train	GAGCCCTCACTCCAGCCCCAGCTGCAGGTGAGCCACCCTCATGCCTCTCCTCCTCCCCCTGCTACTCCACACTCCTCAGATGCCCCCGTGGCCTCCCTCCTTTTTCTCTCCCACACTGTATCACCCCTGGCTTCCTCTCTGCTGTTTCTCCTTCTCTCTCTGACTTCCCGCATCCTTTTCTCATTTGTCTATTTCTCACTCCCTTCCTGGTTCTGTTCTTTCTCCCTTCCTCTTCCCCATGTCTATTTCTT
13922	chrX	147938705	147938956	+	0	train	TGGTGAGAACTGTTTATGGAAAAAAATGTTGTGGACTCATGGATTAAAAGCAAAGGTGCCATTTTACAAAGAGGACTATAACGGCAAGTGGATTGGATATGTCTCATTGCCGGGCAGCCCATTATTCCAGAACACAGAGTAACTTTTTTCTGGGGTCCGTACTAAAGGCCATCATTCCATTCTTGCCTTTCTTCATTTTAATTTACTATTCAGGTCTACTCGTGAAGTGCTTAAATTAGAGTGGCCCTTGG
6694	chr10	47355208	47355459	+	0	train	CTCTGGAAAATGCCAACCCATGGAATTCTAAAGGAGGAAGAGGCACTAGACTGAGGCTGCCAAGAATTAGAACCCTCCATGGGACAAAGATCCTGGCCTCCCCGAGGGGCACACAGGGCCTCACTGTGAGCTCAGCCCCTGAACAGGCTCTGCTTCCCATCCTTCAGGTTCAACATCGGTGGCCCCACATCCTCCATTCCCATCTTGTGCTCCTACTTCTTTGATGAAGGCCCTCCAGTTCTGCTGGACAA
5191	chr7	45918336	45918587	-	0	train	AATTGTGGACTTAAGCCGATGCCTCCAGACCTTGGCATGGTCCACAGGCCCTGGGAGCATGGGCTCTGAATGTAGCCTTTGATCCCCATAGCGGTCTTACAGCCCCTCCAAGTTCATTCTGAAGAAGGAATGGAGTGAGAATCCTGGCTGCAGATCCAGTCTTGAATTTAGTCATATACTTAAAATTCCAATTCAACTGTTAACATTCCAGCATCCATTTTAAGCATCAGACTTTCTTCATTTAGCACTTT
8108	chr12	14883430	14883681	-	0	train	GATTACAGGCCCATGACATCATGCCTGGCTAATTTTTGTATTTTTAGTAGAGATGGGGTTTCACCATGTTGTCCAGGCTGGTCTCGAACTCCTGGCCTCAAGTGATCCACCCACCACAGCCTCCCAAAGTGCCGGGATTACAGGCATGAGCCACCACACCCAGCCAGCTGATTGCTGTTGAATAGCTGGATTTATAAAGACTGAGCATAGGAGGAAATGGCACATCACTCTCATTTTTAATTTATTCATTA
8421	chr14	24575844	24576095	-	0	train	CCTAATTCTGTACTGTTGAGCAAGTTATTTGAATTTGTGTTTCCTCATCTATAAAATGAGAATAATATTAATACCGATCTTGCAGAGTTGCCATGAGAGTTAAATAAGTTAGAGTATTTAAATGTCTTGGAATTGCCCGCACACTATAAGTGCTATAAAAACATGCTTTGTGTAAATAATTTGGCAGCATGTGTCAGACCCTACCTAGGAGGTAAGAATACAGCAATAACAGTACCATCAGCTCATGTCTA
10844	chr16	2123124	2123375	-	0	train	GGGCACCTGAGTCCTACCCAGGGCAGACGCTTCCACACCCTGGGGGCTGGGGGACTGCACCTGGCTCCTGTCTGGGCCCCAGCTTCATTCCACTGCCCTGGGCCCTGGGAGCTCGGCCGAGCGGGGTCCCCAAGACCTTGCTGCATTTCTGGGCCTTGGGCTGGGGTGAGGGCCGGGAGAAGGAGCCAGCCTGGAGCCTGGCACGCAGGGAGTGCATGGCCAGAACCGGTGACAGGCAGGGCTGCCTGCTG
12915	chr22	20786569	20786820	+	0	train	CCCATCCCGGAGAAGTGCGCAGCAGTGTGGGGAGCTGGAGCTGGGGTGGCTGTCCTGCACCAGCCCCCACGACCCTCAGACCACAGGCACTGCCAAGAGGGAACATGAACCTAGCCGGCCTCTAAGTGCAACGGCTGCCCCTGACAGGTGGTGACAGATATTTTCAAGAGTGACTCTGACCAGCTGTGATTTCCACCTTACATGTTGTCTTTGGATCCTTTCCCTGAATGATATGAGATTGTGCTGGGAAC
2505	chr4	71758236	71758487	-	0	train	AGAAAGAGACATTTGTGACTTCTTCATTGATGCTTTAGATCTAACATTTTAAAGGAGTGCTTTCACTATGTATATCTGGCAAGAAGAGAAGCTGCATTTGGGTACTGAAAATTTGGAAAGATTCATTAGGGATCTCTTGATCTTGAGTCTGTGGCTTGGGTACATAAGATCTATCGCATGGCAGATGTGCTCCCATGCATCTCTTGAGGCCAAGGAGGTGAAATTGTGATATTATTTCTCCAAATAGTGCG
13456	chrX	147924460	147924711	+	0	train	TTTTATTTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTCTATATATATATATATTTTTTTTTTTTTAAAGACAGGATCTCACTCTGTCACCTAGGCTGGAGTGCAGTGGCATGATCATGGCTCACTGTAACCTTGAACTCCTGAGCTTGAGCTATCCTCCCACCTCAGCCTCCCGAGTAGCTGGGACTATAGGCACATACCACTGCACCTAATTTTTTTTTTTTTTTAATAATTTGTTGTAAAGATCAG
1051	chr1	186674934	186675185	-	0	train	TTTCTTTTCGAGATGGAGCCGCCCTCTGTCACCCAGGCTGGAGTGCAGTGGCGCCATCTCGGCTCACTGCAACCTCCGCCTCCTGGGTTCAAGCAATTCTCCTGCCTCAACTTCCTGAGTAGCTGGGACTACAGGCTCACGTCGCACGCATGGATAATTTTTTGTATTTTCAGTATAGACGGGGTTTCACCGTGTTAGCCAGGCTGGTCTCAAACTCCTGACCTAGTGATCCGCCGGCTTCGGCCTCCCGA
6929	chr11	5225979	5226230	-	0	train	TATGTGTGCTTATTTGCATATTCATAATCTCCCTACTTTATTTTCTTTTATTTTTAATTGATACATAATCATTATACATATTTATGGGTTAAAGTGTAATGTTTTAATATGTGTACACATATTGACCAAATCAGGGTAATTTTGCATTTGTAATTTTAAAAAATGCTTTCTTCTTTTAATATACTTTTTTGTTTATCTTATTTCTAATACTTTCCCTAATCTCTTTCTTTCAGGGCAATAATGATACAATG
8210	chr12	49297183	49297434	+	0	train	GAGGCCCTGCGCCAGGCCAAGCAGGAGATGAACGAGTCCCGACGCCAGATCCAGAGTCTAACGTGCGAGGTGGACGGGCTGCGCGGCACGGTGAGTACGAAGCTGCGCGCTCGGGCCCGGGGAGCGGACGATGAAATGTTCTGCAACTGGCCCCTTCCACTCTCCTACCCCAGAACGAGGCGCTGCTCAGGCAGTTGAGAGAGCTGGAGGAGCAGTTCGCCCTGGAGGCGGGGGGCTACCAGGCGGGCGCT
14885	chrX	153864301	153864552	-	0	train	GGGAAGGGGGCTCTGGGCCAGAGGGGTTGCCTGGCACTCCGACTCACCCCTGCTGCCACCCTCTCTCCCTGGCAGAAGAGAAGGGTGGGGCTTCCCTTTCGCCACAGTATGTCAGCTACAACCAGAGCTCCTACACGCAGTGGGACCTGCAGCCTGACACTGACTACGAGATCCACTTGTTTAAGGAGAGGATGTTCCGGCACCAAATGGCTGTGAAGACCAATGGCACAGGTGAGGCGCCGGGGGCCCGC
2761	chr4	71770920	71771171	-	0	train	TCACCTGAGATGACTTTCCCAACCCTTAAAGTGAGAAGGATAAACCACTAAAAGTTAAAGCCACTTAAAACAGAGTAAGCAATTTATATACTTCCTACATTTTTTTCCAAAGATATAACATCACAGGAGGGAAATTTCCTGGGCTTATTTGCATAAATGATGAGTCAGGTTAGGAGAACACTGACTGACTTTGATGTGTCCGCCCTATTACATATGTGATGGACATGTTGAACAAGGGGGTCTCATGATTA
12492	chr19	44929736	44929987	+	0	train	AGCTACTTCGGAGACGGAGGCAAGAGGATCACTTGAGCCCAGGAGGTTGAGGCTGCAGTGAGCTATGTTTGTGCCACTGCATTCCAGCCTGGGTAACAGAATGAGACCCTGTCTCATCCAAAAAAAAAAGAAAGAAAGAAAGAAAGAAAGAAGGAAGGGAGGGAGGGAGGGAGGGAAAATCTAGTCAGGCCTAAACTTAGAAAGATTGTTTGGAGGCCAGGTGCAGTGGCTCATGCCTGTAATCCCAGCAC
13395	chrX	147922601	147922852	+	0	train	GTTTTGAATGCTAATAAAGGCTATTTTCAGGGTAGCTGTTGGTAAGATTGTAAAGTACATATAAACTCCTTAGTCAAAGTGTAGATGTGGCTATGATCTTAGGATTTTACTAAACTCTGATGGATGGTTAACAGTTATCATTTTTTTGGCTCTTATATACCAAGAAAATTAATAATATATCAAAAGCAGGCTGCAAATCTATAGAGACAGAAAGTAGATTAGTGATTGCTTGTGCTGGGGCTGGTGGGGAG
14432	chrX	149489801	149490052	-	0	train	GAACCCTCCATTTGTGTGGCAGAGCTCTTGTAACCCCCATTTTTAACCTGGAGGGTTGGAGGACTTTTAGTTTGGGTGGAGAGGATCCAGAACAATCCTGGCAGAGCCCAGCAAGCTCTTCACGCCTGGCTCCAGCCCTCCCACCCCTATCCCCGCTGTCTTCTCTGCCGAGAGCCTGGGCTTTTCAAGTCTTTATCTCCCCCTAAGGCTGTTTCCTACTTTTCCAAAAATGAAACTATCTTTTTAAAAGC
11043	chr16	2130199	2130450	-	0	train	GATGATGGGGAGGGGGCTGGCCTGGAAGGACCCCCAGTGCAGGTGACATTGAAGCCAGGTTTCAAAGCTCCCACAGGGAGCTGCCCAGAGAGAGTCCCCAAGGGGCAAGGTGACTCGGGGGCAGGGGTAGGGCCTCTGTCAGGAGAGCCTAGGAGAGGCCTGTGTCTTCTAGGAAGAGCCCTGGCAGCCGAGCGGAGGCAGTGGTGAGGACCTGCATCCTGCATGTCCAGCTGGCCTCACCCGGGGTCCCT
9250	chr15	50253196	50253447	-	0	train	TCTTCCAGGGATCCCTGGACATTGCCTTTCCCAGTGGTGTGACAAGAGTTAGAGGTGGCCTACCTTGCCTCCTGTCCTAGGAGGCGACAGTAGGAGAGCCTTCGGTTTTCTCATCCTCTTACTTGTATGTTGAACTTTACTTAATGCAGGCTATATCCAAAATACAGTGTGAATTAGGCTAGAATAGAAATGCTTTCTATTCTACCTTCAGAAAAGGAAAAGGGTAGGGTAGGGGAAATCATGGGAGAGAT
11246	chr16	28933665	28933916	+	0	train	TGACAGAGTCTCGCTCTGTTGCCTAGGCTGGAGTGCAGTGGTGCCATCTTGTCTTGGCTCACTGCAACCTCCGCCTCCCAGGTTCAAGTGATTCCCCTGCCTCAGCCTCCTGAGTAGCTGGGATTACAGGTGCCCACCACCACGCCTGGCTAATTTTTTTTTTTTTTTTGAGACGGAGTCTTGCACTGTCACCCAGGCTGGAGTGCAGTGGCACGATCTCAGCTCACTGCAACCTCCACCTTCCAGGTTCA
9710	chr15	90892503	90892754	+	0	train	GGGTGGCACATCGGAGGCAACTTTCCCTGCCTGCCCCATGTGCTCTCTAGGTTCCCCAGCGAGGGTCAAACTCCCAGAGAGCCTGGGTGAGGGGTCCGAACACGGGGGCCCCTCACCCAGGGGTAGGAAGCAGAATGGGTAGGAAGCGGAGAAGAGAACTGCGGGACTGGGAAGGCCGTGGTAGGAGCCCAAGACCGTTTCAGGGGAACTTTGGCGAAGTGTTCAGCGGACGCCTGCGAGCCGACAACACC
14707	chrX	149500531	149500782	-	0	train	GGCCCGACAGGAGCAGTGGCTTCATTCTAGAAGCTGGTGGGTGCTACTTGTTCTAAAACCTTGGGCTCTAACGTGTCAGACTCATTGAATAGCTCCCCAGTGCTCACGCTGGATGAGACTGAGAGTTGTAGGAGATGCTGAGACATGGGAGGGAGAAAAGAGGCTGAGCTCCAGGAGCCTCAGCCCAGAATGGGAGAAAGGCATGCCGTGTATCTGTCTCTGACCTAATGTGTATGTTCAGCAGCTGAGAC
8411	chr14	24575500	24575751	-	0	train	CAATCCTGTACTTACAGCAAAGCATTCTCCTCAATACCTGAGGCTGAAGCTGGCCTTGCCTGGAACAAGGGTTGTTCTCCCTCTTTTGGAGAGGAGGAGGGAGGTGAGGCCTAGGATGGGGAAAAGGGCTCCTTTCAAGACAGCAGTGTTTCCTGTAGAACCCTGGAGCCCCCTCCCAATCTGCTGCCCCATAGACTCCAAGCCTCAGCACCATCTCCTCCCTCTCCTGCACCCTCTCTCCTGCCGTCCCC
13015	chrX	106035081	106035332	-	0	train	TTAATAGACAAGACCACCACTGTTCAAGTGCCCATGATGCACCAGATGGAACAATACTATCACCTAGTGGATATGGAATTGAACTGCACAGTTCTGCAAATGGACTACAGCAAGAATGCTCTGGCACTCTTTGTTCTTCCCAAGGAGGGACAGATGGAGTCAGTGGAAGCTGCCATGTCATCTAAAACACTGAAGAAGTGGAACCGCTTACTACAGAAGGGGTAAATGCCTTAGAGGATGTGTGGGTGGAA
6910	chr11	5225936	5226187	-	0	train	TCTTTTATTTTTAATTGATACATAATCATTATACATATTTATGGGTTAAAGTGTAATGTTTTAATATGTGTACACATATTGACCAAATCAGGGTAATTTTGCATTTGTAATTTTAAAAAATGCTTTCTTCTTTTAATATACTTTTTTGTTTATCTTATTTCTAATACTTTCCCTAATCTCTTTCTTTCAGGGCAATAATGATACAATGTATCATGCCTCTTTGCACCATTCTAAAGAATAACAGTGATAAT
3585	chr4	154607402	154607653	-	0	train	TTCTGTTCATTCATTAAGGCCCATCCTTTCCCCCACTCTATAGAAGTGTTGTCCACTTGCACAATTTTTTCCAGGAAAGAATCTCTCTAACTCCTTCAGCTCACATGCTTTGGACCACACAGGGAAGACTTTGATTGTGTAATGCCCTCAGAAGCTCTCCTTCTTGCCACTACCACACTGATTTGAGGAAGAAAATCCCTTTAGCACCTAACCCTTCAGGTGCTATGAGTGGCTAATGGAACTGTACCTCC
14858	chrX	153863251	153863502	-	0	train	AAAGATGAGACCTTCGGCGAGTACAGGTGAGCCGGGGCAGGAGTGGGTGCTGGCACTCAGCTCCACCCTGGCCTTCACATCTCACCCCCTCTCTCTCTTCTCTGTGCTGCCGGGTGACTTCAGGTCCCTGGAGAGGTAAGGCAGGGGCGGTGGGGAGGGGCCAACAGCAAGGTCTCCCTATAGACAATGTGCGCCTGGTGTGCCATTGTCCCCTGGCTGCCGGGGCTCTGACAGGACTCTCACTTCCCAGC
13287	chrX	147918253	147918504	+	0	train	ATTAGTGAAAGACCTTAGAATTCACAGGAGGTATTTGTCCTTCACGCGAGTGTAGACCAAACGTAACCTATGAGTTTCTTTTATTCCACTTATTAAAGCAGCAACCAAAGGTATTATATACCTTCTGTATTCACTTAAAATGACTGATTTTGAAAAAGTCATGCAAACATCCATTTACAGATAAGCCTCATTAACTCAAAGGCAGTGGCCCTGTTGGGCTCTGATGATTACTCAAACCATTTCTGACACTC
12020	chr18	46087302	46087553	-	0	train	TTATAGTAGCCTCCAATTTAAAACCACAATATTGATAAAATTGTAAAATATATTTTCTCAGATGCCATGAAGTACACCATTGTGGTGTCGGCTACGGCCTCGGATGCTGCCCCACTTCAGTACCTGGCTCCTTACTCTGGCTGTTCCATGGGAGAGTATTTTAGAGACAATGGCAAACATGCTTTGATCATCTATGACGACTTATCCAAACAGGTCAAAGGAAATAAATTTTTAGAATCCATTTATTTGTA
5093	chr7	44063289	44063540	-	0	train	TGGGAGGCTGAGGCAGGCAGATCACCTGAGGTCAGGAGTTTGAGACCAGCCTGACCAACATGGTAAAACCCGTTTCTACTAAAAATACAAAATTAGCCGGGTGTGGTGGCGCTCACCTGTAATCCCAGCTACTTGGGAGGCTGAGGCAGAATCACTTCAACCCAGGAGATGGAGGTTGCAGTGAGCCAAGATCGTGCCACTGCACTCCAGCCTGGGCAAGAGGAGTAAAACTCCATCTCAAAAAAAAGAAA
14147	chrX	147946599	147946850	+	0	train	TTATAGGTGAGTGCACTGTAGCTTCAGTCCTTTGCCCACAGTCACATAGCTAGCATATGTCCAAATGATAGTCAAACCTTAGTGTCTGATTCTCTGAATCTATGTTGTCGTCTACTTTTTCTGTAACTTTTAATTATTAAGTAGTTAGGGTTCCTTCCTGACTATGGATTCATTTGGGACCATCACCCAAGATTAGTGAGAGATACTTTTTAAGACAAAGTTTATGATGGAATATTTCTTGGAATTCATAG
14287	chrX	149484913	149485164	-	0	train	AGTGGCCTTACATGTTCTGGCCCCATTATCTCCCTGACCTCATCTTTTTATAAGTATCCAGGCCAGTTGTCCTATAACAGTGTCCCACAGCCTGGATTTATCTGATTGCTTCCACATGACTACATTCAGGGTAACATTTTTGACACGTGTGCTACATGGGCTGTTGTATTCTCCCATTGTGTCACATGGTGGGGGCACTTCATGGCCAGCGTTACTAGTATTGTAAGTTTGAACACTCGGTTGAAGAGCTA
7481	chr11	64255044	64255295	+	0	train	TGGCACCAAGGGGACGAAGGGGGAGTCACTGTCTTATTCTGTGAGTCGGCCGTCACTTACCGAACACCTGCTGTGTTCTAGGCTGCTCTGTGACCTACTGTGTACCAGAGGCAGTTGTGGACCTGGGTTGTGGCTGGGCAGCCCCTGTGTCCCCCACTCACCGCCTCCCCGTGTATACTGGCCCCCCAGGTCTGGTCTGAGGAGCTATTCAAGCTGGCTATGAACATCCTGGCTCAGAACGCCTCCCGGAA
10170	chr16	2099178	2099429	-	0	train	TTTGCTAAGCAGACGCCACGGACGACTGCACAGCAGCACGCCAGATAACTCAGAAGAAGCAAGCACGCGGCTGTGCACGCTTCCGAAATGCACTCCAGAAGAAAATCTCAGTACATCTATAGGAAGTGAAGAGGCTGAGTTAGTCCCTTAGAAACGTCCCAGTGGCCGGGCCGGGTGTGGTGGCTCACGCCTGTAATCCCAACACTTCAGGTGGCCGAGGTGGGCGGATCTGAGTCCAGGAGTTTGAGACC
12728	chr22	20780469	20780720	+	0	train	ATGAAGTAAAAGGATGGGCTGGGCGCGGTGGCTCACGCCTGTAATCCCAGCACTTTGGGAGGCCGAGGCAGGCAGATCACTTGAGGTCAGGAGTTCGAGATCAGCCTGACCAACAGACCAACATGGTGAAAACCTGGCTCTACTAAAAATACAAAAATTAGCTGGGCCTGGCGGTGGGTGCCTGTACTCCCAGCTACTTGGGAGGCTGAGGCAGGAGAATCACTTGAACCTGGAAGGCAGAGATTGCAGTG
5518	chr7	99972335	99972586	-	0	train	CCTGCCTTGGCCTCCCAAAGTCCTGGGATTACAGGCATGAGCCACGGAGCCCAGCCTAGAAATGTTAATTTCTAACGCATGTCAGATTCCATGCACACTGGGCAAGGTTCCATTCCTCCATGGGGTGACTCAGGGATCCAGGCCAATTGCATATTGAGACTCTTTCATATTATCCTGTGGCCTTCAAAGTCGTCACCTCTAGGGATGAGAAACAAAAGGGACAAGCCAGCTGGTAGGGTCTTGGACAAGAA
14363	chrX	149487606	149487857	-	0	train	GTGTGTTTGGTTATTTTTATAACACTACTCACTGTACAATTTCTGTCATGCACATTTTTTGTATCAAAAAGGGTGCTGTAACTTTAAAAGACTGGGTATTTCCAACTTGGAATATTCAAACCATCTTTTGCAAGGGATGTTTTAAATAGGCATGAAGGGTTGTTTTTAATTGAGGTTAAGGATCTGAAATGAGAGGTTTTGGTTTACCCTATCTATGGTATGTCTTAAAAATCAACGAAGATGTCCTTGTC
11669	chr18	31592489	31592740	+	0	train	TATAACAACTGGTAAGAGGGAGTGACTATAGCAACAACTAAAATGATCTCAGGAAAACCTGTTTGGCCCTATGTATGGTACATTACATCTTTTCAGTAATTCCACTCAAATGGAGACTTTTAACAAAGCAACTGTTCTCAGGGGACCTATTTTCTCCCTTAAAATTCATTATACACATCCCTGGTTGATAGCAGTGTGTCTGGAGGCAGAAACCATTCTTGCTTTGGAAACAATTACGTCTGTGTTATACT
8957	chr15	50242486	50242737	-	0	train	GTCCCTTCAGTCTGTCAGTGGGGCAGGAGATGATCCAGTCCAGGCCAGGAAGATCATCAAGCAGCCTCAGCGTGTGGGAGCCGGTCCCATGAAAAGGGAAAATGGCCTCCATCTTGAAACCCTGCTGGACCCAGTTGATGACTGCTTTTCAGAAGAGGCCCCAGATGCCACCAAGCACAAGCTGTCCTCCTTCCTGTTCAGTTACTTGTCTGTGCAGACTAAGAAGAAGACGGTGCGCTCCCTCAGTTGCA
6825	chr11	5225627	5225878	-	0	train	AACTGATGTAAGAGGTTTCATATTGCTAATAGCAGCTACAATCCAGCTACCATTCTGCTTTTATTTTATGGTTGGGATAAGGCTGGATTATTCTGAGTCCAAGCTAGGCCCTTTTGCTAATCATGTTCATACCTCTTATCTTCCTCCCACAGCTCCTGGGCAACGTGCTGGTCTGTGTGCTGGCCCATCACTTTGGCAAAGAATTCACCCCACCAGTGCAGGCTGCCTATCAGAAAGTGGTGGCTGGTGTG
7427	chr11	14970664	14970915	-	0	train	TCATTATAATTTTTTTGAGACAGAGTCTCACTCTGTCATCCATGCTGGAGTGCAATGGCACCGTGTCGGCTCACTGCAACTTCCATCTCCTGGGTTCAAGCAATTCTCCTGCCTCAGCCTCCCAAGTAGCCGGGATTACAGGCGCCCACCACCATGCCCGGCTAATTTTTGTATTTTTAGTAGAGATAGGGTTTCACCATGTTGGCCAGGCTGGTCTCAAACTCCTGACCTCAGGTGATCCACCTGCCTCA
7573	chr11	64263798	64264049	+	0	train	AGCTGGGGGTGGGCGGGGCCTGCCTGGCCAGGGAGTGTGAGGGACAAGGGCCACCCCCAGGGCCTGGGAGTGGCCGAGCTGGATGGCCCCGACTGAGTAGGGAACTGAGTAGGGAACAGATCTGAGGACAAGCTCTGGAATTCCTCATTGAGACCAGAGGTGGCGGGTGGGGGTGGCCTGGGGGCTCTGTCTCTGAGACCTTGGCCTTCTGCCTCCCCCCAGACTATGCGGAGGCCCTGATCAACCCCATT
7898	chr11	119088369	119088620	+	0	train	CAAATTGGGGGACTCAGAGGGTTAGTTCCTAGTATGAAGGAGATGGGGTGGCTGGGCGTTAAGTTCCCCGGGAAATGGCAGATTACATTCTATGGCAAGATCATCCCTAGGCTGGGAAAATTGTTGGAGTGCAGAGGGCTCCCAAGCCCCTTCTCATGCCCAGATGGAAATTCCAGTCCCTTCAGGATCTGCCTAACCTGTGACAGTCTAAAGAGTCTGAGCCGTGGCTGGGAAGGGCAGGACTAATCCAA
2657	chr4	71765373	71765624	-	0	train	AGTCCTGTGAAAGTAATTCTCCATTCCCCGTTCACCCAGGCACTGCTGAGTGCTGCACCAAAGAGGGCCTGGAACGAAAGCTCTGCATGGCTGCTCTGAAACACCAGCCACAGGAATTCCCTACCTACGTGGAACCCACAAATGATGAAATCTGTGAGGCGTTCAGGAAAGATCCAAAGGAATATGCTAATCAGTGAGTGCCTTCATCATAAATAGAACTTTAGGACCTAAAGTATCAGAAATGACTCTAA
11181	chr16	18400139	18400390	-	0	train	TTAGGGGAGCCGTGTGTCCTTGGGGCTTTGCTGGGTGGTCTCGAGGGTGGGAGAAGAATGGGTTCTCCTGGACCAATGGAGCCCGTGCCCCTCGGGGCCACATTGCTCCTGCGCTCCCTGACTGCGGACGCGTGTGTCTCGCGGCTGTCTCTGTGGAGATGGCCTCCTCCTGCCTGGCAACAGCACCCACAGAATTGCATCAGACCTACCCCACCCGTTGTTTGTGATGCTGTAGCTGAGGGCTCCTCTGT
2658	chr4	71765424	71765675	-	0	train	ATATATGAGTTTCCTTTTTCCTTCTCCTCCAGACCTCAGCACTGTCTGCCAAGTCCTGTGAAAGTAATTCTCCATTCCCCGTTCACCCAGGCACTGCTGAGTGCTGCACCAAAGAGGGCCTGGAACGAAAGCTCTGCATGGCTGCTCTGAAACACCAGCCACAGGAATTCCCTACCTACGTGGAACCCACAAATGATGAAATCTGTGAGGCGTTCAGGAAAGATCCAAAGGAATATGCTAATCAGTGAGTG
1481	chr2	10442095	10442346	-	0	train	GCCTGGTTTTCAGGGAGATGTTGATGTTTTTTTGCTTTTGTTACTTTAATGATAAACCTGTCTGTTGATGCCTGGTCTCATGATGTCATGTCACAAGGCCCTGTGATGTTACTCCCCCATGTGAATTTCCCACAATGAAGGCTGCTCTTTCTTTTCTGTTTCACTCTCTTAGATCACCGGCGTAATCAACCCAGCGTTGGACAAATACTTTCCGTCAGACTCTGGAGTGAGAATCATAGCTGAGCCCGGCA
10889	chr16	2125067	2125318	-	0	train	CCCACCCCTCGTGGGGCCAGGGTTGTGGCCGGCCTCCCTGGCCCTCCCCATGGAAGTGGTAGGCGGAGCCAGCAGCCATCTGCCCAGCCCGGGGCTGCACTGTTTTTTTTCAAATGAGCACCGTCCCAAACTGCAGCCCGTTAATTTAAACAGGATCATTTCCGGCCCTGGAAGCCGCCTCACTCTCCTTAAATAGAAAGGAGCACAGCGCAGAGGGAAACAGATGAGGTCATGGCTCGGCTGGCCCAGCG
1484	chr2	10442567	10442818	-	0	train	GGATCACTTGAGCCCAGGAGGTTGAGGCTGCAGTGAGCCATGATCATGCCACTGCACTCAGCCTGGGCTACAGAGTGAGACCCTGTCTCAAAAAAAAAAAAGAAAAAGCATGTTGCTGTGGGCTTCCTAGAGAATATGCTGACTGTAGCACATCATCACCCCAAATGTGCTTTGCTAGACCTATGCTTCCTCTCCTTAAAATACTTGAAATGTTTAGTCACTTAGGAAGTTAAGCCATTATATTGGTGCTT
9770	chr15	90894726	90894977	+	0	train	CTGCACCCCAACCTGGGTGACAGAGAGAGAGAGAGACCTTGACTCGAAAAAGAAAAAAACCTGGGCGCAGTGGCTCACGCCTGTAATTTCAACATTTTGGGAGGCTGAGGAAGGTGGATCACTTGAGTCTAGGAGTTTGACACTAGCCTGGCCAACATGGCAAAACCTGTCTCTACTAAAAATACAAAAAATTAGCGAGGTGTAGTGGTGCAAGCCTGTAATCCCAGCTACTTGGGAGGCTGAGGCACAAG
10537	chr16	2113530	2113781	-	0	train	GCTGGGTACATGGGGGACAGGGCTGTCTCCATCTTGCGGGTACCTGCCTCTTCACCAGGGGCCTTGGGAGGGGCCATCAGAAATGGCGTGACCTGTGCAGCCTGTCCTGGGTTCTGTAAGCCAGTGTAGGTGCCTCCCCTCACTGCTCCGAGCTCTCTGGGTGAGGAGCTGGGGCAAGAGCGCCGGGAGGGTCTGAGAAGACTCAGAGAGAGGTGGACTCTTTGTAGCTGGTACTAGGTTTGCTTTACAGA
5541	chr7	99972595	99972846	-	0	train	TGAGCCCGGACGAAATGTTAATTTGTTTTTTTTGAGACGGAGTCTCACTCTGTCATCCAAGCTGGAGTGCAGTGGCATGATCTTGGCTTGTTGCAACCTCTGCCTCTCTGGTTCAAGTGATTTTCCTGCCTCAGCCTCCAGCATGACTGGGATTACAGGCCCGCACCACCATGCCCAGCTAATTTTTGTATTTTTTAATAGAGATGGGGTTTCACCATGTTGGCCAGGCTGGTCTTCAACTCCTGATCTCA
8711	chr14	75280128	75280379	+	0	train	GTGAGGAACTCTAGCGTACTCTTCCTGGGAATGTGGGGGCTGGGTGGGAAGCAGCCCCGGAGATGCAGGAGCCCAGTACAGAGGATGAAGCCACTGATGGGGCTGGCTGCACATCCGTAACTGGGAGCCCTGGCTCCAAGCCCATTCCATCCCAACTCAGACTCTGAGTCTCACCCTAAGAAGTACTCTCATAGTTTCTTCCCTAAGTTTCTTACCGCATGCTTTCAGACTGGGCTCTTCTTTGTTCTCTT
14403	chrX	149488624	149488875	-	0	train	TGGGCCTCTGGGGCAGGAGCCAGAAACCTCCTGAGCTGCCTGTCAAGGTCATCGGGCTTGTTCCTCCTGATCTGGAGATGGATCCTTGGGGCCCACACAGGCACGCGTTCTCCTTAGCCAGACTTCCCCGTATTTGCTCCTGGCTGCAGCAGCACAGGCTGAGGCCCGGCACCAGATGTTCAATACGATCTTGCAGTCAGGCGGTCCACAGCTATTTCTGTAGCATTTGCCATGTGTCAGACCCTGTGCCA
6639	chr10	47352004	47352255	+	0	train	GAGAGCTCCACCTAGGCAGCACTCACACCTCCACACTGTTCTACCTGTGGTCTGCTGCATCGTCACAATTGGGCAGGGCAGCATTTGCCATGGGATCCCTTGCAAGGAGGGTCTGAGACCAGGGCTTGGGTGCAGGCGCTTTGTCTGGGAGGCGGTTACTGAAGCAGGCATGAAGGAGGGAGCAGGGAGAGTGGGTTGGGAAGCTGACAGGCAGGTGCCTCAAAGCTGTTCTGCTGAAGCCAGGACCCTGA
3185	chr4	87980955	87981206	+	0	train	TCAATGGGCAGTTTTGAGCTGCAGTTTATACACACATGCATAACAGAGTCACCTTTCAATTATCCATGTTAATAGGAAAGTGGTTATAGATTTTAGTACACACATTAAAATATGGATACTCTTCTCTTTTGATAAATCTCATTTCAAATAAAAAAACCAGTCTCATAATTATGTATCTGTATCTATTACATCATTGAATTTAGTAAATAATGTTTAATATGTATAAGGAAAAACAATGTTATTGACATGAA
6263	chr10	47301374	47301625	+	0	train	CCCTGTCAGGAAGCAGGAGCCCCTAGCTGGCGTCAGCTATGGTTCCTGTGTCCAGGGAAGCAGAGTGGACTGAGAAGGGCCTTGGCTGCCCGGGATTGAGGACTTCCACGGACTGGCATGAAGCGTTCAGACCACAGGGGTTTAAGGAAAGGCCCAGGGGCTTAGGCAGTGGGAACACTCTGCCTATCTCCGGCCTCTGTCCCACCTTTCCACTTCCTCAGCCCCTTGGATGGAGCTGGGCAGAGGGCACT
8553	chr14	74888036	74888287	+	0	train	ACCATATATGTCTACATCTCTCACCCCTCAGAGGTAACCACTGTTAATGGCTTTGTGCATACAGGGTGAGCTGATTCTCCAGTTTATATTATTTACTGTAAAGGCCATATCAGTAAACGCCTTGGTCTTTGAGGTTCGTTTTGAATGAATCCAGAATGTGTTGATGTGCTGTTTTCTAGCAAGAGAACCGAGTAGGAGATACCATATGGGGTTTGTGTGTGGTTTTTTTGTTTTTGGGTTTTTTTTTTTTT
14989	chrX	153867239	153867490	-	0	train	TTCCAGGGAACCAGACCTCTACCACCCTCAAGCTGTCGCCCTATGTCCACTACACCTTTAGGGTTACTGCCATAAACAAATATGGCCCCGGGGAGCCCAGCCCGGTCTCTGAGACTGTGGTCACACCTGAGGCAGGTGAGTCAGGGTGGCACCCACTCCCATGCCACCTGGAAGGGGCTCCGGGCTGTGAAGGGAGGGCTTAGAGAGGTGCCATGCCCAAGTTCCTCTGCTTCCAAATTTCCAGGGGGAGC
4741	chr6	32949425	32949676	-	0	train	CTTCTGACATTTTCTCCTGCATTGTGACTCACGAAATTGACCGCTACACAGCAATTGCCTATTGGGGTGAGGCTTTCTCCCTGGAATTCTGGTCCTTTTGGGGGCAAAAAGGGATAGATCCATGGGAGGAGGCTTCTTTCTCCACTGGTACCTTGTTTAGTCCATTCCTACCCTAAGCCCATCCCAGTCTCCCATGTCATCCCAGACACCCACGTCATTTCCCTGGGTGGGAGGCTCCCTAACTAGGTCCC
4834	chr6	33069253	33069504	-	0	train	ACTACAGCAAGGGCTTGCATCCTCTCTTCTCAGGAGAGAGAAAGGTGAGCAGAGTGAGGCTGGTCAGTGGTGTGATACCCCTCTCTGTGATTCAGAGCTGCCATAAAATCTAAGGCTGAGGTAGAGGACCACCCTCCCCTAAGAGGTGGAGCCTTTGTGATTCATCCCAGAAGAGGGGCCTAACCTGGTGCTGTCTCCTTCCAGATCCCCCTGAGGTGACCGTGTTTCCCAAGGAGCCTGTGGAGCTGGGC
10980	chr16	2128368	2128619	-	0	train	GCACCGCAGGGTTGCAGCCACTCCTGGTCTCATTTTACACACCAGGAAATTGAGGCTCTTTGAGAAGCCGTGGTGATGATTTCATCAGCATGCTCTGGGGCAGACCCCTGCAGCCGCACAGGGTGCCTGGGGCCCACACTAGTGCCCTGGTTTATAGACAGACAGAGGTGGCAGTGGCGCTTCCGAGTCGGGCTGCGATGTGCTTGCACTCCCCGAGGGGCTGAGGGGCCCTGCGCCCAGGTGCAGCTGCT
9773	chr15	90894760	90895011	+	0	train	GACCTTGACTCGAAAAAGAAAAAAACCTGGGCGCAGTGGCTCACGCCTGTAATTTCAACATTTTGGGAGGCTGAGGAAGGTGGATCACTTGAGTCTAGGAGTTTGACACTAGCCTGGCCAACATGGCAAAACCTGTCTCTACTAAAAATACAAAAAATTAGCGAGGTGTAGTGGTGCAAGCCTGTAATCCCAGCTACTTGGGAGGCTGAGGCACAAGAATCGCTTGAACCTGGGAGGTGGAGGTTGCAGTG
13989	chrX	147940751	147941002	+	0	train	TAGCCTGTGCAAAACACTGTATGACTTGTCAAAGGTAATAGAGAAAATACGTTGTTTGGCTTATAATTTTTTTAAAAATGGAAGTCTTTATAAATTTAAGTACCAATATACCAGTTTTCAATAAAATTTTATACTTCCCTTTATTCTTCTCTTAAACCCTTACACTCAGTTTAGGCAATCCTGTACATAGCCTGTTAATCCATTTGATCCTTTCTAGCATTTTGGTTTTTTCCAGACAAAAATCTGTGTCT
12184	chr18	46093128	46093379	-	0	train	TTTTTTTTCCTTAGTAGAGATGGGATTTTGCTGTGTTGGCCAGGCTGGTCCAAACTCTTGGCCTCAAGTGGTCCGCCCGCCTCCCAAAGTGCTGGTATTACTGGCGTGAGCCACTGCACCCAACCCCTGTATGCTAAGATGAAGAATACCACCTATACTACCCTGTGTGCCTTGAAAGAGAAAGACACACTTTGTTAACAGTTGATTTTTATTTTTTATTTTTTATTTTTTTGAGATAGAGTCTTGCTCTG
5629	chr7	99974395	99974646	-	0	train	ACTTTTTTGGTTATTTTAGTTTTTAAAAGTATTTGATTATTTATTTATTTATTTATTTTTGAGACAGAGTCTCACTCTGTCACCCAGGCAGGAGTGCAGTGGCATGATCTCGGCTCACTGCAACCTCCGCCTCCCAGGTTCAAGCAATTTTCCTGCCTCAGTCTCCTGAGTAGCTAGGACTACAGGCACCTGCCACCACACCTGGCTAATTTTTTTGTATTTTTAGTAGAGACGGGTTTCATCATGTTGGC
706	chr1	159304708	159304959	+	0	train	TCAATATTATTCCTCCACCCTATTTTCCTCTATCTTTTCTGCCTAGATTCAGGTATATATTATGTGGTCAAACAGCATGACATATATGTGAACATTTCAAAGAGCTGTGTATCTGGAATAGGATCAAAAGGTTTGACTTAAAGTTTTGCTCTGCATAATCCATATGGCAGGACCTGAATATTAGGTTGTACTCTTCGTTATGAAACATATCTGGGTACATTTCCTTATGTCCTCTGTTGTTACTTAAGAAC
12195	chr18	46094313	46094564	-	0	train	CTCCTGACTAGCTGGGATTACAGGCATGCACCACCATGCCCGGCTAATTTTTTGTATTTTAGTAGAGACAAGGTTTCACCATGTTGGCCAGGATGGTCTCGATCTCCTGACCTTGTGATATGCCTGCCTCGGCCTCCCAAAGTGCTGGGATTAAAGGCGTGAGCCACTGCACCGGTCCACTGATAGGTTTTATTTTTTCAAAGGCAGTAGCTACTAGATATTTGGCAATCTGTGAACTTGCACATTACAGA
951	chr1	173911965	173912216	-	0	train	ATCCTTCCTACCCTTCATTCTTCTTTTATCCTTTTATTCATCAGAACACAAGAGTTGAGCATTTATGCTGTCCCAGGTACTGTGCTTGAAGGAGTTAACAACTGAGGTGGCTATTAGTCAGAGACTGACCAGCATGTGCTCACCACCCATGTTAACTAGGCAGCCCACCAAACCCACCACCATTTTTTTTTGACTTCTATAGGTATTTAAGTTTGACACCATATCTGAGAAAACATCTGATCAGATCCACT
13271	chrX	147917860	147918111	+	0	train	AAGATTTTGAACGTAGGTCTGATTCACAGCAAAACCGTTAACCACTAAGTACACTGACTCCAGTAAGAGCCCTAGTCCTCACCCAATACACTTTAATTCCCCTGTGCATTCATTCAAATTCATTGAATTTGCTGCCTTTGGAAACCTCTCAGGAACCTCCTCAACCTCTCTTCTCTACAGACATCAGCTTTGCCTGATAGGTAGGGATCATAGCAAAACACAGTTTTCCAAGGTGGTGATAGGTGGAGTGA
7259	chr11	5254159	5254410	-	0	train	AGAAGGTGCTGACTTCCTTGGGAGATGCCATAAAGCACCTGGATGATCTCAAGGGCACCTTTGCCCAGCTGAGTGAACTGCACTGTGACAAGCTGCATGTGGATCCTGAGAACTTCAAGGTGAGTCCAGGAGATGTTTCAGCACTGTTGCCTTTAGTCTCGAGGCAACTTAGACAACTGAGTATTGATCTGAGCACAGCAGGGTGTGAGCTGTTTGAAGATACTGGGGTTGGGAGTGAAGAAACTGCAGAG
9521	chr15	50263678	50263929	-	0	train	GGAGATTTTCCTTTTTGTTCATTTGCTCACTTAGAAACTCCCTCCATCATTCATTCACTCCCTCACTCTTTTTTTAAATTCTGAAGATAAGGTCGTGCAGGGGATCAGATCCTACCCTCGGCCCATGAATCTGTCTCTCTCTTTTTTTTGAGTCCAGGCTGGAGTGCAGTGGCACGATCTCGGCTCACTGCAACCTCCGTCTCCCAGGTTCAAGCGATTCTCCTGCCTCAGCCTCCCAAGTAGCTGGGATT
6476	chr10	47311126	47311377	+	0	train	CCTGGACCCCAGAGTGTGGCAATCTGTGACTAGCACAGTGCTAGCCTCAGCAGCTCTCCCACTGTAGATTCCCTCCTCCTTGCAGCTCAGGGCAGGAATCCACAGGGAATAGGCCTTCACTGTGCACAGGGTCTGCAGGACACACAGCACTGGGCTGTTCTGTCTGTTGCACCAGCCTAAGGATTTCCAGTTCTCACCTTCGCTCCAGACTGGGCTGTGGCTGAGTCCATCCCTCTCCCTGCAGCCTGCTC
11147	chr16	2133276	2133527	-	0	train	TTTTCCCCAGGGAAGAGGGGTCAAGCTGGGAGAGGTGAAGGACACAGATCACAGCTGCTGGCAGGTGTTCAAGGGTCCAAGAGCGTTGCTGTCTGGGTGTCACCAGTAGCCTTCCTGGGGGGCTCACGCAGGTGCCTCTCCACTTGTGGCTCCCTGGCTGCTGAAGCTCAGCAGGGACAGCTGTGTCCAGTTCCAGGTGGAGGACAGCCGGGGCTTCTGAGGCCACAGCCTGCCTTGGGTTAATGATGCTG
12962	chr22	43160984	43161235	+	0	train	ACTCGGAGGTGGGCAACGCTCCTGGCCTTGTTCCTAATGGTGCTCTGAACTGCGGCCTCTGTTTCAGGTACGGCTCCTACCTGGTCTGGAAAGAGCTGGGAGGCTTCACAGAGAAGGCTGTGGTTCCCCTGGGCCTCTACACTGGGCAGCTGGCCCTGAACTGGGCATGGCCCCCCATCTTCTTTGGTGCCCGACAAATGGGCTGGGTAAGTGTGGCCACAGCATGTGTCCCTGATCCCTGGATCCGACCC
15786	chrX	154314314	154314565	+	0	train	ATACCAATATAAAAATTCTAAATAGAACATTAGCCCATCTCAACTTCATTAGTACATTAAAAAAATTAAATGAATCAGTGAGCTCAAATTAGTAATGCAGATGACTCATGCCAGGAGATGTAGGAATAGAATTCAGTGTTCTAGTGTTAGGTTAAAGGAAGGAGGGAAGTGTGATCATCATCATACACACTAAACAAAAGCAATTCATAAAACTCAACATTTTTTTTTTGAGATGGAGTCTCGCTCAGTCA
12141	chr18	46091206	46091457	-	0	train	CTCCAAAATATAAAATTGAATAAACTTTCACCCCAAATCCAAGAAGTTTAAGTCTGTTCTGTCTTGCTGTTAAGATTATTTTATACAAAAGCAAAGCATTTGATAACTTCAACAAAATTTTTCTGGTGGAAATTGGAAGTCCGTGACTTTGTATTTCTAGTGAGTAGACATATTGTTCAAAGGCGAGTTTTGGTGTAGCTATCAGAAGCTCTCTAATGTAAGTCTACTTTGTATTGGGTTGAGCCTCATCT
6543	chr10	47349173	47349424	+	0	train	CACCAGCAGCCAGACCAGGGGCGTGGCCGAGGACATCGCGCACATCCTTAAGCAGATGCGCAGGGCCATCGTGGTGGGCGAGCGGACTGGGGGAGGGGCCCTGGACCTCCGGAAGCTGAGGATAGGCGAGTCTGACTTCTTCTTCACGGTGCCCGTGTCCAGGTCCCTGGGGCCCCTTGGTGGAGGCAGCCAGACGTGGGAGGGCAGCGGGGTGCTGCCCTGTGTGGGGACTCCGGCCGAGCAGGCCCTGG
2273	chr4	71749129	71749380	-	0	train	TCATGCTTCCTAAAAGCCCCTGGTTTTTCATTTCTTTGAGTTGGTTTTCAATATAACATATTGCCATTCAAATCATTACAATGATGAGAAGACAGGTTTGTAGCTCTCTCACTTCTTGCTTTATCGACAGTAAATTTATAGCAGCTGAGTAGCCTTTCAGTAATTGCTAAAGTAGAGTGCGATGAATGGAGTTGTTGAGATACTTGTGCACATTTGATTATTCAGAGTACACAGAGATTAAATCCTCTGCT
9942	chr16	2092369	2092620	-	0	train	CTGGAGCGTGTGACTGATGCTGTGGCAGGTCTGAGGAGCTCTGGCCATGGATGGCCCACGTGCTGCTGCCCTACGTCCACGGGAACCAGTCCAGCCCAGAGCTGGGGCCCCCACGGCTGCGGCAGGTGCGGCTGCAGGAAGGTGAGCTGGCAGGGCGTGCCCCAAGACTTAAATCGTTCCTCTTGTTGAGAGAGCAGCCTTTAGCGGAGCTCTGGCATCAGCCCTGCTCCCTAGCTGTGTGACCTTTGCCC
7332	chr11	6393221	6393472	+	0	train	GGGGTTCTATGCTCTTTCCCCATACCCCGGTCTCCGCCTCATCTCTCTCAATATGAATTTTTGTTCCCGTGAGAACTTCTGGCTCTTGATCAACTCCACGGATCCCGCAGGACAGCTCCAGTGGCTGGTGGGGGAGCTTCAGGCTGCTGAGGATCGAGGAGACAAAGTGAGGGCCAGTAGTGGGAACACGGTGGTGCTGGGGGACAAGCAGGCTCCTGTTGAGCTGGAGCACCTCTGGGCACAGAAGTTTT
3124	chr4	73998092	73998343	-	0	train	CCCAGCTGGTCCTGCCGCTGCTGTGTTGAGAGAGCTGCGTTGCGTTTGTTTACAGACCACGCAAGGAGTTCATCCCAAAATGATCAGTAATCTGCAAGTGTTCGCCATAGGCCCACAGTGCTCCAAGGTGGAAGTGGTGTAAGTTCTGTGCTGCTGTGTCCGCTGTGACCTTGGCAAGAGAGAAATCCCGCAGCCTGGGTCTTCAACCTTGGTATCTCATGAGTGTATCTTCTTTTTCTTTCCTTCAGAGC
7649	chr11	69774401	69774652	-	0	train	CACCTGCTCGGGGGCCCCGGGAACCGGGGCGGACTCGGGCTCCGGTCCCTTCTGACGCGGGGCTGGGGACGCAGACACTCTTGGCTCCGGCAGCCCAGCGCAACCCCTGAGGTCGGGCGCCGCCTCCCGCCTTCAGAAACTCGGGCTCCGAGCGCCGAATTCCAGCGCCTTCGCCCGTGGGCACAGGGCGCGCGGTGCAGCCACAGGGGGCCCGAGACACGCGCCCCGGCCTGGCCCAGGCTGGGGAACCG
5691	chr7	99975752	99976003	-	0	train	TGTCCTGCTGTCTCTGCTGCTGCTTCTGGGTCCTGCTGTCCCCCAGGAGAACCAAGATGGTGAGTGGGGAAAGCAAGGGATGGGTGCTGGAGAGGACTGGAAGGAGGTGAGGAACAGGACATGTGGCTGGGAGACAGGCTGGATGCAGCTGGGATACCCTGGCATACGGCAGGAATGGGTGCCCAAGGCTGTCAACTCCCTCAGCTCACACACTTCCAGGAGCATTCAGGGAGCCTCTGCGCTGGCCCGAA
14808	chrX	149504423	149504674	-	0	train	CCCCTCCTTCTCTCCATCCTTTCTTCCCTTCAGCTATTCATCTTTTCCTCCTTATCTTCCTCCATCCAACCATCTGTCCTTTCCTGCATACATCATTCTCTTTTTTTCCTAAATTCATCTTTCCTTTCCACCATCTTTCACTCACTATCTCGCTTCCTCACCCAGGTTGGAGGCCATGACCAAAGCCTAACCCTGCCACCCAGGACTCAGGCTTCCTCCTCGAGCCCCACTCCCACCCTTGCTGAGGCACA
1906	chr2	162145648	162145899	-	0	train	GAGGGGGAGCTTTAGCCCACCGATTGTTATTAGCTCTCTTCCACCAGTTTCAATCCAGAACATTAATGTAGCTTCACGACAAATCCCCTAGCAACCCTCATCTCCTAAACTCCCTAAGCCCCTTCTACATAGAATCCTGAACCCAAAGTTGCCACTGCTTATAGGTGAGAACCATATTGCCAAAGAGACAGATCTTCAATTTAACTTTCACATTTCTTTCAGGAATAACATTGCCAAACGTCACGATGAAT
8579	chr14	74889122	74889373	+	0	train	TTTTTCAGAACTACAGCTGTATGCAGTAAGTACCTGCTTTCTTGGGAATGGAATTTTATGGGAAGAGCAACAATTGATTGAGTTTAATTAAAGAAAAATATTAAAAAGGAAAAATGAACTACTTATGATTTTCTTTTTTGCATTTTTCCTAGAGGATGACTTGGTTACAGTCAAAACCCCAGCGTTTGCAGAATCTGTCACAGAGGGAGATGTCAGGTGGGAGAAAGGTAAGATTTAGTTTCCTATTTTTT
14632	chrX	149497411	149497662	-	0	train	TTGGAACCTCAAATCCTGTACTTGCTTGTATGTCATTTAGAGCATGTAGTCTTTTCTGTTTTAGAATCCTTTTCCATTTTTCCCTATCATGTTTTATGAGGGCCTGAGCATCCCATCCTTTTGACTTTGCAGAGAATGCCCCTCACATGTAATGAGAGTAGAGACCAGCAGTATGCTCTTTGATGTTGCAGGTATCTTGCTTTGATTGGCCCAGGGGATCTTGCTTTCCTAGTAGGAGCTGTCAGCCCCCT
462	chr1	119508132	119508383	+	0	train	ACAGAGAGAGAGGAAAACAATAAGGGGGAGAGAGAGTCGGGGAGAGAGAGTCGGGGAGAGAGACTCAGAGAGAGAGACAATGAGAAAGACAGAGAAATAATGAAAAATATATAGTGAGAGAGAGAGCATCAGTGAGAGAATGTGTGTGCACAGTGCACAGCACAGAGCAGAGACAGAGTAAGAGGCAGTATAAGGCCAGACATGCCTCATTTAGATTTTGCATATATGGCTTTTTTTTAAAAAAAAAAAAA
11416	chr16	29462303	29462554	+	0	train	GTACCTGTTCTTCTGGAAGGGCCTCCTCGCTTCTGCCAGGCTCATCACATCTTTTTTTTTTTTGAGACAGAGTCTTGCTCTGTCACCCTGGCTGGAGTGCAGTGGCATGATCTCAGCTCACTGCAACCTCCGCCTCCCCAGTTCAAGTGATTCTCCTGCCTCAGCCTCCTGAGTAGCTGGGATTACAGGCGTGTGCTACCACACCCGGCTAATTTTTGTATTCTTTTTAGTAGAGACGGGGTTTCACCATG
15041	chrX	153869967	153870218	-	0	train	CTGCGCCCCAGTGGCCCCATGCCAGCCGACCGTGTCACCTACCAGAACCACAACAAGACCCTGCAGCTGCTGAAAGTGGGCGAGGAGGATGATGGCGAGTACCGCTGCCTGGCCGAGAACTCACTGGGCAGTGCCCGGCATGCGTACTATGTCACCGTGGAGGGTATGGACCTCCTGGGACAGTGGCCTGTGATGCCCACTGTCATGGGGGAGGGGAGGGTCTGCGCCGGGTCAAGAGCCAGCTGCTGGCT
12150	chr18	46091426	46091677	-	0	train	ATTATAAAATAAGATTTTGGTGTTTTAGGATCTAAGTCTTCTCAGTTATTCATTTAGCGATTCAAAGATTGTATCAGATTTTTAAAAAACTTTGTCTTAATTTGCCTCAGATTTTTTTCTTGTTATTATCATAGATTGTAAAGAGAATAAACTTAAGGGTTTGAATTAGCCAAGTTTTTTTACTCTAGGGACTGATTTATACATAACTTGTAGCCACTGGCTCCAAAATATAAAATTGAATAAACTTTCAC
9469	chr15	50261573	50261824	-	0	train	GGTAGGAGAATCGCTTGAACCCGGGAGGCGGAGGTTGCGGTGAGCTGAGATAGCGCCGTTGCACTCCAGCCTGGGCAACAAGAGCGAAACTCCATCTCAAAAAAAAAAAAAAGTGACAAAAATATGCCTAGAGGCAGGGGTGGGAAAATATTCAAACTCAGAAAGTGATGGCAATTTTGACTTTTTTTTTTTTTTTTTTTTGAAGCAATAGAGACAAGGTCTTGCTATGTTGCCCAAGCTGGTCTCCAACT
973	chr1	173913014	173913265	-	0	train	AAACACTAGATCTTTAATCTGTCCTTGGCTTGGCTGCATGACAGTCTTTCTTCAAGTTGGATCACACTTTGGAAGCAGAGTTCATCAATAGGGAGGCATGAGTCCCTTCAAGATGGTATACGGTGCTTATTTGAAACTTGGACACTAAAGTCTGTGGGTCTTAGGAGGGTTCCTTCTATTCTAGTGGTCAATTTCCATGGAACTTCATCACCTTTGCTCAGGGCTCTGGGGTGAGTTAACCCAAGTCTTCA
15733	chrX	154312618	154312869	+	0	train	CGAGAGAAAGAGCAGATGCCATTATCAAATTAATTGAGAGCCAGATACAGACCAGCAGGAATCTTGACCCACAGCCCCCCATTGAGGACTCACCTGAAGTCAACATCACAGATGTAAGGATGACCTCTCCACCTGATTACAGAGTTGGTGACAAGGTAGGCAGAAAGGTGGATAATTGAATAAATCAGATACGTCAACTGCTTTGTTCTCTAATGTTGTTCGTATGTACACAGGATTCTTCATTTATTCTG
1524	chr2	10444021	10444272	-	0	train	TTCAGACTGAAATACAGTTGGTGCAGAGTCTGGGGGTGCCTCCAGAGAGGATTATCTATGCAAATCCTTGTAAACAAGTATCTCAAATTAAGTATGCTGCTAATAATGGAGTCCAGATGATGACTTTTGATAGTGAAGTTGAGTTGATGAAAGTTGCCAGAGCACATCCCAAAGCAAAGTGAGTTATTCCCCCATCTGAGGGCAAGATCGGGAGCATAAGATATGTGGATTCTTATCAAACAAACTTAAAT
13282	chrX	147918136	147918387	+	0	train	GGAAGGTATGAGTGTATCTGTGGGTGGGTGAGTGGTGGATAAGGGGAAGGACAGAGCCAAAAGCGACGGCTATTGGAAAAACTATGATGAGAAACAGGAAGATGGAACCTTGTTGGAATTAGTGAAAGACCTTAGAATTCACAGGAGGTATTTGTCCTTCACGCGAGTGTAGACCAAACGTAACCTATGAGTTTCTTTTATTCCACTTATTAAAGCAGCAACCAAAGGTATTATATACCTTCTGTATTCAC
7196	chr11	5226706	5226957	-	0	train	ATGAAGTTGGTGGTGAGGCCCTGGGCAGGTTGGTATCAAGGTTACAAGACAGGTTTAAGGAGACCAATAGAAACTGGGCATGTGGAGACAGAGAAGACTCTTGGGTTTCTGATAGGCACTGACTCTCTCTGCCTATTGGTCTATTTTCCCACCCTTAGGCTGCTGGTGGTCTACCCTTGGACCCAGAGGTTCTTTGAGTCCTTTGGGGATCTGTCCACTCCTGATGCTGTTATGGGCAACCCTAAGGTGAA
6348	chr10	47305507	47305758	+	0	train	GTCTGGCCCACCGCCCCTTCCTCCAGCTTCTGTTGGTGCATGCATCTAAGCCCAGCGTTCTCACTGATTGCCCTAGTGGCTGTTTGAGGGGCAGTGGGTGAAGGGACAGATCTCCAAACCGTCTTGGAATCTTTTTATATAGGGCAGAGCATCTAAGGGGCTGCTCTTGGCCTGAGAGTTGACCAGCAGTGGGGCTGGACAGGCCTTTGGATGAGGAAGTCCCTTCCCCCAGTACCATCCCCACTGCCCTC
12280	chr19	2249794	2250045	+	0	train	GCCGGGTCCTCCTAGGGAAGATCAGGGGCTGGCAGAGCCCCCACCCTGGGCAGGGAGGCTGTGGTCTTGTTCCTAGGACTGGGTTGCGGGTCCGTGGCCTGGAAGGTGGGCACCACACTCTGTCCTGTCCCCGAAGCCCAGCTCTTAGACTTGCCCCTGCCTCGGTGCCAGGGAGAGAGCTGCTGCCTTCTCCCCACCCCTGAAGACGACGCAGGGCTCGGGGCCAGTGGAACCCTTCTTCCCACAGCCCC
9198	chr15	50251219	50251470	-	0	train	CCTTTGGGAAACTAACTGGACCTCTCTGAGCCTTGCTTTCCTTATCTGTAAAATGAGAATAGTGGCATTGATTCATAGGGTATTGTGAAGCTTAAATGGGAACACGTGTGGATCACCTAGCAAAATAGCCGTGTGGCAAGTGCTCAATATATGGTAGGCATAGTATTTCTGAAAGCTCTTCTTTGAAGCTGCAGAGATTTGTTAGGACAGTGGTTAGGCATGTAGGTGCTAAAGCCAGATGGCATGTGTTG
13999	chrX	147941037	147941288	+	0	train	TATCAAGGAAATCTAAGCATTTCATGAAGTTCTTTGTTTAAATGTTTACCCTATTTTTCTCTTTACTGTTATTCAAAATACAAACTATTTGTGTCTTCTCATAAATGTTCAGTTTAGTTAGTGTGATGCGGTTATGCCTCCTTAAAATTTCAAACTGGAAGATAGGAACGAGGAGGCTACAAGGTGTAGTGGAGCAAGCACTGGATGGGCAATGCAGCAATTTACTTATAAGACTTCAGCAAATGAACTAG
874	chr1	173906360	173906611	-	0	train	CTTTCCCTGAGGACAGAATAGTGTGGCCACATGCCTAATTGTAATGGATGAAGAGCAAATGGAAGGTAAGAAAGGGAAGCTGGTGAGTGTGCATCAGTGTCTTAAAGTGTGCTCCAACTAGAGCACTAGACTACACTGGAGGAAACGAAAAGGTGGTCAAATAAATGCATATCCTCTCATGGGAGATGAACAGTACACACTGACATGCTGAGGTCTGACAAGTCCCACAGTAAAGAAGACGGTTGAATATC
4334	chr6	32040303	32040554	+	0	train	CTTGCCTCACCGGCACTCAGGCTCACTGGGTTGCTGAGGGAGCGGCTGGAGGCTGGGCAGCTGTGGGCTGCTGGGGCAGGACTCCACCCGATCATTCCCCAGATTCAGCAGCGACTGCAGGAGGAGCTAGACCACGAACTGGGCCCTGGTGCCTCCAGCTCCCGGGTCCCCTACAAGGACCGTGCACGGCTGCCCTTGCTCAATGCCACCATCGCCGAGGTGCTGCGCCTGCGGCCCGTTGTGCCCTTAGC
16371	chrX	156004885	156005136	+	0	train	ACATATGTGTGTGCCTGTGCCTTGCATTTGTGTGAGTGTGCACATGGGCATGCCTATGTGTATGAGTGTGTATGTGAGTGTGGTGAGTTGCTTCTGTGCACACACTTTTGTTTATGAGTGTGCATGCAAGTGTGATGAGTGTGAAAGTGTTCCTGTAGACATGTTTGCCTGTGTGTGCATATGTGTATTTGTGGGCAAACGCAGCTGTGTCTGTGAGTGTGAGTGTGCCTTCTGTGTGTGTGTGTGCACGT
11585	chr17	41584765	41585016	-	0	train	GATTGACAATGCCCGTCTGGCCGCGGATGACTTCCGCACCAAGTGAGTTTGAAATGGTGGGCCAGAACATCCAGTGTCCCCAGAGTAGGGCATTTTTGGAGCAGTGTTTCCCAAATAGAACTAGCCAGTACCAGGATAGGTGCATGAAAACTCCCTGGGGTGCTTATAAAAGAATAAGACTCTTGGGCCCCACCCTTGGAGTTTTGATTCAGCTATTTATAGCAGGTTACCTGGGTGATTCTGGTCCACAG
9407	chr15	50258625	50258876	-	0	train	CCCTTCCTTTTATTGCTGGGCAGGGGCGGCATGAGGGGTAAGTACTGGGGACTAATGTGAATACCACCCAGAATCCCACAACTGGGTTACTGTAGTGATTTTCCTTTTGCATATTTCCTTCTGGACTTTGCATATATGAAAAGGCTCCACTGATGATCTAGCATAGGACATGGGCCACATTAGTGACAAGAACACCAGTGAACAAGTCTGATTTTCATGGAGTTATGTCACTTATTTCTGCTTAAAAGTGT
11717	chr18	31593844	31594095	+	0	train	ATCAGGAGACCCTTTGCATCCAGCAGAAGAGGAACTGCTAAGTATTTACATCTCCACAGAGAAGAATTTCTGTTGGGTTTTAATTGAACCCCAAGAACCACATGATTCTTCAACCATTATTGGGAAGATCATTTTCTTAGGTCTGGTTTTAACTGGCTTTTTATTTGGGAATTCATTTATGTTTATATAAAATGCCAAGCATAACATGAAAAGTGGTTACAGGACTATTCTAAGGGAGAGACAGAATGGAC
11030	chr16	2129756	2130007	-	0	train	GTCTGCAGCAGTTGGGAATGTGGGGCACCCGAGCTCCCACTGCAGAGGCGACTGTGGAGACAGAGAGCACCTGCAGGTCATCCATGCAGTATCGGCTTGCATCCAGATCATACAGGGAACACTATGATTCAACAACAGACAGGGACCCCGTTTAAACATGGACAAGGGGTCACTCACGCCTGGAATCCCAGCAGTTTGGGAGGCCAGGGTGGGTGGATCGCTTGAGCCCAGGAGTTTGACACCAGCCTGGG
1353	chr1	225836674	225836925	+	0	train	AAGTAAGCCAGGGACAGGACAAATACCGCATGATTCCACCTATAGAAGGAATCTGAAATAGTCAAACTCAGCAGAAGCAGAGAGGAGAAGGGTGGTTGTCAGGGGCTGGGGAATAGGGGAAACGGCATTGCTGATCGTTGCTGTGTAAAGTTTGAGTCCTGCCAGCCGAATACATCCTAGAGATGGGCTGTACAGCAGAGCGCCGTCAGTGTTGGACACTTACAATCTGTTAAAAGGGTAGGACCCATATT
15442	chrX	154303309	154303560	+	0	train	TTAAGCAACCCTCCCACCTCAGCCTCCCAAGTAGCTTGAACTGCAGGTGCATGCCACCCTGCCCAGCTAATTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTGGAAAGACGAGGTTTCACCATGTTGCCCATGCTGGTCTCAAACTCCTGGGCTCAAGCAATCCTCCTGCATCATCCTCCAGAAGAGCCGGGATTACAGGTGTGAGCCCCCACGCCTGGCCTCTCCCTCCTTTTTATCATTCCCTCTTCCC
15358	chrX	154300395	154300646	+	0	train	TATCCATGAGCATGAGATATCTTTGTTTGTGTCATCTGTGATGTCTTTCAGTAGTGTTTTGTAGTTTTTCTTGTAGCTGTCTTTCACCTCCTTGGTTAAGTATATTCCTAGGGTGGGTTTTATTTGTTGTTTTTGCTGTTTACAGCTGTTGTAAAAGGGATTGAGTTCTTGATTCTCAAGCTTGGTCGTTGTTGGTGTAGTGCTACTGATTTGTGTGCATTTATTTTGTAACCTGAGACTACTGAATTTGT
11122	chr16	2132450	2132701	-	0	train	TCCTTCATGCCACCGAAGGCCTGGTGTGTTCACATTTTTGGTTTAATAGTTTGAATTAAGAGCCAAATAAGGTCCACACACTGCAATTAGTTGATGTCTTTTTTTTTTTCTTTTTTTTTTTTTTTTTGAGACGGAGTCTTGCTCTTGTCTCCAGGCCGCAGTGCAGTGGCATGATCTCAGCTCACCGCAACCTCCGACTCCCTGGTTCAAGCGATTCTCCTGCCTCAGCCTCCCGAGTACCTGGTAGCTGG
13140	chrX	147913991	147914242	+	0	train	GTAGCACATTATGGCCTTGAAGTACTTATTGTTCTCTTCCAGCAACTTATGATTTGCTCCAGTGATTTTGCTTGCACACTGACTGGAATATAAGAAATGCCTTCTATTTTTGCTATTAATTCCCTCCTTTTTTGTTTTGTTTTGTAACGAAGTTGTTTAACTTGAAGGTGAATGAAGAATAGGTTGGTTGCCCCTTAGTTCCCTGAGGAGAAATGTTAATACTTGAACAAGTGTGTGTCAGACAAATTGCT
2831	chr4	71773111	71773362	-	0	train	CACACTGGCTTTCAAAAGTTCAGAATTAAACAAGACTTACCAGAATGTTTCACAATATAGATAAAAAGTGAAACATGCTATTGCAGAAAGCAAAAATTCTGTTAGTGAAATTTGTAAACACTTAGTACTGTGGGGGGTCACTTCATGTGAAAGTGATTTCATCAGAGTCACTAAGATTATGCCAATATTTACTAGGGCCATATCAAAGTCCAAATCACCTTCTTTATAACCGAATGGTACATTTGAAAACA
5464	chr7	99971351	99971602	-	0	train	ACATCCCACCCACCCCACAGCACAGGTGCTCCCCGCTTCCCCATCAATTGCCCCATCCTCATCCCAGGCCTCAGGTCACACAGGAAGTGATGGCAGAGTCACTTCCTATCCAGGCACCTATGACCTCTCACCTCCACACCCCACCCATCGGAGGCTGATACCCCCGTGAGAAGGCATCAGACTCACCCCTGTCCAGGGAGGTTGCCTGGAGAGTGAGCCACTCTCAAAGTCACTCAGACCTGGGCTCACCT
12993	chrX	106034349	106034600	-	0	train	TACTCCCAGTGCTTCAGTCCAACATTGGCTGTACCTGAGTGCATTCCTTTCAGAATAACTTCTGTGAGTTTGGTGCCATGTATTGACTTACATAGTAATGGTTATCAATACTCAGGGAAGAAGCAGAGTCCAATATAATACCATCATGAGAGAGAGAAGGAGAGAATCATAAGCTTGATATGGTGATTGCCATGTGTTCCCTTCCTCTTTTCCCACAGATGGGTTGACTTGTTTGTTCCAAAGTTTTCCAT
13571	chrX	147928448	147928699	+	0	train	AGGTTGTACATTCCCGTGTGGATTTCTATTTTGAAGTAATATCTAATTTTGAGTAATTTAATTAAAATGTTTTCACTATGTGTTCAGTATGTTTCTGTTGGTCATAAATTTTTTCACATAGATTATTTATTTTAAAATAACTGAATAGGGAGAACTTCTTATTCTTACTTTAAAAATTGTGATTAGAAGTGACTTTTATTTATTTCTCAGTTTTATGTGATAGAATATGCAGCATGTGATGCAACTTACAA
2313	chr4	71750124	71750375	-	0	train	GCCTCCATTTAAATTTCTATAGTCAATTTATCTACTCCTAAGAGTAGAAATGGAAATGGCCTATTTCATTGCCAGGTAGCTGCATTATGAGAACATTCTTCTATAAATTAAGCTGTGATATTCCCTTTGTAATTTCTCCTGCAATCACCCTGGCTTTCTCTATCAACTAGAAAATGTCTATTCTCTGTTCGCAATAGCAGCTCTTTTATACTTGAAGTGACTTTTTAAAAATTAAGCTGTGTAAAGTATTT
11527	chr17	34362486	34362737	-	0	train	TCAGAAGGCAAGACCGGGCTCTAACAATTGGTCACTCTTGGGCAAGTCACTTTAGCTCTCAAACTCTACTTTCTCTATCAGTGAAATGGAGTTGATGATGTCTGCCCTCCAAGACTGTTTGGAGAATACCAACCTAGTAAGAGGCATGAAAGGGGGTGCAAACAGAAAAACTAGGAGGAAGAAGCTGGGATTGGAATGCAGGTCTCTTGCGGGATGTGGTGTGGGAGGAGAATGCACAAATGGACAGAGTG
15294	chrX	154298244	154298495	+	0	train	CTCTTTCACCCAGGCTGGAGTGCAGTGGTGCAATCATAGTTCACTGCAGCCTCGAACTCTCGTGGGCTCAAGCGACTCTCCTTCCTCTCAGCCTCTTGAGTAGCTGGAACTACAGGTGCACACCACCATGCCCTGCCCATCTATTAATAGCTAGTTTTGTTGACCTTTTATAAAGAACCAGCTTTGGCTAGGGGAGGTGGCTCACGCCTGTAACCTCAGCACTTTAGGAGGCCGAGGCAGCTGGATCACTT
8224	chr12	49297413	49297664	+	0	train	GGGGGCTACCAGGCGGGCGCTGCGCGGCTCGAGGAGGAGCTGCGACAGCTAAAAGAGGAGATGGCGCGGCACCTGAGGGAGTACCAGGAGCTCCTCAACGTCAAGATGGCCCTGGACATCGAGATCGCCACCTACCGCAAGCTGCTGGAGGGCGAGGAGAGCCGGTGAGGGTGGAGCTGCTGGGGCGGGGCAGGGCGGGGTCGGGACTGGGCCGGGCAGGGCGGGGCCTGGGCAGGGGCGCTGACAACTTG
9662	chr15	90889599	90889850	+	0	train	AGCTGAACGAGCTGACTGTGGAGAGCGTGCAGCACACGTGGGTGGTGGCTTTGCACCTGGGCTGCGGCGGGGCTCCCAGCAGACCACGAGTGTTTATGTAGGCAGGGCTAGGTCGTGGAGACTGTCCACACAGAGCTGTCACCAGGTGGCCGGGCTTGCTTGGCTCTACAGGGATGCACTGGACCTGGGTTGAGGGGGCAGGAGGGCTCGGTTCTAATGCTGCCCTTCTCTTGGGTGCAGGCTGACCTCAG
7943	chr11	119090721	119090972	+	0	train	AATCTTCCTGCCTCAGCCTCCTAAGTAGTTGGGATTACAGGTGCCACCACACCTGGCTAATTTTTGTATTTTTAGTAGAGACTGGGTTTCACCATGTTAGTCAGGCTGGTCTCAAACTCCTGACCTCAGGTGATCCACCAGCCTCGGCCTCCCAAAGGGCTGGGATTACAGGCTTGAGCCCCGCACCCGGTCAGTACTTCCATTTTTATATGCTACTATATTGTCTTGACTTTTACAATGAATATGTAGTA
14505	chrX	149492533	149492784	-	0	train	TGCTTTGTTTTGTTCATTCAGCAAGTGTAGTGAGCACCTCTTATGTACCCAGCAGTGGCCTAGTACCATTTTGTCTAAAGCCTCTGGAATCTGTGTCTGTGTCCTGTTGCAACAGGACTCTTAATAACTTGAAGCCCAGATCATGTCAATTTCTCGCCTTGAAAGACCTCAGTGACTCCATTGGCCTAGAACTTGGAGTCTCCACTCCCTGGATAGACCCACCAGGTCCATATTATCTGGCCCGGACCTAA
7560	chr11	64262604	64262855	+	0	train	GGACTCTCAGCATCCGCCTCACCCTCCTTGGCCCACCCCCAGGTGATCTCAGGGCAGTTCCTGTCCGACAGGAAGGTGGGCATCTACGTGGAGGTGGACATGTTTGGCCTCCCTGTTGATACGCGGCGCAAGTACCGCACCCGGACCTCTCAGGGGAACTCGTTCAACCCCGTGTGGGACGAAGAGCCCTTCGACTTCCCCAAGGTGAGCCTGGCCCCTGCACCCGCCCAGGCACAGGCAGATCCAGCCCA
15204	chrX	153906156	153906407	+	0	train	TACCCTGGGTATCGCCGCCTGCCAGGTGCTCATCTTCCGGGAGATTCATGCCAGTCTGGTGCCAGGGCCATCAGAGAGGCCTGGGGGGCGCCGCAGGGGACGCCGGACAGGCAGCCCCGGTGAGGGAGCCCACGTGTCAGCAGCTGTGGCCAAGACTGTGAGGATGACGCTAGTGATTGTGGTCGTCTATGTGCTGTGCTGGGCACCCTTCTTCCTGGTGCAGCTGTGGGCCGCGTGGGACCCGGAGGCAC
6666	chr10	47353258	47353509	+	0	train	CTGCTTTCCTGGGCTCTAAAACTGGCTGCTCCTCCTGACACTGAGTAGGACCTCCAACTCTTACAGATCCCTTCCCCTGAAGTATTTGAAGAGCTGATCAAGTTTTCCTTCCACACTAACGTGCTTGAGGACAACATTGGCTACTTGAGGTTTGACATGTTTGGGGACGGTGAGCTGCTCACCCAGGTCTCCAGGCTGCTGGTGGAGCACATCTGGAAGAAGATCATGCACACGGATGCCATGATCATCGA
11255	chr16	28933850	28934101	+	0	train	CTGTCACCCAGGCTGGAGTGCAGTGGCACGATCTCAGCTCACTGCAACCTCCACCTTCCAGGTTCAAGTGATTCTCCTGCCTCAGCCTCCCGAGTAGCTGGGATTAAAGCCTGGCTAATTTTTTTTGTATTTTTAGTAGAGATGGGGTTTCATTATGTTGGCCAGGCTGGTCTCAAACTCCTGACCTCGTGATCCACCCGCCTCGGCCTCCCAAAGTGCTGGGATTACAGACATGAGCCACAGGGCCGGGC
9016	chr15	50244790	50245041	-	0	train	CATGTGTTGTAGGAGGGATAAAAGCACAGTGGGAGGAGATCTACCTACTGTCACTTGTACCTTGGGAAAGTCACTTTCCCTTGGTTCCTTCTAAAAAATGGAAAGATTGAGCTGGGCACAGGGGCACGTGTCTGTAGTCCCAGCTACTTGGGAGGCTGAGGTGGGAAGCTCAATTGCTCCCACCATAGCCCAGGTGGGAGTTCTGGGCTATGGTGTGCTATGCCAATTGAGTGTCCGCACTAAGTTCGGCA
8612	chr14	74893112	74893363	+	0	train	AGACTTTGTATCCAAAGGCCTGCTTTGCATCTCAGCTTTCTGTTACTAGCTGTGTGTCTTTAGGCTAGTCACTTAATTTATCTGAACAGATTGTTCATCTGTGAAGTTGGAGTGATACTAATATACCTCAGAATATCTTGAAGATTAAGTAACAGTACATATGAAAGCACTGGCACAAACTCAGCAGATGTTTCTTTCCTTGTCTGATGCAGCTTTATCCTCTTTTCATTTTCAGTGTCTGCAGTAAAACC
11234	chr16	28933280	28933531	+	0	train	CCCCCTGACTCTGTGTCCAGGGGCCCCCTCTCCTGGACCCATGTGCACCCCAAGGGGCCTAAGTCATTGCTGAGCCTAGAGCTGAAGGACGATCGCCCGGCCAGAGATATGTGGGTAATGGAGACGGGTCTGTTGTTGCCCCGGGCCACAGCTCAAGACGCTGGAAAGTATTATTGTCACCGTGGCAACCTGACCATGTCATTCCACCTGGAGATCACTGCTCGGCCAGGTAGAGTTTCTCTCAACTGGGA
12921	chr22	20786616	20786867	+	0	train	GGCTGTCCTGCACCAGCCCCCACGACCCTCAGACCACAGGCACTGCCAAGAGGGAACATGAACCTAGCCGGCCTCTAAGTGCAACGGCTGCCCCTGACAGGTGGTGACAGATATTTTCAAGAGTGACTCTGACCAGCTGTGATTTCCACCTTACATGTTGTCTTTGGATCCTTTCCCTGAATGATATGAGATTGTGCTGGGAACTCTAGCCCTCTGTGTGCTGACCTCCAGAATCTGACAACTTTCCTTTC
15521	chrX	154305759	154306010	+	0	train	ACTGAGCCTGGGGGAACTTGAGTGTAACCGGGAGGGACATCCCACCCCTGTCAGTTAGCTTCCTCTTCTATGTTTCTGCACATCTGACCTGAAACTGATGGGCACTTCTCATGAGATTGGGCAAAATACCAGCCACTTCATGAAGTATCATGGGTGATAGGAAAGAAAAAGAACCTTAAAAACAAGGTCAGTTTGAGAATGTGTGGCTGGTTTGATGCCATCTCAAAGTACACTGTGTCCCGAAGCAAATT
3553	chr4	154606622	154606873	-	0	train	TTTGGCGATGATCCTAGTGACAAGTTTTTCACATCCCATAATGGCATGCAGTTCAGTACCTGGGACAATGACAATGATAAGTTTGAAGGCAACTGTGCTGAACAGGATGGATCTGGTTGGTGGATGAACAAGTGTCACGCTGGCCATCTCAATGGAGTTTATTACCAAGGTATGTTTTCCTTTCTTAGATTCCAAGTTAATGTATAGTGTATACTATTTTCATAAAAAATAATAAATAGATATGAAGAAAT
6937	chr11	5225994	5226245	-	0	train	TTACTATTTGGAATATATGTGTGCTTATTTGCATATTCATAATCTCCCTACTTTATTTTCTTTTATTTTTAATTGATACATAATCATTATACATATTTATGGGTTAAAGTGTAATGTTTTAATATGTGTACACATATTGACCAAATCAGGGTAATTTTGCATTTGTAATTTTAAAAAATGCTTTCTTCTTTTAATATACTTTTTTGTTTATCTTATTTCTAATACTTTCCCTAATCTCTTTCTTTCAGGGC
16402	chrX	156006228	156006479	+	0	train	GGCAAGAGTGTGCATGTTAGTGTATGTGTGCAGATGTGTGACTGTGTGCATGTGTGAGTGTGTGGTGGGTGGGCCATCAAGGGCCGCCCTTGTCTGGTTCCTCCCCTCCCCTCTCCACTGCCTGGTCCTGGACGGGGTGGGCTTTTCGAGTCTCCACCCTGGTCCAGAAGAGGGTTCTCAAGTTGCCAGGGGAGAGCAGGGAAGGGGGGTCTGAGGCAGAGGCTGAAGATAAGGGCAGCTTGGTCCCGACC
4696	chr6	32939884	32940135	-	0	train	AGCTGAATATAGGAAAAATCAACTTTTTTTCTTCTATATGCTCACACTCAACACTTCTTTGACCAACTGTGTGAGGTTTTTTTTTTTTTTTTACTCATACCAACCAATTCTCCTATATTAGCTGGATATCCTATAATTCAATTCCATTGTGACATTAACTAGAGTTAACATAGACACCAAAGGTTAAAGACTCAGTCCCATAAGACTGCCTCCATTTCAGACACCAATCACAAGTAGTAGGTTCCCAAATT
1817	chr2	112833569	112833820	-	0	train	GGGTTTCGCCATGTTGGCCAGGCTGGTTTTGAAGTCCTGACCTAAATGATTCATCCACCTCGGCTTCCCAAAGTGCTGGGATTACAGGCATGAGCCACCACGCCTGGCCCAGAGAGGGATGATCTTTAGAAGCTCGGGATTCTTTCAAGCCCTTTCCTCCTCTCTGAGCTTTCTACTCTCTGATGTCAAAGCATGGTTCCTGGCAGGACCACCTCACCAGGCTCCCTCCCTCGCTCTCTCCGCAGTGCTCC
9995	chr16	2093757	2094008	-	0	train	GTGGAGGGTCTGCGGAAGCGCCTGCTGCCGGCCTGGTGTGCCTCCCTGGCCCACGGGCTCAGCCTGCTCCTGGTGGCTGTGGCTGTGGCTGTCTCAGGGTGGGTGGGTGCGAGCTTCCCCCCGGGCGTGAGTGTTGCGTGGCTCCTGTCCAGCAGCGCCAGCTTCCTGGCCTCATTCCTCGGCTGGGAGCCACTGAAGGTGAGGGGGCTGCCAGGGGTAGGCTACAGGCCTCCATCACGGGGGACCCCTCT
8967	chr15	50242704	50242955	-	0	train	CGTCTCTTCCTCATCCCGGCCACTATCCAGGACAAGTTAATCATCCGTTTCACTGTGACATCCCAGTTTACCACTAGGGATGACATCCTGAGAGACTGGAATCTCATTCGAGATGCTGCCACTCTCATCCTGAGTCAGCACTGTACTTCCCAACCCAGCCCTCGGGTTGGGAACCTCATCTCCCAAATCAGGGGTGCCAGAGCCTGGGCCTGTGGAACGTCCCTTCAGTCTGTCAGTGGGGCAGGAGATGA
4596	chr6	32936193	32936444	-	0	train	AGGGCCAGAGAACAGGATCTCAGATCAGCTGCTGTAACCAGGTTTCCCCTTGTGGGAAGTGTTGTTTCTTGCTGGGCAGTTGGGAAGGGAATGGAGAACAGAGAAGAGAGTGGAAATCACATGCTCACTTGAACTTTCCTGGGGAACGTCTCCTCACAGCGTGCACAAGAGCCTCCCTTTAGAAATGGAGTGTTCATTTTATCATGGGAAAAGAATCTGAGTGGGACATGATTCAGAACAGGACCGGCCCA
6295	chr10	47303238	47303489	+	0	train	CCTGCTCCCCAGTGGTGGGTTTCTGCTGCCTCAGTGGGGCAGCACCTCTTCCTGAACAACAAATGCCTCCATCCCCACAGCCTGCGATACGTAACTATACCTCCCCCGCCACCGGGCAGTGCAGCTCAGCTGCCAAGTGCTGGCTGTGAGCTGGCCTCGTGGGGTCCAAATCCCAATTCTGCGATCTGGGCAAGGGACTGAACTTGTCCATGACTCAGTTTTTTTCACTTCTAAAATGGAGCTAATAGTGC
6606	chr10	47351153	47351404	+	0	train	TGCATCCATGCCCACCCAGATGGCCATGAGTGCCACCACAGGCAAGGCCTGGGACCTGGCTGGTGTGGAGCCCGACATCACTGTGCCCATGAGCGAAGCCCTTTCCATAGCCCAGGACATAGTGGCTCTGCGTGCCAAGGTGCCCACGGTGCTGCAGACGGCCGGGAAGCTGGTGGCTGATAACTATGCCTCTGCCGAGCTGGGGGCCAAGATGGCCACCAAACTGAGCGGTCTGCAGAGCCGCTACTCCA
4670	chr6	32939191	32939442	-	0	train	TGTCCCCACAGAATTGGGGTACTTCACCTCCTGGCATGTGGATGTGTTTACCAACTGAGAAGTTCTCTGAACTCCATAGTTCCGGGATTTTTATGGAGGCTTCATCATGTAGGCATGACTGATTATTAACTCAATCTCCAGCCCCTTCCCCTTCAGGGAGTATGGGGGATGGGACTAAAAGTTCCAGACTTCTAATCATGACTTGGTCTTTCTGGTGACCAGCCCCTCCTGCAGGAGCCCACCAAGAGTAC
3888	chr5	132679774	132680025	+	0	train	ACCATGAGAAGGACACTCGCTGCCTGGGTGCGACTGCACAGCAGTTCCACAGGCACAAGCAGCTGATCCGATTCCTGAAACGGCTCGACAGGAACCTCTGGGGCCTGGCGGGCTTGGTAAGCTGCACTGTATTCCTGGCAAGCCGGCCGCGTGGCTCCTGGTGGACAGCAGCCTCACTTCTAAACACTCCTTAGGAGCTGCAGCACCCTTGGTCAACCCATTCATTCATTCACTCATTCAATAAGTATTTG
790	chr1	159307044	159307295	+	0	train	CATTTGATAACTTAAAAAATATATTGATGCTCATGTCTCATTTCTTGAGATTCTGATTTAATTGGTTTGGGGTGCAGCCTGGGTATACGTATTTTTCATAGGTCTTTCACATAATGGTAATGGGTAGCCAATATTGAGAATCACTTGTCTAGGTGATCTTTAAATGATTTCTGGATGTAATATTCTGAGGCTCTATAATTTGAGACTAATCACAAAAATCGGTACAGTTTATAAACAGACTAACAGAACCA
9103	chr15	50247536	50247787	-	0	train	AACAAACCTCCCTGCATCAACAGGCATTTATTGAAGGCCTATTTTGCTTAATTTATTTCTATTCAATTCAACAATTACAAGGTATTGTGCTAAGAAAGGGACACAGAGGTAAATGAGACAGGTGCAGCCTCTCTCCCATGTTCTTCATGGAAGAAGCGGGACTTAACTAAGAACTCTGGCCACTGGCATTGTTCTCCCACTTCTACAAGAGTCCACATGCAAGACAAAGCCATTGATAAAAGTAGTGGGGG
5105	chr7	44064330	44064581	-	0	train	CCAAAGGTGGCATCTGCCAAGGGACACCCAGCTAGGAAACGGAAGGGCTGGGCTTAGAGCATCTGGCTCCAAATCCCAACTTACTGTGGGGCCCTGGACAAGCCACCTCCATCTCTGGGCCTCTCCCTTTTCCGGGGTGGTGGGGAGCTCCCCCTGGTACTGAATTCCTCTTGATGTAGGCTTGGACCCCTCGCAGGGCCCTCCCCCATCAGGTCCTCAGAATCCCTGCATGAGCTTCACCACCTATCTCC
416	chr1	109659924	109660175	+	0	train	GCAGTGAGAGGCCTGCAGGCAGAGCAGCCTGTGAGGTGTGTGGCACCACCTGGGTACCAGGCCTGGGGCCTGCCCCTCACTCATGGGGAACCATCCCTCACCCGTGCTGAATTTGTTTGAGAGCAGCAAATCCTACTTTTAGTACAGATGTGAGAATTTGAGGCATTAGTCCAACAAGTTTTTCAGCCTAGAATTTGTTTTCCTTTCCCACTACCCATCAAGGGATCTGGTTACTCAGCTAGTTCCCATCA
10911	chr16	2125662	2125913	-	0	train	AGGAGGGCCCCTTGAGCCTCAGTGTGCCCATCAGGAGCGTAAGGTCAGTGCAGCACCTGCCCACACAGGCTGTGAAGGGTGGGAGTGGAGAGGGATGCAAGGGGGTCACAACGCCTGGCTCCATGTCAGCTGCGTGCAGGGGCACCAGGAGCCGGCCCTCATTCTCCCCTTGAACTGGAAGGGTGGCCCCGACCCCAGCGGCAGGTAGCATACGTATGAAGCGCTCTCCTTCCTACACCCCACAGGTGGGC
2708	chr4	71768205	71768456	-	0	train	ATATTGTCTTATTTTCCCCTCAGGTCACTAGTCCTGTACAGTAGAAAATTTCCCAGTGGCACGTTTGAACAGGTCAGCCAACTTGTGAAGGAAGTTGTCTCCTTGACCGAAGCCTGCTGTGCGGAAGGGGCTGACCCTGACTGCTATGACACCAGGGTAGGTTTCTGTGGCTGGCCGTCTCTGTGGCAGCCCAGAGAAGGAAGCCAAAATAGGCCTTTATACCACGTGTTGAAAAAATTTTAGACAGCAAC
8996	chr15	50244215	50244466	-	0	train	GCTGGGCATGGTGGTGCATGTCTGTAGTCCTAGCTACTCAGGAGGTGAAGGTGGGAGGATTACTTGAGCTCAGGAGTTCAAGGCTGCAGTGAGCTTTGATTGTGCAACATGCCTCTGTACTCTAGCCTGTGTGACAAAGCAAGACCCTATCTCTCAAAAAAAAAAAAAAAAAAAAAAAAAAGAGGTTCAGCTAACTCCCTTGCCCTCTCTTGAAATAGAGCAGTAAGTATGAGAACTAACTGATAGAATTA
14213	chrX	149482808	149483059	-	0	train	TGGAAGAGGATCCGTACCTCCCTGGTAATCCCCGTGAACTGATTGCCTATAGCCAGTATCCCCGGCCTTCAGACATCCCTCAGTGGAATTCTGACAAGCCGAGTTTAAAAGATATAAAGATCATGGGCTATTCCATACGCACCATAGACTATAGGTATACTGTGTGGGTTGGCTTCAATCCTGATGAATTTCTAGCTAACTTTTCTGACATCCATGCAGGGGAACTGTATTTTGTGGATTCTGACCCATTG
14032	chrX	147942230	147942481	+	0	train	ATGATAACCTCTGCTTCACTGAGACGCCTACTACTACTATAGTTATGAATCTGATTCCCCGTAGCCAGTAAATGAGATGTTTATGAGTGGATTGGAAGGACAGTGGTTAGAAATGAGGTTTCATCTGGTTTTAACAGAACCCAACATACTGGAAGTGCTTTTTGTAAGGGTCTGTCATTGGACCCCAGGTGTATATTTCTATCAACTAAATGTATACTGTTTCTCGTACTTGTTGTAGCAGAGTTGTAGTC
10527	chr16	2113314	2113565	-	0	train	ACTCTTTGTAGCTGGTACTAGGTTTGCTTTACAGATGGGGAAACTGAGGCACAGAGAGGTTGAGGCATTAGTAGTACTACATGGCTGGCTGGAGAGCCGGACAGTGAGTGTCCCAGCCCGGGCTTGGCTCCCATGGCATGCAGAGCCCCGGGCACCTCCTCTCCTCTGTGCCCCGCGTGGGACTCTCCAGCCCGACGGGAGGTGTGTCCAGGAGGCGACAGGCTAAGGGCAGAGTCCTCCACAGAGCCCAG
12461	chr19	44928918	44929169	+	0	train	TGCGATAGCTCATGCCTGTAATCCCAGCACTTTGGGAGGCTGAGTCGGGCAGATTACCTGAGATCGGGGGTTCGAGGCCAAGTTGGGCAGATCACCTGAGGTTGGGAGTTTGAGACCAGCCTGACCAACATGGAGAAACTCCGTCTCTATTAAAAATACAAAATTAGTCAGGCATGGTGGTGCATGCCTCTATTCCCAGCTACTTGGGAGGCTGAGGCAGGAGAATCACTTGAACCTGGGAGGCGGAGGTT
4878	chr6	33070496	33070747	-	0	train	AAACCTGCACATGTACTCCTGAACATAAAATAAATGTTGAAATATTTTTAAAAAGGAAACAAAAGTTTGGAACAAATGCCAAAATAACTGTACTGTACTTTTGAATTTATATGCCCCAAATGAAAAATATTATCAACAAAGCTATACATTCTACAGTTTCATGTTCATAAACTAAGACAGAAACTTTAAAACTGTCAAGAGCCCTAAAATTTGAAGGATATTTTCTTCTTCCTCTCAATTTTGTATTTTTT
5847	chr7	150857904	150858155	+	0	train	ATGCATGCCACTGGCTACGTCCACGCCACCTTCTACACCCCCGAGGGGCTGCGCCACGGCACTCGCCTGCACACCCACCTGATTGGCAACATACACACTCACTTGGTGCACTACCGCGTAGACCTGGATGTGGCAGGTAGGACTCAAAGCGAGACTCTCCCGTTCAAACATCTGCATCCAGCCAATAACTTAAACTCCCAGGAGACGGCACTATAACTCCCTGGTGGTGGAAAGTTAGGAGCATTTGCCAA
6742	chr11	4386055	4386306	-	0	train	GTTTGGGATAGGAGTAGGAGACAGGAGTCTCAAACTCTCTTTCCCCCAGGAGTGAGTCCTGGAACCTGAAGGACCTGGATATTACCTCTCCAGAACTCAGGAGTGTGTGCCATGTGCCAGGGCTGAAGAAGATGCTGAGGACATGTGCAGGTGAGGCAAGTTCTAGTTTTGCGGGGGATAATGGGGTGCAGAGTAGATCCCAGGGTCAGGGAGCCTGGATGGCAACTTGGAGGAGAGATGGCAGGTCAGAG
4698	chr6	32939957	32940208	-	0	train	CTTGGTCAGTAGATGTTCCCTGAGCAGGAAATCTGTGCCAGACTAGCTGGATGTCACCAAGGCTTAGGTTCTGAGCTGAATATAGGAAAAATCAACTTTTTTTCTTCTATATGCTCACACTCAACACTTCTTTGACCAACTGTGTGAGGTTTTTTTTTTTTTTTTACTCATACCAACCAATTCTCCTATATTAGCTGGATATCCTATAATTCAATTCCATTGTGACATTAACTAGAGTTAACATAGACACC
3191	chr4	87981096	87981347	+	0	train	TTTCAAATAAAAAAACCAGTCTCATAATTATGTATCTGTATCTATTACATCATTGAATTTAGTAAATAATGTTTAATATGTATAAGGAAAAACAATGTTATTGACATGAAGATTATACTCACATATTTGGCTTGAAAATATCTATAAAAATAATTTCTGTTGCAAAGTAAGAAATGTTCTTCAGAATGTTATTAATCCCTGTGTTAAAAGAGAAATTGGAAGATGCTCACTTTAGCTCCTAAAAGCCATGG
9264	chr15	50253599	50253850	-	0	train	ATACATGTACTAGATTGTTATATAAAATGTGTTTCTTACGAGCCATGGTAAAACACAGTCTGAAATATACACTGTTCTGGTGGGATGGATGGGAGTAGATCACGTCTGTGGTGAAATCTTTCTCAGGCCAAATGTGTCACACGAGCCCACCCTCCTGTGTTAACTTGCCTTAATTTTCCCCTAGGTCTGTGCAACACTAGGGACCACTGGGGTCTGTGCATTTGACTGCCTGTCAGAGCTGGGCCCCATCT
7339	chr11	6393405	6393656	+	0	train	AACACGGTGGTGCTGGGGGACAAGCAGGCTCCTGTTGAGCTGGAGCACCTCTGGGCACAGAAGTTTTATTTTCCTGGCATTCCCAACAAGTGTTCCCTGGGGATTCAGCTCATGGTCACTGTTGAAAGCCTTCATTCAGTCCCCCTTTCTCTAGCCAGGGCTGCCTGGACCCCTGGATGCCCTGATTACCATCCTTAATTCTCCCTACTAGGTGCATATAATTGGCCACATTCCCCCAGGGCACTGTCTGA
5875	chr7	150860619	150860870	+	0	train	ACATTGAAAATGAGGTACTGCCCTGTCCCCAGCCCTGCCCGGTGCTGGCCCTGCCTCCTTCCAGCTCAGCCCAGGACCATCCTCATCACCATCAGGGAGTCCCAACCACCCTTTGCCCAGATCTGTCCCCAGTCGCAGGAGCTGTGCCTTGCTGTGTGGACGGCAAGTTCAGAGGTCACAACAGAGCTGCTCATCTCTTTAAAAAGGGGCTGGAGAGGAATTCAGCAAGTTTCCAGGCAGAACTGAAAATG
9240	chr15	50252998	50253249	-	0	train	ATTCTACCTTCAGAAAAGGAAAAGGGTAGGGTAGGGGAAATCATGGGAGAGATTAAAGGCAGAGAGGAAACCCAGATGGGACTGTTTAATCCCCAAGCAAGAAACCACTAGGAAAGAGAGATACCTTCCCTCTAACTTTTGTATTCGGTGCACTGACATTGCTCTCTGTGCTGGTGGGAAATTGATTTCTGTGCCAGGATGTTCTAGGCTGAGGCCAAGAGGGTGCTGGCCCTTTACCCAGTGGAAGAAAA
567	chr1	119511805	119512056	+	0	train	AGAGTTCAAGACTGCTAACTTTAGTTTTTTAGATGATAAAACTAGGACTGAGAGAGGGCAAGTAACTTGTCCAAGGTCCCCCAGGTAAGTAAGCAGGTAGGAGAGTTAGACTTTAAACTCATCCCTGTGTGACTCCAAAGGCTCTTTCTACTGTGACTCCAAAAGCTCTTTCTACTGTGGTTCCAATCAAAAGTCAACTAATTTCTGACTTCAGACTCTTGATACCCAGACACCCCTTGCCTCCCAGGCCA
5666	chr7	99975052	99975303	-	0	train	GACATTTACAAATTGTCAGAAAAGGTGTTATATGTTTGTTATATAACAATCACTTTGGAATGTTAATCTGATTCTGTGCCAAAATCTGAATTACTCAGGGTTCTCCAGAGAAACAGAACTAATAGGTGGTACACATATACATATATATGTACGTACACATACATACATACACTGTATACACATGGATACACACACACATAGGAAGAGATTTACATATATGTATACAAAAGAGAGAGAGAGTAGAGATTTAT
11565	chr17	41583626	41583877	-	0	train	GGAGATGGACGCTGCACCTGGCGTGGACCTGAGCCGCATTCTGAACGAGATGCGTGACCAGTATGAGAAGATGGCAGAGAAGAACCGCAAGGATGCCGAGGAATGGTTCTTCACCAAGGTGGGTGTCATTTGAGGTGGAAGGAACCCAGACCACCTGCCTTCTGGGGCCTTCTGGTGTGAATGGCATTCTCTTTTTTGCAGACAGAGGAGCTGAACCGCGAGGTGGCCACCAACAGCGAGCTGGTGCAGAG
7956	chr11	119091127	119091378	+	0	train	TTAGCCTCCCTATGTCATTTTCCTTATCTGTAAAGTGGGGATAATAATACTACCTTCCTCACAGGGTTGTTGTGAAGATGAAATGAGCTGACATATGGAAAGTACTTTTAGAGCAGTGTCTGGCATGTAGTAAGTATGATGTAACTGTTAGCTGTTAACATTAAGCTGAGAGCTGGAAGATGACTGAAAGTCAGCCAGCTAGAGAGGGAAAGACAGACTCAGGCAGAGGGAACCGCACGAGGCCCCAGATT
13361	chrX	147920992	147921243	+	0	train	TTTACAGCGGGAGAAGGAGTTTGGCTATGGTCCTGCAGGCAGGGAGAAACCACTGAAGGGGAGAAACATACAGAATAAATTTGAGTAAGAAGCTTAAATCTCCCCAAGCCCTTTTAAAATAAATTAAGATATAAAGCCTTTGGGTTTCTTTATGCTTTGTCCTATTCTTCTAATTGTCCAAAACAAAACAAAAACCTTCCTTTTCTGTACCTATTAAAAGGTTAATTTTATAAAGTTACAGACAGCATGCT
2397	chr4	71753797	71754048	-	0	train	ATCTAACATCTGTATAAGAAGAAGCTAAGTTTTTCTGGAGGAAACATGTCATGTGTAGCAAGTGGGTGGCCAATTAAAACAGCCATAAAAATATAGATGACGAAATCTATCTGCAGGAACTTGAGTCAAAATGGCAAAATGAGCACATAGCTTAGCAACATTCTTACCCCAGAATTTTCTGTAGATATTGCTGTACAAAATTGAGGACCTATAAAACTGCTAAAAATGTTATCTAACTTCATGGCATATTG
3313	chr4	121822291	121822542	-	0	train	AGAAAAAGAAGCTCAGAAGAAGCCAGCTGAATCTCAAAAAATAGAGCGTGAAGATGCCCTGGCTTTTAATTCAGCCATTAGTTTACCTGGACCCAGAAAACCATTGGTCCCTCTTGATTATCCAATGGATGGTAGTTTTGGTAAGTTTTAAGGAAAATCTGTGTGGAATTACGGTAATTATAATTATTAGATGATATTAATGATTTGCATTTTAAGATGTCTTATGTAGTCATTTTGACTATAGTGAGTTA
11586	chr17	41584773	41585024	-	0	train	CTTCTGCAGATTGACAATGCCCGTCTGGCCGCGGATGACTTCCGCACCAAGTGAGTTTGAAATGGTGGGCCAGAACATCCAGTGTCCCCAGAGTAGGGCATTTTTGGAGCAGTGTTTCCCAAATAGAACTAGCCAGTACCAGGATAGGTGCATGAAAACTCCCTGGGGTGCTTATAAAAGAATAAGACTCTTGGGCCCCACCCTTGGAGTTTTGATTCAGCTATTTATAGCAGGTTACCTGGGTGATTCTG
9631	chr15	90888392	90888643	+	0	train	CACCATGCCCAGCCCTGACCTCTGTTTTAATAAGGCCACTCTGGCTGCTGTGCTGCAAATAGACTTCAGGGAGCAAGGACAGAAGCTGGGAGGCCAGAGAGCAGGCTCTTGCCATAATCCAGATCCAAGCTTTTGGCCACTAGGACGGGGAGGTAGCAATGGAGGTGAGGCGCGGTCAGGTCCTGGAAGGTGAAGCCAGTGGGATTTCCCTATGGATTGGAAGTGGGGCGTGAAATAGAGGAGTCAGGGGT
5577	chr7	99973558	99973809	-	0	train	TGAGTCGCTGGGACTACAGGTGTGTGCCACCACGCCCGGCTAATTTATGTATTTTTAGTAGAGATGGGGTTTCACCATGTTGGCCAGGCTGGTTTCGAACTCCTGACCTCAAGTGACCCACCTGCCTCAGCTTCCCAAAGTGTTGGAATTACAGGCATGAGCCACCACACCTGGCCCCAGTTAAATTATTATTCACTGGAGTCACTTTGTTGTGCTATCAAATAGTTTTCTAACTATTTTTTTTGTACCCA
11375	chr16	28937165	28937416	+	0	train	CCCCACATGGAGGGGGTTGGAGCGGTCTGTGGCCCGAATAGTGGACTGGGCCCTGGAGGAGAGGGGGCATGACTCGGTTCCCCATCCCCATCCCCAAACCCCCAGGCCCAGAAGAAGAGGAAGGGGAGGGCTATGAGGAACCTGACAGTGAGGAGGACTCCGAGTTCTATGAGAACGACTCCAACCTTGGGCAGGACCAGCTCTCCCAGGGTAAGGCTGCCCTCCCCCGTGGCCCCCCACCTCTGCGGTGG
5684	chr7	99975634	99975885	-	0	train	ACATGTGGCTGGGAGACAGGCTGGATGCAGCTGGGATACCCTGGCATACGGCAGGAATGGGTGCCCAAGGCTGTCAACTCCCTCAGCTCACACACTTCCAGGAGCATTCAGGGAGCCTCTGCGCTGGCCCGAAATAAGACCTTCAGGAATCTGAATCTAAAACCCCTAGTTTACAGTGAAAACAAAGACTCCAAAGACCAAGCGACCTGCTTGGGGTAGACAGTCAGGACGGAGTAGGAACCATATGCCTG
5287	chr7	99967293	99967544	-	0	train	TGGGAAAACATGGAGAAGCACACGGAGCATTCACAACTTATTGCCGTCAGAGTCAATACATGGGTGAGGTGGGGATTGGGCAAGAGGGAAAGCGTCAGCCTTCCCTGATATTCTGGAAAGTCTCCCGGGGCTGGGGGTGGGCAGGTACAGAGCTTCGAGCTCTGCTGATCGCTGACATCCAGGGGTGGGGGTAGGAAGAGACCTGGGCCGGGAGAAGTCCACCTCAAGCCTGCAGTGTCACACTCTATCCC
7667	chr11	116830624	116830875	+	0	train	CTGGCCTCTGCCCGTAAGCACTTGGTGGGACTGGGCTGGGGGCAGGGTGGAGGCAACTTGGGGATCCCAGTCCCAATGGGTGGTCAAGCAGGAGCCCAGGGCTCGTCCAGAGGCCGATCCACCCCACTCAGCCCTGCTCTTTCCTCAGGAGCTTCAGAGGCCGAGGATGCCTCCCTTCTCAGCTTCATGCAGGGTTACATGAAGCACGCCACCAAGACCGCCAAGGATGCACTGAGCAGCGTGCAGGAGTC
13322	chrX	147919519	147919770	+	0	train	ATAGCTGAGTAAAAATAAGCAAGATCAAAATGATAAGAAAATGTTGATTTCTCCCCTTTTGAACCAGTAACTAACTATAAGGGTATATACCCATGCTTAACTTAAAAATAATTATTTAGCCACCTTGGGTATAGCACAACATATGGATGCCATTATAGTCCACCTTGATCTTACAAGGAAGCTTTCTTTTAGCGTAGCTTTACTTTTATTTAAGCATTATTGAAGAAGCTTGGTATCTCTGTTTAAGTTGC
3840	chr5	132677838	132678089	+	0	train	CCCGTGTCCCTTCCACCTCGACTCGCCTACAAAGCCCAGAGAGGTCTGTTTCTTGGCCCCCAGAGCCCAAAGATACTGACACACTCTTACATTTCCAACTAGAATCAGGAACGAGGAGTGACTCTCAGTCAGTTCATTAAGTAAATGTCTTTCTAACCGCTCTGCCCATGGGACATCACGCCCCACAGGGGAAAGGGGAAGCTTCTGTAGCCTGGGATTCTGGTGCCTCAGTCTGGGTCTAGACTTTCCTG
7183	chr11	5226684	5226935	-	0	train	GGGCAGGTTGGTATCAAGGTTACAAGACAGGTTTAAGGAGACCAATAGAAACTGGGCATGTGGAGACAGAGAAGACTCTTGGGTTTCTGATAGGCACTGACTCTCTCTGCCTATTGGTCTATTTTCCCACCCTTAGGCTGCTGGTGGTCTACCCTTGGACCCAGAGGTTCTTTGAGTCCTTTGGGGATCTGTCCACTCCTGATGCTGTTATGGGCAACCCTAAGGTGAAGGCTCATGGCAAGAAAGTGCTC
13623	chrX	147930127	147930378	+	0	train	TCCATCAATGAAGTCACCTCAAAGCGAGCACATATGCTGATTGACATGCACTTTCGGAGTCTGCGCACTAAGTTGTCTCTGATAATGAGAAATGAAGAAGCTAGTAAGCAGCTGGAGGTATGTCACTTTCCCTAGCACTGCTTGTAAGGGTACCTAGGAACGATTAACTGTATCATTCCACAACTTGAATATGGGGAGCTTGTCATTTATTTACTGCTTCTTAACAATTCCTTAGGTTCTATAGTTGGAAA
3276	chr4	121820184	121820435	-	0	train	CTACTTTAGGGATGGCAATGTGCTGGGGCTTAGGGCAGTGGAGGAGGATGAAACTGTTCTAATCTAGAACACTAATTTTCCCCCAACTGGATTCCTTGAACATGATATAGTACACTAGGTTACAAATTTTAAAACATTTTCCTTCCAATATTACTTTCTGGCATATAAGCAGTGTCTCTTTTATTAAGAAGTAAAGGCTGGGTGCGGTGGCTCGCGCCTGTAATCCCAGCACTTTGGGAGGCCGAGGTGGG
1991	chr3	49025134	49025385	-	0	train	AAGCTGCAGTCCTGATGCAGATGTAGCCAAAAGCTCCTGGGTCCTAAATATGGCCACAGGGTCCACTGCCCGTCCCCATTAACCCTATCCACCCATGTGTTCCTCCATCTCAACAGTGCTGGCCTGTGGGCGGCCCCAAGCAACAGCAGTGTACAAGGTGTCAGAGTATGCACGGCGCTTTGGTGTTCCGGTCATTGCTGATGGAGGAATCCAAAATGTGGGTCATATTGCGAAAGCCTTGGCCCTTGGGG
12547	chr19	50876283	50876534	+	0	train	CCTCTCACATGATCACACTCCTGTTTTCTAACTCACTGTCTGTATTTCACCACGACTATATCTCCCCGACCCCTGTGCTTTTCTCACTGTTTCTTTTTCTTCCCTTTGGAGTCTCCCTTATCCTCCCCTGCCCCATCTACCTTTCCCCATTTTCTCTCTCCTCATGCATCCACCCCCTTCCTCCCCAGGAATAGCCAGGTCTGGCTGGGTCGGCACAACCTGTTTGAGCCTGAAGACACAGGCCAGAGGGT
9411	chr15	50258728	50258979	-	0	train	ACCATGTTAGCCAGGATGGTCTCAATCTCTTGATCTCGTGATCCACCTGCCTCGGCCTCCCAAAGTGCTGGGATTATAGGCATGAGCCACCGCACCCGGCCTGCCCTTCCTTTTATTGCTGGGCAGGGGCGGCATGAGGGGTAAGTACTGGGGACTAATGTGAATACCACCCAGAATCCCACAACTGGGTTACTGTAGTGATTTTCCTTTTGCATATTTCCTTCTGGACTTTGCATATATGAAAAGGCTCC
11007	chr16	2129321	2129572	-	0	train	TCTCAAAAAAAAAAAAAAAAAAAAAAAAAAATCACAGGATCTGAACAGAGATTTCTCCAAAGAAGACGCACAGATGGCCAACAGCGTGTGAGAAGATGGTCGGCCTCATTAGTCATGAGGGAAACGTAAATCAAAACCACTGTCCAGCCGGGCGCGGTGCCTCACGCCTGTAATCCCAGCACTTTAGGAGAGCAGATGGCTTGAGGCCAGGAGTTTGAGGCCAGCCTGGGCAACATAGCGAGACCAATAAA
9047	chr15	50246170	50246421	-	0	train	TCTGCTTCATTAGCATTCCTGTGATCACCTACTCAAACACCTTTACAAAACTGTGCTCAAAGTTCTTCCAATGTCCCACAATAGAAGAAATGGAATTAAAATAAAGTTGTTTGATATGATTTGATTTTATCTCTTCAACTTGACGGTGATGATAATTATTGACACTTGCGTAATGCCTTCAATACAAAAATAAATTCTAGTTCTTCATTAGATCCTTAGCACAGTGCCTGGGGTGGTCACTCTAACCTGGC
1635	chr2	79157742	79157993	-	0	train	ATACAGAACTGACATTACTTTTGAGGTTCACAAGCTAATCACAAATGCTACATCAATTATTGTTCTGCAAATAATATATTACCTTGAGTTGTTCCAAAGGTCTTATGTTTATTGGCTGGAATTTTCCAATAGCAATGAGGAGTCAAGGAAGAGTTTCCTACTCACCGGCAGCATCTGGAATAGCAGACCAACTTTCCTCATGCTGGGGAGCAAATCAGGTGTTGCAGCTAAGGGGCCATGCAAGAAGAGCT
12237	chr18	46095893	46096144	-	0	train	GGCCAAGGTGGGTGCATCACCTGAAGTCAGGAGTTCGAGATCAACCTGGCCAACATGGCGAAACCCTGTCTCTACTAAAAATAAAAAAATTAGCCGGGCATGGTGGTGGTCGCCTGTAGTCCCAGCTACTCAGGAGGCCGAGGCATGAGAATCGCTTGAACCTGGGAGGTGAAGGTTGCAATGAGCCAAGATCGTGCCCCTGCACTCCAGCTTGGGTGACAGAATGAGACTCCGTCTCAGAAGGAAAAAAA
15147	chrX	153874723	153874974	-	0	train	TACTTACTGGCCAGGCTTGTGGAGGTCGGGAGGTATTTGACCACGACATTTGATTGCATTGGGTCATGTGTATGTGTGAGTGGGGCTGAATGTAAGTACACACTTGCGTGTGGGGGGTGTTACCGTGATGGTGTTTTCTCCTGTAAGTAGCTGTCAGGCCGTGTGAGGGGCATGTCACAGGGTATCATGTGTGAGGGCATTTGTGTCTCTTGAGCTACCAGGCTCCAGTCCCTTCCCAGCCTCACCTTGAC
14234	chrX	149483331	149483582	-	0	train	TGATCCTGTGGGAGTCAGAATATCCTGTTCCTCAACAGACTCTTTTACCTAGTGGTTGTAGCCTTAATTGATGATCCTTGTCTGAGTCAGTTATTAAATCGGTGGTTCCAAAATTCAGGTTGTTTTAAAAAGCGCCAACAGCCTCGTGGGGCCCTAATTTTGCATCCTGCTATTTGATTGGATGAGTAATTAATGCAGGGTGAGGTGCCGAGGTGGTGTTTCTAAACGTCTGTTGCTAAAGATAAATGTTG
11642	chr18	31591951	31592202	+	0	train	TGTCTGAGGCTGGCCCTACGGTGAGTGTTTCTGTGACATCCCATTCCTACATTTAAGATTCACGCTAAATGAAGTAGAAGTGACTCCTTCCAGCTTTGCCAACCAGCTTTTATTACTAGGGCAAGGGTACCCAGCATCTATTTTTAATATAATTAATTCAAACTTCAAAAAGAATGAAGTTCCACTGAGCTTACTGAGCTGGGACTTGAACTCTGAGCATTCTACCTCATTGCTTTGGTGCATTAGGTTTG
1555	chr2	79085436	79085687	-	0	train	CTGTTTCTGAGTGTGCACACAGGCCTGGTTATTCTATTGATTTTTGAGTGACCATGGCCCCTGTTCTGGCCCTTCTCCATCTAGAACCGCCGCTGGCACTGGAGTAGTGGGTCCCTGGTCTCCTACAAGTCCTGGGACACTGGATCCCCGAGCAGTGCTAATGCTGGCTACTGTGCAAGCCTGACTTCATGCTCAGGTGAGAGGCAGACAATCTATCCACCTGTTGCCATTTCCTTCCCACTTATCTCTGG
3180	chr4	87980814	87981065	+	0	train	TCATTCATATATTTGCTAAGCAAAGAGTAAATTTATTTTCCTTAAGATTCAATTTGAATATACTAAGAATATTAAAGCAAGTTAGATAAATTACCCAATATATTTGTCAATTTGAAATTTGATAGACATTAGTTGTTTAATTCAATGGGCAGTTTTGAGCTGCAGTTTATACACACATGCATAACAGAGTCACCTTTCAATTATCCATGTTAATAGGAAAGTGGTTATAGATTTTAGTACACACATTAAAA
2051	chr3	49027994	49028245	-	0	train	CAGAAGGGCAACGATCATTAGCAAGCGCTCCTGGGAATTGCACTGAGGTGGGGTGGGGTGGGAGTAGGGGTTTATTCTAATTTAGTATTCTTTCTTCCCACCATGGGGTTCAGTTACTGAGAAGACCCTGAGATTCTGTTTCTTAAAGCAGCAGCAATAGACCAGGTGTACAGTGCCTCCAGCCTACCCATGTCTCTAAGATGTGTTGGTGTGATTTGGTCTTGTGGCACTGCCAAAGGGATCGATAAGCA
6535	chr10	47348796	47349047	+	0	train	CTCACCAGCCTCTCAGAAGAGGAACTGCTTGCCTGGCTGCAAAGGGGCCTCCGCCATGAGGTTCTGGAGGGTAATGTGGGCTACCTGCGGGTGGACAGCGTCCCGGGCCAGGAGGTGCTGAGCATGATGGGGGAGTTCCTGGTGGCCCACGTGTGGGGGAATCTCATGGGCACCTCCGCCTTAGTGCTGGATCTCCGGCACTGCACAGGAGGCCAGGTCTCTGGCATTCCCTACATCATCTCCTACCTGCA
1417	chr1	225843052	225843303	+	0	train	AGTAAAATGAGACTGAATCCTGCCTTGGGGGTAGGGACTCTGTCTTGGCCATGCTGTAGCCCCATCACCTAGAACAATGCCGGGTGAACAGTGGGAGAGCCCAGTACATAGATGTTGAATGAATGCGCAAAGGACTGTGAGAGTCCAGTGGAAGGAGGATTCTTTGTCTAAGCGGCAGAGAGGGGAAATGACCAGGAGAATGTATCTGATTTGGGCCTTGGAGGATAAGTAGGAGTTTGCTGGATGGAGGA
11881	chr18	31597445	31597696	+	0	train	AGTTGATTCTAGGGAATTGGCCTTAAGGGGAGCCCTTTCTTCCTAAGAGATTCTTAGGTGATTCTCACTTCCTCTTGCCCCAGTATTATTTTTGTTTTTGGTATGGCTCACTCAGATCCTTTTTTCCTCCTATCCCTAAGTAATCCGGGTTTCTTTTTCCCATATTTAGAACAAAATGTATTTATGCAGAGTGTGTCCAAACCTCAACCCAAGGCCTGTATACAAAATAAATCAAATTAAACACATCTTTA
6546	chr10	47349255	47349506	+	0	train	CGGACTGGGGGAGGGGCCCTGGACCTCCGGAAGCTGAGGATAGGCGAGTCTGACTTCTTCTTCACGGTGCCCGTGTCCAGGTCCCTGGGGCCCCTTGGTGGAGGCAGCCAGACGTGGGAGGGCAGCGGGGTGCTGCCCTGTGTGGGGACTCCGGCCGAGCAGGCCCTGGAGAAAGCCCTGGCCATCCTCACTCTGCGCAGCGCCCTTCCAGGGGTAGTCCACTGCCTCCAGGAGGTCCTGAAGGACTACTA
3664	chr4	154609889	154610140	-	0	train	GGTAGCCCAGCTTGAAGCACAGTGCCAGGAACCTTGCAAAGACACGGTGCAAATCCATGATATCACTGGGAAAGGTAACTGATGAAGGTTATATTGGGATTAGGTTCATCAAAGTAAGTAATGTAAAGGAGAAAGTATGTACTGGAAAGTATAGGAATAGTTTAGAAAGTGGCTACCCATTAAGTCTAAGAATTTCAGTTGTCTAGACCTTTCTTGAATAGCTAAAAAAAACAGTTTAAAAGGAATGCTGA
9092	chr15	50247212	50247463	-	0	train	GGTTAAACAAACACAAACCAAAATGAAGTGGTGGAAGCGGTAACTATGATTTTTTTTAACATTTAAAAAAAATAATTATTATGCTTACATAATAATTACGTATATATTTATGGGGTATGTGTGAGGTTTTGATACAGGCATACAATGTGCAATGATCAAATTAGGGTAACTGGGTATCCACCACTCAACCATTTATCATTTCTTTGTGTTAGAAACATTCTAGTTCCATTCTTTAAGTTAGTTTGAAATGT
3390	chr4	122453557	122453808	-	0	train	CCTCTGGAGGAAGTGCTAAATTTAGCTCAAAGCAAAAACTTTCACTTAAGACCCAGGGACTTAATCAGCAATATCAACGTAATAGTTCTGGAACTAAAGGTAAGGCATTACTTTATTTGCTCTCCTGGAAATAAAAAAAAAAAAGTAGGGGGAAAAGTACCACATTTTAAAGTGACATAACATTTTTGGTATTTGTAAAGTACCCATGCATGTAATTAGCCTACATTTTAAGTACACTGTGAACATGAATC
5301	chr7	99967617	99967868	-	0	train	GAGGGTAACACTGTCTGTAAGAGGCAGAGCTGGGACTCAAATTCCAGATTTCAGATTCCAAATCCCATCGTTTTTTATCTCTACAATGATGCCTCCCATCTGGGTGGTGGAGAGAAGGGAGGCGTGTAAAATGTCAGCCCCAGAAGGACAAGAGCAAGCCAGTGTGAGCGGAATTGATGGCTGCAAGCTGAGACTTGGATTGGAGACGTAGTGAGACTCAGGATTGTGCAGTGCTGCAGGGAAGTGGTTGC
13697	chrX	147932030	147932281	+	0	train	ACATTTTTTTACCTAAATAAGTAACATTGAGTTTGTCAGACCGAGTGAGCTGGTTTTTACTGGGGAACATTTGACCTGCTTCTTAGTTATTATATTGCTGATGTCATTGGTTAACTAAGGCAGCCTTGTGATAAGTTTTCAGCAGTTACCTTGGAAAATGTCAACAAGTTCTTTCCTATTAACAAAAACATATAAAATGTCATAGACCTTTAGTAACCTAAAAAGTACTTAGGCTGTGAGAGGTATTTTCT
3858	chr5	132679070	132679321	+	0	train	AGGACACCTTTCCCTGATCTGTGCACTTATCTCTTGCTGCCTGGCAAAATGTCTTAGCTCCTCACTTGGGCCATGTGCTGCTCTCCTCTCCCATGGGGAGAGCCACACGGAGAGTGCTGGCCAAAGCAGCAGAGTTCAGGCCAAAGGATGTGCACTCATTTATTCAACAGGCATGCAGGATTTCCAGGGAAAGCTGGATTTTAAAACCTCTGGGAACAAGAGCAGAACCTGACTGAGAGCTCATGTGGGCA
11063	chr16	2130880	2131131	-	0	train	CTTGGGGGTGCCACCTCCATGGTGTCACCTCCGTGGTGCTGTGAGTGTGTGCTTTGTGTTTCTTGTAAATTGGTCGTTGGAGCCGACATCCCATTGTCCCAGAGGTTGTCCTGGCTGGCACTGGCCTAGGTGTAGATGTCATCAGCTCAGGGCCCCCTGCTCTAAAGGCCACTTCTGGTGCTGGTTGCCACTCACCCTGGCTGGGGGTCACCTGGGTCTGCTGCTGTCTCGCAAATGCTGGGGTCCAGGAC
6225	chr9	130703400	130703651	+	0	train	GGTGGGACTAGTTTATGAGAGCAACCTCTGGACTACATCTTACCTCTCCCAAATTTCAGCCAGCTTTTGGATGAATGTCAGAATAATATCTTTGTTCTTGTCGACTATATCCCAGAGCATCTTGCTGATTTATTAGGCTGCTTCTAGCAGATCATATCTTGTAACGCTTAGTCTCTGAGACATGAAAGGAATACAGAAAGTTTATGTTAACGGCTTGATTTATGTTTGGATTGGCTTGGTGGTCTGTTTAT
5043	chr6	37173212	37173463	+	0	train	AATGGGGCTATTAGTCTTCATGGGACAGTCTTTGAAATTCTGGAGAGCTTCACTCTCCAGTAGATTCTGTCACCCTTGGCTTAGAATTGTAGGTGAGTGATTTACACTTGAGCTGGCCTCATAAATCACATGGTTTGCACTTGAGCTTTCCTTGGGAGGTCAGAGGAAGGCATGTGTGAGCATATTAAGAAGAAAAGACAATCTGGCTTCTCCAAAAACTTTTTTAAAGGTACCAACAGAAACCTGATAAT
10377	chr16	2107259	2107510	-	0	train	GGTGACCCTGAGCCCGTGGTGGCTGCTCCTGCTGCTGTCAGGCGGGGCCTGCTGGTGCCCCAGAGTGGGCGTCTGTTCCCCAGTCCCTGCTTTCCTCAGCTGGCCTGATTGGGGGTCTTCCCAGAGGGGTCGTCTGAGGGGAGGGTGTGGGAGCAGGTTCCATCCCAGCTCAGCCTCCTGACCCAGGCCCTGGCTAAGGGCTGCAGGAGTCTGTGAGTCAGGCCTACGTGGCAGCTGCGGTCCTCACACCC
13079	chrX	147912336	147912587	+	0	train	GGGTGCCAGGGCACGCTCGGCGGGATGTTGTTGGGAGGGAAGGACTGGACTTGGGGCCTGTTGGAAGCCCCTCTCCGACTCCGAGAGGCCCTAGCGCCTATCGAAATGAGAGACCAGCGAGGAGAGGGTTCTCTTTCGGCGCCGAGCCCCGCCGGGGTGAGCTGGGGATGGGCGAGGGCCGGCGGCAGGTACTAGAGCCGGGCGGGAAGGGCCGAAATCGGCGCTAAGTGACGGCGATGGCTTATTCCCCC
12389	chr19	41879017	41879268	+	0	train	CAAGGTCCCAGCATCATTGATGGTGAGCCTGGGGGAAGACGCCCACTTCCAATGCCCGCACAATAGCAGCAACAACGCCAACGTCACCTGGTGGCGCGTCCTCCATGGCAACTACACGTGGCCCCCTGAGTTCTTGGGCCCGGGCGAGGACCCCAATGGTACGCTGATCATCCAGAATGTGAACAAGAGCCATGGGGGCATATACGTGTGCCGGGTCCAGGAGGGCAACGAGTCATACCAGCAGTCCTGCG
7899	chr11	119088636	119088887	+	0	train	TGCTCGCATACAGACGGACAGTGTGGTGGCAACATTGAAAGCCTCGTACCCTGGCCTGCAGTTTGAAATCAGTGAGTTTTCTGGAAAGGAGTGGAAGCTAATGGGAAGCCCAGTACCCCGAGAGGAGAGAACACAACATTTCTGGCTTTGCCTATAGCTAAAGCCCGTCCCGCTGCCCCGAGATTCCTTCTGGGCTGCTCCCAGTTCTGAAGGTGCTTTCCTCTGAATACCTCCAGCTCTGACTACCTGGA
11178	chr16	2133986	2134237	-	0	train	AATGGAGCCCGTGCCCCTCGGGGCCACATTGCTCCTGCGCTCCCTGACTGCGGACGCGTGTGTCTCGCGGCTGTCTCTGTGGAGATGGCCTCCTCCTGCCTGGCAACAGCACCCACAGAATTGCATCAGACCTACCCCACCCGTTGTTTGTGATGCTGTAGCTGAGGGCTCCTCTGTCTGCCAGGCCGGTCACTGGGGACTCTGTCCAGGGCCTGGTGGTTCCTGCTTCCCAGCACCTGATGGTGTCCATG
3806	chr5	132674702	132674953	+	0	train	TGTAATAGAAAGCTTACATGCTGTAGTCCTGACTCAGATCCTGGTCAAAGAAAAGCCCTCTTGGGTTTTACTTAGCTTTGGCATAGTGCCTGGAACGTAGGAGGCACTCAATAAATGCCTGTTGAATGAGAGAATTTTTCTGGCCCATACATTTCTGAAAAACCAAATACTCTCACAGAAACAGATATTGAGATGACAGGTTGAGGGAGCTTTCATTTTGTCTAAGAGACTTCCTATGGCAACAGAAAAGG
7442	chr11	62613358	62613609	+	0	train	GCTGCTGGCGCTGGCTGGTGGCGTCATCCTCCTCTGTAGTGGGCACCTCCTGGTCCAGCTAAGGCACCTTGGCACCTTCCTGGCTCCCTCCTGTCAGTTCCCTGTCCTGCCCCAGGCTGCCCTGGCAGCGGGCGCGGTGGCTCTGGGCACAGGACTAGTGGGTGTAGGAGCCAGCCGGGCAAGTCTGAATGCAGCTCTATACCCTCCCTGGCGAGGGGTCCTGGGCCCGCTGCTGGTGGCTGGCACGGCTG
14622	chrX	149496846	149497097	-	0	train	CTTTAATGAGCTTCCCTAGTTAGCAGCATTTCACATGTATTGTCACACAACTCGTTGCTGGGAGAAGTAAGTATCCTATGTGACTTCCCTGGAAGAGGACTTGAAGCTTGTGCCTAGTTTCCTCTAGACTTTGCCTCATGTGCCTTTTTTCTTTGCTGAGTGTGTTTCACTGTAATAAGTCATAGCTGTGAGTGAGACTAAATGCTGAATCCTATGAGTTCCCCTAGTGAATTGCTGAACCTGAGGGTGGT
6456	chr10	47310055	47310306	+	0	train	GCGCCCCCACCGCGCGGCCTGTGGCAGGCCAAGGACATCTCCCCCATCGTCAAGGCGGCCCGCCGGGATGGCGAGCTGCTCCTCTCCGCCCAGCTGGATTCTGAGGAGAGGGACCCGGGGGTGCCCCGGCCCAGCCCCTATGCGCCCTACATCCTAGTCTATGCCAACGATCTGGCCATCTCGGAGCCCAACAGCGTGGCAGTGACGCTGCAGAGATACGACCCCTTCCCTGCCGGAGACCCCGAGCCCCG
16235	chrX	155998593	155998844	+	0	train	GAGTTTGAGACAGTATCAAGAGTGGGTATCCTCTCAGCGTGTCTTCAAGTAGCATCCCAGAGCCCCGCTCCTGGGTACCCACAACATGAACACCGTCCAGAAGCAAGGGCAGCCTCTGCAGGTGGGGCGGGGGTTGGAAAACATTTATCAAACGGTTGAGTTGGGTGCAGGGGATGCAACATGATCAAACAGGGTCTTGCCTCCAGGAGCCTCATGTAGGGACAACGCACAGTGATGACCTTCAACTGCAG
10195	chr16	2100287	2100538	-	0	train	GTATGGGGTGGACAGCCGGAGCGGCCACCGGCACCTGGACGGCGACAGAGCCTTCCACCGCAACAGCCTGGACATCTTCCGGATCGCCACCCCGCACAGCCTGGGTAGCGTGTGGAAGATCCGAGTGTGGCACGACAACAAAGGTTTGTGCGGACCCTGCCAAGCTCTGCCCCTCTGCCCCCGCATTGGGGCGCCCTGCGAGCCTGACCTCCCTCCTGCGCCTCTGCAGGGCTCAGCCCTGCCTGGTTCCT
656	chr1	159303116	159303367	+	0	train	ATTCTCTCTCCTAGACACTTTGGCATGATCTCGCTCAATAATTACATTATTATTATTATTGCCATTTTATAATTGAGGATGCTGAAACTCAGTGATTTTCTGGTGGTTACATGGCTAAGGAACTGGATTTCAACGTAAGTTCCTTGGATCTAAGTCCAGTTCTCTTCTGACTATATCACCCTTTTGTTATCACCATGTATCTACTTCTTTGGTCTCTGTTCAAATTTGCACTACATCCCCTTGTTCCAGGA
9109	chr15	50247967	50248218	-	0	train	GAGCCTGTGTTCCTCTGTTTTCTCATGGCGAGGAAGGCTGGTTTCTACTTTGCAGATCTGCCTAGGAGACTTGCGGTGTGTGGTTCAGGTGTCTGGGCCTGTTCCTCAGCTCCTGGGACTTGATTCTCTTTGTTAACCAGAGGAGCGCTGAGCACAACATGCCCTGAGGGCTGCCTGATGCCTGTCCTCTGCTGAGTGTCTCACCGTTCCCCTTGCTGTCACTGAGCACTGGGGAGGGTTGGCCCCAGAGT
7748	chr11	116836986	116837237	-	0	train	CCCCCACCTCCAAGCTTGGCCTTTCGGCTCAGATCTCAGCCCACAGCTGGCCTGATCTGGGTCTCCCCTCCCACCCTCAGGGAGCCAGGCTCGGCATTTCTGGCAGCAAGATGAACCCCCCCAGAGCCCCTGGGATCGAGTGAAGGACCTGGCCACTGTGTACGTGGATGTGCTCAAAGACAGCGGCAGAGACTATGTGTCCCAGTTTGAAGGCTCCGCCTTGGGAAAACAGCTAAAGTAAGGACCCAGCC
4240	chr6	31355713	31355964	-	0	train	GGAAGACAGTCCCTAGAATACTGATCAGGGGTCCCCTTTGACCCCTGCAGCAGCCTTGGGAACCGTGACTTTTCCTCTCAGGCCTTGTTCTCTGCCTCACACTCAGTGTGTTTGGGGCTCTGATTCCAGCACTTCTGAGTCACTTTACCTCCACTCAGATCAGGAGCAGAAGTCCCTGTTCCCCGCTCAGAGACTCGAACTTTCCAATGAATAGGAGATTATCCCAGGTGCCTGCGTCCAGGCTGGTGTCT
3674	chr4	154610103	154610354	-	0	train	TATAATTTTACATTTTCCTCAAGAATGGAATAATTTATCAGAAAGCACTTCTTAAGAAAATACTTAGCAGTTTCCAAAGAAAATATAAAATTACTCTTCTGAAAGGAATACTTATTTTTGTCTTCTTATTTTTGTTATCTTATGTTTCTGTTTGTAGATATTTGCAGGAAATATATAATTCAAATAATCAAAAGATTGTTAACCTGAAAGAGAAGGTAGCCCAGCTTGAAGCACAGTGCCAGGAACCTTGC
5055	chr6	37173419	37173670	+	0	train	TTCTCCAAAAACTTTTTTAAAGGTACCAACAGAAACCTGATAATTCCTGGCTGTTTTGCCAGGGAGTAAAAAGTTAAAAGCTCTTTTAGCATCTTCTTTAAGGCAGCAGCTCCAAATATTTTGGTACCAGTGACCTCACTGTGGGTGGTGTTCGTGTTTGTAAGTTGGTAGGTGAATTGAATCATTTCATCATGCTCAGTGGTGTCTCATCAAAATCTCTTGTCATCATCCTTCCTATTTCTGGTGAGTGG
7893	chr11	119088277	119088528	+	0	train	AGTGATTCGCGTGGGTACCCGCAAGAGCCAGGTGGGTGCAGGAGCCGGGGTGGAGGAGGTTTGTCAGAACAGTTATGATGCTCACAGCATCACAAATTGGGGGACTCAGAGGGTTAGTTCCTAGTATGAAGGAGATGGGGTGGCTGGGCGTTAAGTTCCCCGGGAAATGGCAGATTACATTCTATGGCAAGATCATCCCTAGGCTGGGAAAATTGTTGGAGTGCAGAGGGCTCCCAAGCCCCTTCTCATGC
14320	chrX	149485971	149486222	-	0	train	TGAGAAAGCACTTCCAGAATACCAGTTGGACCAGATACCCCTCCTGGATGTGTGGGCAACCTGTATGGGTTCTTGTGAGTTCTATCCTGACTTGATCTGCCCAGTGCATTTCAGGGACAGGTTGTACACTGTCTCTGAAATGACGTATGACGTTTGCTCTTTTTGGAGTGCCTCTTCTATGAACTATGCTAAACCAGATTGCCCCCTTTCTGGACTTAAGCATTTAATCCATCCTTAACATCTAAATCTTG
14698	chrX	149499828	149500079	-	0	train	GTGTATATATAGATGATGAACAAGTGGTCTGAGGCGGGTGCTGGGCTGTCCTGCCCAAGGGTGGGGTGAGCAAGGCAGGGCTGTGAAAAGGCAACACTCTTTGAGATGGAACAGCAGCTGACACAGCCCCTCGGTCTTTGTGGTATACAAATCATGGTCAGAACTAACTTTGGTCTCACGGCCAGCATGTCTACTTTAGGAAGCAAAGCAGGAGGTTTCATTCTGTGCCTAGTGTAGCTGAAGTTCTGGAA
15631	chrX	154308916	154309167	+	0	train	TTGAACATGTGATTTGTGCATGACCGGGAAGGGCCATGGGAAAATGTCAAGAATCATACTTTACGTTTTCTGCTTTGTACTTGGGCTCCCCGTCTCTATTCCCAAAGCTCCAGCCACCCTTGCCATACTGTATGGATTACCTATGACTCCCTGGGCAGGGACAGTGGATGTCTTCGTCACCTTAGTCACCTGAGTTTCCAGCCCAGGCTGAGCCCTGGGCTTGACCCATTTTCCACTTGTAACGTTGATCG
1266	chr1	192810660	192810911	+	0	train	GTTAAGGTTGAGTCAGTTTTTCCATTGCATACAATGTTTTCAAGAGGCTTAGTTCCCAGAGAATTTATGGCTCCCCTATAAATATTTACTTTGCATTGACAGCAAAGTACTTATATTTTTGCAGCAGAGTGCCAACATAAGCCTTTTGCCTACTGGTATCTTAGTCTTTAAAAAGCTAAATTTTGAGAATTACAGGTTTGAGCGAAATCTAGAAAATTCATTTAGAAATAATTAAAATGTGAGGGATAGGA
4194	chr6	30491120	30491371	+	0	train	TGACTCTTCCCCTCAGAGCCCCCAAAGACACACGTGACTCACCACCCCATCTCTGACCATGAGGCCACCCTGAGGTGCTGGGCCCTGGGCTTCTACCCTGCGGAGATCACACTGACCTGGCAGCAGGATGGGGAGGGCCATACCCAGGACACGGAGCTCGTGGAGACCAGGCCTGCAGGGGATGGAACCTTCCAGAAGTGGGCAGCTGTGGTGGTGCCTTCTGGAGAGGAGCAGAGATACACGTGCCATGT
14504	chrX	149492530	149492781	-	0	train	TTTGTTTTGTTCATTCAGCAAGTGTAGTGAGCACCTCTTATGTACCCAGCAGTGGCCTAGTACCATTTTGTCTAAAGCCTCTGGAATCTGTGTCTGTGTCCTGTTGCAACAGGACTCTTAATAACTTGAAGCCCAGATCATGTCAATTTCTCGCCTTGAAAGACCTCAGTGACTCCATTGGCCTAGAACTTGGAGTCTCCACTCCCTGGATAGACCCACCAGGTCCATATTATCTGGCCCGGACCTAACTT
594	chr1	119512772	119513023	+	0	train	TACTTCACAATGTTGGTCATTTCCCCCTTGAAGGTAAACTAACTCCAAATTTTCTCTGCCACAAGATTAATTTTTGAACACATGAGGCTGTCTTTCCAGAAACTCAAATTGCACAGAGACATTAAAAGGGCTAAGAATACAAAAATTTGTAAATAAAAATGATCCTTTCCCTAGGAAATTCCTTCCTTCTTCAAGGGAAGGCACTATGGGTACCTCCATGAGCTTTGCTTCTACACAGCTCAGGGGAGGCT
15701	chrX	154311520	154311771	+	0	train	TACGGTTAGTAGAATCAGAAAGCGTGCTGTGTGTATGAGCAGGTGCAAGAAATAAGAGTGCATACCTACACACGTGCCTAGGCCCACACTCTTCTGGAAGAACCCTGGGAATTGGGGAAGACGGGAGCTTGGAATCACCAACTATCACATTTCAAGGCTGACAGCTGAAATTGTTTATCTCAGGTTTAACTCATCACAAAGAGTGCGGCTCCTGGATTTTTTTAAAAGACTATAGACATTTTAGAATAAAA
8915	chr14	102084628	102084879	-	0	train	TGGTTCTGATGAGGAAGAAGAAAAGAAGGATGGTGACAAGAAGAAGAAGAAGAAGATTAAGGAAAAGTACATCGATCAAGAAGAGCTCAACAAAACAAAGCCCATCTGGACCAGAAATCCCGACGATATTACTAATGAGGAGTACGGAGAATTCTATAAGAGCTTGACCAATGACTGGGAAGATCACTTGGCAGTGAAGGTGAGTGACTGATGGGTGCTTCAAGCTTGTCCTTAGATTATCATCTTTCTCC
13130	chrX	147913675	147913926	+	0	train	GTAAAAGGAGGCAAAAGGTAACCCCGCGTTTATGTTCTTAACAGTCTCATGAATATGAAATTGTTTCAGTTGACTCTGCAGTCAAAATTTTAATTTCATTGATTTTATTGATCCATAATTTCTTCTGGTGAGTTTGCGTAGAATCGTTCACGGTCCTAGATTAGTGGTTTTGGTCACTAGATTTCTGGCACTAATAACTATAATACATATACATATATATGTGTGAGTAACGGCTAATGGTTAGGCAAGAT
16292	chrX	156002282	156002533	+	0	train	GAGGTTGCAGTGAGCCAAGATCACACCACTGCACTCTAGCCTGGGTGACAGAGGGAGACTGTCTCAAAAAAAAAAATTTGCTTAGAATTTGTCTGTGTGACCCTGGGCAAGTCATTTCCCCTCCTTGGGACTCAGTTCCTTGCCTGTGAACGGGGACAGTGCTTCTCCCTTACAGAGCTGTTGTGAGAATTAAAGTAGAAAATGTACCTATGGTGGTTGTTGGTAGTGACCAGTTCCCCCAACCCTGACTC
423	chr1	109660304	109660555	+	0	train	CTCTGAATCCTTAGAAATTACACGGCTATTTGATCCTGGAAAGATCGTGCAGAGCACACCTGAGTGTCATACAGCCTGGTCTGAGGTAGTGGGGTTGGAGATGAGGTGGGTTGGGGCACAGTGGTGTTTAGCTCAGGTACCAGGTGGGGAGGTTTAGACTTTCTGCTTTAAAAGGAATGATTAGAGCCTGGTCTGGCGTTTCTTTTGCTGGTCCAACACACCTTGACCACTTTCATCCAGGTTTTGCCAGG
57	chr1	67686937	67687188	+	0	train	TTAAGGAGTCAGCTTCATTCTCTGCCAGTCAGAGCTAAAAATAGAAATTGTGTAGGAGACAAACCTTGTTAATTCCCTAGAAATACATTAAGAGGATAGAGTGGAATTTTTTTTCTCTGCAATCTTGCATTTTTTTAATGGCTCTTTTTTTTTTTCCTGATAAAAACCTTTGGTAGGTAGGGAAGTTATGTTTTCAGGGGTAAATGTGCTACTTTTGTCTTCTAAATTTTGCTCTTTTTTGACTGGTCTAG
3574	chr4	154607088	154607339	-	0	train	TGTGTGTGCAACACATAACATTTCAATAAAAGTAGAAAATATGAAATTAGAGTCATCTACACATCTGGATTTGATCTTAGAATGAAACAAGCAAAAAAGCATCCAAGTGAGTGCAATTATTAGTTTTCAGAGATGCTTCAAAGGCTTCTAGGCCCATCCCGGGAAGTGTTAATGAGCTGTGGACTGGTTCACATATCTATTGCCTCTTGCCAGATTTGCAAAAAACTTCACTCAATGAGCAAATTTCAGCC
3569	chr4	154607028	154607279	-	0	train	ACATCTGGATTTGATCTTAGAATGAAACAAGCAAAAAAGCATCCAAGTGAGTGCAATTATTAGTTTTCAGAGATGCTTCAAAGGCTTCTAGGCCCATCCCGGGAAGTGTTAATGAGCTGTGGACTGGTTCACATATCTATTGCCTCTTGCCAGATTTGCAAAAAACTTCACTCAATGAGCAAATTTCAGCCTTAAGAAACAAAGTCAAAAATTCCAAGGAAGCATCCTACGAAAGAGGGAACTTCTGAGAT
5722	chr7	101130548	101130799	+	0	train	CCCCACTTCTTCAGGCTGTTCCGGAGCACGGTCAAGCAAGTGGACTTTTCAGAGGTGGAGAGAGCCAGATTCATCATCAATGACTGGGTGAAGACACACACAAAAGGTGAGCAGGCAGGGAAAGGAAACCCATTTCCTGGGCCTCAAGAGAAAGGGAATTTGGAAATAAATCCACATATCCCAGTTGGGTGCAGTAGTTCACACCTGTAATCCCAGCCCAACACTTTGGGAGGTCTAGGCGAGAGGAAGGC
5076	chr6	42178875	42179126	+	0	train	GATTGACGTCAACGGGGATGGTGAGGGGGCCGAGGAGGGGCTCCCCAGCGGAGGGGTCACCATGGATGTGGGGTCACCAGGGGTGGAAGGTCACTAAAGGAGAGGGTGAGGAAGGGAGGAGAGGCCCAAAGGCCCCCGTGCTGGTCACTTCCTCCACCTGCCTCTGCCCCAGCCACAAAGTTGGCTTTTAGGGGCCCCTGGACCAGAATCTGGGCTCTGGGTTCCTCTGCTTGCTGCACCCGCAGCAGGGG
2264	chr4	71748785	71749036	-	0	train	TCTGTTTGAACAGCATTCTCTTTAATTTCCAATGCCTAACGTGAATGTCCCATTTGTCCTATAATATTTGGATCCTAATTTGTGAGATCTCCTTGTCATACATTTGCTTGAAAAAACTTTTCCTTAAAAGTAATATCCATCTACTCCTTTCACTGCAAGTCACTTATATCCAGGGATCAGTCTCTACTTGTCATTCCAAATACACTTGGCACAACTACTGGAAGACTGGGAAAAGGGGCTTCTGAACCATA
8275	chr12	57232194	57232445	+	0	train	TCCTCTCACTTCGCAGTCTCACCCACGGCTACATGTCTGACGTCAAGCGGATATCAGCCACGTCCATCTTCTTCGAGTCTATGCCCTATAAGCTCAACGTGAGTGCTCTAGGGTGTGGGGAGGGGCTCTTGGCCCTGGTGGTGGTCCTCCCCTGGAGAAGCTGAGGGCCTGGAGCGCCGGGCCGTCCTTAGGGTTAAGGAGGAGAGTGAGCTGCCCTGCTTCCTTCTCAGGGCTTTAGCTGTTTGTGTGTC
3778	chr5	132543012	132543263	-	0	train	TATAGAGATCTGTTATAAATAATAAGATTCTGAGCACATTAGTACATGGGTGATAACTACATCACCAGCAAACATTCTGTTAAAAGTTATGAATGCTGGTGTGCTGTAAAAATGATTGTATTTCCTTTCCTCTCCAGACTCTGAGGATTCCTGTTCCTGTACATAAAAATGTAAGTTAAATTATGATTCAGTAAAATGATGGCATGAATAAGTAAATTTCCTGTTTTAAGCTGTAAATCATTAGTTATCAT
11169	chr16	2133773	2134024	-	0	train	TGGTGGTTCCTGCTTCCCAGCACCTGATGGTGTCCATGAGAGCAGCCCCTCAGGAGCTGTCCGGGAGAGAAGGGCGCTGGTGGCTGCTGAGCGGAGAGCAAGGCCCGTGTTCTCCAGGCCCTTGGCACAGCAGTGGAGCCCCCGCCCCTGCCTTGTGTTGTCCTCTTAGGCTCTGGTCCTGGGGTTTGGAGGAGGGGGACCCTGGGAGTTGGTGGCCTGTCCCAGCCTGAGCTGGCAAGATTCCGAATGCC
1149	chr1	186678160	186678411	-	0	train	ATTTCAGCCAGATCACATTTGATTGACAGTCCACCAACTTACAATGCTGACTATGGCTACAAAAGCTGGGAAGCCTTCTCTAACCTCTCCTATTATACTAGAGCCCTTCCTCCTGTGCCTGATGATTGCCCGACTCCCTTGGGTGTCAAAGGTGAGTAAGAAGAATCCATTAGAGATGTATTAACTATAAGACGGGCTGCATTGCTGCCAAAAAAAAAAATTGACCTTAGACTACCATTTATTTATTAACA
10313	chr16	2105007	2105258	-	0	train	CAGGGCAGACGGGCAGCTTGGCCGAGGAGCTGAGCCTCCAGCCTGGGCTCCTTCCTGCCATGGCGTTCCTCGGTCTCTGACCTGCTTCAGTAGCCTCAGCCGTTCTGTCCTGTGTGAACGCAGGGTGCCTCTCGGGGGACCCAGGGTGTAAAGAGGGGCCCAGATGTGGGGAGGGACTAAGAAGATGCTGCTCTGTGCCCTCCACTCTCCCCTCCCCTCCCCTCCCCCTTCCCTCCCCTAGCCCCTCCCCT
7004	chr11	5226128	5226379	-	0	train	TGCTTTCTTTTTTTTTCTTCTCCGCAATTTTTACTATTATACTTAATGCCTTAACATTGTGTATAACAAAAGGAAATATCTCTGAGATACATTAAGTAACTTAAAAAAAAACTTTACACAGTCTGCCTAGTACATTACTATTTGGAATATATGTGTGCTTATTTGCATATTCATAATCTCCCTACTTTATTTTCTTTTATTTTTAATTGATACATAATCATTATACATATTTATGGGTTAAAGTGTAATGT
13566	chrX	147928364	147928615	+	0	train	TAGCTAAAGTGAGGATGATAAAGGGTGAGGTAGGAAAATGCCTATTTAAATTTTTTTCTTATATTGTTTCCTTTTTTTAAACCCAGGTTGTACATTCCCGTGTGGATTTCTATTTTGAAGTAATATCTAATTTTGAGTAATTTAATTAAAATGTTTTCACTATGTGTTCAGTATGTTTCTGTTGGTCATAAATTTTTTCACATAGATTATTTATTTTAAAATAACTGAATAGGGAGAACTTCTTATTCTTA
16227	chrX	155998210	155998461	+	0	train	TTCTACCAACCCCGAGGCATTTGGGTGTTTTAGGGCAGAAGAGGAGCCAGGAGGATGGGTATGCCCCACTGGAGCTGTGTGTGGGGCAGCAGGTGAGGGTGGGATTCCAGAGGGAGGGTCAACCCAGCCAAGCAGAGGAAGGGGAGGAGAGGGTTTGTTTTGGAAAGAACATCACCCTCTCAGTTTCCTGGGGTCTGGATAGCCTGTTCTTGTGATGAGCTGGAGGATGTGGGCCCTGCTTGGATCCTCCT
8593	chr14	74891327	74891578	+	0	train	ATGTCAAATTCTAAAAGAAAAGTGTATTTTTTAAAAAATAAACCTTATACTCTGTTCTTTCCATGGGTGTTTCTTAGCAGGAGCTGAGAAACTGGACCTTTTCTTATAGATATACAGATTGAGCATACCTAATCTGAAAATTCAAAATCTGAAATGCTCCAAAACTGAAAATCCAAAATCTGAAATGCTTCAAAACTTTTTGAGCACTGACAGAATGCTCAAAGGAAATGCTCATTGTCTTTTATGGGGAA
7467	chr11	64254478	64254729	+	0	train	CGGGCCCCAACATGGTGAGGGTGGGCGCTGGTGCAGCTCGCTCAGGCCAAGGACCCCTGTCCCAGACCCCTGCCCTGCCCGTGGGGGTGGGGCCCGGCTGCAGGCTGACCTCTCCCACTGCTGCCCAGACAGGAAGTGCTCAGGGCAGGAGCTGAGGCTGGGGCTATAGCCAGGGGCTGGCCATTCCCTGGCCTGGGTAGAGAGCTTGGAAGCTCCTGGCTATTGCCCTGCAGCCTCTCATACTCAGCCTG
8133	chr12	14884110	14884361	-	0	train	CTTTCCTCTTCTTCCATCCCTGTATTTAAACTATCACAGTGTCTAAATTGATAAATAATAACATAATGAATCATGGATAAATATTGATATAATGAATCTTTTTTTTTTAATTTCAGAATCACATGAAAGCATGGAATCTTATGAACTTAGTAAGTGAATATTTAACTTCTTTATTCAAATCCCTTGCATTAAAGAACCTCTTCTTATTTTTAAATAAACAAGATGGAAAGATATATAACAGGGAGGGAAAA
15284	chrX	154298098	154298349	+	0	train	AGGAATTTATCCATTTCTTGTAGGTTAGTCCATTTGTTGGCATGCAGTTATTCATAGTACACTGTCACTCTTTTTATTTCTGTAGAAGCGGTATTAATGACTCCACTTTAACTTGATTGTACTTTTTTTTTTTAAGCAGGGTCTTGCTCTTTCACCCAGGCTGGAGTGCAGTGGTGCAATCATAGTTCACTGCAGCCTCGAACTCTCGTGGGCTCAAGCGACTCTCCTTCCTCTCAGCCTCTTGAGTAGCT
15594	chrX	154307771	154308022	+	0	train	CGACTGGGAGCCAGAGCTTGTTACGTGAGTGTGGTGGGTGCCATAGGGAGGGAGATGTGGCTAGAGGCATAGTCAAAAGAGACTTGAGTTGAGTGGGGCAGTTCTGCTGGGCTGGAAATGCAGAACCAAAAGGGAGTCTCTGGGGGGCACAAAGCATGCAGAGAATTCCGGGGAAGGTCAAGAGCAACCAGAGAGTAGCAGAGGGTAGGAAGCAAGAAGCAGGAGATCAGAGAATCGTAAATACCATCCAT
15203	chrX	153906136	153906387	+	0	train	CTGATGGTGTTCGTGGCACCTACCCTGGGTATCGCCGCCTGCCAGGTGCTCATCTTCCGGGAGATTCATGCCAGTCTGGTGCCAGGGCCATCAGAGAGGCCTGGGGGGCGCCGCAGGGGACGCCGGACAGGCAGCCCCGGTGAGGGAGCCCACGTGTCAGCAGCTGTGGCCAAGACTGTGAGGATGACGCTAGTGATTGTGGTCGTCTATGTGCTGTGCTGGGCACCCTTCTTCCTGGTGCAGCTGTGGGC
13900	chrX	147938057	147938308	+	0	train	TATAGTTAATGACATCCCTTGCATTCCTTATACTGCTTTAGGTGTTAGTGGCTTCATCAGTTGTAGCAGGGGAATCCCAGAAACCTGAACTCAAGGCTTGGCAGGTAGGAAAACATTCCTTGAGAAATACACTTTCAGTTTATATTTTAATGTTTATTCCCCTTGTTAACAAAGATTACAAATGATCCTCAGGATTAGGGACTGGAGGGAGGAGTGTTTGTGGGTATGTGGATCCATTGCCCTGGAGTTAA
13661	chrX	147931163	147931414	+	0	train	TCCTTCAGTGATTCTCTCATAACTTTTAGCCCAGAGCTGCCTCCTTACTTTGATTGTGAGTATTATGAAAATATACTCATATGCCTCTTAAACATCGAATTATTCTTAAATGATCTATCATACTTTTGGCTAACCATTTATCAGTGAATTATCAATGAAAAAGAACGAACCAGATTAATGAACCAGGACTCAAAAAACATGACTTCTTGAAAAGTCCTAGAATTATTTTATAGATTTTATTACTCTAGTCT
12115	chr18	46090496	46090747	-	0	train	AAATTTGACAAATTTAAATAGCCACATGTGGTTAGTGGATGCTGTATTGAACATCAGAGATGTAGGTTAAAGTTTTTGGATGTGTAATGATATTCCATGATACTCCAAGTACTTCTCCAGAGTTTTCATTATTGACCAGACTATTCCTGGAATAGCCTTTTTGCTTCAGACATGATTTAAAGATCTAAAATATAGTTTTAATGATGAGAGATGATTTGGTGCTGTTATGTTATGAAAATTTGTATTTGATT
12936	chr22	23894637	23894888	+	0	train	CTGGGAGCTGGGGAGGCGACTCCTGAACGGAGCTGGGGGGCGGGGCGGGGGGAGGACGGTGGCTCGGGCCCGAAGTGGACGTTCGGGGCCCGACGAGGTCGCTGGGGCGGGCTGACCGCGCCCTTTCCTCGCAGTACATCGCGGTGCACGTGGTCCCGGACCAGCTCATGGCCTTCGGCGGCTCCAGCGAGCCGTGCGCGCTCTGCAGCCTGCACAGCATCGGCAAGATCGGCGGCGCGCAGAACCGCTCC
12042	chr18	46087732	46087983	-	0	train	ATCAGTAGGTAAGACAATGTAAAACAGTAGTACATTTAAAGGTTTTGTTGGTTTCTTTAAAGTGGAAGCTGTTTGATTCTGATTAACTCTTTTTTTGTTGTTTGAGACGGAGTTTCACTCTTGTTGCCCAGGCTGGAGTGCGATGGCACGATCTCAGCTCACTGCAACCTCTGCCTCCTGGGTTCAAGCGATTCTCCTGTCTCAGCCTCCCGAGTAGCTGGGATTACAGGTGCATGCCACCACGTCCGGCT
13567	chrX	147928371	147928622	+	0	train	AGTGAGGATGATAAAGGGTGAGGTAGGAAAATGCCTATTTAAATTTTTTTCTTATATTGTTTCCTTTTTTTAAACCCAGGTTGTACATTCCCGTGTGGATTTCTATTTTGAAGTAATATCTAATTTTGAGTAATTTAATTAAAATGTTTTCACTATGTGTTCAGTATGTTTCTGTTGGTCATAAATTTTTTCACATAGATTATTTATTTTAAAATAACTGAATAGGGAGAACTTCTTATTCTTACTTTAAA
8125	chr12	14883909	14884160	-	0	train	TCTTATTTTTAAATAAACAAGATGGAAAGATATATAACAGGGAGGGAAAAGGGGGCCTCTTTTGGAAAACTAAAGTAAATTTTTAAATCTAATGACTATAAAAATTGCCAAAGGAGCAATTTTTTAAGTTTGAAGTAGTGCAATATGGGATTTAAGCTACAGGCGACATATTTAGAAGCCATAAAATCTCATTTGGAAATTTTAAATTGGCACCACGTCAACTGCACAGATGGAAAACGAGGAGTAATGAC
4037	chr6	29666319	29666570	+	0	train	TTGGAGCTACACCACTTAACATGTATTTGTGAGTGACTTCTGGGTTCAGAAGTTCTTCTCACTATTGAGTGATAAAGAAAAAAAATAACTCCATGATGAAAGAGTTTTACATCTTACGGAATGCTTTCATATGAATAATCGGACCTAGCATTTCCCTATGAGCTAACTATGCCATATAGTAACCCCATTTTACAGAGGATACAACTGAGGCCAGGAGTAGTTCAGTGACTTACTCAAACCGATATAACTTA
7866	chr11	119085497	119085748	+	0	train	AGTATCAGCCAAGCCTCCGAACTGCACACAAACGTCTTAGAAGTGCGCCTTCTTTTTGTGTTATAGTGGTCTCCCAGCCACAGCCAACGCTCCAAGTCCCCAGCTGTGACACACCTACTGAATTACTACCGTGGGTGGGAGGCCGCCGTGGGCCTTTCCATTACGAGCCTGCTTGCCGAGCCCTGGGCTTGTGCACAGACAAACTGCAGAGCTGGTGGAGGCCACTGCCAGGCCGAGATAAGAAAGAGATG
12517	chr19	50874516	50874767	+	0	train	AAGCCTCCCGACCTGGTCCAGCCACCAACCCGCTAACGCAGGGAATAGCTACAGAATTGCCAGCCCTCCCAGGACCCCTTGCTTGTGTCCTGGACTCCCAGTCCTGGTCCTCTGCCCCCATGTCTCTTCAAACCCACAGCTCAGCTCCCTCCCCTATCCAATTCTTTTGGGTCTGATCCCCCTGACCCAGCACCCCCTCCGCAGGTGCCGTGCCCCTCATCCAGTCTCGGATTGTGGGAGGCTGGGAGTGT
11682	chr18	31592955	31593206	+	0	train	CCTGCCATCAATGTGGCCGTGCATGTGTTCAGAAAGGCTGCTGATGACACCTGGGAGCCATTTGCCTCTGGGTAAGTTGCCAAAGAACCCTCCCACAGGACTTGGTTTTATCTTCCCGTTTGCCCCTCACTTGGTAGAGAGAGGCTCACATCATCTGCTAAAGAATTTACAAGTAGATTGAAAAACGTAGGCAGAGGTCAAGTATGCCCTCTGAAGGATGCCCTCTTTTTGTTTTGCTTAGCTAGGAAGTG
12929	chr22	23894510	23894761	+	0	train	GCCTCCGTGCCGGACGGGTTCCTCTCCGAGCTCACCCAGCAGCTGGCGCAGGCCACCGGCAAGCCCCCCCAGGTTTGCCGGGAGGGGACAGGAAGAGGGGGGTGCCCACCGGACGAGGGGTTCCGCGCTGGGAGCTGGGGAGGCGACTCCTGAACGGAGCTGGGGGGCGGGGCGGGGGGAGGACGGTGGCTCGGGCCCGAAGTGGACGTTCGGGGCCCGACGAGGTCGCTGGGGCGGGCTGACCGCGCCCT
3712	chr4	154612052	154612303	-	0	train	ACTTGTAACTTTTTAAAAACATAGTCTAGGTTTTACCTATTTTTCTTAATAGATTTTAAGAGTAGCATCTGTCTACATTTTTAATCACTGTTATATTTTCAGGGTAGTTATTGTCCAACTACCTGTGGCATTGCAGATTTCCTGTCTACTTATCAAACCAAAGTAGACAAGGATCTACAGTCTTTGGAAGACATCTTACATCAAGTTGAAAACAAAACATCAGAAGTCAAACAGCTGATAAAAGCAATCCA
7179	chr11	5226673	5226924	-	0	train	TATCAAGGTTACAAGACAGGTTTAAGGAGACCAATAGAAACTGGGCATGTGGAGACAGAGAAGACTCTTGGGTTTCTGATAGGCACTGACTCTCTCTGCCTATTGGTCTATTTTCCCACCCTTAGGCTGCTGGTGGTCTACCCTTGGACCCAGAGGTTCTTTGAGTCCTTTGGGGATCTGTCCACTCCTGATGCTGTTATGGGCAACCCTAAGGTGAAGGCTCATGGCAAGAAAGTGCTCGGTGCCTTTAG
14474	chrX	149491435	149491686	-	0	train	AGGTGTCTTTCATCCCCAGGAGGACCAAAGTTCCACAGGTTTCAGACTGAAGACTTCATCTACCAGAAAGTATAAGTAGGCCAGGGCTCAGCATAATCCTGCTGGAAGGCTAGATGTATATCTTTTCTCTTGACTGCAAGTGAGAACGGGTGAGTCTCATGATTGTCCTCACCCTGGCAGTGATGAGAAGAAGCTGGTGGGTCCAGCTGATAAGTCAGGGGCTGGTCTGCGAAGACACGGCTCTCTTCTCA
5026	chr6	37172638	37172889	+	0	train	GAGATACCTTTCTTGGTTGTGCAGACATGCATCCCTTCATCCTTCGCAGGCGGTCCTGCCTCACAGGGCCTCAAGTTTTGGGTCTGCGGCCAGCTGTGTTTGTTTCTTGGAGCAGTTCATAAAGAATTTCAGTTTATGGTTTGGGCTAGCAGAGAGGTGGGTAATGCTTTGGGTTGGAGAGATGCCGTAAGGTGCGCCTCCACTCTCCTTAGCCCAGAGGGAAAAATGGAGTTCACCTAGCTCCTGAGAGA
16340	chrX	156004117	156004368	+	0	train	GCACTTCAGTCATACCAGGAAGGACTCCAATAAGATGCTGGGAAAAGCTTCCAGCAGCAGACTGTGAAGGAAAGGGAAAGCAAGATTTAGAAACCACCTAGTCTAGGTGCAGAGGCCAGAGGAAGTCATTGCTGTCCTGTCCCGCCTGGGGCTTTTGTGGACCAGTCTCCCAGTGAGGTGCCTGGTCTGAGAGGGCCTTGACCATTCCCCTTGGGAGTCTTTCAGACCCCAGTCTTGTGTGTTCTGACTGA
7503	chr11	64258893	64259144	+	0	train	ACAGGCAAAGATGGCTGAGTACTGCCGCTCCATCTTTGGAGACGCGCTACTCATCGAGCCTCTGGACAAGTACCCGGTACGGGAGCTCGGAGCGAGGGGGGTGGCATGGGGGCCAGGGTGCTGCGGACCCGGTCCAGGGCCCTGACTCGTCCATGCCTGCCCAGCTGGCCCCAGGCGTTCCCCTGCCCAGCCCCCAGGACCTGATGGGCCGTATCCTGGTGAAGAACAAGAAGCGGCACCGACCCAGCGCA
13193	chrX	147915805	147916056	+	0	train	CACAGAGAGGTTAGGTAAGTGACTTACTACCAAGTGTCAGGGCCATTAAGGGTCAGGATTCTGAATTCCTGAAATGATGAAATTTAGCTTGAAGAAATTGGTTTGATTTCCTGCTTAGTTTTCAATTTCATGGTGGTCTTTGATTGTATTTTGTGCTATAACACTGCCTTAGCATCCTATAACTATAGTTACAGTGTTATATTACCATTTTTTATTGTTAATACAAAGCCATCATGAAATAATTCAGTTTA
11312	chr16	28935452	28935703	+	0	train	ACTGGCTGCTGAGGACTGGTGGCTGGAAGGTCTCAGCTGTGACTTTGGCTTATCTGATCTTCTGCCTGTGTTCCCTTGTGGGCATTCTTCATCTTCAAAGAGGTGAGTCATGTCCCCAGTGGGTCTGTCCAAACCCTACTCCATCTTCCCCAGGATAAGCCGGCTCTGGCCAGTCTGACAACCATCTTTCTTTCCTCCCATCCCTCCCTTCAAGACCCCAGAATCCTGTTCTCCCCAGTCTTCCTCTAGCC
13825	chrX	147935965	147936216	+	0	train	GCCACTTTCAGACTTAGATTATCTTTTATAAATGTGCCTTCCATACAGAATTCTGAAAATATTTACCATGGACAAAGAAGCAGATTTAAATCAGGTACAGACAAGAATTGAAAGTGTTGAATAAGTGAATATATTTCCCATTCTCTCTACTTTTTTCTTAATTCTTATAGAACTTTGCAGAAAGAATGATACATACCACAGTACCAGCAAGTTTAAAGCGTACCCTTTTGTGCAAAAATAAGAGGTTATCT
16034	chrX	154324040	154324291	+	0	train	CCTTTCCAAGCACTCATGGGCGTTTCTTTGATTTTTTTTTTTATAAGTGGGCTCCCGCTTTACATATTATTAGGTACTTTTTTGTGTGGTTTTTGTGTTTTGTTTTTTGGGTTTTTTTTCTTTGGAGACAGAGTCTCACTCTGTCGCCCAGGCTGGAGTGCAGTGGCACAATCTCGGCTCACTGCAGCCTCCGCCTCCCAGATTCTCCTGCCTCAGCCTCCTGAGTAGCTGCGACTAGAGGTGCACTCCAC
16374	chrX	156005038	156005289	+	0	train	TGTAGACATGTTTGCCTGTGTGTGCATATGTGTATTTGTGGGCAAACGCAGCTGTGTCTGTGAGTGTGAGTGTGCCTTCTGTGTGTGTGTGTGCACGTGAATGTGGTGAGTGTGTCTGTGTGTTAACACAAGTGTGTTCAAGAGTGTGTTATATGAGCATATAATGCATGTGTGTATTCTCGAGGGCTGAGGGACCCAGCCCCACCTTCACCACCTGCTAACTGTCCCCACCCCCACAGCAGGCCCAGCAC
5782	chr7	101136171	101136422	+	0	train	TCGAGACCAGCCTGGGCAACATGGCAAAACCCTATCTCTACTAAAAATACAAAAATTAGGCAGGCGTGGTGGCATGTGCCTGTAGTCCCAGCTACTTGGGAGGCTGAGGCAGGAGAATCACTTGAATCCAGGAGGCAGAGGTTGCAGTGAGCCGAGATCACGCTGCTGCACTCCAGCCTGGGCAATAGAGCATGACTCTGAAGAAAAGAAAGAAAGAAAGAGAGAGAGAGAGAAAAGAAAGAAAGAAAGAA
9340	chr15	50255264	50255515	-	0	train	TTATTATGCTACATACTTCTTTACATAAATAGCTTTCTTTTTATGGAGTAACTCCCAGGCACTTAATATCTGTGGGATCTGGTCCACACCACTCTCCATGACAGTGTCTGCCAGGCATTTTTAGCAGGATTCTATCAGCTGTAGGTCACTTGAAGATTTCAAGAAGCAAAGACATCAAAGACTGGGTGAATGAATGATAAGGAACCATAAAGACTCCTGTGAAGTTCCTGAGGTGGCCCCGAAGATTTCCA
16454	chrX	156009698	156009949	+	0	train	AGGAGGGCTTCCTGGAGGAGGAGGGATGCTGGGCTTGCCAGAAAGGAGGCAGCTGCTCCCAGGATGAGTTCTGAACATGCTACCTGAGCCCTTCCCTCCTCCCGTGCTCTGTTCCAGACTTGGATGGGGGCCCACGGGGCCGGTGTGCTGTTGAGCCAGGACTGTGCTGGCACCCCACAGGGAGCCTTGGAGCCCTGCGTCCAGGAGGCCACTGCACTGCTCACTTGTGGCCCAGCGCGTCCTTGGAAATC
8683	chr14	74901288	74901539	+	0	train	CAACTAAATGCTAATTGTAAAACATTAACTATAATTTCAGGCCTAAGTTCCATTTCTTAGTTTCTAATAGCTAAGGCACTGATTATCAAATTGTGGTCTGGATCAGCATCATCTGGGACCTTATTAGAAATGCATATTCTTAGACCCCATCCCAGACTTAAAGAAGAAGTGCAGATGATCCTGATGCATATTCAAGTTTGAGAACCACTGAGCTGAGGAGGCTTGTTTGCTTCTATGGAGTGGGGGATATA
1425	chr2	3577504	3577755	+	0	train	GGATTCTGAATGATTTATTCAAGAATCAGGAAGTAACTCCATAGAAGGGTTTGCTCAGTCAATTGTTCGTCTAAGTTGTTTAGCCTTCTCGAACTTTGAACTTACCCTGCCATTCTTCTTGCTTTTAAAGCAGTATGGCAGTTACAGCTTTTTGTCAATTTAAAGTCTTTTTTCATTTTGTTACATGATAATTTTTACCTTACAGAGGAGAATTCTGCCTAAGCCAACTCGAAAAAGCCGTACAAAAAATA
11074	chr16	2131127	2131378	-	0	train	CCACCATGCCCAGCTAATTTTTTGTATTTTTAGTAGAGACGGGGTTTCACCGTGTTAGCCAGGATGGTCTGGATCTCCTGACCTCGTGATCCTCCCGCCTCAGCCTCCCAAAGTGCTGGGATTACAGGCTTGAGCCACCGCCTGTCTTTTAAATGTCCGATGATGTCTAGGAGCTTCCCTTCCTCTCTTTTTCCTTGTGCAATTTGTTGAAGAAACTGGCTCCTGCAGCCTGGATTTCTCGCTGTGTCTTG
4619	chr6	32937419	32937670	-	0	train	AGAAAGAAAGGGGCAATGAAGGAATGGGAGAGAAGAAAGAGTAATGCAGGAATACATTCTAACGGTTCCCCTTCAAGGGGCAGCATGGCAGAGGGGGCTGGGGTGGAAAGTGGGTTGCAAAATCTACGAAGAGTTGCGATAGGGAAGAAACCAGGTTGAGGAAGCAGCCAGAATGTCACCCTCCTTCCTAAACATGTTTTTTTCTCCTATGCAGGGCCACCATCTGTGCAAGTAGCCAAAACCACTCCTTT
5150	chr7	44159013	44159264	-	0	train	TGCCAGCCTCAGGCAGCTCTCCATCCAAGCAGCCGTTGCTGCCACAGGCGGGCCTTACGCTCCAAGGCTACAGCATGTGCTAGGCCTCAGCAGGCAGGAGCATCTCTGCCTCCCAAAGCATCTACCTCTTAGCCCCTCGGAGAGATGGCGATGGATGTCACAAGGAGCCAGGCCCAGACAGCCTTGACTCTGGTAAGGGTCACACCAAAGTTAGGGACTTTGCACTGGGAGAGCAGCACCCAGGGCAGGGC
12224	chr18	46095259	46095510	-	0	train	ACCCCGTCTCTACTAAAAATATAAAAAATTAACCTGGTGGAGGTGGCGCACCCCTGTAGTCCCAGCTACTCGGGAGGCTGAGAATCGCTTGAGACTAGGAGGTGGAGGATGCAGTGAGCCGTCTCAAAAAAACAATAATAATAAAGTACAGACTGCATTTTAAAAATATGCGAGTGTTTACTGTGTACTAGGAGCTGTATAAAAGGTTTTCACATATATCAAAAGCTACTTTCCCATTTGACATTTTAAAC
15051	chrX	153870997	153871248	-	0	train	CCACCTCCCTTCTCCTGCCTAGCACCATCTGGACCAGGAGGAGAGTGTCAGCCCGTCTGTCCCTTCTAGGTGCCCCCAAGTGGCCAAAGGAGACAGTGAAGCCCGTGGAGGTGGAGGAAGGGGAGTCAGTGGTTCTGCCTTGCAACCCTCCCCCAAGTGCAGAGCCTCTCCGGATCTACTGGATGAACAGCAGTGGGTGCCGGCGAGCGGCTGCCTCGTGGGGTCGGGGTGTTAGCGGGGGTGTCTTCCTG
12820	chr22	20783012	20783263	+	0	train	TATGTAAAAGAGGGATAACAAAACGCACACAACTTGCATGTTGCTAGGAGCAGAAATGAGATAATACAGGAAAGGTGCTGAGAAGAATGCCCGGCACATGGCCAGTTCTCAACTACTAGTCACCCATTACTATTAGTTACTCACATCTTAGAGCTAACATAGACATGGGCTTATTCCTGGATACACAGCACTGTCCCCATATCTACAGTGGTGATCCTAAGGGCAACATGGCATCACCCAAATGTCTTGTT
16394	chrX	156005643	156005894	+	0	train	GACAGAAACACCTGCCAACTCTGGGGCTTCCTGGGAACCTGTAGTTAGTGGCTGCTGTTAGGAGTGAGGGTGGCAGGGCTGCACACCAGGGCTGGGCTCCTGCCTGGAGGCTGGACATGACCTCAGTGTCCTTAATGGGGGCTGGACTGACCCTTGCGCACTGCAGTGCTGAGATGGCCCAGGGACTTTATGACCCACCTTGTGGCAGATGGGAAGAGTGAGGCCCAGGAGTGTGGTTCACACAAGGTCCT
11554	chr17	41583031	41583282	-	0	train	GCAGGAGATCGCCACCTACCGCCGCCTGCTGGAGGGCGAGGACGCCCAGTGAGTCTTGGCCCTCCCCTTAGTCCGCCCCCCCCATGGCACTCTCACGGCCCCACCATGTATCTAATGATCCTGTCCTTTTCTATTTTCACAGCCTCTCCTCCTCCCAGTTCTCCTCTGGATCGCAGTCATCCAGAGATGGTAAGACCCTCCTCCTCTGCAGGCCTGGGCTCCAGGCCACCCTCTGTACCCCAAGCAGGTCT
8647	chr14	74895200	74895451	+	0	train	CAAAGATAGGCTGGGCTTGTCTTTCATTTTACCCAGCTCTGACAGTGAAATGGAAAACAAGTTGTGTTTTTAAATAACCTTAGAATGTTAACCAAAGGTACTAATTCACTAGGTATGCCCATTTCTGTAGTGAGCTAATTGTTGACTTACCTGTTTTCTTCAAGATTTCAATGTGTTCAAGTATCAGCACTGTCAGGAAGAGAAATTTGACAAACGTGTTTTGAGTGCGTACTGTATACAGAGTACTGTAC
15681	chrX	154310947	154311198	+	0	train	GTGAACCGCCTGGGACACAGTGGTGCATTGCCCGCCGAGCACTGCATAAACATCTATCAGAGGCGCTGCGAAGCCTTTGGGTAACTGTATTCTCTTGTGCTTGATTTCCATTCTGTCCTGCCCCCTTCATCTCTTGTAACCCAGCCTGCTCCTCTGTTGGCAGGTGGAACACTTATGTGGTGGACGGCCGGGACGTGGAGGCACTGTGCCAGGTATTCTGGCAGGCTTCTCAGGTGAAGCACAAGCCCACT
11595	chr17	41584995	41585246	-	0	train	ACCATGAGTTAGCAAAGTCTTAGGACAGGCCTGGGGCATCTGTTTTCCTTTGGGCTGCTATGGTCAAGTTTTGTGGGGGAAAAGGGGGATTCAGGCAAGAACATGAAGCAAGAGCTTAATGTAGGCTACAGTGAAGTCCAGCTTGTGAAGTCCATTTGACAAATTACCTGTGCCTTTTCCATCCTGCAGATTCTCACAGCCACAGTGGACAATGCCAATGTCCTTCTGCAGATTGACAATGCCCGTCTGGC
5207	chr7	45919071	45919322	-	0	train	TACTGATACACATCGTCATTCTTGGGCTTTAGCAATCATCATGATTACCACCTTAGTAGCACTGTAGTATAGGTTGATGTGAGTTATAAGATTATAAAAAGATCTAAGTGACTTCTAGAATCTATTTGACAAAAAAAGGTAAATTTTCGACAGTCAAAAGTCACAATTATCTGTTGCTTAAATAGAACTGTTTTGTCTTCATGCCCTAGTCTGCAGCCCAGGCATTAAGAAGAAACCAAGGAAATTTAAGA
4137	chr6	30007282	30007533	+	0	train	TCGGTGGGCGGGGCTGACCGCGGGAACTGGGCCAGGGTATCACATCCTCCAGGGAATGTTTGGCTGCGACCTGGGGCCCGACGGGCGTCTCCTCCGCGGGTATGAGCAGTATGCCTACGACGGCAAGGATTACATCGCCCTGAACGAGGACCTGCGCTCCTGGACCGCCGCGGATACCGCGGCTCAGATTACCCAGCGCAAGTATGAGGCGGCCAATGTGGCTGAGCAAAGGAGAGCCTACCTGGAGGGCA
13228	chrX	147916770	147917021	+	0	train	ACTGTCGCCCAGGTTGGAGTGCAGTGGTGGGATCTCAGCTCACTGCAACCTCCACCTCTCGGGTTCAAGCCAGTCTCATGCTTCAGCCTTCCGGGTAGCTGAGATTACAGGCATGTGCCACCATTCCTGGCTATTTTTTGCATTTTTAGTAGAGATGAGGTTTCGCCATGTTGGCCAGGCTGATCTCAAGTTATCTGCCTGCCTCAGCCCTCCAAAGTGCTGGAATTACAGGCATGAGACACCGGGTCCAG
7641	chr11	69774133	69774384	-	0	train	TCTGAAGGTCCGGGACTGGGTGCGGCCGCCGGGGGTCCCCTACACAGGCAAGCTAATCTGAGCTAGCGCAGGCTTGGGCTCCGGAGGCCCTAGAGGGCAGCTTGGGCTCTGGAGGCCCTTGGGGGCGGCTGCGCCGGGAACCCTGGCCCTTTATCCCCAACCCCACCCCAGAAATAGGGTCCCCGGAGGCGAACAAGCCGAGGGGCGGAGTGGGCCAGGGATCACCTGCCCCGCAATGACCTGCGCCCCGC
8150	chr12	14884598	14884849	-	0	train	GTTTCAGTTCTCTCTCTGAACTGGCATCGTGCCCAGGGTGAGCTGTCAGCTGGAGCTAGTGGTTTCTGTGGCTGCCAATTTAACACAGGTTCTTAAGAGGCTTTCGGAACCCTCTTAGAAACCTGCCCTAGTAAGCCCAGCAGAGCAACTGCCCTGTAGTTCTCTTGCCTGGAGAAACCTGGCTGTCTTCTGGATCCTTCTTAATCCTCTTTGACCCTGTTCTCAAACAGGCTCTGAATAAATCAGAGAAG
14266	chrX	149484138	149484389	-	0	train	AGCCGAGATTGCGCCACTGAACTCCATCCTGGAGACAGGGCTAGACCCCGTCTAAAAAAAAAGAATGAAACAAGTGTTAAGAGTGTTAGTGATTATATGGAAAAATTGGAACACTTGTGCATTGCTGGTAAGAATGTAAAATGGTGCAGCCACTGTGGAAAAGAATTTGGTGGATCCTCAGAGTTAAACATAGAATTACTCTATGGCCCAGAAGTTCCACTCCTAGGTATATATCCACAGAGCTGAAAACA
6716	chr10	47356984	47357235	+	0	train	GACACAGACCTAAACCCCCGGGCCTCCTCCATCACTGCAGGCCCAGGCAGGATAGAGAAGACAGGTGCTCCAGGGTCCTGACATGACCCCCATCCTGAAGGGCCTTATGTCTTCCAGGTGAACGCTATGGCTCCAAGAAGAGCATGGTCATTCTGACCAGCAGTGTGACGGCCGGCACCGCGGAGGAGTTCACCTATATCATGAAGAGGCTGGGCCGGGCCCTGGTCATTGGGGAGGTGACCAGTGGGGGC
1540	chr2	10444420	10444671	-	0	train	CACCTGTGACCACTCTTGTTTCAGGATGATAAGGATGCCTTCTATGTGGCAGACCTGGGAGACATTCTAAAGAAACATCTGAGGTGGTTAAAAGCTCTCCCTCGTGTCACCCCCTTTTATGCAGTCAAATGTAATGATAGCAAAGCCATCGTGAAGACCCTTGCTGCTACCGGGACAGGATTTGACTGTGCTAGCAAGGTAAGCGATAGCAGCAGGCCTCAAAAGCGTTGTATAAAATGGGCCTGGTATTC
10964	chr16	2127857	2128108	-	0	train	TCCCACCTCTCCCTCCCTCCAGCCCCTCCCACCTCTCCCTCCCTGCCAGCCCCTCCCACCTCTCCCTCCCTGCCAGCCCCTCCCACCTCTCCCTCCCTGCCAGCCCCTCCCACCTCTCCCTCCCTGCCAGCCCCTCCCACCTCTCCCTCCCTGCCAGCCCCTCCCACCTCTCCCTCCCTGCCAGCCCCTCCCACCTCTCCCTCCCTGGCTCATCCCTGCTGTGTCCCTTCTCTCTAGTTTCCTGTTCAGTT
12315	chr19	2251025	2251276	+	0	train	GACACGGGGCAGGAGCGGGCGGGGGCGGCGTGGCCTCGTGGCCGCTCTCAACTCCTCCAATTGCGGGTTCCAGGCCATCCGCGGAACTCGAGGAGTCGCCACCCAGCGCAGACCCCTTCCTGGAGACGCTCACGCGCCTGGTGCGGGCGCTGCGGGTCCCCCCGGCCCGGGCCTCCGCGCCGCGCCTGGCCCTGGATCCGGACGCGCTGGCCGGCTTCCCGCAGGGCCTAGTCAACCTGTCGGACCCCGCG
7376	chr11	14968165	14968416	-	0	train	CTAGGCTAGCTGGACAGAGGATATGGTGGGTGGTCCCTTTGACCAAGCTCAAGCAGGAAGAACAGGGGTCCTAAGGAGCAGGTAAGCACCTCTAGGACTTGATGCTGCAAACTCCGCTCCTCTTCCAGGTAAGACTGAGGAATTTTTTATTTTCCTAAGAAAGGGTATTTGGTGCCCGTGACTGGGGTGTAGATTTTATAGTCCTTTGTGAATGGGGCTGGGTGTGGGACCATAATTCACTCCAGTGTCAT
10917	chr16	2125911	2126162	-	0	train	TGGCCTCAGGATGGCTCGTACCATCATTGGCTGTGCCCACAGCCGAGTGGGTGATGGGATTCCGGCTGCCCCGCTGGATCTGTGCTGCTGCCCTCTCCAGGGCACTGCTGTGCCCGCACAGCCGGGCGCAGATGGCCAGTTTGCTTGCCCCCCCCCCCACCATCCTCTTCCTACCTTGGCTTCCTCCATTGACACACTGGACCCTGCTGGCTGCCCGGGGAGGTGTTTGGGGGATGGTGTTGGGGGAGGAG
7485	chr11	64255318	64255569	+	0	train	CCCCAGGCCACCCGAGGGGGAGCCGGGGGGTTCACGTGGCCGTTTTCAGGGTGTGACCTCTTCATCTGCCTTCCCAGATACACGAAGCTGAAGCTGCAGGTGAACCAGGATGGTCGGATCCCCGTCAAGAAGTGAGCACCCCTTCCCCCAGCACCTTCCTCCTGCCCTGACCTTGGTGACCTTTGTCCTCCACTGACCCTGAACCCCTCCTGCCCGCATCAGCATCCTGAAGATGTTCTCAGCAGACAAGA
10975	chr16	2128293	2128544	-	0	train	ATGATTTCATCAGCATGCTCTGGGGCAGACCCCTGCAGCCGCACAGGGTGCCTGGGGCCCACACTAGTGCCCTGGTTTATAGACAGACAGAGGTGGCAGTGGCGCTTCCGAGTCGGGCTGCGATGTGCTTGCACTCCCCGAGGGGCTGAGGGGCCCTGCGCCCAGGTGCAGCTGCTTGGGTGCTGCCAGCCCCTCCCACCTCTCCCTCCCTGCCAGCCCCTCCCACCTCTCCCTCCCTGCCAGCCCCTCCC
10445	chr16	2109675	2109926	-	0	train	TGTCGTATACACTTGGTCCTTGGAGGAGGGGCTGAGCTGGGAGACCTCCGAGCCATTTACCACCCATAGCTTCCCCACACCCGGCCTGCACTTGGTCACCATGACGGCAGGGAACCCGCTGGGCTCAGCCAACGCCACCGTGGAAGTGGATGTGCAGGTGCCTGTGAGTGGCCTCAGCATCAGGGCCAGCGAGCCCGGAGGCAGCTTCGTGGCGGCCGGGTCCTCTGTGCCCTTTTGGGGGCAGCTGGCCA
12178	chr18	46092899	46093150	-	0	train	TTTGAGATAGAGTCTTGCTCTGTCTCCAGGCTGGAGTGCAGTGGGGCTATCTCAGCTCACTGCAACCTCCTTCTCCCAGGTTCAAGTGATTCTCGTGCCTCAGCCACCCAAGTAGCTGGGATTACAGGCATGCACCACCATGCCCGGCAAACTTTTGTGTTTTTAGTAGAGATGGGGGTTTCACCATGTTGCCCAGGCTGGTCTCGAACTCCTGGCCTCAAGTGATCCACCCCCTTTGGCATCCCAAAGTG
2648	chr4	71764991	71765242	-	0	train	ATCAATTTTCCCAAACTCTTACTAACCCTAATCCTATTATCTGAACCAGAATCACTTGAGTATCCTGTCCACATGATATATTTTCAAATATTTGACAGCCCCTGTGGTGATTTCCCTGATGCTAAGGATCATTGCGTATGTCATATGATAGAAGCAATTTTATTGCTGTACACATTTTAAATACTTAATTTCCAGTAAAGCATAAAAAATAAACAGCTTCTCTCTTCACTCCTATACATCCCAGAAACATG
15740	chrX	154312820	154313071	+	0	train	TTGTTCTCTAATGTTGTTCGTATGTACACAGGATTCTTCATTTATTCTGGTCTCCATGGACGAACCAATTACTAGAGTGACATGTAACTAACAGTTCGTAGGCAATAAAGAGAGGTTTGCCATTTTTAAAAAATACATTTAGGGGTCCAAAGTGCAGCTGTGTCACGTGGATAGATACTGTGTAGTGGTGAGGTCTGGGCTTTCAGTGCACCCAGCATCGGAGTAGTGGACATTGCACCCAGTAGGTACTT
1800	chr2	112832607	112832858	-	0	train	TATACCTAAACAACATGTGCTCCACATTTCAGAACCTATCTTCTTCGACACATGGGATAACGAGGCTTATGTGCACGATGCACCTGTACGATCACTGAACTGCACGCTCCGGGACTCACAGCAAAAAAGCTTGGTGATGTCTGGTCCATATGAACTGAAAGCTCTCCACCTCCAGGGACAGGATATGGAGCAACAAGGTAAATGGAAACATCCTGGTTTCCCTGCCTGGCCTCCTGGCAGCTTGCTAATTC
15578	chrX	154307276	154307527	+	0	train	GGGGATGAAGAGGCTTTCTATCGGGCAGCAGCACACCAGGCCCTGTTCGGCTGCCAGAATTTGTGTGTTCATCAGATTTAGCTGGCTGATGAAATTGGAACAAAAGAGAAAACACATGAAGCATAGTCCATTGTGGTTTGCCTGGGAACGGGGTAAACAGCTATACTCCAGGCAGAGCAGATCTTTATTCTAGCCTTTCACCTCCCTGAATCGTCTGCCCCGCAGCATAAGGGTTCTTGAGAAAGACAGCA
4691	chr6	32939737	32939988	-	0	train	TGTGACATTAACTAGAGTTAACATAGACACCAAAGGTTAAAGACTCAGTCCCATAAGACTGCCTCCATTTCAGACACCAATCACAAGTAGTAGGTTCCCAAATTACCCACATCTTCTGTCCAACTTGCCTACAAATCAGAGGTTCCCATGACCCCCTCCTTGGGGTTGGTAATTTGCTAAAGTGGCTTATGGAACTCAGGAAAAGTTTACTTATTATTGTAGATTTTTTACAAAGGATATTTTAATTGATA
1435	chr2	3579717	3579968	+	0	train	GTGGTTTTGGAGATGAGCAAGAGTGTTAAAGGTTATTGTAGTGATTTAGTTGAGAAATAATACAAGTGAAAATAAAGATGTATAAAAGTCCTCTAATCTCACATTCCCTAGAGATGGTCACTCATTGCCTTGGTGAGTTTGTAGTTTATCCCAGAGTAACATTTAAAATTGACTGAGCTTTTGAAATAAATCAGTAAGCTATTACCCTATTCTAGGTAGGCAAAGGCTTTAGTTTGGTACGTGAAATATAA
4354	chr6	32181185	32181436	-	0	train	CTGGGGATCCTGGGAGGCCTGGGGACAGCCGCCCTGCTCATTGGGGTCATCTTGTGGCAAAGGCGGCAACGCCGAGGAGAGGAGAGGTGAGTGGAGAAAGCCAGACCCCTCAGACCTAGGGCTTCCAGGCAGCAAGCGAAGAGGGGTCGGGGGGTGGAACGACAACGTGCCGCATTCCCCCCAATCTTTCTCCTCAGGAAGGCCCCAGAAAACCAGGAGGAAGAGGAGGAGCGTGCAGAACTGAATCAGTC
10315	chr16	2105078	2105329	-	0	train	AGTGCATGGTAGGATGGCCCCACCTGCTCACCCTGCCCCGCATGCCTGCCAGGGCACTGGGTTCAGCCCCCCAGGGCAGACGGGCAGCTTGGCCGAGGAGCTGAGCCTCCAGCCTGGGCTCCTTCCTGCCATGGCGTTCCTCGGTCTCTGACCTGCTTCAGTAGCCTCAGCCGTTCTGTCCTGTGTGAACGCAGGGTGCCTCTCGGGGGACCCAGGGTGTAAAGAGGGGCCCAGATGTGGGGAGGGACTAA
15985	chrX	154322797	154323048	+	0	train	GTATGCAAAACCCGGTCTCTGGATATAGAAAAGATAGAGCAGAAAAGGGAGAACAAAGCCTGAGGACCTTCCTAAAGAGAACGGGTTCAGAGACTTAGCAGGCCAAACTCAGGCCACCAATGCTCAGCCATCGTGGACAGTAACTTGGAGTGATCCATCTGAAAAGTCAGCTGAGACTTTTCAGTTGACTAACCCGAGGTGTCACAGTGTGATCTCAAGGGAGACAGGCGTGCAAGAACACAATGTAAATG
7745	chr11	116836976	116837227	-	0	train	CAAGCTTGGCCTTTCGGCTCAGATCTCAGCCCACAGCTGGCCTGATCTGGGTCTCCCCTCCCACCCTCAGGGAGCCAGGCTCGGCATTTCTGGCAGCAAGATGAACCCCCCCAGAGCCCCTGGGATCGAGTGAAGGACCTGGCCACTGTGTACGTGGATGTGCTCAAAGACAGCGGCAGAGACTATGTGTCCCAGTTTGAAGGCTCCGCCTTGGGAAAACAGCTAAAGTAAGGACCCAGCCTGGGGTTGAG
13548	chrX	147927593	147927844	+	0	train	TCTGTTGTCTCTTTTCCTATTAACCTTGATTTGCTAGGGTCTTCATGAAAGGTTCCATGCTGTACATCTTAAGTGAACTTTGAAGTGGTCATCAAAGAGATCAGTTAGAAGAACTGATCTGCAGTGTGTGACAGACCAGGAAAGTGATATCCAGAGGATTGCCACCATGAAGTTGTCATTTTTAGTGGAGAGGAAGCAGGAGACCAGAAAGTGACCAAGGATCTGTCCATTTAGTTCCAGGAGATATGACT
9808	chr16	164468	164719	+	0	train	CCCCCCGTCCCAGGCTCTTCCTCAGCCACCCGCAGACCAAGACCTACTTCCCGCACTTCGACCTGCACCCGGGGTCCGCGCAGTTGCGCGCGCACGGCTCCAAGGTGGTGGCCGCCGTGGGCGACGCGGTGAAGAGCATCGACGACATCGGCGGCGCCCTGTCCAAGCTGAGCGAGCTGCACGCCTACATCCTGCGCGTGGACCCGGTCAACTTCAAGGTGCGCGGGGCGCGGTGCGGGCGGGGCGGGGCG
6410	chr10	47307062	47307313	+	0	train	CTGAAGACGGGTGTCATTGCCATTACCAGTCTGTCTCCCCTCTAGGCTGAACTCCTCCTGGGCAACTTTGTTATCTGTCTCCCCTCTAGGCTGAACTCCTCCTGGGCAACTTTGTTATGACCCTGGTCAGCCCAACTTAGCCTGGAGTAGGAGTTTTGCAAATCTTTATCAGGAAAACAAAGAGAAGAAGAGGAAGAAAGAAAGGAAAAGAGGGTGGATAGAAAAGGGGAAAGAGGGGAAGAAAAGAGAGG
6952	chr11	5226026	5226277	-	0	train	AAAAAAAAACTTTACACAGTCTGCCTAGTACATTACTATTTGGAATATATGTGTGCTTATTTGCATATTCATAATCTCCCTACTTTATTTTCTTTTATTTTTAATTGATACATAATCATTATACATATTTATGGGTTAAAGTGTAATGTTTTAATATGTGTACACATATTGACCAAATCAGGGTAATTTTGCATTTGTAATTTTAAAAAATGCTTTCTTCTTTTAATATACTTTTTTGTTTATCTTATTTC
10150	chr16	2098561	2098812	-	0	train	CAGGCTCAGAGCCTTCACGATAGAATTTTTCTAAGCAGTTAAGGAAGAATTAACACCAATCCTTCACAGACTCTTTCCAAGAATACAGCAGGTGGGAACGCTTCCCATTCATACGGAAACGGGAGGCCGCACCCCTTAGGAATGCACACGTGGGGTCCTCAAGAGGTTACATGCAAACTAACCCCAGCAGCACACAGAGAAGGCGCATAAGCCGCGACCAGGAGGGGTTGCTCCCGAGTCCGTGGCAGGAA
13695	chrX	147931961	147932212	+	0	train	AGAAAATATATACTGATACCTTTTAATAAAAATGAAGCGAATAAAATTTTCTATAAAGATGATATTTAAACATTTTTTTACCTAAATAAGTAACATTGAGTTTGTCAGACCGAGTGAGCTGGTTTTTACTGGGGAACATTTGACCTGCTTCTTAGTTATTATATTGCTGATGTCATTGGTTAACTAAGGCAGCCTTGTGATAAGTTTTCAGCAGTTACCTTGGAAAATGTCAACAAGTTCTTTCCTATTAA
8540	chr14	74885841	74886092	+	0	train	GTGTGCCCACTCTCCACAGGTGTTTGGATTATAATGAAAGTTAAGTTGGAATCCTCTTTTTATGGGATATAGAAGGGTGCTCCTAGATATTCTGTTTGGATGTAAGTGCAGCTGAAAGGCTCGAGGTTACATTTATTGTGGAGATTGAGAGCATCCTCTCAGTAGCATAGTTACTCATTTTTAAAAACAGTAAAAGCATAAAACTGGTTGAGTCCTGGGACTGACTGACTAGTTTGGAGGGAATGCTGTTA
6118	chr9	130699702	130699953	+	0	train	TCAAGATATTAATGACAGGCCGAGTGTGGTGACTCATGCCTGTAATCCCAGCGCTTTGGAAGGCCAAGGAGGGAGGACTGCTTGAGCCTAAGAGTTCAAGACCAGCCTGGGCAACATAGCAAGACCCCCCTCTCTACAAAAAAATTAAAAAAGAATTAGCCGGGCGTGGTGGCACATGCCTGTAGTCCTAGCTACTCAGGAGGCTGGGGCCAGAGGGTTGCTTGAGCCCAGGAGCTCAAGGCTACAGTAAG
16435	chrX	156008166	156008417	+	0	train	CCCTGCTGGGCTGTTGGTTCATGCCCCCTGGGTGGGAGGAGGGGGAGAGGGAGAGCTCCAGTGAGTGGTCTCTGGTTTTTCCCCTCAGACTCCTCACTTTGGGCAAAGGACAAGAGGCAGTGAGGGCCCCTCCCTGGGGTCTGGGCCAAGCTGACCACTCTTCTCCAGAATCTTCCCTCCCTGTCCCCTTCACACTGTGGCTCCAGCTTACTATGCAGAAAAATCCTTTTCTCTCTCAATGAGGAGCGTAG
2942	chr4	71778442	71778693	-	0	train	TGGCCCTCATGGTTTGGGCCAACAAAGTTTCTTTAAGTCTGTCTTTAAAGTCTCATTTCACTTGTCAACATTTTAAAACCTGGAGATTCCACAAAAAAATCTAGATTTCTAGCTTCTCTGGAAACACAGGAATGTGTGGTAATAAGGGCCAAATTTTCACATGGCAATGATGAACTGGAGCTGAATTGTCACTCCTCCTTTTGGACTAAACCCTCAGTTTTCACCACTGAGCATTTAATGCTTTCTTCCCT
2415	chr4	71754274	71754525	-	0	train	TTATTGTTTTCTTACAGGGCCCTCTACTAAAGAAGGAACTATCTTCTTTCATTGACAAGGGACAAGAACTATGTGCAGATTATTCAGAAAATACATTTACTGAGTACAAGAAAAAGTAAGAAACTTGTTCTGGCTGTATCCTCCAAATTTATCAATAATATTTTCATAGTACTATGAATTGAAAGCATAGTTGAACACTTAAGCTTGTCTTCAGTGAACAACAACAAAAGGAGGTTCAATTAGGAGTTTTT
7812	chr11	118341082	118341333	-	0	train	CACCTCCAGCCCTAAAAGTCTCCCACAAGACTTGACTTGTGAGGAGCTCTAGGGATGGATTCAATAAAATTTTTTTCCTGTTCTCCTCCTTCTAACCAAGGGTGGAAAACTGTACATGGGATTTGCCCTGGTCCCGTTCTGGACCACTTGGCTGATAATTCAGGGGCTGGTAAACCAGCTGGTCTCCACCCACAGCCATCCCTGGCCCTGGGAAAGGCTTCAGTGTTGAGAGCCCCGCTTTCTTGGTTGTC
6563	chr10	47349607	47349858	+	0	train	GCCTGCAGGCTGCGTCTGAGGATCCCAGGCTCCTGGTGCGAGCCATCGGGCCCACAGAAACTCCTTCTTGGCCCGCGCCCGACGCTGCAGCCGAAGACTCACCAGGGGTGGCCCCAGAGTTGCCTGAGGACGAGGCTATCCGGCAAGCACTGGTGGACTCTGTGTTCCAGGTGTCGGTGCTGCCAGGCAATGTGGGCTACCTGCGCTTCGATAGTTTTGCTGACGCCTCCGTCCTGGGTGTGTTGGCCCCA
1975	chr2	162148513	162148764	-	0	train	TTTTCTCTTTTTTCCTCCTTTAAAAAATAGCATTATTTCTTCCTAAATGTAAAAGCCATACATGTTCATGATGGAAAGTATGAAAAATATGCAAAAAATATTAAGTACTCAAAATTCCTCTGTCCAAAGAAAGCTATTCAAAGTAAACAAATTTGGTGTATTACTTTCCTGTGTTTTACGTAAACTGTACATAAATATCTCTTGGCTCATTATATGGCTTTGTATCATGCTTAATATCTTAACATTGTATT
1518	chr2	10443480	10443731	-	0	train	AGAGCTAAATATCGATGTTGTTGGTGTCAGGTGAGATTTTGGTGGGATAGCTAGAGGTCAAGACATTGAACAGTTTGAGTTTTACAGGCTTTCTCCTAGTGTTTGCTATTATTTTAAGAAATACTAAGACACAGTGTCTCGTCTCTTTATTTTACCCCAGCTTCCATGTAGGAAGCGGCTGTACCGATCCTGAGACCTTCGTGCAGGCAATCTCTGATGCCCGCTGTGTTTTTGACATGGGGGTGAGTATA
1410	chr1	225841429	225841680	+	0	train	CTGCCACCATGCCTGGCTAATGTTTTGTATTTTTAGTAGAGACGGGGTTTTACCATGTTGGCCAGGCTGGTCTCGAACTCCTGACCTCAGGTGATCCACCCGCCTCGGCCTCCCAAAGTGCTGGGATTACAGGTGTGAGCCACCGCACCCAGCCTGTCCCACTGTCCCAGCCTTTTTTTTTTTTTTTTTTGAGACAGAGTCTTGCTCTGTTGCCCAGGCTAGAGTGCAGTGGTACGATCTCGGCTCACTGC
4622	chr6	32937540	32937791	-	0	train	TTCCTTATTTCAAATACTTTCCAGTTTATGTACTTGAAATAAATACAACAACTTCTAGAACAGCTTGCAGTTCAGATCTGGCTTTTACTAATTGTAATCAAACATAATTCTGGAGGAGGAAAGAAAGAAAGGGGCAATGAAGGAATGGGAGAGAAGAAAGAGTAATGCAGGAATACATTCTAACGGTTCCCCTTCAAGGGGCAGCATGGCAGAGGGGGCTGGGGTGGAAAGTGGGTTGCAAAATCTACGAA
8388	chr14	24574616	24574867	-	0	train	CTGCAGAGCTGGTGGACCATAGCTCCTGCAGCCCAGACCTACCTCTTGCTTTTGCAGCAATATAAATGTCACCCTGGGCGCCCACAATATCCAGAGACGGGAAAACACCCAGCAACACATCACTGCGCGCAGAGCCATCCGCCACCCTCAATATAATCAGCGGACCATCCAGAATGACATCATGTTATTGCAGGTACCACCTACCTGGCCCTCTGGCTCCTTCCTAGTGTGTCCGGGGACAATGGAGGAGG
8236	chr12	49297925	49298176	+	0	train	CCCCCTTGAACCCTTTATCCTGCTTTCTTCAGTGCCTGAGGTGGAGCCTCCCCAGGACAGCCACAGCCGGAAGACGGTTCTGATCAAGACCATTGAGACCCGGAATGGGGAGGTGAGGCAGGTCCCCTAATGCCAGGACCCCACCATTTCCCATATTATTTCTGAGCCCCAGCCTGGCTGTCGCTAATCTTACTTAGGGTGGGTAATTCTTGGGGAGAGACAGAGAAGGAGACAGAGAACGTGGATAGATA
14148	chrX	147946605	147946856	+	0	validation	GTGAGTGCACTGTAGCTTCAGTCCTTTGCCCACAGTCACATAGCTAGCATATGTCCAAATGATAGTCAAACCTTAGTGTCTGATTCTCTGAATCTATGTTGTCGTCTACTTTTTCTGTAACTTTTAATTATTAAGTAGTTAGGGTTCCTTCCTGACTATGGATTCATTTGGGACCATCACCCAAGATTAGTGAGAGATACTTTTTAAGACAAAGTTTATGATGGAATATTTCTTGGAATTCATAGCAGCTC
8567	chr14	74888903	74889154	+	0	validation	TGCTGAGTCAGAATCCCTGGGGGAGAGGCCTAGACAAGTGAACTTTGAAAAAGTTCCTCCCCAAGTGACACTGATGGACACCCCTGGTCAAGAGTCACTGTTTAAGGGGAAGGTGACCCATGGGGCCTTAACTAGACTAAAATGTGAGTGGTTCGCCTGTTAAAAGGAGTTAACGTGTGTTTCTTTTGTAGCATTAACAACAGTGTCTTCAGTGTTCGCTTTTTCAGAACTACAGCTGTATGCAGTAAGTA
13520	chrX	147926688	147926939	+	0	validation	TGTTGGTCAGGCTAGTCTCGAACTCCTGACCTCAGGTGATCCACTCGCCTCGGCCTCCCAAAGTGCTGGGATTACAGATGTGAGCCACCATGCCCAGCTGGTCCTTGAATTCTTATTTGCTGCACTAGGTAGTAAAGGTGTAGCAAAAACAGGACAGAATCTTCTGCCTTTGATGAAATCACATTCCAGTAAGGATAGCTAATATGACAGATGTGAGTCTTAAAGAAAAAAAAAGGGAGAGGATGAAAGGT
1244	chr1	192810173	192810424	+	0	validation	ATTGGAAGACCCGTTTGAGCTACTTCTTACAAAATTCCTCTACTCCTGGGAAGCCCAAAACCGGCAAAAAAAGCAAACAGCAAGCTTTCATCAAGTAAGTTGAGAATCCTGTGCTTGCAAATATCAATAGTTAGCTGCTGAACTGAAAAGGGGAACTCTGATGTGCGTAAGCTAACATACAGAACCTCTCTTGCAGGCCTTCTCCTGAGGAAGCACAGCTGTGGTCAGAAGCATTTGACGAGCTGCTAGCC
11672	chr18	31592572	31592823	+	0	validation	TACATCTTTTCAGTAATTCCACTCAAATGGAGACTTTTAACAAAGCAACTGTTCTCAGGGGACCTATTTTCTCCCTTAAAATTCATTATACACATCCCTGGTTGATAGCAGTGTGTCTGGAGGCAGAAACCATTCTTGCTTTGGAAACAATTACGTCTGTGTTATACTGAGTAGGGAAGCTCATTAATTGTCGACACTTACGTTCCTGATAATGGGATCAGTGTGTAATTCTTGTTTCGCTCCAGATTTCT
15945	chrX	154321271	154321522	+	0	validation	AGCATTCTGGCTGGCTGCACAGCCTGCTCAGGTGGGAGGGGGGCTGGGACCACCATCACCACCTTCCTGCCTGGGGCAGCGAGAACCCTGGAGCGTGACACGAGGCAGGAACTCTTACAAACACACCCTTCTCAGAGTAGTAGTCACCTGGCAGGTGGGCTAGAAGGAAACCACAGGAAACGGGAAGCCAGAGAGGGGATTGAAACAAGGTGTGCAAGGTGGTGGGGGAGAAAACCAGGGGGGTGGGGGGG
15119	chrX	153873736	153873987	-	0	validation	CTGCAGAGACAAAGCTTTCAGAGGCAGCCTGCTGCTCCCTCCCTCCCTCTTCTCCCTTCCTCGAGGCCGCCCTGGCCAGTGCTCTGCAGAGTCCTGGGGCTGGGACAGCAGCTCTGGCCTCTGCCCCAGAGAGCTTGGTGGCTGTCTGGGAGGAAGGAGGGAGAAGAGGAGGGCTGGCGCCAACCCCGCAGGCTATGGAAACTCAGAGGCAGCAGGGCCTTTGGGATAGTTTTCAGGGGGAGGGGGCAGCA
14047	chrX	147943036	147943287	+	0	validation	TATTCCAGTATATTTTTATCTGATGAAAAGGAGAAAGGTTTTATTAAGTAAAATGTCAAATTATTTTTACTGTTATCTTGTATATTTTAAATAGGAAGTAGACCAGTTGCGTTTGGAGAGATTACAAATTGATGAGCAGTTGCGACAGATTGGAGCTAGTTCTAGACCACCACCAAATCGTACAGATAAGGAAAAAAGCTATGTGACTGATGATGGTCAAGGAATGGGTCGAGGTAGTAGACCTTACAGAA
7419	chr11	14970146	14970397	-	0	validation	TGAAACTATTTTCAGGTTCAGCACTAACTGTAAGTTTTTGCATTTAATTTTTAATAATGGCTGCACTCAGCTTGCAAAATTCTTGAAAATTTAACCATTAGCTTTCACAAGCCTATACAAACTGGCTCCAGCACACCACTGTTTAGAGGCCACACCAGTGCCTGGGTCCTGAGGAGGACACTGGCCTTGTGCCCTGTCCCCTAGGACTCCCGCTGGCCACATCCTCAGGGGAAGAAGCAAAGACCAGGAAG
14747	chrX	149502431	149502682	-	0	validation	GCATGAGTATTTGAGAAAAAAAATGGAGAAAAAAGTATTGGAAGACAAGAGCACCATAGGGACAATAATGCTTATCATTTGGGAATGACGAATGTTTTCAGTCTCTTTCCCTTTGCTCAGTGGTTACACCGGCAGGCTTGGCAGCCTTCTGCAGGAGCTCAGCTGGCCAGCCCAGGTTGAGAGTGACATTCAGCTCTACTAAGTCAGGCCAGTTTCAGAGGTGAACCTGTCAGGAATCATATTGCACAGCC
12001	chr18	46086719	46086970	-	0	validation	TTTGGATAATTCTAAGGCATTGGTTGAGACTAATGGTATATTGACTGATTTGCTAAATGCATTTTACTTTTTATTATGGAAAATTTCAAATACATAGAAAAGTTAGGAGAATGGCATAGTGAAACCCCATGTATCTGTCACCCAGTTATTGCATAGAGTAGCATGTTTCGTTCTAGTTTCATTTTTAACCACTACTGTTCTCCCTTCTAGATAATTTTGAATCAAATAGAAATTATATCACTTCATTTGTA
8882	chr14	102083242	102083493	-	0	validation	CTCTGTGGGTGTGTTTTCTACTCAGGTAGCACTGTTACAACTGGTATTGATCTAGGCAAGATAATTAACATGAACTAGGTCATTTTCTGTCTTAGGTTCTGCCTAGGTATCTGGCTAGCAAGAAAAGTCAGAGCTAGATGAAACCATTCTTAACTGTTAAAAGGTCTAAAAGTAACTTTGTAATACCTCAGGTGAGACCAAGGACCAGGTAGCTAACTCAGCCTTTGTGGAACGTCTTCGGAAACATGGCT
16329	chrX	156003874	156004125	+	0	validation	CTGGGGCGGGGCCGCTTGGCAAGAACATCCTGGCTGCTTGGGGGTTTGGAGCAGGGCCTTGCAGCCTGTGAGTGGCCCAGTGAGTGTTCTCAGTCCCAGCCGAGTGAGATCCAGGGCTGGGGGCAGGCTTGGCCCTTGGGAGGGGAGGGCCCATATGGTTACTGCAGGGGCAGGGTTTTGGCAGGAAATAAACATGCACGGCTGCTAGTTGGGGCAGGGGCTGGCACTTGAGTCATGTGAAATGCACTTCA
15835	chrX	154316123	154316374	+	0	validation	TGGAGTGGAGTGCTCAGCTCACTGCAAGCTCCGCCTCCTGGGTTCACGACATTCTCCTGCCTCAGCCTCCCAAGTAGCTAGGACTACAGGTGCCCACCACCACACCCGGCTAATTTTTTGTATTTTTTAGTAGAGACGGGGTTTCACTGTGTTAGCCAGAGTGGTCTCAATCTCCTGACCTCGTGATCCGTCCGCCTCGGCCTCCCAAAGTGCTGGGATTACAGACATGAGCCACCGCGCCCAGCCAGTAT
6165	chr9	130700923	130701174	+	0	validation	AAATATGGAAAAGTAAGTCGGGCTCTTGATGTTCCTGTTTGCTGACTGAGACTACAAGGCTATTTTTGAATCCCCATAGCTCTCTGGAATTCTGGCCTAAAGAACCCCAGTAGCTAAGCATTAATAGAGGCTGGCATCCCACAAACTGATCGTGTTCCTTAAACGTAACATCAGGACGGTCAGGGTTCACAGGGTCATGGGTCAGTAGCCTTGTAAAGAACAAGTTTTATCCTTTTTCTCCAAGGAGACTG
8738	chr14	94563750	94564001	+	0	validation	CCCTGGGGGCCTGCTCACACAGCCGCAGCCAGATCCTTGAGGGCCTGGGCTTCAACCTCACCGAGCTGTCTGAGTCCGATGTCCATAGGGGCTTCCAGCACCTCCTGCACACTCTCAACCTCCCCGGCCATGGGCTGGAAACACGCGTGGGCAGTGCTCTGTTCCTGAGCCACAACCTGAAGTTCCTTGCAAAATTCCTGAATGACACCATGGCCGTCTATGAGGCTAAACTCTTCCACACCAACTTCTAC
11073	chr16	2131095	2131346	-	0	validation	GTAGAGACGGGGTTTCACCGTGTTAGCCAGGATGGTCTGGATCTCCTGACCTCGTGATCCTCCCGCCTCAGCCTCCCAAAGTGCTGGGATTACAGGCTTGAGCCACCGCCTGTCTTTTAAATGTCCGATGATGTCTAGGAGCTTCCCTTCCTCTCTTTTTCCTTGTGCAATTTGTTGAAGAAACTGGCTCCTGCAGCCTGGATTTCTCGCTGTGTCTTGGGGGTGCCACCTCCATGGTGTCACCTCCGTGG
5902	chr8	22163685	22163936	+	0	validation	GCACGACTCCTTTCCTTCCCACCCCACTGCCAAGCTGCTGGGCTCAGCTGAGTCCACTCACTACCTGGTGGCTTCTGACTCTAGCACAGCCCCTCTTTACTGATGAGAAAACTGAGGCTCAGAGAGATTGCCTGATATACCTGAAGTCCCACAATAAGGGCTGCACATGGGATAGAAACTCACTTCCTACATTCCAGATGGAATGCTCTCTGCAGGCCAAGCCCGCAGTGCCTACGTCTAAGCTGGGCCAG
7627	chr11	69773440	69773691	-	0	validation	CAGGGCAGGTGGTGAGAGCACCAGCTGTTGTGGGCTGGCCATGTCCCCTTCTCACCCTGTGTGGGTCTTGACACCTTAACTGCTCAGCAGAGACATCTCAGCCCAGGGTGGGGGGTGGGACAGAAGGGGGTTCTGACCCCTGGCTTCAGGCTGGGTACCTTGCCCAAGAGGTGCCCCAGCCCTGACACTGCCCTGCTTTGCTGCAGCCCTTCTTCACCGATGAGTGCACGTTCAAGGAGATTCTCCTTCCC
7219	chr11	5226751	5227002	-	0	validation	AGGAGAAGTCTGCCGTTACTGCCCTGTGGGGCAAGGTGAACGTGGATGAAGTTGGTGGTGAGGCCCTGGGCAGGTTGGTATCAAGGTTACAAGACAGGTTTAAGGAGACCAATAGAAACTGGGCATGTGGAGACAGAGAAGACTCTTGGGTTTCTGATAGGCACTGACTCTCTCTGCCTATTGGTCTATTTTCCCACCCTTAGGCTGCTGGTGGTCTACCCTTGGACCCAGAGGTTCTTTGAGTCCTTTGG
9091	chr15	50247167	50247418	-	0	validation	ATGATTTTTTTTAACATTTAAAAAAAATAATTATTATGCTTACATAATAATTACGTATATATTTATGGGGTATGTGTGAGGTTTTGATACAGGCATACAATGTGCAATGATCAAATTAGGGTAACTGGGTATCCACCACTCAACCATTTATCATTTCTTTGTGTTAGAAACATTCTAGTTCCATTCTTTAAGTTAGTTTGAAATGTACAGCAAATTATAGTTAACTATAGTTGCCTTATTGTGCCGCCTAA
10055	chr16	2095571	2095822	-	0	validation	TCCGCGGGGATTTCTGACGGCAGCTCAGACTCCGCATCCACACAGAGCGCGTGGCCCTCACCCTCCCGGCTTCCTCAACCCTTGGCCGTCCCTTGCTCGGACAGTGCTTCGGGCTGACCAGGTCGGAGGCTTGGGTTTGTCCTGGACCCCTCTGCGTCCTTCCTCACTGCAGCCTCCAGCGCGTCCCGTGGCTCCTTTCCCAACGCAGAGCACGGCCTTCCCTGCGCCTGAGCCTGCACCCTCCGTCCTGG
3219	chr4	87982344	87982595	+	0	validation	ATACAAAGTAACATGCTAGTATTATTTCAGCCAGATTTAGACAATTTTTAGTATAAGATGACCTAAAAGCTAGAGAGTGGAAAAGGATTACCATATTCCCATCCCTAGCCGTTCATATAATTATTCTTCATTTGTGCCGTGATTCAGTACCCTGATGCTACAGACGAGGACATCACCTCACACATGGAAAGCGAGGAGTTGAATGGTGCATACAAGGCCATCCCCGTTGCCCAGGACCTGAACGCGCCTTC
5882	chr7	150860720	150860971	+	0	validation	CCAACCACCCTTTGCCCAGATCTGTCCCCAGTCGCAGGAGCTGTGCCTTGCTGTGTGGACGGCAAGTTCAGAGGTCACAACAGAGCTGCTCATCTCTTTAAAAAGGGGCTGGAGAGGAATTCAGCAAGTTTCCAGGCAGAACTGAAAATGACCAAAGGCTAGAGTGGCCTCCAGTGGTCAGTACTCAGCCCTGCCCACTGAAGCCCACCCTGTCTCCTGCAGGACCTGGTGGCCTGGGTGACGGTGGGCTT
8166	chr12	14885128	14885379	-	0	validation	TTGTTTAAACCACTTGCCTTTGAGAAAATCCATTTTTATGTGAAGTATTAAGTATAGCCCTTTCTAGGGACTGGACAATCTCATGAACTTACTATGTTTGTTCAGTTAATTAATTTTAAAATAAAGTTTTACATCAAAAGAATTTTAGAAAAGAATCATTTTCATAACTCCTGTTGTCAGAAAATAAATTTTGCCTGTTTTCTATATGTCATTAAATATACCTGCATTTGTTCAAAGCTTATAAAAGGAAA
11064	chr16	2130936	2131187	-	0	validation	TCCTTGTGCAATTTGTTGAAGAAACTGGCTCCTGCAGCCTGGATTTCTCGCTGTGTCTTGGGGGTGCCACCTCCATGGTGTCACCTCCGTGGTGCTGTGAGTGTGTGCTTTGTGTTTCTTGTAAATTGGTCGTTGGAGCCGACATCCCATTGTCCCAGAGGTTGTCCTGGCTGGCACTGGCCTAGGTGTAGATGTCATCAGCTCAGGGCCCCCTGCTCTAAAGGCCACTTCTGGTGCTGGTTGCCACTCAC
7963	chr11	119091612	119091863	+	0	validation	ATGATTGTGTGAAAGCGATATACGTTCAGTAGAAACTGTACTTAGTACCCATACAGCCATTCTGTTTTTTACTTTCAGTACAGTATTCATTACATGAGATATTCACTTTATTGTAAAACAGGCTTGGTGTCAGATGATTTTGTCCAACTATAATAGGCTAATCTTAAGTGTTCTGAGCACATGTAAGGTAGGCTAGGTGTATTAAATGCATTTTCAGCTTGTTTTCAACTTAACAATGGGTTTATCAGGAT
8548	chr14	74887039	74887290	+	0	validation	GTGTGTTCCTTTTCATCTAGCAATTCCATCTCTAGGAATCTATACTGAAGAACTCCTGGTATATTTATCTCAGATTTAATATCTGATAATATAAATATTATCAGATTTATCTTAAAATGGCAAAGAGCAAGGGATTGGCTAAACTATAGCATGTTCACATGTTCACACTATGAAAAGTGGGAAAAAGCAGGTTACAAAATAATTTCATTTTCTACTGTGTGGTATGATCCCATTGTAAGTATTCTTTTTGC
10729	chr16	2119586	2119837	-	0	validation	CAGTCATACCTGGTTGTGGTCCCACAGTGGGACCACCCTGTTGGGTTCAGAACAGGAGATGGGGGCCCCTCGAGTCTGTGTGGGGGCTGTGGACAGGGTTGGGAGACCTTGGCTCTGTGGGGGACTGTGGACAGGGGATGGGGGGCCTTGGCCCTGCGTGGGATGGGTTGGGGGTCCGTGCCCTTCCTGGCCCTGGGTGGACAGGTCCATGTGGCACTCGGCATAGGGCTGAGATGGGTGCAGAGGGCTGA
15842	chrX	154316412	154316663	+	0	validation	TCAGATTATAGGTGTAGTGGCACCAGCCTGCAGTCCCAGCTACTCGGGACTCTAAGGCAGGAGGATGCTTGATCCCAGGACTTAAACGCTGCAATGTACCACTGCATTCCAGCCTGTGAAACAGCGAGACCCCATCTCAAAAAAGAAAAAGAATCAGGAGGTCTTATATAGAATACATATTTCTGGATTCTTTTGAAAAATGGGAAGAACTACTACCATAGGGTGGCATTCCCTCCTGACGCCAGTTGACT
5376	chr7	99969101	99969352	-	0	validation	ACTACAACTTCTGTCCCCCAGGTTCAAGCGATTCTCCTGCCTCAGCCTCATGAGTAGCTGGGATTACAGGCATGTGCCAGCACACCCAGCAAATTTTTGTATTTTTAGTAGAGATGAGGTCTTACCATGTTGGCCAGGCTGGTCTCAAACTCCTGACCTCAGGTGATCCTTTGGCCTCAGCCTCCCTAACTGCTGGGATTACAGGCATGAGCCACTGCGTCCAGCCTAATTTTATATTTTTGGTAGAGATG
10783	chr16	2121441	2121692	-	0	validation	GGCAGAGTGACCCCCGAGGTGCTTGAGGCCGAGGGGAGGTGGAGTTCTCGGTTTGCCCCAGCTCTCTGTCTACTCACCTCCGCATCACCAGCTCCAGGACCTGGTTTGTAACTCGGGCAGCTCTGAAAAGAGAGACATGCTGCCGCCCTGTGGTTTCTGTTGCTTTTTCTTCACTGACTACTGACATGGGATGTTTTTCCTACGGCTGTGACCAATTGTGCTTCTTCTAATTGCCTGGTTTTTCTTTTTTT
2040	chr3	49027768	49028019	-	0	validation	GCACTGCCAAAGGGATCGATAAGCAGAGACCCCATGCTTCAGATCAAGAGCCTGATGAAAGTAGTTCAAAGATGCGATGCCCTTTCTCACCATCCCTTTCCAGAAATATGAACAGGGATTCATCACAGACCCTGTGGTCCTCAGCCCCAAGGATCGCGTGCGGGATGTTTTTGAGGCCAAGGCCCGGCATGGTTTCTGCGGTATCCCAATCACAGACACAGGCCGGATGGGGAGCCGCTTGGTGGGCATCA
12180	chr18	46092956	46093207	-	0	validation	TGAAAGAGAAAGACACACTTTGTTAACAGTTGATTTTTATTTTTTATTTTTTATTTTTTTGAGATAGAGTCTTGCTCTGTCTCCAGGCTGGAGTGCAGTGGGGCTATCTCAGCTCACTGCAACCTCCTTCTCCCAGGTTCAAGTGATTCTCGTGCCTCAGCCACCCAAGTAGCTGGGATTACAGGCATGCACCACCATGCCCGGCAAACTTTTGTGTTTTTAGTAGAGATGGGGGTTTCACCATGTTGCCC
791	chr1	159307123	159307374	+	0	validation	TGGGTATACGTATTTTTCATAGGTCTTTCACATAATGGTAATGGGTAGCCAATATTGAGAATCACTTGTCTAGGTGATCTTTAAATGATTTCTGGATGTAATATTCTGAGGCTCTATAATTTGAGACTAATCACAAAAATCGGTACAGTTTATAAACAGACTAACAGAACCACAAAATAATAGAATTGGAAGGCAATTTAACTAGTGCAATTTCTTCATTTTGCCTAACAGGCATGTAAGAAATGATGATT
13350	chrX	147920723	147920974	+	0	validation	ATGCTATGAGAATGCAGAGGACAGGAATTTAACCTAGACTTCTCAGTTCTCCCGTTGAAGTTGACATCTGAACTGAAATCTGGAAGACCAGTAAGAGATAGCTATGTAAAAAGAGGGGAAGGACAATAGGAAAGAAGGAATGGAGAGAGGCCCAGAAACTACAGAGTATGGCACAAGTAGTTTAGCATTGTTGGGTCACAAATTCTAAGGATATGAAAGATAAAGTTGAAGAAGTGGTGGGGCTGGGGAGG
629	chr1	119513667	119513918	+	0	validation	TGGCTGTAGTACGACCAAATCTCAGACAGAACCACAGAAGAATGTACCCTGAGTCTGTTACAACCACCATATTTGGGAGTGGGGGGTGGGGCACATAGATCTGTGTTCGTGGTTGGCACCTCTTAGGGATATATCCTGACAGTGACAATATGCTCTTCATGGACAGGTACCCAGCTCCTGTTAGAGGCCTGTGTCCAAGCTAGTGTGCCAGTCTTCATCTACACCAGTAGCATAGAGGTAGCCGGGCCCAA
2675	chr4	71766367	71766618	-	0	validation	TTAGAAATTTTACCTTAAAAAGATGACTAACTTATAAATTAAGTATAAGAATGTCAAAGTACTGTTTCCCATATAAATCCCAAAGTAATTTTTTCCATGATGACTATTTCTATTCTTTTTTCCTCTATGGTGACTCTATTTTAGATTTTAATTTCAAATATGAAATTTTTCCAAAAAGAACATTTTGATGCTACGATTGTTTTGATGGTGCTTAGTCTGCTCACAGGTATCAGTGTATCATTGTTGAAGCA
12645	chr21	39347834	39348085	-	0	validation	GTTAGGAGACTTAGGACGACAAAAAATATTAAATATTAAGTCTACAAAGGAAATTTATTCTTTGCGTTGCGTACATTGTGGCTGGTGCTTGGCTTTTAATTCGTGCTTGTACTTCCTTTTTTGACAATAAAAGAGTCAAGATAGCACCGAGGCCAGGAGAAAGGGAACGTGTAAGTTTTTATATATACAGTTTCCAAGCCAACTTCGGGAAGCCTTAACCTTTTTACGGGGTGGGGGTGGGGAGGTAAAAA
9062	chr15	50246417	50246668	-	0	validation	TCCTATTAGCAACCCCATTGCCTACCATATGAGGCCCACACTCCTGGGCCTGGCTTTCAAGGACTTGACGATTAGCTCCTACCTTAGCTTGGGCATTTCATCGACCCATCCAGACCTCATGCTCAGTCCATCTCTGCACCTTCACCCGTGTCCTTCACCTGAACTACCCTCTCCTCTCCAAATGTTGTGAATGGATTAAATTCTGCTTCCACTAGAAGCTTTTCTGCCTATTCCATCTGTTCTGGTTTCTG
14598	chrX	149495850	149496101	-	0	validation	TTGGTGACTTGTTGTGTGTGGACACAGAGGTGGGGAACCAGCCTGGGTGACCGTTCCTAACCCAGGGGATACTGTTCAGGTGACAGCTCATTAATTGTAGAGTGATGGGCAACTCCCGAGCAGCCCAGAATGCTTGGTTTGCTGGTACTTGCAGGTTCCCTCTGTGGGAAAGCCCATCCTGGCTAGCTGTCACTCTCAGGGGGTGGCTTGGTGAACAGGCCCTTGGTGCAGAGGGAACTGGGACACCATCA
5175	chr7	45917236	45917487	-	0	validation	ATAGATATTGATGACTCTCATGTGTTTGTCTCTCTTGGGCGTTTTAAGGAAATGCTAGTGAGTCGGAGGAAGACCGCAGCGCCGGCAGTGTGGAGAGCCCGTCCGTCTCCAGCACGCACCGGGTGTCTGATCCCAAGTTCCACCCCCTCCATTCAAAGATAATCATCATCAAGAAAGGGCATGCTAAAGACAGCCAGCGCTACAAAGTTGACTACGAGTCTCAGAGCACAGATACCCAGAACTTCTCCTCC
1437	chr2	3579773	3580024	+	0	validation	ATAATACAAGTGAAAATAAAGATGTATAAAAGTCCTCTAATCTCACATTCCCTAGAGATGGTCACTCATTGCCTTGGTGAGTTTGTAGTTTATCCCAGAGTAACATTTAAAATTGACTGAGCTTTTGAAATAAATCAGTAAGCTATTACCCTATTCTAGGTAGGCAAAGGCTTTAGTTTGGTACGTGAAATATAACAAGCTTATTTGAAAATGAGTGTTAATTTGACTTCAGGAACCTGGGATTTGCATTT
9903	chr16	2091101	2091352	-	0	validation	CGGCAGCGTCTCACCCCTCGCAGCGCCCCGCCCCCTCGCAGCGTCCCGCCCCCTCGCAGGGCCCCGCCCCGGCAGCGTCCCGCCCCCTCGTAGGGCCCCGCCCCGGCAGCGTCCCGCCCCCTCGCAGGGCCCCGCCCCGGCAGCGTCCCTCCCGCCCTCCTGACCGCGCCCCCCACAGGTGTGCCTGCTGCTGTTCGCCGTGCACTTCGCCGTGGCCGAGGCCCGTACTTGGCACAGGGAAGGGCGCTGGC
11198	chr16	2134759	2135010	-	0	validation	GGGGTGAACGGTGTGAGCAAAGACCGTGAGGCTGGAGGCTGGCCACGGGAGGTGTGAGGGGTAGGGGCAGGGTGGGAGGTGGGCTCGCGGGTGGGCTGGGGTCATGAAGGGCCTCAGGCGCTCTGCTATTGGGTTCCAAGGCTATCCTGAGAACAGGGGTGAGGGGGGATTGCCGTGGGGGGTTAAAGCCTTGTCATGTTCGCTTTCGGGAGATAAAAACAACAGGTGGCCTTTATGGAGACGCTGCCCAG
1431	chr2	3579656	3579907	+	0	validation	GCTAAGTAAATGATGTGATCATATCCGTGCTTTAGAAAGATTATCTTGACATTGGAAGGGGGTGGTTTTGGAGATGAGCAAGAGTGTTAAAGGTTATTGTAGTGATTTAGTTGAGAAATAATACAAGTGAAAATAAAGATGTATAAAAGTCCTCTAATCTCACATTCCCTAGAGATGGTCACTCATTGCCTTGGTGAGTTTGTAGTTTATCCCAGAGTAACATTTAAAATTGACTGAGCTTTTGAAATAAA
6908	chr11	5225931	5226182	-	0	validation	TATTTTTAATTGATACATAATCATTATACATATTTATGGGTTAAAGTGTAATGTTTTAATATGTGTACACATATTGACCAAATCAGGGTAATTTTGCATTTGTAATTTTAAAAAATGCTTTCTTCTTTTAATATACTTTTTTGTTTATCTTATTTCTAATACTTTCCCTAATCTCTTTCTTTCAGGGCAATAATGATACAATGTATCATGCCTCTTTGCACCATTCTAAAGAATAACAGTGATAATTTCTG
8470	chr14	24632250	24632501	-	0	validation	CCCTGTGACTCCACCCCCATCCTCACTCTGCTCTCTGTGCAGCTCCATAAATGTCACCTTGGGGGCCCACAATATCAAAGAACAGGAGCCGACCCAGCAGTTTATCCCTGTGAAAAGACCCATCCCCCATCCAGCCTATAATCCTAAGAACTTCTCCAACGACATCATGCTACTGCAGGTGAGGCACACTCCTGCCACTCTTGCTCTTCTTGGTCCAGTTGGTTCCACTCCCCCTGGAATGCCGGCCCTTC
11695	chr18	31593063	31593314	+	0	validation	TATCTTCCCGTTTGCCCCTCACTTGGTAGAGAGAGGCTCACATCATCTGCTAAAGAATTTACAAGTAGATTGAAAAACGTAGGCAGAGGTCAAGTATGCCCTCTGAAGGATGCCCTCTTTTTGTTTTGCTTAGCTAGGAAGTGACCAGGAACCTGAGCATCATTTAGGGGCAGACAGTAGAGAAAAGAAGGAATCAGAACTCCTCTCCTCTAGCTGTGGTTTGCAACCCTTTTGGGTCACAGAACACTTTA
12636	chr21	39347615	39347866	-	0	validation	CTTTTTACGGGGTGGGGGTGGGGAGGTAAAAAGTTGTGATCTCTGAGAAAATAACCGCCACTACTCTGGAAGTGTTCATCAGCAGTTATACAAAACCGTGATTTTGGCTGCTCCCTAACAAATTCGTGATTGCATGATTCGAATTGCAGGTCTGTAGAATGAAGTTGGCTTTGGGTGTACGTGTGTTTGATAACTTGCAGGGTGGAAAAGCAGAACATGTGTAAAACAAGTATAAGCTTTGTGTTTGGATA
10959	chr16	2127671	2127922	-	0	validation	CCTCCCACCTCTCCCTCCCTGGCTCATCCCTGCTGTGTCCCTTCTCTCTAGTTTCCTGTTCAGTTTCAGGAAGGAGGCTGGGAACCCAGATGTAGGGAATTTGCGCCCTGGAGTCAGACCTGGGTTCACGTCCCAGCGCCTCCACCTCTGGTGTGACCTTGGTCCAGTCTCTCAGCCTCAGTTTCCTCACCTGTAAAGTGGGCTCCATGATTAGATGCACCCTGCAGGGCAGTGTAGCAGTGACCTGGCTC
6898	chr11	5225915	5226166	-	0	validation	ATAATCATTATACATATTTATGGGTTAAAGTGTAATGTTTTAATATGTGTACACATATTGACCAAATCAGGGTAATTTTGCATTTGTAATTTTAAAAAATGCTTTCTTCTTTTAATATACTTTTTTGTTTATCTTATTTCTAATACTTTCCCTAATCTCTTTCTTTCAGGGCAATAATGATACAATGTATCATGCCTCTTTGCACCATTCTAAAGAATAACAGTGATAATTTCTGGGTTAAGGCAATAGCA
12682	chr22	19964049	19964300	+	0	validation	GTGTGTAAAGATGGCCTCCTGTCTGTGTGGGCGTGGGCACTGACAGGCGCTGTTGTATAGGTGTGTAGGGATGGCCTCCTGTCTGTGAGGACGTGGGCACTGACAGGCGCTGTTCCAGGTCACCCTTGTGGTTGGAGCGTCCCAGGACATCATCCCCCAGCTGAAGAAGAAGTATGATGTGGACACACTGGACATGGTCTTCCTCGACCACTGGAAGGACCGGTACCTGCCGGACACGCTTCTCTTGGAGG
1915	chr2	162145969	162146220	-	0	validation	CCTCCACGGTGCTCTGCACACACTTGATGTTGATATGGGGTCATCTTCTTTGCTTTCCTTGTTACTAGCAGATCACACATTTTCTTTAATCCATTATCAAGTGGTCCTTTTGGTAACAATTCAAGATGGTCCTTTGCTTTCCCTCAAATCAATTAACTAACTCTGACCCCATACTTACTAATAACAAACAGCTGCCTTGCTTGCAAGTCACATAGTGCAGTGGATAGCCTGTATCCATTGAGACTGAAGTC
6541	chr10	47348995	47349246	+	0	validation	ACTGCACAGGAGGCCAGGTCTCTGGCATTCCCTACATCATCTCCTACCTGCACCCAGGGAACACCATCCTGCACGTGGACACTATCTACAACCGCCCCTCCAACACCACCACGGAGATCTGGACCTTGCCCCAGGTCCTGGGAGAAAGGTACGGTGCCGACAAGGATGTGGTGGTCCTCACCAGCAGCCAGACCAGGGGCGTGGCCGAGGACATCGCGCACATCCTTAAGCAGATGCGCAGGGCCATCGTG
4164	chr6	30490333	30490584	+	0	validation	TTATCTCACCCTGAATGAGGACCTGCGCTCCTGGACCGCGGTGGACACGGCGGCTCAGATCTCCGAGCAAAAGTCAAATGATGCCTCTGAGGCGGAGCACCAGAGAGCCTACCTGGAAGACACATGCGTGGAGTGGCTCCACAAATACCTGGAGAAGGGGAAGGAGACGCTGCTTCACCTGGGTAAGAGGGTCCACAGGGCTACTCTCCCATCTCCTTCTTGGGCTAGGACTGTGCCCACAGCTGACAGAC
9977	chr16	2093417	2093668	-	0	validation	TAGAGAGCCCGGCTGTGACGCCTGTGAGCGCACGTGTGCCCCGCGTACGGCCACCCCACGGCTTTGCACTCTTCCTGGCCAAGGAAGAAGCCCGCAAGGTCAAGAGGCTACATGGCATGCTGCGGGTGAGCCTGGGTGCGGCCTGTGCCCCTGCCACCTCCGTCTCTTGTCTCCCACCTCCCACCCATGCACGCAGGACACTCCTGTCCCCCTTTCCTCACCTCAGAAGGCCCTTAGGGGTTCAATGCTCT
635	chr1	119513772	119514023	+	0	validation	TTCGTGGTTGGCACCTCTTAGGGATATATCCTGACAGTGACAATATGCTCTTCATGGACAGGTACCCAGCTCCTGTTAGAGGCCTGTGTCCAAGCTAGTGTGCCAGTCTTCATCTACACCAGTAGCATAGAGGTAGCCGGGCCCAACTCCTACAAGGAAATCATCCAGAATGGCCATGAAGAAGAGCCTCTGGAAAACACATGGCCCGCTCCATACCCACACAGCAAAAAGCTTGCTGAGAAGGCTGTACT
2336	chr4	71751728	71751979	-	0	validation	ACTTTGTTTTAATATGACATAAGAAAAAAATGCAACTGAAGAATTTCTAAAGGTCACATAAATGTTTCATTACTATAAGAGTTATTTTCTAAAAACTAAAAAATGATTTTAAAGAATATTAAAATTTTAGTCATTTTTTATTACGAATTTCTTTTTTAGACACTATCAATTGTCTCCAAACTCTGAAAGCTAAAGATATTAGAAGTCCTATATATCTTCCAGTTTCCCTCCCTACTCCCCTCAATTTTGGA
13121	chrX	147913428	147913679	+	0	validation	AATAGAAGGTGCGTGTGGAAGGAAGGCTGGAAAATAGCTATTAGCAGTGTCCAACACAATTCTTAAATGTATTGTAGAATGGCTTGAATGTTTCAGACAGGACACGTTTGGCTATAGGAAAATAAACAATTGACTTTATTCTGTGTTTACCAATTTTATGAAGACATTTGGAGATCAGTATATTTCATAAATGAGTAAAGTATGTAAACTGTTCCATACTTTGAGCACAAAGATAAAGCCTTTTGCTGTAA
16023	chrX	154323776	154324027	+	0	validation	ACAAACGAACCCGGGTTCCCTGGCCCTGGCTTGCCTTTGGAACAGCTTAGGGCGCCAAGAGTGTGTGGGACCTCATGCAAGGGCATCAGATGGACAGAATCAAATTCCTGTTCAGAACCTAGCTGCACGGGAGACGAGGCAGTGTGGTCTTGATCTTTCCAGCCTTTCCCACTAACGTTCAGCAGGTTTCCCTTGCCACCCTGGACCTCGCCCATGCGCCACACAGGTAACCCCTGGCACAGTCGGGATGG
15834	chrX	154316077	154316328	+	0	validation	TTTAAAAAAAATTTTTTTAGACGAAGTCTCGCTCTGTCGCCCAGGCTGGAGTGGAGTGCTCAGCTCACTGCAAGCTCCGCCTCCTGGGTTCACGACATTCTCCTGCCTCAGCCTCCCAAGTAGCTAGGACTACAGGTGCCCACCACCACACCCGGCTAATTTTTTGTATTTTTTAGTAGAGACGGGGTTTCACTGTGTTAGCCAGAGTGGTCTCAATCTCCTGACCTCGTGATCCGTCCGCCTCGGCCTCC
1357	chr1	225836823	225837074	+	0	validation	GTTTGAGTCCTGCCAGCCGAATACATCCTAGAGATGGGCTGTACAGCAGAGCGCCGTCAGTGTTGGACACTTACAATCTGTTAAAAGGGTAGGACCCATATTAAGTGTTCTCGCCATATAATAAAAAGTGTAAACACTAAGGAAATCAATGGACTTTATAGTTAATAATATCTCAGTCTGGTTCGTTAATTGTAATAAATGTACCATGCTAACATAAAATATTAATAATGGAGGAAACCCGGTGGGTGTGA
14954	chrX	153866198	153866449	-	0	validation	CTGCTTCTTTCATTTCCTTGAATCTGGAACATGACCAGATGGTAACTTGAAGTACCCGGTCAAGTTTTTTTTCTGGTTTCTCCTTTTTTTCTTTTCTTTTTTCTTTTCTTTTCTTTTCTTTCTTTTTTTTTTTTTTTGAGATGGAGTCTTGCTCTGTCATCCAGGCTGGAGTGCAGTGGAGCGGTCTTGGCTCACTGCAACCTCTGCCTCCCGGGTTCAAGCGATTCTCCTGCCTCAGCCTCCTGATTAGC
13801	chrX	147935245	147935496	+	0	validation	TTAATGGTGTCAGATCTGTTCTATGACCAGTCAAAGAGTTTCTAGGGGTGAATGAGGCAGCCACACACAGCAAGAATGTCAAAGCTATGCTCTGATTACAAGAGCCACAAGAATTTTTTGTGAGAAGGCTTTTCACTGAGTTCTTATCTAGCATTGGCTCAAAGCAAATTTTTGCTTCCATCTTTTCCCCCAACTTTGCCTTGAGCTTTGCCTTTAAAGGCGAAAAGGAAAAAATGGAAGATGAGATGGTG
14899	chrX	153864691	153864942	-	0	validation	TCGGACACACAACCTGACCGATCTCAGCCCCCACCTGCGGTACCGCTTCCAGCTTCAGGCCACCACCAAAGAGGGCCCTGGTGAAGCCATCGTACGGGAAGGAGGCACTATGGCCTTGTCTGGTAAGCTGGAGGAATGACCTGCCAACAGGGAGGCCCCGGCCAGCCGGGTCCAGGAGGGAGGGCCTTAGACTTCCTGGCAGCTGCCACACTCTCCTCGTTCCCCTGCATCCCCCCAGGGATCTCAGATTT
4540	chr6	32189066	32189317	-	0	validation	CCTCCACACCCTCTCCAGAGCCCCTAAAGCTCCTCTCCACTGCTCAGCCAGACACTAGGTGCATCAAAGCCTCCCACCTGCTCAGCCCCAGGACCCCTTCACACACCCTACACTGATCTCCCCAGTTAGCTCGGCACCCCCAGCCCCACTCTGCCACCTCAAACTCTGACTCTTCTCAACCCCAGCCTCTGTCTCTCTCCCTCTGAAACCTACCAAGTCACTTTCCTTTCTCCATCCACTCCCAGATTCCT
5705	chr7	101128888	101129139	+	0	validation	GTTAAGTTTGCCCATCTACAAAATGAGGATTATTTGCTGTCCTAAAGAATTCATGAGCCGGGCGCGGTGGCTCAAACGCCTGTAATCCCAGCACTTTGGGAGGCCAAGGCGGGCGGATCATGAGGTCAGGAGATCAAGACCATCCTGGCTAACACAGTGAAACTCCATCTCTACTAAAAATACAAAAAAAATTAGCCAGGCGTGGTGGCAGGCGCCTGTAGTCCCAGCTACTCGGGAGGCTGAGGCAGGAG
14766	chrX	149502838	149503089	-	0	validation	GAGGGTGTGCTGTTGGGTCTTTCAGCTCCTCGGGTGGTAGGGAGGACACAGGCTGGGGAGGTGGCAGTGTTGGGTGGAGTCCAGCTCAGGGCCCCACCCTCTCCCCTGCAGGGTACCTGTCAGTAAACCTAGGGGTGGGGTGAGGACACCTGAGGGCCTCCCGTGTGGCCAGCATTGCTGTTGCTGACTTATTGCCCAATGAGGGGGCTGTGCTTAGGTAGGGGCTGCTATCCACTCTGGAAATTAAGTCA
3196	chr4	87981195	87981446	+	0	validation	ATTGACATGAAGATTATACTCACATATTTGGCTTGAAAATATCTATAAAAATAATTTCTGTTGCAAAGTAAGAAATGTTCTTCAGAATGTTATTAATCCCTGTGTTAAAAGAGAAATTGGAAGATGCTCACTTTAGCTCCTAAAAGCCATGGTATGTACTGTGAATGCAAAGATTCTGAAACTAAATAAAAAGAAAGATAGTAAAAGACTAATGTGCTATAAAGGCTAAGGGAAAATAAAAACCCATATAT
1222	chr1	192809556	192809807	+	0	validation	AATATACTCGTTTAAAATCCATTTTTGTTTTTTAATTATAGAGCAGATCTCACCCAGTCCGAATGTGGAACAAATAATTGTTATGCAGCGTCTGCTTAAAAGAAGTGTCGTAGGTGGGGAAGGAAGAGCGCAGGGGAATCAGTCACCCACCTCTTTGTACAGTCTCTGGCGTGGTCCAGAACCTCCTGCTCTAAAGAGAGAAGCGTGGGCCGGCTCCAGACAGTTCCATGTCTGTCCTTTTCATTAAAGTG
10335	chr16	2105744	2105995	-	0	validation	ACGGCAGCGCAACGGGGCTCACAGTCTGGCTGCACGGGCTCACCGCTAGTGTGCTCCCAGGGCTGCTGCGGCAGGCCGATCCCCAGCACGTCATCGAGTACTCGTTGGCCCTGGTCACCGTGCTGAACGAGGTGAGTGCAGCCTGGGAGGGGACGTCACATCTGCTGCATGCGTGCTTGGGACCAAGACCTGTACCCCTGCCTGGAGCTTTGCAGAGGGCTCATCCCGGGCCCCAGAGATAAATCCCAGTG
13649	chrX	147930968	147931219	+	0	validation	TCAAGACTGTGTTCTTATAAAAACCCTGTGAAATTTGCCACCTGTCAACTTGATGTATGATGCTGAACTCTTAGCAGAAGGGGAAAAGTTTAATCTCCTTTTGCCACTTTGTTTTTTCTTAAGGCCATGGAGGTCTGTTTAATTGGTCTTTGCTCTCATTCTTTCTTGATGTCCCTGTGTTCATGATTTTGTGGTTCCTTCAGTGATTCTCTCATAACTTTTAGCCCAGAGCTGCCTCCTTACTTTGATTG
12364	chr19	41878233	41878484	+	0	validation	TAGGTTGACACATACAACTTACAGAGAATGAGGGCCAGGCACAGGGTCACAGGCCAGGGCAGGCCACAGACTGGCTCCTCAGCCCAGGCAGGGAGAGGCCAGGGAGCCAAGAGTTTGAACCCAGTGCCACTCCTGACTGCCTGGTGATGCTGGCAACCCGCCTGCCCTCCCAGAGCCTCAGCCATCCCTCCTGTAAAATGGGGCTAAGGAGAGAACCTACTTCTAGGGTTCTGTGAATGATTACACAAGAA
13584	chrX	147928827	147929078	+	0	validation	TGCTGAGAACTTGGAAGTGATATGCAATTAGTTTAGAAGAATTTCTAGTAGTTTAAAATTTTTTAAACTACTAGAAATTGATTTTAAATTTGGTGGACTTTGGCAAAAATGGTTTGGTTTGCAAGAGCAATGGAGAAAAAACTTATTTAACTTTTCCTACTTATAGAATCAAGGCAGCCTTGCGCTTGTTTAAACTTACTTGGCTATACACGTGTCTGATGTACATAATGTACATTTGTTTTCTAAACTAA
9096	chr15	50247382	50247633	-	0	validation	AGCGGGACTTAACTAAGAACTCTGGCCACTGGCATTGTTCTCCCACTTCTACAAGAGTCCACATGCAAGACAAAGCCATTGATAAAAGTAGTGGGGGAAGCTGAAGACACTGTCCTTTCTGCCTATGTGTGTCTTTGTCCTTCCACCTTTCCTCTAGGGAATTCCTTCTTGGTTAAACAAACACAAACCAAAATGAAGTGGTGGAAGCGGTAACTATGATTTTTTTTAACATTTAAAAAAAATAATTATTA
9396	chr15	50258285	50258536	-	0	validation	GTGCATGCTCTGCCTCCAGGTGGTACATTGGCAGAGCCCCCATATGCACGCCTACTACCCAGCCCTCACCTCTTGGCCCTCCCTGCTAGGAGACATGCTGGCTGATGCCATCAACTGCTTGGGATTCACCTGGGTGAGTAGCAACGGCTGTAACTACTCGCAATGGGGAACGTAGCAGGGAGTGGTATCCTAGGGCAGACATATTTTGTCTTGGTTTCTGAATATGGCTCCACAATAAATATTAATGAGGG
1329	chr1	225835468	225835719	+	0	validation	AAGCTCCGCCTCCCAGGTTCACGCCATTCTCCTGCCTCAGCCTCCCGAGTAGCTGGGACTACAGGCGCCCGCCACCATGCCCAGCTAATTTTTTGTATTTTTAGTAGAGACGGGGTTTCACCGTGTTAGCCAGGATGATCTTGATCGCCTGACCTCGTGATCCACCTGCCTCAGCCTCCCAAAGTGCTGGGATTACAGGTATGAGCCACGGCACCCGGCCTTTGTTTTTTCTTTCCTTTTTTTTTTTTTTT
2734	chr4	71768843	71769094	-	0	validation	TCAAAGATAATACGTAAAGCATAGCTTTCTAAGGTCATTAAAATGCAATTGTACATTCTAGGAATCTGGGAAACCAAGGCAATGAGGTACAGAGCAAGTACATTTGGGGAGGCTTAAGAAGGACAACACCAGATTGGTTTGATGGAAGCCCTAAGTTCAATTTCTACTGCTTTCCAAGATAAATTTGCCTAGAACAAAAACTGCTTTTTAAATTAAACCTCATTTTTTTCATTAGCCAAATAAAGAAGTAA
9879	chr16	2090165	2090416	-	0	validation	ATTCTCCGCTGGCGCTACCACGCCTTGCGTGGAGAGCTGTACCGGCCGGCCTGGGAGCCCCAGGACTACGAGATGGTGGAGTTGTTCCTGCGCAGGCTGCGCCTCTGGATGGGCCTCAGCAAGGTCAAGGAGGTGGGTACGGCCCAGTGGGGGGGAGAGGGACACGCCCTGGGCTCTGCCCAGGGTGCAGCCGGACTGACTGAGCCCCTGTGCCGCCCCCAGTTCCGCCACAAAGTCCGCTTTGAAGGGAT
10957	chr16	2127610	2127861	-	0	validation	AGTTTCAGGAAGGAGGCTGGGAACCCAGATGTAGGGAATTTGCGCCCTGGAGTCAGACCTGGGTTCACGTCCCAGCGCCTCCACCTCTGGTGTGACCTTGGTCCAGTCTCTCAGCCTCAGTTTCCTCACCTGTAAAGTGGGCTCCATGATTAGATGCACCCTGCAGGGCAGTGTAGCAGTGACCTGGCTCAGCCACTGGCAGCCCCAACAATCATACCTTGTTAAAGTAGCTCTGTCGGTTCCCTCAGGGG
4284	chr6	32039371	32039622	+	0	validation	CCGGCACCCCTGTGGCCATTGAGGAGGAATTCTCTCTCCTCACCTGCAGCATCATCTGTTACCTCACCTTCGGAGACAAGATCAAGGTGCCTCACAGCCCCTCAGGCCCACCCCCAGCCCCTCCCTGAGCCTCTCCTTGTCCTGAACTGAAAGTACTCCCTCCTTTTCTGGCAGGACGACAACTTAATGCCTGCCTATTACAAATGTATCCAGGAGGTGTTAAAAACCTGGAGCCACTGGTCCATCCAAAT
10012	chr16	2094299	2094550	-	0	validation	AATCATAGTTTGCAGGGTTGAAGGGGGGCTCATTGCACCCTGAGAGACTGTGCACTGCTGTAAGGGCAGCTGGTCAGGCTGTGGGCGATGGGTTTATCAGCAGCAAGCGGGCGGGAGAGGGACGCAGGCGGACGCCTGACTTCGGTGCCTGGAGTGGCTCTTGGTTCCCTGGCTCCCAGCACCACTCCCACTCTCGTTTGGGGTAGGGTCTTCCGGCTTTTTGTCGGGGGGACCCTGTGACCCAAGAGGCT
3750	chr5	132542458	132542709	-	0	validation	TTAAAATATATAACAAATGCCCTATATTAATAATTTCTGCATACTTAAATAATTATGACTATATGATGGTGTTGTATGCATTTGAATATGTCCTGGTCATATTAAAATGTAAAATATATAGTTTTATTAGTCTAAATAGAATAAAACTACCAGCTAGAACTGTAGAAACACATTGATATGAGTTTAATGTATAATGCATTACACTTCCAAAACATTTTTTTCCAGTTACATAATTAAGTTATATCCTTTAT
10100	chr16	2096996	2097247	-	0	validation	CTCTAGATGAAGACCTGATCCAGCAGGTCCTTGCCGAGGGGGTCAGCAGCCCAGCCCCTACCCAAGACACCCACATGGAAACGGACCTGCTCAGCAGCCTGTGAGTGTCCGGCTCTCGGGGGAGGGGGGATTGCCAGAGGAGGGGCCGGGACTCAGGCCAGGCAGCCGTGGTTCCCGCCTGGGGTAGGGTGGGGTGGGGTGCCAGGGCAGGGCTGTGGCTGCACCACTTCACTTCTCTGAACCTCTGTTGT
15885	chrX	154318968	154319219	+	0	validation	TATAGGCTTCTGAGCATTAGGTTTTTTTTTCTGGACTGAGATAGAATTTTGATTATTATTAGCTGGTATGTTAGTGGATGAGTATACTTGTAGTTTGAATGTGATAGCATTGAGAACAGATTCTATTCCTGTTTTTCATCTTCATGATGTGCTATGAAACCTTACTTACATGAATAAGTAGATGGGAAGGGAAGTTTTGTTAGAGGAATGTCACTCAGTTCTTTGAATTCCTTCCCAGTTGTAAGTCAAAA
1746	chr2	96144385	96144636	-	0	validation	GTGTGTGTTGGGGGAGGGGGTGGGGCATGGCTGCTCGCGCGTGCCTGTATGGGTCTGTGTATGTTTGCATGTGTATGTTGGGAGCATGAAGGGAAATGTATGTCCCGGTGTGTCCTCCGCACATTCCTGAGACCTGTCTCAGGTCAGGAGGACTGGCTGAGGAGTCCTCTTGTCTCGGCCAGCCCATGGGGTCTCCACCCGCTGCTGCTCCAGCTCTCCGGGGCTGGGCTGGAAAGGCCTCACCGCCCCTC
13325	chrX	147919549	147919800	+	0	validation	TGATAAGAAAATGTTGATTTCTCCCCTTTTGAACCAGTAACTAACTATAAGGGTATATACCCATGCTTAACTTAAAAATAATTATTTAGCCACCTTGGGTATAGCACAACATATGGATGCCATTATAGTCCACCTTGATCTTACAAGGAAGCTTTCTTTTAGCGTAGCTTTACTTTTATTTAAGCATTATTGAAGAAGCTTGGTATCTCTGTTTAAGTTGCTTCTGTATCAGTGTTTTCCAGGGCCCTGTC
5348	chr7	99968363	99968614	-	0	validation	TAGCCCCTCTCTGTCTGTTGTATATATTGGAGTAATAACCTATTTGTCTTGATAAAGGGATTGCATGCTTGAATTGCAAAAACCTTTATTTCTTTTGGGTTGCCCAATGTGCAAGACTAAGAGTTATTTTGATAAATTTCTCACCAGGCTGACTGTCTCTCTGTGGGGTCGGGGGAGTTTTCAGGGTCTCACGTATTGCAGGGAAGGTTTGGTTGTGAGATCGAGAATAACAGAAGCAGCGGAGCATTCTG
6034	chr9	130696749	130697000	+	0	validation	GGTAGAGCCAACAGGTTTTGCTACTGGACTTGTCCTGGAGCGTGAGAGAGAGAGAGGAATCAAGGAACTCTCCGAGGTTTTTGGCCTGAGCAACTGAGGGGATGGAGTCACCGTTGACTGAGATGGGGACAACTCATCCCCTATCTCGGTTTGCTATGGAGGACTGGAGGGGGCTGCGGGAAAGCTTAGTTTTGTGCTTGATACATTTGAAAATGCCTGTTAGTCATCTAGATGGCAGAGTTGGGTAGCTA
12773	chr22	20781663	20781914	+	0	validation	GTGGGTGGAATCAAAGGCTGAGTTCTAACAGGCTTGCGGCAGACACACACACAGAGACCACATGTACATGATGAACACACATATCCTTTTCATTACAGGTTATTAGTACAAGTTTTGGAATTGAGCAAACAAGAGTCTAAGCGCTGGTTTCACCACTTCTCGTTTGTGTGACCTCAGACAAGTCATTCAACATCTCTATGACTCAGTTTCCTTATCTTTATCACAGAGATGACACCCACTCTGACAGGGCC
5052	chr6	37173389	37173640	+	0	validation	GAGCATATTAAGAAGAAAAGACAATCTGGCTTCTCCAAAAACTTTTTTAAAGGTACCAACAGAAACCTGATAATTCCTGGCTGTTTTGCCAGGGAGTAAAAAGTTAAAAGCTCTTTTAGCATCTTCTTTAAGGCAGCAGCTCCAAATATTTTGGTACCAGTGACCTCACTGTGGGTGGTGTTCGTGTTTGTAAGTTGGTAGGTGAATTGAATCATTTCATCATGCTCAGTGGTGTCTCATCAAAATCTCTT
7131	chr11	5226564	5226815	-	0	validation	ATTTTCCCACCCTTAGGCTGCTGGTGGTCTACCCTTGGACCCAGAGGTTCTTTGAGTCCTTTGGGGATCTGTCCACTCCTGATGCTGTTATGGGCAACCCTAAGGTGAAGGCTCATGGCAAGAAAGTGCTCGGTGCCTTTAGTGATGGCCTGGCTCACCTGGACAACCTCAAGGGCACCTTTGCCACACTGAGTGAGCTGCACTGTGACAAGCTGCACGTGGATCCTGAGAACTTCAGGGTGAGTCTATGG
13636	chrX	147930422	147930673	+	0	validation	TCAAAATTAAACAGCAGCTTATGTATATTAAGGGACTTCTGGTACTTTCACTTTTACTAGAAGTTTATCAAAGTTGCTAATAAAATGCGGCAATGCTTCTTCCATATAACATGTTGTCATTAAAAATACTAGAGAATATTTAAATATCTAATCATATTCCTCTCATTATGTATGTTTATCTTTGTTTAAAAGGGTATAGAACTTCTTCATTCTAATTGGTTGTCATTCTTTAAAACTCCTGTCTTCAGATT
11763	chr18	31594827	31595078	+	0	validation	CACTCCAGCCTGGGTGACAAGAGTAAAACTCTGTCTCAAAAAAAAAAAATTATACCTACATTCTCTTCTTATCAGAGAAAAAAATCTACAGTGAGCTTTTCAAAAAGTTTTTACAAACTTTTTGCCATTTAATTTCAGTTAGGAGTTTTCCCTACTTCTGACTTAGTTGAGGGGAAATGTTCATAACATGTTTATAACATGTTTATGTGTGTTAGTTGGTGGGGGTGTATTACTTTGCCATGCCATTTGTT
6602	chr10	47350957	47351208	+	0	validation	CCTTGCCCCAGGTCGCCGGCCAGCGCTACGGCTCACACAAGGACCTCTACATCCTGATGAGCCACACCAGTGGCTCTGCGGCCGAGGCCTTTGCACACACCATGCAGGACCTGCAGCGGGCCACGGTCATTGGGGAGCCCACGGCCGGAGGCGCACTCTCTGTGGGCATCTACCAGGTGGGCAGCAGCCCCTTATATGCATCCATGCCCACCCAGATGGCCATGAGTGCCACCACAGGCAAGGCCTGGGAC
9926	chr16	2091923	2092174	-	0	validation	CCTCCCGGCCCCAGGGTCCACACGTGCTCGGCCGCAGGAGGCTTCAGCACCAGCGATTACGACGTTGGCTGGGAGAGTCCTCACAATGGCTCGGGGACGTGGGCCTATTCAGCGCCGGATCTGCTGGGGTGAGCAGAGCGAGGGCCCCGGGCGTCTACGCCAAGGACAAGGGAGTAGTTCTCCAGGAGTGCCGCGGCCTCCTGACCAGCCTGGCTCCGGGGTGCCGGAAGGGCTGGGGTGCGGCACCCACG
16218	chrX	155998098	155998349	+	0	validation	ATTGAACTGACCTTTGTGGGGGTGCCTTTGTGGGGTGAGAATGGGCATACTTGGGCCTCAGTTGTGATGGCCATGGGGTGGAGCTGAGGTCTAGGCCCAGGGCTGGGAAAGCTTCTACCAACCCCGAGGCATTTGGGTGTTTTAGGGCAGAAGAGGAGCCAGGAGGATGGGTATGCCCCACTGGAGCTGTGTGTGGGGCAGCAGGTGAGGGTGGGATTCCAGAGGGAGGGTCAACCCAGCCAAGCAGAGGA
11689	chr18	31593034	31593285	+	0	validation	CCAAAGAACCCTCCCACAGGACTTGGTTTTATCTTCCCGTTTGCCCCTCACTTGGTAGAGAGAGGCTCACATCATCTGCTAAAGAATTTACAAGTAGATTGAAAAACGTAGGCAGAGGTCAAGTATGCCCTCTGAAGGATGCCCTCTTTTTGTTTTGCTTAGCTAGGAAGTGACCAGGAACCTGAGCATCATTTAGGGGCAGACAGTAGAGAAAAGAAGGAATCAGAACTCCTCTCCTCTAGCTGTGGTTT
7493	chr11	64257117	64257368	+	0	validation	CCTCAGCCTCCCAAGTAGCTGGGATTACAGGCATTCGCCACCACGCCCGGCTAATTTTGTATTTTTAGTAGAGATGGGGTTTCTCCATGTTGGTCAGGCTGGTCTCAAACTCTCAACCTCAGGTGATCCGCCTGCCTCGCCCTCCCAAAGTGCTGGGATTATAGGCATGAGCCACCTCGCCCAGCCTGCAGGGTGCTTTTAAGAGGCTGGAAGGGGGTTGGGTGCGGTGACTCATGCCTGTAATCCCAGAA
FP015872	chr17	62423696	62423947	+	1	test	CTGCCTGGTCCCCTCTCCTAGAGAGCCCTTCGCCAACCCTGGGACAGCCCCAAACGACAGCTCCCGAGCGCCACCCGGACCAGACTCCAGTGCCGCCCTGTCCCGCCACCCTTTTTCCCGCCCAGACTCCGCGTCTTCTGAAAGAGGGCGTGGTCCGCAGGAAGTTCTCACGTTACGGAAAGTAAAGTGTTTCCGGCTCCGGTGTCATGGCCGGCTCCTACCCTGAAGGTGCACCTGCAGTCCTCGCCGAT
FP006065	chr5	177603442	177603693	+	1	test	TTTTCCAGTTTCTGAAATCAGTGCCTGTGCTGTACCCAGCCCTCATCTTTCTTTCCCTCTCCATCTCCCATCTTTTATCATGAGCTGCCTCTTGGACTAAAGGTTTGAGGCTCGTTCAGTGAACCGCCACTGACAGTTTGGGCCTTGCATTGCTTCGGGGGGCACGTCTGATCTTCCCAGAGTAAACATTGGTCATTTTACCTGGACTAATGCCAGCACACACCCGTGTCTGGGTGGCCTTTCTGGTCATT
FP000289	chr1	22143046	22143297	-	1	test	CCAGTCCCGCGGGCTAAGGCGGCAGCCGCCGCCGCTCCGGCCCGCCCCCGGGGGGGTGTCCCGGGCCGCTGGGCCCGCCCAGGTAACCCCATCCTCGGCCCGCCCCCGGCCCGCCCCCCCGGCCGCCCGCCCGCCCGCCCGCCTCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCATCCCGGCTCTGGGGCGGCGCTGACAGTCTGGTCCGCGCCGGGCAGCGGGCGCAGCAGCGGGCAGGCTGCCGGCAG
FP009070	chr9	93531955	93532206	+	1	test	TTCTTTGAGCATATGCTTTGGCCACAGCCTGCCAAATGAACTCTTCCTAAACATCTTTATTTAGGCAAGTTGTTAAGCAGTTGATTTGGAATGGAATTGCAATGTCATTAACTGGAGATAATACAGTATATTTCTGTGAGTGTATTGTAAATTCAACATGAAAAATGTGCAATGTTATTCTGCTTTCTTCCATGCAAAGGGTGAAATCAAAATTGCTGTTTCTATTGAAGATGAAGCCAACAAGGACCTGC
FP000686	chr1	47189985	47190236	-	1	test	TTGTTCCTCATACCCACCTCCTTAGCTGGCCCCTGGAAGATGACCTCGAATGCCAGCCCAAGTGAGCTGGGCACCAGCCAGGGGAGGCCAGGTCTGGCAGCTCATTACCATGGGAGGGAGGGCACGTTCTCTCCGACGCCCGTCTTCGTGTCTCCTCCCTCCCTCGCCTTCCTCCTTCCTAGCTCCTCTCCTCCAGGGCCAGACTGAGCCCAGGTTGATTTCAGGCGGACACCAATAGACTCCACAGCAGC
FP012593	chr12	132329497	132329748	-	1	test	CCGCCCCCCGGACCTGCTGTCCTGGGGTCCCCGGGGGGGCGCGGTGGGGGGGGCGGGGCAGGCGCGCGGGGCGGTCCCGGGGGGCGGCCTTGAGGGGTGGGGGAGGCGGGGCGGGGCGGTCCTGGGGCCGCGGTTGCAGATGAGGTGAGGTGAGGCCGCGTCACTCTGCACCGGCGCGGTGGCTGCGGGGCGGGCAGGACAGGAGCCGGCACAGACACCGAGCGCCGCCCGCCCGCGCCTTCCCCGCCGCC
FP008976	chr9	74887429	74887680	-	1	test	CGGCTTTCAAGTTCTCTGGTCTCCACCCAAAGATGATTATCCTATCTAAGGTAGTGTACGCTACTACAGGGCAGGTCCCGCAGCCCCCGTTTCAAGAGCTGGGATGCGGCCATAGCGGAGGTGGTGTCAGAGCGGAAACCTAGCGGGGGTCGGGGGTTCAGCTCGGGGCGGTGGGAAACACCCCGGGAGAGGAGGCAGCTCTGATTCCGCTCCGGGCCGGAGGGAGAGGAGTTCGGAGGTGGCTTGAGCTG
FP010265	chr10	118046639	118046890	-	1	test	CGCCCGCCCGGTGTTAGGAGCCGCCGGCCGGGCCAGGCCGAACCAAGCGCCCGCAGCCCTGCAGCCCGCTGCCCGCCCCACGTCCCGCCGGCGCCGGGGCGGCCGCGTTCCCCGCCCTCCCCCACTCCACCCCTGGCTACCCGCTCCCGCCCGAGAGGAGGAGCTGAGGCGAGGGGGAGGAGGAGGAGGGAGAGGTTTGCGAGAGGGAGCGAGGCCGCGTTAGCCGCCGGCGGTTCGTATTACCGGGGTCG
FP010112	chr10	99913980	99914231	-	1	test	AAAGGAAGGTGTTGATCCAGGTACTGCTGGGCCCGCTGGTGCACTTCTCAGTCCAGTACGCTTAGGGTTCCAGCTCCATCCTTTAGGACAAGCCCTGCTGTGATAGCCCGGGTGACTTGCGCGATTCATATGTAAATGTGGGAAGCCTGGGGGAAATGCCGACCCTATTCAAAGGAGTTTGGGGGGTTGTGGAGCATTCCAGAGCTCTGGGAAGATTCTGTCCCGCCTGCTTACCCTCAGGCCCACACCTC
FP013654	chr15	34988217	34988468	-	1	test	CTCTCCCCTGCAATGACAGGGTGAATGGCGGTGGGGTTGGGGGCACGTCTCCAGAGTCCCAGTCCGCTCGGGACCCGCGGACCGCCCGGAGGCGACAACGGCCCTGAAATTGGGGGGAGCTTGTTTCCAGCCGGACTGTGTCCCTGGGCGCGCCAGAGCGCCGCGCCTGCGCGCTCAGCCCGTTGCCGCGCCGGCCCTGCGGACGTGCGCGCGCTGCCTTCGCGGCACCTGGGCCTGAGGTGCGTGCCTCC
FP017149	chr19	35868323	35868574	+	1	test	TCCACAACGGGCGGAGCCCATAGCCGGACTCCTGGCTGGGCCCTTCATGGGGCGGGACGCCTGGAATCTCGAGGGGCGGGGGCCTGGCGCAGGCTCCCGCCCGGGGTTCCCGAGCTGCTCCACTCTGCGCGAAGCCGCCACGCTATTGTCCTGACCAGGAAGGCGGGGCCGGCGCGGGGCGGGGCTGGCGGCGCCGGCGCAGCCCGGGGGCGGCGGGAGGAGGAGGTGGCGGCGGTGGCGCTGGGAGCTCC
FP007154	chr7	2631787	2632038	+	1	test	CGCTGTGGGGCGGGATCGGCGGGGGATAGACCAATGGTGGGTAAGAGGCGGGACAGAGGGCGGGGCGCGCGGGGGCTGGGCTGGCGGGAGGCGGGCCAATGGAGGCTCGGGGGCGGGGCCGGCGGGCGGACGGGCGGGGGTGCGGAGGGGGCGGCGGCGGCGGCGGCGAACAAAGAGGCGGCGGGCGCGGGCGGCCGAGCGGAGCCGAGCGCAGCCGAGCCGGGCCGAGCCGGGCCGGGCCGGGCCCAGGA
FP001382	chr1	156751624	156751875	-	1	test	CGGCTCCGGGTTGGGCGTTGGGCCGTGGCAGGCCCTGGGCGTCAAGAAGCTGCACGCATCAAACCGCCGCGGGAGGGAGCGCGAGGTTGGGGGGCGGGGGGAGGAGGAGGAGTGGGTCCGGGAGGAGGGAGGAGGAGGAGTGGGGACCGGGCGGGGGGTGGAGGAAGAGGCCTCGCGCAGAGGAGGGAGCAATTGAATTTCAAACACAAACAACTGCACGAGCGCGCACCCACCGCGCCGGAGCCTTGCCC
FP006773	chr6	99394155	99394406	-	1	test	CGAGACCAGCCAACATGGAGAATCGCTTGAACCCGGGCGGCGGAGATTGCAGTGATCCCAGAATCCGAGATGGGGACTGTACTCCAGCCTGGGCAACAGAACAGACTCTGCTAAAAAAAAAAAAAAAAAAAAAAGTCGGCTGCAGACGGCGAAAGTGATCTCGGAACCTTTATGTTGGGGATCTTTTTTCGGGAGGGACCGGAAAAGAGGTGGGATCGTTTGTCGCGATGTGGAGTGGCCGTAAGCTGGGC
FP016669	chr19	6481736	6481987	-	1	test	TGAGGGCCCAGGCACGATGGGGGGAACCAGACACAGTCCAGCCTGGCGGGGACGGAGCTGGGGGTCTGCCTTCCCAGGCTGACCACTTATTGCCCACCCATCCCCAACATAATTAGCAGGCAGGATGCCTGGGTTTCTCACGGGGGTGGGGGCAGGGCCTCTGTCCGGGGACGTCACCCGTTAAACTTCCTCTCTCAGCCACACAGGAAGCTGAGCCGGCTTGGGGCCCAGCATACACAGGCCCCCAGGAC
FP019221	chrX	47467599	47467850	-	1	test	CTCATTCATTTTACACCAGATCTTGTGAATTTTCCCCATGTTTTGCTTTAATACTTGGTGTGGGTTTTACCGAACTTTTCTCTTCCGCAGCATCCAGCACAGTGAGCGGCCCCTGGGCACCACCAGAAAACATTTGTGAGTGAACGTGGAGTGATGACCATGTGGACACTGTCTTTCCAGGGTTTCTTGGAGCCTCTGCAGAGTCTGGGGGCCAGGCTTCTTAGTGGAAACTGCAGGATCTTCCTTCTGAC
FP001473	chr1	163202953	163203204	-	1	test	TTCCTTGTGATTGGCCTGAGATTAGGAGTTCAAACAACCAATGACCTGTGCACAGTTTAGGGCCTAACCCTGCCCTGGCCGGTTAGCAAGAGAGTGTTGTGATAAGAGCAGATGAAAGCCTTATGCCACAGTAGTGCCTGTAGCAGAGAAAGGATTCCCTCCTCCACCTCAGGGCCCACTGCTGCTGCTCTCCAGCCTGCAGTTTCCTCTAAGGCTCTGGATTGGCTGGAAGAGCAACAGAGGGCTGGGAA
FP013271	chr14	63852813	63853064	+	1	test	GTCCCAAAAGACGGGGAGAAAGTTGCCCGCCCCGGGCTGCGGCGGGGCCTGCTGCTCCGCCCCGGCGCTGGCGGGCGCGCTCTACACTGGCCGCCGAGGGGCGAGCGCGGGACCCGGAGGCGGGCGGCGGGCGGGGAAGGCGCGGGGCGTGGTTTGCCGCTTTCCGAGCGCCAAGGAGCGGAGCGCGCTCAGCTGCGCCCAGAGCCTTCGGCCGGACCTGAAAAAGCGAGAGGGAGAGCGAGCAAAAGGCG
FP015623	chr17	42682330	42682581	+	1	test	ATTTCCAGGCACCCCTCCCCCAAACCACCTCCCTTTTCTTGAAATTTTCAGGATGCTTCTGGCCCTAGGTTCCCTGTCCTCACTCCTGTGGCCCTAGTCCCGGGCAGCCTCGGGTTACTTGGGGTTTTCCCGGTGGCGAGAGGCTCTCTACCCCCGCCCTTTCGCAGCAGGGCTGAGGCACGCGCTTGCGCGGGGTCCGGGAAACCGGCGCGTGCCAGGAGACAGAGGCTGGGGAAGGGGGGAGGTGAGAG
FP006309	chr6	30882733	30882984	+	1	test	CTGGCACTGCCATGCCACTTAGCTGGGGTCAGCGTGGGCCTGGGGTGTGGAATGTCCCACCAGGGTATGACGGGCTGTAGCTTGCCTGGCAGGCCTGTTGGGGCTTTCCCAGAGCACAGCTCCTGGAAGGAGGGGCTGTGGGCTGCCAGGTGAGGTGACTTGGGAAGCCTTGGCCCCACCCCCAGGCTGGCCCCACCCCCAGTCCAGCGTCTCCTGGGCCTAGATTCCCCAGCTGCTGTTCTCTGGAGGGG
FP004656	chr4	39977860	39978111	-	1	test	GGGCTCGCTTCCCGCCACCCGGGCTCCTCCTCCTCTCGCCCTCTCCGGTCGGGTCCTCCGGGAGGCGCCCCGCGCGGCAGGTATTGGCGCCAACCGGGTGGCGGCGCTGTCCGCCCTGCGCGCGGCCGCCTCGGGCCCGAGGGAGGCGGATGCACGGCCGGCGGAGGAAGGGGAGGGAGCGAGGAGCGCGCGCTGCTCTCGCGTGCTCTCGCGCCGCTCGCGTGACCGGCCGGTGTGTGCGCGAGGCCCCG
FP004426	chr3	196503731	196503982	-	1	test	GGGTGTCTGAGGCCTCGCAGAAATCTCAGGGAAAGGTCAGGCTCCACCCAGGCTGCACTAGGCGGGGCTGCGACGCCAGGGCGGGACCGTCTCCGGCTGGGAGGGAGGGCGGGCGGCGCGCAGACCCCGGAAGTCCCGCCCCGGGAACGTTCGCCCTTGGGCAGTTTTCTGGGTGGGAAGTGCGATTTTGGCGGGCGGTTGTGACGTTGCTAGCGCTTGTCCGGTGGCTGCTGCGCTGCCGCAACGAATAG
FP015744	chr17	48100988	48101239	-	1	test	TGGGGGAGGGGGCGCGGCGGCGGCGGCGGCGGGAGCCCTGCGTGAGGGAACGCGCTTTCGAGGCGGAGGTTAGGAGCGGGGAGCGCGCCCGGGTCCAGCGTCCTGCTTCTCCGCTTCCCGCGCTGAGCTCTTCGCCTGTCGCTGAGGCGTCGGTGCCAGCTGCGTGAAGGATGGAGAGGGCGGGGCGCGAATCCTGAGCCAGAGACTGAGTGCTTGGGGGTGGGCCGAGCACTTGGGGGCCGCTCTTCGGG
FP018256	chr20	58514842	58515093	-	1	test	CTTGGGGGAGGCCTCCTTGCCCACTGCGAAGACTTTGGGGAAGGCGACGGCCAGGGGACGGACATCTGGGGGAGGGCTGGGGGTCTGAATGGCAGGAGCTGCACCAGGGGGCTGTGCGCGCTGCGCGGGTGTGCGCGCCTCGCCGGGGGCTGTCTCCACCTCTTCTGGCAAGTTGGACCTATGAGTGCGCAGCTCCGCTTACAACTATCAACAGCCGGGAAGGCTGAGCGCGTGTGAGCGCCGAGGGGGGC
FP007581	chr7	95485742	95485993	+	1	test	TATTACTGAATGGATCAGTTAATATATAACCAGTTTAAAGGACCTGAAAATGTAGTGACAGCCAAGAAGGATATTTTGAAGTTTGAAATGATCCCTATATAAATAGAACGGATCAGCATAACTTTGGGATAAAATTAGCCGACAGTTTGTGGACTCTCCAGCATGCGCCTGTTTGCTCGGTGCTGTTCTCTCGATAAATCACAACAAAGCTTCCAGAGGGAGAGGAAGGATGGACGGCACCACTGCCCCTG
FP006349	chr6	31652203	31652454	+	1	test	TAACTCACTATAGACCCGAAACGGCACTCACGGGGCGACAGACCTGCTAGCTGACTGCCCGCGTCTACTGCCTTCCCACGGTGTTCCAGCAGAACGGCACAACTAACCCACAGCCAAACACACACACACACACACACACACACACACACACACACACCCACCACCCCGCGGCTCCGCCCCCGACTTCCCCACGGACCGTCACTTCCGGTCTCCCCCAAACCTGCCACCGACGGCCACTTCCGTTTCCCCGA
FP004542	chr4	5711001	5711252	+	1	test	AATGCGTAGCATATTCTAAGCCTCAAAAAACTTAAAGGAGCCTTCTAAGCCTCAGTAAAGTTAAAGGGGTGGAAACCGGGCCCCCCTCCGGAAGGGTCCGGGGCTGGGAGGCGCGTCTGTCTCTGGGCATGCTCAGTGCAGGGGCAGGGCTGGGGCGGGGGTGGCTGAAAGTTTTGAGCGGTGATCCAGGCTCCTCCCTCCGGCTCGGCGAAGCAGGGAAGGGGAGAGAAGCAGGAGTCGGGAGACTGCAC
FP010840	chr11	61356750	61357001	-	1	test	CTGGCCTTCGCCAGGGCCAGCCATAGCCTTGAGTTTGCTGCTCTGTCTGGGCACTGCCAGGTGCTGGGGTGGGGCCTCAGTCATTGTGACTGAAGATCAGGCCCACCCAGGCATTGAGGCCTCGGGCGGGGGGTGGTGCCCAGGCTGATGCAGGGGAACTGAAGCAAAAAGATTCCATCCCACAGGCCAAGAGCTAAATCAGTGTTACCTCCTTTAGCCAGAGAACTGGGTGCATCTGAGCCAGTGGAGAT
FP011133	chr11	73142015	73142266	-	1	test	GGAGCGGCGGCGCAGGCGGCCGACCGGGAGCGCGGACAACGCGGCTGCGGCTAACCCGCCGCCCGGCCGGCTGAGACCCTCGCGCCCCTGCTCCTGGGGCCCCCGCCCCCTCGCCGTCTCCGCCTCCTTCCTCACACACCCCCGGGCCGCCCGGGCCGCCCGGGCCCCAAGCCTTCCCGGTGCTCCTCCCTCGTCTCCTCACACTCGCTCTCTGGCTGGGGTCCGCCTGCCGCCCGCTGGCCTGCTCCCTC
FP000997	chr1	109213731	109213982	+	1	test	GATCACAGTTCCTGCAGACTGTAGCAGAGCTGGGGAGCAGGAACCAGAGGTATAGTCCCACGCCCACCCACTCTCCCCTCCCCTCCCTCCGCGGCCCCGCCTCCAACCAGCCGCTTCCGGTCGCGGGGAGGGTCGCCTGCGGGCTCCAGCTGCGCCTGCGCAGGAAGGGCGGGTCAGCGCGCCGGCGCAGTGCGGCGGTCACAGGCTGAGTGCTGCGGCGCGATCCTTGCTTCCCTGAGCGTTGGCCCGGG
FP015560	chr17	39980606	39980857	+	1	test	CCTCTCGTGGACGCCCACAGAAATGCACTCGAGAGGAACATAAAAGCATCACACTTAAGCGCTCGAAAGAAAGCCCCACGAACCCATAAAGGCTCCTCCCCGCTCCTTCCTCGAAGCTCCTCCTTCATCCTGCACCTCGGCTGTGGATTTATTTCCCCTTTGTTGACTCGGCCATCGGCCTGCCGGGCCTGGCGTTTCCCAGAAGGCCCAGCGCCGGGAAGGGGTTTGCAGCTGCTCCGTCATCGTGCGGC
FP005857	chr5	142621283	142621534	-	1	test	GAAAGAGTAGGGAGGTTAGGGTTGGGGGATGTGTCCAAATAAGGCTTGCTCGAGGAGGAAATATCCGAAATATCCAGCTAAATTTTTGAAGGGTGAGTGTGACTTAGTCAGACCATGGTGAAAGGGTGGAGGAAAGGAGGGAAGGAGGGAGGAAGGGAGGGAGGGAGAAAGAGAAAGAGAAAAAGAGAGAGAGAAAAAATACTGTTGGCAGCAGCACAATGTTTGGGCTAAGACCTGGGTGAGTATGAGGG
FP003988	chr3	122022196	122022447	-	1	test	ATGAGATTTCCCGAGAACTTCTGGAGGAAAGTGGGAATCTTCTCCGTCAGCGGGGTGGGGGACGGTGTTTCAGCGAGCAGGAGGCGGCAGCAGGTAGGGAAGGTGGCCGCAGTCCCCCGGGAGGCGGGGGCGGAGCAGGAAACGGCGGCGCGCGGGGCGGGAGGCGGAGCCGTGGGGAGCGCCGCAGGTGGGGACGAGCCGGGCGGCACCTGCCCCGGGACCAGAGCGGACGCTCCCTCCCCGCTGCGCCG
FP010841	chr11	61361828	61362079	+	1	test	GCAGAAGACACTGCCCTGAATGCCTACAGCCGGCTGGCAATGCCGCCCCAGAGCAGAGCGGCGGGTGCCGCCTCGCCTTCTTATAGGCTCGCAGCACTTCCAGTCGCGCGCGAGGCCTCCTGGGAAGTGAAGTCCGGAGACCAGGGGTGGCCGGCTCCCAGCGCGCCAAGCTGCTAACCCCCTAACTCCCGTTCCTCCTCGGCGTTTCTTCTCCAAGTCCGAAGTCAGGGTCCAGGACTGCTGCGATGCAT
FP002019	chr2	10043371	10043622	+	1	test	CCGATCACGCCGCGGCCAGGCTCCGATCACGCGGCCCCCCGCGGCGCTCATTGGCCGGCCGGGCACGAATTGGCTGGTGGTGGCCGGCCCCGCCCCGGCAGGGGCGCGGTGTATTTTGGTTCGCCTCCGCGCCCCGCCCCGCCAGCCGCTCCCTCCGCGGCCGCCCCGCCCCTCCCGCGCCGCGAGGGCCGCGCCGGGGCAGAGCCGCGCGGGCGGGCGAGGCGCGTGCCGGCCGCAGGAGCTCCGGGTTG
FP003218	chr2	231709698	231709949	+	1	test	CCGCTCCTGTTGTCGGCGCCGCCTCGGTCCCACTGCCCGCCCTGGGTAGCGTCTCCGCCCTTGGCGGGAGCGGGGCGCTCTCAGACTGACTGGCTCTTTCTTAATATTTCGGCCCTCGTCCGCGCCCGTCGTGCCCCTGCAGGGATTGGCGCGAGTCACCTTGGCGTCTCCTTAACCCTTGTGTCCCTGGCGTCATCTCTGACTCTCCCAGGGGCGACTTCTTGGCAGAGCGGAGCTCGGGGCCCGGATCT
FP008373	chr8	66870570	66870821	+	1	test	ATAACATCATTTGCCTGAGATAACAACAAAAACCAAACGTAGACCTATTTTTTCTCTGCTTCGCTGGGAGGGAGCAAGAGTGAAGCTATTAAAACGTAAGGGAACAAGCCCGGCTAGTTCCCCTGCTGACCCGAGGGCGCGGGCGGTCCCGGCAGGCCCCGCGACGCAGCCAACGGCCGGGACGTGCGCGCATGCGCGCTAGGACTCCGCTCCGCCTACGCTGCAGGCGGAGAGCAACCGCCAAGCTTGGT
FP005566	chr5	96875722	96875973	+	1	test	AGGTTAGGGTTAGACATCCATGTACTGTTGGTCCATCAGGCTGTTGGAATTACAGCAGCAGAACCCTTGAAGGAGGGATCACAATGGGAGCAATGTGGGATGAATGGGGATAAATGCTAAATCTGGGTACTGGAAAGGATAAAGAGAGGGCAGAGCAAAGGCCAGAGGTTTCATCTTTGTGGAAGGTCTGTATTCAGAGCAGAGAGGAAGTTGAAGCCCAACTCAAACAGGCAGATAAAGAGAGATCAAAG
FP008727	chr8	144853475	144853726	-	1	test	GGCTATAATTCACATACCATATAATTCACCCATTTTGTACAATTCAATGATTTTGGCATATTATTTTTTAACTATGGTAAAGTACATGTTAAATAAAATGGCTCGGCCATTGTTTCGGACCGAGCTCCTGCGCTAGGGCCCAACAGAGCAGCCCAAAGCGGAACCGAGTCACGCGCGCCGGACGCCACCCGACAGGAGGGGAAACGGGGCAGCTCTCAAAACCCTGGAGGCGCAGACCGTCCACACGGGAC
FP013143	chr14	39267165	39267416	+	1	test	GAGGGGTGGGGTGGCGAGGACAGGGTACGTCGCAGGCTTGTGCGGGTCGGGCTCGGACCTGCGCTGCCTCGGGATGTAAAGTATAACAAGAGGGTCGGGATGGGCAGCGTAGGCCTGTGAGGCCTGCGGGTGCCCCTGTCCCCCAGCTCCCCCCGCAGCCGGCTCCGCAGTGGTCCACTCCGGTTGCCGGGTGCGGATTCGGGTTCCGGACCGAAGGCTGTGTGTTCTCCGCCGTTTATTGTGGCCCCGAC
FP008343	chr8	57994322	57994573	+	1	test	ATGGGGTGGGATGGGCATCAGTAGCGGGGTGGAGATGTTGAGAGAGAGGCAGGGCTTGGAGTGGAGGTGCGAGAAGGGTGGGGACCCGGAAGGTGGGGTGGGGGCGGGATGGAGGCGGGGCGGGGTGTACGAGGGGCGTGTACACTGGCTCAGGGACACGCGCTCTCGGCCACAGCAACTGGCTCCAAGTTCCCTCCCTCACTCTCCGGCGAAGCCTGCCTGAGCCCTCCCACTCGGTGCAGACCGAACCA
FP003009	chr2	201071588	201071839	-	1	test	AGATACTGTAGAAAAGAGAAGGGCGGGGAAGATAATATGTTGTCCTTGCTTGGGGACGGAATCCTTAGACACCGCACACAGGGCGGGGCTGCCGGCATACTGCAGCCAATAAGGGAGGCTGTTGTTGAGAGCGTGTTCTCACCCAATGAGAGGGTTGGTTGTTGCGCTGCAGTTGGCAGGCTGCTGCGGGAGGCGGCGGCGGTAGGAAGCCGGAGACAGCAGGGTGACAGGTAGGCGGCGGTGCTGTCTTT
FP010468	chr11	3855463	3855714	+	1	test	AAAGACTAGCGCGGGCCGGGGGTCCGGGAGAGCCCGCTAGGGGCGGGGATTCCGGGGAGCCGTCTTCACCGGTTATTCCGGGATCCAGCTGGGCGCTGGGGCTGGCCCGGGCTTCGCTGGGGACCGGGCGGCGCGGGGCGGGCGCGGAGACGCACGCCCCCGCCCGCCCCGGGCCCGCCCCGCGCCGCCCGCCCGCCTGGAAGCCGCTGTCCTGGGCCTGGCCGGTGTGCGTCCGCCTGCTGGACCTGGGC
FP001359	chr1	156149856	156150107	+	1	test	GCGAATCCAGGAGCCACAGAAGGTGGGGCGCAGTGGGACATGGGTAGGAGGAGGAAGAATATGGGGAGATTCTAGTCCCTCCCACTTAGAGATGCGGTCGCCATGGTGACTGAGGACCAGTGAGGCGGGATGGGGTTGAGAATGGGGGTGGGGGTGTGGCAGGGGCTGAGGCACTGAGAGACCGGAAAGCCTGGCATTCCAGAGGGAGGGAAACGCAGCGGCATCCCCAGGCTCCAGGTAAGGGAGCGGGG
FP011087	chr11	68841865	68842116	-	1	test	GGAAACTGAGGCTCGGAGGGTGAGGAATCTCGTCCTAGACCCCATTCCGGCAGAGGTCGAGGACGCGAGCACCCGCTCGAGCCTCAGTTTCCCTCCCCTCCTAGCCCCGCGGACGCCCCGCCCCGCCGAGGCCCCGGGCCCGCCCTCCGCGCGCCGGGAGTCACCCGCCTGCAGGCGCCGCAGCCGGACCCGCCTCAGCCAATCCGCTGCTGCCGGCGTCGGGTGCGCTCGGCCTCGCCCGCGGCCCTCCT
FP012248	chr12	85036205	85036456	-	1	test	GCTGTTCACCTCCCTCAACGGGAATCCCAGTCTTAAGACCCCGGCTCACCCCTCAGCAGCGAGATTCCTTGGCTTTGTTGTAAGACAGCGCCCATAGCCGCTACACAAAGCGTATCCATGGCAACAGCCAATCACAAGGAAGAGACACCGGTTTTTCCCTACAGGTGCTCTAGAAGGACAAACAATCTCGATTCTAAATTGAAACGAACGCAGCATTTCAGGGACTGGATGAGGAGCTTACGGTTTTTTAC
FP004809	chr4	80184092	80184343	+	1	test	AGGAGCCTGAGCGTTCCGGTACCCAACCCTCTCTCTTTCTTGACCTTCCTCCCCTCTGCCCCTTCATCCTGGGCGCGGAGGGCGGAACAGTTTCTGGACCAGAGCACCCGCGAGAAGCAGGGAGCGGCCGCACAGCAGCCAGAAGAGGGCGCCAGCACCCCGGGGCCGGGGCTACAGGGCTGGCTCTGCCCCGGCGTCCCGCCCCCGCCCCGGGGAACGCTGCCTCTGCCGCAAAGATGCCCCCTTCCGCT
FP014054	chr15	78507377	78507628	+	1	test	GGGATACTCACTGTGAGAAGGCTGGGCGGAGTTGCAGAAAGTCAACAGAAGCCGAATCTCTGAATTTCTGTTCGCAGCCTCCTAGGCGGGGCCGGGAAAAAAATCCAGTAGTCTGCGCTGACTGGGCGGCGAGGGACCGGGAGGAGCCAATCAGAAGTCAGGACTCGCGGGGCTTGGAGGAGGGGCGCGGGCGCTGCGGCCCCTGCTCTACCTCCTAGCGCCGGTGCGCGGCCGAGGCCGCACTACCTGTC
FP007082	chr6	159750872	159751123	+	1	test	GGGTTTTGCTTTGGGTTGCGTGCACGCGCACGCACGTGTGTGTCTTCGCTTTACTGTTTTTCTGTTCTAATTTTGAGGAAGAAAGATTCCCAGAAATAACCTGTCCAGTTCAGTTAAAATATTTCATTCATGTGAGAATGCACACATTTTAGAGTGTTTAGAGGTATATCTTTCAAAGTGCTTTCTATTATGTTTGTGGGTTAATCGATTTTAGGAATGGGTTGATTTTTTTGTTTGTAATTTTAAAAAAT
FP006294	chr6	30571241	30571492	+	1	test	GACAGTCGTAAACGCCATGTGTTTACGCGACTGGAGCAAGCGGACGCCGGCCCCGCTCCGTCATTGCAGGCCACGCCTCCACTGAACCAGGGCCACGCCCCCGAGATGACGGCGAAGCTCGCACGTGCGCAGCCCGGGGGCGGGGTTGGCCGCGCCAGCTTGGAGAGCCAGCCCCATCGGGGTTCCCCGCCGCCGGAAGCGGAAATAGCACCGGGCGCCGCCACAGTAGCTGTAACTGCCACCGCGATGCC
FP014832	chr16	67644937	67645188	+	1	test	CCTCCCGGCCACGCCACTGCCCACCCCGACGTCTACCCTGGCGGAGAAGCTCCTGGGCGACGGGGCGGGCGGGCAGGTGGGACTCGGCCCCCCTCCCACAACCCCGCTCCCGGGCAAGCTCTCGAGCCGCGAGGCCGGGGCGGGGAGGGGCCGGGCCGGGGGCGGCCTGGCAGGAAGCGGCGCGCACCTTCCGCCGCCGGAGGAGCAGGTGGCTGCCGTGCGGGTCTGGGCCCCAGGCTTCCTGTGTGCGC
FP003277	chr2	238847905	238848156	+	1	test	CTGAAAGTGCCGCGCCGGGGGGCGGGGGCGGCCGGCGAAGGCCCCAGAACTTGTCCTGCCCCCGGCCACCGCGGCCAATCAGCGCGCCGCCCTAGCTCCTGACAACTATTTAGCAACCCAGCCCAGCTAGAGTTTCCAAAAAAGTTAGAATAACTTCCTCTCCCGGAGACCTCGGTTTTGCACAAGCCGGCCTTGAAATCAGAGCCTTTCCAGCAACTCCGAGAGCGTGTGCTCGGCGACCGCGGGCTTGG
FP003383	chr3	12609263	12609514	-	1	test	ATTTTTCAGATAGAAATAATACCTACTTCATAGGTTTGTTGTATGAATTAAATAAATTATTGTTGTATGGATTAAATAAAGTTGTGTTTATATGGCATGTGATAAATGGTAGCTGTTGTTATTTCTATTGAACTTTGATCTTGTTTAAACATTTCATGTTTTTTTTAAATCCTTTCTAGTAAAAAAGCACGCTTAGATTGGAATACTGATGCTGCGTCTTTGATTGGAGAAGAACTTCAAGTAGATTTCCT
FP018542	chr21	46228713	46228964	-	1	test	GTCGCCTGGCACACCCCGGGGTCACGCTCGCGGCGCTCTGATTGGTTGCGTGGGCGTCGGCCCACCTAAGCCTGAGCGCCTGCCGAGGCCTGCGCCTGCGTAGTGCGCGCGGGAGGGGCGGGAGGGGCGGGAGGGGCGGGAGGGGCGGGGCTGGGCGGCAGGTCCCGGGTGCGGACATCTGGCAGCTGGCAGTGGGCGGCGTAGAGCACTGCAGCAGCAATGACGGAGGGCACGTGAGTCCCCTCGCCCCG
FP014101	chr15	83011564	83011815	-	1	test	GAGGCCTCAGCTCCGCAGCTCCACTTCCTGGGGCTCGGCCCGTTCCAGAACAAAAGAGACTCGCGCATGTTCTGCGGCCACGAGACTCGCTCCCGCCGGCCTTAAGGTGTCGCCCTCTGGCCAGAGCGGATGCGCCGGGCGGTGGGGCCGCTCCTGCCGGCTACCTGCGCAAGCGCAGAGGGCTCTTCCGGCGCCTTCCCAGGCGGGGATGCTGCGGCTCCGCAGCGGGCTGAGGCACCTTCGGGCAACAC
FP016884	chr19	13952308	13952559	+	1	test	CGGGGCGGGAATACGGCAACGGGCCGCGGGGGCTGTTACTGTCGCTACCAATCAAGAGTCGAGATGGCTTTGATGGACAGGCATAGCGCGAGTGCGGGCTTTCGCCCAACCGGCGGGCCGCTGTCCCGAAAAAGGACCAATGAGGAGGCGGCAGGGGTGGGGCGAAGGGGCCGGTTGCTCCGGAAGTGGAGGGAGGGGGTGAAAATGGCGCCCAGCTCGAAATCGGAGCGGAACAGCGGGGCTGGGAGCGG
FP016056	chr17	77376014	77376265	+	1	test	GGGGTGATGTGGACAGGCAGCTTCCGAATCAGGGTAGAGAAAAGTCACCACTAGCTAGCAGGGGAGAAGTCAGTATGGAGGAGGCGGACCTTGAGGGAGAGTAGGAATTGGATTGCAAGAGGAAGGAGAGCCTTCTGGCCAGCAGCAGCCAGCAGCAGTGGGGGAGGCTGGAATGAGCTGGCTGGAGAGGGGGCTGGGGCATAAGGAGGGGCCTGCCTGTGAAGATCATATGGGCCAGGCTGCGGAGGGCC
FP004934	chr4	105895240	105895491	+	1	test	ACCCCCAAGAGCCACTGCCGTCCCGCAGCGCCCCTGCCCCCGAGTTGCCTGCCCCGCTGGGCCCCCGGGAGGAGCGGAGCGCGCTCACCCTTCGCCCGGGGCTGGGAGGGCGGCGAGTCGGGCGCACGCGCACCCCCTGCCCGCCCCTGGCGCCCCTCCCCGCGGGCGGTGCAGCTACCCCTGCAGCGCCTCCCCTAGCTAGAAGGGAGCGGGAGGGGGCTCCGGGCGCCGCGCAGCAGACCTGCTCCGGC
FP004640	chr4	37890914	37891165	+	1	test	GGAGGAGGACACCGAGTCCCCCTCCCAGCTCCCCGGGGACCGAGTGGGGAGATCCCGGCTCCTGTCTTCCCCTCGCCTCCAGCGCGCTCGCCCAGGCTGGGAGGAGGAAACCAGAGCCGCGCGCAGACACCTCCTCCTTCTCCTCCTCTTCTTCCTCCTCCTCCTCCTCCTCCTCTTCGGCTGCTGCTCCTGGTGCCGCCACCGTCCGCCGGTGCCTGTTGCTGCCGCCGCCGCGGGACCTGCTGTGTCCT
FP000931	chr1	92836044	92836295	+	1	test	GCAAGTGGATCTGGTGAAAGGGTGGGTGTGGAAGGAAATTTTCTTTTCCAGATGTCAGTGGTCCTTACGGTTATGACATAAGCTATTTTAATTTTAGAGCAGTTTGAATAATTGAAACCAGCATTTACATTGGTTTCTTGAATAGCTTCTCAATAGGTTTGGCATGGACAAGATCTATGAAGGCCAAGTGGAGGTGACTGGTGATGAATACAATGTGGAAAGCATTGATGGTCAGCCAGGTGCCTTCACCT
FP018457	chr21	38804982	38805233	+	1	test	CAGATAAACTCAGGTTTCCACCTTGAATTTCTCATCAATCTGACGTGGCAGGTTATTCATCCTAAGGTTCAGTTCCCACCTGTGTAAAGTGGGAGCCGCAAGTCCTCCGAGAGTGACGATGATGTGCGTGGAGTGCCCAGCCCAGATTGAAGCGCGGCCAAGGCGGGTCGCTATCTGGGCACCGCTCAGCTCCAGAGGGCGCCACTCCCGCGGAGCCTGCGGGATCGGGGCTTCCCGGGAGCAGCGCGATC
FP000451	chr1	32221548	32221799	+	1	test	GCTCACAAATGTTGCCTTATGTAACTGAAGGAATCACAATAAAATGTGAATAATGGCAGCCATAGATTGAGCACTTACTATGTTCTAGGCCTAATGCTAAGTGCTCTACGTATCTCACCTAATTCCTGCAACTGACCTATCAGATGGTTATTATTATAATCGCCATTTACAGATGATGAAACTGAGGCTCAAGGACAACTGACTTTTCCGAGAATTTCTAGTGAGTAAGTGGTGGGGCTGACCCAGACCTG
FP009263	chr9	121286406	121286657	+	1	test	ATCCACTGTCTCAACCCCAGCCATGCTTCTCCAAACTGTTTACGCTTCCTTGTGCAGGCCACACCCCCAAGCCTTTGCGTCTGTTGTTCCCACCTTCCCAAAAGGGAAACTCCATTTATCCTCCAAGGCTCTGCCCCAATGCCCCTCCTCCGTGGCTCTGTTGGGCCATGGGTGCTCCCTCCTTTGTGCTCCCAGGGCCCACAACTGCTTGAATGACTATTGGTGTTCCTGTTTGTGTGCCCTCTGGGCCG
FP015622	chr17	42681792	42682043	-	1	test	CCTCTTTCCTTCCTTTCTTCTTCTTTTTTTTTTTTTCGTAAAATCCAGATCCTGCACAAGAGAGTGAAGTCTTTTGGAGGAGGTGGAATGGGAAAGTGACCCCGTAGACCCTAATCCCCACCTCCCTCTCGGCCCCTCTCCCCGCCTCTTCCTTCCTGTCCCGCTAGTAGTTTAACTTCTACCCCCTGTGGCTATACCTCAGAAACCTGTGTAGCCAGAGATGGGGACGGAGGCCACAGAGCAGGTGGGAG
FP011654	chr12	9733046	9733297	-	1	test	CAGAAAACTAACTGAAAAACGAGAACCTACTGTATGGTTAGTAATTTCTTCCATGTCATACAAGTATTCGAGAAATCTGCTACCTTGATTAGTAAGACTGAACACATTGGTTTTGTCATTTATTCATGGAGGAAGTCCACCACCCACTTGGGGAGCAGAAGGAAATTTGCCATCTCAATTTACTTATCAGAAGTTTCTTTGCAGAAATATGATTGTCCCTTCAGTGGGACATCATTTGTGGTCTTCTCTCT
FP010826	chr11	60334741	60334992	+	1	test	CACTGCTCCATAATATCAGTTTCTTTCTCTAACACACAGGCCAAGTCTGGACTAATTGGCAGAAACTGACTTACCAGTTTTGCAGCCAGAACTTGGGAGCTCTGTACTTTTAAGAGCTAAATCTATTTTTTCTTCCGTAGTTGGCAACACCATTATGACATCACAACCTATTTCCAATGAGACCATCATAATGCTCCCATCAAATGTCATCAACTTCTCCCAAGCAGAGAAACCCGAACCCACCAACCAGG
FP005752	chr5	139198257	139198508	-	1	test	CTCGTGCAAGAGTCTGCAGAGGGCGGGGCTAGGGGCGGGGCTCCCACCTCGGTGCGGGGCGGGGGGCGCCGCGCGCTGGTCTGGCGGCCCGCGATTGGGCGAGGCCGCGAGCGGCGTGACAGCTGCCAGCTCACCGGCTCAGACGGTCGCCGCCGCGTTTGCGCAGGGGGAGCTGGTCGCCGCCGCGGCCGCCTGGAATTGTGGGAGTTGTGTCTGCCACTCGGCTGCCGGAGGCCGAAGGTAGGGGCGGT
FP004375	chr3	186806412	186806663	-	1	test	GGGACAGGGACACCTCCTAGGCCATGCCTGTTCCAGTCCAGTTCTGCCTGAAAGTCCGGCTGGCTCATCACCTGCCTAAATAAAACCGTATACGGGCAAACTCCCTCCGCAAGCAGCGCGCCCCAGCACCGGAAGTGACGCGTTACGTGCCCGCGTATTCCTACCGGCGTATTCCCGCCCTGCTTTTCGCCCGCCGTTCCGTGGCGGGAACTGAGGCGACTGTGGGGACATCAGTGATCGTAAGTCTCCTG
FP010920	chr11	64234339	64234590	+	1	test	GCAGCGCCCCGCCCTCCGGGTAGTGGCGGCGGCGACTGGGGAGCCCAGCCTCCTGGGCGGTGCGTCCCCTTTCCCCTGCCGCGGCGGGAGGCGGGAGGGGGTGTGTGGAGGAGGCGGGCCCCGCCGACGGCCTCGCCCCCCCACCCCGCCGCCCCGCCCCCGCCCCACGGGCCGGTGGGGAGCGCGTGTCTGGGTCACATGAGCCGCCTGCCCGCCAGCCCGGGCCCAGCCCCCCGCCGCCCCCGCCGTCC
FP013539	chr14	102331378	102331629	+	1	test	GCCAGGACTACAGGCGCGCGCCACCGCGCCTGGCTAATTTTTTGTATTCTTAGTAGAGATGGGGTTTCACCATGCTGGCCAGGTCGGTCTCAAACTCCTAACCTCAAATGATCCACCCGCCTTGGGCTCCCAAAGTGCTGGGATTTCAGGTGTGAGCCACGGCGCCCGGCCACTAAAAGGTTTTTATGGGGTATATAAAGCATGATTGTTTTATTTTACTTTTGGGTGTGCAACACTGTCTAGAATGGTCA
FP011880	chr12	48852091	48852342	-	1	test	TCCCGGACGCTAACGTCTGGCAAACCACATTGCCCGGCAGACCCCGCAGCCTGGGGCCGGCGCTCAAGTCATGCATTCTGGGATTGGTAGTTTCCAGGCCTCAGTGTCGGCCTGTGCCGCCCAGCTCCAGGAACTCGCCTTCCCAGCTTGCCTCGCGGCCGGGGGGGTGGGGGGTGTTCATCTCCGCGACCAGGAAACGGGAAAGATGGCGACGGCTCCGCGACGTTGAGGCCGCGTTGGGCGGTTCAGAC
FP009242	chr9	115090859	115091110	-	1	test	TACTCTGTGCTTCTAAATCCCCAATTCTGCTGAAAGTGAGATACCCTAGAGCCCTAGAGCCCCAGCAGCACCCAGCCAAACCCACCTCCACCATGGGGGCCATGACTCAGCTGTTGGCAGGTGTCTTTCTTGCTTTCCTTGCCCTCGCTACCGAAGGTGGGGTCCTCAAGAAAGTCATCCGGCACAAGCGACAGAGTGGGGTGAACGCCACCCTGCCAGAAGAGAACCAGCCAGTGGTGTTTAACCACGTT
FP011210	chr11	83034424	83034675	-	1	test	AATACGGCCCCTTTAAGCTTGTTAAGATTCCTCACTAGCCTTGTATGCAGCTGTGGGCAGAGAGCTTTGGCAGAAGGAAAGACTTTTTGTTTTTTTTTAAATGACTGTTTCATAATCTGTTGGCTCAAATCTGCTGCTGCCACACCCTGTGCCTCTCAGGGTCCCTGGGTCCTGTTGCAGTTATCATATGCCTCACCTTTAACCTAAATGACTCTGCGAAGTGGCTGCTATGGAAGGATTTACTGAGTGCA
FP002952	chr2	190469292	190469543	+	1	test	AAGGCCACATTCGTGTTTACATATTCTCATTTTTATCCTTCCTTCCCCCATTGTAGTGAAGTGGTTACAATCCCAGGCATAGTTCTCCTACAGCCACCTGGAAGGAAGCCAGTAATGTCATTACTGATGGCAGTGCGCCCAGACTATCTTAAAAAAGCATCATCTCTGCTACAAATTAAGTTTCAGGAAGGGTGATGTTCACTTTTATATTAACAAGCATTTTGAAAAGGCTGAGGGCAGATATGTTAGAA
FP005590	chr5	110738806	110739057	+	1	test	TCTCTTTCTGGAATCTCTTTGGTACTGTGTTTCATTAATCCCACAACCATATTCCCACCTATTCCCTAACGACAACAAACTTTTAAGGTCCAGGTTACCGCCGCGTCTCCCTAGCAACCGTGCCCTTTAATGGTTGCCGGAAGAGGCTATAATCACGTGCTCCGAAGACTTCCGGGTTTCCAACGTGACTTCCGGTTGTCAGAATTTACCCCTGACGCGGCGGCGGCCGACGGGAAGCTGTGTGTGCTTAG
FP012828	chr13	72781932	72782183	+	1	test	CCGGATATTAGTAACACTTCTCGCGGGAAAATGCCGAGATAAACATTGGTTGGCTCTCTGACGGCCGCGGGGATTGGGCGAGAGTGGGGAGGGGCGGCAAAGCAAGGCGAGGAGCCATTTCCGCCCGTGGCCTAGGCGGGGCTTCCTTCTGGACGCCACCTCACAACGTGTGTTGCTTTCCGCTCCTCGGAACATCCGGGAGAGTTGACTTCCGGCGGCTTGTGGGAGTGCTGGTTCTGTCCTCCTTGCGG
FP015801	chr17	51166325	51166576	+	1	test	GGGCCGGGCGGGTGGGGCGTTCCTGCGGGTTGGGCGGCTGGGCCCTCCGGGGTGTGGCCACCCCGCGCTCCGCCCTGCGCCCCTCCTCCGCCGCCGGCTCCCGGGTGTGGTGGTCGCACCAGCTCTCTGCTCTCCCAGCGCAGCGCCGCCGCCCGGCCCCTCCAGCTTCCCGGTAAGGCGGTGGGGGCGCATCCCCTGGCGACTCCTCCCGTTCCCTCTTCCGCTTGCGCTGCCGCAGGTGGGCCCGGTCT
FP013649	chr15	34338006	34338257	-	1	test	TCTGAGGGTGAGGAGCAGCGCCCGCCCTCTAGGATGCTTCCAGAGTGAGTGTGCAGCACAGCCGGACCGAGGAAAAGGATTATTCCCCTCTTCACGCGGGTCGGGCCGTGCCCTGGACTACAGCTCCCGTCGTGCCCCTGGCCACTCGTATTCGGCCCCGCCCCGCCTGCCGTGTCCCGCGGCGAGCGCAGGCGCGCTGCACTCGGTCACCGCGGGCTGCGGCGGCTGGGCGGGGCTTTCGGGCGCCCGCG
FP006755	chr6	89412222	89412473	-	1	test	CGTGTGTCGGTGTGAGTCTGGGAGGGTGCAGGGGCCCCGCGCCTCCCCGGTCTCCTCGTGCGTGTGCGTGTGCGTGTGCATGTGTGTCCTCGGCCCAGCGCCCCGCCGAGTGGTGCCCGCGGCCGGGCGAGCGCGCGCGCGCCCGTGCGGGGTGTGTATGTGAGTATGTGACAGTGTGTGTCAGGTGACTCGGGTCACGCAGTCTCTCTCTCTCTCCCTCCTCCGGGAGGAACTGCCGCGCTCCGGCTGAC
FP007920	chr7	141789982	141790233	+	1	test	ATATGAGCTCATAACTGGCACTGTATAGCACCTGAGCATGTTTTGATTTGGTCTTTCTTGTTGAGGGGCCTTGTTACGTTATACCCTGGTGTCTAGACTTTGGCCCAAAACCTTTGAGATGGAGAGCCGGTGTCTGTGGTACTGTTTGCAATCTCTGGTTCTACACTCAGGGTTCATTCATGTAGATGCAAATACAGATGGTATCATAACTTCCAACCTCTCTGTTCCTCATCACAGCTGTTCAGTCTCGC
FP008024	chr7	152759587	152759838	+	1	test	CCCAATCTCTCCAGCAGCAAGGTTGGACGTATGGGGCCAAGAAGCCTCGCTGCCAGCACTAAAGATGGCGCGCCGGAGTGGGGGTGGGGGTGTTGGGGGAGGCGGGGCGTGCGCGCGTTCTGATTGGCCGACCGGGGAGTGACGTCACGTGTCGGCCGCCGAGCATCCGGGCTCCCGGCAGCGGCGCTGCGGCGGCTCGCGGGAGACGCTGCGCGCGGGGCTAGCGGGCGGCGGAGCGGACGGCGACGGGG
FP015065	chr17	996918	997169	+	1	test	GGGGAAAGCTGTCCAAGCCCAATTCCATTTTCACAAGAGGCTTTCTTTTTGGAAACATGATAGCGGTCTCGCTGGTGTGCGCACCAGGAGCGTTGCGGCCGCGGCTTCCTCCTGCGGCGAATCTGCCGTTGCATCACAGTGGCTAGTCTCAGGGCCCGCAACGTGACGCTTGTCGAATCTGCTGCGGGGAGAAGGACGCGAGGGTTGCTTGGGCAGCGACTGTCATGGCGGCGGCCGCCCCCAATGCCGGA
FP012402	chr12	110025565	110025816	+	1	test	TTCAGTTGATAGTTTATATACTTTCTCTGAAGGATCCTAATGATAGTTAACCATTTCTCATTTTTATTTTGCTGGATTGTTTTCTGTTTTTTGCTTCAGCATTCTTGCTTTTGCTGTGCTTACTTTTGGAGTTTTGATTCCCTGTGTCACTGTTTTCTTTCGCATACACCTCTCAGGTTTACACAGTAAACAATGTGAATGTGATCACCAAAATACGCACAGAACATCTGACCGAGGAGGAAAAAAAGAGA
FP004558	chr4	6998885	6999136	+	1	test	CTTTGCAAGCTTACTAAAGGGTGGATAATGCCTGAGCCTTCTGGAGTCCGTCAGGTGTGAAGTGAGCGCTGGAGCTGATGCTCGCTCTGCTTCAGGGCGGTTTTCCTGGTGTGTTCGCAGTGTGACAGGAAAATCACCTTGCCTGGAAATGGTCCATATCAGTTAGTAGATTGTTGTTTTTCTAAATTGTGATTTCTAGAGCTGAAAGAAGCCCAGCGAAGGAAGAAGCAGCTGGAAGAAAGATGCAGAGT
FP006502	chr6	37499839	37500090	-	1	test	ACTTTTCCACCCTCCACCCATCTAGGCTGAGGTCTCAAGCTAGAGGTTGCCCCTCATAAACATGTCCGCACCAAGCCGGCCGCGCCCCTCAACAAGATGGCGACGGCAGCAGCCCCACCTCCCGCTGACTCGGTGGCCAATGAGGGAGAAGCACGAAGACTTGAGCATGCGCACATAGCGACTTGGTGGGCGCGTCCAGTGATGACTGGGGGATCCCGGCAAGTAACATGACTAAAAAGAAGCGGGAGAAT
FP003320	chr3	3179640	3179891	-	1	test	TTCTGTCCCCGGGACGGAGCGCAGGCCTGTAATTGTCCCTCCCGCCTAGGCCATCACTTTCAGGCTTGGGTACGCGCCGCCGTCTGCTCCCCTGGGGCCGCCAGGGGGCGCTGTGGCCCCGGTGCGCGGCAGCCGCGCGACACGGGCCCTCCCTCGGAGTCTTCGGCACCGCCCTGTCCCAGCCTCCTTTGCGGGTAAACAGACATGGCCGGCGAAGGAGATCAGCAGGACGCTGCGCACAACATGGGCAA
FP002585	chr2	105399363	105399614	-	1	test	GCCCCGAGAGGGAGAGGTGGGGATAGGACCTTCCTTTAGTCCCCTCCGTAGTGGAACGAAGGATTCTTTATGGTCATCTGTGGTGTTCATTTCGTAAAGAAGCATCTGCAGGGTAAGAAGGAAAAACGTCCACCTTGCAAATATATCCCAGGCACATGCCTCCTGAGAAGTGACCCCCTCCTCCCTCCGCCTCCCCCGGCACGTCCTGGGGCTTCTCCAGTCTCCCGCTCCTGGGACCAGGCAGAGATCCC
FP005826	chr5	141373715	141373966	+	1	test	TACACTCTTCTAAATGCTTCATTATCTTGGGGAGGGAAATATTATGAGTGTCATCTCTGCAGATTTAGCAGAAATAAAATCCTCTGTGTGATAGTTTCACAAAACGATGCAGTATTAAGTTAGGACTCTAAGCGTCGCTGTTGACCAACCTGGGCAAGAAAATCAACGGAAACTCAAGTTACATCCTCCAACAACAAAGCAAATTAGACGGGAAAGCAGGAAAGCTGTGCAGAAATTCTGACCTGAAACGC
FP000746	chr1	54548054	54548305	+	1	test	GCAGCCTCGTCCTTCCTCTCCGCTAGGCGGGCACTGGAGCTTTCTGTGCAGGGCTCCTAGGAGAAGGGGGGTAGAGGGCAGTCTGAAGAGAGGCGGGACGCGGGGTGATAACAGCTGGCTCTGGTGGGCGGGCGGGAGCTGGGGAGGAGGAGCAGGAGAGGCCCACAGGCTTCATTTGGAGTCAGGCCTGGCTGTTGCTCAGGTGACCAGCTTGTGTCTCTGGGAGGGCGCTGCTTTCCCCGGCCACCCGG
FP017001	chr19	18606748	18606999	-	1	test	GGGGGCTTTGGGCCGCGGGATAGGGCGCGCGCGGTTCTCGGTGGGGCGGAGGCGGGACGGGCGGGGCGCGCGGTGCCCGCGGGCGGGCGGGCGGCGGGGAGGCCGGTGCGGGGCCCGCCGCCCCCCCGGGGCCGGGCCGGGCCGGGGGCGGGGCCGGGCGGGGCCAGCGGCGCATTAGCGCCTTGTCAATTCGGCTGCTCAGACTTGCTCCGGCCTCCGCGTCCGCGCCCAGCGACGTGCGGGCGGCCTGG
FP019731	chrX	153646983	153647234	+	1	test	GGGAGCCGCAGCTGGTCCCCTGCTGGGCCCCTGACCTGCCCTTGGCCTCTGCCCCGACCCCGTCCCGCCTCGGTGGAAGCCCCAGCTCAGCCGACTCGCAGGTCCCACCCGACTGCTTCGGCTAGCACCCGTTGGCTCACCGGCCCGCCGGGCCCGCCCCCGGAGCCCCGCCCCGGCCCGCCCCTCGGAGTTGTAACTCCACGTCCGAAGGCAGTTTCCAAGGTGGAAGCTGGGTCCGGCTGCCAGGAAGC
FP002939	chr2	189441251	189441502	+	1	test	TATATAATAAAAACTGATTGTCTCTAAATCTCTGTTTTGCTTTCACCTTTTCTTACAAGTGTCTTAGTTTGACATTGCCCACACAAAAGGTGCCTGCACGGAAGCGGAGTCCTACCACCGAGTCTCAGGCCTAGAAGGGCCCGGGGGTGGGTTCTGTGGGCTGGGAACAGCCTTATTACCCGGCTCAACGAAGGAAGTTCAGGGCGGGACCAGAGATTGGCCATTCCGCTACTGCGCAAAGATGGTGGAGG
FP015169	chr17	4997534	4997785	-	1	test	CAGAAGGGCGAGCAGACGCCGCAACCCAGGAGGGTTGCAGACGCCTGAAGAACAGGGAAACGTCGGGGGCTGACCCGCGGGGGCACCCAGGCCCCGGCCCAGTTCCCGTCGCCGCCGTTGGAGGCGCGTCCAGGGGGCGGAGCCTCCTGCCGAGGCCCGCCCAGAAGGTGGCTCCGCTCAGGCCCCGCCCTCAGACGAGCACCGCCCGCCGGCTCCTCCCGCGGCTCGAATCGCCGTCTCTCTCCTCCCCG
FP009676	chr10	23438941	23439192	+	1	test	GGGCGCGGGCCGGAGTGGGCGGGGCGAGCAGGCGCGACTGAATCGGGCGCCGCAAAGAGCCGCGCATTCCAGTGAGTCCACGTGACGCGGCCGCGAGGCCTGAGGTAAACAACCGCGGCCCCGCCTCCCGGGCCTCCGCGCGCACCGCGCTCCTCCTCGCCGGCGGGACGCGCTCCAACGGGGCGGGCGGCTTCCTCCGTGAGTCCCCAGCGGCCGCCGCGGGCCGAAGCAGCTGCAGCGGGCGCGGCGCC
FP009346	chr9	127916881	127917132	-	1	test	CCGCGCTCGTAGGATCCGCCTGCGGCGCGCAGGCCCCGCCCCCGGCGCGCGAGTCTCCGCCCCTCCTCGCCGGCCCCGGCCCCGGCCCCGCCCCCTGCCTCTGGCCGCGCGGATCAGCTTCCAGCCCAGTCGGCCCGGCCCGGGGGCCATGGAGCTCCGAGCGGCGGATCGCGAGCCTCCTGCGAACCCCAGCCTGCACGCCCGGTTAGCATTCGGCCGGGAGATGCGGCAGTGGAATCTGGAAGGGCGGT
FP003957	chr3	119186473	119186724	+	1	test	CTGCACACTCCTTGCCTGAAAGAACTCCTGAAAGGCACTGACCAGGCTCTGGAAGGCTATATTCTTTTCATTCTTTATAAAAGTTATGTGATTACCAAGCTCAATAATCAAATGTCCTTCATCGCTAAAAGATTATATGGCTTTGGTTTGGATGCTTTGGGAACAAGGAGGCAATACGCAATGGCTTGGTTTACTCATGTTACAGAAACTGTAGCAGTTATTTGGTAATGCTTTCAACAGTGCCTTCAGCT
FP014062	chr15	79090721	79090972	-	1	test	TGAGTGTGTGCGTGTGCGTGTGCGTGTGTGTGTGTCTGGGGAGGGGGAGGAGGCACCAAGTTGCAGCCTGCGCGCGGCTCTGCGCCTCGGCGGGGCTGCGTGCGTGTGTCCCTGTCCGCGCTGCTGAAACGGGCTCTTCGAGGGATCGGCGTCACATGACTCCGCCTGCCCCTCGCCAGCGCGCAGATCGCCAGTCCCTCAGTTTGCCCGGCACCGGAGGAGGGTCGGGCGGCATCTTCCGGGTACTGGGG
FP007548	chr7	90466166	90466417	+	1	test	AAGTCAAAGAAACCTTGCTTGTTCAGTTTCATATTCTAGCCCCTTGTGCAAACACCCTCAAAATTCAGTATTGGTGGCCCTCATTGGCTAGTTATATCTTGTAATATAATTTCTTTCCTCCTTTCTCAGGCCCAGCATGTCCCCAGCTGGATTTAACCCTAGTTATTGGCCTAGCAGCCAAGAGAAGAGGTCGGGGTTCTCCTGAGAGGACACCTATTAAGATACCCTTATTGTGTTTTCCAGCGGAAAGG
FP006890	chr6	123803802	123804053	+	1	test	GCCGCGCTGGGGGGCGTGAGGCGAGCGGCGCGGAGAGCGGCAGGGGCGAAACTTCGCGGGCCAGATGCCCGAGGGCGCGGCGGCGCTGCCAGGCTGCCGCTGCTGCCCCTGCGGGCCCCGAGCGCGCCTCCGCAGGCGGCACTGCCCGCGGCGCGGCGTGTGCACCGAGCGAGTGAAGGTATGTGTGGCGGGCGCGGCTGGAGCTGCCGCCGCCGCCGCCGCCGCGCCAGCAGGTCCTAATGCCTGTCACT
FP007064	chr6	158017204	158017455	+	1	test	TCCAGAAGAAAAGGAAGTCATTAAAGGACAGTATGGCAAGCTCACGGACGCGTACGGCTGCCTGGGGGAGCTGAGGCTGAAATCTGGTGAGTAGCCGCTCGCTGGAGGAGCAGGCGCCAGGCTCCCCGGTGGGCAGGAGCCTCTGTGTCGGAAGGGGCCTCAGTGCAGGCATTCTGTTTGACCGCTCTTCTCTCTCTCTTCTTTTTTTTTTTTTTTTTTTTTTCTCCGAGATAGAGTTTTGCTCTTGTTGC
FP019447	chrX	100732069	100732320	-	1	test	TTAGACCCCTATCCTCAGTCCCCAGGCTCCCATTCCTAACCTGGAAGGACATTCCTCTTGTGTCTCAAGGCTTTGGGTCCCAGAGCTCCCTTTTGGGCTGAGTGGCCCCCTGCTAAGGAGCCCGGGGCGCTGTGCGGCCCCTTTAAGAGTCCAGCTCCTCCCCGCTCCAGGGAAAGTTGCAGCTTTTTCCGCCTCCCGGGAGGGTGAGAAGGGCCGGCTGCCGGAGCTGGGTTGCGATCTTCCCGGAGCCG
FP016932	chr19	16173515	16173766	-	1	test	TAACCCCAGCCTCACCTTGTGGTGGCAGATGTGGCCTCCCCACTGAGGTCTTCTCTGTGACATCCCACAGCCCTTATCCTTACCCATGGACACATTCAGGCTGTTGTCCAGAGAGCCTGGCTGTAATTTGAGTCTCATCAAGCAGGGGGTGATGGCGTAGGTGGGTGGGTGACAGCCAGGGCAGGGAGATTTGCCCAGACACTGGTACACAACAGCCTGGCAACCCGTGTGGCCCCGAGGAAGCTGGGAGC
FP005984	chr5	168291929	168292180	+	1	test	GGCCGGCGGCGGGAGGAGCGGGCGGCGCCGGGTCGGGGCTGCAGGGCCGCATGGACAGCGGCGCCACCCCGGCCGGCCCCTACTAGGGCCCCCCATCTGCGGGCGCCACCCCCCGGATCATGGTGCCTCGGCGGCCGCCCGGGCTAAGAGCGGCCGGCTGGAGCCGCTGAGCCCCCGCTGCGGCCGGGAGCTGCATGGGGGAGCGCCGGCAGCGCTTGGGAAGATGCCCCGGCCGGAGCTGCCCCTGCCGG
FP018036	chr20	33686332	33686583	-	1	test	GCGCGCGCCCCGCCCCGCCATTGGCCGTACCGCCCCGCGCCGCCGCCCCATCCCGCCCCTCGCCGCCGGGTCCGGCGCGTTAAAGCCAATAGGAACCGCCGCCGTTGTTCCCGTCACGGCCGGGGCAGCCAATTGTGGCGGCGCTCGGCGGCTCGTGGCTCTTTCGCGGCAAAAAGGATTTGGCGCGTAAAAGTGGCCGGGACTTTGCAGGCAGCGGCGGCCGGGGGCGGAGCGGGATCGAGCCCTCGCCG
FP006532	chr6	41736197	41736448	-	1	test	TCAGCAAGGGATCTTGTCCCTTTGGACTTCATCCCTGTCCTCCTCAGGAGCCTCTTAGCAAAAGATCCAGAAACCAGGAGCAACATTGTATGTGGGGAAGGAGGAGAGAGAGGAAGAAAAGGAGGAGGGGAAGGAGAAGAAAAGCAGGGGAGGGGCTGGGGGAGGAGACAGGGAAAAAGGGGCGGGGAAGAGGAGAAAGTAGAGAATGATGCCTCCGCACCCTGTGAACTTCCAACAAGGGAAGGTGACAT
FP002820	chr2	167135772	167136023	+	1	test	ACCTGAAGCATATTGAAAACATCCCTCATGACAGAAATCCCTCCCTCTTTCATTTCTGCCTTCTAACCCCCCTAAAAAACACATCTGTTTCCTACTGATAGCTTTTAACATACAACCCTTTAATGTCTTGTAACAGGAAGTGGAAATTGAGCGAAGTTTGTGCTCGCCAGCTTTTAAGAGTCACCCTGGGAGCCAGCTGGAGGATTCTGTGAAAGATTCAGACAAGAAAGGCAAGGAAACATCTTTTGACA
FP007931	chr7	143299776	143300027	+	1	test	AAGGTCTCTTCTTTTATCATTGACCACAGGAATATACATAAAGCGTTTTATCTTCTAATAGCTTAGGGTTTCAAAAAGAAACCAAACTTTGATGCTTATGTTGGTGCTGACCTTAGTGCACAACACTAAACATTCCTTTCTTTTAGGAAATGCAAGAGAAACTGCAGAATTTTGCACAGTTACCTGCACACCGAGTCACGGACTCCTGCATCGTGGCACTCCTCTCGCATGGTGTGGAGGGCGCCATCTAT
FP008740	chr9	2015146	2015397	+	1	test	GAGTGACAGGCGCGTCCCGCCAACCCGCGCCCGGACGGGCAGGGAGGAGCGGCGCGCGGGGCCAACTGCGGCGCGTCTTCCGGCGCCCGCGGAGGAGGCGAGGGTGGGACGCTGGGCGGAGCCCGAGTTTAGGAAGAGGAGGGGACGGCTGTCATCAATGAAGTCATATTCATAATCTAGTCCTCTCTCCCTCTGTTTCTGTACTCTGGGTGACTCAGAGAGGGAAGAGATTCAGCCAGCACACTCCTCGC
FP007285	chr7	29806310	29806561	+	1	test	TAGGCCTCAGTCTCTCCGCAGCTGAGACTTGGGACTCTGTACCCGGGAGGCGCGCCCGCCGACTTGGGACCCGCCTCCAGGGTCCCAGAGCGACCCCGCCCGCCGGGGCCGTGTCCCTGCCGCGGGGCGGCGGCGGCTCCCAGGCAGGAGCCTGCGAGCAGGTGGGGGCGGCGGCCCCGCCTCCGCGTCCCGGCAGCACCAGAGCCGGTGGCCGGGGAGCGAGCCGGGCGCACCGAGCGCAGCTCGGCGGT
FP002545	chr2	99304498	99304749	-	1	test	ACAGGATGTTGCTATGTTGCCCAGGCTGGTCTTGAAGTGCCTTGACCTCCTAAAGTGTTGGAACCACAGACGTGAGCCACTCCACCCAGCCTAAAACTTCATCTTCTTTGGTAAATGTGCTTTCATCTTCAGCATTCCAGACTTAGTACCATCCTCCCAGGCACTTATGTCCTCTTCTTTGCTCTCATCCTTGGCGTCTTGTCCCAAACTGAGTCCTGTCAATTCCACCTCTAAAATAGATCTGTAATATC
FP001832	chr1	223712338	223712589	+	1	test	GGCGCCGGGCCGCGCCATCCCGGGAGCTGTCCGCAGATGGCAGCACCGGCCCCGGGTCGCGGCGTTCCCGGCGCTCGGCAGGCCGCAGGATGGCCTGGTCCCGGGCCGGGAGCCCAGCAGGCCGGGAGCGGCTGAGGCCACACCCCGCGGGCCGGGCCGCTTCCCTCCGGTGAATCATCGCTCGCAGCGGCGGCGCCCGCAGTGGCCGCAGCAGCGCGCCGGGCCCTGGCCGCGCCCCAGCCGAGCGCAGC
FP008918	chr9	37034217	37034468	-	1	test	TCTCCCGGCTTCCCGCTCTACTCCGGCCGGGCCGGGTCCGCCACGTCTGGCGCGCTGAGCAGGCCCGGCCGCGCAGCGCCTACCCTTTCCTCGCTCCGGGCCGGCAGTGTGGGGCGGCGCGCTGGGGGCGCGGCGTGTCTGGGGACATCTTGTGATGTTGGCGAGAACAGGACATGATCTCACATGGCGAGAAGCTCTTTAGTTCCTTAATCATTTCACGGTGCCTTCGGACGCTTTTTTTCCACCTAAAA
FP011496	chr11	129024137	129024388	-	1	test	GTCTCCCACCTACTTAGCAAACAACTGAAAAGCAATTAACTTTTAAAAAGAGAAAAGACATAATTTAATATCAATTTCATACTGGTGTTTTCTGAGCAGGAAGTGCTCACCTGTGTGGGGTTGATGGGTGGAGCTCTTCTGGTGAAGCCTGTTCTGCAAGTGTGAGTTTAGGCTGGGGCGTTTCTGTGGTGTTGCAAAGCAGTATGTGCTGAGAGAGGAGGATTAAGCTCCTGGAGGCAGAGCTCTCCCAC
FP014023	chr15	75368556	75368807	-	1	test	GGCCCACTATTTCGGACAACATAGGAGAGCCTCAAAAACAGTTTCTAGCTTCCAACTCGGCTCAAAGTTCAGACAGCAGCTGGGCTTCCGAGAGCCGCCGGTGTTCCAAGAAAAGCTGCCCGGGACTGGGAGGCGCCCTCTGGACGCCGGAACCTTTGGTTTCCTGCCGTGGAGCGGAACGGTAGCGCCAGGGGCCTTCCGGAAAGAGGAGCGAGAGCCCGGCGATGGCGGCTGCGCCGGCCTTGAAGCAC
FP002159	chr2	28751653	28751904	+	1	test	CGGCAGCGGGAGAAGCCGCTCCAGTGCGGTTGCGCGGCGGCCCAGGCGTCGAGGGGAGCCTCGGCGGGGAGCGCACCGCGCGCCTGCGCGGAGAGCTGCGTGACGCGGCGGCGCGCAAGGGACGTGCGGAGTGAGTGGCGCTGCGGGTGGGGCCGTCGGCGGCGCTGGTGAGCTTTGCGGAGCTGGGCGGTGCCGAGGAGGAGGAGGTGGCGGCCTGGGTCTGACGCGGCCCTGTTCGAGGGGGCCTCTCT
FP002903	chr2	178453076	178453327	+	1	test	CAAATCACTATCTACTGTGATCCTTATCATAGACATTGGTCATATTTTTACGTTATTAGTTTTCTGTGTATATTTTAGTGCTATAGCCTGCCATTCCAAGTTAAACATGCTCTTAGCATATCATTTTACACCGTTGATCTATAATTCATTACCTGAGCACATGAGCAGAGGCAGGGAATTATACAGTGGTAAACTACTTGGTGAAACCATGCTTTGTATTTTTAAGATGTATAATTATATTTAAAAACAAG
FP004142	chr3	138707481	138707732	-	1	test	TAGATTTGATAAGAAGAATGTCATTGATGACTTAAAATACCAATTTGGTTATTGTGACAACCATGAAAACCAGATTACATGTGATTAAGAAGTCCATATGTAGTGATGAGATTATAGTTTCTATGGGCCACCTAATTTTGATGGAGAGAGAGCAGTTCCAATTAAAGTCTTTTCAAGATTGAGAACTAAATTTGAATTAAGAGGTAAAGTAGTCATTCAGTTTTGTTAACTTTCTCATTTTTAAAACATGT
FP014120	chr15	85380404	85380655	+	1	test	TCCCTTAGCAACCTGAGTGCCCCGTCCAGGAGCCTTACTATTGGTCTCATGCAGCGAGATGGGCAGTTCTGCCGTGCAATCCCAGCTCGCAGTTCTTGCTCCGCGTGTACTCACGGGAGGACTCGCAGGCGTTACCGCCTTCCCGCGTGCCCCGCCCACCCTCCCGAGCGTCTTTCGGCGCATGCGCGGACTGGAGCTGTGTGCAGGGCCAGCGCGGAGCCCGAGCAGCCGCGGTGAAGCGCCTGTGCTCT
FP009888	chr10	70233305	70233556	-	1	test	GAGGGGCGGGGCTGGGGAGTGCGGACACCCGGGCCTGGCGGAGTGCGCCTGCGCACGGGGTTGGCCCGGCGGGGGTGGGAACACTAGCAGAGCCGCGTTAAAGGCGCTCCCCGCCCCGCCCGCCGGTCCAGTGCTCGCAGTGCGCAGGCGTGGGGCTCTCTCCTTGTCAGTCGGCGCCGCGTGCGGGCTGGTGGCTCTGTGGCAGCGGCGGCGGCAGGACTCCGGCACTATGAGCGGCTTCAGCACCGAGG
FP016899	chr19	14529397	14529648	+	1	test	CCTTCTCTTTAGCCCCTCCTACAGGCGTTCCGCCCCCTTCGATTGGTCTCGCAAGTTGATTGGTCAATCCCCTGGTGCTAGTCCTAGTCTTGGTTCGGACCGGCCCCAAGGAGCAGGGGCGAACGTGGGCGCCTCGTGCCCTGATTGGCCGACGGGGCGCGCGCGGCCTGGAGGGGCGGGGCGGACGCAGAGCCGCGTTTAGTCTATCGCTGCGGTTGCGAGCGCTGTAGGGAGCCTGTGCTGTGCCGCGC
FP016200	chr18	3874769	3875020	-	1	test	ACTTAGAATTCGCTTAATTTAGGCCATTGCCATGATTCTATAGAAAATGAGAAGTCTATAAGTATTTAGCCTCTGCCTGACATTGACTTAGGAATCAAATACTGTAGCTTGAAATCGTTGCAAGTCCCAGTTAATACGGTCAATTGTATTATGAGTGATAAGTGCTCACTCACAAGCTGTCTGCAGTTAATAAAAAAAGGGACTCCTTTTTTTTCCAATATTGTTATTTTTAAATGTCAAGAATCAGATGT
FP015423	chr17	28726867	28727118	-	1	test	CTGGCTCTCAGCTGCTCCTCCTTTCGCCCAGAGTTTATACCTAAACTCTCCCCGAAAGCAGCTGCCTCGCTCGGAGGCGGGAGGGGGAGCCCGCGCCACAGTGAGGGCAGGCTGACACCTAGTGGCTAGGAGCCGCAACTGCTGCAAGCTACCGGCTTGGAGAGCGCGGGTCACTCCAGCTGTCCCGGCAAATCGGAGGTCTTGCTTGCCCAGCCTCCGGGGAGCCGGGCCGAGGGCCGGGACTGTCTGAG
FP017838	chr20	833699	833950	+	1	test	GGGGGCGGCGTGCTCACTTCCCCTTCCACGGCGCCCCAGGGACTTTCCCTGCGGTGGGGGCCTTCGACAGTGAGGGCCACCCGCTCAGCTCGACCGCGGAGGGCAGCTCCAAAGGGGACCCCAAAGGTGCCCACGCGGGGCTGGGGCCTCCTGGGCGTCGTTGGGAGCGGCCACTACCGGCCCGGGTCCGAGCTGTCAGCCTCTCCAAAGCCTGCGCGAGAGGAGCCGGGACACGCCTAGCGCGGGCTCCA
FP003501	chr3	37451940	37452191	+	1	test	GCCCCGTGCGCGGCGGGGAGGGGGCAGTTCGGGGCGCCCCGGCGGGTCCCAGGGGCTGACTCCCGCCCGGGAAGTGCGCGCAGGGCTCACGGCCCGCAGGGGGCGCTGCCGCGCCGTCCAGCCGCCGCCGCCCGGGCCGCCCACCTCGCGGGCGCGCCCCTCCGCAGACCAGCGCCGCCCGGCCCGGGACCCGCTCGCGCAGACTTCTCGGGCGGACTGAGGACGCCGCCGCTCGGGGCCGCCCCTGTGCT
FP003833	chr3	72887913	72888164	+	1	test	GAAGTGGTTTTGAGTCTGAGCTACCAGAAGCCCGAGAGCCTGGCTGCGGGCGGGCGGAGGGGCGGCTACTCCACACAGTACGGTAGGCGCAGGCTGCTCCTCCTGTAACCCAGCTCTCACCGCCTCCCCCTCCTCTCCTCTCTCTCCTCCTCCTTCGCCGTCGCCGCCGCCGCCGGCCGCCCGCCGGCCGCCACGACCCCAGTCCCCGGCGGGCGGGCGGAGGAGGCGACCGCCGCGCGCTGCTGCACTCA
FP001046	chr1	111542025	111542276	+	1	test	TCCTTTCTTTAAAGAGGAATACCTCTCTTAACAGCTTAACTCAGGGTGGTGCAAAGAAGCTTTTTTTTTTTTTTTGAAGGGTTTTTTCAAATGTTTATTTTATGTACAAAGAACTATCATGTTTTTAATTGAGTAGATGCCTTGGATAATCCTTTGAAGGAAGATCGTTTAGACCAGCTTAATGAAACAGACATCCTTCGCGTACTGACGGAAACACTGGCGGCACATATTGAGGCCGTATTTCAGGATCA
FP012779	chr13	48233005	48233256	+	1	test	CCAGGCGGCCCAGAGGGACACCCATCCCTCTCCGAGGCGCGCCGGGATCCGGCGCCCAGCTGCGGGGCGAGGGCAGGGTCCCGCGCAGCACCCCACCCCCGGCCGCTGGCCCCGCCCCCGCCCCCGCCCCGCCTCCCAGGGGATGTGCGAGCAGTCTCCGCCCTCGCGCGGGAGCTGGGAGGCTGCGAGATCCCTACCGCAGTAGCCGCCTCTGCCGCCGCGGAGCTTCCCGAACCTCTTCAGCCGCCCGG
FP005260	chr5	14143660	14143911	+	1	test	GGGCGGCACGCGGCGCTAGGGGCGCGGGGCCCGAGGCGGGCGCGGCCGCGGGCGCCGCCGCAGCCATGAGCGGCAGCAGCGGCGGAGCCGCCGCCCCCGCCGCGTCCTCCGGCCCCGCCGCGGCGGCCAGCGCGGCTGGCTCGGGCTGCGGGGGCGGTGCCGGCGAGGGGGCAGAGGAGGCGGCCAAGGACCTGGCCGACATCGCGGCCTTCTTCCGATCCGGTGAGTGCAACTGCGGCCGGCCCGCCCAG
FP004380	chr3	187291782	187292033	-	1	test	CACACACACACACACACAGAGTGATACAAATACCTGCTTGAGCCCCTCAGTTATTTTCTCTCAAGGGCTGAAGTCAGCCACACAGGATAAAGGAGGGAAGGGAAGGAGCAGATCTTTTCGGTAGGAAGACAGATTTTGTTGTCAGGTTCCTGGGAGTGCAAGAGCAAGTCAAAGGAGAGAGAGAGGAGAGAGGAAAAGCCAGAGGGAGAGAGGGGGAGAGGGGATCTGTTGCAGGCAGGGGAAGGCGTGAC
FP009194	chr9	112332938	112333189	-	1	test	GGTCCGGCCGCGGTGCCGAGGTCCGCCCTCCGCGCCCCGAGTCCGGGGAGCCCAGCTCGGCCTCCCGACACTTCCTCTCAGAAAATGGTGCGCGGCGCTGGGGGAGGCGGAGGCGGCGCGCGGGGCGGGCGTCCTCCTGCCGGGGGAGTTGCTGTTTACCGGCGAGGTGGGGCGAGCGGCCGACGGCGGGTGCACGCGTCGGTACACGTCTGGGCGGCGACCTGGTCGCAAGCCATGGTCCCACGCGGGGT
FP001894	chr1	230678487	230678738	+	1	test	TACTCACATGGTTTGATTCTTTCTCTTCTCTCTCTTTCTTCTTCAGAATTTTCTCCTTCTTTCTCCTTTGTGAGGCAATGACGAACTATGTCATTTAATCTGCTGTCAATTCTAATTCCAGTGGGCTGGGGTAGGGGGCCCTAGAACATGAGCACTGCCATTTAAAGAGCACCAGACTTAGAGTGCCTGGGTCTCAGATGGGCTTTGCCATGGAAGACACGGAAGACATATCACAGAACCACTTTGGGCAT
FP007460	chr7	74093024	74093275	+	1	test	CACCAAAGCCCAGGCAAAGGGTGCTTCAGCCACTTCCTGTTGCAGGCTCAGACCAAGTCCCCTGGCACCCACGCGGCTGCAGCCTCCTCCTGTGCGCTGCAGCCACGCTGGCCCCACCCTCTGCAGCCTCCAATCCTGAGCCCCTGAGGGAGGATGGGGAAGCAGCTGGTCTGGCCACCCCTGCCCTCCCTTAGACCTCCAGAGCCCCCAGTGTAGCCACAGAGGATGCTGTTGGCTTCAGCCCCAAGAAG
FP002388	chr2	71276392	71276643	+	1	test	GGAATGTGCTCCCCCAATCCCAGACCGCTCCTGGTGGGGCGGGCGCGAGCGCAGGACCGCGGAGCCTGCTGCGTTGCTCGGCTGCAGGGGAGGAGCTGGGCCCCGCCCCCGGCTTCCCGCGCTCCGCGTCCATTGGCTGGCCTCGGGGGTGGGCGGGTCCCGCCGCACGGAGTCCTGCGCCTTGAGGCTCAAGCGGCGGCGTGCAAGGCGTGTGAGGCGCGTAGAGAGCCGCGCGCGCCGTCTGCGATGCC
FP001672	chr1	202193565	202193816	+	1	test	CCTGGGACACCCTATCCAAACAAATGCAGGTCTCTGACGGGCTGAGCTTTGCTCGCTTGGGTTTCTTTGCTTTCTTCCTTTTTTTTTTTTCTTTCCTGGATGAAATTCCTGCCAGCTCCGAAGGAAGGAGGGAAGAGGAGCGAGCCGCAGTGTAGAAAGTTTCCTTGACTCCTCCTCCGGCTGGGTCTCCCTCCCTTGCCAAGCCCAGCCTGTGAAACTGAATAACGAAGATCACTCAACAATGCCTGCCC
FP010398	chr11	695539	695790	+	1	test	TCCCGCCGCCGGCGGAAGCCGAGTCAGCCCGAGGCCGAGCCGAGACGAGCCGAATGTCCCCGAGGCCGAATGCTCCCGAACGTCGGTTCTCCACCTCTTCCCTTCCGAAAGTGCCCGAGCGGTGCCGGACGGACTAATCGGGCCTCGGCCGTGGCTCGGACGTCCGCTCCCGAAACGCGGCGCGGTCGGGCCCCTTCGTCAGGAGACGCGAAAATGGCCGAAGGAGCGCGAGCGCGCGGGCCGAGAGGCTG
FP002614	chr2	110678012	110678263	-	1	test	CACCCCGCGCTCGCTCAGCCAACGGCCAGGTTTCGGTTCAACCGAAAACCACGGGAAGTGGGAGGAGCTACTGGCTCAAGGGAGGGAGGTGGGACTTGACCTCCGAGCAACGGCCCCGGTGATTGGCCAACCTTCTGCCGCCGCCACCAATGGGCAGGCGCCCTGAAACGTTCGGCGAGCCGACTGCGGCTGCGCGGGGTATTCGAATCGGCGGCGGCTTCTAGTTTGCGGTTCAGGTTTGGCCGCTGCCG
FP000168	chr1	11926352	11926603	-	1	test	CTTAACTCCACTGGACTTCACTCTTCCGGCCTGGGAACGAGTGGTCCCGCCCCCGGCAGCTTTTGTCCCGCCCTCGCCGTGGCTCCGCCCCCTTCTCTCCGGCCTGCTCTCTCTCCTCCCTCCGCCCTCCCGTTTTCCATCTGTCGTAAAAGACACGGCGACGCTTGACCGCCGGCGTCGTTTGACGGCTGGAGGCCACCGAGAAGAAGGAGGAAGCAGGGGTGGCGGCTGCTGCGGCGGTTGCGGCGGGG
FP004324	chr3	183635422	183635673	+	1	test	GTCAGCAGCCCTTTGGGTGGGTCTCGTCCTCTTGGGCACCCGACACCCACCCCACGGCGGGAGCCCCAGGTGGCCCCGGAGGCCACCGGGAAGCGTGAACTACATCTCCCAGGGTTCCCCGGGGCGGAGGACGCCCACCCGGATTGGCCAGGGTTCGCTGACGCTCAGTGTTTTGGCCCGGACGGTCACATGTTTCCTTTGTTGTGAGCTGCGGCAGAGACTGGTGGCTGGAGGAGACGCCGGCGCTGGAG
FP016353	chr18	47150442	47150693	-	1	test	TTTTCCTCGTGCATCCCGGCCCAGGAGGCCGCGCCCTTGGGAAAGCGGGTCTCCAGAAGGGCCTAGCTTTCCGTCAGCGGCGGTGCAGACAGAGGACAACACGGCGCGGTAGCTGCACCGCCTCGGCGGAAATCCCGGGCGAAGGGGTGGGGCGGAGCCGGGAAATCCGGGGCTTCCGGCGCGGCCAGGGCCCAACTTCCGGCGTCCGTCGGGCAGCAGCGGGGCTGTCTATCCCGGCTGAGGACCCGCGG
FP015916	chr17	65055611	65055862	-	1	test	GGTGACCCGAGTCACATGGGGTCGCATGCGCTCCCAACTCCCTGCTTCCTTGGCCGTCTCCTAAGCAAGCGAGTTCTGGAACCGTCGCAATAAGGGAACTTTTTGCCCGAGACTTCTTGTCAAGGGCGTAGGGTGACAGCCTCCTTTTGGGCGAGAATATGAAACTTGAGGAGTGTCAGGTTGACTGTACTGGAATCAGGAGAGAGTGGACCCATACAGTTAGCATGAAAATGTAGCATCTGTGCTTTTTC
FP018708	chr22	29767168	29767419	+	1	test	AGAAGAAAAATTGGGTAATAAATCTTATCAGATTATCTGGGCTGTCCTGATATGAACCTTTTGAGTTTTTTTCCTCATTCCCAGGATAGGAATACCTCTGTGGTTCAAAACACGAAGGCTGAGATTCAGAGGAACTACAAATCCCATAAAGTGTTGCACGTCTAGGGCCTTTGGGGAACGAGGTCGGCCCCAGCGCAGGCGCGGTGGCGCGAGTTGGACTGTGAAGAAACATGGCGGCCGCGACGTTGACT
FP008970	chr9	72363861	72364112	-	1	test	TTCCAAAATCCATTGGAACATGGGGCCGGGGGTTCTCGGCGTCGTCTTTTGGGCTTCTCGACGTCGCTGCGTTTTTATCCTGAGATCCGTTTTCATGGAGGAAATGGAACACCTCCTTTGTTTGGTGCATTTAGGCTGATGAATGTGAAATTTTTTGCTCGCTTTCCACGCACTCCGTGGAGGTGTGTTTTCGGGGAAGGGTCGGATTATCTTTATTGATTGAGTGTCTCCTAAGGGATAGCATCTAAGGA
FP015914	chr17	64897428	64897679	-	1	test	GTGCATCCTGGCGCCCATTTTGCAGGTCAGTTGCTCTCCCTGGAAGGAAGAGTGTTCTCGGATTTCACCTTAAAGGAGGAAGGCTGCCAGAACTGAACTAGCACTTCTGAATATCCTGAGGCGAGGTCCGGTGACTTCCTTGGGAAGCTCTGCCGCGCCCCCATCCCACCCTACCCCACCCTACCCCACCACAGCAGGCGCTGGAGTCCTGGGACCACCAGGATCTGAGGCCCAAATCCTTCCTCACTAAG
FP011972	chr12	53180847	53181098	-	1	test	GAGAAGCGCGCGAGATGCGTGCTCGCGCCCGCGTCCCCTCCGAGAGCCCTCTCCCGGGCCGGGGACGCGGCGGGACGCGCGGGCAGGGGGCGGTGTGCGCGCACGGCCTTTCCCCTCCCTCCCCCGGAGATCCCCCTCCCTTCGCCACGCGCCCCTCCTCCCCTCCCCTCCCCCTTCCCGCGCGGCACCCCACGGCTTGCGAGCTACTGAAGCGCTGCCGTCTGTACGGACGACGCTGCGGTCTTTGGGGC
FP014213	chr16	335292	335543	-	1	test	CTATGTTATTGGTGTGCCGTGGTACTGGGTGCTGTGTTATTGGCGTGCTGTGGTACTGAGTGCTATGTTATTGGTGTGCTGTGGTACTGGGTGCTGTGTAATTAGTGTGCCGTGGTACTGGGTGCTGTGTAATTAGCGTGCCATGATACTGGGTGCTGCATTATTGGCATGCTGTGGTACTGTGTGCTGTGTTATTAGTGTGTCCTGGTACAGACTAGTGGCCGAGGTGTGCCTGTGTGGAGATCTGAGGG
FP007113	chr6	169751521	169751772	-	1	test	CACCGCAGGCGCGTGCTCGCTCGCCCACTCCGCGTAGCCGAGCACCACCCAGCCTGGCGCCGACTCCCTGCCTAGCTCGGGATCGAACCGGGCCCTACACCCGCCCTACCTCCATAACTTCCGGTGCCCCGCTGCGGCGCGCGTGAATGACGCAGGCGCCTGCGCGCCCCTCCTCCAGGCGCTTTGGAGCCCTGGCGGGCACTTCCTACCGTACGAGGCGCAGGTGGGAGACTTCCGCCCTCGCGGGACTG
FP010489	chr11	6401570	6401821	-	1	test	GCACCTATGGGGGACCAGCTGTGGGCAGAAGCCCTGGAGGAGCTGTGATGGGCAGGGAGGACGTGGGTGGGGTGGGGACCAGGCTCAGCACTGGATGGTACTGGGAGACACTGAGGTGCCCCTTCCCCAACAGGGACTTTGCCTACGTAGCTCGTGATAAGCTGACCCAGATGCTCAAGTGCCACGTGTTTCGCTGTGAGGCACCTGCCAAGAACATCGCCACCAGCCTGCATGAGATCTGCTCTAAGGCA
FP003381	chr3	12545435	12545686	-	1	test	CATAAACAATGGAAATAAACAAATAGAAACTAGCTCTTTAAGATACAGTCTATGGGGAGGCCTGGCTACTGGTTAATCCTATCTGGTTTGGCTTCTGAAGCCATCAGTCAGGATGACATCACCATTGATCAGGTTCCTTGGGGCGGTGTGCATTCGCCAGGTGTATAATGAGGAAAAGGAAGTCTCCGGAAACCTCCCCTAGCATTCCAGGAGGCGAAAGCTATGCACTGCGCAGAGGCTGGGAAGGCTTT
FP004485	chr4	1347007	1347258	+	1	test	AGCTCAGGGCACGTGCTAGGGGGCGGGGCCTTGACCGGAAAGGGTGGGGCAACGGCGAGAAAGGCCGCGTCGAGCGGTAGGTGCAGTCGTCGCTGTGGTGACCCCAGGTCCTGATGCGTCCGGCCCAGGGCCAGCCAATGGCGGTAGAGGTCTTGGGCACCTGACAAGAGGGGCGGTCCCCCCGCCGGAAGGGGCGGGGCGAGGCGAGCGGCCGCGTCAGCGGTAGGTGGGGCTGTGGTTACGCTGCCGGG
FP008069	chr8	9150605	9150856	-	1	test	ACGGGACCTAGCACGATCCCCGTCCCGGGCTGGCGGAGGCCAGGCGGCCGCGCGGGGGTGCTGGTGTCGCGCTGCTTGCTCCGGCCGGGGTTGCCCCTTTGCCGGCCCCGCCCGGGTGCGATAACGGGCTCCTCCTCCTCGTCGTCTTCCTCCCACCGCCGACATCTCCGGGAACCCAGCCCAGGCCCTGCCTCCCGGACACACCGACGCTCACGTAGTCGCGCTTGCCACAACCCTGCGGGCTCTCCGAT
FP012187	chr12	68809077	68809328	+	1	test	CTGATCCAGGTAAGCACCGACTTGCTTGTAGCTTTAGTTTTAACTGTTGTTTATGTTCTTTATATATGATGTATTTTCCACAGATGTTTCATGATTTCCAGTTTTCATCGTGTCTTTTTTTTCCTTGTAGGCAAATGTGCAATACCAACATGTCTGTACCTACTGATGGTGCTGTAACCACCTCACAGATTCCAGCTTCGGAACAAGAGACCCTGGTTAGTATTTTTGTCTCGTGTAACTTTTAAGAATAA
FP019381	chrX	71301530	71301781	+	1	test	AGTGGTGAGCTGACCTCACCCAAGTTCAAAGCCCTACTCTGCCTGATCCTTTTTTCCTGAGCCTCAGAGCTAAAATGCCCCCGAGCTCTTTCCTATTGGCTGGAAAGACGAATTGAAGTTCCCTTGCCCATGTTAGGAGGTGTACGCCTCCTGAACTAAAGATAGAAACAGCTGGCCCTTCCAGGCAGCTAAAAGCCTCCAGACTAAGAGGTGTTCCCCATTCGGCAGCCAGACTCCTTGAAATACCCTTT
FP018915	chr22	42553717	42553968	+	1	test	GCGCGGACAGTGTGGCGGGAAGGCCGGGCCTGGCGGGTGGGGGCGCGGCCTGGCGGTGGGGGTGGACCTTCCTGGGGTGGAGCCCGCGGCTGTGGGAGCAGGGCCCGCCGGGGCTCGGGCAGGGCGGTGGGAAGGCGGGGCCTTCTGAGTGGGGGCGGGGACTGCTGGAGTTGCGGGGCCTGCCTGGGGTAGGGCGGGGCAGGACAGCTTGGAGATAGGGCCCGGAATTGCGGGCGTCACTCTGCTCCTGC
FP013312	chr14	69153068	69153319	-	1	test	GGCCAGCCCCCCGCGGAGGGGAGGGGCGCCGACGCCGCGTTCTCGGGGCCCCGCAGCCGGCGGAAGGGCCGCGCATCGCGGAAGCGCGGCGGCAGCTGAGGATCAGCTGCTGGAGGAGGCGGAGAGGCAGAGGGAGGAGCCTCGGGGGGAAGGAGAGGGAGGGGACCGTCAGGAAGCCGCGAACGCCGCCGAGTGTCTGCACACCTCGCTCTGCCTGCCATGGCTGGTTAAAGAACCATCCGGATCGCAGC
FP008945	chr9	64638892	64639143	+	1	test	GAGCGCTGCTCCTGCTCCTCCAGGGGAGAGGTCAGTAAGGCAGGGATTTGGCTGGTGCCACTTGAACCAAGTCCAGATGCACTGCCCAAAATAACATCCCTCGTCTGGCCAGCAGTGCAGTGGAGACCGAGTTCTGAAGCAGGCCTTTGTGAGGTCAGAGGTGGAGTTCTGGGTGGGCACTCTGGGCCTCACAGTCCCAGAGGAAAGGCCTGTCCTCCAAGCCAATGTGCAGCCTGACTCTGGGGCCAGCG
FP013631	chr15	31067858	31068109	-	1	test	CACACACTCTTTCACAGTATATCCGTGTATCCTATGACACCAAGCCAGACTCACTGCTCCATCTCATGGTGAAAGATTGGCAGCTGGAACTCCCCAAGCTCTTAATATCTGTGCATGGAGGCCTCCAGAACTTTGAGATGCAGCCCAAGCTGAAACAAGTCTTTGGGAAAGGCCTGATCAAGGCTGCTATGACCACCGGGGCCTGGATCTTCACCGGGGGTGTCAGCACAGGTAAGAGCAGGCCCTTTCCC
FP012318	chr12	100200605	100200856	+	1	test	TACGAAATGAGCGAGAAAGCACATAGTAGTCAAATCTCGCCACCACATAAACAGGTTAAGTTTGGAAGCACAGCCCCTAACATTCGGTGCTGCTTAGCATCTCCGACTTCCGGCGTCCTCCGTTGCTAAGGTGACCAGAGGCGCGTCCTAACGCTCCGCCCAACAAACCATAGAGTACCGGAAAAAGCTCGAGCAGAGGGGTCGGAAAGGGAAAACAACTACGGCTGCGGTGTGGTTGGTGGTGAGATGAC
FP017024	chr19	19192518	19192769	-	1	test	TTCTTTTCCCCTCCTCACCGACTCGGAGGGAATGGGCGGGGCCTAAAGGGCTCTGAGAATTTCACTACGCCTGCGCCCGCTCACCTTTACCCTGCCCACAGATTTTTCCAGAGGGAGCCAATTGGCTCCGGCTTCGGGTAGATGGGCGGGGCTATCTCAAGAAGAAGAAGTAGGGGAAGTACGCGGGGCGTCTGGAACTTAAAGGGGCAGCGTACCTTGGCCGTCCCGTTCCACAACAAGGTCCCTTCTGC
FP007196	chr7	8262318	8262569	-	1	test	GCAGGGGCGGGGTCACTCTGGGCGGCGGATCCGAGCGGGGGATACCCCAGGAGATGGGGGTCGAGGAGAGACCCCGGGGAGTAGAGAGAGAGAAACTCACTCCCCGAGTCCCCGACCCTCCCCAAGCAAGGTGAGGTCAACGCCACCCACCTTCCCCCGGTCACCCTGGGAGGGGTCGCTCCGGCGGGCGTCTGCTGGAGGGTAGAAGCGGGGAAGGGAGGCAGCGGCGGCGCCCAGGCGGCGGTAGGTAG
FP001068	chr1	113904843	113905094	-	1	test	ATCCGCGATGAGAACGGCGTGGAGGGGGAATTAAGGCTGGAACTGGAGAGGGAGGGGTCTGCGGTCAGTCGGACGCGGCGCTGGAAATCCGGCGGAGGACCTAGAAGTAGCGGGAGCGATCTGTGGGCGGGGCGAACGGCCTGCCGGCCGACGGTAGGGCCTGAAGAGCCTAGGGTGGGGCTCGCGGTGTCGCGGAGGGCGTGGAAGTGTGTCAGGGCCAGCGTGCCGGCCCTACGGAAGCCGAGCCTGAG
FP015786	chr17	50150661	50150912	-	1	test	CCAGCCCCGCTGACACCGTCTCGGACGATGTGAGCCGGCGACACCGCGGCTGCTTCGCGGGCAGGGCGCGCGCTCCCACCGGCCCCAGACGCCGCTTTGCTAATTTGGGTGGGGGCGACCGTACCAGGCCGCTCGGGCTTTTGAAGCGGGGCTGGCTGTCTTCGGCAGCTTGATTGGGGGAGGCGACCCACACACAGGGCGCTTGGAGATTTTTTTTTTTTCATTTTTTTTTTTAAAGCGGGATTAACTGT
FP000940	chr1	93447936	93448187	+	1	test	GGTGACCAGGTGCCCGCTTGGGGACCCCGCCGGCGGTGGCCAGGCTAGCCCAGGTCCTCCCCGATCCCAGGATGCACTGCGCCCCGCCCCCTCCCCGCGCCGCGTTCCGCCTCCCCTCGGCCCCGGCCCCCTCCCGCCCAGCGCGGGGAGGCGGGGACGCGTCGGCGGCGGAGGCTTCTCCAGTCGCGTCTTTCTCACTCACTGGGGAGCCCGGCGGTGGCGGCACCTTTCGAGGTAGACCCGCTGAGCTG
FP005868	chr5	144170672	144170923	+	1	test	TCGCGCAGGCGTGTTAGCAGGTTGGGCACCGGTGGGCGGGAGGAGGAGACACGAGTATGGAGCTTGCGCAGTATCTACCCAACCTGCGGCTGGCCCGTGCCAAAGAGGAAAGTGCTTCCGGCAGCCTGGTTAGGGGTGGAACCAGGAAAGCGCAGGCGTAGCAATTGCTGCTTACTCGTAGGTAGGAGGAGGAGTTTCTCAGACCTAGGTGATGGAGGAAGACTGGGAGGCGCTAAAATGAGACAGACGGA
FP009011	chr9	84671886	84672137	+	1	test	CTGTGTCACCTCTACTGGACTGTTATTGTTATCATTTTTTTCCTGAAGGATGCTGTCATTCCATACATCTCATTAACATGAGATTTTATTCGTATACGTGGAAGATTGTAAGGCAAAGTTTCTCCCAGTTTTCCTTATTGTGTTTGATACTGAGATAGGCAGCCCTAAAGTGGGGCTTGGGCTGTGGCCTTCCGGTTGTCGAGTGCGCATGCATTTTTCCTTCCACAGAAAACATACAACCATGTTTCTGC
FP011783	chr12	29148991	29149242	+	1	test	CAACTCCCCTCCCCCAAACCACACAAATGCGGGAGTCATAGCCAGTGGATCCGCTGGTGCTGAACGAATGGAAGCGCGGGGCGGGCCTGCGCGCGCGGCCGCCCTGCCCACGATGCTGTCGCCCCCGCGACCTGTAGGGCCTTCGCTGGCCCCTCACCCTCCTCCCGCCCCGGGCTGAGGCTGCCGCCGGTTTCGCGCCCAGTGCCGAGGCAGCGGCGGAGCCGGGGAAGGTCGGAGGGAGGGTGGTTTCC
FP006044	chr5	177086674	177086925	+	1	test	GGGTCGGGGGCGGGGCTCTCCAGGTGGGCGGGGATCTTGGCCACCCCTGGCCACACCTCTCTCCGGCTCGAGCTGGTCTAGGCGGGGCGGGCCCGAGGGGGTGTGGCAGGAGGTGGGCGGGCCCGGGTGGGGGGGGGGGGGGCGTGGAAGGAGGGGCGGGCCCGAGCAGGAGGGGGCGGGCCCGAGGGGCGGGGTGGGACAGGAGGTGGGCCGCTCGCGGCCACGCCGCCGTCGCGGGTACATTCCTCGCT
FP006960	chr6	137044981	137045232	-	1	test	GGCTGAGTGCCTAACTGTATATCGAGAAATAACGCTCTATTTCATCCGTCTTCCGATATTGAACTTTAACTGACCATCAGACCGGCCTGAATACCAGAATCTCTTTGCAATGCACTCGCCTCGCTCATTACCAGAAGGAGGTTTTACCTGCCTAGTGACCTGGTTGGTCTGGGCATGAACCCCGAGAGGTGGAAACTCTCAGACTCTCAGGACTGACAGCCGAGTGGCTGGGGACTATAAATCCGCGAGAA
FP002226	chr2	42567939	42568190	+	1	test	GTGTTGGACAGTTGGTTCTGCTCAGCCCAGTTTTGGGGTAAACGATGACGCAGTTAGGGTGGCCTAGGAAGGGAGTTTTTCTACCTCTGTCCCCAGCTGGGTCTCTGGCGAGGCGGGAAGCACCCGGAATCTTCCTGGCCCTAGAGCCTGCAGGCTCCAGGCCGGCCCCTTGAATCTCACCGCGAGGAAGGCACCCTGCTGCCTGCACTTATTTGCATCCAAGAGTTTGCATTGAGACTGGCGCTTGCCTA
FP017729	chr19	55376625	55376876	+	1	test	AACCTGAGACGCCCCCACTCTTCCACCAGGTCCCTAAAGAATCAACACCCTAGACCAGGCATGGCCAGGCTGGTGTGAAGCCCTTCCACGTCCGTAGCACACCCCTGACTCCCTAGCAACAGAACTCACAATCCTGGGGCCTGGCGGGCAGGCGGGCGGGGCCGGTTGCTGGGGTGATGTCCTGTAAGGCCTCTTCCCACAGTGGCCCCAGGGGTCTGGGGAGGTGACATGTTGGGCTGTGGGATCCCAGC
FP013315	chr14	69259959	69260210	+	1	test	TCCCACTCGGCGCGCTTCTCTCCCAGTGCGCAGTGGCCTGGTGGGTCAGCCGGCGGCGGCTGGAGCGCGGGGCCGGCCCTGCGCACGAATGAATGGGCGCCCGGGGGACGCGCGCGCTCGGGGCTGAAGGGCATTAGGACCGTGAGGATCGCTCCGCGATCCTGTCTCTCCCTATCACCCCCCCGCCCCCCCACCTCTCTCCTTTTTCTGCTCTGCAGGACTGAGCAGCTAGGCGCGAGCGAAAACAAACA
FP008680	chr8	143944660	143944911	-	1	test	CTGGCGCCGTGCGTCATGTTCCAAGTCTGGGTGGTGACGGTCCTTGCGCGCCCTCCTCGGATATTTATATCCCCTGGGCCCCTGCCCACTGCTCCCCTCCCCCACAAGCTGCTGCTGACAGCAGCACGGCCGTGCCCTCCTCCCACCCGACGCTGCCGCCAGTGGCTTGTGCCTCCTCCGAGGGGCCTGTGACCTCCCACACCCCTGGCCCGCTCCGTCTGCCCCGTGGGCTCCTGCCACCGTCCCCGATG
FP015475	chr17	32927952	32928203	+	1	test	TCTGCCGCCCGGACGGGCCACCGCGGGGGCGCGAGGAAGGGGTGTTGGGTCGCCAGGGCCCGCCTCCCAGCTGCCCCGGCCGCCGGGGGGCCTGGCGGTGACGGCGGCGCCTGGCGGCGGGGATTTGGCGCGGGCCGGGGGCCGGGGGCCGGGGCGCGGGGGCGCGAGGCTGGATTCCTAGGGCCGCGGCGCTTCCCGGCATGCTCCGCTGCAGGCCCGCGCCCGCGCCCGGACTTTGCCATCGGCGGGGC
FP013526	chr14	100568253	100568504	-	1	test	GAGGGCAGCGGGTCCCTGATTGAGGAGTCGAGGCTCCAAATGCCATCGCCCTCGGATTGTGTGAGTGTCAGGTTCTGCAATCCTGCACGGCCCCGAATTCCCCAACATTCTCCGACTCGGAGGTTCTGCGATTCCGGGGTTCGGCTTCCTCCGGGAGGGAGCGCGGGGCAGGAAGGGTTAACGGGCGGGGCGGGGGGAGCAGCGGGGCCGCGGCCGCCGGAGAGTTAACGGGGACAGGCCGAGTCCCCCTC
FP011290	chr11	102185838	102186089	+	1	test	TAGTGGAGAAGAAGCAGGTATATAGGTTTACCTTTCTTCTCTGAACACAGCCTTAAATATGTTTCAATGAATTGTTTTATGTTAAGATGTGTTTTCTTTTTAAATTAGTACCCCCTCCCACTTTTTTTTAGGTGCACCAAATAAAAACCATGATTTTTTTTTTTTTTTCTGTATTATAGGTCCTCTTCCTGATGGATGGGAACAAGCCATGACTCAGGATGGAGAAATTTACTATATAAACCATAAGAACA
FP001476	chr1	164589520	164589771	+	1	test	AGAACCATAGAGCAAATTACTTCTTCCAGAGTGCTAAATGATCCCCTGGTGCCCTGGAAAGAGACCGTTTACCTCCAAAAGGTGAGTTGTTTATTGCCAGAGAGTAGAATATTGGCTGTAATGCCTGCTCATCCATCCTTCTCCATCCCTGCGAGGTATTTAGGGTGGAGGAGATATTATCCACCTGTCACCATGGGGAAGGAGACTGACAAATACTTTTCTGATTCCTGTTCCTAAGATTGAATTGCTGG
FP005628	chr5	119355737	119355988	+	1	test	CTCGGCCCTGTGTCTCGGGAGAGTCGGCTGCCGTCTGGGCCTGTGCCCCACCTGCGTGCGCCCGGGCCGAGCCAGCCCCGCCAGGTCGCCTCCGCCCGGCTGCCGGCCCGAGCGCGCGGCTCCGGGGGCGGACTCCCGCCGCCCAGCTGACACGCGATTCGTCACTCTTGCAGAACTTTCCCCCTGAAAATCCCTGCACCAGAACCCGCTCTCCCGCCCCGGGGAGGTTTTGATTTTAGTGGCTTTCTTCC
FP008637	chr8	141136552	141136803	+	1	test	TCCAGAGGTCCTGTCCATTTTCGTGCCTCCTTTTATCAGTAAAGAGGACAGTCAAATGGCCGGTGCCAACTGCGGCACTCTCGGTAAAACCCGGATGCGCTCCTTGAGAAAGAAGAGAGAGAAGCCCAGACCAGAGCAGTGGAAGGGCCTCCCGGGGCCCCCCAGAGCGCCAGAGCCTGAGGATGTCGCCGTCCCGGGCGGCGTGGACCTCCTCACCCTGCCGCAGCTGTGCTTCCCAGGTATGTCTAGGA
FP000639	chr1	44355206	44355457	-	1	test	ACAAGTCCCATAAGGCCGGCGGCCGGGAGTCACCATCTTGACATACCTTAGCGCGCATGTCGGCGGGAGAGCGAGCGTTTGGGGAGAAGAAGCCCCGCCTGACTGCAGCCCATTGGTCAAAGGGGTCGGCAGCGGGCTCAGCGGTTGGCCACAGCGCCCGCCCGTCCGCCGCGTCGGTCGGGGTCAGAGTGCGCGGAGGTGAGTCGGTGTGTTGTGGACTCGTGGTGCTGGCTGCCGCTGTTCAGGCGGCC
FP017183	chr19	36916240	36916491	-	1	test	GCTGCGGGGACTCGGCAGCGTCACCTGCCGGAAACACCCGAATGTTCATCCCGCGCGCAGTTTCTGAGATGCTGGGTGAAGGCGACCCGCAGATAGGTCTGTGACAGACGCCTAAAGCGCCGAACCATCCCTCCGCGCCAGCGACGTTTCCGAGGACAACACCTCCCAGCAGGCCCCGCGGCCCCACTCAACTTCCGGCCAGAACACATCTGCTTTCCTGTGTGAGGGTGAGGCTTGGGCCGGGTTCCCCG
FP004075	chr3	129278783	129279034	+	1	test	CCTGGCGCCGGGAGCGCGGGCTGAGGCGGGGCTGACGTGAGGGGCGGTGACGCGGGTGGCCAAAGCGGAGCGGAGCGGGGGTGAGGAGAGTCGAGGGAGGTGACGCGCGCTGCCGGGGCGAGGTGAGGGGAGGGGAGGCGACGCGGGCGGTCGGGGTATGGCGAGGGGAGGGGAGGGGAAAGGAAGCGACGCGAGCTGACAGAGGGGAGCAGAGGCCCGAGCGGACGCGAGGCGACGCGGAGAGGGCGGCC
FP019766	chrX	154428476	154428727	+	1	test	TCCCCTTTCCCACTCGGAGCGTGAGCGCCTGGAGACACGTACAGCCAACCAGTGAGAAGGAGTGGCCGCGAGTGGCATGCACTTGGTCCAATTACCTGCGGCCCTGCCGGTCGGCCCGCGCTGGGGCCAATGGAGGTGCGAGGCGGGGCTCGGGCGGGGGCAACGGTCACCTGATCTGCGGCTGTCGAGGCCGCTGAGGCAGTGGAGGCTGAGGCTATGATGGCGGCCATGGCGACGGCTCGAGTGCGGAT
FP005537	chr5	90409705	90409956	-	1	test	TAGATTGGGGGGGCGGGGGGGCGTTAGGAGGTGGGCGGGGCACAGAAAAAAAATTTTTTTTTCCTTGATTGACGTAAAACCTACCCAATGAAAATAGAACTGGTAAAGCTTCCGCCCCCTAATAGGCAGAACCATAGCGACATTTCAACCAATCCAGCTCAAAAGTGTCTATGTTGTGGGAGGTCCCTGAGGCCGCTGAGGTCGTTCGTGTCTGTTGAACGGCTGTGGGCGTCTTGCTGCCTTGGGTAGGG
FP006923	chr6	131249945	131250196	+	1	test	TAGCCTCTTTCCCCACACTTAGCCTATCATTTCTATTGTTCATGCAGTTTGCTAGGTAGTTAATGGTACTTCATACAAGTTGTGATTCCAGTGTTCTGCTATGTTGGGTTTTCTATTCATAAACCACAATCTCCTAGGGATAAAAGAAGGCTCTTCCTCTTTAAGTAGCAGACAGAAAATATGGTTTTTTTGGGGGTTTGAATTTTCATGTGTAATTACTGCAGTTGTCTGTAGGAGAAAGGGCTCATTGT
FP004885	chr4	94757754	94758005	+	1	test	CCGCTCCAAGTGCGTTGAGAAGTGTAATCCCTCCGCTAAGTACTTGGCTCTCAGTCTCCAATCACTAGTGGGAGCTGGAGAGGGCAAGTGAAAAAAAAAAATTCCAGAGGCAGCCGAGGAGCGAGAGGGAAAAAAAAAAAAAAAAAAAAAAAAAGTCCCGCGGTGGCGGCGGCGGCGGCGGCGGAGCGGCCGCGGCGGCCAGAAGTTGACGGCGCAGCCGGGCGCGGGGCGCGGAGTCGGCGGGGCCTCGC
FP017614	chr19	51571082	51571333	+	1	test	TCCGTATGGGTGCGGCCATGCCGTCCTCCCTACCAAGTTTCTGAAGTTGAATGAACCGCGGAATAAAGGTACGCCGTTATACGCCATGATAACTGTTGGCGGAGCGTAGCTTCAAAAGTGTTAGTGGGCATGCGCGCGGTTCAACAGCGCCTACGGAAATCCTAACGGGAATGTAACCCAAACCGGAAGTGGCCTAGAGCGACCATTTAGCTTCTGTTGTTAAGTGGATCTAAGCCTATGTCGCTTACTGG
FP017314	chr19	41860054	41860305	+	1	test	GGAAACCCAGGCCTCCACGCGCGACCCCTTGGCCCTCCCCTTTACCTCTCCACCCCTCACTAGACACCCTCCCCTCTAGGCGGGGACGAACTTTCGCCCTGAGAGAGGCGGAGCCTCAGCGTCTACCCTCGCTCTCGCGAGCTTTCGGAACTCTCGCGAGACCCTACGCCCGACTTGTGCGCCCGGGAAACCCCGTCGTTCCCTTTCCCCTGGCTGGCAGCGCGGAGGCCGCACGGTAAGCGGGGGCTCCG
FP002706	chr2	131475918	131476169	+	1	test	GCCTCAGCAACCGCGCAGCCTTTGATGACCCCCGCTGCCTTCCCGCCAATCCTACATCCAATCAGAGAGCGTCCCCAGTACACATGTTGAGCAATGGCCAATCAGAACTGGGATCCGGCCCTCAGCCCGCCTCCCAGGAACTCCGAGCCAATGGCGGCCTGGCACCGGCGGGCCAATCCCGTGCGGCGCGCACAGGCAGGAGGTTGCAGTTGGGCGCTCAGCAGCTGTGGCAGCCGGTTGAGGTCTGGCAG
FP004915	chr4	102577266	102577517	+	1	test	CTCATCTCCAGCCCACCTTGCATCCTCCTCTTGCCATCTCTGCTCTCTGAATAACTGAGGCATATTTATACCTTTGTATGTGTCTTCCTTTACCAAAAACACCCCACCGTTTCCATTTCACCCTCTCGAAAACCAATGCAGAAATCAAGACTTTGTCTAAAGAACACCTCTGTTTTACCTTCCCCAACTCACCTTATTATAAGGCATTTTTCTTTCTTTGGGCTCTGCAGACACATTATAAACTATTATCT
FP015586	chr17	40937029	40937280	-	1	test	AAAGTACTTTCCTTCAAACAGTGACTGCCACAAAGGCATCAGATATTCACCACCTTCTCGGCTGCCTCAGCACAGCAAGCTTTATTCTGGGACCTGAGATCCTGTTCTGAGCTGGCTTTCCCTTCTCCAGGCTCGCTCACCCTCCCTTTAGAGGTGGGGCTTATTGGGGGTGGAAAAAGGTGTGTGTATCTCTGTTGTTTAATGTCTGGTTTTTTGTTGCTGTTGTTGTTGTTTTGTTTTGGTTTTGGTTT
FP003799	chr3	64023421	64023672	-	1	test	TTGGGCTGCACCCACTCTGGGACTCCGCAGGGGGCCTTGAGGGTGGAAAAGCCCCCGTGACCACACAGGGGCGCAGGCGCCCAGGCCGGGTGAGGCGTTCCCGGAGGTGTCCAGCAGAGGACGCTGCCGGCGGAGACGGGACCGGAAGCCGGGCCGCAGGCGGCCGGGCGTATTCGCCGACTCCTCCCGCTTCCGCTGCCGCAGCCGGTCGTAACCAAGTTGTGTCCTGTCAGCCGCTGTCCCCTTCGCCG
FP007908	chr7	140640760	140641011	-	1	test	CCAGGAAGGGTTAAGCCACCTCTCCCCCACCCCGAGAGGCGAGGGGGCGAGTCCGGCAGAAGGTCCTGTTTACCGCAGCTCCGCGCGGGGCCGGGCCCTGGGAGAGGGGTTGCCGTGGCAACCGGCCGGGCGCCCGCCAGCTGCGGATTAGCTCACTGGGCCGGGCGGGATGGGTCGGGAGGAGGGGGCGCGCGTCGCGCAGATCGTCGCGGAGCCACGGCAGGAGGAGGCAGGGGCCGCGGGCGAGCCTG
FP015763	chr17	48908206	48908457	+	1	test	AACACTTGAATTGGGCTCTCCGCTACTTCCGCACAGCGCAGGACCCACACGAGCCCAGAACCATCACGGCCTGCTAGTTCCCACCCCTCCCGTCCTGGAGGACCGTAGGTGGACGCTCTTGCAGAGCGCCTCTCGCTGGTTGGGGCGGGGGTGGGCGGAGCCAGCACCGTCTGGGCTGTGGAAGCGGAGGGGGTGGGGACACTCTGGCCCGGTTCTCGGTGGTGCGGGAGCGGGCGGGAGCAGCGGCCGCT
FP002637	chr2	113756563	113756814	-	1	test	ACCAGAGCCTCGCATTACTTCCCCTAGCTCCGCCAAGCCTCGCTGCATACGCGACCGCGCGCAGTCCGCCGTCGCGCCCCTATGACGTCACCCGCGCGCCACCGCTTCCGGGCTGGGGACTGGGGGCGGGGCCGCGCGGAGGGCAAGCCGGGGGCGGGGCCGGGGCCAGAGCAGATCTCCGGGCCTCCGCGGCCATAGCTGACTGTGCCGTCCCTTCCCCTCACCCGCTCCACGCCCTCCTGGGCCGAGTG
FP007300	chr7	30771230	30771481	+	1	test	GGCCCCGAGTCCGGAGCCGTGGACGGGGCCCTAGGCGACACCAGGGCGCCCTGACTGCGGCCTCCCACTCCGAGACGCGAGTTTGCGGGACGCTCCACCCCTTCCCTCCCGCCCCAGCCCGCCAGCCCCAGGGAAGGGGAGGAGTTCTTGCCGCGCCGACGCCGCCGTCGCCACGGCAACGCGGCCATACTGCGCCGGACAGACCCAGTTGCCTGGTGCTGCGGCCCGGCGTGGGCCTCGTGGGCAGAGCC
FP007929	chr7	143288232	143288483	+	1	test	GCACCCCACTTCACCCCATTGGACCGCGCGGCCGCCGCTAGAGCTCTGCGCCTGCGCACGCACCGGGCCGGGGACTGGGTGGCCTGGTGTGTGGGCGCGGCAGGGCGCAGGCGCAGGCGCAGTGTGCGTCCGCGTCTGAGGGGAGGGATGTGGGGGAAGCGACGGCCCCCGGTTTGTTTGGGCTGTGGGCGGTGCGCAGCGGAGAGCCCGGGAAAAGCGGGAAATGGCGGCGCCGAGCGCGGGGTCTTGGT
FP005672	chr5	132257495	132257746	+	1	test	GCCGGGTTGCCGGCAGCCGGGGCTGAGGGCGGGTCCTCAGAGCCCCCCGAAGTGGGAGGGTCGGGCTGGCCGCGGGGCCAATCCAGGCGCACAGGGAGCGGGGGCGGGGGCTGGGCCGAGTGGGGACAGGGGCGGGCGGGCCGGGGGCGGGCGCGGCGTCCAGGCGGCGGCGGCGGCTGCGGCGGCGGCGGCTCCTCCTCAGAGTCCGGCTCAGGCTCCGGCTGCGGCTCCAGCCCGCGATGCCCCATTCC
FP006931	chr6	132763404	132763655	-	1	test	CTGGCCCATACAACTCCTTGAAAGAAAAGCTCAGGAATCAGAATTGCCAGGCTAGAATAAAATAGACACTCTTTCTCCTGGTGTCTGACTGAAGATGCAGCATCCTTGAACCAGCAGCTGCTGGGACTCCTGTCTGTCATCTCAAAGGGTACGGCCCACCCAGAAAGTGAAATCAAAACAGGAAGTCACCAGGGGTGACTGGAGGAGCACAGGCCTTGGAAAGGAAAGCAGCTGAGATCCAGAGGAGTGGA
FP019623	chrX	132413556	132413807	-	1	test	AGGCAGGTATTGGTCACACACCATCCCCCTCTTCATCCTAGCCTAAAGGGGGAGCTCCAGCAGACATCGAGGGTATTTAAATATACTGTTGCGCCACCGGTGGGTGGGAAGGTGGAGTGGCTGCCAGGGTAGCGAGCCGTGTGAAGCTTTTGGTGGGAGGAGCTATGGATCAAAAATTGCTGTCTCCAACAGTCAACTTCCTTAAATTTCCTTGCAGCTGATCAGGTAAGCTGGGGCTGAGAGTGTCTTCT
FP002171	chr2	31233919	31234170	+	1	test	AGGCGGAGACTGAGACACCGGCACTCTCCCACTGGACTGGGTCAGCATCCCTCTCCCCGCGCGGCCGCAGGTCTAGAGCTGAGCGCCTGCCCACAAACATGGCGGCGCCCTGCGCGGCTTCCCGTCGCCGCAACCGTGGGGCCGGCCCTGCCTTGCCAGTACTAGGGGACTTCCTCTGCGCGCCGGCTTCCTGCCCAGCTGGCATTTAAACCACCGCCTGGGGCTGCAGGATGCTGCTGCGGATGCAGAGC
FP011029	chr11	66568529	66568780	-	1	test	TTGCTACGCCGAATAAAAGGAGCTGGCATATCATGCCTAACATATAACAAGCACGTGATAGAGGTCAGTGACTAGCTTTAGCCTCAGGCCAGCAGCCTCCCGCTCTCCCTACCCTGCGCGCCCCACGTGCCTCCTGTAAACAAGAGAACGCGCAGGCGCCGCAAACCAGCGCCCATTGGTCGGCGTGCCGTCGCCCCGCTGGAGGGAGGACTCAGGCCCCGCTGGCCGCGGGCTCGGTACCCGGTGGGTCG
3934	chr6	29659229	29659480	+	0	test	CCCTGTTTTGAAGCAGCCCTTCTCATGACAGGCTTGCTTGCCAAGGTTCCCTCTGACCTTAAATCTCTTCCTTTTGGTGTCTTGGACAGGGCAGTTCAGAGTGATAGGACCAAGACACCCTATCCGGGCTCTGGTCGGGGATGAAGTGGAATTGCCATGTCGCATATCTCCTGGGAAGAACGCTACAGGCATGGAGGTGGGGTGGTACCGCCCCCCCTTCTCTAGGGTGGTTCATCTCTACAGAAATGGCA
9811	chr16	176758	177009	+	0	test	TGGGGTAAGGTCGGCGCGCACGCTGGCGAGTATGGTGCGGAGGCCCTGGAGAGGTGAGGCTCCCTCCCCTGCTCCGACCCGGGCTCCTCGCCCGCCCGGACCCACAGGCCACCCTCAACCGTCCTGGCCCCGGACCCAAACCCCACCCCTCACTCTGCTTCTCCCCGCAGGATGTTCCTGTCCTTCCCCACCACCAAGACCTACTTCCCGCACTTCGACCTGAGCCACGGCTCTGCCCAGGTTAAGGGCCA
121	chr1	109604886	109605137	-	0	test	GAGGTCCACAGCTTGCTGATGAAAGGGATACTCCTATCCCTTGCCACAGCTTGTTCTCTTCCCTTCCCTTTGGTAGTTTTAACTTCACATTAGAGCACTCTGAATATCGTCTAATCAAAATGTCTTACAGAGCTATTCACTTCCCATCTTTAAGCCTAAAGATTACAGTCTATGAGACTTTCCATCTTTAACCCTAAAGTCTATGAGTCTATGAGGTTTATTAAAGTCTATGAGACATTAATAAAACAAGT
7534	chr11	64261892	64262143	+	0	test	AGGTGGAGGCTGATGGGGTGGGCCCTGGCACCTGTGTGGCCCCTGACCACCAATCTCAGATACAACAAGCAGCAGCTCAGCCGCATCTACCCCAAGGGCACCCGCGTGGACTCCTCCAACTACATGCCCCAGCTCTTCTGGAACGTAGGGTGCCAGCTTGTTGCGCTCAACTTCCAGACCCTCGGTGAGCCCTGGCCCCCTCCATCTTGACCCCGACCCTCAGTCTTACTGACTTCTGACCCACGATCCTG
6930	chr11	5225980	5226231	-	0	test	ATATGTGTGCTTATTTGCATATTCATAATCTCCCTACTTTATTTTCTTTTATTTTTAATTGATACATAATCATTATACATATTTATGGGTTAAAGTGTAATGTTTTAATATGTGTACACATATTGACCAAATCAGGGTAATTTTGCATTTGTAATTTTAAAAAATGCTTTCTTCTTTTAATATACTTTTTTGTTTATCTTATTTCTAATACTTTCCCTAATCTCTTTCTTTCAGGGCAATAATGATACAAT
6081	chr9	130698259	130698510	+	0	test	TACAGCTGGGGCCATGGACTAGGGCCCAGTGGGCTGGGGGGAGCCGTGGGACCCTTTGTTCCACCAGAGGACTTTGATTTACACTGAGGTTGCCCCTTTGACTCCTGTTTGTCTGCTGTGAAGTTTGCTGCCTAGATGTGTATGTAGACTTTTCACCCTGTCCAGGTCTCCCGAAAGAGGGAGCAGTTGGCATATGGTAGGATCAGAAACATCCATGGGGTGGGATCCCAACAGAGAGTTGGGGAGAGAAG
14225	chrX	149483106	149483357	-	0	test	ACGTCTGTTGCTAAAGATAAATGTTGTAAATTAAAAAAAGAAAACATATGGAGCCCAGACAGGTTCCTTTACTGCTCCTGCCTGGCCATGGCAGGCTTTTATAATGTAACCCATTCTGCTCTGTCGCTTCCTGTTTCAGGCAGGCAATCCATGGACCTTGTGGAACTTGTGTCTCTTTTTCCCACGCTGGCTGGACTTGCAGGACTGCAGGTTCCACCTCGCTGCCCCGTTCCTTCATTTCACGTTGAGCT
8763	chr14	94564506	94564757	+	0	test	GATCAAATGATTAACCCATGTATATACTCCTTAGAAAAGGGTCTAAAGAGATGGGCAAACTACAGCCCATGGGCCAAATGTGTTCAGCCACTGGTTTTGTAAATAAAGTTTTATTGGCACACAGCCAGGCCCATTCATTTATGTATTGTCTGTGGCTGACTTCATGCTACAATGGCAGGTTGATTCATTACTGCAGAGATTGAATGATCTGCAATGCCTAAAATAGTTAATATCTCTATAAACCTTTTGGT
11247	chr16	28933686	28933937	+	0	test	CCTAGGCTGGAGTGCAGTGGTGCCATCTTGTCTTGGCTCACTGCAACCTCCGCCTCCCAGGTTCAAGTGATTCCCCTGCCTCAGCCTCCTGAGTAGCTGGGATTACAGGTGCCCACCACCACGCCTGGCTAATTTTTTTTTTTTTTTTGAGACGGAGTCTTGCACTGTCACCCAGGCTGGAGTGCAGTGGCACGATCTCAGCTCACTGCAACCTCCACCTTCCAGGTTCAAGTGATTCTCCTGCCTCAGCC
8091	chr12	14883026	14883277	-	0	test	AACAAAGCCTATATGACAAGTCTCTAAGACACATGGATTGATTACTGATTTCATTTGATCAGGAAGTTAATGAAATCTACTTTATACTCTCCTTTAATTTTTGCCAATCTCCGTTTATATGAGTTGCATAAGTTAAGGCACTTTCAAATATATTTGTGTCAAGGAATATTCACGGAAATATTTCCAGCTATGTGTCGCTAAAACTGCATTTATTTATTTTCTGTTCTAAGATCCCTTCATTAACAGGAGAA
12751	chr22	20780903	20781154	+	0	test	GTTTTAGCATTTGGCCAGCCTGGATTTGAGTTTTCTCTTTTCCTTTCCCAATTATCAATAAGCAGGAATATAGACAAAAGGCTAAAGAAATGCACCTGTGAACTATTCAGCTTGAGCAGCTGACATTGACACCTACAAGTGCTTTTCAGGATACTTTTGAACTACTGGGCAGGTGGGATGGAGAAATAAATTACTATTTCCCCAGCAACTGTTCTGGGCTGAGCACAAGGGCACTTTTTAAGGAGGTCACC
13977	chrX	147940466	147940717	+	0	test	TATACCTGCTGTTCACCAGATCCTGTGCTAGGGAATTAGTCGTTATTTGCATTTTTCAGATTAATCTATCATTACTTTTATAGGATCATTGTTGCAATTTCTTTTTCAGGGTATGGTACCATTTGTTTTTGTGGGAACAAAGGACAGCATCGCTAATGCCACTGTTCTTTTGGATTATCACCTGAACTATTTAAAGGTGAGAACAGAAAGAACTTTAACTTCTAATCCTTTTGTACTAAAATATACAAACT
5502	chr7	99972111	99972362	-	0	test	GCCAGCTGGTAGGGTCTTGGACAAGAAGAAAGACATCACTTCTGCTCACATTCTCTTTTGACAAAACTCAGTCACATGGTCCCAATATATCTTCGAGGTGGCTGAGTAATGTTATCTTCCTATGTGTCAAGCAGAGGAAATAATGTAGTGAAGACACAGGATGGTCTCTGAAATATCATCTCAGGCATGAAAGTAGAGCATATTCACTTGAGTGAGCCTCCAGTGGTGTGAAGTTGATGGCAGGAGAAAGA
16057	chrX	154324693	154324944	+	0	test	GTATGCATTTCACTGCAGCTGAGTTATGCCTCGGTGAAGACACTGAATAAAGATGGATCATAGCGTATGAACGAACAGTAGCGACGTTTCTTGCTATATTCGTTTATAATGCATCCATTAATATGTATGTAAATTAACAAATACTTTAGACATACAAGTGTTCATTTATACTGGTAAATGTTTATGTGCTCAAATTGGTCATAAAAATGTATTTACATTTTATTGGGAAAGGGGCCAGTCGTCTTGTATTC
2760	chr4	71770895	71771146	-	0	test	TTAAAGTGAGAAGGATAAACCACTAAAAGTTAAAGCCACTTAAAACAGAGTAAGCAATTTATATACTTCCTACATTTTTTTCCAAAGATATAACATCACAGGAGGGAAATTTCCTGGGCTTATTTGCATAAATGATGAGTCAGGTTAGGAGAACACTGACTGACTTTGATGTGTCCGCCCTATTACATATGTGATGGACATGTTGAACAAGGGGGTCTCATGATTAGTAACTAGCTCTTCCCTGTAGCGTG
9975	chr16	2093295	2093546	-	0	test	CGGGTGAGCCTGGGTGCGGCCTGTGCCCCTGCCACCTCCGTCTCTTGTCTCCCACCTCCCACCCATGCACGCAGGACACTCCTGTCCCCCTTTCCTCACCTCAGAAGGCCCTTAGGGGTTCAATGCTCTGCAGCCTTTGCCCGGTCTCCCTCCTACCCCACGCCCCCCACTTGCTGCCCCAGTCCCTGCCAGGGCCCAGCTCCAATGCCCACTCCTGCCTGGCCCTGAAGGCCCCTAAGCACCACTGCAGT
10363	chr16	2106576	2106827	-	0	test	GCAGCAGCGGCTCCAAGCGAGGGGTGAGTGTTGAGCGGGGTGTGGGCGGGCTGGGGATGGGTCCCATGGCCGAGGGGACGGGGCCTGCAGGCAGAAGTGGGGCTGACAGGGCAGAGGGTTGCGCCCCCTCACCACCCCTTCTGCCTGCAGCGGTGGGCTGCACGTACGTTCAGCAACAAGACGCTGGTGCTGGATGAGACCACCACATCCACGGGCAGTGCAGGCATGCGACTGGTGCTGCGGCGGGGCGT
9904	chr16	2091102	2091353	-	0	test	CCGGCAGCGTCTCACCCCTCGCAGCGCCCCGCCCCCTCGCAGCGTCCCGCCCCCTCGCAGGGCCCCGCCCCGGCAGCGTCCCGCCCCCTCGTAGGGCCCCGCCCCGGCAGCGTCCCGCCCCCTCGCAGGGCCCCGCCCCGGCAGCGTCCCTCCCGCCCTCCTGACCGCGCCCCCCACAGGTGTGCCTGCTGCTGTTCGCCGTGCACTTCGCCGTGGCCGAGGCCCGTACTTGGCACAGGGAAGGGCGCTGG
14080	chrX	147944250	147944501	+	0	test	TTATTATAGTAATTCTTTATTAATGATCAATTATTGTAATGGGAGTGGGAGGGCAGGTTGTGGACCAAACATCAGGCAAGCATGTATCTGCCTTCAGCTAGATCCAATCCATGACCCACAAGAGACCCTTCCCTGAAGCAGTTGCCATGGTGGCCTCCTTTGCCACCCTCTCAGAAATGACCAAAATATATACATTCCCCTCTGTCCTGGGCCAAATCTTGCTAAAAGGAGCTCTCTTTTTAGAGGTTTTG
14088	chrX	147944469	147944720	+	0	test	TGCTAAAAGGAGCTCTCTTTTTAGAGGTTTTGTTTATCAGGGTATCACACTTAACTGTCGTGGATAATCTTTTCCTCCAGAGAGTATAGATGTTTAAAGGAACACAGCTATTTAATGATATGGCACATCAAGGTTTGAACTTAGGTGGCTAATAACATACCTTTTTAAAAATGAGAATGAAACATGTTTATAACCAAAATTAACCTCAAATATTGCAAAGCCCTTCATTTTGAGTTTAAAATTTTGTTTTA
12664	chr21	39348350	39348601	-	0	test	ATTCGTGCGTCTCCGTCTCCGCAGGTCAGCTCCGCCGAAGGCGCCGCCAAGGAAGAGGTGAGTGCGGGCCTTCTGCGGGGGGTGGTGGGTTTCCCGTGAGCCGCTGGCCTGCCTTCTCTTCTCGCTGACTCTCCTTTTTCTTTCTCCAAGCCCAAGAGGAGATCGGCGCGGTTGTCAGCTGTAAGTAAAGCGAGCCCCGTAACCGTTCGTTTTCCGCGGGTCGTCCCGGGTGAGGACGCTCAGTGCTGCTT
14722	chrX	149501477	149501728	-	0	test	GCTAACCACTATAGGCAGCACTTCCCTTCAGATCCCTCACAGGCCTCACTGGCTCCTAGAAGTCCTCCCTGCCAGTCCTCACCTGACTCACAATAACCTCTGTGCTTGGAGGATCCCCCACTTGGCACTTCGATGCTTTTGAAAGCTGGAAGTCCTTCCTTGGTTTCCAGTGCTGTTATACTGTCTTCACAAATAAACTATAAACTACTAAGCTTAAGGGTGGGTTTCTGCTTCATGAATGAGCACTTGAA
12729	chr22	20780486	20780737	+	0	test	GCTGGGCGCGGTGGCTCACGCCTGTAATCCCAGCACTTTGGGAGGCCGAGGCAGGCAGATCACTTGAGGTCAGGAGTTCGAGATCAGCCTGACCAACAGACCAACATGGTGAAAACCTGGCTCTACTAAAAATACAAAAATTAGCTGGGCCTGGCGGTGGGTGCCTGTACTCCCAGCTACTTGGGAGGCTGAGGCAGGAGAATCACTTGAACCTGGAAGGCAGAGATTGCAGTGAGCCGAGACTGTGCCAC
10129	chr16	2097903	2098154	-	0	test	TCGCGAGGGGCTGCCATCACGGACGGTGCAGATGTCCCATATATCCAGCATTCTAGGACATTCTGTCAGATGGCACCGGGCTCTGTCCTGTCTGCTGAGGAGGTGGCTTCTCATCCCTGTCCTGAGCAGGTCTGAGCTGCCGCCCGCTGACCACTGCCCTCGTCCTGCAGGTGGCTGGGAGCCCGAGCCCCACACCTGCCGGGCAGCAGGTGCTGGACATCGACAGCTGCCTGGACTCGTCCGTGCTGGAC
11055	chr16	2130708	2130959	-	0	test	TTCTGGTGCTGGTTGCCACTCACCCTGGCTGGGGGTCACCTGGGTCTGCTGCTGTCTCGCAAATGCTGGGGTCCAGGACTGGGCACATCGAGGGACTTGGTAGGTGCTTGGTTCACTGATGTAAAATATAGGAGCACCCGGGGCCTTGCCCTTTCCCACCTGCATCCCTGAATGACAGGAGAGTGTGGGAGAGTGTAGGGACAGCAGGCGCAGACCCCGGGGCCCCTGCCTGGGATTGGCGTCGGGGAAGA
14894	chrX	153864561	153864812	-	0	test	GAGGAATGACCTGCCAACAGGGAGGCCCCGGCCAGCCGGGTCCAGGAGGGAGGGCCTTAGACTTCCTGGCAGCTGCCACACTCTCCTCGTTCCCCTGCATCCCCCCAGGGATCTCAGATTTTGGCAACATCTCAGCCACAGCGGGTGAAAACTACAGTGTCGTCTCCTGGGTCCCCAAGGAGGGCCAGTGCAACTTCAGGTTCCATATCTTGTTCAAAGCCTTGGGAGGTAAGCGTGAAAGGGGGCCTTGG
13914	chrX	147938449	147938700	+	0	test	TATTTTGCTTCTAGACATATGTACAGGTTTACTCTGCCTGGAATGCTTTTTCTCCTCCACATGTCACCTCATTTTGACCTTATTCCTTACAAAGCTTTCCTTGTGTACTTGCCGAAGATCCCCACCCCCTTGACTGGTTCAAGAGCCTTCCCCTTCTGTTTTGCAATGTGCTCACCCCTAGCATAGCACTTAAACATAGATGTTTATTTGTTGATCTCCTCAACCACATTGTAAGTTCTTTGAGGGCAGGG
13188	chrX	147915681	147915932	+	0	test	GAGAAGATTGGGATCGTGATTATAATAATAGTTAACAGGGGATGAGTACTTTCTAGGTGCCAGGCACTGTTCTCTCTGATACTTTATTTGATGTATTGTTGTTATTCCCATTCTTTAAATGATGCACAGAGAGGTTAGGTAAGTGACTTACTACCAAGTGTCAGGGCCATTAAGGGTCAGGATTCTGAATTCCTGAAATGATGAAATTTAGCTTGAAGAAATTGGTTTGATTTCCTGCTTAGTTTTCAATT
16432	chrX	156008037	156008288	+	0	test	TCTCCCCGGATCACATGATGGCACCACAGCTGAGGAGTGGGCTCTGCACTTCCCCCCCTTCCACCCATGTTGGGCTCCTACAGCCCAGGCACCAGTGAGCAACTTGGGGGTTGCATCAGCCCCTCCCCTCCCTGCTGGGCTGTTGGTTCATGCCCCCTGGGTGGGAGGAGGGGGAGAGGGAGAGCTCCAGTGAGTGGTCTCTGGTTTTTCCCCTCAGACTCCTCACTTTGGGCAAAGGACAAGAGGCAGTG
14249	chrX	149483676	149483927	-	0	test	TCAGTCACAAAAGGACAAATACTGTATAATCCCACTCCTATGAGGTATCTAGAGTAGTCCAGTTCATAGACATAGAAAGTAGAATGGTGGTTTCCAGTTGCTGGGTGAGGATGGAGAAAGGGGAGTTGTTACTTAATGGGGACAGAGTTTCAGTTGTGTAAATTAAGAGGAGTTCTGGAGATAGATGGTGGTAATGATGGCACAACAGTATAAGTGTACTTAATTCCACTGAACTGTATACTAAAAAGTGG
6311	chr10	47303696	47303947	+	0	test	AGGGTTAAGGTTAGACCACCCCTAATTCCCACTGAATGACGTGTGTTTAAATGTAAGCTTTGTCACCTACCAATGTGATACTGAGTTGGTTTTCTTCATCTCTAAATTGAAAATAATACTACATACTCACCTGTCAAAAATGCCAAAATTCAAGTTGATTTCAGTTCCCCAAGTATTCCCTAAGCACTTCGTTTGGTAGCAGATGCTGTAAGCTCAGGGAGCACAAAGATGCCCAGAAAAGCTCATAAACT
14325	chrX	149486240	149486491	-	0	test	ACAGCTCATCAATCTAGTGAACCACTAGATCTGTAAAATCTACCAGGTGATTCTACTCTGCCCTCCCCAGCAGTGAAGGGCAAGTTTACTTTGACTTCCATGTGCCAGTCCTTCAAAGCTTTGAAAACAGCACTGATGCCCCCGACCCTGTCCATCCACCAATATCTTCTCTTCTCTAGGCTGAAATCCCCAGCCCTTCAACCATTCGTCACAAGGTAGTGTTTAGACGCGTGGTTGTCCTGGTCTCTGTC
11241	chr16	28933573	28933824	+	0	test	ATCTTAGATTCCCCCAACCCGAGGGCTACTCCCAGCCTCACCCCAAACCCCAACTTCCACACAGAACACTGACTCCAAGTCTTTCTTTTTTTTGACAGAGTCTCGCTCTGTTGCCTAGGCTGGAGTGCAGTGGTGCCATCTTGTCTTGGCTCACTGCAACCTCCGCCTCCCAGGTTCAAGTGATTCCCCTGCCTCAGCCTCCTGAGTAGCTGGGATTACAGGTGCCCACCACCACGCCTGGCTAATTTTTT
6740	chr11	4386035	4386286	-	0	test	ACAGGAGTCTCAAACTCTCTTTCCCCCAGGAGTGAGTCCTGGAACCTGAAGGACCTGGATATTACCTCTCCAGAACTCAGGAGTGTGTGCCATGTGCCAGGGCTGAAGAAGATGCTGAGGACATGTGCAGGTGAGGCAAGTTCTAGTTTTGCGGGGGATAATGGGGTGCAGAGTAGATCCCAGGGTCAGGGAGCCTGGATGGCAACTTGGAGGAGAGATGGCAGGTCAGAGCAGGGGGAACAGAGATGGAG
11833	chr18	31596007	31596258	+	0	test	TTCCCTTGTGAAAGCCAAGCTTAAAAAAAGAAAAGCCACATTTGTAACGTGCTCTGTTCCCCTGCCTATGGTGAGGATCTTCAAACAGTTATACATGGACCCAGTCCCCCTGCCTTCTCCTTAATTTCTTAAGTCATTTGAAACAGATGGCTGTCATGGAAATAGAATCCAGACATGTTGGTCAGAGTTAAAGATCAACTAATTCCATCAAAAATAGCTCGGCATGAAAGGGAACTATTCTCTGGCTTAGT
12642	chr21	39347754	39348005	-	0	test	GCTGGTGCTTGGCTTTTAATTCGTGCTTGTACTTCCTTTTTTGACAATAAAAGAGTCAAGATAGCACCGAGGCCAGGAGAAAGGGAACGTGTAAGTTTTTATATATACAGTTTCCAAGCCAACTTCGGGAAGCCTTAACCTTTTTACGGGGTGGGGGTGGGGAGGTAAAAAGTTGTGATCTCTGAGAAAATAACCGCCACTACTCTGGAAGTGTTCATCAGCAGTTATACAAAACCGTGATTTTGGCTGCT
2086	chr4	39456688	39456939	-	0	test	TAAGATTCCCCCATCTACCCTTGCTCTTAGCCTTAAGTTCCTTGAAGGCAGGGGGCCTGCCTTTGGCTTGGTGGTCTCAATTCCTGCCATGTGCTGTACCTTAGTAACTCAGTAAATGTTAGTCTAATGTCTAAGATCAGGGCTTTGGGGTTCTCATTAAAAGAAGGCAGCATTGTTCATTGAGGGCATAAGAGCTAGTGAAATAAAGCCTCCGCAAGGAGCTTTACAGTCCACTGAATAGACAGAACAAT
10094	chr16	2096850	2097101	-	0	test	CGGGACTCAGGCCAGGCAGCCGTGGTTCCCGCCTGGGGTAGGGTGGGGTGGGGTGCCAGGGCAGGGCTGTGGCTGCACCACTTCACTTCTCTGAACCTCTGTTGTCTGTGGAAAGAGCCTCATGGGATCCCCAGGGCCCCAGAACCTTCCCTCTAGGGAGGGAGCAGGCTCATGGGGCTTTGTAGGAGCAGAAAGGCTCCTGTGTGAGGCTGGCCGGGGCCACGTTTTTATCTTGGTCTCAGAGCAGTGAG
14710	chrX	149500764	149501015	-	0	test	TGATTCTCCGTATAGCTGGTCTTTTCCACCTTATCATCCTTCCTCTGAGAAGTATGAAAACACTAAGGTAAGGCTGTGAAAGGGACATTTCTGAAGAGGAACCACTTTTTCCTTTGTCACATAAACTACTGGGTATACTGCATGTTCTGTGAAGCTGGTTATATACCACGAAGTTGTGGGTTTCATTTGTGATAATGTTTTGACAGAAGTAAGTGTTGGGATCTTCAGCATTAGGCCCGACAGGAGCAGTG
11497	chr17	34360666	34360917	-	0	test	TCAGCCCCACCCTGATACACCAAATTGAAACCAGGAGAGGGGTCCAGGAAATTCAATTCATAAGCTCCTGGTGCTTTGGCTGTTCCCCAGCGTGCAAACCACACATCCTGTGCAGCAACTTCATTTACAGAGGGGAGCCCAAGGCCTAGCAAGAGCAGTTTAGGGGACCTGGCAGCCGGAGGAGGCGGGGCTTGTGCGTTGCCCACCCAGTGGTGGCTTGGGTGGCTCAGCCTTCTCTCTTATTCTCTGTT
7596	chr11	64265976	64266227	+	0	test	CTGGAGCTGCGGGAGGCCCAGGTGGACGCAGAGGCCCAGCGGAGGCTGGAACACCTGAGACAGGTAGGGGGCCTGCAGTGGCCAGGGAAAGCCTGCTGGATAGACCCGTCGTCAGCCCGGCATCACCTGTCAGCTCCCTGTGTCCACAGGCTCTGCAGCGGCTCAGGGAGGTCGTCCTTGATGCAAACACAACTCAGTTCAAGAGGCTGAAAGAGATGAACGAGAGGTGAAAGCCGAGGATTGTCTATGGG
14737	chrX	149502095	149502346	-	0	test	GATTTTGGAGAGAGGAGGAAACACAGCAGAGGGAGGAGATTGTGCTTTGGGAGGGGAAACCAGGCAAGGACTGAGTGATGTTGCTTGACCTCCTCCCTCCTGATGATTGGATTTTGGCACCAGGCTCTAAGCACCATGTCCTTCTCAGAGAATAAACCTTTCCCTGCAGCCCTTGTGGGAACACATTTTTCAGCTGAATCTTTGTTCTCAAGACCTTTTTCTGTTAGCATTGGGCCTGCTTCAAGATGACA
14678	chrX	149498505	149498756	-	0	test	CATCCTTTTGGGTGTGAAGTGGTTTTGATTTGCATTTCCCTAATGACTAATGATGTTAGCCACATTTCCATGTTCAATTGGCCATTTGTATATCATCTTTGGAGAAATGTCTATCCAAATCCTTTGCCCATTTTTAATTGGCATTTTTTAATTGTTGAATTAGAGATGGATTTTTTTTTATAATTAGTCTATCGGGGTTTCTTTTAATTAGTTATGATCTGGAGAAAGAGTATTAGTAGCAAGAAACCAAG
8037	chr11	123059366	123059617	-	0	test	CTAGACAAGTCACAGATTCATGATATTGTCCTGGTTGGTGGTTCTACTCGTATCCCCAAGATTCAGAAGCTTCTCCAAGACTTCTTCAATGGAAAAGAACTGAATAAGAGCATCAACCCTGATGAAGCTGTTGCTTATGGTGCAGGTAACAATGGTATCTCAATTAACCCTAAAGGCAGGCAGGCCCAAGGTGACTCGCTGTGATGAGTGATTGTTAAACATTCGTAGTTTCCACCAAAAGCTTGGCTAAT
1157	chr1	186678369	186678620	-	0	test	TGCCAGATTATAATGTGCAGAGTATATGTATTTTATTAAAGATGTATTTCAAGTGGCCATTAGACTATAAAGTGTAGTTGTTTAAAAATAGATTTTTTTTATTTTGGAGTTACATTCAACCTCAGGTGCCACTTTCCACATTTTACAATAAAAATAATGGTTGATTTACTTAACAAATGAGAATAAATAAAACATTTTTTTCTTTGAAAATTTCAGCCAGATCACATTTGATTGACAGTCCACCAACTTAC
2656	chr4	71765298	71765549	-	0	test	GAAAGCTCTGCATGGCTGCTCTGAAACACCAGCCACAGGAATTCCCTACCTACGTGGAACCCACAAATGATGAAATCTGTGAGGCGTTCAGGAAAGATCCAAAGGAATATGCTAATCAGTGAGTGCCTTCATCATAAATAGAACTTTAGGACCTAAAGTATCAGAAATGACTCTAATCTAACCCCCTACTCTGTGGAAACACCTCTTCTACAGCAACCTGAACAAATTCTGATAACTTGACTGTCATCTGA
8284	chr12	57232574	57232825	+	0	test	GGTTGGTGGGGGGGGCTGGAGACTGGGCACCTCCCCAGGGGGTGGTGAGGAGGTGTGGGAGGAGGGCAGCCTTGGGCAGGCCTCTCCGGGCCCTCCCCAGGCTGAGGCCTTGCCTCTGTACCTGCCCAGGTGTGTGATGAAGTCAAAGCACACCTGCTGGCAGACATGGCCCACATCAGTGGCCTGGTGGCTGCCAAGGTGATTCCCTCGCCTTTCAAGCACGCGGACATCGTCACCACCACTACTCACAA
14616	chrX	149496650	149496901	-	0	test	ACTAAATGCTGAATCCTATGAGTTCCCCTAGTGAATTGCTGAACCTGAGGGTGGTTTTGGGAACCCAACACCTCACCACACTTTTCTAGCCAGGCAGGAGGTGGGGACAGGAACAGAAGGGCGTCTGTTTGTATGAAGGAAGAGAGTTGTTGCTGCTCAGTTTGTAGGAAACAGAAGTCGTATGATTTCATTGCCATTGGCATCTCATGAGAGTAACTTAAGGCTGGACTTCCAGGTCAGGGCCGAGCACG
14368	chrX	149487707	149487958	-	0	test	TCATTTGCAGCCCCAGCGTCATCTATTAGACGGAGCCAGCAAGGTCTCATGCTGTGCGGTACATTTCAACACTTTCTCAAATTTACCCGTGGCAGCTTTTGGTGTGTTTGGTTATTTTTATAACACTACTCACTGTACAATTTCTGTCATGCACATTTTTTGTATCAAAAAGGGTGCTGTAACTTTAAAAGACTGGGTATTTCCAACTTGGAATATTCAAACCATCTTTTGCAAGGGATGTTTTAAATAGG
6369	chr10	47305990	47306241	+	0	test	GAGCAAAAGGCATGGGCTTGGGGTCAGACAGAGCTGGGTACAAGCCCCGACTCCACCCCTTGCCAACTGTGTGACTTTAGGCAAATTGCACACCCTCTCTGAAGCTTCAGCTCTTTGTCCACAGAATGAATGACCGCTCCCTACTCTCTTCATGGTGAAAATGATGAATCAAGCATGCAGAACACTCAGCATAGGACCAAGCACATACACTTGGAGCTCAGAATGTCATAATTAATAAACAATTTTCTTCC
8018	chr11	123058901	123059152	-	0	test	GACTGTCCTCATCAAGCGTAATACCACCATTCCTACCAAGCAGACACAGACCTTCACTACCTATTCTGACAACCAGCCTGGTGTGCTTATTCAGGTATGTTTCTGTACTTCTCTTGTTTGGCTTACTGATAACAGATAAAGGGAAGTCTTGACTGACTCGCTATGATGATGGATTCCAAAACCATTCGTAGTTTCCACCAGAAAGTCTTATGTTGGCCAGTTCCTTCCTTGGATGTTTGAGCGACCATTCT
2996	chr4	71781625	71781876	-	0	test	GACAGGTTCCAATTAAGCCAGCCCTCTCCTATCAAACAGCATTGGGAATCTTGTTCCCAAGCCACTGGGTACATCTTGGAGAAGAGAGCTTACCAAATCTGCTAAATCTGGTTAGTACCATAAGAAAATGTCCACTTTTGCTACCCTATGAAAAGCAACAAGGCAGTATTTCCCAGTTGCCTACAAGTAGCTTCTTCCCCTAGGACTGAAATGGATGAAATACTAATATACAATATATCTTTCCCTCTTCC
1962	chr2	162148026	162148277	-	0	test	AGGGCCTGAATGAGTTTAAACCTTTTTTAATATCTTGCCAACTTGCTTTCCAGAAAGTTCATACACATGCACACTCTCATCAGCAGTATAGAAATACTCCTCATAATATCCTCCTCAGCATTCAAGCTTTGCTAATTTCATAGGTAAAAATAGTATCTCATTTCTGAGTATTAATTACATTGACTATTTTCCCATGTATTTACCATGCCAACCAACTGTTGATTAGTGGCATAGGGATTATTCCTGATACA
14267	chrX	149484141	149484392	-	0	test	CTGAGCCGAGATTGCGCCACTGAACTCCATCCTGGAGACAGGGCTAGACCCCGTCTAAAAAAAAAGAATGAAACAAGTGTTAAGAGTGTTAGTGATTATATGGAAAAATTGGAACACTTGTGCATTGCTGGTAAGAATGTAAAATGGTGCAGCCACTGTGGAAAAGAATTTGGTGGATCCTCAGAGTTAAACATAGAATTACTCTATGGCCCAGAAGTTCCACTCCTAGGTATATATCCACAGAGCTGAAA
10909	chr16	2125633	2125884	-	0	test	ATCAGGAGCGTAAGGTCAGTGCAGCACCTGCCCACACAGGCTGTGAAGGGTGGGAGTGGAGAGGGATGCAAGGGGGTCACAACGCCTGGCTCCATGTCAGCTGCGTGCAGGGGCACCAGGAGCCGGCCCTCATTCTCCCCTTGAACTGGAAGGGTGGCCCCGACCCCAGCGGCAGGTAGCATACGTATGAAGCGCTCTCCTTCCTACACCCCACAGGTGGGCTCGTCTCCAGACGGCCCTTTTTGAGCTGG
9406	chr15	50258592	50258843	-	0	test	AGGGGTAAGTACTGGGGACTAATGTGAATACCACCCAGAATCCCACAACTGGGTTACTGTAGTGATTTTCCTTTTGCATATTTCCTTCTGGACTTTGCATATATGAAAAGGCTCCACTGATGATCTAGCATAGGACATGGGCCACATTAGTGACAAGAACACCAGTGAACAAGTCTGATTTTCATGGAGTTATGTCACTTATTTCTGCTTAAAAGTGTCAGCCCATCTCTAGAGTGACACTTCACCTTGCC
13763	chrX	147934031	147934282	+	0	test	GTGTACAGAATGTATGCATAGCGCAAGATAAAGTGCTAATCTCAAGACAGTTGGGTGATGAATAATATTTATTTTCTTCTTTACAGCTTGTCAAGCAATTGCATACCTTTTGTAGTTAGAAAGGTTATTGTAAATTTAGGAATTCTTATGGAAAAAGAGCTGGTAGTTGTTTTGTTTTTTTTTTTTAATAGCAAATGAGTAGAGAATTATGTGTCAGATTCATTAAAGTTGGTTTTGAAGATGGCATGATG
9246	chr15	50253158	50253409	-	0	test	GTGACAAGAGTTAGAGGTGGCCTACCTTGCCTCCTGTCCTAGGAGGCGACAGTAGGAGAGCCTTCGGTTTTCTCATCCTCTTACTTGTATGTTGAACTTTACTTAATGCAGGCTATATCCAAAATACAGTGTGAATTAGGCTAGAATAGAAATGCTTTCTATTCTACCTTCAGAAAAGGAAAAGGGTAGGGTAGGGGAAATCATGGGAGAGATTAAAGGCAGAGAGGAAACCCAGATGGGACTGTTTAATC
9413	chr15	50258825	50259076	-	0	test	CATTCTCCTGCCTCAGCCTCCTGAGTAGCTGGGACTACAGGCTTCTGCCACCCCGCCTGGCTAATTTTTTTGGTATTTTTAGTAGAGATGGGGTTTCACCATGTTAGCCAGGATGGTCTCAATCTCTTGATCTCGTGATCCACCTGCCTCGGCCTCCCAAAGTGCTGGGATTATAGGCATGAGCCACCGCACCCGGCCTGCCCTTCCTTTTATTGCTGGGCAGGGGCGGCATGAGGGGTAAGTACTGGGGA
6904	chr11	5225922	5226173	-	0	test	TTGATACATAATCATTATACATATTTATGGGTTAAAGTGTAATGTTTTAATATGTGTACACATATTGACCAAATCAGGGTAATTTTGCATTTGTAATTTTAAAAAATGCTTTCTTCTTTTAATATACTTTTTTGTTTATCTTATTTCTAATACTTTCCCTAATCTCTTTCTTTCAGGGCAATAATGATACAATGTATCATGCCTCTTTGCACCATTCTAAAGAATAACAGTGATAATTTCTGGGTTAAGGC
10870	chr16	2124140	2124391	-	0	test	GCAGAGCCCAGGGGGAGGGCAGGAGAGCCAGCGCCTGGCTGGGAACACCCCTGAGGGGCCGAGGCTCCAGGGCGAGGGGGCCCGACCTGGGGTTCACACGCCCGGGTGGCGGGCAGACCCGCTGCAGCATGAGACACGTGTCAGCTACCTCGGGCCGGCAGGCTGGCCCTGCTGCCCACAGCCCTGGGACGTGGCCCCACCTGTGACGGGTGTGGAGGGGCAGCCTCCAGGCCTGGCCACACCCTCTGCTG
461	chr1	119508120	119508371	+	0	test	TGGGTGTGTGTGACAGAGAGAGAGGAAAACAATAAGGGGGAGAGAGAGTCGGGGAGAGAGAGTCGGGGAGAGAGACTCAGAGAGAGAGACAATGAGAAAGACAGAGAAATAATGAAAAATATATAGTGAGAGAGAGAGCATCAGTGAGAGAATGTGTGTGCACAGTGCACAGCACAGAGCAGAGACAGAGTAAGAGGCAGTATAAGGCCAGACATGCCTCATTTAGATTTTGCATATATGGCTTTTTTTTA
2122	chr4	69956529	69956780	-	0	test	AGTCTGTTGAGAATTCTGATTTAGATAAAGTAATTAAGGCTTACAAAAGCCGGAATTAAATTTAATAATATGATTGAATTTGGAAAAAAAAGCTAAAAAATGTTCTGTCATTTTCCTTGTGCACATCTCTTTTACACAAGCCTTACTTCACATCTTGTTTTTGCTATAAGTATATATGAAGGCAAAAGACTGAGATGCTTATTTCACTACTTACAACATTCTTAAGGCAAGTTTTCTTACTAAGAGGTTAT
2626	chr4	71764355	71764606	-	0	test	TAAACTGAGATAGATAATGAAAAATTAATCTAGCAATTACATTCATTTTAATTAATTCAATTAGCAGCTTAATTAACAAGCAGATACCATGGGTGATGCACCTGACCAGACATTAAGTGAATAAAGGGACAAATGGCCCCTCCTCTCACACAAACCACATAGAAATTTGTAAATCTATTTTTAATGAATGAATATACTGATTAATAGTGGCCAGTATTATAACCTTTTCTCTAAATTCTTAGTGTTTTTTA
8141	chr12	14884407	14884658	-	0	test	GGATCCTTCTTAATCCTCTTTGACCCTGTTCTCAAACAGGCTCTGAATAAATCAGAGAAGAAGGTTCTCTGGAGACTTCTGTACAGCACTTAAAGTGTCTTATTTTGCTTGTCTGAAGACGTCATAGCCCTTGGGAAATTTTAGCTGAAAATGGCCACTCCCTCCTTCAACATCAGAGAAACTAAAATATAGAGATATCCACAGCAAGGCCAGAGCTAGAGAAAAACCTCATAAATCCTAAATTCCTGAAA
7336	chr11	6393384	6393635	+	0	test	AAAGTGAGGGCCAGTAGTGGGAACACGGTGGTGCTGGGGGACAAGCAGGCTCCTGTTGAGCTGGAGCACCTCTGGGCACAGAAGTTTTATTTTCCTGGCATTCCCAACAAGTGTTCCCTGGGGATTCAGCTCATGGTCACTGTTGAAAGCCTTCATTCAGTCCCCCTTTCTCTAGCCAGGGCTGCCTGGACCCCTGGATGCCCTGATTACCATCCTTAATTCTCCCTACTAGGTGCATATAATTGGCCACA
15812	chrX	154315101	154315352	+	0	test	CTTCCTTCTGTAGTCGTTCCTTCTAAATGCATTTGAAGAAACTCTACCACCTGATTGTCTCTGTCTTCTAGATAGCTACTCGGAAAGCATGCGGTCTGGCTCTGGCTAAGCTGGGCTACGCGAACAACAGAGTCGTTGTGCTGGATGGTGACACCAGGTACTCTACTTTCTCTGAGATATTCAACAAGGAGTACCCTGAGCGCTTCATCGAGTGCTTTATGGCTGAACAAAACATGGTGAGTGTGTAGTGT
1208	chr1	192809117	192809368	+	0	test	TGGACAAGAGCGCAGGCAGTGGCCACAAGAGCGAGGAGAAGCGAGAAAAGATGAAACGGACCCTGTGAGTATGGCTTTCTTCCCTCTCCCGCCACCCCCTGCCCCACACTGCAAGCTGCAAACGCGGTACTTTCGGGCTCGCCTTTGACGTTAGGAAACTAGCCTGAGCCTATGCAGGGAAAAAAAATCGAAAAGGTCAATTTGTTAAGTAAGGTTAAATCTGGGTGATGCTCGGGTACAGTTTAAGAACC
14949	chrX	153866098	153866349	-	0	test	TTCTTTTCTTTTCTTTTCTTTCTTTTTTTTTTTTTTTGAGATGGAGTCTTGCTCTGTCATCCAGGCTGGAGTGCAGTGGAGCGGTCTTGGCTCACTGCAACCTCTGCCTCCCGGGTTCAAGCGATTCTCCTGCCTCAGCCTCCTGATTAGCTGGGATTACAGGCACGCGCCACCACACCCAACTAATTTTCGTATTTAGTAGAGATGCGGTTGCACCATGTTGATCAGGTTGGTCTCGAACTCCAGACCTC
1373	chr1	225838331	225838582	+	0	test	GGAGCTGTCCTGAGCCCTGGAAACCCAGCAGTGGCTGTACAGACCTGGCCCAGCTGTCAGGGGGCACCTCTAAGGAAACCGGGAGGCAATAATCGTAGCTCCCTTGCAGGGAGGTTGTGAAGGCTGAGTGAGGACATCTGTGCACCTGGAGCACAGTGTGAGTGTGAAACCAGTGTCAGCCCTTATTACTGTCAATACCATGAAGGGGCGGCGGGGGCACTAAGGGTGGCAGGACTCAATATCTAGGCTCT
3013	chr4	71782174	71782425	-	0	test	CATTAACTTTCAATATTTGGGGAGAGTAGTCCTGATCTATCAATTCTCTATTCTCTAATACCTGAGCATGACCCAAAGACCAAGTCAGCAGTTTACTTAGAAGCTAGCTGTGTTCTTCCTGGCAGGAAGTTACAAAGTGTACAGATTAAGGGAGTTCTGAGAACTCTTAGGTTCTGAGTAGCCTTGTATTGTGGGGGATTGGTTTGAATTGCAGTTTTCTCTGCCTGTTATAGTTCTAGAGAGCTGACCAC
2133	chr4	69957108	69957359	-	0	test	TGCTCAACCAAGAACTTCTACTTAACCCCACCCACCAGATCTACCCTGTGACTCAGCCACTTGCCCCAGTTCATAACCCCATTAGTGTAAGTCCAAATTTACTGGCTTTGCTGTTTCATTCAAGATGTGTATGTGATGGTAGAATAAAAGAATAAATGTAGAGTAAATGAATTAAAAAAACAGTTTAGATAAGTGATTCTTTTATTATTATACTTTAAGTTTTAGGGTACATGTGCACAACATGCAGGTTA
13276	chrX	147918007	147918258	+	0	test	CTCAGGAACCTCCTCAACCTCTCTTCTCTACAGACATCAGCTTTGCCTGATAGGTAGGGATCATAGCAAAACACAGTTTTCCAAGGTGGTGATAGGTGGAGTGATAGTGCTCTGGAGATGGCCAAAGAAGGAAGGTATGAGTGTATCTGTGGGTGGGTGAGTGGTGGATAAGGGGAAGGACAGAGCCAAAAGCGACGGCTATTGGAAAAACTATGATGAGAAACAGGAAGATGGAACCTTGTTGGAATTAG
12632	chr21	39346790	39347041	-	0	test	GAAATATCTGCCTGTTTCCCTCTTTACATTTTTCTTGTTTCTTTCCTTATTTATCTTTGTCCATCTTGAGATCTACTGTAAAGTGAATTTTTTAATGAAAACAGTTCCAAGTTTTACTCTCAGTGGGTTTGGGACATCAGATGTAATTGAGAGGCCAACAGGTAAGTCTTCATGTCAGTGTTTGTTGAGGAACGAGCCTATGAGGTCAGTTTTCCCAAAAGGAAAAAGGGCAGAAGGGATTTGTTCATTTT
1450	chr2	10441334	10441585	-	0	test	TACACTGTTGCTGCTGCCTCTACGTTCAATGGCTTCCAGAGGCCGACGATCTACTATGTGATGTCAGGGCCTGCGTGGTAAGTAAGCCATGCATGTTGATGGTGCTGCCAAGAATAGGCACCTTCTTGGATGTGTGCTTCTTGTCTAGACGAATAAGAAATTGTCTTGCCTAAGATTAAATATATATGGATATTTTTCCTAAGAAAAGTTTTAGAAAAGACTGATGAGTGTATTTCTATGTAATTGGAATA
14470	chrX	149491373	149491624	-	0	test	CCAGAAAGTATAAGTAGGCCAGGGCTCAGCATAATCCTGCTGGAAGGCTAGATGTATATCTTTTCTCTTGACTGCAAGTGAGAACGGGTGAGTCTCATGATTGTCCTCACCCTGGCAGTGATGAGAAGAAGCTGGTGGGTCCAGCTGATAAGTCAGGGGCTGGTCTGCGAAGACACGGCTCTCTTCTCACCCCTCTTTGGGGTGGAAGATTAATTTTTGTCCTTAGCATTTGTGAACCAGGTTGGGAATGA
15342	chrX	154299943	154300194	+	0	test	GGTAGATAGCCAGTAGTGGAATTTCTGGATTGAATGGTAGATAGATGGTAATCTACTACTTTTAGTATTTTTTTTTTTTTTTTTTGTCTTGAGACGGAGTCTTGCTCTGTCGCCCAGGCTGGAGGGCAGTGGCGCAATGTTGGCTCACTGCAACCTCTGCTGTCCGGGTTCAAGCGATTCTCGTGCCTTAGCCTCCCGAGTAGCTGGGACTACAGGCACACGCCACCACTGCCAGCTAATTTTTGTATTTT
15491	chrX	154305003	154305254	+	0	test	GCCACAGATGAATTTAAGCATGTGAAAGAATACTCATGTCAGAGGCACAAAGGAAACTTGCCCCGAGTCCACGGTGCTCTGCGGTTAGGAGCTGGCCTCACTGTGCACAGGGGGAGGGGTGGTGAGCTTACCGGGCTTTAAAGCGCCCTTCCAACCTGTGAGCCCTGCATTCCTTACCCTTGGTTGGATTCTTCCCGGGCTGGGAGAAATGACCGCTTCTATGAGGAGACCATGTGCCGAGGTCGTGTGCT
10924	chr16	2126355	2126606	-	0	test	AGCAGCGGGGGCGCCACAGTCTCCCTGCAGAGTGAGCGCAGCTGGAAAATGCAGCTCACGCCCTTTCCCAGAACACCTCGCTCTTCATGGCTTGGCAGCTGTCCTTGCCTAGGGGCCAGGGTGCCCAGGCACTGGTGGCAGGAGAAGGGCTACATCTGGGGCTGAGGCGGGCTGGGTCCTTTTCTCCCTGCAGCTCCCGAGGCCCAGCCCTGGCCCAGCCTGGCATTCCTGACCTTAGCAGCGCCATGATC
7286	chr11	6391389	6391640	+	0	test	CCCAATGTGGCTCGCGTGGGCTCCGTGGCCATCAAGCTGTGCAATCTGCTGAAGATAGCACCACCTGCCGTGTGCCAATCCATTGTCCACCTCTTTGAGGATGACATGGTGGAGGTGTGGAGACGCTCAGTGCTGAGCCCATCTGAGGCCTGTGGCCTGCTCCTGGGCTCCACCTGTGGGCACTGGGACATTTTCTCATCTTGGAACATCTCTTTGCCTACTGTGCCGAAGCCGCCCCCCAAACCCCCTAG
9611	chr15	90887844	90888095	+	0	test	TCATGGGGCTATTTGGAGAAAGACCATTCCAGAAAGGAGGACAGCAATTACACAGGCCTTGAGGTAGGAGAGTACCAGGGACTAATAGCCAGGAACCAGTGGTGCCTCTGAGAGTGAGGGAGGGGGAGAGTCATACACGAGGCTGGAGGAGGCAGGCGTCAAGGGCTACTGGGTGATAGAGGGTCTAGCAGGGCCATGGTGAGGACTTTGGCTCTGGGTGAACAAGAATGGCATGATCTGACCTCTGTTTT
4093	chr6	29669443	29669694	+	0	test	CCATGCCCAGCTAATTTTTGTATTTTTAGTAGAGACGGGGTTTCACCATGTTGGCCAGGCTGGTCTTGAACCCCTGACCTCAGGTGATCTGCCTGCCTTGGCCTTCCAAAGTGCTGGGATTACAGGCATGAGCCACCAGGCCCAGCCCAATAACCTTTAATTTCAACATACTAATAAACATAAACAGTATTTCAAGATTTCTGCAATAACTCTAATGGGAATGAAAACATCTGTGGCTTCCATTGGTAATT
149	chr1	109605897	109606148	-	0	test	TAGGTCGTATGTTGGGCATACCTATGAAAATAGCTGCTTCTTCCCTCAGGATGTTTGATGTGGGAGGGCAGAGATCCGAGAGAAAGAAGTGGATCCACTGCTTCGAGGGAGTCACCTGCATCATTTTCTGTGCAGCCCTCAGTGCCTATGATATGGTGCTGGTGGAAGATGACGAAGTGGTGAGTGGCCTTTGCATCAAGCAGCTTTGGTAGAACAAGTTCTCCCCATGACCCTTTCTCTAAGCCTTGTGT
12507	chr19	44930577	44930828	+	0	test	CCAGGTTCAAGCAATTCTCCTGCCTCAGCCTCCTGAGTAGCTGGGATTACAGGCGTGCACCACCACGGGTGGCTAATTTTTGTAATTTAGTAAAGACGGGGTTTTGCCATGTTGGCCAGGCTGGTCTCGAACTCCTGACCTCCAGTGATCTGCCTGCCTCGGCTTCCCAAAGTGTTGGGATTACAGGCGTGAGCCAAATGCCCAGCCAAGGGTAAAGTGTTTAGACTTCAACGTGCTTTGGTCCACCTGGG
14578	chrX	149495047	149495298	-	0	test	CTCAAATGTATTTGATTCCTGTTTCACTGAGAGTTTGGAAAGCAATGCATCCATCTGTTTGTTTTCTAATTTTGTGTTTCTGGGCTCCTGATGGTGAAGTGGCTTGAACTGGCTCTGAGGCTGCACCTGGATGCTTTATTAGTGATGGTCACTTAAAATGACCCTTCAAAATGGACTGTGTGCCAGGAAGGGCGCTGATTCATGTGTGGTATCGCCTTTAGTCCTCATGGGGACTTAGTGAGGTACCTGCT
9483	chr15	50261850	50262101	-	0	test	CTGAACCACACATGTTTAGAGGCTCTTCCTGAGGCCAGCTTGAGGTTTGCCTTCATCATTCCCAAAGAACTTTTGACTTCAATGTGACAAAAAGGCCAGGCGCGGGGGCTCACACCTGTAATCCCAGCACTTTGGGAGGCCAAGGCAGGTGGATCACCTGAGGTCGGGAGTTCAAGACCAGCCTGACCAACATGGAGAAACCCCGTCTCTACTAAAAATACAAAATTAGCTGGGCATGGTGGCACATGCCT
13223	chrX	147916547	147916798	+	0	test	GTTTTCTTTTTTGGTTTTCTTTCTTCCTTTCTTCCTTTCTTTTCCTTTCTTTCCTTTCTTCCTTTTTTCTTTCCTTTTCTTTTTTCCTTTCTTTCCTTCATTTCTCTCCTTTGCTCCTTTAGTTTGTTTCTTTGTTTCTTTGGTTTGTTTCTTTGTTTCTTTGGTTTGTTTCTTTGTTTCTTTAGTTTGTTTCTTTGTTTCTTTGTTTCTTTCAACAGGTCTCACTGTCGCCCAGGTTGGAGTGCAGTGGT
4391	chr6	32182907	32183158	-	0	test	TTAGCTGGCACTTGGATGGGAAGCCCCTGGTGCCTAATGAGAAGGGTGAGTCCTAAGGTGCCCCCCAAGCTGCCTTCTCCCTGATCTCACTCCCACACCCACCCTGGGATAATTTGTCTTATCCTCCCATCATAGGAGTATCTGTGAAGGAACAGACCAGGAGACACCCTGAGACAGGGCTCTTCACACTGCAGTCGGAGCTAATGGTGACCCCAGCCCGGGGAGGAGATCCCCGTCCCACCTTCTCCTGT
13583	chrX	147928783	147929034	+	0	test	TGTGCCAGAAGACTTACGGCAAATGTAAGTTGATACACAAGAAATGCTGAGAACTTGGAAGTGATATGCAATTAGTTTAGAAGAATTTCTAGTAGTTTAAAATTTTTTAAACTACTAGAAATTGATTTTAAATTTGGTGGACTTTGGCAAAAATGGTTTGGTTTGCAAGAGCAATGGAGAAAAAACTTATTTAACTTTTCCTACTTATAGAATCAAGGCAGCCTTGCGCTTGTTTAAACTTACTTGGCTAT
3372	chr4	122452798	122453049	-	0	test	GCAGCATAAGTATAGTAGTAAAAGACATTCCTAAAAGTAACTCCAGTTGTGTCCAAATGAATCACTTATTAGTGGACTGTTTCAGTTGAATTAAAAAAATACATTGAGATCAATGTCATCTAGACATTGACAGATTCAGTTCCTTATCTATGGCAAGAGTTTTACTCTAAAATAATTAACATCAGAAAACTCATTCTTAACTCTTGATACAAATTTAAGACAAAACCATGCAAAAATCTGAAAACTGTGTT
15185	chrX	153905662	153905913	+	0	test	CTGAGCAATGGCCTGGTGCTGGCGGCCCTAGCTCGGCGGGGCCGGCGGGGCCACTGGGCACCCATACACGTCTTCATTGGCCACTTGTGCCTGGCCGACCTGGCCGTGGCTCTGTTCCAAGTGCTGCCCCAGCTGGCCTGGAAGGCCACCGACCGCTTCCGTGGGCCAGATGCCCTGTGTCGGGCCGTGAAGTATCTGCAGATGGTGGGCATGTATGCCTCCTCCTACATGATCCTGGCCATGACGCTGGA
15959	chrX	154322001	154322252	+	0	test	AAGAACTGGAAGGCTGGGCGCAGTGGCTCATGCCTGTAATCCCAGCACTTGGTGAAACCCCATCTCTACTGAAAATAGAAAAATTAGCCAGGTGTGGTGGCACGCACCTGTATTCCAGCTACTCAGGACGCTGAGACAGGAGAATTGCTTGAACCCAGGAGGCAGAGGCTGCAGTGAGCCAAGATCACACCACTGCACTCCAGCCTGGCGACAGAGAGAGGCTTTGTCAAAAAAAAAAAAAAAAAAAAAAG
1512	chr2	10443355	10443606	-	0	test	AAGACACAGTGTCTCGTCTCTTTATTTTACCCCAGCTTCCATGTAGGAAGCGGCTGTACCGATCCTGAGACCTTCGTGCAGGCAATCTCTGATGCCCGCTGTGTTTTTGACATGGGGGTGAGTATACGTGACCCTGTTAGGGAAGGGCGGGACACAACTGACAATAACTAGTCTTAATTCTAGAGTTAACTTTTTATGGCAGTTGGTTCTGTATTACATGGGTTTCAGCCTATCTGCTGCATACATTTTTG
14718	chrX	149501383	149501634	-	0	test	AACCTCTGTGCTTGGAGGATCCCCCACTTGGCACTTCGATGCTTTTGAAAGCTGGAAGTCCTTCCTTGGTTTCCAGTGCTGTTATACTGTCTTCACAAATAAACTATAAACTACTAAGCTTAAGGGTGGGTTTCTGCTTCATGAATGAGCACTTGAACTTGGAATTCTAAAACAAGGTTTTGAGTCCTGGCTCCCTTATCTAATATCTGGTTAGCATAGGTAAGTCACATACCTTCCCTGAGCCTCCCTCT
3562	chr4	154606919	154607170	-	0	test	TAATGAGCTGTGGACTGGTTCACATATCTATTGCCTCTTGCCAGATTTGCAAAAAACTTCACTCAATGAGCAAATTTCAGCCTTAAGAAACAAAGTCAAAAATTCCAAGGAAGCATCCTACGAAAGAGGGAACTTCTGAGATCCCTGAGGAGGGTCAGCATGTGATGGTTGTATTTCCTTCTTCTCAGTACTGCAGACTATGCCATGTTCAAGGTGGGACCTGAAGCTGACAAGTACCGCCTAACATATGC
11609	chr17	63930061	63930312	-	0	test	CTGCTACATGAACAGCGCCTCCGGCAATGTGAGCTGGCTCTGGAAGCAGGAGATGGACGAGAATCCCCAGCAGCTGAAGCTGGAAAAGGGCCGCATGGAAGAGTCCCAGAACGAATCTCTCGCCACCCTCACCATCCAAGGCATCCGGTTTGAGGACAATGGCATCTACTTCTGTCAGCAGAAGTGCAACAACACCTCGGAGGTCTACCAGGGCTGCGGCACAGAGCTGCGAGTCATGGGTGTGTGCCAGG
7735	chr11	116836128	116836379	-	0	test	CCTCCACCTTCAGCAAGCTGCGCGAACAGCTCGGCCCTGTGACCCAGGAGTTCTGGGATAACCTGGAAAAGGAGACAGAGGGCCTGAGGCAGGAGATGAGCAAGGATCTGGAGGAGGTGAAGGCCAAGGTGCAGCCCTACCTGGACGACTTCCAGAAGAAGTGGCAGGAGGAGATGGAGCTCTACCGCCAGAAGGTGGAGCCGCTGCGCGCAGAGCTCCAAGAGGGCGCGCGCCAGAAGCTGCACGAGCTG
6113	chr9	130699594	130699845	+	0	test	TAAGTCATTTCTTTGGGTCTTGTAACTTACGGTGGCCTGTGCTTTAGACTAGGCTTTTGTTAATGGTGTTAACTTTTAGCGCAGGGAGTGGAGAAGGTGTTGTTCACGTCAAGATATTAATGACAGGCCGAGTGTGGTGACTCATGCCTGTAATCCCAGCGCTTTGGAAGGCCAAGGAGGGAGGACTGCTTGAGCCTAAGAGTTCAAGACCAGCCTGGGCAACATAGCAAGACCCCCCTCTCTACAAAAAA
2091	chr4	39456851	39457102	-	0	test	GTTTCTGAGTTTTCTGTCTGCTGCCATAATGTATGTCATTATTCTTTCCAACCCACCGTCTCTCCCCTCAAGATAAAAAGTTCATGCTGAAATGGTTTTCTTTGCATTATACCTTCTAGAATGCACCTGATAATGGCACCCTGTACATCTATTATAAATCTGTTAAGATTCCCCCATCTACCCTTGCTCTTAGCCTTAAGTTCCTTGAAGGCAGGGGGCCTGCCTTTGGCTTGGTGGTCTCAATTCCTGCC
12585	chr19	50877489	50877740	+	0	test	AGGGGAGAGCCCTGCGGCACCTGGGGGAGCAGAGGGAGCAGCACCTGCCCAGGCCTGGGAGGAGGGGCCGGGAGGGCGTGAGGAGGAGCGAGGGGGCTGCATGGCTGGAGTGAGGGATCAGGGGCAGGGCGCGAGATGGCCTCACACAGGGAAGAGAGGGCCCCTCCTGCAGGGCCTCACCTGGGCCACAGGAGGACACTGCTTTTCCTCTGAGGAGTCAGGAGCTGTGGATGGTGCTGGACAGAAGAAGG
4872	chr6	33070397	33070648	-	0	test	TTTGAATTTATATGCCCCAAATGAAAAATATTATCAACAAAGCTATACATTCTACAGTTTCATGTTCATAAACTAAGACAGAAACTTTAAAACTGTCAAGAGCCCTAAAATTTGAAGGATATTTTCTTCTTCCTCTCAATTTTGTATTTTTTTCTACCTTTTCTATAATAAGAAAAAGAAAATGTCCATTCCCCCACCCCCATGACTCTAAAAACAATTTTACATCTGTGTCATAGAAAAATTAAGATCTT
9488	chr15	50262236	50262487	-	0	test	GCTCTGCCATGGTGCTCATGCTAAAGTGGCATCGACTCCCTTCCTGTGGCGGGGCTCCCTGCCATGTGCCAGCTCTGTGCTGGCCACATGACATGCTATCTAATTTAATCTTCCCAGTGGCCCTGTTTTACAGATGAGGAAACTGCTGATGCTCAGAGAAGTTAACTGTCTTGCCCAAAGTCACAGAATTAGTACCTGGTAGATCCAGGATTTTACCCTAGACCAGTCTTACTTCACAGGCTGTGCCGTTT
4315	chr6	32039909	32040160	+	0	test	CGGGTTCTTTTGCATACCCCAGTTATGGGCCTGTTGCCACTCTGTACTCCTCTCCCCAGGCCAGCCGCTCAGCCCGCTCCTTTCACCCTCTGCAGGAGAGCCTCGTGGCAGGCCAGTGGAGGGACATGATGGACTACATGCTCCAAGGGGTGGCGCAGCCGAGCATGGAAGAGGGCTCTGGACAGCTCCTGGAAGGGCACGTGCACATGGCTGCAGTGGACCTCCTGATCGGTGGCACTGAGACCACAGCA
14040	chrX	147942422	147942673	+	0	test	ATATTTCTATCAACTAAATGTATACTGTTTCTCGTACTTGTTGTAGCAGAGTTGTAGTCTTTAGGAAGTTTTCAAGGCATGATGTAAGACCTAGCTGTCTGGTCAGCTATTAAATAGTTTTGTTTTCCCCTCATGAGGAATAGTCAGCATTTTCAGTTGATGAAAAAATCATTGTTCCTTAACCATAGTATATGTTCTTTTAAAACAAAAGATGGCCCAAACTCAATGCTTTGTAAATTAAGATATTAACT
2900	chr4	71776814	71777065	-	0	test	TAAATTACTAATCTTTGAAATAGAAATTTTCCTATACAAGATAATGCAATCTGTTGGACTTCAAATGCATTTAAATGGCTCGGGGATCTGTTTGACAAAAACATGTGTGATGATAGAGACATAAATAATGATACACATCCAATATATCCTCTCCATTAGATTGCATAATTTGATTTCGGTTTATACAGTATAACTTCAGTAGTCATTAAGACGTGTATTTTGTTAGGAAACAGTGAAAGAAAATCACCCAT
396	chr1	109659527	109659778	+	0	test	AGCATGGATGCAGCTGGCAATAGTAGGACTGACACACGGTGGCATTGACGTCGAGTACGAAACCCACAGGCAGTATTCATAGCTACTCCCAGAAGCTTTGCACGATCAGACCCCCACGTGGGGAATCCTGAGAGCCAGAGCTGTGGCCAGAGCTGGATTAGGGTACATATGTGGGTGCCCCTGTTGAAGGAGTGTATGTTGAAGTGCTCTGTGCTGGGGCACTCTCCTTCTTTATCTTTTTTCCTCTCTTT
9694	chr15	90891709	90891960	+	0	test	GTGGCAGGGCTTGGGGAGTGTGGGTCAGGCCCACCCAGCGTCTGAGCAGAAAGGGCTTTCCAGGCCCTCCGTCTACATACAAGATGCAGAGTGAGTGACCCTCAGGGCCAGCCTTGCTCTAGGTTTGGAATGTCAGGGCCACTCCTATGCCATGGGCTGTACACACCAGGTTGGTGCTTACCTGGTCAGGGCACCTGCCTGGACCCCGTAGTCATCTCAGTGTGCTCCCCACGTGGTCCCACCCCTGGTCA
1869	chr2	162144330	162144581	-	0	test	AGAAAAATCTAATATAGATGAAGCTTTAACCTCTTAATACTGCATTTTGCAAGGCTGTTCCTGCAAGCTCTGGTTTTATAGGATATGATATATTTAGTTGAATTACAGACTAATAATCTCAACAATAGTTTCTGTATTGTCAATATACTAAAATCTTCAAAACAGCCTAGAAGATTGAAAAGGGCATGAAATTATGCAGGCTTGGTATTAGATCCCAGCTCTGCTACTTTCTCTTTCCAGTAGTCACTGGT
6388	chr10	47306696	47306947	+	0	test	CGGGAGCCCAGGGGACCCATTGTGGGAGGTTCCATTTGAGCAGATACCTGAAGAGGTTGAGTGGTACTGGATGGGGAACCATGAAGGTGGAAGGAAGCGTGAGGCACAGTCCTGAGAATAAAGCCACATGTTGGAAGGAGAAGAACGCCAGTGTGACGGGGGCAGAGGGGTGGGGTTGGGCTGTGGATAGGAGATAAAGTTGCAGAGGTAGGCAGGCCAGATCAAGTAAGGCCTTGGTAAGGGGTTGAGTT
7985	chr11	119092282	119092533	+	0	test	GACCTTGGGATGCTACCGGTCCAAAAGGCGCTGGGGAGCAAGTAGATAGAGGTGGTCCCATGCTTTGCGCCATTGGTTGGGGAAAGATCAGGCCTGATGTCCTAGGATGTTTTTCCATCAGGGGGCCTTGGGCGTGGAAGTGCGAGCCAAGGACCAGGACATCTTGGATCTGGTGGGTGTGCTGCACGATCCCGAGACTCTGCTTCGCTGCATCGCTGAAAGGGCCTTCCTGAGGCACCTGGTAGGGCCTG
8786	chr14	94566354	94566605	+	0	test	CCACTCTCCAAAACAACTATAATAACAGATTCGTCCTTCTCAAATGTCCTCCCCACTCATGACTTTCACTTAACAATCTTGTCTCACTTTTCATTGAGAAAACAGAATCAGTTAAACAGGAAATCTCTCATCTTCTCACCTCCAAATCTACTCACCTCCAAATCTTCTCACCTCCAAATCTACTCGCCTCCCTGCATTTGAACCCACAAGCTCAGATGCCCTCCTCTCATCATGGCAGTAGTGACCCACCT
2117	chr4	69956447	69956698	-	0	test	GAAAAAAAAGCTAAAAAATGTTCTGTCATTTTCCTTGTGCACATCTCTTTTACACAAGCCTTACTTCACATCTTGTTTTTGCTATAAGTATATATGAAGGCAAAAGACTGAGATGCTTATTTCACTACTTACAACATTCTTAAGGCAAGTTTTCTTACTAAGAGGTTATTTATTTATTTATTTATTTATTTATTTTACACAAGCCTTACTTCACATCTGGTTTTTGCTGTAAATATATATGAAGGCAAAAG
700	chr1	159304633	159304884	+	0	test	CCCCTGCATCTCTTTTCTTCTACCCCCTTCCCTTTTGATTACTTGTATGCCTTCTTTCAATATTCTAGTCATCTCTCAATATTATTCCTCCACCCTATTTTCCTCTATCTTTTCTGCCTAGATTCAGGTATATATTATGTGGTCAAACAGCATGACATATATGTGAACATTTCAAAGAGCTGTGTATCTGGAATAGGATCAAAAGGTTTGACTTAAAGTTTTGCTCTGCATAATCCATATGGCAGGACCTG
14110	chrX	147945394	147945645	+	0	test	TTGTGTAACTGGAATCACTGGGGTATTGATTACATTTCAGATTCTGTGGTTGGTTTAAATAGAGGGTGTGGGAATAAAGAACTTCCAGTAAGCATTTCAGAATCAGTAACTGTTGAACCTTTTGAAAATATTCTCATAGGAAACGACGATCACTCCCGAACAGATAATCGTCCACGTAATCCAAGAGAGGCTAAAGGAAGAACAACAGATGGATCCCTTCAGGTAAAACCTGTCTGCCTCTTTTCATCTTA
15626	chrX	154308630	154308881	+	0	test	ATTCAGTATGCTTGCAAGTTGGGCCTCTCCAGTCAGTGCAGCAGCTCTTTGGCATGACCTTGTGGCCCACAGAGGGCCACATTTTCTGCTTGGGATAGTGATGACCCTGTGCCTGCAAACAGGATGTTGGGGGTGGGGTAGATGCTAGTAGAAAGGGATTCATTTCTAGCGTGGGCACTTCTAGATGAGCACGTGTAAGATCTCAGTTGACCTGAGTTTTATAAACCAGGGAACCAGAGTATTCTCTTTCT
8978	chr15	50243271	50243522	-	0	test	AAAGCACATGCCTTCTGGGTTCTGGGGCTGAGTTGAAAGAGGGCCCTGAGATGAGTTCTCGAGGTGGATCCCATGTGACAGGAAACAGGGCGCCATTGGAAATTTTCCAGCTTGGAATATGAGCAAAATCAAATGGAGATGACAATGAGGCAGAAGCCGCTTACAACGGCTGTGACGGGAAGATGGCCCTGGTTTAAAACCATTTAGCAGTCTGAAAGTTTATGCATAGTCTCTTGTCTGATGATTAATCT
10781	chr16	2121407	2121658	-	0	test	GGAGGTGGAGTTCTCGGTTTGCCCCAGCTCTCTGTCTACTCACCTCCGCATCACCAGCTCCAGGACCTGGTTTGTAACTCGGGCAGCTCTGAAAAGAGAGACATGCTGCCGCCCTGTGGTTTCTGTTGCTTTTTCTTCACTGACTACTGACATGGGATGTTTTTCCTACGGCTGTGACCAATTGTGCTTCTTCTAATTGCCTGGTTTTTCTTTTTTTGTTTTTGGAGTTTTCTCTTTCTTTCCTCCCTCCC
9971	chr16	2093111	2093362	-	0	test	CCTGCCAGGGCCCAGCTCCAATGCCCACTCCTGCCTGGCCCTGAAGGCCCCTAAGCACCACTGCAGTGGCCTGTGTGTCTGCCCCCAGGTGGGGTTCCGGGCAGGGTGTGTGCTGCCATTACCCTGGCCAGGTAGAGTCTTGGGGCGCCCCCTGCCAGCTCACCTTCCTGCAGCCACACCTGCCGCAGCCATGGCTCCAGCCGTTGCCAAAGCCCTGCTGTCACTGTGGGCTGGGGCCAGGCTGACCACAG
4104	chr6	29670318	29670569	+	0	test	CTGTTTGTAATTGTGCCGGTTCTTGGACCCTTGGTTGCCTTGATCATCTGCTACAACTGGCTACATCGAAGACTAGCAGGTGCAGTGGCTGGGCAGCAGGCAAGACCACCAAATAGTGGGGGACCAAGTCAGCTCTGAATGGGAAGCCAAAAGAGAATAGAACCAGGACTCAAGATTAGGGGAGCTGGGATTTCCTTATTCCTCTGTCCCCATGCCCAACCCCAGGCTCTTCTGAGAAACTGTGAAGAGAA
839	chr1	161222733	161222984	-	0	test	CCGTGACTGACTATGGCAAGGACCTGATGGAGAAGGTCAAGAGCCCAGAGCTTCAGGCCGAGGCCAAGTAAGTCTCAGGGCAAGGGGTTCAGGGGCTGTGGAACTGTGGAGAGAAAGAAGGGAAGATGAGAGGTCCCACAGAAGTCTGAACCCAGGGGTGGGGATTAGGGCAGATTAGGCTTAAATTGCAGAGAAAAAGTATTTCATCACCCAAAGATCCCACACGTCTCTTAGATAGAGAGGAACAGCAA
11358	chr16	28936647	28936898	+	0	test	GCTCCACCCCCACTCTCCTCATCCCTCCAATCCGCTGTGCGCCAAGCCTTCTGGAGCTCGGAACTCCGCCCCCGGGGCGGGGAGTCCCGCCCAGCTATGAGCCCCGCCTCTAGAACCAGACCCCGCCTCCAGGGCTCAGAGCCACGCCCCCAGGACCCAGAGCCTGAAGTCGTAATCAAGAGCAGAACTTCGCCCCAGAACTGAAGGCCTCGGCCCTAGATTTAGATTCCGCCCCAGGGTTCAAGGCCGGG
829	chr1	161222587	161222838	-	0	test	TGAACCCAGGGGTGGGGATTAGGGCAGATTAGGCTTAAATTGCAGAGAAAAAGTATTTCATCACCCAAAGATCCCACACGTCTCTTAGATAGAGAGGAACAGCAAGAACTGGGCCTTGAATTTCAGTCTCTAGAGTCTGTCCCTCTACCTAGCAAAGGTCTTGACTCTATTCCTACCTAGGGGCTTTGCCATGCGATGGACCAGGCACTAGAGTTTGGGGACCTGAGTCAGTCTGCTCTGACCTCCCACCA
1163	chr1	186678498	186678749	-	0	test	TTAAAATAAATTTGGATTTGGATCAGCAAGAAAATAACTTTCCATGATTCTAAAGTGGGTGCCATACTCAGCCATTCCTTTCATAGGCCTCTTGGATAGTGAGCAGATGGCTACCTGAAAAATCAATATTGCCAGATTATAATGTGCAGAGTATATGTATTTTATTAAAGATGTATTTCAAGTGGCCATTAGACTATAAAGTGTAGTTGTTTAAAAATAGATTTTTTTTATTTTGGAGTTACATTCAACCT
12263	chr18	46097427	46097678	-	0	test	GAATAAATGAAGGTGGTTTTTGGAATACTTACGTAGGCTTTCTCTTAGTGCTGAATACGAAACGACCGTGAAAACGTCCTGCAGGTGCAAAGAACCATTTCATGTGGCCAGCACTGTGACAACAGTGAAGCTGGTTTCTAAATTGTCTCTCCGGCAGAGTGTGTGCTTTGTGGCACTTGGACGTGTTTAAAGAATTTGTCCTTGAGTTTTCCACCTGGTTTTGAGTAATTTTTAGTGATGGAGAATTCAAG
6698	chr10	47355316	47355567	+	0	test	GCACACAGGGCCTCACTGTGAGCTCAGCCCCTGAACAGGCTCTGCTTCCCATCCTTCAGGTTCAACATCGGTGGCCCCACATCCTCCATTCCCATCTTGTGCTCCTACTTCTTTGATGAAGGCCCTCCAGTTCTGCTGGACAAGATCTACAGCCGGCCTGATGACTCTGTCAGTGAACTCTGGACACACGCCCAGGTTGTAGGTACGTGGAGAAGCTTTCTCCTTTCTGCTGTCATTCTAGAAGCTTCTGG
15495	chrX	154305136	154305387	+	0	test	GGCTTTAAAGCGCCCTTCCAACCTGTGAGCCCTGCATTCCTTACCCTTGGTTGGATTCTTCCCGGGCTGGGAGAAATGACCGCTTCTATGAGGAGACCATGTGCCGAGGTCGTGTGCTAGGAAGCCAGTTGCTGTGAGAAATGACCAGTGTCATGTCTGTCTTTCAGCCACCCTACATCATGTAGCAGTTCTTCTGAGATCATGTCTGTGCTGTTCTTCTACATCATGAGGTACAAGCAGTCAGATCCAGA
6101	chr9	130699155	130699406	+	0	test	TCGTGATTATCACTGATTCCCCTCAAGTGGCTTGGGTTACCTACTCTTGCTATAGTTCTCAGTGCTTGAATATTGTGGTTGGCCTCAAACATCTCAAAAGGAAGTGAAAGAGAAGTGATGGAGACCTTTCTGAGGATCTGGCTTAAGTAATGTCATTTCTTTGTGTATTTCAGAGGAGAAGATCTGCAGAAGATGAGCTTGCAATGAGAGGTTTCTTACAGGAAGGGGACCTTATCAGTGTATCCTGCGCT
15039	chrX	153869874	153870125	-	0	test	GGCGAGTACCGCTGCCTGGCCGAGAACTCACTGGGCAGTGCCCGGCATGCGTACTATGTCACCGTGGAGGGTATGGACCTCCTGGGACAGTGGCCTGTGATGCCCACTGTCATGGGGGAGGGGAGGGTCTGCGCCGGGTCAAGAGCCAGCTGCTGGCTGCGGGCTCAGGGCGGCTCTCCCCTTCCTCCCAGCTGCCCCGTACTGGCTGCACAAGCCCCAGAGCCATCTATATGGGCCAGGAGAGACTGCCC
11146	chr16	2133220	2133471	-	0	test	GCTGGCAGGTGTTCAAGGGTCCAAGAGCGTTGCTGTCTGGGTGTCACCAGTAGCCTTCCTGGGGGGCTCACGCAGGTGCCTCTCCACTTGTGGCTCCCTGGCTGCTGAAGCTCAGCAGGGACAGCTGTGTCCAGTTCCAGGTGGAGGACAGCCGGGGCTTCTGAGGCCACAGCCTGCCTTGGGTTAATGATGCTGCCGAGAGGTGGTGGCTTTTGGAAAAGATGGCGTACTGCAAAACGTGCTGCTCTGCG
16265	chrX	156000568	156000819	+	0	test	TCTGTCCCAGGTGGCAGACACTGGTTTCCCCTCCTGCTCTCACAACCGGCCTGTTACCAGGTGTTGTCTGAGCTGTGGTGAGGCTTCCCTGGTGACATTCAGGAGCAGGGAGCCTGTGAGTAAGGGTGTATGCATCTGCCCTGACTGCCTGGCCCTGTGGTCAAGGATGGGGGAAGGCAGCTCTGCCTGCAGCTCCACCCCATTTATAAAGCACTGTGGTGCCTTCTGCTGGGGCATGTGCTGAGTGGTGC
14252	chrX	149483750	149484001	-	0	test	TCTTAAAAAGGAAGGAAACTGACACATGCTACATCATGGATGAGCCTTGAGGACATTATGCTAAGTGAAAAAAGTCAGTCACAAAAGGACAAATACTGTATAATCCCACTCCTATGAGGTATCTAGAGTAGTCCAGTTCATAGACATAGAAAGTAGAATGGTGGTTTCCAGTTGCTGGGTGAGGATGGAGAAAGGGGAGTTGTTACTTAATGGGGACAGAGTTTCAGTTGTGTAAATTAAGAGGAGTTCTG
4609	chr6	32936514	32936765	-	0	test	CTTAGATTTCAACTATTCTGGTGCCAGAAGCAGATGGGAGCTGAAGGAATGATGAAGGTTGAAGAAGGGGGGCTTTTCTTGGTGTGGGGCAGTACTGCATTTGGCCTGCTCTACCAAGCATACGGGAGTAGTAAAGCCACGGCTGGCAGACCATTTGGCATGCATGCTCAGGGGCCAGTGGATAAAGAATTACTTACAGTTCAAACACTGTTTGAACTCAGTGTCGGGAGTAGTTAAAGGTATCGTGAGAA
11693	chr18	31593051	31593302	+	0	test	AGGACTTGGTTTTATCTTCCCGTTTGCCCCTCACTTGGTAGAGAGAGGCTCACATCATCTGCTAAAGAATTTACAAGTAGATTGAAAAACGTAGGCAGAGGTCAAGTATGCCCTCTGAAGGATGCCCTCTTTTTGTTTTGCTTAGCTAGGAAGTGACCAGGAACCTGAGCATCATTTAGGGGCAGACAGTAGAGAAAAGAAGGAATCAGAACTCCTCTCCTCTAGCTGTGGTTTGCAACCCTTTTGGGTCA
11018	chr16	2129587	2129838	-	0	test	TCACTCACGCCTGGAATCCCAGCAGTTTGGGAGGCCAGGGTGGGTGGATCGCTTGAGCCCAGGAGTTTGACACCAGCCTGGGCAACAGGGTGAGACCCCGGTCTCTAAAAAATAAAAGAACATTGGCCGGGCGTGGTGGTATGCATCTGTGGTCCCAGCTATTCAGGAGACTGAGGTGGGACATCACTTGAGCCGAGGAGGTCAAGGCTGCAGTGAGCTGTGATCACACCACTGCACTCCAGGCTGGGTCA
11068	chr16	2131009	2131260	-	0	test	GGATTACAGGCTTGAGCCACCGCCTGTCTTTTAAATGTCCGATGATGTCTAGGAGCTTCCCTTCCTCTCTTTTTCCTTGTGCAATTTGTTGAAGAAACTGGCTCCTGCAGCCTGGATTTCTCGCTGTGTCTTGGGGGTGCCACCTCCATGGTGTCACCTCCGTGGTGCTGTGAGTGTGTGCTTTGTGTTTCTTGTAAATTGGTCGTTGGAGCCGACATCCCATTGTCCCAGAGGTTGTCCTGGCTGGCACT
14159	chrX	147946791	147947042	+	0	test	TTAGTGAGAGATACTTTTTAAGACAAAGTTTATGATGGAATATTTCTTGGAATTCATAGCAGCTCAAAGTAAGTGTTAACTATTGGTGGGTTCTTAAGGACCAGCATGTCCAAGGGATAGATAGATTGGTAAATTACTGTATGTACCTGTCTTTTACAAAGCTATAAGATATTTAAGTACCCGTTTTGTAATTTTCCACTTAAACTGATATGTTTGATGAACTTGGCAGTCTAACTGCATTGGTTAAAAGG
15400	chrX	154301755	154302006	+	0	test	TTTTTTTTTTTTTTTTTAGTAGAGACGGGGTTTCTCCATGTTGGTCAGGCTGGTTTCGAACTCCCGACCTCAGGTGATCCGCCCACCTCGGCCTCCCAAGATGCTGGGATTACAGGCATGAGCCACTGTGCCCGGCCTAAGTCCCCATTTTCTTTTTTCTTTTTTTTTTTTAAATGAGACAATTATTTATTTTAGGCAGATAGTCAACTTAAAAATCATTCAGCGTGGGCACTCCTGAAACTGTCAGGATC
10021	chr16	2094622	2094873	-	0	test	CTGAGAGCCCGGGACTCGGCGTCTCGCAGTTGGTCTCGTCCTCCCCCTCAACGTGTCTTCGCTGCCTCTGTACCTCTTCTCTAGCAGCTCTGGGACCGGGCATATCAGCATGGTGGCCCGATGCAGTGGCACAGCCTCGGTGGTCACTGGCTCCTGGAGACACAAGCAGATCTCTGGCCTCAGGGAGCCCTACACACTGTTGGGATTTGAAAGGCATTCATATGTTTCCTTGTCCAGAAGTTAATTTTAGG
13753	chrX	147933795	147934046	+	0	test	TTACGTAGTTTCAATGCTAATTTAGCCTGCTTTTTTGTCAGCATAGACTCATAAACATCTTCCTTGTCATTAACAAACACTAGAAATGATAAATGTATCTCCCAAAGAAAATTAGCTTGAAGATAAAGTGCAAATAGTAGGAAAATAATGCAGAAGTGTTAGCTCAACTGTGTACTACAGACAACTGGAGAAATGTGTTTGTAATGTTACTAAGGTAACTTGCATACCAATTGTATGTGTACAGAATGTAT
6817	chr11	5225615	5225866	-	0	test	AGGTTTCATATTGCTAATAGCAGCTACAATCCAGCTACCATTCTGCTTTTATTTTATGGTTGGGATAAGGCTGGATTATTCTGAGTCCAAGCTAGGCCCTTTTGCTAATCATGTTCATACCTCTTATCTTCCTCCCACAGCTCCTGGGCAACGTGCTGGTCTGTGTGCTGGCCCATCACTTTGGCAAAGAATTCACCCCACCAGTGCAGGCTGCCTATCAGAAAGTGGTGGCTGGTGTGGCTAATGCCCTG
11675	chr18	31592634	31592885	+	0	test	CCTATTTTCTCCCTTAAAATTCATTATACACATCCCTGGTTGATAGCAGTGTGTCTGGAGGCAGAAACCATTCTTGCTTTGGAAACAATTACGTCTGTGTTATACTGAGTAGGGAAGCTCATTAATTGTCGACACTTACGTTCCTGATAATGGGATCAGTGTGTAATTCTTGTTTCGCTCCAGATTTCTAATACCACAAAGAATAAATCCTTTCACTCTGATCAATTTTGTTAACTTCTCACGTGTCTTCT
14071	chrX	147943815	147944066	+	0	test	CCGTGCTTCTTTCTTCTCTCCCAGTACCTTTCAGTGAACCTGATCCCCACCTGCCAGTGCCTACTGTCTTTCTTGCTCAACAGTGTATCTCCTTTGTAACTTGCTAATGATGGTATAAGGTATAATCCATTTCACGCATATTTGCATTTCAAAAAGGAACTCTTTAAGATGGAAAAGCCTTCAAGATCCACAGGAATGAAGCTATAATGGTGAAGCTATAATTGGTCATTAGCTCCAACAGAGGAAGAGAG
4231	chr6	31161404	31161655	+	0	test	GTTTTTGTCCTCCATATTTGCCTGGTGCCCCACCATCAACAGGTACTTTGGTCAATAATGTCCGACTCCCAAGAGGTCACAGGCTGGAATTGAGTGATGGAGACCTCCTGACCTTTGGCCCTGAAGGGCCCCCAGGAACCAGCCCCTCGGAGTTCTACTTCATGTTCCAACAAGTACGAGTCAAGCCTCAGGACTTTGCTGCCATTACCATCCCACGGTCTAGGGGAGAAGCCCGGGTTGGGGCTGGTTTC
9863	chr16	180855	181106	+	0	test	CCGGCCAGCTTCCAGGTGAGCGGCTGCCGTGCTGGGCCCCTGTCCCCGGGAGGGCCCCGGCGGGGTGGGTGCGGGGGGCGTGCGGGGCGGGTGCAGGCGAGTGAGCCTTGAGCGCTCGCCGCAGCTCCTGGGCCACTGCCTGCTGGTAACCCTCGCCCGGCACTACCCCGGAGACTTCAGCCCCGCGCTGCAGGCGTCGCTGGACAAGTTCCTGAGCCACGTTATCTCGGCGCTGGTTTCCGAGTACCGCT
10397	chr16	2108231	2108482	-	0	test	TGCTTTGTGTTTGTCGTGTCATTTGGGGACACGCCACTGACACAGAGCATCCAGGCCAATGTGACGGTGGCCCCCGAGCGCCTGGTGCCCATCATTGAGGGTGGCTCATACCGCGTGTGGTCAGACACACGGGACCTGGTGCTGGATGGGAGCGAGTCCTACGACCCCAACCTGGAGGACGGCGACCAGACGCCGCTCAGTTTCCACTGGGCCTGTGTGGCTTCGACACAGGTCAGTGCGTGGCAGGGCCG
15389	chrX	154301280	154301531	+	0	test	TGAGGTGGGGTGGATCACCTGAGGTCAGGAGTTTGAGACCAGCTTGGCCAACATGGTGAAACCCTGTCTCTACTAAAAATGCAAAAATTAGCCGGGCGTGGTGGTGCACCTCTGTAATCCCAGATACTTGGGAGGCCGAGGCAGGAAAATCACTTGAACCCGGAAGGTGGAGGTTGCAGTGAGCCGAGAGAGTGCCACTGTACTCCAGCCTGGGCAACAGAGTGAGACTGTGTCTCAAAAAATAAATAAAT
12826	chr22	20783236	20783487	+	0	test	AACATGGCATCACCCAAATGTCTTGTTAGTCACTACAGAATCACAGTGTGAGGGATGAAGGCCATCAAGACAGAGCTGAGGCTGGCAGGGTGGCTCATGCCTATAATCCCAGTGCTTTGGAAGGCTGAGGCAGGAGGATTGCTTGAGGCCAAGGGTTTGAGACCAGCCTAGGTAACATAGCAAGACCCCATCTACAATTAAAAAAAAAAAAAAAAAGACAGAAAGAAAAAATAGCCAGGCGTGGCATGTGC
4028	chr6	29665421	29665672	+	0	test	AAGTCAAAGGTTCTCTTCATATTATTGTGGTGTATCGCCTACAAGCATAATTAAAATAAACACTAAATTTCAGTTTAAAGTTTACTGAAAATAAATATGTATTTTTTATTCCCTATTTAAGCTTTGAATCCCCTGACTTCCTATACCATTACCACTGTCCTAGTTCAGGTTCATGTTGTTTTTTACTTTAATTGTTATCACAGTCTCTTAACATTTCTCCCTATGTTCTCCAGTCCTGTAGGTGCTAAATC
4226	chr6	30492165	30492416	+	0	test	TGATGGGGACCTGATCCCAGCAGTCACAGGTCACAGGGGAAGGTCCCTGCTGAAGACAGACCTCAGAAGGGCAGTTGATCCAGGACCCACACCTGCTTTCTTCACGTTTCCTGATCCTGCCCTGGGTCTGCAGTCACAGTTCAGGAAACTTCTCTGGGATCCAAAACTAGGAGGTTCCTCTAGGACCTTATGGCCCTGCCTCCTCCCTGGCCCCTCACAGGACATTTTCTTCCAACAGGTGGAAAAGGAGG
4337	chr6	32040373	32040624	+	0	test	CTGGGGCAGGACTCCACCCGATCATTCCCCAGATTCAGCAGCGACTGCAGGAGGAGCTAGACCACGAACTGGGCCCTGGTGCCTCCAGCTCCCGGGTCCCCTACAAGGACCGTGCACGGCTGCCCTTGCTCAATGCCACCATCGCCGAGGTGCTGCGCCTGCGGCCCGTTGTGCCCTTAGCCTTGCCCCACCGCACCACACGGCCCAGCAGGTGACTCCCGAGGGTTGGGGATGAGTGAGGAAAGCCCGAG
7416	chr11	14970032	14970283	-	0	test	ATACAAACTGGCTCCAGCACACCACTGTTTAGAGGCCACACCAGTGCCTGGGTCCTGAGGAGGACACTGGCCTTGTGCCCTGTCCCCTAGGACTCCCGCTGGCCACATCCTCAGGGGAAGAAGCAAAGACCAGGAAGCCTGGCTGCTTATCCTGGGGAGGGGCAGGCAGGGGCTCACAGCCTGCACTGAGTTTGCTTCCCCTCCACAGGTCTGCCCTGGAGAGCAGCCCAGCAGACCCGGCCACGCTCAGT
6178	chr9	130701333	130701584	+	0	test	CAGTGGATGTTATTGCTGGCAAAGCTTGCTCTGTCATCTGAGCCTCTAAATGGAATTTCACTGCTTTTCTTGTACGAGGTTGCATTATTGAAGCTATTTACTATTTTAAGGAGAGCATACTTCCTAAAACAAATTAGGATTTACTAGTCTCTTAGCTTTATCTGGGTGAACGTTTTTATTTTCTGAGTGGAACAAAAACCTCTTCCCTTAATGGAGTGGCTGCAGACAGCCACGTTGATGTACATGGAGTA
12865	chr22	20784564	20784815	+	0	test	CTTCCCCGGGTCACTTGAGAAAATAACAGAATCAGCGATGCTGAGCGCCCCTCCCAGTACTTGGAACCTAGGAGGCACTCAAAAAAAGATTGGCTCAACTCTTCCCTGCCCAGGAAATTCCAAGGTCCTCTTAGCCTACCGAGGACACATCATTCATGATTTCCTCTATTATTATTCGTTACTTTGTAGTTAAAACTGCAGGTGTTAAGTACTTATTGAGATTATTATTGGGTCATGGCAGAAAGAATGGA
9953	chr16	2092657	2092908	-	0	test	GCCTTTAGTCCAGCCAGACCCTAGGGGACATGTGGACATGTGTAGATACCTTTGTGGCTGCTAGAACTGGAGGTAGGTGCTGCTGGCATCAGTAGGCAGAGGGGAGGGACACAGGTCCGTGTCTTGCAGTGCACAGGACGGGCCCATGACAGACAACTGTCTGCCCCAGAACATCCCCAGGATAAGGCTGAGAAGCCCAGGTCTAGCCGTGGCCAGCAGGGCAGTGGGAGCCATGTTCCCTGGGTCTCTGG
6670	chr10	47353374	47353625	+	0	test	TAACGTGCTTGAGGACAACATTGGCTACTTGAGGTTTGACATGTTTGGGGACGGTGAGCTGCTCACCCAGGTCTCCAGGCTGCTGGTGGAGCACATCTGGAAGAAGATCATGCACACGGATGCCATGATCATCGACATGAGGTCAGTGGCCAGGGGTCAGTGCTTCCTAGCCAGGACGCAGGGCTGCCAGGGGACAGTCAAAGCTATGGGCCACAGCAGGGAAGAAAAGGAACCCTGTGACACAGCAGAGG
10203	chr16	2100533	2100784	-	0	test	GTTCTTTCCATGTAACTTAATCATGTCCTTGAGGTCCTGCTGTTAATTGGACAAATTGCAGTAACCGCAGCTCCTTGTGTATGGCAGAGCCGTGCAAAGCCGGGACTGCCTGTGTGGCTCCTTGAGTGCGCACAGGCCAAAGCTGAGATGACTTGCCTGGGATGCCACACGTGTTGGGCAGCAGACCGAGCCTCCCACCCCTCCCTCTTGCCTCCCAGGTACCACGGCCCACGTGGGCATCATGCTGTATG
1479	chr2	10442013	10442264	-	0	test	ATGTCATGTCACAAGGCCCTGTGATGTTACTCCCCCATGTGAATTTCCCACAATGAAGGCTGCTCTTTCTTTTCTGTTTCACTCTCTTAGATCACCGGCGTAATCAACCCAGCGTTGGACAAATACTTTCCGTCAGACTCTGGAGTGAGAATCATAGCTGAGCCCGGCAGATACTATGTTGCATCAGCTTTCACGCTTGCAGTTAATATCATTGCCAAGAAAATTGTATTAAAGGAACAGACGGGCTCTGA
388	chr1	109658681	109658932	+	0	test	TGAAGCCAGTTCCAGCTGTGGGGAAGATGGCTGCTTGCTCGTGGCCAGCTGGGGCCATACACAGCCCTGGGGAGGCCACATCTGTGCAGGGAGCTTGTGTCTGAGGGTGGTGACAGCTGTTTTCTGCCTCAGGAGAAACTGAAGCCAGAATACTTGGAGGAACTTCCTACAATGATGCAGCACTTCTCACAGTTCCTGGGGAAGAGGCCATGGTTTGTTGGAGACAAGGTAATGGGGGCATGTGATGAGGA
59	chr1	67687014	67687265	+	0	test	TAGAAATACATTAAGAGGATAGAGTGGAATTTTTTTTCTCTGCAATCTTGCATTTTTTTAATGGCTCTTTTTTTTTTTCCTGATAAAAACCTTTGGTAGGTAGGGAAGTTATGTTTTCAGGGGTAAATGTGCTACTTTTGTCTTCTAAATTTTGCTCTTTTTTGACTGGTCTAGTCAAGTGACAGCCCGATTATTTTGCTACTCCTTAAAAGTACTATTCTGTCTCTTGGAGTATGGTTGATGGCAATTCC
3679	chr4	154610221	154610472	-	0	test	TATGCCCCATTCTAAGTAAAAAGATTCAGGTCCACATTGTATTCCTGTTTTAATTGATTTTTTGATTTGTTTTTCTTTTTCAAAAAGTTTATAATTTTAATTCATGTTAATTTAGTAATATAATTTTACATTTTCCTCAAGAATGGAATAATTTATCAGAAAGCACTTCTTAAGAAAATACTTAGCAGTTTCCAAAGAAAATATAAAATTACTCTTCTGAAAGGAATACTTATTTTTGTCTTCTTATTTTT
735	chr1	159305449	159305700	+	0	test	TAGACAGGGTCCCTGACTTCTTGGAGCACAGAGCAGTATGGGAAGAGGACATTAAATAAAGAATTACATAAGTAATTAATTTAAATTATACATGTTTTGAAGAAGTTTTTTTTTGACAACTATAATTAACACTAGAACTGGGAAGTTTCTATAAGGTAAGAGAGGACAAAATAGACACTCTCCTAAGCTAAAATTCCCAAGAAAGACTGTTTATTTTCCCCTAACTAACTAGAACTAGCAACAGAAGATCT
6733	chr11	4385678	4385929	-	0	test	CAAAATCCCCCCACCACAGGCACAGACTTAGTGAACTCCCCCCATGCAAGGCCTGACTGTGGTCCTCTCTCTGCAGTCCACATCACTCTGGATCCAGACACAGCCAATCCGTGGCTGATACTTTCAGAAGATCGGAGACAAGTGAGGCTTGGAGACACCCAGCAGAGCATACCTGGAAATGAAGAGAGATTTGATAGTTATCCTATGGTCCTGGGTGCCCAGCACTTTCACTCTGGAAAACATTACTGGGA
10475	chr16	2111320	2111571	-	0	test	GTGCGGCGGCCCAGGCGGATGTGCGCGTCTTTGAGGAGCTCCGCGGACTCAGCGTGGACATGAGCCTGGCCGTGGAGCAGGGCGCCCCCGTGGTGGTCAGCGCCGCGGTGCAGACGGGCGACAACATCACGTGGACCTTCGACATGGGGGACGGCACCGTGCTGTCGGGCCCGGAGGCAACAGTGGAGCATGTGTACCTGCGGGCACAGAACTGCACAGTGACCGTGGGTGCGGCCAGCCCCGCCGGCCAC
11621	chr17	75778610	75778861	-	0	test	GGATTTCAAAACCGACCTGAGGTTTCAGAGCGCAGCCATCGGTGCGCTGCAGGTAAGACAAAGGCCTGGAGCCGGGGGAGGGCTGGGCGGTTTCCGCTCCCCGAGTGGGATTAATAGTGCGGCTCTCGTCCTCAACAGGAGGCTAGCGAAGCGTACCTGGTGGGTCTGTTCGAAGATACCAACCTGTGTGCCATCCACGCTAAGAGAGTCACCATCATGCCCAAAGACATCCAGTTGGCTCGCCGGATACG
6607	chr10	47351225	47351476	+	0	test	CGACATCACTGTGCCCATGAGCGAAGCCCTTTCCATAGCCCAGGACATAGTGGCTCTGCGTGCCAAGGTGCCCACGGTGCTGCAGACGGCCGGGAAGCTGGTGGCTGATAACTATGCCTCTGCCGAGCTGGGGGCCAAGATGGCCACCAAACTGAGCGGTCTGCAGAGCCGCTACTCCAGGGTGACCTCAGAAGTGGCCCTAGCCGAGATCCTGGGGGCTGACCTGCAGATGCTCTCCGGAGACCCACACC
10747	chr16	2120430	2120681	-	0	test	CGCCCAGGCTGGACTGCAGTGGCACAATCATAGCTCACTGCAGCCTCGACTTCCCTGGCTCAAGCGATCCTTCCTCCTCAGCCCCCCGAGTAGCTGGAACTACAGTTACACACTACCATGCCTGGCTGATTCTTTTTTTCCTTGTAGAGATGGGGTCTTGCTATGCTGTCCATCCTGGTCTCAAACTCCTGGCCTTCCCAAAGCACTGGGTTTACAGGCATAAGCCACCACACCCAGTTTCCTTTTCTTCT
10709	chr16	2118662	2118913	-	0	test	GCCAGTGGGGGGCTGGCATAGACCCTTCCCACCAGACCTGGTCCCCAACACCTGCCCCTGCCCTGCAGAAACCTGAGTGGGAACCCGTTTGAGTGTGACTGTGGCCTGGCGTGGCTGCCGCGATGGGCGGAGGAGCAGCAGGTGCGGGTGGTGCAGCCCGAGGCAGCCACGTGTGCTGGGCCTGGCTCCCTGGCTGGCCAGCCTCTGCTTGGCATCCCCTTGCTGGACAGTGGCTGTGGTGAGTGCCGGTG
6176	chr9	130701230	130701481	+	0	test	CCTCCAGGGGGCGCTGCAGCTCAGCACTGGGGCTGAGGCTGTCCTGGGAACAAAGGCAGGCTGGGTCTGGTTAAACAGCCCTTAGACAAACGTCCAATCAGTCCAGTGGATGTTATTGCTGGCAAAGCTTGCTCTGTCATCTGAGCCTCTAAATGGAATTTCACTGCTTTTCTTGTACGAGGTTGCATTATTGAAGCTATTTACTATTTTAAGGAGAGCATACTTCCTAAAACAAATTAGGATTTACTAGT
157	chr1	109606372	109606623	-	0	test	CATCCCCCATTTCCTCTAGTTAATGACATATGAAGGTTTGAAGGAAAGAATCCTAAATTTAACAATATGGCTTCTAGTCCTGATTTCTATCACCAAATTAGCTGTGTGACTTTGGGTAAGATACCTTTTACGTCTCTTAGCCTCGTCTGTGGAAAATTTGTCATTGATGCTAATCACTTTTCTCTAGCTACCTGAACCAATTAGAACGAATTACAGACCCTGAGTACCTCCCTAGTGAGCAAGATGTGCTC
9287	chr15	50254179	50254430	-	0	test	CTGAGCAGCTCAGACATGGGAGCTTGTCTCCAGCAACTGTAGGGAAGACAAGCTTCCAATGATCATATTTGGTTCCCGAGTCTTAGGCTTCAGCAGTTTCTGGGGGGTGATCATTCCAGGATGACTCTTTGCTGACAGGTTTCTGTGGCCCTTCTAGGCTCACTCCTCTGTGGAAAAGGCTGGTTTGATTTCCCTTGTGAAGATGAAATTTCTGCCTGTGGATGACAACTTCTCACTCCGAGGGGAAGCTC
2562	chr4	71761442	71761693	-	0	test	GGCACCAAATACCTAGACTGCACACAGCACGGGAACCCTGAGCCTGGTCCAAGAAACCACTTTTTCCTCCTAGGTCTCCAGGCCTGTGATGGGATGGGCTGCTGTGAAGACTTCTGACATGTCTTGGATACATTTTCCCCCATTGTATTGGGGATTAACATTCAGCTCCTTGTTACTTATGCAAATTTCTATAGCCAGCTTGAATTTCTCCTCAGAAAAAGGGCTTTTCTTTTCTCTCACATTGTCTGGCT
13526	chrX	147926900	147927151	+	0	test	GTGAGTCTTAAAGAAAAAAAAAGGGAGAGGATGAAAGGTGATTTAGGGACAGGATTGCTGTTTCAGAGAGTAGTCAGGGAAGATCTCTCAATGTCCAGGCCTGTGTTTTTCTCTCTACAGTTTTAAAGTGGACATAGGATATAGTGTTATTTATTTTCCAAAAATGTAACCAATCTCTACTAGGTGTAAGCAGCTGGGCTTGGTTTTGCAAGGGATCTTAATAGGGAACAGAAACACTGTGTATGCTTAGC
14007	chrX	147941253	147941504	+	0	test	GCAATTTACTTATAAGACTTCAGCAAATGAACTAGCTGTGTGACTTTAGGCAAGTCACTTCCCTACACTAGATGCTTCTTGAAATTCCTTTTGGCCAAAATGTTCTATACTTTAATGAAATGGAAACTGATCTACTTCATTTTCATAATGTTTTATTTGTATCCATACATATCAAAATTATAATGTAGCTGGTATTAAATGTTTTTAAAAAGCATTTCAACTCTATTTATGGCATTAAGAATTTTTTTGCT
3844	chr5	132677876	132678127	+	0	test	GAGAGGTCTGTTTCTTGGCCCCCAGAGCCCAAAGATACTGACACACTCTTACATTTCCAACTAGAATCAGGAACGAGGAGTGACTCTCAGTCAGTTCATTAAGTAAATGTCTTTCTAACCGCTCTGCCCATGGGACATCACGCCCCACAGGGGAAAGGGGAAGCTTCTGTAGCCTGGGATTCTGGTGCCTCAGTCTGGGTCTAGACTTTCCTGAAAAAACGTTAAAATATGAACTGCATTCCTAGAATTTA
1665	chr2	79159134	79159385	-	0	test	GCCCAGTGTATCTTGGATGCTGCTTTCCTGCCTCATGCTGCTGTCTCAGGTTCAAGGTGAGATTGCTTTGCCTCTAGCACTGGGTTCCCTATGAATCCTCAGAGCTAACAAGAGGAGGAAGGCTCCTGTGTGTCATGTGAGGTAATGACGTGGTGTCTAATGAACCTGCCTGCAGTTCTTGCATCATCTCTCCTTCCTTCAGGTTAACTTGCAGTGGGAGGCTCCATGGTGGTCCACTAACAGTGGAATGA
1965	chr2	162148203	162148454	-	0	test	ATACTGTTTCATCTCATAAATGTACAATATTTGTTTAGCTACTTATTTGGGAAGCTTAAGAGGTCTTAAAATTTATTCCACAAAGAATGTTGCAAACAACATTTTATATACACACTTTCCTACATTTCTTACTATTTTTTTGCACATAGGTTTCTAGTTGTAGAAATACTGTGTCAAAGGGCCTGAATGAGTTTAAACCTTTTTTAATATCTTGCCAACTTGCTTTCCAGAAAGTTCATACACATGCACAC
13349	chrX	147920710	147920961	+	0	test	GGATAAGCATAGGATGCTATGAGAATGCAGAGGACAGGAATTTAACCTAGACTTCTCAGTTCTCCCGTTGAAGTTGACATCTGAACTGAAATCTGGAAGACCAGTAAGAGATAGCTATGTAAAAAGAGGGGAAGGACAATAGGAAAGAAGGAATGGAGAGAGGCCCAGAAACTACAGAGTATGGCACAAGTAGTTTAGCATTGTTGGGTCACAAATTCTAAGGATATGAAAGATAAAGTTGAAGAAGTGGT
2417	chr4	71754299	71754550	-	0	test	CAAGACAAATAATTTTGTTATGCAATTATTGTTTTCTTACAGGGCCCTCTACTAAAGAAGGAACTATCTTCTTTCATTGACAAGGGACAAGAACTATGTGCAGATTATTCAGAAAATACATTTACTGAGTACAAGAAAAAGTAAGAAACTTGTTCTGGCTGTATCCTCCAAATTTATCAATAATATTTTCATAGTACTATGAATTGAAAGCATAGTTGAACACTTAAGCTTGTCTTCAGTGAACAACAACA
3909	chr5	132682286	132682537	+	0	test	CTAGAAGTGTAAGTAGTATGCACCCAAAATAGGCAAAACCTGCTGGCCTAGTGATAGAGACAACTCCCAGTCAGGCTAGACTGGAGGCCTTGGTTTTATAAGTGTTCAGGTGACAAGTGCCACAGTAGGCTTGATCAAGTAGACAGGCAGGCAAGACAAATGCTTACCAATGCAAGCTAATGAAATGTTTCTTTTGCAGAATTCCTGTCCTGTGAAGGAAGCCAACCAGAGTACGTTGGAAAACTTCTTGG
10099	chr16	2096971	2097222	-	0	test	GGTCCTTGCCGAGGGGGTCAGCAGCCCAGCCCCTACCCAAGACACCCACATGGAAACGGACCTGCTCAGCAGCCTGTGAGTGTCCGGCTCTCGGGGGAGGGGGGATTGCCAGAGGAGGGGCCGGGACTCAGGCCAGGCAGCCGTGGTTCCCGCCTGGGGTAGGGTGGGGTGGGGTGCCAGGGCAGGGCTGTGGCTGCACCACTTCACTTCTCTGAACCTCTGTTGTCTGTGGAAAGAGCCTCATGGGATCC
465	chr1	119508167	119508418	+	0	test	GTCGGGGAGAGAGAGTCGGGGAGAGAGACTCAGAGAGAGAGACAATGAGAAAGACAGAGAAATAATGAAAAATATATAGTGAGAGAGAGAGCATCAGTGAGAGAATGTGTGTGCACAGTGCACAGCACAGAGCAGAGACAGAGTAAGAGGCAGTATAAGGCCAGACATGCCTCATTTAGATTTTGCATATATGGCTTTTTTTTAAAAAAAAAAAAACAAAACATTTACCTCTGTTGCTCATCATCAAAAAG
12074	chr18	46089239	46089490	-	0	test	GGTAAATGGTAATGGGGACATTAGAATCTAATACGAGGATTAATAGTAAAATTTCAGCTGGGAAGTGAAGGGGGCCAGCCCCTCCACACCTGTGGGTATTTCTCATCAGGTGGGACGAGCTAGCTCAGTCGGTAGAGCATGGGACTCTTAATCCCAGGGTCGTGGGTTTGAGCCCCATGTTGGGCACCAGATGAAGGGGGCCAGCCCCTCCACACCTGTGAGTATTTCTCGTCAGGTGGGTTGAGAGACTG
5046	chr6	37173243	37173494	+	0	test	TTGAAATTCTGGAGAGCTTCACTCTCCAGTAGATTCTGTCACCCTTGGCTTAGAATTGTAGGTGAGTGATTTACACTTGAGCTGGCCTCATAAATCACATGGTTTGCACTTGAGCTTTCCTTGGGAGGTCAGAGGAAGGCATGTGTGAGCATATTAAGAAGAAAAGACAATCTGGCTTCTCCAAAAACTTTTTTAAAGGTACCAACAGAAACCTGATAATTCCTGGCTGTTTTGCCAGGGAGTAAAAAGTT
15939	chrX	154320935	154321186	+	0	test	TCTTGGTAGCCTTCCTCCTTCACAGAGAAAGATTAGCATGTGATTGGTTAAACCACGAATTTCCTTATGGTTCCATGAAGTTTGATTATTCAAGCCTTTTTCTTGAAAAAGCCATATTGTACTTTAAAAGTGTCAGACTCTCAGGGCAACTTTTGAAGTGCATCTCTCGGTGGTACTGTTGTTTCAGACAGTGAGAAACAGTTGCTTATCCCGAGTGCCCCTTGTGTGTTGAGCTGCGTGCAGGTTATGGA
8101	chr12	14883310	14883561	-	0	test	CTCCCAAAGTGCCGGGATTACAGGCATGAGCCACCACACCCAGCCAGCTGATTGCTGTTGAATAGCTGGATTTATAAAGACTGAGCATAGGAGGAAATGGCACATCACTCTCATTTTTAATTTATTCATTATTTTTATAGTGTTTAAACTGTTCATGTATCGGCAATCTAGTTATGCTTCATAAATCCTCAGGACAGAGAATTTCTCCTCAAAAGGAATTTAAAATCTACCAAGTAGAAATACAGAAATTA
11611	chr17	63930070	63930321	-	0	test	GAAAATGCACTGCTACATGAACAGCGCCTCCGGCAATGTGAGCTGGCTCTGGAAGCAGGAGATGGACGAGAATCCCCAGCAGCTGAAGCTGGAAAAGGGCCGCATGGAAGAGTCCCAGAACGAATCTCTCGCCACCCTCACCATCCAAGGCATCCGGTTTGAGGACAATGGCATCTACTTCTGTCAGCAGAAGTGCAACAACACCTCGGAGGTCTACCAGGGCTGCGGCACAGAGCTGCGAGTCATGGGTG
4333	chr6	32040279	32040530	+	0	test	GAGCACTGTGCGGCTGGGGCTGTGCTTGCCTCACCGGCACTCAGGCTCACTGGGTTGCTGAGGGAGCGGCTGGAGGCTGGGCAGCTGTGGGCTGCTGGGGCAGGACTCCACCCGATCATTCCCCAGATTCAGCAGCGACTGCAGGAGGAGCTAGACCACGAACTGGGCCCTGGTGCCTCCAGCTCCCGGGTCCCCTACAAGGACCGTGCACGGCTGCCCTTGCTCAATGCCACCATCGCCGAGGTGCTGCG
8046	chr11	123059706	123059957	-	0	test	TGAGGATGGAATCTTTGAGGTCAAGTCTACAGCTGGAGACACCCACTTGGGTGGAGAAGATTTTGACAACCGAATGGTCAACCATTTTATTGCTGAGTTTAAGCGCAAGCATAAGAAGGACATCAGTGAGAACAAGAGAGCTGTAAGACGCCTCCGTACTGCTTGTGAACGTGCTAAGCGTACCCTCTCTTCCAGCACCCAGGCCAGTATTGAGATCGATTCTCTCTATGAAGGAATCGACTTCTATACCT
5861	chr7	150859285	150859536	+	0	test	AGCCTACTGTTAACCGGAAGCCTTACCACTAACAGAAACATTCCGTGAACACATATTTTGTAGGTTCTGTGTGTCAGATACTGTGTTCTTACAATAAAGTAAGCTAGAGAAAAGAAAATGTTATTAATAAAATCCTGGCCGGGCACGGTGGCTCACGCCTGTAATCCCAGCACTTTGGGAAGCTGAGGCAGACAGATCACGAGGTCAGGAGATCAAGACCATCCTGGCTAACACGGTGAAACCTCATCTCT
11552	chr17	41582967	41583218	-	0	test	CCCTTAGTCCGCCCCCCCCATGGCACTCTCACGGCCCCACCATGTATCTAATGATCCTGTCCTTTTCTATTTTCACAGCCTCTCCTCCTCCCAGTTCTCCTCTGGATCGCAGTCATCCAGAGATGGTAAGACCCTCCTCCTCTGCAGGCCTGGGCTCCAGGCCACCCTCTGTACCCCAAGCAGGTCTAGGCATTGGCTAGGGGCTCCGTGAGGGGCTGAGCTCTAGTGCTGTCACCCAGTTTCCCTTGTGA
4026	chr6	29665362	29665613	+	0	test	TGGGATTACAGGTGTGAGCCACCGCACCTGGCCAATATTTGTGATTTTTATTGACGACAAAGTCAAAGGTTCTCTTCATATTATTGTGGTGTATCGCCTACAAGCATAATTAAAATAAACACTAAATTTCAGTTTAAAGTTTACTGAAAATAAATATGTATTTTTTATTCCCTATTTAAGCTTTGAATCCCCTGACTTCCTATACCATTACCACTGTCCTAGTTCAGGTTCATGTTGTTTTTTACTTTAAT
15341	chrX	154299922	154300173	+	0	test	ATAATGACTTTTTTTCCTTTGGGTAGATAGCCAGTAGTGGAATTTCTGGATTGAATGGTAGATAGATGGTAATCTACTACTTTTAGTATTTTTTTTTTTTTTTTTTGTCTTGAGACGGAGTCTTGCTCTGTCGCCCAGGCTGGAGGGCAGTGGCGCAATGTTGGCTCACTGCAACCTCTGCTGTCCGGGTTCAAGCGATTCTCGTGCCTTAGCCTCCCGAGTAGCTGGGACTACAGGCACACGCCACCACT
11913	chr18	31597950	31598201	+	0	test	TTCGTATTTCATTGCTTGTTATACATAAAAATATACTTTTCTTCTTCATGTTAGAAAATGCAAAGAATAGGAGGGTGGGGGAATCTCTGGGCTTGGAGACAGGAGACTTGCCTTCCTACTATGGTTCCATCAGAATGTAGACTGGGACAATACAATAATTCAAGTCTGGTTTGCTCATCTGTAAATTGGGAAGAATGTTTCCAGCTCCAGAATGCTAAATCTCTAAGTCTGTGGTTGGCAGCCACTATTGC
15482	chrX	154304611	154304862	+	0	test	AAGAAAATGGGGGAGGGACAGTCCCCATTTTCTTAATGGGGACTTTCTTAGTGAAATTTGAAATGAGAGATTATTAATATGAGCTGGGCCTAACATCTTTTTTTACTTGCAGTTGGGGCAGAGCTCCAGGGAGTTGGTTTTGTCCCCTGCTGTCTTGTCCCCTCCCCCTTTTTAATTAATTAGGGCCTGGCTGGACTCACATGCAGGCCGCTTTCTAAGAGCAGCAGTTACTTTGGCAGCATTCAGAAAGG
12809	chr22	20782872	20783123	+	0	test	AGCTTCACTCACAGGCAGGCAGGAGCTGTCTGGTACTTCAACCTCCAAGACACCTCCTGCTCATCTCATCCTGGCTGCTCTACCCACCAGCTAGAAACCTTGAACAAGTTACTTCACTTCTTTGTGCCTCTGTTTCCTCATATGTAAAAGAGGGATAACAAAACGCACACAACTTGCATGTTGCTAGGAGCAGAAATGAGATAATACAGGAAAGGTGCTGAGAAGAATGCCCGGCACATGGCCAGTTCTCA
8015	chr11	123058867	123059118	-	0	test	ACCAAGCAGACACAGACCTTCACTACCTATTCTGACAACCAGCCTGGTGTGCTTATTCAGGTATGTTTCTGTACTTCTCTTGTTTGGCTTACTGATAACAGATAAAGGGAAGTCTTGACTGACTCGCTATGATGATGGATTCCAAAACCATTCGTAGTTTCCACCAGAAAGTCTTATGTTGGCCAGTTCCTTCCTTGGATGTTTGAGCGACCATTCTTCCTTAGCAGGACCCTAGCACTGTCACAGACCTG
4149	chr6	30007865	30008116	+	0	test	ACAGCAACCTTGGGCACCAGGACTTTTCCTCCCGGGCCTTGTTCTCTGCCTCACACTCAATGTGTCGGAGTCTGACTCCAGCTCCTCTGAGTCCCTTGGCCTCCACTCAGATCAGGACCAGAAGTCCCTGCTACCCTGCTCAGAGACTAGAACTTTCCAAGGAATAGGAGATTATCCCAGGCGCCTGTGTCCAGGCTGGTGTCTGGGCTCTGTGCTCCCTTCCCCACCCCAGGTGTCCTATTCATCAGGAT
15775	chrX	154314007	154314258	+	0	test	GGGGGGACCATTTTTAGGAATACATAAAATATCAAAATTGACTAAAAGAGGTATAAAAACCTTAAATGGATCAATAGCCCTAGAAAAAAATGGAGAAAATTATCCAAGTGCTATACCCCCAAAGCACCAGGCTCACCAGGTTATATAAGACTGACTCATTTTTAACCTTTATGAAAAGAGTAATTTCTAAACTGTTTCAAATTTTTCCAAATACGGAGGAAGATGAAAATGTCATCCCGATTTACTTATGA
5081	chr6	42179071	42179322	+	0	test	CCTGGACCAGAATCTGGGCTCTGGGTTCCTCTGCTTGCTGCACCCGCAGCAGGGGCTCTGACTTCTCCTCACGTGGGCTCTGTCCCTGCCCCTGGCAAGAACCCGGTTCTGTGCTCTGGACTGCAGAAATGAACACCCTCCTCCCCCTGATTCCCTTTCTCTCTACCCCAGGGGAACTCTCCCTGGAAGAGTTTATAGAGGGCGTCCAGAAGGACCAGATGCTCCTGGACACACTGACACGAAGCCTGGAC
5638	chr7	99974493	99974744	-	0	test	TTTGATTCAGGCATGCAATGTGAAATAATCACATCATCAAAAATGAGGTATCCATCCCTTCAAGCTTTTATCGTTTGTGTTACAGACAATCCAATTATACTTTTTTGGTTATTTTAGTTTTTAAAAGTATTTGATTATTTATTTATTTATTTATTTTTGAGACAGAGTCTCACTCTGTCACCCAGGCAGGAGTGCAGTGGCATGATCTCGGCTCACTGCAACCTCCGCCTCCCAGGTTCAAGCAATTTTCC
"""

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `8`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `81b29e578672…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `NucleotideTransformerPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download. **This checkpoint ships its own model code**, and the manifest lists the two Python files the loader executes (`modeling_esm.py`, `esm_config.py`), so `verify_snapshot` has re-hashed them before `from_pretrained` imports them with `trust_remote_code=True` — the tokenizer loads natively with `trust_remote_code=False`. Digest verification proves the executed code is the pinned upstream code byte for byte; it is not a safety claim about that code, and this is the one row of the fleet where remote code runs at all (accepted for this row on 2026-09-20). The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "nt-v2-50m-multi-species",
  "modelId": "InstaDeepAI/nucleotide-transformer-v2-50m-multi-species",
  "revision": "81b29e5786726d891dbf929404ef20adca5b36f1",
  "files": [
    {
      "path": "README.md",
      "bytes": 6339,
      "sha256": "e526d7b98f106bc2ca9ba73fa166ff5fd62853812e9757a6692eeedf25e42923"
    },
    {
      "path": "config.json",
      "bytes": 1064,
      "sha256": "e20f497248c7cb264c7cd4582dbcfd52dc4cbf74a97fc711559b8c8f71c635db"
    },
    {
      "path": "esm_config.py",
      "bytes": 14876,
      "sha256": "a44e859baa08465ecdcd76b0a73f6e4fe011245a212931c403de1149ae9613ec"
    },
    {
      "path": "model.safetensors",
      "bytes": 223642688,
      "sha256": "17e75af297556ea56828716d8aa539e8f12b8b625547204b74171ab91aa33569"
    },
    {
      "path": "modeling_esm.py",
      "bytes": 58205,
      "sha256": "f2b003f45d2fa4f94e92d8bbef927b719dfdd7ab7c7a95743daa2b10ab140cb3"
    },
    {
      "path": "special_tokens_map.json",
      "bytes": 101,
      "sha256": "d6dc30bf018166daab248b0abf7efda6fd1b1e0a2d1bee5b31b23db2ebdaee77"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 129,
      "sha256": "253d338919eba938e50b776f3243cc739462c207fe64e3d2e81cc5e681bee45b"
    },
    {
      "path": "vocab.txt",
      "bytes": 28718,
      "sha256": "c00e0ad166d6ab3f7540ebc92270392e581bb3106763412f29034b49323e1052"
    }
  ],
  "totalBytes": 223752120
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = NucleotideTransformerPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. The promoter sample, its provenance and the split

`sample_dataset` returns the pinned draw from the inline block of the carried `samples.py` — 1,600 training, 200 validation and 400 test windows, half promoter and half not in every split, the train and validation windows drawn from the origin's *train* lists and the test windows from its *test* lists — and refuses to return it unless the block still hashes to `SAMPLE_DIGEST`. Each record carries its genomic interval (chromosome, 0-based half-open start/end, strand) so any window can be looked up in the reference. `validate_dataset` checks every record against the contract (alphabet, 12..6,000 bases, label 0/1, unique ids, both labels in the training split) and reports counts, lengths, GC fractions and a digest; `check_split_disjoint` asserts no sequence appears twice; the training split's table is written to `outputs/nucleotide_transformer_train.csv` (also a valid BYOD file).

Look for: three digests, promoter windows with a mean GC fraction around 0.62 against 0.48 for the negatives — the composition gap the GC baseline of Section 6 lives on — and four refusal probes (a duplicate id, a base outside A/C/G/T/N, a window too short, a training split with one label) each rejected before the model does anything.

In [ ]:
import hashlib
import json
import time

USE_BYOD = False  # @param {type:"boolean"}
SPLIT_SEED = 42  # @param {type:"integer"}

os.makedirs('outputs', exist_ok=True)
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    file_name, payload = next(iter(uploaded.items()))
    byod_csv = Path('work') / 'byod.csv'
    byod_csv.parent.mkdir(parents=True, exist_ok=True)
    byod_csv.write_bytes(payload)
    records = load_byod_dataset(byod_csv)
    splits = split_dataset(records, seed=SPLIT_SEED)
    data_source = 'BYOD (' + file_name + ')'
    raw_rows = {'byod': len(records)}
else:
    t0 = time.perf_counter()
    splits = sample_dataset()
    data_source = f'{CORPUS_NAME}: {CORPUS_REPO}@{CORPUS_REVISION[:12]} ({CORPUS_LICENSE})'
    raw_rows = {'origin_rows': CORPUS_ROWS, 'inline_records': sum(len(v) for v in splits.values()), 'pinned_digest': SAMPLE_DIGEST[:16] + '...', 'seconds': round(time.perf_counter() - t0, 3)}
dataset_manifests = {name: validate_dataset(part, require_both_labels=(name == 'train')) for name, part in splits.items()}
splits = {name: manifest['records'] for name, manifest in dataset_manifests.items()}
disjoint = check_split_disjoint(splits)
train_records, val_records, test_records = splits['train'], splits['validation'], splits['test']
write_dataset_csv(train_records, 'outputs/nucleotide_transformer_train.csv')
print({'data_source': data_source, 'raw_rows': raw_rows, 'splits': disjoint, 'reference': REFERENCE_GENOME, 'interval_bases': INTERVAL_BASES})
for name, manifest in dataset_manifests.items():
    by_label = {label: round(sum(gc_content(r['sequence']) for r in splits[name] if r['label'] == label) / max(1, sum(1 for r in splits[name] if r['label'] == label)), 4) for label in (0, 1)}
    print({name: {'n': manifest['n_records'], 'labels': manifest['label_counts'], 'bases': manifest['bases'], 'gc_fraction': manifest['gc_fraction'], 'mean_gc_by_label': by_label, 'n_with_N': manifest['n_with_N'], 'digest': manifest['digest'][:16] + '...'}})
example = train_records[0]
print({'example': {'id': example['id'], 'label': example['label'], 'region': example.get('region'), 'start': example.get('start'), 'end': example.get('end'), 'strand': example.get('strand'), 'sequence': example['sequence'][:60] + '...'}})

probes = {
    'duplicate id': [{**r, 'id': 'same'} for r in train_records[:8]],
    'base outside A/C/G/T/N': [{**train_records[0], 'sequence': train_records[0]['sequence'][:-1] + 'U'}, *train_records[1:8]],
    'too short': [{**train_records[0], 'sequence': 'ACGTACG'}, *train_records[1:8]],
    'one label only': [{**r, 'label': 1} for r in train_records[:8]],
}
for name, probe in probes.items():
    try:
        validate_dataset(probe, require_both_labels=True)
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})

## 5. Representations and masked prediction through the inference contract

The inference contract is exercised as the representation-only tutorial exercised it. A deterministic in-code pair of 300-base sequences with **exactly the same multiset of bases** — one carrying the 6-mer `ATTCCG` four times, the other the same letters permuted — is embedded beside the first four held-out windows; identical composition means no GC rule can tell the pair apart, so the cosine distance between their vectors is a small look at what the representation carries beyond composition. `validate_inputs` applies exactly the checks `embed` applies (a list of A/C/G/T/N strings of 12..6,000 bases) and returns an input manifest with per-sequence SHA-256s; a sequence with a `U` is validated too and its rejection recorded as a finding. `embed` returns one 512-d **mean-pooled** vector per sequence — a representation, not a prediction; `token_counts` shows the 6-mer tokenisation (251 bases → 47 tokens); `predict_masked` returns the model's distribution at one masked position — its pre-training objective, which is what `<mask>` inside a repeated `ATTCCG` context should recover — and `predict` is refused until an adapter exists, which is the contract's point: the checkpoint is not a classifier.

In [ ]:
import csv
import random

rng = random.Random(7)
MOTIF = 'ATTCCG'
filler = ''.join(rng.choice('ACGT') for _ in range(300 - 4 * len(MOTIF)))
positions = sorted(rng.sample(range(0, len(filler), 30), 4))
with_motif = ''
cursor = 0
for p in positions:
    with_motif += filler[cursor:p] + MOTIF
    cursor = p
with_motif += filler[cursor:]
permuted = list(with_motif)
rng.shuffle(permuted)
permuted = ''.join(permuted)
assert sorted(with_motif) == sorted(permuted) and len(with_motif) == 300
probe_sequences = [with_motif, permuted] + [r['sequence'] for r in test_records[:4]]
probe_names = ['motif_x4', 'same_bases_permuted'] + [r['id'] for r in test_records[:4]]
print({'ceilings': {'MIN_BASES': MIN_BASES, 'MAX_BASES': MAX_BASES, 'MAX_TOKENS': MAX_TOKENS, 'MAX_SEQUENCES': MAX_SEQUENCES, 'MIN_RECORDS': MIN_RECORDS, 'MAX_RECORDS': MAX_RECORDS, 'device': pipe.device, 'remote_code_executed': pipe.remote_code_executed}})
input_manifest = validate_inputs(probe_sequences, names=probe_names)
input_manifest['findings'] = []
try:
    validate_inputs(['ACGUACGUACGU'])
except ValueError as exc:
    input_manifest['findings'].append({'input': 'uracil-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/nucleotide_transformer_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print({'input_manifest': {k: input_manifest[k] for k in ('n_sequences', 'bases', 'gc_fraction', 'n_with_N')}, 'findings': len(input_manifest['findings'])})

t0 = time.perf_counter()
vectors = pipe.embed(probe_sequences)
embed_seconds = round(time.perf_counter() - t0, 3)
counts = pipe.token_counts(probe_sequences)


def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b, strict=True))
    return dot / ((sum(x * x for x in a) ** 0.5) * (sum(y * y for y in b) ** 0.5))


pair_cosine = cosine(vectors[0], vectors[1])
held_out_cosines = [round(cosine(vectors[0], v), 4) for v in vectors[2:]]
checks = {'one_vector_per_sequence': len(vectors) == len(probe_sequences), 'dimension': all(len(v) == HIDDEN_SIZE for v in vectors), 'pooling': POOLING}
if not all(v is True or isinstance(v, str) for v in checks.values()):
    raise RuntimeError(f'embed output failed a sanity check: {checks}')
print({'embed': {'n': len(vectors), 'dimension': len(vectors[0]), 'seconds': embed_seconds, 'tokens': dict(zip(probe_names, counts, strict=True)), 'checks': checks}})
print({'motif_vs_permuted_cosine': round(pair_cosine, 4), 'motif_vs_held_out_cosines': held_out_cosines, 'reading': 'same bases, different order: the distance between the pair is what the model carries beyond composition (sample-sanity, not a measurement)'})
with open('outputs/nucleotide_transformer_embeddings.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['name', 'bases', 'tokens'] + [f'dim_{k}' for k in range(HIDDEN_SIZE)])
    for name, sequence, count, vector in zip(probe_names, probe_sequences, counts, vectors, strict=True):
        writer.writerow([name, len(sequence), count] + [f'{x:.6f}' for x in vector])

masked = pipe.predict_masked(MOTIF * 3 + '<mask>' + MOTIF * 3, top_k=5)
print({'masked_prediction': masked, 'reading': 'the pre-training objective: the distribution over 6-mers at the masked position'})
try:
    pipe.predict(probe_sequences[:1])
except RuntimeError as exc:
    print({'predict_before_adapt': 'refused', 'message': str(exc)})

## 6. Baselines and the frozen model on the held-out windows

Three references frame the adaptation, each scored by `classification_metrics` (carried in `metrics.py`): **accuracy** and **MCC** — the Matthews correlation, 0 for any constant or chance predictor and symmetric in the two classes, which is the headline metric here — with per-class precision, recall and F1. The **majority** baseline answers the training majority label (0.5 on a balanced test split, MCC 0). The **GC-threshold** baseline is fitted on the training split alone — the midpoint threshold and direction that maximise training accuracy — and reads only composition: the build record measured **0.715 / 0.431** with the rule *GC ≥ 0.560 → promoter*, which is the honest floor for a sequence model on this task. The **frozen probe** (`pipe.linear_probe`) fits an L2-penalised logistic regression on the standardised mean-pooled embeddings of the training windows and scores the test windows; the model's weights are untouched, so this is what the pre-trained representation alone knows — the build record measured **0.815 / 0.632**. Expect the probe well above the GC rule: the representation carries more than composition. The cell asserts that ordering.

In [ ]:
METRICS = ('accuracy', 'mcc')
baseline_majority = majority_baseline(train_records, test_records)
baseline_gc = gc_threshold_baseline(train_records, test_records)
print({'majority_baseline': {k: baseline_majority[k] for k in METRICS}, 'label': baseline_majority['label'], 'n': baseline_majority['n']})
print({'gc_threshold_baseline': {k: baseline_gc[k] for k in METRICS}, 'threshold': baseline_gc['threshold'], 'rule': baseline_gc['rule'], 'train_accuracy': baseline_gc['train_accuracy']})
t0 = time.perf_counter()
frozen_probe = pipe.linear_probe(train_records, test_records)
print({'frozen_probe_test': {k: frozen_probe[k] for k in METRICS}, 'positive': frozen_probe['positive'], 'negative': frozen_probe['negative'], 'train_accuracy': frozen_probe['train_accuracy'], 'probe': frozen_probe['probe'], 'seconds': round(time.perf_counter() - t0, 1)})
assert frozen_probe['mcc'] > baseline_gc['mcc'] > baseline_majority['mcc']

## 7. Bounded fine-tuning of the head and the last two blocks

`pipe.adapt` builds a fresh **mean-pooled head** (a 512→512 `tanh` layer and a 512→2 output, 263,682 parameters, initialised from `SEED` so two runs start alike) and trains it together with the **last `TRAINED_LAYERS` encoder blocks** — two by default, 8,660,482 of 53.8 M parameters in all — while the embeddings and the first ten blocks stay frozen. Cross-entropy over the two labels, AdamW without weight decay at a fixed learning rate, gradient clipping at 1.0, batches of 16 in seeded order, no scheduler. Epoch 0 records the untrained head's validation metrics (chance); every epoch is scored on the 200 validation windows and the epoch with the highest **validation MCC** is kept (ties: accuracy, then the earlier epoch); on any exception the frozen weights are restored.

Watch the training loss fall from about 0.51 to 0.14 over six epochs while the validation MCC peaks early — the build record kept **epoch 2** (validation 0.805 / 0.621) and the later epochs were lower: 1,600 windows are enough to overfit two blocks, which is what the selection rule is for. The build record's counter-examples — the CLS-token head the upstream classification class uses (0.7725 / 0.552 on 640 windows, against 0.8275 / 0.658 for the mean-pooled head), four blocks (no better), and the whole encoder (0.83 / 0.686 but peaking at epoch 1 with a 200 MB adapter) — are why the default is two blocks under a mean-pooled head. Training is batched at a small learning rate, so the validation curve is noisy and the same recipe lands within a few hundredths of the numbers below from one GPU run to the next.

In [ ]:
EPOCHS = 6  # @param {type:"integer"}
LEARNING_RATE = 3e-5  # @param {type:"number"}
TRAINED_LAYERS = DEFAULT_TRAINED_LAYERS  # @param {type:"integer"}
SEED = 0  # @param {type:"integer"}


def report(entry):
    row = {'epoch': entry['epoch'], 'train_loss': None if entry['train_loss'] is None else round(entry['train_loss'], 4)}
    if entry.get('val'):
        row.update({'val_accuracy': entry['val']['accuracy'], 'val_mcc': entry['val']['mcc']})
    if 'note' in entry:
        row['note'] = entry['note']
    print(row)


t0 = time.perf_counter()
adapt_result = pipe.adapt(train_records, val_records, epochs=EPOCHS, lr=LEARNING_RATE, layers=TRAINED_LAYERS, seed=SEED, progress=report)
adapt_seconds = round(time.perf_counter() - t0, 1)
print({'trainable_parameters': adapt_result['n_trainable'], 'head_parameters': adapt_result['n_head'], 'total_parameters': adapt_result['n_total'], 'layers': adapt_result['layers'], 'best_epoch': adapt_result['best_epoch'], 'selection': adapt_result['selection'], 'loss': adapt_result['loss'], 'train_accuracy': adapt_result['train_accuracy'], 'seconds': adapt_seconds})

## 8. Held-out evaluation

The test windows were never used for training or epoch selection, and no sequence appears in two splits. The adapted model is scored by `pipe.evaluate` exactly as the probe and the baselines were in Section 6, the four systems are put side by side, and the per-class rows are read. Read it in this order: the **MCC** first (the build record measured 0.632 → **0.693**, past the GC rule's 0.431 and the majority's 0), then accuracy (0.815 → 0.8425), then the per-class rows — the adapted classifier is conservative on the promoter class (precision 0.905, recall 0.765) and permissive on the negatives (recall 0.92), which a threshold on the probabilities could trade but nothing here does. The cell asserts the adapted MCC is above the frozen probe's and above both baselines, and writes `evaluation_report` — the structured verdict — to `outputs/nucleotide_transformer_evaluation_report.json`. Four hundred windows from one seeded draw give **no dispersion estimate**; the deltas are sample-sanity evidence that the adaptation contract works, not a benchmark, and a gain on this benchmark's window convention says nothing about promoters in *your* sequences until you measure it.

In [ ]:
adapted_test = pipe.evaluate(test_records)
adapted_val = pipe.evaluate(val_records)
comparison = {metric: {'majority': baseline_majority[metric], 'gc_threshold': baseline_gc[metric], 'frozen_probe': frozen_probe[metric], 'adapted': adapted_test[metric]} for metric in METRICS}
comparison['delta_vs_frozen_probe'] = {metric: round(adapted_test[metric] - frozen_probe[metric], 4) for metric in METRICS}
comparison['delta_vs_gc_threshold'] = {metric: round(adapted_test[metric] - baseline_gc[metric], 4) for metric in METRICS}
comparison['per_class'] = {'frozen_probe': {'positive': frozen_probe['positive'], 'negative': frozen_probe['negative']}, 'adapted': {'positive': adapted_test['positive'], 'negative': adapted_test['negative']}}
comparison['confusion_adapted'] = adapted_test['confusion']
for key, row in comparison.items():
    print({key: row})
verdict = evaluation_report(adapted_test, frozen_probe, [baseline_majority, baseline_gc], sample_kind='BYOD' if USE_BYOD else 'pinned promoter sample')
print({'verdict': verdict})
evaluation_report_payload = {
    'model': {'id': MODEL_ID, 'revision': MODEL_REVISION, 'key': MODEL_KEY, 'license': MODEL_LICENSE},
    'data_source': data_source,
    'dataset_digests': {name: manifest['digest'] for name, manifest in dataset_manifests.items()},
    'splits': disjoint,
    'baselines': {'majority': baseline_majority, 'gc_threshold': baseline_gc},
    'frozen_probe_test': frozen_probe,
    'validation_metrics': adapted_val,
    'test_metrics': adapted_test,
    'comparison': comparison,
    'verdict': verdict,
    'adaptation': {k: v for k, v in adapt_result.items() if k not in ('history', 'trainable_names')},
    'history': adapt_result['history'],
    'adaptation_seconds': adapt_seconds,
}
with open('outputs/nucleotide_transformer_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report_payload, f, indent=2, ensure_ascii=False)
assert adapted_test['mcc'] > frozen_probe['mcc']
assert verdict['adapted_beats_baselines']
print({'report': 'outputs/nucleotide_transformer_evaluation_report.json', 'mcc_gain_over_frozen': verdict['mcc_gain_over_frozen']})

## 9. Look at the predictions, export the adapter and reload it

Eight held-out windows are printed with their GC fraction, the GC rule's answer, the adapted classifier's label and promoter probability, and the truth — so the numbers can be checked by eye: look for windows the GC rule gets wrong that the classifier gets right, and for the probabilities near 0.5 that the accuracy hides. The pair from Section 5 is classified too (both are synthetic, so there is no truth — only the observation that identical composition can receive different probabilities).

`pipe.save_artifact` writes the trained tensors — the head and the last two blocks, about 35 MB — as `adapter.safetensors`, with a `manifest.json` recording the artifact format, **the licence the adapter inherits (`cc-by-nc-sa-4.0`)**, the base model id and revision, the digest of the base `model.safetensors`, the two remote-code files the base executes, the tensor names, the file size and SHA-256, the training configuration and the epoch history (OUT8). `NucleotideTransformerPipeline.from_artifact` re-verifies the base snapshot (including the model code), checks the artifact manifest, its digest, its licence and its exact tensor set **before** deserialising, refuses any encoder tensor outside the recorded blocks, and overlays the tensors onto a freshly loaded base — a new object from files, not the in-memory model (VER2). The cell asserts identical labels and near-identical probabilities on 64 test windows (VER4).

In [ ]:
import shutil

shown = test_records[:8]
shown_predictions = pipe.predict([r['sequence'] for r in shown])
for record, label, probabilities in zip(shown, shown_predictions['labels'], shown_predictions['probabilities'], strict=True):
    gc = gc_content(record['sequence'])
    gc_answer = int(gc >= baseline_gc['threshold']) if baseline_gc['rule'].startswith('GC >=') else int(gc < baseline_gc['threshold'])
    print({'id': record['id'], 'region': f"{record.get('region')}:{record.get('start')}-{record.get('end')}{record.get('strand', '')}", 'gc': round(gc, 3), 'gc_rule': gc_answer, 'adapted': label, 'p_promoter': probabilities[1], 'truth': record['label']})
pair = pipe.predict(probe_sequences[:2])
print({'synthetic_pair': dict(zip(probe_names[:2], [p[1] for p in pair['probabilities']], strict=True)), 'reading': 'same bases, different order; no truth exists for these'})

artifact_dir = Path('outputs/nucleotide_transformer_adapter')
shutil.rmtree(artifact_dir, ignore_errors=True)
pipe.save_artifact(artifact_dir, metadata={'tutorial': 'nucleotide_transformer', 'data_source': data_source})
artifact_manifest = json.loads((artifact_dir / 'manifest.json').read_text(encoding='utf-8'))
print({'artifact': str(artifact_dir), 'format': artifact_manifest['format'], 'license': artifact_manifest['license'], 'tensors': len(artifact_manifest['tensors']), 'bytes': artifact_manifest['files'][0]['bytes'], 'sha256': artifact_manifest['files'][0]['sha256'][:16] + '...', 'base_remote_code_files': artifact_manifest['base']['remote_code_files']})

reloaded = NucleotideTransformerPipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR, device=pipe.device)
parity_sequences = [r['sequence'] for r in test_records[:64]]
before = pipe.predict(parity_sequences)
after = reloaded.predict(parity_sequences)
parity = {'identical_labels': sum(a == b for a, b in zip(before['labels'], after['labels'], strict=True)), 'of': len(parity_sequences), 'max_probability_difference': max(abs(x - y) for pa, pb in zip(before['probabilities'], after['probabilities'], strict=True) for x, y in zip(pa, pb, strict=True))}
print({'reload_parity': parity, 'reloaded_best_epoch': reloaded.adapter['best_epoch'], 'reloaded_layers': reloaded.adapter['layers']})
assert parity['identical_labels'] == parity['of'] and parity['max_probability_difference'] < 1e-4

result_payload = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'remote_code': {'executed': pipe.remote_code_executed, 'files': list(REMOTE_CODE_FILES), 'verified_before_import': True},
    'snapshot': {'path': str(WEIGHTS_DIR), 'files': snapshot['files'], 'total_bytes': snapshot.get('total_bytes'), 'fetched_this_run': fetched, 'weight_file': WEIGHT_FILE, 'weight_format': 'safetensors, digest-verified', 'weight_sha256': pipe.weight_sha256},
    'data_source': data_source,
    'corpus': {'name': CORPUS_NAME, 'repo': CORPUS_REPO, 'revision': CORPUS_REVISION, 'dataset': CORPUS_DATASET, 'license': CORPUS_LICENSE, 'files': {k: {'bytes': v[0], 'sha256': v[1]} for k, v in CORPUS_FILES.items()}, 'reference': REFERENCE_GENOME, 'sample_digest': SAMPLE_DIGEST, 'split_digests': SAMPLE_SPLIT_DIGESTS},
    'inference_contract': {'input_manifest': input_manifest, 'embedding_dimension': HIDDEN_SIZE, 'pooling': POOLING, 'motif_vs_permuted_cosine': round(pair_cosine, 6), 'masked_prediction': masked, 'output_files': ['outputs/nucleotide_transformer_embeddings.csv', 'outputs/nucleotide_transformer_input_manifest.json']},
    'comparison': comparison,
    'verdict': verdict,
    'artifact': {'dir': str(artifact_dir), 'sha256': artifact_manifest['files'][0]['sha256'], 'bytes': artifact_manifest['files'][0]['bytes'], 'tensors': len(artifact_manifest['tensors']), 'license': artifact_manifest['license']},
    'reload_parity': parity,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'device': pipe.device, 'dtype': 'float32'},
}
with open('outputs/nucleotide_transformer_result.json', 'w', encoding='utf-8') as handle:
    json.dump(result_payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

A GC threshold fitted on the training windows classifies 71.5 % of the held-out windows (MCC 0.431), a logistic probe on the frozen mean-pooled embeddings 81.5 % (0.632), and a bounded fine-tuning of a mean-pooled head plus the last two encoder blocks on 1,600 windows 84.25 % (**0.693**), with a 35 MB adapter that reloads prediction-for-prediction and carries the base licence in its manifest. That is the claim: the adaptation contract works end to end on a real, licence-traced labelled set behind a verified remote-code perimeter, and the numbers it produces are read against two non-neural baselines and the frozen representation rather than in isolation.

The test split is 400 windows from one seeded draw, the validation split that picks the epoch is 200, and the task is one benchmark's convention — 251-base windows centred on EPD non-TATA promoters against windows that contain none. So a gain here says the last two blocks learned that convention's promoter signal beyond composition, not that the classifier finds promoters in arbitrary sequence, handles TATA promoters or other species, or that its probabilities are calibrated (they are not; the per-class rows show a conservative promoter class). The 0.06 MCC gain over the frozen probe is a few times the run-to-run spread the build record observed, not a large margin, and the whole-encoder alternative that scores similarly peaks at epoch 1 and costs a 200 MB adapter — bounded is the point, not the ceiling.

Three things to carry to real data. **Baselines first:** fit the GC rule and the frozen probe on *your* labels before reading any fine-tuned number; if the probe already matches the fine-tuning, the representation was the answer. **Labelling convention:** the windows the head learns from define what "promoter" means; keep the window length and the negative-sampling rule fixed between training and use, or the number means nothing. **Leakage:** keep every sequence in one split (the contract de-duplicates by exact bases) and split by chromosome or locus when windows overlap — the pinned sample inherits the origin's train/test separation, a BYOD CSV does not.

Successful execution proves that the recorded repository revision's package, carried in this standalone notebook, can acquire and digest-verify the pinned model snapshot including the model code it executes, validate the demonstrated dataset contract without leakage on a licence-traced inline sample, execute the inference contract and a bounded fine-tuning, evaluate against two non-neural baselines and the frozen representation on a sequence-disjoint split, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, classification quality on any other task or window convention, calibration of the probabilities, or production fitness; and nothing it produces may be used commercially under the weights' licence.

**Optional experiments (they do not affect the default path):** set `TRAINED_LAYERS = 0` and read how much of the gain the head alone recovers; set `TRAINED_LAYERS = 12` and watch the validation MCC peak at epoch 1 and the adapter grow to the whole encoder; raise `EPOCHS` and watch the selection rule keep an early epoch while the training loss keeps falling; or bring your own CSV through BYOD and read the two baselines before the adapted number.

## References

- Repository README: https://github.com/kurtvalcorza/nucleotide-transformer-genomics-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/nucleotide-transformer-genomics-pipeline/blob/main/MODEL_CARD.md
- Weight provenance and the remote-code perimeter: https://github.com/kurtvalcorza/nucleotide-transformer-genomics-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/InstaDeepAI/nucleotide-transformer-v2-50m-multi-species
- Upstream code and licence: https://github.com/instadeepai/nucleotide-transformer (CC BY-NC-SA 4.0)
- Nucleotide Transformer (Dalla-Torre et al., Nature Methods 2024): https://doi.org/10.1038/s41592-024-02523-z
- Genomic Benchmarks (Grešová et al., BMC Genomic Data 2023; Apache-2.0): https://github.com/ML-Bioinfo-CEITEC/genomic_benchmarks — human non-TATA promoters from the Eukaryotic Promoter Database
- GRCh38 reference genome via Ensembl REST: https://rest.ensembl.org/
- DIMER Notebook Specification 2.0 and Model Card Specification 1.1 (fleet specs in the ml-worker repository)